In [ ]:
# SATIPO research workflow — original development notebook
# This notebook requires the project source datasets and Google Colab/Drive setup.
# Execution is NOT verified as a clean end-to-end run.


from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from io import BytesIO
import zipfile
import pandas as pd
import numpy as np
import re
import unicodedata
import hashlib

# =====================================================
# PROJECT CONFIGURATION
# =====================================================

ROOT = Path('/content/drive/MyDrive/SATIPO_FOREST_PROJECT')

DIRS = {
    'raw': ROOT / '01_RAW_DATA',
    'master': ROOT / '02_DATA_MASTER',
    'analysis': ROOT / '03_ANALYSIS',
    'remote': ROOT / '04_REMOTE_SENSING',
    'outputs': ROOT / '05_OUTPUTS'
}

for folder in DIRS.values():
    folder.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 50)

# =====================================================
# ORIGINAL ZIP AND VERIFIED INTERNAL PATHS
# =====================================================

ZIP_PATH = DIRS['raw'] / 'ESTRUCTURA_SATIPO.zip'

INNER_ZIP = (
    'ESTRUCTURA_SATIPO/'
    'ESTRUCTURA-20260917T213837Z-1-001.zip'
)

INVENTORIES = {
    'Mature': 'ESTRUCTURA/BOSQUE_MADURO.xlsx',
    'Late_secondary': (
        'ESTRUCTURA/BOSQUE_SECUNDARIO_TARDIO.xlsx'
    )
}

if not ZIP_PATH.exists():
    raise FileNotFoundError(ZIP_PATH)

FILES = {}

# Read nested ZIP directly
with zipfile.ZipFile(ZIP_PATH, 'r') as outer_zip:

    inner_bytes = outer_zip.read(INNER_ZIP)

    with zipfile.ZipFile(
        BytesIO(inner_bytes), 'r'
    ) as inner_zip:

        for forest, internal_path in INVENTORIES.items():

            destination = (
                DIRS['raw'] / Path(internal_path).name
            )

            destination.write_bytes(
                inner_zip.read(internal_path)
            )

            FILES[forest] = destination

            print(f'EXTRACTED: {forest}')
            print(f'File: {destination}')
            print(f'Size: {destination.stat().st_size:,} bytes')
            print('-' * 60)

print('\nEXTRACTION COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# LOAD ORIGINAL INVENTORIES
# =====================================================

EXPECTED_COLUMNS = [
    'Parcela',
    'Sub Parcela',
    'N° Árbol',
    'Nombre común',
    'Nombre científico',
    'Familia',
    'DAP (cm)',
    'Altura T. (m)',
    'Observación'
]

EXPECTED_ROWS = {
    'Mature': 483,
    'Late_secondary': 579
}

raw_data = {}

for forest, path in FILES.items():

    df = pd.read_excel(
        path,
        sheet_name='Hoja 1'
    )

    # Validate exact structure
    assert list(df.columns) == EXPECTED_COLUMNS, (
        f'Unexpected columns in {forest}'
    )

    assert len(df) == EXPECTED_ROWS[forest], (
        f'Unexpected number of records in {forest}'
    )

    raw_data[forest] = df.copy()

    print('\n' + '=' * 65)
    print(f'FOREST TYPE: {forest}')
    print(f'Rows: {len(df)}')
    print(f'Columns: {df.shape[1]}')
    print(f'Plot: {df["Parcela"].unique()}')
    print(
        f'Subplots: {df["Sub Parcela"].nunique()}'
    )

    print('\nFIRST RECORDS:')
    display(df.head())

    print('\nMISSING VALUES:')
    display(
        df.isna().sum().to_frame('Missing')
    )

    print('\nOBSERVATIONS:')
    display(
        df['Observación']
        .value_counts(dropna=False)
        .to_frame('Frequency')
    )

print('\nBOTH INVENTORIES VALIDATED')


In [ ]:

# =====================================================
# TREE-LEVEL MASTER DATASET
# =====================================================

COLUMN_MAP = {
    'Parcela': 'plot_id',
    'Sub Parcela': 'subplot_number',
    'N° Árbol': 'tree_number',
    'Nombre común': 'common_name_raw',
    'Nombre científico': 'species_raw',
    'Familia': 'family_raw',
    'DAP (cm)': 'dbh_cm',
    'Altura T. (m)': 'height_m',
    'Observación': 'observations_raw'
}

def clean_text(value):

    if pd.isna(value):
        return pd.NA

    value = unicodedata.normalize(
        'NFC', str(value)
    )

    value = re.sub(r'\s+', ' ', value).strip()

    return value if value else pd.NA


def normalize_observation(value):

    if pd.isna(value):
        return ''

    value = unicodedata.normalize(
        'NFKD', str(value)
    )

    value = ''.join(
        c for c in value
        if not unicodedata.combining(c)
    )

    return value.strip().lower()


frames = []

for forest, raw in raw_data.items():

    df = raw.rename(columns=COLUMN_MAP).copy()

    # Provenance
    df['forest_type'] = forest
    df['source_file'] = FILES[forest].name
    df['source_sheet'] = 'Hoja 1'
    df['source_excel_row'] = np.arange(
        2, len(df) + 2
    )

    # Identifiers
    for col in [
        'plot_id',
        'subplot_number',
        'tree_number'
    ]:

        df[col] = pd.to_numeric(
            df[col],
            errors='raise'
        ).astype('Int64')

    # Structural measurements
    for col in ['dbh_cm', 'height_m']:

        df[col] = pd.to_numeric(
            df[col],
            errors='raise'
        )

    # Preserve original taxonomic labels
    df['species_clean'] = (
        df['species_raw'].apply(clean_text)
    )

    df['family_clean'] = (
        df['family_raw'].apply(clean_text)
    )

    df['common_name_clean'] = (
        df['common_name_raw'].apply(clean_text)
    )

    # Standardized observations
    df['observations_clean'] = (
        df['observations_raw']
        .apply(normalize_observation)
    )

    obs = df['observations_clean']

    # Original inventory classification
    df['status_inventory'] = np.where(
        obs.eq('muerto'),
        'dead',
        'not_recorded_dead'
    )

    # Ambiguous condition flags
    df['flag_cut'] = obs.str.contains(
        r'\bcortado\b',
        regex=True
    )

    df['flag_fallen'] = obs.str.contains(
        r'\bcaido\b',
        regex=True
    )

    df['flag_status_review'] = (
        df['flag_cut'] | df['flag_fallen']
    )

    # Additional ecological observations
    df['flag_liana'] = obs.str.contains(
        'liana',
        regex=False
    )

    df['flag_decay'] = obs.str.contains(
        'pudricion',
        regex=False
    )

    df['flag_crown_damage'] = obs.str.contains(
        r'sin apice|sin copa|copa seca|sin guia',
        regex=True
    )

    # Unique spatial identifiers
    df['subplot_id'] = (
        'P'
        + df['plot_id'].astype(str).str.zfill(2)
        + '_S'
        + df['subplot_number']
          .astype(str).str.zfill(2)
    )

    df['tree_id'] = (
        df['subplot_id']
        + '_T'
        + df['tree_number']
          .astype(str).str.zfill(3)
    )

    # Sampling metadata
    df['census_year'] = 2019
    df['plot_area_ha'] = 1.0
    df['subplot_area_m2'] = 400
    df['subplot_area_ha'] = 0.04

    # Basal area (m2)
    df['basal_area_m2'] = (
        np.pi * (df['dbh_cm'] / 200)**2
    )

    # Slenderness ratio: height / diameter
    # Both measurements expressed in meters
    df['slenderness_ratio'] = (
        df['height_m'] / (df['dbh_cm'] / 100)
    )

    # Pending taxonomic validation
    df['species_accepted'] = pd.NA
    df['genus_accepted'] = pd.NA
    df['family_accepted'] = pd.NA
    df['taxonomic_status'] = 'pending'

    # Pending biomass and functional traits
    df['wood_density_g_cm3'] = np.nan
    df['wood_density_source'] = pd.NA
    df['agb_kg'] = np.nan
    df['carbon_kg'] = np.nan

    frames.append(df)


master = pd.concat(
    frames,
    ignore_index=True
)

master = master.sort_values(
    ['plot_id', 'subplot_number', 'tree_number']
).reset_index(drop=True)

# Validate identifiers
assert len(master) == 1062
assert master['tree_id'].is_unique
assert master['subplot_id'].nunique() == 50

print('TREE MASTER CREATED')
print(f'Records: {len(master)}')
print(f'Variables: {master.shape[1]}')
print(f'Unique trees: {master.tree_id.nunique()}')

display(master.head(10))


In [ ]:

# =====================================================
# QUALITY ASSURANCE / QUALITY CONTROL
# =====================================================

master['flag_missing_dbh'] = (
    master['dbh_cm'].isna()
)

master['flag_missing_height'] = (
    master['height_m'].isna()
)

master['flag_missing_taxonomy'] = (
    master['species_clean'].isna()
)

master['flag_invalid_dbh'] = (
    master['dbh_cm'].notna()
    & master['dbh_cm'].lt(10)
)

master['flag_invalid_height'] = (
    master['height_m'].notna()
    & master['height_m'].le(0)
)

master['flag_invalid_subplot'] = (
    ~master['subplot_number'].between(1, 25)
)

# Separate inclusion criteria by analysis

# Floristic inventory
master['include_floristic'] = (
    master['status_inventory'].ne('dead')
    & master['dbh_cm'].ge(10)
    & master['species_clean'].notna()
    & ~master['flag_invalid_subplot']
)

# Structural inventory
master['include_structural'] = (
    master['status_inventory'].ne('dead')
    & master['dbh_cm'].ge(10)
    & master['height_m'].gt(0)
    & ~master['flag_invalid_subplot']
)

# Conservative structural subset
master['include_structural_strict'] = (
    master['include_structural']
    & ~master['flag_status_review']
)

# Summary table
qa_summary = master.groupby('forest_type').agg(

    total_records=('tree_id', 'size'),

    subplots=('subplot_id', 'nunique'),

    dead_records=(
        'status_inventory',
        lambda x: x.eq('dead').sum()
    ),

    floristic_records=(
        'include_floristic', 'sum'
    ),

    structural_records=(
        'include_structural', 'sum'
    ),

    strict_records=(
        'include_structural_strict', 'sum'
    ),

    cut_records=('flag_cut', 'sum'),

    fallen_records=('flag_fallen', 'sum'),

    liana_records=('flag_liana', 'sum'),

    decay_records=('flag_decay', 'sum'),

    missing_dbh=('flag_missing_dbh', 'sum'),

    missing_height=('flag_missing_height', 'sum'),

    missing_taxonomy=(
        'flag_missing_taxonomy', 'sum'
    ),

    invalid_dbh=('flag_invalid_dbh', 'sum'),

    invalid_height=('flag_invalid_height', 'sum'),

    invalid_subplot=('flag_invalid_subplot', 'sum')

)

display(qa_summary.T)

# Expected original inventory counts
expected_structural = {
    'Mature': 464,
    'Late_secondary': 545
}

for forest, expected in expected_structural.items():

    observed = int(
        qa_summary.loc[
            forest, 'structural_records'
        ]
    )

    assert observed == expected, (
        f'{forest}: expected {expected}, '
        f'found {observed}'
    )

print('\nALL STRUCTURAL CHECKS PASSED')


In [ ]:

# =====================================================
# STRUCTURAL SUMMARY
# =====================================================

inventory = master.loc[
    master['include_structural']
].copy()

summary = inventory.groupby('forest_type').agg(

    n_trees=('tree_id', 'size'),

    raw_taxonomic_labels=(
        'species_clean', 'nunique'
    ),

    dbh_mean_cm=('dbh_cm', 'mean'),
    dbh_sd_cm=('dbh_cm', 'std'),
    dbh_min_cm=('dbh_cm', 'min'),
    dbh_max_cm=('dbh_cm', 'max'),

    height_mean_m=('height_m', 'mean'),
    height_sd_m=('height_m', 'std'),
    height_min_m=('height_m', 'min'),
    height_max_m=('height_m', 'max'),

    basal_area_m2_ha=('basal_area_m2', 'sum'),

    mean_slenderness=(
        'slenderness_ratio', 'mean'
    )

)

print('FOREST STRUCTURAL SUMMARY')
display(summary.round(4).T)

# =====================================================
# SUBPLOT MASTER
# =====================================================

subplot_master = inventory.groupby(

    [
        'forest_type',
        'plot_id',
        'subplot_number',
        'subplot_id'
    ],

    as_index=False

).agg(

    n_trees=('tree_id', 'size'),

    raw_taxonomic_labels=(
        'species_clean', 'nunique'
    ),

    mean_dbh_cm=('dbh_cm', 'mean'),
    sd_dbh_cm=('dbh_cm', 'std'),

    mean_height_m=('height_m', 'mean'),
    sd_height_m=('height_m', 'std'),

    max_dbh_cm=('dbh_cm', 'max'),
    max_height_m=('height_m', 'max'),

    basal_area_m2=('basal_area_m2', 'sum'),

    mean_slenderness=(
        'slenderness_ratio', 'mean'
    )

)

subplot_master['subplot_area_ha'] = 0.04

subplot_master['density_trees_ha'] = (
    subplot_master['n_trees'] / 0.04
)

subplot_master['basal_area_m2_ha'] = (
    subplot_master['basal_area_m2'] / 0.04
)

assert len(subplot_master) == 50

print('\nSUBPLOT MASTER')
display(subplot_master.head(10))


In [ ]:

# =====================================================
# TAXONOMIC MASTER
# =====================================================

floristic = master.loc[
    master['include_floristic']
].copy()

taxonomy_master = (

    floristic.groupby(

        ['species_clean', 'family_clean'],

        dropna=False

    ).agg(

        n_records=('tree_id', 'size'),

        n_plots=('plot_id', 'nunique'),

        n_subplots=('subplot_id', 'nunique')

    ).reset_index()

)

# Taxonomic identification level
def identify_taxonomic_level(name):

    if pd.isna(name):
        return 'unidentified'

    name = str(name).strip()

    if re.search(
        r'\bsp\.?\s*\d*$|\bspp\.?$',
        name,
        flags=re.IGNORECASE
    ):
        return 'genus_or_morphospecies'

    if re.search(
        r'\bcf\.|\baff\.',
        name,
        flags=re.IGNORECASE
    ):
        return 'uncertain_identification'

    if len(name.split()) >= 2:
        return 'candidate_species'

    return 'higher_rank_or_unidentified'


taxonomy_master['identification_level'] = (

    taxonomy_master['species_clean']
    .apply(identify_taxonomic_level)

)

# Validation fields
taxonomy_master['accepted_name'] = pd.NA
taxonomy_master['accepted_genus'] = pd.NA
taxonomy_master['accepted_family'] = pd.NA
taxonomy_master['taxonomic_rank'] = pd.NA
taxonomy_master['match_status'] = 'pending'
taxonomy_master['taxonomic_source'] = pd.NA

# Functional traits
taxonomy_master['wood_density_g_cm3'] = np.nan
taxonomy_master['wood_density_source'] = pd.NA
taxonomy_master['wood_density_match_level'] = pd.NA

taxonomy_master = taxonomy_master.sort_values(
    'n_records',
    ascending=False
).reset_index(drop=True)

print('TAXONOMIC MASTER')
print(f'Unique taxonomic entries: {len(taxonomy_master)}')

display(taxonomy_master.head(20))

print('\nIDENTIFICATION LEVELS:')
display(
    taxonomy_master['identification_level']
    .value_counts()
    .to_frame('Frequency')
)


In [ ]:

# =====================================================
# EXPORT MASTER DATASETS
# =====================================================

OUTPUT = DIRS['master']

tables = {
    'SATIPO_TREE_MASTER_V01': master,
    'SATIPO_SUBPLOT_MASTER_V01': subplot_master,
    'SATIPO_TAXONOMY_MASTER_V01': taxonomy_master,
    'SATIPO_QA_QC_V01': qa_summary.reset_index(),
    'SATIPO_FOREST_SUMMARY_V01': summary.reset_index()
}

for filename, table in tables.items():

    path = OUTPUT / f'{filename}.csv'

    table.to_csv(
        path,
        index=False,
        encoding='utf-8-sig'
    )

    print(f'SAVED: {path.name}')

# Combined Excel workbook
xlsx_path = OUTPUT / 'SATIPO_DATA_MASTER_V01.xlsx'

with pd.ExcelWriter(
    xlsx_path,
    engine='openpyxl'
) as writer:

    master.to_excel(
        writer,
        sheet_name='Tree_master',
        index=False
    )

    subplot_master.to_excel(
        writer,
        sheet_name='Subplot_master',
        index=False
    )

    taxonomy_master.to_excel(
        writer,
        sheet_name='Taxonomy_master',
        index=False
    )

    qa_summary.to_excel(
        writer,
        sheet_name='QA_QC'
    )

    summary.to_excel(
        writer,
        sheet_name='Forest_summary'
    )

# Source file integrity
manifest = []

for forest, path in FILES.items():

    manifest.append({

        'forest_type': forest,

        'source_file': path.name,

        'source_sheet': 'Hoja 1',

        'records': len(raw_data[forest]),

        'sha256': hashlib.sha256(
            path.read_bytes()
        ).hexdigest()

    })

manifest_df = pd.DataFrame(manifest)

manifest_df.to_csv(

    OUTPUT / 'RAW_DATA_MANIFEST.csv',

    index=False

)

print('\n' + '=' * 65)
print('SATIPO DATA MASTER V01 COMPLETED')
print(f'Excel workbook: {xlsx_path}')
print('=' * 65)


In [ ]:

# =====================================================
# PHASE II: TAXONOMIC AUDIT
# =====================================================

tax = taxonomy_master.copy()

# Unique taxonomic identifier
tax['taxon_id'] = [
    f'TAX_{i:04d}'
    for i in range(1, len(tax) + 1)
]

# Original names are preserved
tax['query_name'] = tax['species_clean'].copy()

# Preliminary genus extraction
tax['genus_raw'] = (
    tax['species_clean']
    .astype('string')
    .str.split()
    .str[0]
)

# Names associated with multiple families
family_conflicts = (
    tax.groupby('species_clean')
    ['family_clean']
    .nunique()
    .reset_index(name='n_families')
)

family_conflicts = family_conflicts.loc[
    family_conflicts['n_families'] > 1
]

print('TAXONOMIC ENTRIES:', len(tax))

print('\nNAMES WITH MULTIPLE FAMILIES:')
display(family_conflicts)

print('\nGENUS / MORPHOSPECIES:')
display(
    tax.loc[
        tax['identification_level']
        .eq('genus_or_morphospecies'),
        [
            'taxon_id',
            'species_clean',
            'family_clean',
            'n_records',
            'n_plots'
        ]
    ]
)

print('\nHIGHER RANK / UNIDENTIFIED:')
display(
    tax.loc[
        tax['identification_level']
        .eq('higher_rank_or_unidentified'),
        [
            'taxon_id',
            'species_clean',
            'family_clean',
            'n_records',
            'n_plots'
        ]
    ]
)

print('\nTAXONOMIC RECORD COVERAGE:')

coverage = (
    tax.groupby('identification_level')
    .agg(
        taxonomic_entries=('taxon_id', 'size'),
        individual_records=('n_records', 'sum')
    )
)

coverage['individual_percentage'] = (
    coverage['individual_records']
    / tax['n_records'].sum()
    * 100
)

display(coverage.round(2))

assert tax['n_records'].sum() == 1009
assert tax['taxon_id'].is_unique


In [ ]:

# =====================================================
# PREPARE TAXONOMIC QUERIES
# =====================================================

def prepare_taxonomic_query(row):

    name = str(row['species_clean']).strip()
    family = str(row['family_clean']).strip()
    level = row['identification_level']

    if level == 'candidate_species':

        return pd.Series({
            'gbif_query': name,
            'query_rank': 'SPECIES'
        })

    elif level == 'genus_or_morphospecies':

        genus = name.split()[0]

        return pd.Series({
            'gbif_query': genus,
            'query_rank': 'GENUS'
        })

    elif level == 'uncertain_identification':

        genus = name.split()[0]

        return pd.Series({
            'gbif_query': genus,
            'query_rank': 'GENUS'
        })

    else:

        # Do not infer an unidentified species
        return pd.Series({
            'gbif_query': pd.NA,
            'query_rank': 'UNRESOLVED'
        })


tax[['gbif_query', 'query_rank']] = (
    tax.apply(
        prepare_taxonomic_query,
        axis=1
    )
)

print('PREPARED TAXONOMIC QUERIES')

display(
    tax[
        [
            'taxon_id',
            'species_clean',
            'family_clean',
            'gbif_query',
            'query_rank',
            'n_records'
        ]
    ].head(30)
)

print('\nQUERY RANKS:')

display(
    tax['query_rank']
    .value_counts()
    .to_frame('Frequency')
)


In [ ]:

# =====================================================
# GBIF TAXONOMIC MATCHING
# =====================================================

import requests
import json
import time

from datetime import datetime, timezone
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm.auto import tqdm

GBIF_URL = (
    'https://api.gbif.org/v1/species/match'
)

CACHE_PATH = (
    DIRS['master'] / 'GBIF_TAXONOMIC_CACHE.json'
)

# Persistent cache
if CACHE_PATH.exists():

    with open(
        CACHE_PATH,
        'r',
        encoding='utf-8'
    ) as f:

        gbif_cache = json.load(f)

else:

    gbif_cache = {}


# HTTP session with automatic retries
session = requests.Session()

retry = Retry(
    total=4,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=['GET']
)

session.mount(
    'https://',
    HTTPAdapter(max_retries=retry)
)


def save_cache():

    with open(
        CACHE_PATH,
        'w',
        encoding='utf-8'
    ) as f:

        json.dump(
            gbif_cache,
            f,
            ensure_ascii=False,
            indent=2
        )


def gbif_match(name, rank, family=None):

    params = {
        'name': str(name),
        'rank': str(rank),
        'kingdom': 'Plantae',
        'verbose': 'true'
    }

    if pd.notna(family):

        params['family'] = str(family)

    cache_key = json.dumps(
        params,
        sort_keys=True,
        ensure_ascii=False
    )

    if cache_key in gbif_cache:

        return gbif_cache[cache_key]['response']

    try:

        response = session.get(
            GBIF_URL,
            params=params,
            timeout=30
        )

        response.raise_for_status()

        result = response.json()

        gbif_cache[cache_key] = {
            'query': params,
            'retrieved_at': datetime.now(
                timezone.utc
            ).isoformat(),
            'response': result
        }

        save_cache()

        time.sleep(0.2)

        return result

    except requests.RequestException as error:

        return {
            'matchType': 'API_ERROR',
            'error': str(error)
        }


# Query each taxonomic entry
gbif_results = []

for _, row in tqdm(
    tax.iterrows(),
    total=len(tax),
    desc='GBIF taxonomic validation'
):

    if pd.isna(row['gbif_query']):

        result = {
            'matchType': 'NOT_QUERIED'
        }

    else:

        result = gbif_match(
            name=row['gbif_query'],
            rank=row['query_rank'],
            family=row['family_clean']
        )

    gbif_results.append({

        'taxon_id': row['taxon_id'],

        'gbif_usage_key': result.get(
            'usageKey'
        ),

        'gbif_accepted_key': result.get(
            'acceptedUsageKey',
            result.get('usageKey')
        ),

        'gbif_scientific_name': result.get(
            'scientificName'
        ),

        'gbif_canonical_name': result.get(
            'canonicalName'
        ),

        'gbif_genus': result.get('genus'),

        'gbif_family': result.get('family'),

        'gbif_rank': result.get('rank'),

        'gbif_status': result.get('status'),

        'gbif_match_type': result.get(
            'matchType'
        ),

        'gbif_confidence': result.get(
            'confidence'
        ),

        'gbif_note': result.get('note'),

        'gbif_error': result.get('error')

    })


gbif_df = pd.DataFrame(gbif_results)

tax_gbif = tax.merge(
    gbif_df,
    on='taxon_id',
    how='left',
    validate='one_to_one'
)

assert len(tax_gbif) == len(tax)

print('\nGBIF MATCHING COMPLETED')

display(
    tax_gbif['gbif_match_type']
    .value_counts(dropna=False)
    .to_frame('Frequency')
)

display(
    tax_gbif[
        [
            'species_clean',
            'gbif_canonical_name',
            'gbif_rank',
            'gbif_status',
            'gbif_match_type',
            'gbif_confidence'
        ]
    ].head(30)
)


In [ ]:

# =====================================================
# TAXONOMIC MATCH QUALITY CONTROL
# =====================================================

def assess_match(row):

    match_type = row['gbif_match_type']
    status = row['gbif_status']
    rank = row['gbif_rank']
    confidence = row['gbif_confidence']

    if match_type == 'NOT_QUERIED':

        return 'unresolved_original_label'

    if match_type == 'API_ERROR':

        return 'api_error_retry'

    if match_type == 'NONE':

        return 'no_match'

    if pd.isna(confidence):

        return 'manual_review'

    if confidence < 95:

        return 'low_confidence_review'

    if match_type != 'EXACT':

        return 'non_exact_match_review'

    if rank != row['query_rank']:

        return 'rank_mismatch_review'

    original_family = str(
        row['family_clean']
    ).strip().casefold()

    matched_family = str(
        row['gbif_family']
    ).strip().casefold()

    if original_family != matched_family:

        return 'family_mismatch_review'

    if status == 'SYNONYM':

        return 'synonym_resolution_required'

    if status != 'ACCEPTED':

        return 'status_review'

    if row['query_rank'] == 'SPECIES':

        return 'species_match_candidate'

    if row['query_rank'] == 'GENUS':

        return 'genus_match_candidate'

    return 'manual_review'


tax_gbif['validation_category'] = (
    tax_gbif.apply(
        assess_match,
        axis=1
    )
)

validation_summary = (

    tax_gbif.groupby(
        'validation_category',
        dropna=False
    )

    .agg(
        taxonomic_entries=('taxon_id', 'size'),
        individual_records=('n_records', 'sum')
    )

    .reset_index()

)

validation_summary['individual_percentage'] = (

    validation_summary['individual_records']
    / tax_gbif['n_records'].sum()
    * 100

)

print('TAXONOMIC VALIDATION SUMMARY')

display(
    validation_summary.round(2)
)

print('\nRECORDS REQUIRING MANUAL REVIEW:')

review = tax_gbif.loc[
    ~tax_gbif['validation_category'].isin(
        [
            'species_match_candidate',
            'genus_match_candidate'
        ]
    )
].copy()

display(
    review[
        [
            'taxon_id',
            'species_clean',
            'family_clean',
            'gbif_canonical_name',
            'gbif_family',
            'gbif_rank',
            'gbif_status',
            'gbif_match_type',
            'gbif_confidence',
            'validation_category',
            'n_records'
        ]
    ]
)

print(
    '\nEntries requiring review:',
    len(review)
)


In [ ]:

# =====================================================
# EXPORT TAXONOMIC AUDIT
# =====================================================

TAX_OUTPUT = (
    DIRS['master'] / 'TAXONOMIC_VALIDATION'
)

TAX_OUTPUT.mkdir(
    parents=True,
    exist_ok=True
)

tax_gbif.to_csv(

    TAX_OUTPUT / 'SATIPO_TAXONOMY_GBIF_V02.csv',

    index=False,

    encoding='utf-8-sig'

)

review.to_csv(

    TAX_OUTPUT / 'SATIPO_TAXONOMY_REVIEW_V02.csv',

    index=False,

    encoding='utf-8-sig'

)

validation_summary.to_csv(

    TAX_OUTPUT / 'SATIPO_TAXONOMY_QA_V02.csv',

    index=False,

    encoding='utf-8-sig'

)

with pd.ExcelWriter(

    TAX_OUTPUT / 'SATIPO_TAXONOMIC_AUDIT_V02.xlsx',

    engine='openpyxl'

) as writer:

    tax_gbif.to_excel(

        writer,

        sheet_name='GBIF_matches',

        index=False

    )

    review.to_excel(

        writer,

        sheet_name='Manual_review',

        index=False

    )

    validation_summary.to_excel(

        writer,

        sheet_name='Validation_summary',

        index=False

    )

    coverage.to_excel(

        writer,

        sheet_name='Original_coverage'

    )

print('TAXONOMIC AUDIT EXPORTED')

print(
    TAX_OUTPUT / 'SATIPO_TAXONOMIC_AUDIT_V02.xlsx'
)


In [ ]:

# =====================================================
# PHASE III: ACCEPTED TAXONOMIC NAMES
# =====================================================

from tqdm.auto import tqdm
import requests
import json
import time

ACCEPTED_CACHE = (
    TAX_OUTPUT / 'GBIF_ACCEPTED_NAMES_CACHE.json'
)

if ACCEPTED_CACHE.exists():

    accepted_cache = json.loads(
        ACCEPTED_CACHE.read_text(encoding='utf-8')
    )

else:

    accepted_cache = {}


def get_gbif_taxon(key):

    if pd.isna(key):
        return {}

    key = str(int(key))

    if key in accepted_cache:
        return accepted_cache[key]

    try:

        response = session.get(
            f'https://api.gbif.org/v1/species/{key}',
            timeout=30
        )

        response.raise_for_status()

        result = response.json()

        accepted_cache[key] = result

        ACCEPTED_CACHE.write_text(
            json.dumps(
                accepted_cache,
                ensure_ascii=False,
                indent=2
            ),
            encoding='utf-8'
        )

        time.sleep(0.15)

        return result

    except requests.RequestException as error:

        print(f'GBIF error for {key}: {error}')

        return {}


accepted_results = []

for _, row in tqdm(
    tax_gbif.iterrows(),
    total=len(tax_gbif),
    desc='Retrieving accepted taxonomy'
):

    result = get_gbif_taxon(
        row['gbif_accepted_key']
    )

    accepted_results.append({

        'taxon_id': row['taxon_id'],

        'accepted_key_api': result.get('key'),

        'accepted_name_api': result.get(
            'canonicalName'
        ),

        'accepted_full_name_api': result.get(
            'scientificName'
        ),

        'accepted_genus_api': result.get(
            'genus'
        ),

        'accepted_family_api': result.get(
            'family'
        ),

        'accepted_rank_api': result.get(
            'rank'
        ),

        'accepted_status_api': result.get(
            'taxonomicStatus'
        ),

        'accepted_dataset_key': result.get(
            'datasetKey'
        )

    })


accepted_df = pd.DataFrame(accepted_results)

tax_v03 = tax_gbif.merge(

    accepted_df,

    on='taxon_id',

    how='left',

    validate='one_to_one'

)

assert len(tax_v03) == 163

print('ACCEPTED TAXONOMY RETRIEVED')

display(
    tax_v03.loc[
        tax_v03['gbif_status'].eq('SYNONYM'),
        [
            'species_clean',
            'gbif_canonical_name',
            'accepted_name_api',
            'accepted_rank_api',
            'accepted_status_api',
            'n_records'
        ]
    ]
)


In [ ]:

# =====================================================
# CONTROLLED TAXONOMIC CORRECTIONS
# =====================================================

SPELLING_CORRECTIONS = {

    'Cinnamomun triplinerve':
        'Cinnamomum triplinerve',

    'Miconia donoeana':
        'Miconia donaeana',

    'Brosimun alicastrum':
        'Brosimum alicastrum',

    'Heliocapus americanus':
        'Heliocarpus americanus',

    'Hymatanthus sucuuba':
        'Himatanthus sucuuba',

    'Handroanthus ochraceos':
        'Handroanthus ochraceus',

    'Protium arachouchini':
        'Protium aracouchini'

}

# Explicit family differences to review
FAMILY_CORRECTIONS = {

    'Ampelocera ruizii': 'Cannabaceae',

    'Ampelocera edentula': 'Cannabaceae',

    'Ampelocera sp. 1': 'Cannabaceae',

    'Quiina florida': 'Quiinaceae',

    'Minquartia guianensis': 'Coulaceae',

    'Sloanea sp.': 'Elaeocarpaceae'

}

tax_v03['name_corrected'] = (
    tax_v03['species_clean']
    .replace(SPELLING_CORRECTIONS)
)

tax_v03['family_corrected'] = (
    tax_v03['family_clean']
    .copy()
)

family_mask = (
    tax_v03['species_clean']
    .isin(FAMILY_CORRECTIONS)
)

tax_v03.loc[
    family_mask,
    'family_corrected'
] = (

    tax_v03.loc[
        family_mask,
        'species_clean'
    ].map(FAMILY_CORRECTIONS)

)

tax_v03['flag_spelling_corrected'] = (
    tax_v03['species_clean']
    .isin(SPELLING_CORRECTIONS)
)

tax_v03['flag_family_updated'] = family_mask

print('SPELLING CORRECTIONS')

display(
    tax_v03.loc[
        tax_v03['flag_spelling_corrected'],
        [
            'species_clean',
            'name_corrected',
            'gbif_canonical_name',
            'accepted_name_api',
            'n_records'
        ]
    ]
)

print('\nFAMILY UPDATES')

display(
    tax_v03.loc[
        tax_v03['flag_family_updated'],
        [
            'species_clean',
            'family_clean',
            'family_corrected',
            'accepted_family_api',
            'n_records'
        ]
    ]
)


In [ ]:

# =====================================================
# FINAL TAXONOMIC CONSENSUS - VERSION 03
# =====================================================

def resolve_taxonomy(row):

    original = str(row['species_clean'])

    result = {

        'final_accepted_name': pd.NA,

        'final_genus': pd.NA,

        'final_family': pd.NA,

        'final_rank': 'UNRESOLVED',

        'resolution_status': 'manual_review',

        'resolution_method': pd.NA,

        'operational_taxon': original

    }

    # Preserve unidentified field codes
    if re.fullmatch(
        r'SI-\d+',
        original,
        flags=re.IGNORECASE
    ):

        result['resolution_status'] = (
            'unidentified_field_code'
        )

        result['resolution_method'] = (
            'original_inventory'
        )

        return pd.Series(result)

    # Explicit unresolved spelling
    if original == 'Brosimun cinanomun':

        result['final_family'] = 'Moraceae'

        result['resolution_status'] = (
            'unresolved_species_name'
        )

        result['resolution_method'] = (
            'original_family_only'
        )

        return pd.Series(result)

    # Do not accept doubtful taxonomic status
    if row['gbif_status'] == 'DOUBTFUL':

        result['resolution_status'] = (
            'doubtful_taxonomic_status'
        )

        return pd.Series(result)

    accepted_name = row['accepted_name_api']
    accepted_rank = row['accepted_rank_api']
    accepted_status = row['accepted_status_api']

    if (
        pd.isna(accepted_name)
        or accepted_status != 'ACCEPTED'
    ):

        result['resolution_status'] = (
            'accepted_name_not_verified'
        )

        return pd.Series(result)

    # Check the requested taxonomic rank
    if accepted_rank != row['query_rank']:

        result['resolution_status'] = (
            'rank_mismatch'
        )

        return pd.Series(result)

    # Require exact match or explicit correction
    match_allowed = (

        row['gbif_match_type'] == 'EXACT'

        or (

            row['flag_spelling_corrected']

            and str(
                row['name_corrected']
            ).casefold()

            == str(
                row['gbif_canonical_name']
            ).casefold()

        )

    )

    if not match_allowed:

        result['resolution_status'] = (
            'unverified_name_match'
        )

        return pd.Series(result)

    # Confidence threshold
    confidence = row['gbif_confidence']

    confidence_allowed = (

        pd.notna(confidence)

        and (

            confidence >= 95

            or row['flag_family_updated']

        )

    )

    if not confidence_allowed:

        result['resolution_status'] = (
            'low_confidence'
        )

        return pd.Series(result)

    # Require consistent family
    expected_family = str(
        row['family_corrected']
    ).casefold()

    matched_family = str(
        row['accepted_family_api']
    ).casefold()

    if expected_family != matched_family:

        result['resolution_status'] = (
            'family_requires_review'
        )

        return pd.Series(result)

    # Store verified nomenclatural result
    result['final_accepted_name'] = (
        accepted_name
    )

    result['final_genus'] = (
        row['accepted_genus_api']
    )

    result['final_family'] = (
        row['accepted_family_api']
    )

    result['final_rank'] = accepted_rank

    if accepted_rank == 'SPECIES':

        result['resolution_status'] = (
            'resolved_species'
        )

        result['operational_taxon'] = (
            accepted_name
        )

    elif accepted_rank == 'GENUS':

        result['resolution_status'] = (
            'resolved_genus'
        )

        # Preserve original morphospecies identity
        result['operational_taxon'] = original

    if row['flag_spelling_corrected']:

        result['resolution_method'] = (
            'controlled_spelling_correction'
        )

    elif row['gbif_status'] == 'SYNONYM':

        result['resolution_method'] = (
            'GBIF_accepted_synonym'
        )

    elif row['flag_family_updated']:

        result['resolution_method'] = (
            'GBIF_family_update'
        )

    else:

        result['resolution_method'] = (
            'GBIF_exact_match'
        )

    return pd.Series(result)


resolved = tax_v03.apply(
    resolve_taxonomy,
    axis=1
)

tax_final = pd.concat(
    [tax_v03, resolved],
    axis=1
)

assert len(tax_final) == 163

assert tax_final['n_records'].sum() == 1009

# Resolution summary
resolution_summary = (

    tax_final.groupby(
        'resolution_status'
    )

    .agg(
        taxonomic_entries=('taxon_id', 'size'),
        individual_records=('n_records', 'sum')
    )

    .reset_index()

)

resolution_summary['individual_percentage'] = (

    resolution_summary['individual_records']
    / 1009
    * 100

)

print('FINAL NOMENCLATURAL RESOLUTION')

display(
    resolution_summary.round(2)
)

print('\nPENDING TAXONOMIC RECORDS')

pending_taxonomy = tax_final.loc[

    ~tax_final['resolution_status'].isin(
        [
            'resolved_species',
            'resolved_genus'
        ]
    )

].copy()

display(
    pending_taxonomy[
        [
            'taxon_id',
            'species_clean',
            'family_clean',
            'accepted_name_api',
            'resolution_status',
            'n_records'
        ]
    ]
)


In [ ]:

# =====================================================
# INTEGRATE TAXONOMIC CONSENSUS WITH TREE MASTER
# =====================================================

taxonomy_lookup = tax_final[

    [
        'taxon_id',
        'species_clean',
        'family_clean',
        'final_accepted_name',
        'final_genus',
        'final_family',
        'final_rank',
        'resolution_status',
        'resolution_method',
        'operational_taxon',
        'accepted_key_api'
    ]

].copy()

# Validate unique lookup keys
assert not taxonomy_lookup.duplicated(
    ['species_clean', 'family_clean']
).any()

master_v03 = master.merge(

    taxonomy_lookup,

    on=[
        'species_clean',
        'family_clean'
    ],

    how='left',

    validate='many_to_one'

)

assert len(master_v03) == 1062
assert master_v03['tree_id'].is_unique

# Species accepted only for species-level matches
master_v03['species_accepted'] = (

    master_v03['final_accepted_name']
    .where(
        master_v03['final_rank'].eq('SPECIES')
    )

)

master_v03['genus_accepted'] = (
    master_v03['final_genus']
)

master_v03['family_accepted'] = (
    master_v03['final_family']
)

master_v03['taxonomic_status'] = (
    master_v03['resolution_status']
)

master_v03.loc[

    master_v03['status_inventory'].eq('dead'),

    'taxonomic_status'

] = 'dead_not_identified'

# Species-level analytical subset
master_v03['include_species_analysis'] = (

    master_v03['include_floristic']

    & master_v03['final_rank'].eq('SPECIES')

    & master_v03['resolution_status']
      .eq('resolved_species')

)

# Operational taxonomic subset
master_v03['include_otu_analysis'] = (

    master_v03['include_floristic']

    & master_v03['operational_taxon'].notna()

)

# Confirm all inventory records were matched
assert master_v03.loc[

    master_v03['include_floristic'],

    'taxon_id'

].notna().all()

# =====================================================
# FOREST-LEVEL TAXONOMIC COVERAGE
# =====================================================

forest_taxonomic_summary = (

    master_v03.loc[
        master_v03['include_floristic']
    ]

    .groupby('forest_type')

    .agg(

        n_individuals=('tree_id', 'size'),

        raw_labels=('species_clean', 'nunique'),

        resolved_species_richness=(
            'species_accepted', 'nunique'
        ),

        resolved_species_individuals=(
            'include_species_analysis', 'sum'
        ),

        accepted_genera=(
            'genus_accepted', 'nunique'
        ),

        accepted_families=(
            'family_accepted', 'nunique'
        ),

        operational_taxa=(
            'operational_taxon', 'nunique'
        )

    )

)

forest_taxonomic_summary[
    'species_identification_coverage_pct'
] = (

    forest_taxonomic_summary[
        'resolved_species_individuals'
    ]

    / forest_taxonomic_summary['n_individuals']

    * 100

)

print('FOREST TAXONOMIC SUMMARY')

display(
    forest_taxonomic_summary.round(3).T
)

print('\nTAXONOMIC STATUS BY FOREST')

display(

    pd.crosstab(

        master_v03['taxonomic_status'],

        master_v03['forest_type']

    )

)


In [ ]:

# =====================================================
# EXPORT TAXONOMIC MASTER V03
# =====================================================

V03_OUTPUT = (
    DIRS['master'] / 'VERSION_03'
)

V03_OUTPUT.mkdir(
    parents=True,
    exist_ok=True
)

tables_v03 = {

    'SATIPO_TREE_MASTER_V03':
        master_v03,

    'SATIPO_TAXONOMY_MASTER_V03':
        tax_final,

    'SATIPO_TAXONOMY_PENDING_V03':
        pending_taxonomy,

    'SATIPO_TAXONOMY_SUMMARY_V03':
        resolution_summary,

    'SATIPO_FOREST_TAXONOMY_V03':
        forest_taxonomic_summary.reset_index()

}

for name, table in tables_v03.items():

    table.to_csv(

        V03_OUTPUT / f'{name}.csv',

        index=False,

        encoding='utf-8-sig'

    )

    print(f'SAVED: {name}.csv')


with pd.ExcelWriter(

    V03_OUTPUT / 'SATIPO_DATA_MASTER_V03.xlsx',

    engine='openpyxl'

) as writer:

    master_v03.to_excel(

        writer,

        sheet_name='Tree_master',

        index=False

    )

    tax_final.to_excel(

        writer,

        sheet_name='Taxonomy_master',

        index=False

    )

    pending_taxonomy.to_excel(

        writer,

        sheet_name='Pending_taxonomy',

        index=False

    )

    resolution_summary.to_excel(

        writer,

        sheet_name='Resolution_summary',

        index=False

    )

    forest_taxonomic_summary.to_excel(

        writer,

        sheet_name='Forest_taxonomy'

    )

    subplot_master.to_excel(

        writer,

        sheet_name='Subplot_master',

        index=False

    )

    qa_summary.to_excel(

        writer,

        sheet_name='QA_QC'

    )

print('\n' + '=' * 65)

print('SATIPO DATA MASTER V03 COMPLETED')

print(
    V03_OUTPUT / 'SATIPO_DATA_MASTER_V03.xlsx'
)

print('=' * 65)


In [ ]:

# =====================================================
# PHASE IV: FINAL TAXONOMIC ADJUSTMENTS
# =====================================================

tax_v04 = tax_final.copy()

tax_v04['infraspecific_name'] = pd.NA
tax_v04['field_genus'] = pd.NA
tax_v04['field_family'] = pd.NA
tax_v04['taxonomic_adjustment_note'] = pd.NA

# Preserve original genus and family information
known_family = ~tax_v04['species_clean'].str.match(
    r'^SI-\d+$',
    na=False
)

tax_v04.loc[
    known_family,
    'field_family'
] = tax_v04.loc[
    known_family,
    'family_corrected'
]

# -----------------------------------------------------
# Sorocea: accepted subspecies -> parent species
# -----------------------------------------------------

mask = tax_v04['species_clean'].eq('Sorocea opima')

row = tax_v04.loc[mask].iloc[0]

subspecies = get_gbif_taxon(
    row['accepted_key_api']
)

parent_key = (
    subspecies.get('speciesKey')
    or subspecies.get('parentKey')
)

parent = get_gbif_taxon(parent_key)

print('Accepted subspecies:')
print(subspecies.get('canonicalName'))

print('\nParent species:')
print(parent.get('canonicalName'))
print(parent.get('rank'))
print(parent.get('taxonomicStatus'))

if (
    parent.get('rank') == 'SPECIES'
    and parent.get('taxonomicStatus') == 'ACCEPTED'
    and parent.get('canonicalName') == 'Sorocea pubivena'
):

    tax_v04.loc[
        mask, 'infraspecific_name'
    ] = subspecies.get('canonicalName')

    tax_v04.loc[
        mask, 'final_accepted_name'
    ] = parent.get('canonicalName')

    tax_v04.loc[
        mask, 'final_genus'
    ] = parent.get('genus')

    tax_v04.loc[
        mask, 'final_family'
    ] = parent.get('family')

    tax_v04.loc[
        mask, 'final_rank'
    ] = 'SPECIES'

    tax_v04.loc[
        mask, 'resolution_status'
    ] = 'resolved_species'

    tax_v04.loc[
        mask, 'resolution_method'
    ] = 'GBIF_infraspecific_to_species'

    tax_v04.loc[
        mask, 'operational_taxon'
    ] = parent.get('canonicalName')

    tax_v04.loc[
        mask, 'accepted_key_api'
    ] = parent.get('key')

    tax_v04.loc[
        mask, 'taxonomic_adjustment_note'
    ] = 'Accepted subspecies retained; aggregated to species'

    print('\nSOROCEA SUCCESSFULLY RESOLVED')

else:

    print('\nSOROCEA REQUIRES MANUAL REVIEW')


# -----------------------------------------------------
# Euphorbia: retain original field identification
# -----------------------------------------------------

mask = tax_v04['species_clean'].eq('Euphorbia sp.')

tax_v04.loc[
    mask, 'field_genus'
] = 'Euphorbia'

tax_v04.loc[
    mask, 'field_family'
] = 'Euphorbiaceae'

tax_v04.loc[
    mask, 'taxonomic_adjustment_note'
] = 'Field genus retained; GBIF status requires review'


# -----------------------------------------------------
# Unresolved Brosimum label
# -----------------------------------------------------

mask = tax_v04['species_clean'].eq(
    'Brosimun cinanomun'
)

tax_v04.loc[
    mask, 'field_family'
] = 'Moraceae'

tax_v04.loc[
    mask, 'taxonomic_adjustment_note'
] = 'Family retained; species identity unresolved'


# -----------------------------------------------------
# Updated taxonomic summary
# -----------------------------------------------------

tax_summary_v04 = (

    tax_v04.groupby('resolution_status')

    .agg(
        taxonomic_entries=('taxon_id', 'size'),
        individuals=('n_records', 'sum')
    )

)

tax_summary_v04['percentage'] = (

    tax_summary_v04['individuals']
    / 1009 * 100

)

display(tax_summary_v04.round(3))

assert len(tax_v04) == 163
assert tax_v04['n_records'].sum() == 1009


In [ ]:

# =====================================================
# GLOBAL WOOD DENSITY DATABASE V2.2
# =====================================================

import requests
import hashlib
from pathlib import Path

TRAITS_DIR = DIRS['raw'] / 'EXTERNAL_TRAITS'

TRAITS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ZENODO_RECORD = '20815517'

GWDD_FILES = {

    'species':
        'gwddagg_v2.2_species.csv',

    'binomial':
        'gwddagg_v2.2_binomial.csv',

    'genus':
        'gwddagg_v2.2_genus.csv',

    'metadata':
        'columns_gwdd_v2.2.csv'

}

# Published checksums from Zenodo
EXPECTED_MD5 = {

    'species':
        'a8f0317b62e9412df00bffc329015c89',

    'binomial':
        '2ce721e791fca2b5eb17c31f81a59228',

    'genus':
        'f9e7ccb930ffa22a433ec2f19af035bb',

    'metadata':
        'b1b494e22cbf4e48762d162f10e386b0'

}

gwdd_paths = {}

for level, filename in GWDD_FILES.items():

    path = TRAITS_DIR / filename

    url = (
        f'https://zenodo.org/records/'
        f'{ZENODO_RECORD}/files/{filename}'
    )

    # Download if missing or corrupted
    valid_file = (

        path.exists()

        and hashlib.md5(
            path.read_bytes()
        ).hexdigest() == EXPECTED_MD5[level]

    )

    if not valid_file:

        print(f'Downloading {filename}...')

        response = requests.get(
            url,
            timeout=180
        )

        response.raise_for_status()

        path.write_bytes(response.content)

    checksum = hashlib.md5(
        path.read_bytes()
    ).hexdigest()

    assert checksum == EXPECTED_MD5[level], (
        f'Checksum mismatch: {filename}'
    )

    gwdd_paths[level] = path

    print(
        f'VERIFIED: {filename} | '
        f'{path.stat().st_size:,} bytes'
    )

print('\nGWDD V2.2 DOWNLOADED AND VERIFIED')


In [ ]:

# =====================================================
# INSPECT GWDD V2.2 DATA STRUCTURE
# =====================================================

gwdd = {}

for level, path in gwdd_paths.items():

    df = pd.read_csv(
        path,
        low_memory=False
    )

    gwdd[level] = df

    print('\n' + '=' * 65)

    print(f'DATABASE: {level.upper()}')

    print(f'Rows: {len(df):,}')
    print(f'Columns: {df.shape[1]}')

    print('\nCOLUMN NAMES:')

    for col in df.columns:

        print(repr(col))

    print('\nFIRST RECORDS:')

    display(df.head(5))

    print('\nDATA TYPES:')

    display(
        df.dtypes.to_frame('dtype')
    )

    print('\nMISSING VALUES:')

    display(
        df.isna().sum().to_frame('missing')
    )

print('\nALL GWDD TABLES LOADED')


In [ ]:

# =====================================================
# WOOD DENSITY VARIABLES
# =====================================================

for level in ['species', 'binomial', 'genus']:

    df = gwdd[level]

    density_cols = [

        col for col in df.columns

        if any(

            token in col.lower()

            for token in [
                'wsg',
                'density',
                'wood_density'
            ]

        )

    ]

    print('\n' + '=' * 65)

    print(f'LEVEL: {level.upper()}')

    print('\nDensity columns:')

    print(density_cols)

    for col in density_cols:

        values = pd.to_numeric(
            df[col],
            errors='coerce'
        )

        print(f'\nVariable: {col}')

        print(
            'Valid observations:',
            values.notna().sum()
        )

        display(
            values.describe().to_frame(col)
        )

print('\nMETADATA:')
display(gwdd['metadata'])


In [ ]:

# =====================================================
# TREE MASTER V04: FUNCTIONAL TRAITS PREPARATION
# =====================================================

lookup_cols = [

    'taxon_id',

    'final_accepted_name',
    'final_genus',
    'final_family',
    'final_rank',

    'resolution_status',
    'resolution_method',

    'operational_taxon',
    'accepted_key_api',

    'infraspecific_name',
    'field_genus',
    'field_family',

    'taxonomic_adjustment_note'

]

lookup_v04 = tax_v04[lookup_cols].copy()

# Remove previous lookup fields before updating
master_v04 = master_v03.drop(

    columns=[
        c for c in lookup_cols
        if c != 'taxon_id'
        and c in master_v03.columns
    ]

).merge(

    lookup_v04,

    on='taxon_id',

    how='left',

    validate='many_to_one'

)

assert len(master_v04) == 1062
assert master_v04['tree_id'].is_unique

# Update accepted taxonomy
master_v04['species_accepted'] = (

    master_v04['final_accepted_name']

    .where(
        master_v04['final_rank'].eq('SPECIES')
    )

)

master_v04['genus_accepted'] = (
    master_v04['final_genus']
)

master_v04['family_accepted'] = (
    master_v04['final_family']
)

master_v04['taxonomic_status'] = (
    master_v04['resolution_status']
)

master_v04.loc[

    master_v04['status_inventory'].eq('dead'),

    'taxonomic_status'

] = 'dead_not_identified'


# Updated analytical subsets
master_v04['include_species_analysis'] = (

    master_v04['include_floristic']

    & master_v04['final_rank'].eq('SPECIES')

    & master_v04['resolution_status']
      .eq('resolved_species')

)

master_v04['include_otu_analysis'] = (

    master_v04['include_floristic']

    & master_v04['operational_taxon'].notna()

)


# -----------------------------------------------------
# Identify palms
# -----------------------------------------------------

master_v04['flag_palm'] = (

    master_v04['family_accepted']
    .eq('Arecaceae').fillna(False)

    | master_v04['family_clean']
      .eq('Arecaceae').fillna(False)

    | master_v04['observations_clean']
      .str.contains('palmera', na=False)

)

# Conservative subset for dicot/tree allometry
master_v04['include_tree_allometry'] = (

    master_v04['include_structural_strict']

    & ~master_v04['flag_palm']

)

# Trait assignment fields
master_v04['wd_match_level'] = pd.NA
master_v04['wd_reference_taxon'] = pd.NA
master_v04['wd_reference_version'] = pd.NA
master_v04['wd_uncertainty'] = np.nan

print('PALMS REQUIRING SEPARATE ALLOMETRY')

display(

    master_v04.loc[

        master_v04['flag_palm'],

        [
            'tree_id',
            'forest_type',
            'species_clean',
            'species_accepted',
            'family_clean',
            'dbh_cm',
            'height_m',
            'observations_raw'
        ]

    ]

)

print('\nBIOMASS ELIGIBILITY')

display(

    master_v04.groupby('forest_type').agg(

        total_records=('tree_id', 'size'),

        strict_inventory=(
            'include_structural_strict',
            'sum'
        ),

        palm_records=('flag_palm', 'sum'),

        eligible_tree_allometry=(
            'include_tree_allometry',
            'sum'
        ),

        resolved_species_individuals=(
            'include_species_analysis',
            'sum'
        )

    )

)

# Save intermediate master
V04_OUTPUT = DIRS['master'] / 'VERSION_04'

V04_OUTPUT.mkdir(
    parents=True,
    exist_ok=True
)

master_v04.to_csv(

    V04_OUTPUT / 'SATIPO_TREE_MASTER_V04.csv',

    index=False,

    encoding='utf-8-sig'

)

tax_v04.to_csv(

    V04_OUTPUT / 'SATIPO_TAXONOMY_MASTER_V04.csv',

    index=False,

    encoding='utf-8-sig'

)

print('\nMASTER V04 SAVED')


In [ ]:

# =====================================================
# PHASE V: WOOD DENSITY REFERENCE TABLES
# =====================================================

import pandas as pd
import numpy as np
import re
import unicodedata

WD_VERSION = 'GWDD_v2.2'
WD_PRIMARY = 'wsg_est_trunk'
WD_ALTERNATIVE = 'wsg_est'


def normalize_taxon(value):

    if pd.isna(value):
        return None

    value = unicodedata.normalize(
        'NFC', str(value)
    )

    value = re.sub(r'\s+', ' ', value)

    return value.strip().casefold()


# -----------------------------------------------------
# Species-level reference
# -----------------------------------------------------

wd_species = gwdd['binomial'].copy()

wd_species['name_key'] = (
    wd_species['binomial']
    .apply(normalize_taxon)
)

# -----------------------------------------------------
# Genus-level reference
# -----------------------------------------------------

wd_genus = gwdd['genus'].copy()

wd_genus['name_key'] = (
    wd_genus['genus']
    .apply(normalize_taxon)
)

# -----------------------------------------------------
# Validate reference tables
# -----------------------------------------------------

for label, df in [
    ('Species', wd_species),
    ('Genus', wd_genus)
]:

    assert df[WD_PRIMARY].notna().all()
    assert df[WD_PRIMARY].gt(0).all()

    duplicates = df.loc[
        df['name_key'].duplicated(keep=False)
    ]

    print(f'\n{label} reference')
    print(f'Records: {len(df):,}')
    print(f'Duplicate keys: {len(duplicates)}')

    if len(duplicates) > 0:
        display(duplicates)

        raise ValueError(
            f'Duplicate taxonomic keys in {label}'
        )


# Lookup dictionaries
species_lookup = (
    wd_species
    .set_index('name_key')
    .to_dict(orient='index')
)

genus_lookup = (
    wd_genus
    .set_index('name_key')
    .to_dict(orient='index')
)


# -----------------------------------------------------
# Family-level reference
# Equal weight per genus, excluding palms
# -----------------------------------------------------

wd_family = (

    wd_genus.loc[
        wd_genus['family'].ne('Arecaceae')
    ]

    .groupby('family')

    .agg(

        wd_trunk_mean=(
            WD_PRIMARY, 'mean'
        ),

        wd_general_mean=(
            WD_ALTERNATIVE, 'mean'
        ),

        wd_trunk_sd=(
            WD_PRIMARY, 'std'
        ),

        wd_trunk_q25=(
            WD_PRIMARY,
            lambda x: x.quantile(0.25)
        ),

        wd_trunk_q75=(
            WD_PRIMARY,
            lambda x: x.quantile(0.75)
        ),

        n_genera=('genus', 'nunique')

    )

    .reset_index()

)

wd_family['name_key'] = (
    wd_family['family']
    .apply(normalize_taxon)
)

family_lookup = (

    wd_family
    .set_index('name_key')
    .to_dict(orient='index')

)

print('\nREFERENCE TABLES READY')

print(f'Species: {len(species_lookup):,}')
print(f'Genera: {len(genus_lookup):,}')
print(f'Families: {len(family_lookup):,}')

display(wd_family.head(10))


In [ ]:

# =====================================================
# HIERARCHICAL WOOD DENSITY MATCHING
# =====================================================

master_v05 = master_v04.copy()

# Recover verified nomenclatural aliases
alias_lookup = (

    tax_v04[
        [
            'taxon_id',
            'name_corrected',
            'gbif_canonical_name'
        ]
    ]

    .set_index('taxon_id')

    .to_dict(orient='index')

)


def assign_wood_density(row):

    result = {

        'wd_trunk': np.nan,

        'wd_general': np.nan,

        'wd_level': 'not_eligible',

        'wd_reference_name': pd.NA,

        'wd_reference_family': pd.NA,

        'wd_reference_n': np.nan,

        'wd_reference_n_type': pd.NA,

        'wd_family_sd': np.nan,

        'wd_source': pd.NA

    }

    if not row['include_tree_allometry']:
        return pd.Series(result)

    result['wd_level'] = 'unmatched'

    # -------------------------------------------------
    # LEVEL 1: Accepted species
    # -------------------------------------------------

    if row['resolution_status'] == 'resolved_species':

        accepted = row['species_accepted']

        key = normalize_taxon(accepted)

        if key in species_lookup:

            ref = species_lookup[key]

            result.update({

                'wd_trunk': ref[WD_PRIMARY],

                'wd_general': ref[WD_ALTERNATIVE],

                'wd_level': 'species_accepted',

                'wd_reference_name': ref['binomial'],

                'wd_reference_family': ref['family'],

                'wd_reference_n': ref['nb'],

                'wd_reference_n_type': 'records',

                'wd_source': WD_VERSION

            })

            return pd.Series(result)

        # ---------------------------------------------
        # LEVEL 2: Verified nomenclatural aliases
        # ---------------------------------------------

        aliases = alias_lookup.get(
            row['taxon_id'], {}
        )

        candidates = [

            aliases.get('name_corrected'),

            aliases.get('gbif_canonical_name'),

            row['species_clean']

        ]

        for candidate in candidates:

            key = normalize_taxon(candidate)

            if key in species_lookup:

                ref = species_lookup[key]

                result.update({

                    'wd_trunk': ref[WD_PRIMARY],

                    'wd_general': ref[WD_ALTERNATIVE],

                    'wd_level': 'species_alias',

                    'wd_reference_name': ref['binomial'],

                    'wd_reference_family': ref['family'],

                    'wd_reference_n': ref['nb'],

                    'wd_reference_n_type': 'records',

                    'wd_source': WD_VERSION

                })

                return pd.Series(result)

    # -------------------------------------------------
    # LEVEL 3: Genus
    # -------------------------------------------------

    genus_candidates = [

        row['genus_accepted'],

        row['field_genus']

    ]

    # Original genus may be used when nomenclature
    # was successfully resolved
    if row['resolution_status'] in [
        'resolved_species',
        'resolved_genus'
    ]:

        genus_candidates.append(
            str(row['species_clean']).split()[0]
        )

    for genus in genus_candidates:

        key = normalize_taxon(genus)

        if key in genus_lookup:

            ref = genus_lookup[key]

            result.update({

                'wd_trunk': ref[WD_PRIMARY],

                'wd_general': ref[WD_ALTERNATIVE],

                'wd_level': 'genus',

                'wd_reference_name': ref['genus'],

                'wd_reference_family': ref['family'],

                'wd_reference_n': ref['nb'],

                'wd_reference_n_type': 'records',

                'wd_source': WD_VERSION

            })

            return pd.Series(result)

    # -------------------------------------------------
    # LEVEL 4: Family
    # -------------------------------------------------

    family_candidates = [

        row['family_accepted'],

        row['field_family'],

        row['family_clean']

    ]

    for family in family_candidates:

        key = normalize_taxon(family)

        if key in family_lookup:

            ref = family_lookup[key]

            result.update({

                'wd_trunk': ref['wd_trunk_mean'],

                'wd_general': ref['wd_general_mean'],

                'wd_level': 'family',

                'wd_reference_name': ref['family'],

                'wd_reference_family': ref['family'],

                'wd_reference_n': ref['n_genera'],

                'wd_reference_n_type': 'genera',

                'wd_family_sd': ref['wd_trunk_sd'],

                'wd_source': (
                    WD_VERSION + '_family_derived'
                )

            })

            return pd.Series(result)

    return pd.Series(result)


wd_assignments = master_v05.apply(
    assign_wood_density,
    axis=1
)

# Avoid duplicate columns if cell is rerun
master_v05 = master_v05.drop(

    columns=wd_assignments.columns,

    errors='ignore'

)

master_v05 = pd.concat(

    [master_v05, wd_assignments],

    axis=1

)

assert len(master_v05) == 1062

assert master_v05['tree_id'].is_unique

print('WOOD DENSITY MATCHING COMPLETED')

print('\nMATCH LEVEL BY FOREST:')

display(

    pd.crosstab(

        master_v05['wd_level'],

        master_v05['forest_type']

    )

)

print('\nUNMATCHED ELIGIBLE TREES:')

unmatched_wd = master_v05.loc[

    master_v05['include_tree_allometry']

    & master_v05['wd_trunk'].isna()

].copy()

display(

    unmatched_wd[
        [
            'tree_id',
            'forest_type',
            'species_clean',
            'species_accepted',
            'genus_accepted',
            'family_accepted',
            'dbh_cm',
            'height_m'
        ]
    ]

)


In [ ]:

# =====================================================
# WOOD DENSITY QUALITY CONTROL
# =====================================================

eligible = master_v05.loc[
    master_v05['include_tree_allometry']
].copy()

assert len(eligible) == 1002

# -----------------------------------------------------
# Species and operational taxon-level audit
# -----------------------------------------------------

wd_taxon_audit = (

    eligible.groupby(

        [
            'forest_type',
            'operational_taxon',
            'wd_level',
            'wd_reference_name'
        ],

        dropna=False

    )

    .agg(

        n_trees=('tree_id', 'size'),

        basal_area_m2=(
            'basal_area_m2', 'sum'
        ),

        wd_trunk=('wd_trunk', 'first'),

        wd_general=('wd_general', 'first'),

        reference_n=(
            'wd_reference_n', 'first'
        )

    )

    .reset_index()

    .sort_values(
        'n_trees',
        ascending=False
    )

)

print('DOMINANT TAXA AND WOOD DENSITY')

display(wd_taxon_audit.head(30))


# -----------------------------------------------------
# Family disagreements between GBIF and GWDD
# -----------------------------------------------------

family_comparison = eligible.loc[

    eligible['family_accepted'].notna()

    & eligible['wd_reference_family'].notna()

].copy()

family_comparison['family_disagreement'] = (

    family_comparison['family_accepted']
    .apply(normalize_taxon)

    !=

    family_comparison['wd_reference_family']
    .apply(normalize_taxon)

)

family_disagreements = (

    family_comparison.loc[
        family_comparison['family_disagreement'],

        [
            'species_clean',
            'species_accepted',
            'family_accepted',
            'wd_reference_name',
            'wd_reference_family',
            'wd_level'
        ]

    ]

    .drop_duplicates()

)

print('\nFAMILY DISAGREEMENTS:')

display(family_disagreements)


# -----------------------------------------------------
# Coverage before community imputation
# -----------------------------------------------------

wd_coverage = (

    eligible.groupby('forest_type')

    .agg(

        eligible_trees=('tree_id', 'size'),

        matched_trees=(
            'wd_trunk',
            lambda x: x.notna().sum()
        ),

        total_basal_area=(
            'basal_area_m2', 'sum'
        )

    )

)

wd_coverage['coverage_pct'] = (

    wd_coverage['matched_trees']

    / wd_coverage['eligible_trees']

    * 100

)

matched_ba = (

    eligible.loc[
        eligible['wd_trunk'].notna()
    ]

    .groupby('forest_type')
    ['basal_area_m2']
    .sum()

)

wd_coverage['matched_basal_area_pct'] = (

    matched_ba

    / wd_coverage['total_basal_area']

    * 100

)

print('\nWOOD DENSITY COVERAGE')

display(wd_coverage.round(3))


In [ ]:

# =====================================================
# COMMUNITY-LEVEL IMPUTATION
# =====================================================

reference_levels = [

    'species_accepted',

    'species_alias',

    'genus'

]

reference_trees = master_v05.loc[

    master_v05['include_tree_allometry']

    & master_v05['wd_level'].isin(
        reference_levels
    )

].copy()

assert len(reference_trees) > 0

# Pooled abundance-weighted community mean
community_wd_trunk = (
    reference_trees['wd_trunk'].mean()
)

community_wd_general = (
    reference_trees['wd_general'].mean()
)

# Distinct operational taxa for sensitivity range
reference_taxa = (

    reference_trees[
        [
            'operational_taxon',
            'wd_trunk'
        ]
    ]

    .drop_duplicates()

)

community_q25 = (
    reference_taxa['wd_trunk'].quantile(0.25)
)

community_q75 = (
    reference_taxa['wd_trunk'].quantile(0.75)
)

print('COMMUNITY REFERENCE VALUES')

print(
    f'Pooled trunk WD: '
    f'{community_wd_trunk:.4f} g/cm3'
)

print(
    f'Pooled general WD: '
    f'{community_wd_general:.4f} g/cm3'
)

print(
    f'Taxon-level Q25: '
    f'{community_q25:.4f}'
)

print(
    f'Taxon-level Q75: '
    f'{community_q75:.4f}'
)

# Missing eligible trees only
impute_mask = (

    master_v05['include_tree_allometry']

    & master_v05['wd_trunk'].isna()

)

master_v05.loc[
    impute_mask, 'wd_trunk'
] = community_wd_trunk

master_v05.loc[
    impute_mask, 'wd_general'
] = community_wd_general

master_v05.loc[
    impute_mask, 'wd_level'
] = 'community_pooled'

master_v05.loc[
    impute_mask, 'wd_reference_name'
] = 'Satipo pooled community'

master_v05.loc[
    impute_mask, 'wd_reference_n'
] = len(reference_trees)

master_v05.loc[
    impute_mask, 'wd_reference_n_type'
] = 'inventory_trees'

master_v05.loc[
    impute_mask, 'wd_source'
] = 'GWDD_v2.2_Satipo_pooled'

# Main master fields
master_v05['wood_density_g_cm3'] = (
    master_v05['wd_trunk']
)

master_v05['wood_density_source'] = (
    master_v05['wd_source']
)

master_v05['wd_match_level'] = (
    master_v05['wd_level']
)

master_v05['wd_reference_taxon'] = (
    master_v05['wd_reference_name']
)

master_v05['wd_reference_version'] = np.where(

    master_v05['include_tree_allometry'],

    WD_VERSION,

    None

)

# Complete coverage check
assert master_v05.loc[

    master_v05['include_tree_allometry'],

    'wood_density_g_cm3'

].notna().all()

print('\nCOMMUNITY IMPUTATION COMPLETED')

print(
    'Imputed trees:',
    int(impute_mask.sum())
)

display(

    pd.crosstab(

        master_v05['wd_level'],

        master_v05['forest_type']

    )

)


In [ ]:

# =====================================================
# ABOVEGROUND BIOMASS: CHAVE ET AL. 2014
# =====================================================

CARBON_FRACTION = 0.47


def chave_agb(dbh_cm, height_m, wd):

    return (

        0.0673

        * (

            wd

            * dbh_cm**2

            * height_m

        )**0.976

    )


mask = master_v05['include_tree_allometry']

# Initialize outputs
master_v05['agb_kg'] = np.nan
master_v05['agb_Mg'] = np.nan

master_v05['carbon_kg'] = np.nan
master_v05['carbon_Mg'] = np.nan

master_v05['agb_general_wd_kg'] = np.nan

master_v05['agb_equation'] = pd.NA

# Main estimate
master_v05.loc[
    mask, 'agb_kg'
] = chave_agb(

    master_v05.loc[mask, 'dbh_cm'],

    master_v05.loc[mask, 'height_m'],

    master_v05.loc[
        mask, 'wood_density_g_cm3'
    ]

)

# Convert kg to Mg
master_v05['agb_Mg'] = (
    master_v05['agb_kg'] / 1000
)

# Carbon
master_v05['carbon_kg'] = (

    master_v05['agb_kg']

    * CARBON_FRACTION

)

master_v05['carbon_Mg'] = (

    master_v05['carbon_kg']

    / 1000

)

# Alternative GWDD density estimate
master_v05.loc[
    mask, 'agb_general_wd_kg'
] = chave_agb(

    master_v05.loc[mask, 'dbh_cm'],

    master_v05.loc[mask, 'height_m'],

    master_v05.loc[mask, 'wd_general']

)

master_v05.loc[
    mask, 'agb_equation'
] = 'Chave_et_al_2014_D_H_WD'

# Validation
assert master_v05.loc[
    mask, 'agb_kg'
].notna().all()

assert master_v05.loc[
    mask, 'agb_kg'
].gt(0).all()

assert master_v05.loc[
    ~mask, 'agb_kg'
].isna().all()

print('INDIVIDUAL BIOMASS ESTIMATED')

print(
    'Trees with biomass:',
    master_v05['agb_kg'].notna().sum()
)

print('\nLARGEST BIOMASS CONTRIBUTORS')

display(

    master_v05.loc[mask]

    .sort_values(
        'agb_kg',
        ascending=False
    )

    [
        [
            'tree_id',
            'forest_type',
            'species_accepted',
            'dbh_cm',
            'height_m',
            'wood_density_g_cm3',
            'wd_level',
            'agb_kg',
            'carbon_kg'
        ]
    ]

    .head(20)

)


In [ ]:

# =====================================================
# CELL 29: SUBPLOT AND TAXON-LEVEL BIOMASS
# =====================================================

import pandas as pd
import numpy as np

# -----------------------------------------------------
# 1. Verify previous processing
# -----------------------------------------------------

if 'master_v05' not in globals():
    raise NameError(
        'master_v05 no existe. Ejecutar las celdas 24-27.'
    )

required_columns = [
    'tree_id',
    'forest_type',
    'plot_id',
    'subplot_number',
    'subplot_id',
    'operational_taxon',
    'include_tree_allometry',
    'agb_Mg',
    'carbon_Mg',
    'basal_area_m2',
    'wood_density_g_cm3'
]

missing_columns = [
    col for col in required_columns
    if col not in master_v05.columns
]

if missing_columns:
    raise ValueError(
        f'Columnas faltantes: {missing_columns}. '
        'Revisar la ejecución de las celdas 24-27.'
    )

# -----------------------------------------------------
# 2. Rebuild biomass dataset
# -----------------------------------------------------

biomass_data = master_v05.loc[
    master_v05['include_tree_allometry']
].copy()

assert len(biomass_data) == 1002, (
    f'Se esperaban 1002 árboles, '
    f'pero se encontraron {len(biomass_data)}'
)

assert biomass_data['agb_Mg'].notna().all()
assert biomass_data['carbon_Mg'].notna().all()
assert biomass_data['wood_density_g_cm3'].notna().all()
assert biomass_data['agb_Mg'].gt(0).all()

print('BIOMASS DATASET VERIFIED')
print(f'Eligible trees: {len(biomass_data):,}')

display(
    biomass_data.groupby('forest_type')
    .size()
    .rename('n_trees')
    .to_frame()
)

# -----------------------------------------------------
# 3. Subplot-level biomass
# -----------------------------------------------------

subplot_biomass = (

    biomass_data.groupby(
        [
            'forest_type',
            'plot_id',
            'subplot_number',
            'subplot_id'
        ],
        as_index=False
    )

    .agg(
        n_eligible_trees=('tree_id', 'size'),
        agb_Mg=('agb_Mg', 'sum'),
        carbon_Mg=('carbon_Mg', 'sum'),
        basal_area_m2=('basal_area_m2', 'sum'),
        mean_wd=('wood_density_g_cm3', 'mean'),
        mean_dbh_cm=('dbh_cm', 'mean'),
        mean_height_m=('height_m', 'mean')
    )

)

subplot_biomass['subplot_area_ha'] = 0.04

subplot_biomass['agb_Mg_ha'] = (
    subplot_biomass['agb_Mg']
    / subplot_biomass['subplot_area_ha']
)

subplot_biomass['carbon_Mg_ha'] = (
    subplot_biomass['carbon_Mg']
    / subplot_biomass['subplot_area_ha']
)

subplot_biomass['basal_area_m2_ha'] = (
    subplot_biomass['basal_area_m2']
    / subplot_biomass['subplot_area_ha']
)

subplot_biomass['density_trees_ha'] = (
    subplot_biomass['n_eligible_trees']
    / subplot_biomass['subplot_area_ha']
)

assert len(subplot_biomass) == 50

print('\nSUBPLOT BIOMASS')
display(subplot_biomass.head(10).round(4))

# -----------------------------------------------------
# 4. Taxonomic contribution to biomass
# -----------------------------------------------------

taxon_biomass = (

    biomass_data.groupby(
        [
            'forest_type',
            'operational_taxon'
        ],
        as_index=False,
        dropna=False
    )

    .agg(
        n_trees=('tree_id', 'size'),
        basal_area_m2=('basal_area_m2', 'sum'),
        agb_Mg=('agb_Mg', 'sum'),
        carbon_Mg=('carbon_Mg', 'sum'),
        mean_wd=('wood_density_g_cm3', 'mean')
    )

)

taxon_biomass['agb_contribution_pct'] = (

    taxon_biomass['agb_Mg']

    / taxon_biomass.groupby('forest_type')
      ['agb_Mg'].transform('sum')

    * 100
)

taxon_biomass['abundance_pct'] = (

    taxon_biomass['n_trees']

    / taxon_biomass.groupby('forest_type')
      ['n_trees'].transform('sum')

    * 100
)

taxon_biomass['basal_area_pct'] = (

    taxon_biomass['basal_area_m2']

    / taxon_biomass.groupby('forest_type')
      ['basal_area_m2'].transform('sum')

    * 100
)

# Biomass dominance relative to numerical abundance
taxon_biomass['biomass_abundance_ratio'] = (

    taxon_biomass['agb_contribution_pct']

    / taxon_biomass['abundance_pct']

)

taxon_biomass = taxon_biomass.sort_values(
    ['forest_type', 'agb_Mg'],
    ascending=[True, False]
).reset_index(drop=True)

print('\nTOP 10 BIOMASS CONTRIBUTORS PER FOREST')

display(
    taxon_biomass.groupby('forest_type')
    .head(10)
    .round(4)
)

# -----------------------------------------------------
# 5. Mass balance verification
# -----------------------------------------------------

tree_totals = (
    biomass_data.groupby('forest_type')['agb_Mg']
    .sum()
    .sort_index()
)

subplot_totals = (
    subplot_biomass.groupby('forest_type')['agb_Mg']
    .sum()
    .sort_index()
)

taxon_totals = (
    taxon_biomass.groupby('forest_type')['agb_Mg']
    .sum()
    .sort_index()
)

mass_balance = pd.DataFrame({
    'tree_level_Mg': tree_totals,
    'subplot_level_Mg': subplot_totals,
    'taxon_level_Mg': taxon_totals
})

mass_balance['difference_Mg'] = (
    mass_balance['tree_level_Mg']
    - mass_balance['taxon_level_Mg']
)

assert np.allclose(tree_totals, subplot_totals)
assert np.allclose(tree_totals, taxon_totals)

print('\nBIOMASS MASS BALANCE')
display(mass_balance.round(6))

print('\n' + '=' * 60)
print('CELL 29 COMPLETED SUCCESSFULLY')
print('=' * 60)


In [ ]:

# =====================================================
# CELL 30: BIOMASS PARTITIONING BY DIAMETER CLASSES
# =====================================================

import pandas as pd
import numpy as np

assert 'biomass_data' in globals(), (
    'Ejecutar primero la celda 29'
)

biomass_partition = biomass_data.copy()

# -----------------------------------------------------
# 1. Diameter classes
# -----------------------------------------------------

dbh_bins = [10, 20, 30, 40, 60, 80, np.inf]

dbh_labels = [
    '10-<20',
    '20-<30',
    '30-<40',
    '40-<60',
    '60-<80',
    '>=80'
]

biomass_partition['dbh_class'] = pd.cut(
    biomass_partition['dbh_cm'],
    bins=dbh_bins,
    labels=dbh_labels,
    right=False,
    include_lowest=True
)

assert biomass_partition['dbh_class'].notna().all()

# -----------------------------------------------------
# 2. Biomass by diameter class
# -----------------------------------------------------

dbh_biomass = (
    biomass_partition.groupby(
        ['forest_type', 'dbh_class'],
        observed=False,
        as_index=False
    )
    .agg(
        n_trees=('tree_id', 'size'),
        agb_Mg=('agb_Mg', 'sum'),
        carbon_Mg=('carbon_Mg', 'sum'),
        basal_area_m2=('basal_area_m2', 'sum'),
        mean_height_m=('height_m', 'mean'),
        mean_wd=('wood_density_g_cm3', 'mean')
    )
)

dbh_biomass['abundance_pct'] = (
    dbh_biomass['n_trees']
    / dbh_biomass.groupby('forest_type')
      ['n_trees'].transform('sum')
    * 100
)

dbh_biomass['biomass_pct'] = (
    dbh_biomass['agb_Mg']
    / dbh_biomass.groupby('forest_type')
      ['agb_Mg'].transform('sum')
    * 100
)

dbh_biomass['basal_area_pct'] = (
    dbh_biomass['basal_area_m2']
    / dbh_biomass.groupby('forest_type')
      ['basal_area_m2'].transform('sum')
    * 100
)

print('BIOMASS PARTITIONING BY DBH CLASS')

display(dbh_biomass.round(4))

# -----------------------------------------------------
# 3. Contribution of large trees
# -----------------------------------------------------

large_tree_results = []

for forest, df in biomass_partition.groupby('forest_type'):

    for threshold in [30, 40, 50, 60, 80]:

        subset = df[df['dbh_cm'] >= threshold]

        large_tree_results.append({
            'forest_type': forest,
            'dbh_threshold_cm': threshold,
            'n_large_trees': len(subset),
            'abundance_pct': (
                len(subset) / len(df) * 100
            ),
            'agb_Mg': subset['agb_Mg'].sum(),
            'biomass_pct': (
                subset['agb_Mg'].sum()
                / df['agb_Mg'].sum()
                * 100
            ),
            'basal_area_pct': (
                subset['basal_area_m2'].sum()
                / df['basal_area_m2'].sum()
                * 100
            )
        })

large_tree_summary = pd.DataFrame(
    large_tree_results
)

print('\nLARGE TREE CONTRIBUTION')

display(large_tree_summary.round(4))

# -----------------------------------------------------
# 4. Biomass concentration among individuals
# -----------------------------------------------------

concentration_results = []

for forest, df in biomass_partition.groupby('forest_type'):

    ranked = df.sort_values(
        ['agb_Mg', 'tree_id'],
        ascending=[False, True]
    )

    for fraction in [0.01, 0.05, 0.10, 0.20]:

        n_top = int(np.ceil(len(ranked) * fraction))

        top = ranked.head(n_top)

        concentration_results.append({
            'forest_type': forest,
            'top_fraction_pct': fraction * 100,
            'n_trees': n_top,
            'actual_abundance_pct': (
                n_top / len(ranked) * 100
            ),
            'agb_Mg': top['agb_Mg'].sum(),
            'biomass_pct': (
                top['agb_Mg'].sum()
                / ranked['agb_Mg'].sum()
                * 100
            )
        })

biomass_concentration = pd.DataFrame(
    concentration_results
)

print('\nBIOMASS CONCENTRATION')

display(biomass_concentration.round(4))

# -----------------------------------------------------
# 5. Largest biomass individuals
# -----------------------------------------------------

top_biomass_trees = (
    biomass_partition.sort_values(
        ['forest_type', 'agb_Mg'],
        ascending=[True, False]
    )
    .groupby('forest_type')
    .head(10)
)

print('\nTOP 10 INDIVIDUAL TREES PER FOREST')

display(
    top_biomass_trees[
        [
            'forest_type',
            'tree_id',
            'operational_taxon',
            'dbh_cm',
            'height_m',
            'wood_density_g_cm3',
            'agb_Mg'
        ]
    ].round(4)
)

# -----------------------------------------------------
# 6. Mass balance
# -----------------------------------------------------

original_totals = (
    biomass_partition.groupby('forest_type')['agb_Mg']
    .sum()
    .sort_index()
)

class_totals = (
    dbh_biomass.groupby('forest_type')['agb_Mg']
    .sum()
    .sort_index()
)

assert np.allclose(
    original_totals,
    class_totals
)

print('\nALL BIOMASS PARTITIONING CHECKS PASSED')


In [ ]:

# =====================================================
# CELL 31: BIOMASS DISTRIBUTION FIGURES
# =====================================================

import matplotlib.pyplot as plt
import numpy as np

assert 'dbh_biomass' in globals()

colors = {
    'Late_secondary': '#D28C45',
    'Mature': '#28664F'
}

forests = ['Late_secondary', 'Mature']

forest_labels = {
    'Late_secondary': 'Late-secondary forest',
    'Mature': 'Mature forest'
}

fig, axes = plt.subplots(
    1, 3,
    figsize=(16, 5)
)

x = np.arange(len(dbh_labels))
width = 0.36

# -----------------------------------------------------
# A. Abundance distribution
# -----------------------------------------------------

for i, forest in enumerate(forests):

    df = (
        dbh_biomass[
            dbh_biomass['forest_type'] == forest
        ]
        .set_index('dbh_class')
        .reindex(dbh_labels)
    )

    axes[0].bar(
        x + (i - 0.5) * width,
        df['abundance_pct'],
        width=width,
        color=colors[forest],
        label=forest_labels[forest]
    )

axes[0].set_ylabel('Relative abundance (%)')
axes[0].set_title('(a) Stem abundance')

# -----------------------------------------------------
# B. Biomass distribution
# -----------------------------------------------------

for i, forest in enumerate(forests):

    df = (
        dbh_biomass[
            dbh_biomass['forest_type'] == forest
        ]
        .set_index('dbh_class')
        .reindex(dbh_labels)
    )

    axes[1].bar(
        x + (i - 0.5) * width,
        df['biomass_pct'],
        width=width,
        color=colors[forest]
    )

axes[1].set_ylabel('Relative aboveground biomass (%)')
axes[1].set_title('(b) Biomass allocation')

# -----------------------------------------------------
# C. Absolute biomass
# -----------------------------------------------------

for i, forest in enumerate(forests):

    df = (
        dbh_biomass[
            dbh_biomass['forest_type'] == forest
        ]
        .set_index('dbh_class')
        .reindex(dbh_labels)
    )

    axes[2].bar(
        x + (i - 0.5) * width,
        df['agb_Mg'],
        width=width,
        color=colors[forest]
    )

axes[2].set_ylabel(
    'Aboveground biomass (Mg ha$^{-1}$)'
)

axes[2].set_title('(c) Absolute biomass')

# -----------------------------------------------------
# Figure formatting
# -----------------------------------------------------

for ax in axes:

    ax.set_xticks(x)
    ax.set_xticklabels(
        dbh_labels,
        rotation=35,
        ha='right'
    )

    ax.set_xlabel('Diameter class (cm)')

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.grid(
        axis='y',
        alpha=0.18,
        linestyle='--'
    )

    ax.set_axisbelow(True)

axes[0].legend(
    frameon=False,
    fontsize=9
)

plt.tight_layout()

figure_dir = DIRS['outputs'] / 'BIOMASS'
figure_dir.mkdir(
    parents=True,
    exist_ok=True
)

fig.savefig(
    figure_dir / 'Fig_Biomass_Diameter_Classes.png',
    dpi=600,
    bbox_inches='tight'
)

fig.savefig(
    figure_dir / 'Fig_Biomass_Diameter_Classes.pdf',
    bbox_inches='tight'
)

plt.show()

print('BIOMASS FIGURE SAVED')


In [ ]:

# =====================================================
# CELL 32: EXACT BIOMASS DIFFERENCE DECOMPOSITION
# =====================================================

import pandas as pd
import numpy as np

assert 'dbh_biomass' in globals()
assert 'biomass_data' in globals()

# -----------------------------------------------------
# 1. Prepare diameter-class data
# -----------------------------------------------------

decomp_data = dbh_biomass.copy()

decomp_data['mean_tree_agb_Mg'] = (
    decomp_data['agb_Mg']
    / decomp_data['n_trees']
)

secondary = (
    decomp_data[
        decomp_data['forest_type'] == 'Late_secondary'
    ]
    .set_index('dbh_class')
    .reindex(dbh_labels)
)

mature = (
    decomp_data[
        decomp_data['forest_type'] == 'Mature'
    ]
    .set_index('dbh_class')
    .reindex(dbh_labels)
)

assert (secondary['n_trees'] > 0).all()
assert (mature['n_trees'] > 0).all()

# -----------------------------------------------------
# 2. Exact symmetric decomposition
# -----------------------------------------------------

decomposition = pd.DataFrame({
    'dbh_class': dbh_labels,

    'N_secondary':
        secondary['n_trees'].to_numpy(),

    'N_mature':
        mature['n_trees'].to_numpy(),

    'mean_AGB_secondary':
        secondary['mean_tree_agb_Mg'].to_numpy(),

    'mean_AGB_mature':
        mature['mean_tree_agb_Mg'].to_numpy(),

    'AGB_secondary':
        secondary['agb_Mg'].to_numpy(),

    'AGB_mature':
        mature['agb_Mg'].to_numpy()
})

decomposition['observed_difference_Mg'] = (
    decomposition['AGB_secondary']
    - decomposition['AGB_mature']
)

# Abundance component

decomposition['abundance_component_Mg'] = (
    (
        decomposition['N_secondary']
        - decomposition['N_mature']
    )
    *
    (
        decomposition['mean_AGB_secondary']
        + decomposition['mean_AGB_mature']
    ) / 2
)

# Mean individual biomass component

decomposition['individual_biomass_component_Mg'] = (
    (
        decomposition['mean_AGB_secondary']
        - decomposition['mean_AGB_mature']
    )
    *
    (
        decomposition['N_secondary']
        + decomposition['N_mature']
    ) / 2
)

decomposition['reconstructed_difference_Mg'] = (
    decomposition['abundance_component_Mg']
    + decomposition['individual_biomass_component_Mg']
)

decomposition['residual_Mg'] = (
    decomposition['observed_difference_Mg']
    - decomposition['reconstructed_difference_Mg']
)

assert np.allclose(
    decomposition['observed_difference_Mg'],
    decomposition['reconstructed_difference_Mg']
)

total_difference = (
    decomposition['observed_difference_Mg'].sum()
)

decomposition['net_difference_contribution_pct'] = (
    decomposition['observed_difference_Mg']
    / total_difference * 100
)

print('BIOMASS DIFFERENCE DECOMPOSITION')

display(decomposition.round(4))

# -----------------------------------------------------
# 3. Whole-forest component contributions
# -----------------------------------------------------

decomposition_summary = pd.DataFrame({
    'component': [
        'Stem abundance',
        'Mean individual biomass',
        'Total observed difference'
    ],

    'contribution_Mg': [
        decomposition['abundance_component_Mg'].sum(),
        decomposition[
            'individual_biomass_component_Mg'
        ].sum(),
        total_difference
    ]
})

decomposition_summary['contribution_pct'] = (
    decomposition_summary['contribution_Mg']
    / total_difference * 100
)

print('\nTOTAL COMPONENT CONTRIBUTIONS')

display(decomposition_summary.round(4))

# -----------------------------------------------------
# 4. Independent whole-plot decomposition
# -----------------------------------------------------

plot_stats = (
    biomass_data.groupby('forest_type')
    .agg(
        N=('tree_id', 'size'),
        total_agb_Mg=('agb_Mg', 'sum'),
        mean_tree_agb_Mg=('agb_Mg', 'mean')
    )
)

s = plot_stats.loc['Late_secondary']
m = plot_stats.loc['Mature']

plot_abundance = (
    (s['N'] - m['N'])
    *
    (s['mean_tree_agb_Mg']
     + m['mean_tree_agb_Mg']) / 2
)

plot_individual = (
    (s['mean_tree_agb_Mg']
     - m['mean_tree_agb_Mg'])
    *
    (s['N'] + m['N']) / 2
)

plot_decomposition = pd.DataFrame({
    'component': [
        'Total stem abundance',
        'Mean individual biomass'
    ],
    'contribution_Mg': [
        plot_abundance,
        plot_individual
    ]
})

plot_decomposition['contribution_pct'] = (
    plot_decomposition['contribution_Mg']
    / total_difference * 100
)

assert np.isclose(
    plot_abundance + plot_individual,
    total_difference
)

print('\nWHOLE-PLOT DECOMPOSITION')

display(plot_stats.round(4))
display(plot_decomposition.round(4))

# -----------------------------------------------------
# 5. Save outputs
# -----------------------------------------------------

decomp_dir = DIRS['analysis'] / 'BIOMASS_DECOMPOSITION'

decomp_dir.mkdir(
    parents=True,
    exist_ok=True
)

decomposition.to_csv(
    decomp_dir / 'DBH_BIOMASS_DECOMPOSITION.csv',
    index=False
)

decomposition_summary.to_csv(
    decomp_dir / 'BIOMASS_COMPONENT_SUMMARY.csv',
    index=False
)

plot_decomposition.to_csv(
    decomp_dir / 'WHOLE_PLOT_DECOMPOSITION.csv',
    index=False
)

print('\nCELL 32 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 33: WOOD DENSITY SENSITIVITY ANALYSIS
# =====================================================

import pandas as pd
import numpy as np

assert 'biomass_data' in globals()

sensitivity_data = biomass_data.copy()

required_columns = [
    'tree_id',
    'forest_type',
    'dbh_cm',
    'height_m',
    'wood_density_g_cm3',
    'wd_level',
    'agb_Mg'
]

assert all(
    c in sensitivity_data.columns
    for c in required_columns
)

assert len(sensitivity_data) == 1002

assert (
    sensitivity_data['wood_density_g_cm3'] > 0
).all()

# -----------------------------------------------------
# 1. Reference values from GWDD matching
# -----------------------------------------------------

POOLED_WD = 0.5526
WD_Q25 = 0.4625
WD_Q75 = 0.6500

rho = (
    sensitivity_data['wood_density_g_cm3']
    .to_numpy(dtype=float)
)

D = sensitivity_data['dbh_cm'].to_numpy(dtype=float)
H = sensitivity_data['height_m'].to_numpy(dtype=float)

community_mask = (
    sensitivity_data['wd_level']
    .eq('community_pooled')
    .to_numpy()
)

imputed_mask = (
    sensitivity_data['wd_level']
    .isin([
        'genus',
        'family',
        'community_pooled'
    ])
    .to_numpy()
)

secondary_mask = (
    sensitivity_data['forest_type']
    .eq('Late_secondary')
    .to_numpy()
)

mature_mask = (
    sensitivity_data['forest_type']
    .eq('Mature')
    .to_numpy()
)

# -----------------------------------------------------
# 2. Chave et al. (2014) allometric equation
# -----------------------------------------------------

def estimate_agb_Mg(density, diameter, height):

    return (
        0.0673
        * (density * diameter**2 * height)**0.976
        / 1000
    )

baseline_check = estimate_agb_Mg(rho, D, H)

assert np.allclose(
    baseline_check,
    sensitivity_data['agb_Mg'],
    rtol=1e-5,
    atol=1e-8
), 'Baseline equation does not match previous AGB'

# -----------------------------------------------------
# 3. Define sensitivity scenarios
# -----------------------------------------------------

scenarios = {}

scenarios['Baseline'] = rho.copy()

# Community-level imputations only

r = rho.copy()
r[community_mask] = WD_Q25
scenarios['Community_low'] = r

r = rho.copy()
r[community_mask] = WD_Q75
scenarios['Community_high'] = r

# All non-species-level imputations

r = rho.copy()
r[imputed_mask] *= 0.80
scenarios['Imputed_low_20pct'] = r

r = rho.copy()
r[imputed_mask] *= 1.20
scenarios['Imputed_high_20pct'] = r

# Common wood density across all trees

scenarios['Common_density'] = np.full(
    len(rho),
    POOLED_WD
)

# Conservative opposing perturbations:
# reduce secondary imputations, increase mature ones

r = rho.copy()
r[imputed_mask & secondary_mask] *= 0.80
r[imputed_mask & mature_mask] *= 1.20

scenarios['Opposing_reduce_gap'] = r

# Opposite scenario

r = rho.copy()
r[imputed_mask & secondary_mask] *= 1.20
r[imputed_mask & mature_mask] *= 0.80

scenarios['Opposing_increase_gap'] = r

# -----------------------------------------------------
# 4. Estimate AGB under each scenario
# -----------------------------------------------------

sensitivity_results = []

for scenario_name, scenario_rho in scenarios.items():

    assert np.isfinite(scenario_rho).all()
    assert (scenario_rho > 0).all()

    agb = estimate_agb_Mg(
        scenario_rho,
        D,
        H
    )

    agb_secondary = agb[secondary_mask].sum()
    agb_mature = agb[mature_mask].sum()

    sensitivity_results.append({
        'scenario': scenario_name,

        'secondary_AGB_Mg_ha':
            agb_secondary,

        'mature_AGB_Mg_ha':
            agb_mature,

        'difference_Mg_ha':
            agb_secondary - agb_mature,

        'secondary_mature_ratio':
            agb_secondary / agb_mature,

        'secondary_excess_pct':
            (agb_secondary / agb_mature - 1) * 100
    })

sensitivity_summary = pd.DataFrame(
    sensitivity_results
)

baseline_difference = (
    sensitivity_summary.loc[
        sensitivity_summary['scenario'] == 'Baseline',
        'difference_Mg_ha'
    ].iloc[0]
)

sensitivity_summary['gap_change_pct'] = (
    (
        sensitivity_summary['difference_Mg_ha']
        - baseline_difference
    )
    / baseline_difference * 100
)

print('WOOD DENSITY SENSITIVITY RESULTS')

display(sensitivity_summary.round(4))

# -----------------------------------------------------
# 5. Biomass contribution by WD assignment level
# -----------------------------------------------------

wd_biomass = (
    sensitivity_data.groupby(
        ['forest_type', 'wd_level'],
        as_index=False
    )
    .agg(
        n_trees=('tree_id', 'size'),
        agb_Mg=('agb_Mg', 'sum'),
        basal_area_m2=('basal_area_m2', 'sum')
    )
)

wd_biomass['biomass_pct'] = (
    wd_biomass['agb_Mg']
    / wd_biomass.groupby('forest_type')
      ['agb_Mg'].transform('sum')
    * 100
)

wd_biomass['abundance_pct'] = (
    wd_biomass['n_trees']
    / wd_biomass.groupby('forest_type')
      ['n_trees'].transform('sum')
    * 100
)

print('\nBIOMASS BY WOOD DENSITY ASSIGNMENT LEVEL')

display(wd_biomass.round(4))

# -----------------------------------------------------
# 6. Save sensitivity results
# -----------------------------------------------------

sensitivity_dir = (
    DIRS['analysis'] / 'BIOMASS_SENSITIVITY'
)

sensitivity_dir.mkdir(
    parents=True,
    exist_ok=True
)

sensitivity_summary.to_csv(
    sensitivity_dir / 'WD_SENSITIVITY_SUMMARY.csv',
    index=False
)

wd_biomass.to_csv(
    sensitivity_dir / 'WD_BIOMASS_CONTRIBUTION.csv',
    index=False
)

print('\nROBUSTNESS CHECK')

if (
    sensitivity_summary['difference_Mg_ha'] > 0
).all():

    print(
        'Secondary forest retains higher AGB '
        'under all tested scenarios.'
    )

else:

    print(
        'AGB ranking changes under at least '
        'one sensitivity scenario.'
    )

print('\nCELL 33 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 34: TAXONOMIC DIVERSITY AND HILL NUMBERS
# =====================================================

import pandas as pd
import numpy as np
from pathlib import Path

# -----------------------------------------------------
# 1. Locate latest tree master
# -----------------------------------------------------

required_columns = {
    'tree_id',
    'forest_type',
    'operational_taxon',
    'species_accepted',
    'taxonomic_status',
    'include_floristic'
}

candidate_names = [
    'master_v04',
    'master_v03',
    'tree_master_v04',
    'tree_master_v03',
    'master'
]

diversity_master = None

for variable_name in candidate_names:

    obj = globals().get(variable_name)

    if isinstance(obj, pd.DataFrame):

        if required_columns.issubset(obj.columns):

            diversity_master = obj.copy()

            print(
                'Using dataframe:',
                variable_name
            )

            break

# If necessary, recover saved master from Drive

if diversity_master is None:

    master_files = sorted(
        DIRS['master'].rglob(
            'SATIPO_TREE_MASTER_V0*.csv'
        ),
        reverse=True
    )

    for file in master_files:

        candidate = pd.read_csv(file)

        if required_columns.issubset(
            candidate.columns
        ):

            diversity_master = candidate

            print('Loaded:', file)

            break

assert diversity_master is not None, (
    'No compatible taxonomic master found'
)

assert diversity_master['tree_id'].is_unique

# -----------------------------------------------------
# 2. Prepare floristic inventory
# -----------------------------------------------------

floristic_mask = (
    diversity_master['include_floristic']
    .astype(str)
    .str.lower()
    .isin(['true', '1', '1.0'])
)

diversity_data = diversity_master.loc[
    floristic_mask
].copy()

assert len(diversity_data) == 1009

assert (
    diversity_data['operational_taxon']
    .notna()
    .all()
)

print('\nFLORISTIC INVENTORY')

display(
    diversity_data.groupby('forest_type')
    .size()
    .rename('n_individuals')
    .to_frame()
)

# -----------------------------------------------------
# 3. Hill diversity function
# -----------------------------------------------------

def hill_diversity(counts, q):

    counts = np.asarray(
        counts,
        dtype=float
    )

    counts = counts[counts > 0]

    if len(counts) == 0:
        return np.nan

    p = counts / counts.sum()

    if np.isclose(q, 0):

        return float(len(p))

    elif np.isclose(q, 1):

        return float(
            np.exp(
                -np.sum(p * np.log(p))
            )
        )

    else:

        return float(
            np.sum(p**q)**(1 / (1-q))
        )

# -----------------------------------------------------
# 4. Define taxonomic scenarios
# -----------------------------------------------------

species_only = diversity_data.loc[
    diversity_data['taxonomic_status']
    .eq('resolved_species')
    &
    diversity_data['species_accepted'].notna()
].copy()

without_SI = diversity_data.loc[
    ~diversity_data['taxonomic_status']
    .eq('unidentified_field_code')
].copy()

taxonomic_scenarios = {

    'All_operational_taxa': (
        diversity_data,
        'operational_taxon'
    ),

    'Resolved_species_only': (
        species_only,
        'species_accepted'
    ),

    'Operational_without_SI': (
        without_SI,
        'operational_taxon'
    )
}

# -----------------------------------------------------
# 5. Calculate diversity metrics
# -----------------------------------------------------

hill_results = []

for scenario, (data, taxon_col) in (
    taxonomic_scenarios.items()
):

    for forest, group in data.groupby(
        'forest_type'
    ):

        counts = (
            group[taxon_col]
            .value_counts()
            .to_numpy()
        )

        N = counts.sum()
        S = len(counts)

        p = counts / N

        H = -np.sum(p * np.log(p))

        simpson_concentration = np.sum(p**2)

        f1 = int((counts == 1).sum())
        f2 = int((counts == 2).sum())

        hill_results.append({

            'scenario': scenario,

            'forest_type': forest,

            'n_individuals': N,

            'observed_richness': S,

            'hill_q0': hill_diversity(
                counts, 0
            ),

            'hill_q1': hill_diversity(
                counts, 1
            ),

            'hill_q2': hill_diversity(
                counts, 2
            ),

            'shannon_H': H,

            'simpson_1_minus_D':
                1 - simpson_concentration,

            'pielou_evenness':
                H / np.log(S)
                if S > 1 else np.nan,

            'hill_evenness_q1_q0':
                np.exp(H) / S,

            'dominant_taxon_pct':
                p.max() * 100,

            'singletons': f1,

            'doubletons': f2

        })

hill_summary = pd.DataFrame(hill_results)

print('\nHILL DIVERSITY RESULTS')

display(hill_summary.round(4))

# -----------------------------------------------------
# 6. Diversity profiles
# -----------------------------------------------------

q_values = np.linspace(0, 3, 61)

profile_results = []

for scenario, (data, taxon_col) in (
    taxonomic_scenarios.items()
):

    for forest, group in data.groupby(
        'forest_type'
    ):

        counts = (
            group[taxon_col]
            .value_counts()
            .to_numpy()
        )

        for q in q_values:

            profile_results.append({

                'scenario': scenario,

                'forest_type': forest,

                'q': q,

                'effective_taxa':
                    hill_diversity(counts, q)

            })

hill_profiles = pd.DataFrame(
    profile_results
)

# -----------------------------------------------------
# 7. Mature / secondary diversity ratios
# -----------------------------------------------------

diversity_ratios = []

for scenario, group in hill_summary.groupby(
    'scenario'
):

    s = group.loc[
        group['forest_type']
        .eq('Late_secondary')
    ].iloc[0]

    m = group.loc[
        group['forest_type']
        .eq('Mature')
    ].iloc[0]

    for metric in [
        'hill_q0',
        'hill_q1',
        'hill_q2'
    ]:

        diversity_ratios.append({

            'scenario': scenario,

            'metric': metric,

            'secondary': s[metric],

            'mature': m[metric],

            'mature_secondary_ratio':
                m[metric] / s[metric],

            'mature_excess_pct':
                (m[metric] / s[metric] - 1)
                * 100

        })

hill_ratios = pd.DataFrame(
    diversity_ratios
)

print('\nDIVERSITY CONTRAST BETWEEN FORESTS')

display(hill_ratios.round(4))

# -----------------------------------------------------
# 8. Export
# -----------------------------------------------------

diversity_dir = (
    DIRS['analysis'] / 'TAXONOMIC_DIVERSITY'
)

diversity_dir.mkdir(
    parents=True,
    exist_ok=True
)

hill_summary.to_csv(
    diversity_dir / 'HILL_DIVERSITY_SUMMARY.csv',
    index=False
)

hill_profiles.to_csv(
    diversity_dir / 'HILL_DIVERSITY_PROFILES.csv',
    index=False
)

hill_ratios.to_csv(
    diversity_dir / 'HILL_DIVERSITY_RATIOS.csv',
    index=False
)

print('\nCELL 34 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 35: SAMPLE COMPLETENESS AND RAREFACTION
# =====================================================

import numpy as np
import pandas as pd

from scipy.special import gammaln

# -----------------------------------------------------
# 1. Sample coverage estimator
# -----------------------------------------------------

def sample_coverage(counts):

    counts = np.asarray(counts, dtype=int)

    counts = counts[counts > 0]

    n = counts.sum()

    f1 = np.sum(counts == 1)
    f2 = np.sum(counts == 2)

    if n <= 1:
        return np.nan

    if f1 == 0:
        return 1.0

    if f2 > 0:

        A = (
            (n-1)*f1
            /
            ((n-1)*f1 + 2*f2)
        )

    elif f1 > 1:

        A = (
            (n-1)*(f1-1)
            /
            ((n-1)*(f1-1) + 2)
        )

    else:

        A = 0.0

    return 1 - (f1/n)*A

# -----------------------------------------------------
# 2. Bias-corrected Chao1
# -----------------------------------------------------

def chao1_bc(counts):

    counts = np.asarray(counts, dtype=int)

    counts = counts[counts > 0]

    n = counts.sum()

    Sobs = len(counts)

    f1 = np.sum(counts == 1)
    f2 = np.sum(counts == 2)

    if n <= 1:
        return float(Sobs)

    return (
        Sobs
        +
        ((n-1)/n)
        *
        f1*(f1-1)
        /
        (2*(f2+1))
    )

# -----------------------------------------------------
# 3. Exact individual-based rarefaction
# -----------------------------------------------------

def log_comb(n, k):

    if k < 0 or k > n:
        return -np.inf

    return (
        gammaln(n+1)
        - gammaln(k+1)
        - gammaln(n-k+1)
    )

def rarefied_richness(counts, m):

    counts = np.asarray(counts, dtype=int)

    counts = counts[counts > 0]

    N = counts.sum()

    m = int(m)

    if m < 1 or m > N:

        return np.nan

    denominator = log_comb(N, m)

    expected = 0.0

    for ni in counts:

        if N-ni < m:

            p_absent = 0.0

        else:

            p_absent = np.exp(
                log_comb(N-ni, m)
                - denominator
            )

        expected += 1-p_absent

    return expected

# -----------------------------------------------------
# 4. Coverage at interpolated sample sizes
# -----------------------------------------------------

def interpolated_coverage(counts, m):

    counts = np.asarray(counts, dtype=int)

    counts = counts[counts > 0]

    N = counts.sum()

    m = int(m)

    if m < 1 or m > N:

        return np.nan

    if m == N:

        return sample_coverage(counts)

    denominator = log_comb(N-1, m)

    uncovered = 0.0

    for ni in counts:

        if N-ni < m:

            p_absent = 0.0

        else:

            p_absent = np.exp(
                log_comb(N-ni, m)
                - denominator
            )

        uncovered += (
            ni/N
        ) * p_absent

    return float(
        np.clip(1-uncovered, 0, 1)
    )

# -----------------------------------------------------
# 5. Evaluate taxonomic scenarios
# -----------------------------------------------------

completeness_results = []
rarefaction_results = []
standardized_results = []

for scenario, (data, taxon_col) in (
    taxonomic_scenarios.items()
):

    forest_counts = {}

    for forest, group in data.groupby(
        'forest_type'
    ):

        counts = (
            group[taxon_col]
            .value_counts()
            .to_numpy(dtype=int)
        )

        forest_counts[forest] = counts

        N = counts.sum()

        Sobs = len(counts)

        estimated_richness = chao1_bc(
            counts
        )

        coverage = sample_coverage(
            counts
        )

        completeness_results.append({

            'scenario': scenario,

            'forest_type': forest,

            'n_individuals': N,

            'observed_richness': Sobs,

            'singletons':
                int((counts == 1).sum()),

            'doubletons':
                int((counts == 2).sum()),

            'sample_coverage':
                coverage,

            'sample_coverage_pct':
                coverage*100,

            'chao1_bias_corrected':
                estimated_richness,

            'richness_completeness_pct':
                Sobs / estimated_richness
                * 100

        })

        sample_sizes = np.unique(
            np.linspace(
                1,
                N,
                150
            ).astype(int)
        )

        for m in sample_sizes:

            rarefaction_results.append({

                'scenario': scenario,

                'forest_type': forest,

                'sample_size': m,

                'expected_richness':
                    rarefied_richness(
                        counts, m
                    ),

                'sample_coverage':
                    interpolated_coverage(
                        counts, m
                    )

            })

    # Common sample size

    common_n = min(
        c.sum()
        for c in forest_counts.values()
    )

    for forest, counts in (
        forest_counts.items()
    ):

        standardized_results.append({

            'scenario': scenario,

            'forest_type': forest,

            'common_sample_size':
                common_n,

            'rarefied_richness':
                rarefied_richness(
                    counts,
                    common_n
                ),

            'coverage_at_common_n':
                interpolated_coverage(
                    counts,
                    common_n
                )

        })

# -----------------------------------------------------
# 6. Output tables
# -----------------------------------------------------

completeness_summary = pd.DataFrame(
    completeness_results
)

rarefaction_curves = pd.DataFrame(
    rarefaction_results
)

standardized_richness = pd.DataFrame(
    standardized_results
)

print('SAMPLE COMPLETENESS')

display(
    completeness_summary.round(4)
)

print('\nRAREFACTION AT COMMON SAMPLE SIZE')

display(
    standardized_richness.round(4)
)

# -----------------------------------------------------
# 7. Verify rarefaction identities
# -----------------------------------------------------

for scenario, (data, taxon_col) in (
    taxonomic_scenarios.items()
):

    for forest, group in data.groupby(
        'forest_type'
    ):

        counts = (
            group[taxon_col]
            .value_counts()
            .to_numpy(dtype=int)
        )

        assert np.isclose(
            rarefied_richness(
                counts,
                counts.sum()
            ),
            len(counts)
        )

        assert np.isclose(
            rarefied_richness(
                counts, 1
            ),
            1
        )

# -----------------------------------------------------
# 8. Export
# -----------------------------------------------------

completeness_summary.to_csv(
    diversity_dir / 'SAMPLE_COMPLETENESS.csv',
    index=False
)

rarefaction_curves.to_csv(
    diversity_dir / 'RAREFACTION_CURVES.csv',
    index=False
)

standardized_richness.to_csv(
    diversity_dir / 'STANDARDIZED_RICHNESS.csv',
    index=False
)

print('\nALL RAREFACTION CHECKS PASSED')

print('CELL 35 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 36: TAXONOMIC DIVERSITY FIGURE
# =====================================================

import matplotlib.pyplot as plt
import numpy as np

# -----------------------------------------------------
# 1. Plot configuration
# -----------------------------------------------------

plt.rcParams.update({

    'font.family': 'DejaVu Sans',

    'font.size': 11,

    'axes.labelsize': 11,

    'axes.titlesize': 12,

    'axes.spines.top': False,

    'axes.spines.right': False,

    'legend.frameon': False,

    'savefig.dpi': 600

})

forest_order = [
    'Late_secondary',
    'Mature'
]

forest_colors = {

    'Late_secondary': '#D58E45',

    'Mature': '#27634F'

}

forest_names = {

    'Late_secondary':
        'Late-secondary forest',

    'Mature':
        'Mature forest'

}

scenario = 'All_operational_taxa'

fig, axes = plt.subplots(
    1,
    3,
    figsize=(16, 4.8)
)

# -----------------------------------------------------
# 2. Panel A: Hill diversity
# -----------------------------------------------------

ax = axes[0]

plot_hill = hill_summary.loc[
    hill_summary['scenario'].eq(scenario)
]

x = np.arange(3)

width = 0.35

metrics = [
    'hill_q0',
    'hill_q1',
    'hill_q2'
]

for i, forest in enumerate(
    forest_order
):

    row = plot_hill.loc[
        plot_hill['forest_type']
        .eq(forest)
    ].iloc[0]

    values = [
        row[m]
        for m in metrics
    ]

    bars = ax.bar(

        x + (i-0.5)*width,

        values,

        width=width,

        color=forest_colors[forest],

        label=forest_names[forest]

    )

    ax.bar_label(
        bars,
        fmt='%.1f',
        padding=3,
        fontsize=9
    )

ax.set_xticks(
    x,
    [
        'q = 0',
        'q = 1',
        'q = 2'
    ]
)

ax.set_ylabel(
    'Effective number of taxa'
)

ax.set_xlabel(
    'Diversity order'
)

ax.set_title(
    '(a) Hill diversity'
)

ax.legend(
    fontsize=9
)

ax.set_ylim(
    0,
    plot_hill['hill_q0'].max()*1.22
)

# -----------------------------------------------------
# 3. Panel B: Diversity profiles
# -----------------------------------------------------

ax = axes[1]

for forest in forest_order:

    d = hill_profiles.loc[

        hill_profiles['scenario']
        .eq(scenario)

        &

        hill_profiles['forest_type']
        .eq(forest)

    ].sort_values('q')

    ax.plot(

        d['q'],

        d['effective_taxa'],

        color=forest_colors[forest],

        linewidth=2.5

    )

ax.set_xlabel(
    'Diversity order (q)'
)

ax.set_ylabel(
    'Effective number of taxa'
)

ax.set_title(
    '(b) Diversity profiles'
)

# -----------------------------------------------------
# 4. Panel C: Individual-based rarefaction
# -----------------------------------------------------

ax = axes[2]

for forest in forest_order:

    d = rarefaction_curves.loc[

        rarefaction_curves['scenario']
        .eq(scenario)

        &

        rarefaction_curves['forest_type']
        .eq(forest)

    ].sort_values('sample_size')

    ax.plot(

        d['sample_size'],

        d['expected_richness'],

        color=forest_colors[forest],

        linewidth=2.5

    )

    ax.scatter(

        d['sample_size'].iloc[-1],

        d['expected_richness'].iloc[-1],

        color=forest_colors[forest],

        s=45,

        zorder=5

    )

common_n = (
    standardized_richness.loc[
        standardized_richness['scenario']
        .eq(scenario),
        'common_sample_size'
    ].iloc[0]
)

ax.axvline(

    common_n,

    color='gray',

    linestyle='--',

    linewidth=1.2,

    label=f'Common n = {common_n}'

)

ax.set_xlabel(
    'Number of individuals'
)

ax.set_ylabel(
    'Expected taxon richness'
)

ax.set_title(
    '(c) Individual-based rarefaction'
)

ax.legend(
    fontsize=9
)

# -----------------------------------------------------
# 5. Final styling
# -----------------------------------------------------

for ax in axes:

    ax.grid(

        axis='y',

        linestyle='--',

        alpha=0.25

    )

    ax.set_axisbelow(True)

fig.tight_layout()

# -----------------------------------------------------
# 6. Export publication-quality figures
# -----------------------------------------------------

figure_dir = (
    DIRS['outputs'] / 'FIGURES'
)

figure_dir.mkdir(
    parents=True,
    exist_ok=True
)

for extension in [
    'png',
    'pdf',
    'svg'
]:

    fig.savefig(

        figure_dir
        /
        f'FIG_TAXONOMIC_DIVERSITY.{extension}',

        dpi=600,

        bbox_inches='tight'

    )

plt.show()

print(
    'FIGURE SAVED IN:',
    figure_dir
)

print(
    'CELL 36 COMPLETED SUCCESSFULLY'
)


In [ ]:

# =====================================================
# CELL 37: BETA DIVERSITY AND TAXONOMIC TURNOVER
# =====================================================

import numpy as np
import pandas as pd

# -----------------------------------------------------
# 1. Define taxonomic comparison scenarios
# -----------------------------------------------------

beta_scenarios = {}

for scenario, (data, taxon_col) in (
    taxonomic_scenarios.items()
):

    d = data.copy()

    d['beta_taxon'] = (
        d[taxon_col].astype(str)
    )

    beta_scenarios[scenario] = d

# Additional conservative scenario:
# unresolved taxa are not assumed to be identical
# between the two plots.

d = diversity_data.copy()

d['beta_taxon'] = (
    d['operational_taxon'].astype(str)
)

unresolved = (
    ~d['taxonomic_status']
    .eq('resolved_species')
)

d.loc[unresolved, 'beta_taxon'] = (

    d.loc[unresolved, 'forest_type']
    .astype(str)

    + '::' +

    d.loc[unresolved, 'operational_taxon']
    .astype(str)

)

beta_scenarios[
    'Plot_specific_unresolved_taxa'
] = d

# -----------------------------------------------------
# 2. Beta diversity functions
# -----------------------------------------------------

def calculate_beta(x, y):

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    px = x > 0
    py = y > 0

    # Shared taxa
    a = int(np.sum(px & py))

    # Unique to secondary forest
    b = int(np.sum(px & ~py))

    # Unique to mature forest
    c = int(np.sum(~px & py))

    jaccard = (
        (b+c)/(a+b+c)
        if a+b+c > 0
        else np.nan
    )

    sorensen = (
        (b+c)/(2*a+b+c)
        if 2*a+b+c > 0
        else np.nan
    )

    turnover = (
        min(b,c)/(a+min(b,c))
        if a+min(b,c) > 0
        else np.nan
    )

    nestedness = sorensen - turnover

    bray_curtis = (
        np.abs(x-y).sum()
        /
        (x+y).sum()
    )

    # Relative-abundance Bray-Curtis

    xr = x / x.sum()
    yr = y / y.sum()

    bray_relative = (
        np.abs(xr-yr).sum()
        /
        (xr+yr).sum()
    )

    return {

        'shared_taxa': a,

        'secondary_unique': b,

        'mature_unique': c,

        'secondary_richness': a+b,

        'mature_richness': a+c,

        'total_gamma_richness': a+b+c,

        'jaccard_dissimilarity':
            jaccard,

        'sorensen_dissimilarity':
            sorensen,

        'simpson_turnover':
            turnover,

        'nestedness_component':
            nestedness,

        'turnover_pct':
            turnover/sorensen*100
            if sorensen > 0
            else np.nan,

        'nestedness_pct':
            nestedness/sorensen*100
            if sorensen > 0
            else np.nan,

        'bray_curtis_raw':
            bray_curtis,

        'bray_curtis_relative':
            bray_relative

    }

# -----------------------------------------------------
# 3. Construct abundance matrices
# -----------------------------------------------------

beta_results = []

beta_matrices = {}

taxon_membership = []

for scenario, data in (
    beta_scenarios.items()
):

    matrix = pd.crosstab(

        data['forest_type'],

        data['beta_taxon']

    ).reindex(

        ['Late_secondary', 'Mature'],

        fill_value=0

    )

    beta_matrices[scenario] = matrix

    x = matrix.loc[
        'Late_secondary'
    ].to_numpy()

    y = matrix.loc[
        'Mature'
    ].to_numpy()

    results = calculate_beta(x, y)

    results['scenario'] = scenario

    beta_results.append(results)

    # Identify shared and exclusive taxa

    for taxon in matrix.columns:

        ns = int(
            matrix.loc[
                'Late_secondary',
                taxon
            ]
        )

        nm = int(
            matrix.loc[
                'Mature',
                taxon
            ]
        )

        if ns > 0 and nm > 0:

            membership = 'Shared'

        elif ns > 0:

            membership = (
                'Secondary_only'
            )

        else:

            membership = (
                'Mature_only'
            )

        taxon_membership.append({

            'scenario': scenario,

            'taxon': taxon,

            'n_secondary': ns,

            'n_mature': nm,

            'membership': membership

        })

beta_summary = pd.DataFrame(
    beta_results
)

beta_membership = pd.DataFrame(
    taxon_membership
)

# -----------------------------------------------------
# 4. Display results
# -----------------------------------------------------

print('TAXONOMIC OVERLAP')

display(

    beta_summary[[

        'scenario',

        'secondary_richness',

        'mature_richness',

        'shared_taxa',

        'secondary_unique',

        'mature_unique',

        'total_gamma_richness'

    ]]

)

print('\nBETA DIVERSITY')

display(

    beta_summary[[

        'scenario',

        'jaccard_dissimilarity',

        'sorensen_dissimilarity',

        'simpson_turnover',

        'nestedness_component',

        'turnover_pct',

        'nestedness_pct',

        'bray_curtis_raw',

        'bray_curtis_relative'

    ]].round(4)

)

# -----------------------------------------------------
# 5. Validate beta partitioning
# -----------------------------------------------------

assert np.allclose(

    beta_summary[
        'sorensen_dissimilarity'
    ],

    beta_summary[
        'simpson_turnover'
    ]

    +

    beta_summary[
        'nestedness_component'
    ]

)

assert np.allclose(

    beta_summary[
        'total_gamma_richness'
    ],

    beta_summary[
        'shared_taxa'
    ]

    +

    beta_summary[
        'secondary_unique'
    ]

    +

    beta_summary[
        'mature_unique'
    ]

)

# -----------------------------------------------------
# 6. Export
# -----------------------------------------------------

beta_dir = (
    DIRS['analysis'] / 'BETA_DIVERSITY'
)

beta_dir.mkdir(
    parents=True,
    exist_ok=True
)

beta_summary.to_csv(

    beta_dir / 'BETA_DIVERSITY_SUMMARY.csv',

    index=False

)

beta_membership.to_csv(

    beta_dir / 'TAXON_MEMBERSHIP.csv',

    index=False

)

for scenario, matrix in (
    beta_matrices.items()
):

    matrix.to_csv(

        beta_dir
        /
        f'ABUNDANCE_MATRIX_{scenario}.csv'

    )

print('\nALL BETA DIVERSITY CHECKS PASSED')

print('CELL 37 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 38: TAXON CONTRIBUTIONS TO BETA DIVERSITY
# =====================================================

scenario = 'Resolved_species_only'

matrix = beta_matrices[
    scenario
].copy()

secondary = matrix.loc[
    'Late_secondary'
]

mature = matrix.loc[
    'Mature'
]

# -----------------------------------------------------
# 1. Relative abundances
# -----------------------------------------------------

p_secondary = (
    secondary / secondary.sum()
)

p_mature = (
    mature / mature.sum()
)

# -----------------------------------------------------
# 2. Additive Bray-Curtis contributions
# -----------------------------------------------------

taxon_contributions = pd.DataFrame({

    'taxon':
        matrix.columns,

    'n_secondary':
        secondary.values,

    'n_mature':
        mature.values,

    'relative_secondary':
        p_secondary.values,

    'relative_mature':
        p_mature.values

})

taxon_contributions[
    'relative_difference'
] = (

    taxon_contributions[
        'relative_secondary'
    ]

    -

    taxon_contributions[
        'relative_mature'
    ]

)

taxon_contributions[
    'bray_contribution'
] = (

    taxon_contributions[
        'relative_difference'
    ].abs()

    / 2

)

total_bc = (
    taxon_contributions[
        'bray_contribution'
    ].sum()
)

taxon_contributions[
    'contribution_pct'
] = (

    taxon_contributions[
        'bray_contribution'
    ]

    / total_bc * 100

)

taxon_contributions[
    'dominance_direction'
] = np.where(

    taxon_contributions[
        'relative_difference'
    ] > 0,

    'Secondary',

    np.where(

        taxon_contributions[
            'relative_difference'
        ] < 0,

        'Mature',

        'Equal'

    )

)

taxon_contributions = (

    taxon_contributions

    .sort_values(

        'contribution_pct',

        ascending=False

    )

    .reset_index(drop=True)

)

taxon_contributions[
    'cumulative_contribution_pct'
] = (

    taxon_contributions[
        'contribution_pct'
    ].cumsum()

)

# -----------------------------------------------------
# 3. Verify mass balance
# -----------------------------------------------------

reference_bc = (

    beta_summary.loc[

        beta_summary['scenario']
        .eq(scenario),

        'bray_curtis_relative'

    ].iloc[0]

)

assert np.isclose(
    total_bc,
    reference_bc
)

assert np.isclose(

    taxon_contributions[
        'contribution_pct'
    ].sum(),

    100

)

print('RELATIVE BRAY-CURTIS:')

print(round(total_bc, 6))

print('\nTOP 20 COMPOSITIONAL CONTRIBUTORS')

display(

    taxon_contributions
    .head(20)
    .round(4)

)

# -----------------------------------------------------
# 4. Contribution concentration
# -----------------------------------------------------

concentration_results = []

for k in [1, 3, 5, 10, 20]:

    contribution = (

        taxon_contributions
        .head(k)['contribution_pct']
        .sum()

    )

    concentration_results.append({

        'top_taxa': k,

        'cumulative_contribution_pct':
            contribution

    })

beta_concentration = pd.DataFrame(
    concentration_results
)

print('\nCONCENTRATION OF DISSIMILARITY')

display(
    beta_concentration.round(4)
)

# -----------------------------------------------------
# 5. Export
# -----------------------------------------------------

taxon_contributions.to_csv(

    beta_dir
    /
    'TAXON_BRAY_CURTIS_CONTRIBUTIONS.csv',

    index=False

)

beta_concentration.to_csv(

    beta_dir
    /
    'BETA_DIVERSITY_CONCENTRATION.csv',

    index=False

)

print('\nCELL 38 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 39: BETA DIVERSITY FIGURE
# =====================================================

import matplotlib.pyplot as plt
import numpy as np

from matplotlib.patches import Circle

plt.rcParams.update({

    'font.family': 'DejaVu Sans',

    'font.size': 11,

    'axes.spines.top': False,

    'axes.spines.right': False,

    'legend.frameon': False,

    'savefig.dpi': 600

})

fig, axes = plt.subplots(

    1, 3,

    figsize=(17, 5.2),

    gridspec_kw={

        'width_ratios': [
            1, 1.1, 1.5
        ]

    }

)

# -----------------------------------------------------
# 1. Panel A: Taxonomic overlap
# -----------------------------------------------------

ax = axes[0]

row = beta_summary.loc[

    beta_summary['scenario']
    .eq('Resolved_species_only')

].iloc[0]

circle1 = Circle(

    (0.40, 0.5),

    0.29,

    facecolor='#D58E45',

    edgecolor='none',

    alpha=0.75

)

circle2 = Circle(

    (0.68, 0.5),

    0.29,

    facecolor='#27634F',

    edgecolor='none',

    alpha=0.75

)

ax.add_patch(circle1)
ax.add_patch(circle2)

ax.text(

    0.26, 0.50,

    str(int(row['secondary_unique'])),

    ha='center',

    va='center',

    fontsize=20,

    fontweight='bold'

)

ax.text(

    0.54, 0.50,

    str(int(row['shared_taxa'])),

    ha='center',

    va='center',

    fontsize=20,

    fontweight='bold'

)

ax.text(

    0.82, 0.50,

    str(int(row['mature_unique'])),

    ha='center',

    va='center',

    fontsize=20,

    fontweight='bold'

)

ax.text(

    0.27, 0.14,

    'Late-secondary',

    ha='center',

    fontsize=10

)

ax.text(

    0.81, 0.14,

    'Mature',

    ha='center',

    fontsize=10

)

ax.set_xlim(0, 1.08)
ax.set_ylim(0, 1)

ax.set_aspect('equal')

ax.axis('off')

ax.set_title(
    '(a) Species overlap'
)

# -----------------------------------------------------
# 2. Panel B: Beta partitioning
# -----------------------------------------------------

ax = axes[1]

scenario_order = [

    'All_operational_taxa',

    'Operational_without_SI',

    'Resolved_species_only',

    'Plot_specific_unresolved_taxa'

]

scenario_labels = [

    'All taxa',

    'Without SI',

    'Species only',

    'Plot-specific\nunresolved'

]

plot_beta = (

    beta_summary

    .set_index('scenario')

    .loc[scenario_order]

)

x = np.arange(
    len(plot_beta)
)

turnover = (

    plot_beta['simpson_turnover']
    .values

)

nestedness = (

    plot_beta['nestedness_component']
    .values

)

ax.bar(

    x,

    turnover,

    color='#557D9B',

    label='Turnover'

)

ax.bar(

    x,

    nestedness,

    bottom=turnover,

    color='#C7C7C7',

    label='Nestedness'

)

ax.set_xticks(

    x,

    scenario_labels,

    rotation=25,

    ha='right'

)

ax.set_ylim(0, 1.05)

ax.set_ylabel(
    'Sørensen dissimilarity'
)

ax.set_title(
    '(b) Beta diversity partitioning'
)

ax.legend(
    fontsize=9
)

ax.grid(

    axis='y',

    linestyle='--',

    alpha=0.25

)

ax.set_axisbelow(True)

# -----------------------------------------------------
# 3. Panel C: Dominant contributors
# -----------------------------------------------------

ax = axes[2]

top = (

    taxon_contributions

    .head(12)

    .sort_values(
        'contribution_pct'
    )

)

colors = [

    '#D58E45'

    if direction == 'Secondary'

    else '#27634F'

    for direction in
    top['dominance_direction']

]

ax.barh(

    top['taxon'],

    top['contribution_pct'],

    color=colors

)

ax.set_xlabel(

    'Contribution to Bray–Curtis (%)'

)

ax.set_title(

    '(c) Compositional contributors'

)

for label in ax.get_yticklabels():

    label.set_fontstyle('italic')

    label.set_fontsize(9)

ax.grid(

    axis='x',

    linestyle='--',

    alpha=0.25

)

ax.set_axisbelow(True)

from matplotlib.patches import Patch

ax.legend(

    handles=[

        Patch(

            facecolor='#D58E45',

            label='Higher in secondary'

        ),

        Patch(

            facecolor='#27634F',

            label='Higher in mature'

        )

    ],

    fontsize=9,

    loc='lower right'

)

fig.tight_layout()

# -----------------------------------------------------
# 4. Save publication-quality figure
# -----------------------------------------------------

for extension in [
    'png',
    'pdf',
    'svg'
]:

    fig.savefig(

        figure_dir
        /
        f'FIG_BETA_DIVERSITY.{extension}',

        dpi=600,

        bbox_inches='tight'

    )

plt.show()

print('CELL 39 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 40: INSTALL AND PREPARE iNEXT
# =====================================================

import subprocess
import shutil
from pathlib import Path
import pandas as pd

# -----------------------------------------------------
# 1. Check R installation
# -----------------------------------------------------

if shutil.which('Rscript') is None:

    subprocess.run(
        ['apt-get', 'update', '-qq'],
        check=True
    )

    subprocess.run(
        ['apt-get', 'install', '-y', 'r-base'],
        check=True
    )

print('Rscript available:', shutil.which('Rscript'))

# -----------------------------------------------------
# 2. Install R packages
# -----------------------------------------------------

install_script = """
packages <- c("iNEXT", "ggplot2")

for (p in packages) {

    if (!requireNamespace(p, quietly = TRUE)) {

        install.packages(
            p,
            repos = "https://cloud.r-project.org"
        )

    }

    stopifnot(
        requireNamespace(p, quietly = TRUE)
    )

}

cat("iNEXT version:",
    as.character(packageVersion("iNEXT")),
    "\\n")
"""

subprocess.run(

    ['Rscript', '-e', install_script],

    check=True

)

# -----------------------------------------------------
# 3. Prepare output directories
# -----------------------------------------------------

inext_dir = (
    DIRS['analysis'] / 'INEXT_DIVERSITY'
)

inext_dir.mkdir(
    parents=True,
    exist_ok=True
)

# -----------------------------------------------------
# 4. Export abundance vectors
# -----------------------------------------------------

inext_records = []

scenario_names = [

    'All_operational_taxa',

    'Resolved_species_only',

    'Operational_without_SI'

]

for scenario in scenario_names:

    matrix = beta_matrices[
        scenario
    ]

    for forest in matrix.index:

        abundance = matrix.loc[
            forest
        ]

        abundance = abundance[
            abundance > 0
        ]

        for taxon, count in (
            abundance.items()
        ):

            inext_records.append({

                'scenario': scenario,

                'forest_type': forest,

                'taxon': taxon,

                'abundance': int(count)

            })

inext_input = pd.DataFrame(
    inext_records
)

inext_input_path = (
    inext_dir / 'INEXT_INPUT.csv'
)

inext_input.to_csv(

    inext_input_path,

    index=False

)

print('\nINPUT VERIFICATION')

display(

    inext_input.groupby(

        ['scenario', 'forest_type']

    ).agg(

        n_individuals=(
            'abundance', 'sum'
        ),

        richness=(
            'taxon', 'nunique'
        )

    )

)

print('\nCELL 40 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 41: COVERAGE-BASED HILL DIVERSITY
# =====================================================

r_script = r'''
library(iNEXT)
library(ggplot2)

args <- commandArgs(trailingOnly = TRUE)

input_file <- args[1]
output_dir <- args[2]

dat <- read.csv(
    input_file,
    stringsAsFactors = FALSE
)

set.seed(2026)

scenarios <- unique(dat$scenario)

all_estimates <- list()
all_curves <- list()
all_coverage <- list()
all_info <- list()

for (sc in scenarios) {

    cat("\nProcessing:", sc, "\n")

    d <- dat[dat$scenario == sc, ]

    # -------------------------------------------------
    # 1. Abundance vectors
    # -------------------------------------------------

    forests <- c(
        "Late_secondary",
        "Mature"
    )

    assemblages <- lapply(

        forests,

        function(f) {

            x <- d[
                d$forest_type == f,
            ]

            v <- x$abundance

            names(v) <- x$taxon

            v

        }

    )

    names(assemblages) <- forests

    # -------------------------------------------------
    # 2. Sample information
    # -------------------------------------------------

    info <- DataInfo(

        assemblages,

        datatype = "abundance"

    )

    print(info)

    info$scenario <- sc

    all_info[[sc]] <- info

    # Common reference coverage

    target_coverage <- min(
        info$SC
    )

    # Round downward conservatively

    target_coverage <- floor(
        target_coverage * 1000
    ) / 1000

    cat(
        "Target coverage:",
        target_coverage,
        "\n"
    )

    # -------------------------------------------------
    # 3. Standardized diversity estimates
    # -------------------------------------------------

    est <- estimateD(

        assemblages,

        q = c(0, 1, 2),

        datatype = "abundance",

        base = "coverage",

        level = target_coverage,

        nboot = 200,

        conf = 0.95

    )

    est$scenario <- sc

    est$target_coverage <- (
        target_coverage
    )

    all_estimates[[sc]] <- est

    print(est)

    # -------------------------------------------------
    # 4. Interpolation and extrapolation
    # -------------------------------------------------

    endpoint <- 2 * max(
        sapply(assemblages, sum)
    )

    out <- iNEXT(

        assemblages,

        q = c(0, 1, 2),

        datatype = "abundance",

        endpoint = endpoint,

        knots = 80,

        se = TRUE,

        nboot = 200,

        conf = 0.95

    )

    coverage_curves <- (
        out$iNextEst$coverage_based
    )

    coverage_curves$scenario <- sc

    all_curves[[sc]] <- (
        coverage_curves
    )

    size_curves <- (
        out$iNextEst$size_based
    )

    size_curves$scenario <- sc

    all_coverage[[sc]] <- (
        size_curves
    )

    # Save full R result

    saveRDS(

        out,

        file.path(

            output_dir,

            paste0(
                "INEXT_",
                sc,
                ".rds"
            )

        )

    )

}

# -----------------------------------------------------
# 5. Export results
# -----------------------------------------------------

write.csv(

    do.call(rbind, all_estimates),

    file.path(
        output_dir,
        "INEXT_STANDARDIZED_DIVERSITY.csv"
    ),

    row.names = FALSE

)

write.csv(

    do.call(rbind, all_curves),

    file.path(
        output_dir,
        "INEXT_COVERAGE_CURVES.csv"
    ),

    row.names = FALSE

)

write.csv(

    do.call(rbind, all_coverage),

    file.path(
        output_dir,
        "INEXT_SIZE_CURVES.csv"
    ),

    row.names = FALSE

)

write.csv(

    do.call(rbind, all_info),

    file.path(
        output_dir,
        "INEXT_SAMPLE_INFORMATION.csv"
    ),

    row.names = FALSE

)

cat("\nINEXT ANALYSIS COMPLETED\n")
'''

r_script_path = (
    inext_dir / 'run_inext.R'
)

r_script_path.write_text(
    r_script
)

subprocess.run(

    [

        'Rscript',

        str(r_script_path),

        str(inext_input_path),

        str(inext_dir)

    ],

    check=True

)

# -----------------------------------------------------
# 6. Load results into Python
# -----------------------------------------------------

inext_standardized = pd.read_csv(

    inext_dir
    /
    'INEXT_STANDARDIZED_DIVERSITY.csv'

)

inext_curves = pd.read_csv(

    inext_dir
    /
    'INEXT_COVERAGE_CURVES.csv'

)

inext_size_curves = pd.read_csv(

    inext_dir
    /
    'INEXT_SIZE_CURVES.csv'

)

inext_sample_info = pd.read_csv(

    inext_dir
    /
    'INEXT_SAMPLE_INFORMATION.csv'

)

print('\nSTANDARDIZED HILL DIVERSITY')

display(
    inext_standardized.round(4)
)

print('\nSAMPLE INFORMATION')

display(
    inext_sample_info.round(4)
)

print('\nCELL 41 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 42: STANDARDIZED DIVERSITY CONTRAST
# =====================================================

inext_comparison = []

for scenario in scenario_names:

    d = inext_standardized.loc[

        inext_standardized['scenario']
        .eq(scenario)

    ]

    for q in [0, 1, 2]:

        s = d.loc[

            d['Assemblage']
            .eq('Late_secondary')

            &

            d['Order.q'].eq(q)

        ].iloc[0]

        m = d.loc[

            d['Assemblage']
            .eq('Mature')

            &

            d['Order.q'].eq(q)

        ].iloc[0]

        ratio = m['qD'] / s['qD']

        inext_comparison.append({

            'scenario': scenario,

            'hill_order': q,

            'target_coverage':
                s['target_coverage'],

            'secondary_qD':
                s['qD'],

            'secondary_LCL':
                s['qD.LCL'],

            'secondary_UCL':
                s['qD.UCL'],

            'mature_qD':
                m['qD'],

            'mature_LCL':
                m['qD.LCL'],

            'mature_UCL':
                m['qD.UCL'],

            'mature_secondary_ratio':
                ratio,

            'mature_excess_pct':
                (ratio - 1) * 100,

            'nonoverlapping_intervals':

                m['qD.LCL']
                >
                s['qD.UCL']

        })

inext_comparison = pd.DataFrame(
    inext_comparison
)

print(
    'COVERAGE-STANDARDIZED DIVERSITY CONTRAST'
)

display(
    inext_comparison.round(4)
)

inext_comparison.to_csv(

    inext_dir
    /
    'INEXT_DIVERSITY_COMPARISON.csv',

    index=False

)

print('\nCELL 42 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 43: COVERAGE-BASED DIVERSITY FIGURE
# =====================================================

import matplotlib.pyplot as plt
import numpy as np

scenario = 'All_operational_taxa'

plot_data = inext_curves.loc[

    inext_curves['scenario']
    .eq(scenario)

].copy()

target = (

    inext_comparison.loc[

        inext_comparison['scenario']
        .eq(scenario),

        'target_coverage'

    ].iloc[0]

)

colors = {

    'Late_secondary': '#D58E45',

    'Mature': '#27634F'

}

labels = {

    'Late_secondary':
        'Late-secondary forest',

    'Mature':
        'Mature forest'

}

panel_titles = {

    0: '(a) Taxon richness',

    1: '(b) Shannon diversity',

    2: '(c) Simpson diversity'

}

fig, axes = plt.subplots(

    1, 3,

    figsize=(16, 4.8)

)

for q, ax in zip(
    [0, 1, 2], axes
):

    for forest in [

        'Late_secondary',

        'Mature'

    ]:

        d = plot_data.loc[

            plot_data['Assemblage']
            .eq(forest)

            &

            plot_data['Order.q'].eq(q)

        ].sort_values('SC')

        # Confidence ribbon

        ax.fill_between(

            d['SC'].to_numpy(float),

            d['qD.LCL'].to_numpy(float),

            d['qD.UCL'].to_numpy(float),

            color=colors[forest],

            alpha=0.15,

            linewidth=0

        )

        # Interpolation

        rare = d.loc[

            d['Method'].isin([

                'Rarefaction',

                'Observed'

            ])

        ]

        ax.plot(

            rare['SC'],

            rare['qD'],

            color=colors[forest],

            linewidth=2.2,

            label=labels[forest]

        )

        # Extrapolation

        extra = d.loc[

            d['Method'].isin([

                'Extrapolation',

                'Observed'

            ])

        ]

        ax.plot(

            extra['SC'],

            extra['qD'],

            color=colors[forest],

            linewidth=2.2,

            linestyle='--'

        )

        # Observed point

        obs = d.loc[

            d['Method'].eq('Observed')

        ]

        ax.scatter(

            obs['SC'],

            obs['qD'],

            color=colors[forest],

            s=42,

            zorder=5

        )

        # Standardized point

        std = inext_standardized.loc[

            inext_standardized['scenario']
            .eq(scenario)

            &

            inext_standardized['Assemblage']
            .eq(forest)

            &

            inext_standardized['Order.q']
            .eq(q)

        ].iloc[0]

        ax.scatter(

            std['SC'],

            std['qD'],

            marker='D',

            color=colors[forest],

            edgecolor='white',

            s=60,

            zorder=6

        )

    # Common coverage reference

    ax.axvline(

        target,

        color='gray',

        linestyle=':',

        linewidth=1.5

    )

    ax.set_title(
        panel_titles[q]
    )

    ax.set_xlabel(
        'Sample coverage'
    )

    ax.set_ylabel(
        'Effective number of taxa'
    )

    ax.set_xlim(0.5, 1.0)

    ax.set_ylim(bottom=0)

    ax.grid(

        axis='y',

        linestyle='--',

        alpha=0.25

    )

    ax.set_axisbelow(True)

    ax.spines['top'].set_visible(False)

    ax.spines['right'].set_visible(False)

axes[0].legend(

    frameon=False,

    fontsize=9

)

fig.tight_layout()

figure_dir = (
    DIRS['outputs'] / 'FIGURES'
)

figure_dir.mkdir(

    parents=True,

    exist_ok=True

)

for extension in [

    'png',

    'pdf',

    'svg'

]:

    fig.savefig(

        figure_dir
        /
        f'FIG_INEXT_COVERAGE.{extension}',

        dpi=600,

        bbox_inches='tight'

    )

plt.show()

print('CELL 43 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 44: HEIGHT-DIAMETER DATA PREPARATION
# =====================================================

import numpy as np
import pandas as pd
from scipy.optimize import least_squares
from sklearn.model_selection import GroupKFold

allometry_dir = DIRS['analysis'] / 'ALLOMETRY'

allometry_dir.mkdir(
    parents=True,
    exist_ok=True
)

# -----------------------------------------------------
# 1. Prepare analysis dataset
# -----------------------------------------------------

allometry_data = biomass_data.copy()

required_columns = [

    'tree_id',
    'forest_type',
    'subplot_id',
    'dbh_cm',
    'height_m',
    'wood_density_g_cm3',
    'agb_kg'

]

assert all(

    c in allometry_data.columns
    for c in required_columns

), 'Missing required columns'

allometry_data = allometry_data.loc[

    allometry_data['dbh_cm'].ge(10)

    &

    allometry_data['height_m'].gt(0)

].copy()

assert len(allometry_data) == 1002

assert allometry_data[
    'tree_id'
].is_unique

# -----------------------------------------------------
# 2. Derived variables
# -----------------------------------------------------

allometry_data['log_dbh'] = np.log(

    allometry_data['dbh_cm']

)

allometry_data['log_height'] = np.log(

    allometry_data['height_m']

)

allometry_data['slenderness'] = (

    allometry_data['height_m']

    /

    (allometry_data['dbh_cm'] / 100)

)

# -----------------------------------------------------
# 3. Common diameter support
# -----------------------------------------------------

diameter_ranges = (

    allometry_data.groupby('forest_type')

    .agg(

        n_trees=('tree_id', 'size'),

        min_dbh=('dbh_cm', 'min'),

        max_dbh=('dbh_cm', 'max'),

        mean_dbh=('dbh_cm', 'mean'),

        mean_height=('height_m', 'mean'),

        median_height=('height_m', 'median'),

        mean_slenderness=('slenderness', 'mean')

    )

)

common_min = diameter_ranges[
    'min_dbh'
].max()

common_max = diameter_ranges[
    'max_dbh'
].min()

allometry_data['common_support'] = (

    allometry_data['dbh_cm']

    .between(common_min, common_max)

)

print('ALLOMETRIC DATASET')

display(
    diameter_ranges.round(4)
)

print(
    '\nCommon diameter range:',
    common_min,
    'to',
    common_max,
    'cm'
)

print('\nCOMMON SUPPORT COUNTS')

display(

    pd.crosstab(

        allometry_data['forest_type'],

        allometry_data['common_support']

    )

)

allometry_data.to_csv(

    allometry_dir /
    'SATIPO_ALLOMETRY_INPUT.csv',

    index=False

)

print('\nCELL 44 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 45: NONLINEAR HEIGHT-DIAMETER MODELS
# =====================================================

from scipy.optimize import least_squares

# -----------------------------------------------------
# 1. Candidate models
# -----------------------------------------------------

def power_model(d, a, b):

    return 1.3 + a * d**b


def weibull_model(d, a, b, c):

    return 1.3 + a * (

        1 - np.exp(-b * d**c)

    )


def michaelis_model(d, a, b):

    return 1.3 + (

        a * d / (b + d)

    )


def exponential_model(d, a, b):

    return 1.3 + a * (

        1 - np.exp(-b * d)

    )


model_specs = {

    'Power': {

        'function': power_model,

        'p0': [3, 0.6],

        'lower': [0.001, 0.01],

        'upper': [100, 2]

    },

    'Weibull': {

        'function': weibull_model,

        'p0': [50, 0.03, 1],

        'lower': [1, 0.000001, 0.05],

        'upper': [150, 10, 4]

    },

    'Michaelis_Menten': {

        'function': michaelis_model,

        'p0': [60, 30],

        'lower': [1, 0.01],

        'upper': [150, 1000]

    },

    'Exponential': {

        'function': exponential_model,

        'p0': [50, 0.03],

        'lower': [1, 0.000001],

        'upper': [150, 10]

    }

}

# -----------------------------------------------------
# 2. Model fitting function
# -----------------------------------------------------

def fit_height_model(data, model_name):

    spec = model_specs[model_name]

    d = data['dbh_cm'].to_numpy(float)

    h = data['height_m'].to_numpy(float)

    func = spec['function']

    # Fit on logarithmic scale to accommodate
    # multiplicative variation in tree height.

    def residuals(params):

        prediction = func(d, *params)

        return (

            np.log(h)

            -

            np.log(prediction)

        )

    fit = least_squares(

        residuals,

        x0=spec['p0'],

        bounds=(

            spec['lower'],

            spec['upper']

        ),

        max_nfev=20000

    )

    if not fit.success:

        raise RuntimeError(

            f'{model_name}: {fit.message}'

        )

    params = fit.x

    prediction = func(d, *params)

    residual_log = (

        np.log(h) -

        np.log(prediction)

    )

    # Duan smearing correction for arithmetic
    # mean predictions on the original scale.

    smearing = np.mean(

        np.exp(residual_log)

    )

    prediction_corrected = (

        prediction * smearing

    )

    residual_m = (

        h - prediction_corrected

    )

    n = len(h)

    # Number of curve parameters plus residual variance

    k = len(params) + 1

    rss_log = np.sum(

        residual_log**2

    )

    aic = (

        n * np.log(rss_log / n)

        + 2 * k

    )

    aicc = aic + (

        2 * k * (k + 1)

        /

        (n - k - 1)

    )

    rmse = np.sqrt(

        np.mean(residual_m**2)

    )

    mae = np.mean(

        np.abs(residual_m)

    )

    r2 = 1 - (

        np.sum(residual_m**2)

        /

        np.sum((h - h.mean())**2)

    )

    # Flag parameters close to fitting bounds

    lower = np.asarray(spec['lower'])

    upper = np.asarray(spec['upper'])

    scaled = (

        (params - lower)

        /

        (upper - lower)

    )

    boundary_flag = bool(

        np.any(scaled < 0.0001)

        or

        np.any(scaled > 0.9999)

    )

    return {

        'params': params,

        'smearing': smearing,

        'prediction': prediction_corrected,

        'aicc': aicc,

        'rmse_m': rmse,

        'mae_m': mae,

        'r2': r2,

        'boundary_flag': boundary_flag

    }


# -----------------------------------------------------
# 3. Fit each model for each forest
# -----------------------------------------------------

allometry_fits = {}

model_results = []

for forest, data in (

    allometry_data.groupby('forest_type')

):

    for model_name in model_specs:

        result = fit_height_model(

            data,

            model_name

        )

        allometry_fits[

            (forest, model_name)

        ] = result

        params = result['params']

        model_results.append({

            'forest_type': forest,

            'model': model_name,

            'n_trees': len(data),

            'a': params[0],

            'b': params[1],

            'c': (

                params[2]

                if len(params) == 3

                else np.nan

            ),

            'smearing': result['smearing'],

            'AICc': result['aicc'],

            'RMSE_m': result['rmse_m'],

            'MAE_m': result['mae_m'],

            'R2': result['r2'],

            'boundary_flag':
                result['boundary_flag']

        })

allometry_models = pd.DataFrame(

    model_results

)

allometry_models['delta_AICc'] = (

    allometry_models['AICc']

    -

    allometry_models.groupby(

        'forest_type'

    )['AICc'].transform('min')

)

allometry_models = (

    allometry_models.sort_values(

        ['forest_type', 'AICc']

    )

)

print('NONLINEAR MODEL COMPARISON')

display(

    allometry_models.round(4)

)

allometry_models.to_csv(

    allometry_dir /
    'ALLOMETRY_MODEL_COMPARISON.csv',

    index=False

)

print('\nCELL 45 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 46: SUBPLOT-GROUPED CROSS-VALIDATION
# =====================================================

from sklearn.model_selection import GroupKFold

cv_results = []

cv_predictions = []

for forest, data in (

    allometry_data.groupby('forest_type')

):

    data = data.reset_index(drop=True)

    groups = data['subplot_id']

    cv = GroupKFold(n_splits=5)

    for model_name, spec in (

        model_specs.items()

    ):

        for fold, (train_idx, test_idx) in (

            enumerate(

                cv.split(

                    data,

                    groups=groups

                ),

                start=1

            )

        ):

            train = data.iloc[train_idx]

            test = data.iloc[test_idx]

            assert not (

                set(train['subplot_id'])

                &

                set(test['subplot_id'])

            )

            fitted = fit_height_model(

                train,

                model_name

            )

            observed = test[
                'height_m'
            ].to_numpy(float)

            predicted = (

                spec['function'](

                    test['dbh_cm'].to_numpy(float),

                    *fitted['params']

                )

                *

                fitted['smearing']

            )

            errors = observed - predicted

            cv_results.append({

                'forest_type': forest,

                'model': model_name,

                'fold': fold,

                'n_train': len(train),

                'n_test': len(test),

                'RMSE_m': np.sqrt(

                    np.mean(errors**2)

                ),

                'MAE_m': np.mean(

                    np.abs(errors)

                ),

                'bias_m': np.mean(

                    predicted - observed

                )

            })

            for tree_id, obs, pred in zip(

                test['tree_id'],

                observed,

                predicted

            ):

                cv_predictions.append({

                    'forest_type': forest,

                    'model': model_name,

                    'fold': fold,

                    'tree_id': tree_id,

                    'observed_height_m': obs,

                    'predicted_height_m': pred

                })

cv_details = pd.DataFrame(cv_results)

cv_predictions = pd.DataFrame(

    cv_predictions

)

cv_predictions['squared_error'] = (

    cv_predictions['observed_height_m']

    -

    cv_predictions['predicted_height_m']

)**2

cv_predictions['absolute_error'] = (

    cv_predictions['observed_height_m']

    -

    cv_predictions['predicted_height_m']

).abs()

cv_predictions['bias'] = (

    cv_predictions['predicted_height_m']

    -

    cv_predictions['observed_height_m']

)

# -----------------------------------------------------
# Aggregate out-of-fold performance
# -----------------------------------------------------

cv_summary = (

    cv_predictions.groupby(

        ['forest_type', 'model']

    )

    .agg(

        n_predictions=('tree_id', 'size'),

        MSE=('squared_error', 'mean'),

        CV_MAE_m=('absolute_error', 'mean'),

        CV_bias_m=('bias', 'mean')

    )

    .reset_index()

)

cv_summary['CV_RMSE_m'] = np.sqrt(

    cv_summary['MSE']

)

cv_summary = cv_summary.drop(

    columns='MSE'

)

cv_summary = cv_summary.merge(

    allometry_models[

        [

            'forest_type',

            'model',

            'AICc',

            'delta_AICc',

            'boundary_flag'

        ]

    ],

    on=['forest_type', 'model'],

    validate='one_to_one'

)

cv_summary = cv_summary.sort_values(

    ['forest_type', 'CV_RMSE_m']

)

print('GROUPED CROSS-VALIDATION RESULTS')

display(

    cv_summary.round(4)

)

# -----------------------------------------------------
# Common model family for both forests
# -----------------------------------------------------

common_model_ranking = (

    cv_summary.groupby('model')

    .agg(

        mean_CV_RMSE_m=('CV_RMSE_m', 'mean'),

        mean_CV_MAE_m=('CV_MAE_m', 'mean'),

        mean_delta_AICc=('delta_AICc', 'mean'),

        any_boundary=('boundary_flag', 'any')

    )

    .sort_values('mean_CV_RMSE_m')

)

print('\nCOMMON MODEL FAMILY RANKING')

display(

    common_model_ranking.round(4)

)

cv_details.to_csv(

    allometry_dir /
    'ALLOMETRY_CV_FOLDS.csv',

    index=False

)

cv_summary.to_csv(

    allometry_dir /
    'ALLOMETRY_CV_SUMMARY.csv',

    index=False

)

cv_predictions.to_csv(

    allometry_dir /
    'ALLOMETRY_CV_PREDICTIONS.csv',

    index=False

)

print('\nCELL 46 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 47: HEIGHT CONTRAST AT COMMON DIAMETERS
# =====================================================

# Select the common model with the lowest
# average cross-validated RMSE, excluding
# full-data fits with boundary warnings.

valid_models = common_model_ranking.loc[

    ~common_model_ranking['any_boundary']

]

assert len(valid_models) > 0, (

    'All models require manual review'

)

selected_model = valid_models.index[0]

print(

    'SELECTED COMMON MODEL:',

    selected_model

)

selected_function = model_specs[

    selected_model

]['function']

# -----------------------------------------------------
# 1. Reference diameters within common support
# -----------------------------------------------------

reference_diameters = np.array(

    [10, 15, 20, 30, 40, 50, 60, 80],

    dtype=float

)

reference_diameters = reference_diameters[

    (reference_diameters >= common_min)

    &

    (reference_diameters <= common_max)

]

def predict_height(forest, diameter):

    result = allometry_fits[

        (forest, selected_model)

    ]

    return (

        selected_function(

            np.asarray(diameter, dtype=float),

            *result['params']

        )

        *

        result['smearing']

    )

height_contrast = pd.DataFrame({

    'dbh_cm': reference_diameters,

    'secondary_height_m': predict_height(

        'Late_secondary',

        reference_diameters

    ),

    'mature_height_m': predict_height(

        'Mature',

        reference_diameters

    )

})

height_contrast['height_difference_m'] = (

    height_contrast['secondary_height_m']

    -

    height_contrast['mature_height_m']

)

height_contrast['height_ratio'] = (

    height_contrast['secondary_height_m']

    /

    height_contrast['mature_height_m']

)

height_contrast['secondary_excess_pct'] = (

    height_contrast['height_ratio'] - 1

) * 100

# Biomass ratio implied by height differences
# under Chave et al. (2014), holding D and WD fixed.

height_contrast['height_implied_AGB_ratio'] = (

    height_contrast['height_ratio']**0.976

)

print('\nHEIGHT AT EQUIVALENT DIAMETERS')

display(

    height_contrast.round(4)

)

# -----------------------------------------------------
# 2. Continuous prediction curves
# -----------------------------------------------------

diameter_grid = np.linspace(

    common_min,

    common_max,

    200

)

allometry_curves = pd.DataFrame({

    'dbh_cm': diameter_grid,

    'secondary_height_m': predict_height(

        'Late_secondary',

        diameter_grid

    ),

    'mature_height_m': predict_height(

        'Mature',

        diameter_grid

    )

})

allometry_curves['height_difference_m'] = (

    allometry_curves['secondary_height_m']

    -

    allometry_curves['mature_height_m']

)

allometry_curves['height_ratio'] = (

    allometry_curves['secondary_height_m']

    /

    allometry_curves['mature_height_m']

)

height_contrast.to_csv(

    allometry_dir /
    'HEIGHT_CONTRAST_REFERENCE_DIAMETERS.csv',

    index=False

)

allometry_curves.to_csv(

    allometry_dir /
    'HEIGHT_DIAMETER_PREDICTION_CURVES.csv',

    index=False

)

print('\nCELL 47 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 48: HEIGHT-DIAMETER ALLOMETRY FIGURE
# =====================================================

import matplotlib.pyplot as plt

colors = {

    'Late_secondary': '#D58E45',

    'Mature': '#27634F'

}

labels = {

    'Late_secondary': 'Late-secondary forest',

    'Mature': 'Mature forest'

}

fig, axes = plt.subplots(

    1, 3,

    figsize=(16, 4.8)

)

# -----------------------------------------------------
# PANEL A: Height-diameter relationships
# -----------------------------------------------------

ax = axes[0]

for forest, data in (

    allometry_data.groupby('forest_type')

):

    ax.scatter(

        data['dbh_cm'],

        data['height_m'],

        color=colors[forest],

        alpha=0.20,

        s=12,

        edgecolors='none'

    )

    # Curves restricted to observed common support

    ax.plot(

        diameter_grid,

        predict_height(

            forest,

            diameter_grid

        ),

        color=colors[forest],

        linewidth=2.7,

        label=labels[forest]

    )

ax.set_title(

    '(a) Height-diameter allometry'

)

ax.set_xlabel('Diameter (cm)')

ax.set_ylabel('Tree height (m)')

ax.legend(

    frameon=False,

    fontsize=9

)

# -----------------------------------------------------
# PANEL B: Height contrast
# -----------------------------------------------------

ax = axes[1]

ax.plot(

    allometry_curves['dbh_cm'],

    allometry_curves['height_difference_m'],

    color='#557D9B',

    linewidth=2.5

)

ax.axhline(

    0,

    color='gray',

    linestyle='--',

    linewidth=1.2

)

ax.scatter(

    height_contrast['dbh_cm'],

    height_contrast['height_difference_m'],

    color='#557D9B',

    s=30

)

ax.set_title(

    '(b) Height contrast at equal diameter'

)

ax.set_xlabel('Diameter (cm)')

ax.set_ylabel(

    'Height difference (m)\nSecondary - mature'

)

# -----------------------------------------------------
# PANEL C: Height-implied biomass contrast
# -----------------------------------------------------

ax = axes[2]

agb_ratio = (

    allometry_curves['height_ratio']**0.976

)

ax.plot(

    allometry_curves['dbh_cm'],

    agb_ratio,

    color='#795D91',

    linewidth=2.5

)

ax.axhline(

    1,

    color='gray',

    linestyle='--',

    linewidth=1.2

)

ax.scatter(

    height_contrast['dbh_cm'],

    height_contrast['height_implied_AGB_ratio'],

    color='#795D91',

    s=30

)

ax.set_title(

    '(c) Height-implied biomass ratio'

)

ax.set_xlabel('Diameter (cm)')

ax.set_ylabel(

    'AGB ratio\nSecondary / mature'

)

# -----------------------------------------------------
# Formatting
# -----------------------------------------------------

for ax in axes:

    ax.grid(

        axis='y',

        linestyle='--',

        alpha=0.25

    )

    ax.set_axisbelow(True)

    ax.spines['top'].set_visible(False)

    ax.spines['right'].set_visible(False)

fig.tight_layout()

figure_dir = (

    DIRS['outputs'] / 'FIGURES'

)

figure_dir.mkdir(

    parents=True,

    exist_ok=True

)

for extension in ['png', 'pdf', 'svg']:

    fig.savefig(

        figure_dir /

        f'FIG_HEIGHT_DIAMETER_ALLOMETRY.{extension}',

        dpi=600,

        bbox_inches='tight'

    )

plt.show()

print('\nCELL 48 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 49: SUBPLOT BOOTSTRAP OF ALLOMETRIC CURVES
# =====================================================

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

N_BOOT = 1000
RANDOM_SEED = 2026

rng = np.random.default_rng(RANDOM_SEED)

forest_names = [
    'Late_secondary',
    'Mature'
]

bootstrap_grid = np.linspace(
    common_min,
    common_max,
    200
)

# Store original subplot data

subplot_groups = {}

for forest in forest_names:

    forest_data = allometry_data.loc[
        allometry_data['forest_type'].eq(forest)
    ]

    subplot_groups[forest] = {

        subplot: group.copy()

        for subplot, group in
        forest_data.groupby('subplot_id')

    }

    assert len(subplot_groups[forest]) == 25


# -----------------------------------------------------
# 1. Bootstrap each forest independently
# -----------------------------------------------------

boot_curves = {
    forest: []
    for forest in forest_names
}

boot_parameters = []

failed_iterations = []

for iteration in tqdm(
    range(N_BOOT),
    desc='Subplot bootstrap'
):

    iteration_predictions = {}

    iteration_parameters = []

    try:

        for forest in forest_names:

            groups = subplot_groups[forest]

            subplot_ids = list(groups.keys())

            sampled_ids = rng.choice(

                subplot_ids,

                size=len(subplot_ids),

                replace=True

            )

            sampled_data = pd.concat(

                [groups[s] for s in sampled_ids],

                ignore_index=True

            )

            fitted = fit_height_model(

                sampled_data,

                selected_model

            )

            predicted = (

                selected_function(

                    bootstrap_grid,

                    *fitted['params']

                )

                * fitted['smearing']

            )

            if not np.all(np.isfinite(predicted)):

                raise ValueError(
                    'Nonfinite prediction'
                )

            iteration_predictions[forest] = predicted

            iteration_parameters.append({

                'iteration': iteration,

                'forest_type': forest,

                'a': fitted['params'][0],

                'b': fitted['params'][1],

                'c': (
                    fitted['params'][2]
                    if len(fitted['params']) == 3
                    else np.nan
                ),

                'smearing': fitted['smearing'],

                'boundary_flag':
                    fitted['boundary_flag']

            })

        # Store only successful paired iterations

        for forest in forest_names:

            boot_curves[forest].append(

                iteration_predictions[forest]

            )

        boot_parameters.extend(
            iteration_parameters
        )

    except Exception as error:

        failed_iterations.append({

            'iteration': iteration,

            'error': str(error)

        })


# -----------------------------------------------------
# 2. Bootstrap uncertainty
# -----------------------------------------------------

boot_secondary = np.asarray(

    boot_curves['Late_secondary']

)

boot_mature = np.asarray(

    boot_curves['Mature']

)

n_success = len(boot_secondary)

assert n_success >= N_BOOT * 0.90, (
    'Excessive bootstrap failures'
)

assert len(boot_mature) == n_success

boot_difference = (

    boot_secondary - boot_mature

)

boot_ratio = (

    boot_secondary / boot_mature

)

boot_agb_ratio = boot_ratio**0.976


def percentile_interval(values):

    return np.percentile(

        values,

        [2.5, 97.5],

        axis=0

    )


secondary_ci = percentile_interval(
    boot_secondary
)

mature_ci = percentile_interval(
    boot_mature
)

difference_ci = percentile_interval(
    boot_difference
)

agb_ratio_ci = percentile_interval(
    boot_agb_ratio
)


# -----------------------------------------------------
# 3. Assemble bootstrap prediction table
# -----------------------------------------------------

allometry_bootstrap = pd.DataFrame({

    'dbh_cm': bootstrap_grid,

    'secondary_height_m': predict_height(
        'Late_secondary',
        bootstrap_grid
    ),

    'secondary_LCL': secondary_ci[0],

    'secondary_UCL': secondary_ci[1],

    'mature_height_m': predict_height(
        'Mature',
        bootstrap_grid
    ),

    'mature_LCL': mature_ci[0],

    'mature_UCL': mature_ci[1],

    'difference_LCL': difference_ci[0],

    'difference_UCL': difference_ci[1],

    'AGB_ratio_LCL': agb_ratio_ci[0],

    'AGB_ratio_UCL': agb_ratio_ci[1]

})

allometry_bootstrap['height_difference_m'] = (

    allometry_bootstrap['secondary_height_m']

    -

    allometry_bootstrap['mature_height_m']

)

allometry_bootstrap['height_implied_AGB_ratio'] = (

    allometry_bootstrap['secondary_height_m']

    /

    allometry_bootstrap['mature_height_m']

)**0.976

allometry_bootstrap['positive_interval'] = (

    allometry_bootstrap['difference_LCL'] > 0

)

bootstrap_parameters = pd.DataFrame(
    boot_parameters
)

print('BOOTSTRAP DIAGNOSTICS')

print('Successful iterations:', n_success)

print('Failed iterations:', len(failed_iterations))

print('\nBOUNDARY WARNINGS')

display(

    bootstrap_parameters.groupby(
        'forest_type'
    )['boundary_flag'].agg(
        ['sum', 'mean']
    )

)

# -----------------------------------------------------
# 4. Reference diameter uncertainty
# -----------------------------------------------------

reference_bootstrap = pd.DataFrame({

    'dbh_cm': reference_diameters

})

for column in [

    'secondary_height_m',
    'secondary_LCL',
    'secondary_UCL',
    'mature_height_m',
    'mature_LCL',
    'mature_UCL',
    'height_difference_m',
    'difference_LCL',
    'difference_UCL',
    'height_implied_AGB_ratio',
    'AGB_ratio_LCL',
    'AGB_ratio_UCL'

]:

    reference_bootstrap[column] = np.interp(

        reference_diameters,

        bootstrap_grid,

        allometry_bootstrap[column]

    )

print('\nBOOTSTRAP HEIGHT CONTRAST')

display(

    reference_bootstrap.round(4)

)

# -----------------------------------------------------
# 5. Save outputs
# -----------------------------------------------------

allometry_bootstrap.to_csv(

    allometry_dir /
    'ALLOMETRY_BOOTSTRAP_CURVES.csv',

    index=False

)

reference_bootstrap.to_csv(

    allometry_dir /
    'ALLOMETRY_BOOTSTRAP_REFERENCE.csv',

    index=False

)

bootstrap_parameters.to_csv(

    allometry_dir /
    'ALLOMETRY_BOOTSTRAP_PARAMETERS.csv',

    index=False

)

pd.DataFrame(failed_iterations).to_csv(

    allometry_dir /
    'ALLOMETRY_BOOTSTRAP_FAILURES.csv',

    index=False

)

np.savez_compressed(

    allometry_dir /
    'ALLOMETRY_BOOTSTRAP_DRAWS.npz',

    diameter=bootstrap_grid,

    secondary=boot_secondary,

    mature=boot_mature

)

print('\nCELL 49 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 50: ALLOMETRIC MODEL SENSITIVITY
# =====================================================

model_sensitivity = []

for model_name, spec in model_specs.items():

    function = spec['function']

    predictions = {}

    for forest in forest_names:

        fitted = allometry_fits[
            (forest, model_name)
        ]

        predictions[forest] = (

            function(

                reference_diameters,

                *fitted['params']

            )

            * fitted['smearing']

        )

    for i, diameter in enumerate(
        reference_diameters
    ):

        hs = predictions[
            'Late_secondary'
        ][i]

        hm = predictions[
            'Mature'
        ][i]

        model_sensitivity.append({

            'model': model_name,

            'dbh_cm': diameter,

            'secondary_height_m': hs,

            'mature_height_m': hm,

            'height_difference_m': hs - hm,

            'height_ratio': hs / hm,

            'height_implied_AGB_ratio':
                (hs / hm)**0.976

        })

allometry_sensitivity = pd.DataFrame(
    model_sensitivity
)

sensitivity_summary = (

    allometry_sensitivity.groupby('dbh_cm')

    .agg(

        min_height_difference_m=(
            'height_difference_m', 'min'
        ),

        max_height_difference_m=(
            'height_difference_m', 'max'
        ),

        min_AGB_ratio=(
            'height_implied_AGB_ratio', 'min'
        ),

        max_AGB_ratio=(
            'height_implied_AGB_ratio', 'max'
        )

    )

    .reset_index()

)

sensitivity_summary['all_models_positive'] = (

    sensitivity_summary[
        'min_height_difference_m'
    ] > 0

)

print('MODEL SENSITIVITY: HEIGHT DIFFERENCES')

display(

    allometry_sensitivity.pivot(

        index='dbh_cm',

        columns='model',

        values='height_difference_m'

    ).round(4)

)

print('\nROBUSTNESS ACROSS MODEL FAMILIES')

display(
    sensitivity_summary.round(4)
)

# -----------------------------------------------------
# Save
# -----------------------------------------------------

allometry_sensitivity.to_csv(

    allometry_dir /
    'ALLOMETRY_MODEL_SENSITIVITY.csv',

    index=False

)

sensitivity_summary.to_csv(

    allometry_dir /
    'ALLOMETRY_SENSITIVITY_SUMMARY.csv',

    index=False

)

print('\nCELL 50 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 51: ALLOMETRIC FIGURE WITH UNCERTAINTY
# =====================================================

import matplotlib.pyplot as plt

fig, axes = plt.subplots(

    1, 3,

    figsize=(16, 4.8)

)

b = allometry_bootstrap

x = b['dbh_cm'].to_numpy()

# -----------------------------------------------------
# PANEL A: Fitted height-diameter relationships
# -----------------------------------------------------

ax = axes[0]

for forest, prefix in [

    ('Late_secondary', 'secondary'),

    ('Mature', 'mature')

]:

    data = allometry_data.loc[

        allometry_data['forest_type'].eq(forest)

    ]

    ax.scatter(

        data['dbh_cm'],

        data['height_m'],

        color=colors[forest],

        alpha=0.18,

        s=13,

        edgecolors='none'

    )

    ax.fill_between(

        x,

        b[f'{prefix}_LCL'].to_numpy(),

        b[f'{prefix}_UCL'].to_numpy(),

        color=colors[forest],

        alpha=0.17,

        linewidth=0

    )

    ax.plot(

        x,

        b[f'{prefix}_height_m'],

        color=colors[forest],

        linewidth=2.6,

        label=labels[forest]

    )

ax.set_title(
    '(a) Height-diameter allometry'
)

ax.set_xlabel('Diameter (cm)')

ax.set_ylabel('Tree height (m)')

ax.legend(
    frameon=False,
    fontsize=9
)

# -----------------------------------------------------
# PANEL B: Height differences
# -----------------------------------------------------

ax = axes[1]

ax.fill_between(

    x,

    b['difference_LCL'].to_numpy(),

    b['difference_UCL'].to_numpy(),

    color='#557D9B',

    alpha=0.20,

    linewidth=0

)

ax.plot(

    x,

    b['height_difference_m'],

    color='#557D9B',

    linewidth=2.6

)

ax.axhline(

    0,

    color='gray',

    linestyle='--',

    linewidth=1.2

)

ax.scatter(

    reference_bootstrap['dbh_cm'],

    reference_bootstrap[
        'height_difference_m'
    ],

    color='#557D9B',

    s=27,

    zorder=3

)

ax.set_title(
    '(b) Height contrast at equal diameter'
)

ax.set_xlabel('Diameter (cm)')

ax.set_ylabel(
    'Height difference (m)\nSecondary - mature'
)

# -----------------------------------------------------
# PANEL C: Height-implied AGB ratios
# -----------------------------------------------------

ax = axes[2]

ax.fill_between(

    x,

    b['AGB_ratio_LCL'].to_numpy(),

    b['AGB_ratio_UCL'].to_numpy(),

    color='#795D91',

    alpha=0.20,

    linewidth=0

)

ax.plot(

    x,

    b['height_implied_AGB_ratio'],

    color='#795D91',

    linewidth=2.6

)

ax.axhline(

    1,

    color='gray',

    linestyle='--',

    linewidth=1.2

)

ax.scatter(

    reference_bootstrap['dbh_cm'],

    reference_bootstrap[
        'height_implied_AGB_ratio'
    ],

    color='#795D91',

    s=27,

    zorder=3

)

ax.set_title(
    '(c) Height-implied biomass ratio'
)

ax.set_xlabel('Diameter (cm)')

ax.set_ylabel(
    'AGB ratio\nSecondary / mature'
)

# -----------------------------------------------------
# Mark sparse upper diameter range
# -----------------------------------------------------

large_tree_threshold = 60

for ax in axes:

    ax.axvspan(

        large_tree_threshold,

        common_max,

        color='gray',

        alpha=0.07,

        zorder=0

    )

    ax.grid(

        axis='y',

        linestyle='--',

        alpha=0.25

    )

    ax.set_axisbelow(True)

    ax.spines['top'].set_visible(False)

    ax.spines['right'].set_visible(False)

fig.tight_layout()

for extension in ['png', 'pdf', 'svg']:

    fig.savefig(

        figure_dir /

        f'FIG_ALLOMETRY_BOOTSTRAP.{extension}',

        dpi=600,

        bbox_inches='tight'

    )

plt.show()

# -----------------------------------------------------
# Diameter support diagnostics
# -----------------------------------------------------

print('LARGE TREE SUPPORT')

display(

    allometry_data.assign(

        large_tree=allometry_data[
            'dbh_cm'
        ].ge(large_tree_threshold)

    ).groupby('forest_type').agg(

        total_trees=('tree_id', 'size'),

        trees_ge_60cm=('large_tree', 'sum')

    )

)

print('\nCELL 51 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 52: COUNTERFACTUAL HEIGHT ARCHITECTURE
# =====================================================

counterfactual = allometry_data.loc[

    allometry_data['common_support']

].copy()

assert len(counterfactual) == 997

# -----------------------------------------------------
# 1. Predict heights under each architecture
# -----------------------------------------------------

counterfactual['H_secondary_model'] = (

    predict_height(

        'Late_secondary',

        counterfactual['dbh_cm'].to_numpy()

    )

)

counterfactual['H_mature_model'] = (

    predict_height(

        'Mature',

        counterfactual['dbh_cm'].to_numpy()

    )

)

is_secondary = counterfactual[
    'forest_type'
].eq('Late_secondary')

counterfactual['H_own_model'] = np.where(

    is_secondary,

    counterfactual['H_secondary_model'],

    counterfactual['H_mature_model']

)

counterfactual['H_other_model'] = np.where(

    is_secondary,

    counterfactual['H_mature_model'],

    counterfactual['H_secondary_model']

)

# -----------------------------------------------------
# 2. Preserve individual multiplicative residuals
# -----------------------------------------------------

counterfactual['height_residual_factor'] = (

    counterfactual['height_m']

    /

    counterfactual['H_own_model']

)

counterfactual['H_counterfactual'] = (

    counterfactual['H_other_model']

    *

    counterfactual['height_residual_factor']

)

assert counterfactual[
    'H_counterfactual'
].gt(0).all()

# -----------------------------------------------------
# 3. Biomass estimation
# -----------------------------------------------------

def chave_agb(dbh, height, wd):

    return 0.0673 * (

        wd * dbh**2 * height

    )**0.976


counterfactual['AGB_observed_kg'] = (

    chave_agb(

        counterfactual['dbh_cm'],

        counterfactual['height_m'],

        counterfactual['wood_density_g_cm3']

    )

)

# Verify compatibility with previous biomass estimates

assert np.allclose(

    counterfactual['AGB_observed_kg'],

    counterfactual['agb_kg'],

    rtol=1e-5,

    atol=1e-5

), 'Check biomass equation consistency'

counterfactual['AGB_counterfactual_kg'] = (

    chave_agb(

        counterfactual['dbh_cm'],

        counterfactual['H_counterfactual'],

        counterfactual['wood_density_g_cm3']

    )

)

counterfactual['AGB_change_kg'] = (

    counterfactual['AGB_counterfactual_kg']

    -

    counterfactual['AGB_observed_kg']

)

# -----------------------------------------------------
# 4. Aggregate by forest
# -----------------------------------------------------

counterfactual_summary = (

    counterfactual.groupby('forest_type')

    .agg(

        n_trees=('tree_id', 'size'),

        observed_AGB_kg=(
            'AGB_observed_kg', 'sum'
        ),

        counterfactual_AGB_kg=(
            'AGB_counterfactual_kg', 'sum'
        ),

        mean_observed_height_m=(
            'height_m', 'mean'
        ),

        mean_counterfactual_height_m=(
            'H_counterfactual', 'mean'
        )

    )

    .reset_index()

)

counterfactual_summary['observed_AGB_Mg'] = (

    counterfactual_summary[
        'observed_AGB_kg'
    ] / 1000

)

counterfactual_summary[
    'counterfactual_AGB_Mg'
] = (

    counterfactual_summary[
        'counterfactual_AGB_kg'
    ] / 1000

)

counterfactual_summary['AGB_change_Mg'] = (

    counterfactual_summary[
        'counterfactual_AGB_Mg'
    ]

    -

    counterfactual_summary[
        'observed_AGB_Mg'
    ]

)

counterfactual_summary['AGB_change_pct'] = (

    counterfactual_summary[
        'AGB_change_Mg'
    ]

    /

    counterfactual_summary[
        'observed_AGB_Mg'
    ]

) * 100

print('HEIGHT ARCHITECTURE COUNTERFACTUAL')

display(

    counterfactual_summary.drop(

        columns=[

            'observed_AGB_kg',

            'counterfactual_AGB_kg'

        ]

    ).round(4)

)

# -----------------------------------------------------
# 5. Symmetric architecture contribution
# -----------------------------------------------------

cf = counterfactual_summary.set_index(
    'forest_type'
)

Bs = cf.loc[
    'Late_secondary',
    'observed_AGB_Mg'
]

Bm = cf.loc[
    'Mature',
    'observed_AGB_Mg'
]

Bs_mature_architecture = cf.loc[

    'Late_secondary',

    'counterfactual_AGB_Mg'

]

Bm_secondary_architecture = cf.loc[

    'Mature',

    'counterfactual_AGB_Mg'

]

# Architecture contrast evaluated using
# each forest's own diameter and WD distribution

architecture_effect_secondary = (

    Bs - Bs_mature_architecture

)

architecture_effect_mature = (

    Bm_secondary_architecture - Bm

)

architecture_effect_symmetric = (

    architecture_effect_secondary

    +

    architecture_effect_mature

) / 2

observed_gap_common = Bs - Bm

remaining_gap = (

    observed_gap_common

    -

    architecture_effect_symmetric

)

architecture_decomposition = pd.DataFrame({

    'component': [

        'Observed AGB difference',

        'Height architecture contribution',

        'Remaining joint contribution'

    ],

    'contribution_Mg': [

        observed_gap_common,

        architecture_effect_symmetric,

        remaining_gap

    ]

})

architecture_decomposition[
    'contribution_pct'
] = (

    architecture_decomposition[
        'contribution_Mg'
    ]

    /

    observed_gap_common

) * 100

print('\nSYMMETRIC HEIGHT ARCHITECTURE DECOMPOSITION')

display(

    architecture_decomposition.round(4)

)

assert np.isclose(

    architecture_effect_symmetric

    +

    remaining_gap,

    observed_gap_common

)

# -----------------------------------------------------
# 6. Save
# -----------------------------------------------------

counterfactual.to_csv(

    allometry_dir /
    'HEIGHT_COUNTERFACTUAL_TREE_LEVEL.csv',

    index=False

)

counterfactual_summary.to_csv(

    allometry_dir /
    'HEIGHT_COUNTERFACTUAL_SUMMARY.csv',

    index=False

)

architecture_decomposition.to_csv(

    allometry_dir /
    'HEIGHT_ARCHITECTURE_DECOMPOSITION.csv',

    index=False

)

print('\nCELL 52 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 53: COUNTERFACTUAL MODEL SENSITIVITY
# =====================================================

cf_data = counterfactual.copy()

D = cf_data['dbh_cm'].to_numpy()
H = cf_data['height_m'].to_numpy()
WD = cf_data['wood_density_g_cm3'].to_numpy()

mask_s = cf_data['forest_type'].eq(
    'Late_secondary'
).to_numpy()

mask_m = ~mask_s

observed_agb = chave_agb(D, H, WD) / 1000

Bs = observed_agb[mask_s].sum()
Bm = observed_agb[mask_m].sum()

common_gap = Bs - Bm

model_cf_results = []

for model_name, spec in model_specs.items():

    function = spec['function']

    fit_s = allometry_fits[
        ('Late_secondary', model_name)
    ]

    fit_m = allometry_fits[
        ('Mature', model_name)
    ]

    Hs = (
        function(D, *fit_s['params'])
        * fit_s['smearing']
    )

    Hm = (
        function(D, *fit_m['params'])
        * fit_m['smearing']
    )

    assert np.all(Hs > 0)
    assert np.all(Hm > 0)

    # Exchange architecture while retaining
    # observed individual height residuals

    H_cf = np.where(
        mask_s,
        H * Hm / Hs,
        H * Hs / Hm
    )

    AGB_cf = chave_agb(
        D, H_cf, WD
    ) / 1000

    Bs_cf = AGB_cf[mask_s].sum()
    Bm_cf = AGB_cf[mask_m].sum()

    effect_s = Bs - Bs_cf
    effect_m = Bm_cf - Bm

    architecture = (
        effect_s + effect_m
    ) / 2

    remainder = common_gap - architecture

    model_cf_results.append({

        'model': model_name,

        'secondary_observed_Mg': Bs,

        'mature_observed_Mg': Bm,

        'secondary_counterfactual_Mg': Bs_cf,

        'mature_counterfactual_Mg': Bm_cf,

        'secondary_change_pct':
            100 * (Bs_cf / Bs - 1),

        'mature_change_pct':
            100 * (Bm_cf / Bm - 1),

        'architecture_contribution_Mg':
            architecture,

        'architecture_contribution_pct':
            100 * architecture / common_gap,

        'remaining_contribution_Mg':
            remainder,

        'remaining_contribution_pct':
            100 * remainder / common_gap

    })

counterfactual_model_sensitivity = pd.DataFrame(
    model_cf_results
)

print('COUNTERFACTUAL MODEL SENSITIVITY')

display(
    counterfactual_model_sensitivity.round(4)
)

print('\nARCHITECTURE CONTRIBUTION RANGE')

print(
    'Minimum:',
    round(
        counterfactual_model_sensitivity[
            'architecture_contribution_pct'
        ].min(), 2
    ),
    '%'
)

print(
    'Maximum:',
    round(
        counterfactual_model_sensitivity[
            'architecture_contribution_pct'
        ].max(), 2
    ),
    '%'
)

assert (
    counterfactual_model_sensitivity[
        'architecture_contribution_Mg'
    ] > 0
).all()

counterfactual_model_sensitivity.to_csv(

    allometry_dir /
    'COUNTERFACTUAL_MODEL_SENSITIVITY.csv',

    index=False

)

print('\nCELL 53 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 54: BOOTSTRAP ARCHITECTURE CONTRIBUTION
# =====================================================

bootstrap_cf_results = []

assert len(boot_secondary) == len(boot_mature)

for iteration in tqdm(

    range(len(boot_secondary)),

    desc='Counterfactual uncertainty'

):

    Hs = np.interp(

        D,

        bootstrap_grid,

        boot_secondary[iteration]

    )

    Hm = np.interp(

        D,

        bootstrap_grid,

        boot_mature[iteration]

    )

    if np.any(Hs <= 0) or np.any(Hm <= 0):

        continue

    H_cf = np.where(

        mask_s,

        H * Hm / Hs,

        H * Hs / Hm

    )

    AGB_cf = chave_agb(

        D,

        H_cf,

        WD

    ) / 1000

    Bs_cf = AGB_cf[mask_s].sum()

    Bm_cf = AGB_cf[mask_m].sum()

    effect_s = Bs - Bs_cf

    effect_m = Bm_cf - Bm

    architecture = (

        effect_s + effect_m

    ) / 2

    remainder = common_gap - architecture

    bootstrap_cf_results.append({

        'bootstrap_draw': iteration,

        'secondary_counterfactual_Mg': Bs_cf,

        'mature_counterfactual_Mg': Bm_cf,

        'secondary_change_pct':
            100 * (Bs_cf / Bs - 1),

        'mature_change_pct':
            100 * (Bm_cf / Bm - 1),

        'architecture_contribution_Mg':
            architecture,

        'architecture_contribution_pct':
            100 * architecture / common_gap,

        'remaining_contribution_Mg':
            remainder,

        'remaining_contribution_pct':
            100 * remainder / common_gap

    })

bootstrap_counterfactual = pd.DataFrame(
    bootstrap_cf_results
)

assert len(bootstrap_counterfactual) >= 900

# -----------------------------------------------------
# 1. Summarize uncertainty
# -----------------------------------------------------

metrics = [

    'secondary_counterfactual_Mg',

    'mature_counterfactual_Mg',

    'secondary_change_pct',

    'mature_change_pct',

    'architecture_contribution_Mg',

    'architecture_contribution_pct',

    'remaining_contribution_Mg',

    'remaining_contribution_pct'

]

baseline = (

    counterfactual_model_sensitivity

    .set_index('model')

    .loc[selected_model]

)

uncertainty_rows = []

for metric in metrics:

    values = bootstrap_counterfactual[
        metric
    ].to_numpy()

    uncertainty_rows.append({

        'metric': metric,

        'point_estimate': baseline[metric],

        'bootstrap_mean': values.mean(),

        'bootstrap_median': np.median(values),

        'LCL_95': np.percentile(values, 2.5),

        'UCL_95': np.percentile(values, 97.5),

        'bootstrap_sd': values.std(ddof=1)

    })

architecture_uncertainty = pd.DataFrame(
    uncertainty_rows
)

print('ARCHITECTURE CONTRIBUTION UNCERTAINTY')

display(
    architecture_uncertainty.round(4)
)

print('\nROBUSTNESS OF CONTRIBUTION')

positive_draws = (

    bootstrap_counterfactual[
        'architecture_contribution_Mg'
    ] > 0

).mean() * 100

print(
    'Positive architecture contribution:',
    round(positive_draws, 2),
    '% of bootstrap draws'
)

# -----------------------------------------------------
# 2. Sensitivity to boundary warnings
# -----------------------------------------------------

boundary_by_iteration = (

    bootstrap_parameters.groupby('iteration')

    ['boundary_flag'].any()

    .sort_index()

)

assert len(boundary_by_iteration) == len(
    boot_secondary
)

boundary_free = (

    ~boundary_by_iteration.to_numpy()

)

bootstrap_counterfactual['boundary_free'] = (

    boundary_free[

        bootstrap_counterfactual[
            'bootstrap_draw'
        ].to_numpy()

    ]

)

boundary_sensitivity = []

for scenario, subset in [

    (
        'All_successful_draws',
        bootstrap_counterfactual
    ),

    (
        'Boundary_free_draws',
        bootstrap_counterfactual.loc[
            bootstrap_counterfactual[
                'boundary_free'
            ]
        ]
    )

]:

    values = subset[
        'architecture_contribution_pct'
    ]

    if len(values) == 0:

        continue

    boundary_sensitivity.append({

        'scenario': scenario,

        'n_draws': len(values),

        'mean_contribution_pct':
            values.mean(),

        'median_contribution_pct':
            values.median(),

        'LCL_95':
            values.quantile(0.025),

        'UCL_95':
            values.quantile(0.975)

    })

boundary_sensitivity = pd.DataFrame(
    boundary_sensitivity
)

print('\nBOUNDARY SENSITIVITY')

display(
    boundary_sensitivity.round(4)
)

# -----------------------------------------------------
# 3. Save outputs
# -----------------------------------------------------

bootstrap_counterfactual.to_csv(

    allometry_dir /
    'COUNTERFACTUAL_BOOTSTRAP_DRAWS.csv',

    index=False

)

architecture_uncertainty.to_csv(

    allometry_dir /
    'ARCHITECTURE_CONTRIBUTION_UNCERTAINTY.csv',

    index=False

)

boundary_sensitivity.to_csv(

    allometry_dir /
    'ARCHITECTURE_BOUNDARY_SENSITIVITY.csv',

    index=False

)

print('\nCELL 54 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 55: FULL INVENTORY BIOMASS GAP INTEGRATION
# =====================================================

full_inventory = biomass_data.copy()

assert len(full_inventory) == 1002

full_inventory['in_common_support'] = (

    full_inventory['dbh_cm'].between(

        common_min,

        common_max,

        inclusive='both'

    )

)

# -----------------------------------------------------
# 1. Biomass inside and outside common support
# -----------------------------------------------------

support_summary = (

    full_inventory.groupby([

        'forest_type',

        'in_common_support'

    ])

    .agg(

        n_trees=('tree_id', 'size'),

        biomass_kg=('agb_kg', 'sum'),

        basal_area_m2=('basal_area_m2', 'sum')

    )

    .reset_index()

)

support_summary['biomass_Mg'] = (

    support_summary['biomass_kg'] / 1000

)

print('BIOMASS BY DIAMETER SUPPORT')

display(

    support_summary.drop(
        columns='biomass_kg'
    ).round(4)

)

# -----------------------------------------------------
# 2. Full inventory biomass
# -----------------------------------------------------

full_totals = (

    full_inventory.groupby('forest_type')

    ['agb_kg'].sum() / 1000

)

full_secondary = full_totals[
    'Late_secondary'
]

full_mature = full_totals[
    'Mature'
]

full_gap = full_secondary - full_mature

outside = full_inventory.loc[

    ~full_inventory['in_common_support']

]

outside_totals = (

    outside.groupby('forest_type')

    ['agb_kg'].sum() / 1000

)

outside_secondary = outside_totals.get(

    'Late_secondary', 0.0

)

outside_mature = outside_totals.get(

    'Mature', 0.0

)

outside_contribution = (

    outside_secondary - outside_mature

)

assert np.isclose(

    full_gap,

    common_gap + outside_contribution

)

# -----------------------------------------------------
# 3. Integrate architecture contribution
# -----------------------------------------------------

architecture_point = baseline[
    'architecture_contribution_Mg'
]

remaining_point = (

    common_gap - architecture_point

)

full_gap_decomposition = pd.DataFrame({

    'component': [

        'Height architecture',

        'Remaining common-support contrast',

        'Trees outside common support'

    ],

    'contribution_Mg': [

        architecture_point,

        remaining_point,

        outside_contribution

    ]

})

full_gap_decomposition[
    'contribution_pct_of_full_gap'
] = (

    100 *

    full_gap_decomposition[
        'contribution_Mg'
    ]

    / full_gap

)

assert np.isclose(

    full_gap_decomposition[
        'contribution_Mg'
    ].sum(),

    full_gap

)

print('\nFULL INVENTORY DECOMPOSITION')

display(
    full_gap_decomposition.round(4)
)

print(
    '\nFull observed difference:',
    round(full_gap, 4),
    'Mg'
)

print(
    'Reconstructed difference:',
    round(
        full_gap_decomposition[
            'contribution_Mg'
        ].sum(), 4
    ),
    'Mg'
)

# -----------------------------------------------------
# 4. Propagate architecture uncertainty
# -----------------------------------------------------

full_gap_bootstrap = pd.DataFrame({

    'architecture_Mg':

        bootstrap_counterfactual[
            'architecture_contribution_Mg'
        ],

    'remaining_common_Mg':

        common_gap -

        bootstrap_counterfactual[
            'architecture_contribution_Mg'
        ],

    'outside_support_Mg':

        outside_contribution

})

full_gap_bootstrap['reconstructed_gap_Mg'] = (

    full_gap_bootstrap.sum(axis=1)

)

assert np.allclose(

    full_gap_bootstrap[
        'reconstructed_gap_Mg'
    ],

    full_gap

)

full_gap_uncertainty = []

for component in [

    'architecture_Mg',

    'remaining_common_Mg',

    'outside_support_Mg'

]:

    values = full_gap_bootstrap[component]

    full_gap_uncertainty.append({

        'component': component,

        'mean_Mg': values.mean(),

        'LCL_95_Mg':
            values.quantile(0.025),

        'UCL_95_Mg':
            values.quantile(0.975),

        'mean_pct_of_full_gap':
            100 * values.mean() / full_gap,

        'LCL_95_pct':
            100 * values.quantile(0.025) / full_gap,

        'UCL_95_pct':
            100 * values.quantile(0.975) / full_gap

    })

full_gap_uncertainty = pd.DataFrame(
    full_gap_uncertainty
)

print('\nFULL GAP CONDITIONAL UNCERTAINTY')

display(
    full_gap_uncertainty.round(4)
)

# -----------------------------------------------------
# 5. Save
# -----------------------------------------------------

support_summary.to_csv(

    allometry_dir /
    'BIOMASS_DIAMETER_SUPPORT.csv',

    index=False

)

full_gap_decomposition.to_csv(

    allometry_dir /
    'FULL_INVENTORY_GAP_DECOMPOSITION.csv',

    index=False

)

full_gap_uncertainty.to_csv(

    allometry_dir /
    'FULL_INVENTORY_GAP_UNCERTAINTY.csv',

    index=False

)

print('\nCELL 55 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 56: INTEGRATED ARCHITECTURE FIGURE
# =====================================================

import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(
    2, 2,
    figsize=(12, 9)
)

secondary_color = '#D99043'
mature_color = '#28644F'
architecture_color = '#795D91'
remaining_color = '#557D9B'

# -----------------------------------------------------
# PANEL A: Counterfactual biomass
# -----------------------------------------------------

ax = axes[0, 0]

x = np.arange(2)

width = 0.34

original = [Bs, Bm]

exchanged = [

    baseline['secondary_counterfactual_Mg'],

    baseline['mature_counterfactual_Mg']

]

ax.bar(

    x - width/2,

    original,

    width,

    color=[secondary_color, mature_color],

    label='Original architecture'

)

ax.bar(

    x + width/2,

    exchanged,

    width,

    color=[secondary_color, mature_color],

    alpha=0.40,

    edgecolor='black',

    hatch='///',

    label='Exchanged architecture'

)

ax.set_xticks(x)

ax.set_xticklabels([

    'Late-secondary',

    'Mature'

])

ax.set_ylabel(
    'Aboveground biomass (Mg)'
)

ax.set_title(
    '(a) Architecture exchange'
)

ax.legend(
    frameon=False,
    fontsize=9
)

for i in range(2):

    ax.text(

        x[i] - width/2,

        original[i] + 5,

        f'{original[i]:.1f}',

        ha='center',

        fontsize=9

    )

    ax.text(

        x[i] + width/2,

        exchanged[i] + 5,

        f'{exchanged[i]:.1f}',

        ha='center',

        fontsize=9

    )

ax.set_ylim(
    0, max(original + exchanged) * 1.15
)

# -----------------------------------------------------
# PANEL B: Model sensitivity
# -----------------------------------------------------

ax = axes[0, 1]

sensitivity_plot = (

    counterfactual_model_sensitivity

    .sort_values(
        'architecture_contribution_pct'
    )

)

model_labels = (

    sensitivity_plot['model']

    .str.replace('_', ' ')

)

ax.barh(

    model_labels,

    sensitivity_plot[
        'architecture_contribution_pct'
    ],

    color=[

        architecture_color
        if m == selected_model
        else '#A9B8C4'

        for m in sensitivity_plot['model']

    ]

)

ax.axvline(

    baseline[
        'architecture_contribution_pct'
    ],

    color='gray',

    linestyle='--',

    linewidth=1

)

ax.set_xlabel(

    'Architecture contribution (%)'

)

ax.set_title(

    '(b) Sensitivity to allometric model'

)

# -----------------------------------------------------
# PANEL C: Bootstrap uncertainty
# -----------------------------------------------------

ax = axes[1, 0]

values = bootstrap_counterfactual[
    'architecture_contribution_pct'
]

ci_low, ci_high = np.percentile(

    values, [2.5, 97.5]

)

ax.hist(

    values,

    bins=35,

    color=architecture_color,

    alpha=0.75,

    edgecolor='white'

)

ax.axvline(

    baseline[
        'architecture_contribution_pct'
    ],

    color='black',

    linewidth=2,

    label='Point estimate'

)

ax.axvline(

    ci_low,

    color='gray',

    linestyle='--',

    linewidth=1.5,

    label='95% percentile interval'

)

ax.axvline(

    ci_high,

    color='gray',

    linestyle='--',

    linewidth=1.5

)

ax.set_xlabel(

    'Architecture contribution (%)'

)

ax.set_ylabel(
    'Bootstrap frequency'
)

ax.set_title(

    '(c) Conditional bootstrap uncertainty'

)

ax.legend(

    frameon=False,

    fontsize=9

)

# -----------------------------------------------------
# PANEL D: Full biomass gap waterfall
# -----------------------------------------------------

ax = axes[1, 1]

components = [

    architecture_point,

    remaining_point,

    outside_contribution

]

component_labels = [

    'Height\narchitecture',

    'Remaining\ncontrast',

    'Outside\nsupport'

]

running = 0

waterfall_colors = [

    architecture_color,

    remaining_color,

    mature_color

]

for i, value in enumerate(components):

    bottom = min(
        running,
        running + value
    )

    ax.bar(

        i,

        abs(value),

        bottom=bottom,

        color=waterfall_colors[i],

        width=0.65

    )

    ax.text(

        i,

        bottom + abs(value)/2,

        f'{value:+.1f}',

        ha='center',

        va='center',

        color='white',

        fontweight='bold'

    )

    running += value

    if i < len(components) - 1:

        ax.plot(

            [i + 0.325, i + 0.675],

            [running, running],

            color='gray',

            linestyle='--',

            linewidth=1

        )

ax.bar(

    3,

    full_gap,

    color='#444444',

    width=0.65

)

ax.text(

    3,

    full_gap/2,

    f'{full_gap:.1f}',

    color='white',

    fontweight='bold',

    ha='center',

    va='center'

)

ax.set_xticks(range(4))

ax.set_xticklabels(

    component_labels + ['Net biomass\ndifference'],

    fontsize=9

)

ax.set_ylabel(

    'Biomass difference (Mg)'

)

ax.set_title(

    '(d) Full inventory decomposition'

)

ax.axhline(

    0,

    color='gray',

    linewidth=1

)

# -----------------------------------------------------
# Final formatting
# -----------------------------------------------------

for ax in axes.flat:

    ax.spines['top'].set_visible(False)

    ax.spines['right'].set_visible(False)

    ax.grid(

        axis='y',

        linestyle='--',

        alpha=0.20

    )

    ax.set_axisbelow(True)

fig.tight_layout()

for extension in ['png', 'pdf', 'svg']:

    fig.savefig(

        figure_dir /

        f'FIG_ARCHITECTURE_DECOMPOSITION.{extension}',

        dpi=600,

        bbox_inches='tight'

    )

plt.show()

print('\nCELL 56 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 57: COMMUNITY-WEIGHTED WOOD DENSITY
# =====================================================

functional_dir = DIRS['analysis'] / 'FUNCTIONAL_ECOLOGY'
functional_dir.mkdir(parents=True, exist_ok=True)

functional = biomass_data.copy()

required_columns = [
    'forest_type',
    'tree_id',
    'wood_density_g_cm3',
    'basal_area_m2',
    'agb_kg',
    'dbh_cm',
    'height_m'
]

assert all(
    c in functional.columns
    for c in required_columns
)

assert len(functional) == 1002
assert functional['tree_id'].is_unique

functional['wd'] = pd.to_numeric(
    functional['wood_density_g_cm3'],
    errors='coerce'
)

assert functional['wd'].notna().all()
assert (functional['wd'] > 0).all()

functional['agb_Mg'] = (
    functional['agb_kg'] / 1000
)

functional['stem_weight'] = 1.0

# -----------------------------------------------------
# Weighted statistics
# -----------------------------------------------------

def weighted_stats(values, weights):

    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)

    valid = (
        np.isfinite(values)
        & np.isfinite(weights)
        & (weights > 0)
    )

    values = values[valid]
    weights = weights[valid]

    assert len(values) > 0

    weights = weights / weights.sum()

    mean = np.sum(weights * values)

    variance = np.sum(
        weights * (values - mean)**2
    )

    return mean, np.sqrt(variance)


weight_schemes = {

    'Abundance': 'stem_weight',

    'Basal_area': 'basal_area_m2',

    'Biomass': 'agb_kg'

}

cwm_results = []

for forest, group in functional.groupby(
    'forest_type'
):

    for scheme, weight_col in weight_schemes.items():

        mean, sd = weighted_stats(

            group['wd'],

            group[weight_col]

        )

        cwm_results.append({

            'forest_type': forest,

            'weighting': scheme,

            'n_trees': len(group),

            'CWM_wood_density': mean,

            'weighted_SD': sd,

            'weighted_CV_pct': 100 * sd / mean

        })

functional_cwm = pd.DataFrame(
    cwm_results
)

print('COMMUNITY-WEIGHTED WOOD DENSITY')

display(functional_cwm.round(4))

# -----------------------------------------------------
# Between-plot contrasts
# -----------------------------------------------------

cwm_contrast = (

    functional_cwm.pivot(

        index='weighting',

        columns='forest_type',

        values='CWM_wood_density'

    )

    .reset_index()

)

cwm_contrast['secondary_minus_mature'] = (

    cwm_contrast['Late_secondary']

    - cwm_contrast['Mature']

)

cwm_contrast['secondary_excess_pct'] = (

    100 * (

        cwm_contrast['Late_secondary']

        / cwm_contrast['Mature']

        - 1

    )

)

print('\nWOOD DENSITY CONTRAST')

display(cwm_contrast.round(4))

# -----------------------------------------------------
# Export
# -----------------------------------------------------

functional_cwm.to_csv(

    functional_dir / 'WOOD_DENSITY_CWM.csv',

    index=False

)

cwm_contrast.to_csv(

    functional_dir / 'WOOD_DENSITY_CWM_CONTRAST.csv',

    index=False

)

print('\nCELL 57 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 58: WOOD DENSITY ASSIGNMENT SENSITIVITY
# =====================================================

# Recover assignment level if necessary

if 'wd_level' not in functional.columns:

    assert 'wd_level' in master_v04.columns, (
        'Wood density assignment level not found'
    )

    functional = functional.merge(

        master_v04[
            ['tree_id', 'wd_level']
        ],

        on='tree_id',

        how='left',

        validate='one_to_one'

    )

assert functional['wd_level'].notna().all()

print('WOOD DENSITY ASSIGNMENT LEVELS')

display(

    pd.crosstab(

        functional['forest_type'],

        functional['wd_level']

    )

)

# -----------------------------------------------------
# Sensitivity scenarios
# -----------------------------------------------------

functional_scenarios = {

    'All_assignments':

        functional.copy(),

    'Species_and_genus':

        functional.loc[

            functional['wd_level'].isin([

                'species_accepted',

                'species_alias',

                'genus'

            ])

        ].copy(),

    'Species_level_only':

        functional.loc[

            functional['wd_level'].isin([

                'species_accepted',

                'species_alias'

            ])

        ].copy(),

    'Accepted_species_only':

        functional.loc[

            functional['wd_level'].eq(

                'species_accepted'

            )

        ].copy()

}

sensitivity_cwm_rows = []

for scenario, subset in functional_scenarios.items():

    for forest, group in subset.groupby(
        'forest_type'
    ):

        original = functional.loc[

            functional['forest_type'].eq(forest)

        ]

        for scheme, weight_col in weight_schemes.items():

            mean, sd = weighted_stats(

                group['wd'],

                group[weight_col]

            )

            sensitivity_cwm_rows.append({

                'scenario': scenario,

                'forest_type': forest,

                'weighting': scheme,

                'n_trees': len(group),

                'stem_coverage_pct':

                    100 * len(group) / len(original),

                'biomass_coverage_pct':

                    100 * group['agb_kg'].sum()

                    / original['agb_kg'].sum(),

                'CWM_wood_density': mean,

                'weighted_SD': sd

            })

functional_cwm_sensitivity = pd.DataFrame(
    sensitivity_cwm_rows
)

print('\nFUNCTIONAL SENSITIVITY RESULTS')

display(

    functional_cwm_sensitivity.round(4)

)

# -----------------------------------------------------
# Contrasts across assignment scenarios
# -----------------------------------------------------

functional_sensitivity_contrast = (

    functional_cwm_sensitivity.pivot(

        index=['scenario', 'weighting'],

        columns='forest_type',

        values='CWM_wood_density'

    )

    .reset_index()

)

functional_sensitivity_contrast['difference'] = (

    functional_sensitivity_contrast[
        'Late_secondary'
    ]

    - functional_sensitivity_contrast[
        'Mature'
    ]

)

functional_sensitivity_contrast[
    'secondary_excess_pct'
] = (

    100 * (

        functional_sensitivity_contrast[
            'Late_secondary'
        ]

        / functional_sensitivity_contrast[
            'Mature'
        ]

        - 1

    )

)

print('\nFUNCTIONAL CONTRAST ROBUSTNESS')

display(

    functional_sensitivity_contrast.round(4)

)

functional_cwm_sensitivity.to_csv(

    functional_dir /
    'WOOD_DENSITY_CWM_SENSITIVITY.csv',

    index=False

)

functional_sensitivity_contrast.to_csv(

    functional_dir /
    'WOOD_DENSITY_CWM_SENSITIVITY_CONTRAST.csv',

    index=False

)

print('\nCELL 58 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 59: FUNCTIONAL TRAIT DISTRIBUTIONS
# =====================================================

def weighted_quantile(values, weights, quantiles):

    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)

    valid = (

        np.isfinite(values)

        & np.isfinite(weights)

        & (weights > 0)

    )

    values = values[valid]
    weights = weights[valid]

    order = np.argsort(values)

    values = values[order]
    weights = weights[order]

    cumulative = (

        np.cumsum(weights) - 0.5 * weights

    ) / weights.sum()

    return np.interp(

        quantiles,

        cumulative,

        values

    )


distribution_rows = []

for forest, group in functional.groupby(
    'forest_type'
):

    for scheme, weight_col in weight_schemes.items():

        q05, q25, q50, q75, q95 = weighted_quantile(

            group['wd'],

            group[weight_col],

            [0.05, 0.25, 0.50, 0.75, 0.95]

        )

        distribution_rows.append({

            'forest_type': forest,

            'weighting': scheme,

            'WD_q05': q05,

            'WD_q25': q25,

            'WD_median': q50,

            'WD_q75': q75,

            'WD_q95': q95,

            'WD_IQR': q75 - q25,

            'WD_central90_range': q95 - q05

        })

functional_distribution = pd.DataFrame(
    distribution_rows
)

print('WOOD DENSITY DISTRIBUTION')

display(functional_distribution.round(4))

# -----------------------------------------------------
# Weighted empirical cumulative distributions
# -----------------------------------------------------

forest_colors = {

    'Late_secondary': '#D99043',

    'Mature': '#28644F'

}

forest_labels = {

    'Late_secondary': 'Late-secondary forest',

    'Mature': 'Mature forest'

}

fig, axes = plt.subplots(

    1, 3,

    figsize=(15, 4.5),

    sharex=True,

    sharey=True

)

panel_titles = [

    '(a) Stem-weighted distribution',

    '(b) Basal area-weighted distribution',

    '(c) Biomass-weighted distribution'

]

for ax, (scheme, weight_col), panel_title in zip(

    axes,

    weight_schemes.items(),

    panel_titles

):

    for forest, group in functional.groupby(
        'forest_type'
    ):

        trait_distribution = (

            group.groupby('wd')[weight_col]

            .sum()

            .sort_index()

        )

        x = trait_distribution.index.to_numpy()

        cumulative = (

            trait_distribution.cumsum()

            / trait_distribution.sum()

        ).to_numpy()

        ax.step(

            x,

            cumulative,

            where='post',

            color=forest_colors[forest],

            linewidth=2.3,

            label=forest_labels[forest]

        )

        cwm_value = functional_cwm.loc[

            functional_cwm['forest_type'].eq(forest)

            & functional_cwm['weighting'].eq(scheme),

            'CWM_wood_density'

        ].iloc[0]

        ax.axvline(

            cwm_value,

            color=forest_colors[forest],

            linestyle='--',

            alpha=0.75

        )

    ax.set_title(panel_title)

    ax.set_xlabel(

        r'Wood density (g cm$^{-3}$)'

    )

    ax.set_ylim(0, 1.02)

    ax.grid(

        axis='y',

        linestyle='--',

        alpha=0.25

    )

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[0].set_ylabel(

    'Cumulative weighted proportion'

)

axes[0].legend(

    frameon=False,

    fontsize=9

)

fig.tight_layout()

for ext in ['png', 'pdf', 'svg']:

    fig.savefig(

        figure_dir /
        f'FIG_WOOD_DENSITY_DISTRIBUTION.{ext}',

        dpi=600,

        bbox_inches='tight'

    )

plt.show()

functional_distribution.to_csv(

    functional_dir /
    'WOOD_DENSITY_DISTRIBUTION.csv',

    index=False

)

print('\nCELL 59 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 60: TAXONOMIC DOMINANCE AND BIOMASS
# =====================================================

# -----------------------------------------------------
# 1. Identify available taxonomic columns
# -----------------------------------------------------

taxon_candidates = [

    'operational_taxon',

    'taxon_operational',

    'taxon_final',

    'taxon_id_final',

    'species_final',

    'species_accepted',

    'species_clean'

]

available_taxon_columns = [

    c for c in taxon_candidates

    if c in functional.columns

]

print(
    'Available taxonomic columns:',
    available_taxon_columns
)

assert len(available_taxon_columns) > 0

taxon_col = available_taxon_columns[0]

print('Selected taxonomic column:', taxon_col)

functional['functional_taxon'] = (

    functional[taxon_col]

    .astype('string')

    .str.strip()

    .replace('', pd.NA)

)

# Retain unresolved original labels when necessary

if 'species_clean' in functional.columns:

    functional['functional_taxon'] = (

        functional['functional_taxon']

        .fillna(

            functional['species_clean']

            .astype('string')

            .str.strip()

        )

    )

functional['functional_taxon'] = (

    functional['functional_taxon']

    .fillna('Unresolved_taxon')

)

# -----------------------------------------------------
# 2. Taxon-level structural contributions
# -----------------------------------------------------

taxon_biomass = (

    functional.groupby([

        'forest_type',

        'functional_taxon'

    ])

    .agg(

        n_trees=('tree_id', 'size'),

        basal_area_m2=('basal_area_m2', 'sum'),

        agb_Mg=('agb_Mg', 'sum'),

        mean_dbh_cm=('dbh_cm', 'mean'),

        mean_height_m=('height_m', 'mean'),

        mean_WD=('wd', 'mean')

    )

    .reset_index()

)

taxon_biomass['abundance_pct'] = (

    100 * taxon_biomass['n_trees']

    / taxon_biomass.groupby('forest_type')

    ['n_trees'].transform('sum')

)

taxon_biomass['basal_area_pct'] = (

    100 * taxon_biomass['basal_area_m2']

    / taxon_biomass.groupby('forest_type')

    ['basal_area_m2'].transform('sum')

)

taxon_biomass['biomass_pct'] = (

    100 * taxon_biomass['agb_Mg']

    / taxon_biomass.groupby('forest_type')

    ['agb_Mg'].transform('sum')

)

taxon_biomass['mean_tree_AGB_Mg'] = (

    taxon_biomass['agb_Mg']

    / taxon_biomass['n_trees']

)

taxon_biomass = taxon_biomass.sort_values(

    ['forest_type', 'agb_Mg'],

    ascending=[True, False]

)

taxon_biomass['biomass_rank'] = (

    taxon_biomass.groupby('forest_type')

    .cumcount() + 1

)

taxon_biomass['cumulative_biomass_pct'] = (

    taxon_biomass.groupby('forest_type')

    ['biomass_pct'].cumsum()

)

print('\nTOP 15 BIOMASS CONTRIBUTORS')

display(

    taxon_biomass.loc[

        taxon_biomass['biomass_rank'] <= 15

    ].round(4)

)

# -----------------------------------------------------
# 3. Biomass concentration
# -----------------------------------------------------

concentration_rows = []

for forest, group in taxon_biomass.groupby(
    'forest_type'
):

    biomass_proportions = (

        group['agb_Mg']

        / group['agb_Mg'].sum()

    ).to_numpy()

    effective_biomass_taxa = (

        1 / np.sum(biomass_proportions**2)

    )

    for k in [1, 3, 5, 10, 20]:

        concentration_rows.append({

            'forest_type': forest,

            'top_taxa': k,

            'cumulative_biomass_pct':

                group.head(k)['biomass_pct'].sum(),

            'effective_biomass_taxa_q2':

                effective_biomass_taxa

        })

biomass_concentration = pd.DataFrame(
    concentration_rows
)

print('\nBIOMASS CONCENTRATION')

display(biomass_concentration.round(4))

# -----------------------------------------------------
# 4. Dominant taxon inspection
# -----------------------------------------------------

target_species = 'Pseudosenefeldera inclinata'

dominant_taxon = taxon_biomass.loc[

    taxon_biomass['functional_taxon']

    .str.casefold()

    .eq(target_species.casefold())

].copy()

print('\nPSEUDOSENEFELDERA INCLINATA')

display(dominant_taxon.round(4))

if dominant_taxon.empty:

    print(

        'WARNING: Dominant taxon not matched. '

        'Check the selected taxonomic column '

        'before interpreting results.'

    )

# -----------------------------------------------------
# 5. Export
# -----------------------------------------------------

taxon_biomass.to_csv(

    functional_dir /
    'TAXON_LEVEL_BIOMASS_CONTRIBUTIONS.csv',

    index=False

)

biomass_concentration.to_csv(

    functional_dir /
    'BIOMASS_TAXONOMIC_CONCENTRATION.csv',

    index=False

)

dominant_taxon.to_csv(

    functional_dir /
    'PSEUDOSENEFELDERA_BIOMASS_CONTRIBUTION.csv',

    index=False

)

print('\nCELL 60 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 61: DOMINANT SPECIES EXCLUSION SENSITIVITY
# =====================================================

target_species = 'Pseudosenefeldera inclinata'

dominant_mask = (

    functional['functional_taxon']

    .astype('string')

    .str.casefold()

    .eq(target_species.casefold())

    .fillna(False)

)

assert dominant_mask.sum() == 173

assert functional.loc[
    dominant_mask, 'forest_type'
].eq('Late_secondary').all()

dominance_scenarios = {

    'Full_inventory': functional.copy(),

    'Without_Pseudosenefeldera':

        functional.loc[~dominant_mask].copy()

}


def effective_diversity(weights):

    weights = np.asarray(weights, dtype=float)

    weights = weights[
        np.isfinite(weights) & (weights > 0)
    ]

    p = weights / weights.sum()

    return {

        'q0': len(p),

        'q1': np.exp(-np.sum(p * np.log(p))),

        'q2': 1 / np.sum(p**2)

    }


dominance_rows = []

for scenario, subset in dominance_scenarios.items():

    for forest, group in subset.groupby('forest_type'):

        taxon_summary = (

            group.groupby('functional_taxon')

            .agg(

                abundance=('tree_id', 'size'),

                biomass=('agb_Mg', 'sum'),

                basal_area=('basal_area_m2', 'sum')

            )

        )

        abundance_hill = effective_diversity(

            taxon_summary['abundance']

        )

        biomass_hill = effective_diversity(

            taxon_summary['biomass']

        )

        basal_area_hill = effective_diversity(

            taxon_summary['basal_area']

        )

        wd_abundance, wd_sd = weighted_stats(

            group['wd'],

            group['stem_weight']

        )

        wd_basal_area, _ = weighted_stats(

            group['wd'],

            group['basal_area_m2']

        )

        wd_biomass, _ = weighted_stats(

            group['wd'],

            group['agb_kg']

        )

        dominance_rows.append({

            'scenario': scenario,

            'forest_type': forest,

            'n_trees': len(group),

            'richness': abundance_hill['q0'],

            'abundance_q1': abundance_hill['q1'],

            'abundance_q2': abundance_hill['q2'],

            'basal_area_q2': basal_area_hill['q2'],

            'biomass_q2': biomass_hill['q2'],

            'biomass_abundance_q2_ratio':

                biomass_hill['q2']

                / abundance_hill['q2'],

            'basal_area_m2':

                group['basal_area_m2'].sum(),

            'AGB_Mg':

                group['agb_Mg'].sum(),

            'mean_dbh_cm':

                group['dbh_cm'].mean(),

            'mean_height_m':

                group['height_m'].mean(),

            'CWM_WD_abundance': wd_abundance,

            'CWM_WD_basal_area': wd_basal_area,

            'CWM_WD_biomass': wd_biomass,

            'WD_abundance_SD': wd_sd,

            'dominant_abundance_pct':

                100 * taxon_summary['abundance'].max()

                / taxon_summary['abundance'].sum(),

            'dominant_biomass_pct':

                100 * taxon_summary['biomass'].max()

                / taxon_summary['biomass'].sum()

        })

dominance_sensitivity = pd.DataFrame(
    dominance_rows
)

print('DOMINANT SPECIES EXCLUSION')

display(dominance_sensitivity.round(4))

# -----------------------------------------------------
# Within-secondary changes
# -----------------------------------------------------

secondary_sensitivity = (

    dominance_sensitivity.loc[

        dominance_sensitivity['forest_type']

        .eq('Late_secondary')

    ]

    .set_index('scenario')

)

metrics = [

    'n_trees',

    'richness',

    'abundance_q1',

    'abundance_q2',

    'biomass_q2',

    'basal_area_m2',

    'AGB_Mg',

    'mean_dbh_cm',

    'mean_height_m',

    'CWM_WD_abundance',

    'CWM_WD_basal_area',

    'CWM_WD_biomass'

]

exclusion_changes = []

for metric in metrics:

    original = secondary_sensitivity.loc[

        'Full_inventory', metric

    ]

    excluded = secondary_sensitivity.loc[

        'Without_Pseudosenefeldera', metric

    ]

    exclusion_changes.append({

        'metric': metric,

        'original': original,

        'without_dominant_species': excluded,

        'absolute_change': excluded - original,

        'relative_change_pct':

            100 * (excluded / original - 1)

    })

exclusion_changes = pd.DataFrame(
    exclusion_changes
)

print('\nWITHIN-SECONDARY CHANGES')

display(exclusion_changes.round(4))

# -----------------------------------------------------
# Between-forest contrast
# -----------------------------------------------------

exclusion_contrast_rows = []

for scenario, group in dominance_sensitivity.groupby(
    'scenario'
):

    comparison = group.set_index('forest_type')

    for metric in metrics:

        secondary = comparison.loc[

            'Late_secondary', metric

        ]

        mature = comparison.loc[

            'Mature', metric

        ]

        exclusion_contrast_rows.append({

            'scenario': scenario,

            'metric': metric,

            'secondary': secondary,

            'mature': mature,

            'secondary_minus_mature':

                secondary - mature,

            'secondary_mature_ratio':

                secondary / mature,

            'secondary_excess_pct':

                100 * (secondary / mature - 1)

        })

exclusion_contrast = pd.DataFrame(
    exclusion_contrast_rows
)

print('\nBETWEEN-FOREST CONTRAST')

display(exclusion_contrast.round(4))

# -----------------------------------------------------
# Validation
# -----------------------------------------------------

assert np.isclose(

    secondary_sensitivity.loc[
        'Full_inventory', 'AGB_Mg'
    ]

    - secondary_sensitivity.loc[
        'Without_Pseudosenefeldera', 'AGB_Mg'
    ],

    functional.loc[dominant_mask, 'agb_Mg'].sum()

)

# -----------------------------------------------------
# Export
# -----------------------------------------------------

dominance_sensitivity.to_csv(

    functional_dir /
    'DOMINANT_SPECIES_EXCLUSION.csv',

    index=False

)

exclusion_changes.to_csv(

    functional_dir /
    'DOMINANT_SPECIES_WITHIN_PLOT_CHANGES.csv',

    index=False

)

exclusion_contrast.to_csv(

    functional_dir /
    'DOMINANT_SPECIES_BETWEEN_PLOT_CONTRAST.csv',

    index=False

)

print('\nCELL 61 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 62: TAXONOMIC DECOMPOSITION OF BIOMASS GAP
# =====================================================

taxon_gap = (

    taxon_biomass.pivot_table(

        index='functional_taxon',

        columns='forest_type',

        values='agb_Mg',

        aggfunc='sum',

        fill_value=0

    )

    .reset_index()

)

taxon_gap.columns.name = None

taxon_gap['biomass_difference_Mg'] = (

    taxon_gap['Late_secondary']

    - taxon_gap['Mature']

)

total_biomass_gap = (

    taxon_gap['biomass_difference_Mg'].sum()

)

assert np.isclose(

    total_biomass_gap,

    functional.loc[

        functional['forest_type'].eq(
            'Late_secondary'
        ),

        'agb_Mg'

    ].sum()

    - functional.loc[

        functional['forest_type'].eq('Mature'),

        'agb_Mg'

    ].sum()

)

taxon_gap['net_gap_contribution_pct'] = (

    100 * taxon_gap['biomass_difference_Mg']

    / total_biomass_gap

)

taxon_gap['absolute_difference_Mg'] = (

    taxon_gap['biomass_difference_Mg'].abs()

)

taxon_gap['absolute_contribution_pct'] = (

    100 * taxon_gap['absolute_difference_Mg']

    / taxon_gap['absolute_difference_Mg'].sum()

)

taxon_gap['direction'] = np.where(

    taxon_gap['biomass_difference_Mg'] > 0,

    'Higher_in_secondary',

    np.where(

        taxon_gap['biomass_difference_Mg'] < 0,

        'Higher_in_mature',

        'Equal'

    )

)

taxon_gap = taxon_gap.sort_values(

    'absolute_difference_Mg',

    ascending=False

)

taxon_gap['cumulative_absolute_pct'] = (

    taxon_gap['absolute_contribution_pct']

    .cumsum()

)

print('TOP 20 TAXONOMIC BIOMASS CONTRIBUTORS')

display(taxon_gap.head(20).round(4))

# -----------------------------------------------------
# Positive and negative contributions
# -----------------------------------------------------

positive_gap = taxon_gap.loc[

    taxon_gap['biomass_difference_Mg'] > 0,

    'biomass_difference_Mg'

].sum()

negative_gap = taxon_gap.loc[

    taxon_gap['biomass_difference_Mg'] < 0,

    'biomass_difference_Mg'

].sum()

dominant_gap = taxon_gap.loc[

    taxon_gap['functional_taxon']

    .eq(target_species),

    'biomass_difference_Mg'

].sum()

gap_summary = pd.DataFrame([

    {

        'component': 'Positive contributions',

        'biomass_Mg': positive_gap

    },

    {

        'component': 'Negative contributions',

        'biomass_Mg': negative_gap

    },

    {

        'component': 'Net biomass gap',

        'biomass_Mg': total_biomass_gap

    },

    {

        'component': 'Pseudosenefeldera contribution',

        'biomass_Mg': dominant_gap

    },

    {

        'component': 'Remaining net gap',

        'biomass_Mg':

            total_biomass_gap - dominant_gap

    }

])

gap_summary['pct_of_net_gap'] = (

    100 * gap_summary['biomass_Mg']

    / total_biomass_gap

)

print('\nBIOMASS GAP DECOMPOSITION')

display(gap_summary.round(4))

assert np.isclose(

    positive_gap + negative_gap,

    total_biomass_gap

)

taxon_gap.to_csv(

    functional_dir /
    'TAXONOMIC_BIOMASS_GAP.csv',

    index=False

)

gap_summary.to_csv(

    functional_dir /
    'TAXONOMIC_BIOMASS_GAP_SUMMARY.csv',

    index=False

)

print('\nCELL 62 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 63: DOMINANCE AND BIOMASS INTEGRATED FIGURE
# =====================================================

fig, axes = plt.subplots(

    2, 2,

    figsize=(13, 10)

)

# -----------------------------------------------------
# Panel A: Abundance versus biomass
# -----------------------------------------------------

ax = axes[0, 0]

for forest, group in taxon_biomass.groupby(
    'forest_type'
):

    ax.scatter(

        group['abundance_pct'],

        group['biomass_pct'],

        color=forest_colors[forest],

        label=forest_labels[forest],

        alpha=0.70,

        s=35,

        edgecolors='white',

        linewidths=0.4

    )

limit = max(

    taxon_biomass['abundance_pct'].max(),

    taxon_biomass['biomass_pct'].max()

) * 1.12

ax.plot(

    [0, limit],

    [0, limit],

    '--',

    color='gray',

    linewidth=1

)

for species in [

    'Pseudosenefeldera inclinata',

    'Meliosma frondosa',

    'Ficus insipida'

]:

    rows = taxon_biomass.loc[

        taxon_biomass['functional_taxon'].eq(species)

    ]

    row = rows.loc[rows['agb_Mg'].idxmax()]

    short_name = (

        species.split()[0][0]

        + '. '

        + species.split()[1]

    )

    ax.annotate(

        short_name,

        (

            row['abundance_pct'],

            row['biomass_pct']

        ),

        xytext=(5, 7),

        textcoords='offset points',

        fontsize=9,

        fontstyle='italic'

    )

ax.set_xlim(0, limit)
ax.set_ylim(0, limit)

ax.set_xlabel('Relative abundance (%)')
ax.set_ylabel('Relative biomass (%)')

ax.set_title('(a) Abundance–biomass decoupling')

ax.legend(frameon=False, fontsize=9)

# -----------------------------------------------------
# Panel B: Biomass concentration curves
# -----------------------------------------------------

ax = axes[0, 1]

for forest, group in taxon_biomass.groupby(
    'forest_type'
):

    group = group.sort_values(

        'agb_Mg',

        ascending=False

    )

    x = np.arange(1, len(group) + 1)

    y = group['biomass_pct'].cumsum()

    ax.plot(

        x, y,

        color=forest_colors[forest],

        linewidth=2.4,

        label=forest_labels[forest]

    )

ax.axhline(

    50,

    color='gray',

    linestyle='--',

    linewidth=1

)

ax.set_xlim(1, 30)
ax.set_ylim(0, 100)

ax.set_xlabel('Number of highest-biomass taxa')
ax.set_ylabel('Cumulative biomass (%)')

ax.set_title('(b) Taxonomic biomass concentration')

# -----------------------------------------------------
# Panel C: Signed contributions to biomass gap
# -----------------------------------------------------

ax = axes[1, 0]

top_gap = (

    taxon_gap.head(12)

    .sort_values('biomass_difference_Mg')

)

bar_colors = np.where(

    top_gap['biomass_difference_Mg'] > 0,

    forest_colors['Late_secondary'],

    forest_colors['Mature']

)

ax.barh(

    top_gap['functional_taxon'],

    top_gap['biomass_difference_Mg'],

    color=bar_colors

)

ax.axvline(

    0,

    color='gray',

    linewidth=1

)

ax.set_xlabel(

    'Biomass difference: secondary − mature (Mg)'

)

ax.set_title('(c) Taxonomic contributors to biomass gap')

ax.tick_params(axis='y', labelsize=9)

for label in ax.get_yticklabels():

    label.set_fontstyle('italic')

# -----------------------------------------------------
# Panel D: Dominant species exclusion
# -----------------------------------------------------

ax = axes[1, 1]

secondary_full = secondary_sensitivity.loc[

    'Full_inventory',

    'AGB_Mg'

]

secondary_excluded = secondary_sensitivity.loc[

    'Without_Pseudosenefeldera',

    'AGB_Mg'

]

mature_full = functional.loc[

    functional['forest_type'].eq('Mature'),

    'agb_Mg'

].sum()

values = [

    secondary_full,

    secondary_excluded,

    mature_full

]

labels = [

    'Secondary\nfull inventory',

    'Secondary\nwithout P. inclinata',

    'Mature\nfull inventory'

]

bars = ax.bar(

    labels,

    values,

    color=[

        forest_colors['Late_secondary'],

        forest_colors['Late_secondary'],

        forest_colors['Mature']

    ],

    width=0.65

)

bars[1].set_hatch('///')
bars[1].set_alpha(0.65)

ax.bar_label(

    bars,

    fmt='%.1f',

    padding=4,

    fontsize=10

)

ax.set_ylim(

    0,

    max(values) * 1.18

)

ax.set_ylabel('Aboveground biomass (Mg)')

ax.set_title('(d) Dominant taxon exclusion')

# -----------------------------------------------------
# Formatting
# -----------------------------------------------------

for ax in axes.flat:

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.grid(

        axis='y',

        linestyle='--',

        alpha=0.20

    )

    ax.set_axisbelow(True)

fig.tight_layout()

for ext in ['png', 'pdf', 'svg']:

    fig.savefig(

        figure_dir /
        f'FIG_TAXONOMIC_BIOMASS_DECOUPLING.{ext}',

        dpi=600,

        bbox_inches='tight'

    )

plt.show()

print('\nCELL 63 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 64: WOOD DENSITY STANDARDIZATION
# =====================================================

# Chave et al. (2014) pantropical allometry
# AGB (kg) = 0.0673 * (rho * D^2 * H)^0.976

def chave_agb_kg(rho, dbh, height):

    return 0.0673 * (

        rho * dbh**2 * height

    )**0.976


wd_counterfactual = functional.copy()

# -----------------------------------------------------
# Verify consistency with original biomass estimates
# -----------------------------------------------------

wd_counterfactual['reconstructed_AGB_kg'] = (

    chave_agb_kg(

        wd_counterfactual['wd'],

        wd_counterfactual['dbh_cm'],

        wd_counterfactual['height_m']

    )

)

relative_error = (

    (

        wd_counterfactual['reconstructed_AGB_kg']

        - wd_counterfactual['agb_kg']

    ).abs()

    / wd_counterfactual['agb_kg']

)

print(

    'Maximum relative reconstruction error:',

    relative_error.max()

)

assert np.allclose(

    wd_counterfactual['reconstructed_AGB_kg'],

    wd_counterfactual['agb_kg'],

    rtol=1e-5,

    atol=1e-5

), 'Check original biomass equations before proceeding'

# -----------------------------------------------------
# Reference density scenarios
# -----------------------------------------------------

pooled_basal_area_WD, _ = weighted_stats(

    wd_counterfactual['wd'],

    wd_counterfactual['basal_area_m2']

)

pooled_abundance_WD = (

    wd_counterfactual['wd'].mean()

)

wd_references = {

    'Pooled_basal_area_weighted':

        pooled_basal_area_WD,

    'Pooled_abundance_weighted':

        pooled_abundance_WD,

    'Fixed_0.50': 0.50,

    'Fixed_0.60': 0.60,

    'Fixed_0.70': 0.70

}

wd_standardization_rows = []

for scenario, reference in wd_references.items():

    wd_counterfactual['standardized_AGB_kg'] = (

        chave_agb_kg(

            reference,

            wd_counterfactual['dbh_cm'],

            wd_counterfactual['height_m']

        )

    )

    for forest, group in wd_counterfactual.groupby(
        'forest_type'
    ):

        observed = group['agb_kg'].sum() / 1000

        standardized = (

            group['standardized_AGB_kg'].sum()

            / 1000

        )

        wd_standardization_rows.append({

            'scenario': scenario,

            'reference_WD': reference,

            'forest_type': forest,

            'n_trees': len(group),

            'original_AGB_Mg': observed,

            'standardized_AGB_Mg': standardized,

            'AGB_change_Mg':

                standardized - observed,

            'AGB_change_pct':

                100 * (standardized / observed - 1)

        })

wd_standardization = pd.DataFrame(
    wd_standardization_rows
)

print('WOOD DENSITY STANDARDIZATION')

display(wd_standardization.round(4))

# -----------------------------------------------------
# Between-plot contrasts
# -----------------------------------------------------

wd_gap_rows = []

for scenario, group in wd_standardization.groupby(
    'scenario'
):

    group = group.set_index('forest_type')

    original_gap = (

        group.loc[

            'Late_secondary',

            'original_AGB_Mg'

        ]

        - group.loc[

            'Mature',

            'original_AGB_Mg'

        ]

    )

    standardized_gap = (

        group.loc[

            'Late_secondary',

            'standardized_AGB_Mg'

        ]

        - group.loc[

            'Mature',

            'standardized_AGB_Mg'

        ]

    )

    standardized_ratio = (

        group.loc[

            'Late_secondary',

            'standardized_AGB_Mg'

        ]

        / group.loc[

            'Mature',

            'standardized_AGB_Mg'

        ]

    )

    wd_gap_rows.append({

        'scenario': scenario,

        'reference_WD':

            group['reference_WD'].iloc[0],

        'original_gap_Mg': original_gap,

        'standardized_gap_Mg': standardized_gap,

        'gap_reduction_Mg':

            original_gap - standardized_gap,

        'gap_reduction_pct':

            100 * (

                original_gap - standardized_gap

            ) / original_gap,

        'standardized_secondary_mature_ratio':

            standardized_ratio

    })

wd_gap_sensitivity = pd.DataFrame(wd_gap_rows)

print('\nWOOD DENSITY STANDARDIZED CONTRAST')

display(wd_gap_sensitivity.round(4))

wd_standardization.to_csv(

    functional_dir /
    'WOOD_DENSITY_STANDARDIZED_BIOMASS.csv',

    index=False

)

wd_gap_sensitivity.to_csv(

    functional_dir /
    'WOOD_DENSITY_STANDARDIZED_GAP.csv',

    index=False

)

print('\nCELL 64 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 65: EXACT MULTIPLICATIVE BIOMASS FACTORIZATION
# =====================================================

import itertools
import math

decomposition_data = functional.copy()

required_columns = [

    'forest_type',
    'tree_id',
    'dbh_cm',
    'height_m',
    'wd',
    'agb_kg'

]

assert set(required_columns).issubset(
    decomposition_data.columns
)

assert decomposition_data['tree_id'].is_unique

assert decomposition_data[
    ['dbh_cm', 'height_m', 'wd', 'agb_kg']
].notna().all().all()

assert (
    decomposition_data[
        ['dbh_cm', 'height_m', 'wd', 'agb_kg']
    ] > 0
).all().all()

CHAVE_COEF = 0.0673
EXP_D = 1.952
EXP_H = 0.976
EXP_WD = 0.976


def biomass_factors(group):

    d = group['dbh_cm'].to_numpy(float)
    h = group['height_m'].to_numpy(float)
    rho = group['wd'].to_numpy(float)

    d_term = d ** EXP_D
    h_term = h ** EXP_H
    rho_term = rho ** EXP_WD

    n = len(group)

    m_d = d_term.mean()

    m_h = (

        np.sum(d_term * h_term)

        / np.sum(d_term)

    )

    m_rho = (

        np.sum(d_term * h_term * rho_term)

        / np.sum(d_term * h_term)

    )

    reconstructed_Mg = (

        CHAVE_COEF

        * n

        * m_d

        * m_h

        * m_rho

        / 1000

    )

    observed_Mg = (

        group['agb_kg'].sum() / 1000

    )

    assert np.isclose(

        reconstructed_Mg,

        observed_Mg,

        rtol=1e-8

    )

    return {

        'N': n,

        'Diameter_moment': m_d,

        'Height_factor': m_h,

        'Wood_density_factor': m_rho,

        'Effective_diameter_cm':

            m_d ** (1 / EXP_D),

        'Effective_height_m':

            m_h ** (1 / EXP_H),

        'Effective_wood_density':

            m_rho ** (1 / EXP_WD),

        'Observed_AGB_Mg': observed_Mg,

        'Reconstructed_AGB_Mg':

            reconstructed_Mg

    }


factor_rows = []

for forest, group in decomposition_data.groupby(
    'forest_type'
):

    result = biomass_factors(group)

    result['forest_type'] = forest

    factor_rows.append(result)

factor_table = pd.DataFrame(
    factor_rows
).set_index('forest_type')

print('EXACT BIOMASS FACTORIZATION')

display(factor_table.round(5))

# -----------------------------------------------------
# Multiplicative contrasts
# -----------------------------------------------------

factor_names = [

    'N',

    'Diameter_moment',

    'Height_factor',

    'Wood_density_factor'

]

secondary_factors = factor_table.loc[
    'Late_secondary', factor_names
].astype(float)

mature_factors = factor_table.loc[
    'Mature', factor_names
].astype(float)

factor_ratios = (

    secondary_factors / mature_factors

)

observed_ratio = (

    factor_table.loc[
        'Late_secondary', 'Observed_AGB_Mg'
    ]

    / factor_table.loc[
        'Mature', 'Observed_AGB_Mg'
    ]

)

reconstructed_ratio = factor_ratios.prod()

assert np.isclose(

    observed_ratio,

    reconstructed_ratio

)

factor_contrast = pd.DataFrame({

    'secondary': secondary_factors,

    'mature': mature_factors,

    'secondary_mature_ratio': factor_ratios,

    'relative_difference_pct':

        100 * (factor_ratios - 1)

})

print('\nMULTIPLICATIVE FACTOR CONTRAST')

display(factor_contrast.round(5))

print('\nObserved biomass ratio:', observed_ratio)

print('Reconstructed ratio:', reconstructed_ratio)

# -----------------------------------------------------
# Export
# -----------------------------------------------------

factor_table.to_csv(

    functional_dir /

    'BIOMASS_EXACT_FACTORIZATION.csv'

)

factor_contrast.to_csv(

    functional_dir /

    'BIOMASS_MULTIPLICATIVE_CONTRAST.csv'

)

print('\nCELL 65 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 66: EXACT LMDI AND SHAPLEY DECOMPOSITION
# =====================================================

B_secondary = factor_table.loc[

    'Late_secondary',

    'Observed_AGB_Mg'

]

B_mature = factor_table.loc[

    'Mature',

    'Observed_AGB_Mg'

]

delta_B = B_secondary - B_mature

# -----------------------------------------------------
# 1. LMDI decomposition
# -----------------------------------------------------

def logarithmic_mean(a, b):

    if np.isclose(a, b):

        return (a + b) / 2

    return (

        (a - b)

        / (np.log(a) - np.log(b))

    )


L_B = logarithmic_mean(

    B_secondary,

    B_mature

)

lmdi_contributions = (

    L_B * np.log(factor_ratios)

)

assert np.isclose(

    lmdi_contributions.sum(),

    delta_B

)

# -----------------------------------------------------
# 2. Shapley decomposition
# -----------------------------------------------------

def evaluate_factor_biomass(factors):

    return (

        CHAVE_COEF

        * np.prod([

            factors[name]

            for name in factor_names

        ])

        / 1000

    )


def shapley_decomposition(
    baseline, target
):

    permutations = list(

        itertools.permutations(
            factor_names
        )

    )

    contributions = {

        name: []

        for name in factor_names

    }

    path_rows = []

    for order in permutations:

        current = baseline.to_dict()

        previous_B = evaluate_factor_biomass(
            current
        )

        path = {

            'order': ' > '.join(order)

        }

        for name in order:

            current[name] = target[name]

            current_B = (

                evaluate_factor_biomass(
                    current
                )

            )

            marginal = (

                current_B - previous_B

            )

            contributions[name].append(
                marginal
            )

            path[name] = marginal

            previous_B = current_B

        path_rows.append(path)

    shapley = pd.Series({

        name: np.mean(values)

        for name, values in
        contributions.items()

    })

    return shapley, pd.DataFrame(path_rows)


shapley_contributions, shapley_paths = (

    shapley_decomposition(

        mature_factors,

        secondary_factors

    )

)

assert len(shapley_paths) == 24

assert np.isclose(

    shapley_contributions.sum(),

    delta_B

)

# -----------------------------------------------------
# 3. Integrated results
# -----------------------------------------------------

integrated_decomposition = pd.DataFrame({

    'component': factor_names,

    'LMDI_Mg':

        lmdi_contributions.reindex(
            factor_names
        ).values,

    'Shapley_Mg':

        shapley_contributions.reindex(
            factor_names
        ).values

})

integrated_decomposition['LMDI_pct'] = (

    100

    * integrated_decomposition['LMDI_Mg']

    / delta_B

)

integrated_decomposition['Shapley_pct'] = (

    100

    * integrated_decomposition['Shapley_Mg']

    / delta_B

)

integrated_decomposition[
    'method_difference_Mg'
] = (

    integrated_decomposition['Shapley_Mg']

    - integrated_decomposition['LMDI_Mg']

)

print('INTEGRATED BIOMASS DECOMPOSITION')

display(integrated_decomposition.round(4))

print('\nObserved difference:', delta_B)

print(

    'LMDI reconstructed difference:',

    lmdi_contributions.sum()

)

print(

    'Shapley reconstructed difference:',

    shapley_contributions.sum()

)

print('\nORDER SENSITIVITY')

order_sensitivity = pd.DataFrame({

    'minimum_Mg':

        shapley_paths[factor_names].min(),

    'maximum_Mg':

        shapley_paths[factor_names].max(),

    'mean_Mg':

        shapley_paths[factor_names].mean(),

    'SD_Mg':

        shapley_paths[factor_names].std()

})

display(order_sensitivity.round(4))

# -----------------------------------------------------
# Export
# -----------------------------------------------------

integrated_decomposition.to_csv(

    functional_dir /

    'BIOMASS_INTEGRATED_LMDI_SHAPLEY.csv',

    index=False

)

shapley_paths.to_csv(

    functional_dir /

    'BIOMASS_SHAPLEY_ALL_24_PATHS.csv',

    index=False

)

order_sensitivity.to_csv(

    functional_dir /

    'BIOMASS_DECOMPOSITION_ORDER_SENSITIVITY.csv'

)

print('\nCELL 66 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 67: INTEGRATED DECOMPOSITION SENSITIVITY
# =====================================================

integrated_scenarios = {

    'Full_inventory':

        functional.copy(),

    'Without_Pseudosenefeldera':

        functional.loc[
            ~dominant_mask
        ].copy(),

    'Common_diameter_support':

        functional.loc[

            functional['dbh_cm'].between(
                10, 89.2
            )

        ].copy(),

    'Species_level_WD_only':

        functional.loc[

            functional['wd_level'].isin([

                'species_accepted',

                'species_alias'

            ])

        ].copy()

}

scenario_results = []

scenario_factor_results = []

for scenario, subset in integrated_scenarios.items():

    factors = pd.DataFrame([

        {

            'forest_type': forest,

            **biomass_factors(group)

        }

        for forest, group in

        subset.groupby('forest_type')

    ]).set_index('forest_type')

    s = factors.loc[

        'Late_secondary',

        factor_names

    ].astype(float)

    m = factors.loc[

        'Mature',

        factor_names

    ].astype(float)

    bs = factors.loc[

        'Late_secondary',

        'Observed_AGB_Mg'

    ]

    bm = factors.loc[

        'Mature',

        'Observed_AGB_Mg'

    ]

    gap = bs - bm

    ratios = s / m

    lmdi = (

        logarithmic_mean(bs, bm)

        * np.log(ratios)

    )

    shapley, paths = (

        shapley_decomposition(m, s)

    )

    assert np.isclose(lmdi.sum(), gap)

    assert np.isclose(shapley.sum(), gap)

    for component in factor_names:

        scenario_results.append({

            'scenario': scenario,

            'component': component,

            'secondary_AGB_Mg': bs,

            'mature_AGB_Mg': bm,

            'total_gap_Mg': gap,

            'factor_ratio':

                ratios[component],

            'LMDI_Mg':

                lmdi[component],

            'LMDI_pct':

                100 * lmdi[component] / gap

                if not np.isclose(gap, 0)

                else np.nan,

            'Shapley_Mg':

                shapley[component],

            'Shapley_pct':

                100 * shapley[component] / gap

                if not np.isclose(gap, 0)

                else np.nan

        })

    for forest, row in factors.iterrows():

        scenario_factor_results.append({

            'scenario': scenario,

            'forest_type': forest,

            **row.to_dict()

        })

integrated_sensitivity = pd.DataFrame(
    scenario_results
)

integrated_factor_sensitivity = pd.DataFrame(
    scenario_factor_results
)

print('INTEGRATED SENSITIVITY RESULTS')

display(integrated_sensitivity.round(4))

print('\nSHAPLEY CONTRIBUTIONS BY SCENARIO (Mg)')

display(

    integrated_sensitivity.pivot(

        index='component',

        columns='scenario',

        values='Shapley_Mg'

    ).round(4)

)

print('\nSHAPLEY CONTRIBUTIONS BY SCENARIO (%)')

display(

    integrated_sensitivity.pivot(

        index='component',

        columns='scenario',

        values='Shapley_pct'

    ).round(4)

)

integrated_sensitivity.to_csv(

    functional_dir /

    'BIOMASS_INTEGRATED_SENSITIVITY.csv',

    index=False

)

integrated_factor_sensitivity.to_csv(

    functional_dir /

    'BIOMASS_FACTORS_BY_SCENARIO.csv',

    index=False

)

print('\nCELL 67 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 68: INTEGRATED BIOMASS DECOMPOSITION FIGURE
# =====================================================

component_labels = {

    'N': 'Stem abundance',

    'Diameter_moment': 'Diameter distribution',

    'Height_factor': 'Height dimension',

    'Wood_density_factor': 'Wood density'

}

component_colors = {

    'N': '#D99043',

    'Diameter_moment': '#507B99',

    'Height_factor': '#795B91',

    'Wood_density_factor': '#28644F'

}

scenario_labels = {

    'Full_inventory': 'Full inventory',

    'Without_Pseudosenefeldera':

        'Without P. inclinata',

    'Common_diameter_support':

        'Common diameter support',

    'Species_level_WD_only':

        'Species-level WD only'

}

fig, axes = plt.subplots(

    1, 3,

    figsize=(16, 5.5),

    gridspec_kw={

        'width_ratios': [1.0, 1.15, 1.3]

    }

)

# -----------------------------------------------------
# A. Multiplicative factors
# -----------------------------------------------------

ax = axes[0]

ratios = factor_ratios.reindex(
    factor_names
)

y = np.arange(len(factor_names))

for i, component in enumerate(factor_names):

    value = ratios[component]

    ax.hlines(

        y=i,

        xmin=min(1, value),

        xmax=max(1, value),

        color=component_colors[component],

        linewidth=3

    )

    ax.scatter(

        value, i,

        s=85,

        color=component_colors[component],

        zorder=3

    )

    ax.annotate(

        f'{value:.3f}',

        (value, i),

        xytext=(0, 10),

        textcoords='offset points',

        ha='center',

        fontsize=10

    )

ax.axvline(

    1,

    linestyle='--',

    color='gray'

)

ax.set_yticks(

    y,

    [component_labels[c] for c in factor_names]

)

ax.invert_yaxis()

ax.set_xlabel('Secondary / mature factor ratio')

ax.set_title('(a) Multiplicative contrasts')

ax.margins(x=0.25, y=0.2)

# -----------------------------------------------------
# B. Exact Shapley waterfall
# -----------------------------------------------------

ax = axes[1]

running = 0

for i, component in enumerate(factor_names):

    contribution = shapley_contributions[component]

    next_value = running + contribution

    ax.bar(

        i,

        abs(contribution),

        bottom=min(running, next_value),

        color=component_colors[component],

        width=0.7

    )

    ax.text(

        i,

        (running + next_value) / 2,

        f'{contribution:+.1f}',

        ha='center',

        va='center',

        color='white',

        fontweight='bold',

        fontsize=10

    )

    if i < len(factor_names) - 1:

        ax.plot(

            [i + 0.35, i + 0.65],

            [next_value, next_value],

            '--',

            color='gray',

            linewidth=0.8

        )

    running = next_value

ax.bar(

    len(factor_names),

    delta_B,

    color='#444444',

    width=0.7

)

ax.text(

    len(factor_names),

    delta_B / 2,

    f'{delta_B:.1f}',

    color='white',

    ha='center',

    va='center',

    fontweight='bold'

)

ax.axhline(0, color='gray', linewidth=0.8)

ax.set_xticks(

    np.arange(5),

    [

        'Stems',

        'Diameter',

        'Height',

        'Wood\ndensity',

        'Net gap'

    ],

    rotation=20,

    ha='right'

)

ax.set_ylabel('Biomass contribution (Mg)')

ax.set_title('(b) Exact biomass decomposition')

# -----------------------------------------------------
# C. Scenario sensitivity
# -----------------------------------------------------

ax = axes[2]

scenario_order = list(
    integrated_scenarios.keys()
)

scenario_y = np.arange(
    len(scenario_order)
)

width = 0.18

for j, component in enumerate(factor_names):

    values = (

        integrated_sensitivity.loc[

            integrated_sensitivity[
                'component'
            ].eq(component)

        ]

        .set_index('scenario')

        .reindex(scenario_order)[
            'Shapley_Mg'
        ]

    )

    ax.barh(

        scenario_y + (j - 1.5) * width,

        values,

        height=width,

        color=component_colors[component],

        label=component_labels[component]

    )

ax.axvline(

    0,

    color='gray',

    linewidth=0.9

)

ax.set_yticks(

    scenario_y,

    [scenario_labels[s] for s in scenario_order]

)

ax.invert_yaxis()

ax.set_xlabel('Shapley contribution (Mg)')

ax.set_title('(c) Sensitivity across inventories')

ax.legend(

    frameon=False,

    fontsize=8,

    loc='best'

)

# -----------------------------------------------------
# Final formatting
# -----------------------------------------------------

for ax in axes:

    ax.spines['top'].set_visible(False)

    ax.spines['right'].set_visible(False)

    ax.set_axisbelow(True)

axes[0].grid(

    axis='x',

    linestyle='--',

    alpha=0.2

)

axes[1].grid(

    axis='y',

    linestyle='--',

    alpha=0.2

)

axes[2].grid(

    axis='x',

    linestyle='--',

    alpha=0.2

)

fig.tight_layout()

for ext in ['png', 'pdf', 'svg']:

    fig.savefig(

        figure_dir /

        f'FIG_INTEGRATED_BIOMASS_DECOMPOSITION.{ext}',

        dpi=600,

        bbox_inches='tight'

    )

plt.show()

print('\nCELL 68 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 69: SUBPLOT BOOTSTRAP OF BIOMASS DECOMPOSITION
# =====================================================

from tqdm.auto import tqdm

BOOT_N = 2000
rng = np.random.default_rng(20261115)

boot_data = functional.copy()

assert 'subplot_id' in boot_data.columns

forest_order = [
    'Late_secondary',
    'Mature'
]

# -----------------------------------------------------
# 1. Aggregate sufficient statistics by subplot
# -----------------------------------------------------

boot_data['D_term'] = (
    boot_data['dbh_cm'] ** EXP_D
)

boot_data['DH_term'] = (
    boot_data['D_term']
    * boot_data['height_m'] ** EXP_H
)

boot_data['DHWD_term'] = (
    boot_data['DH_term']
    * boot_data['wd'] ** EXP_WD
)

subplot_stats = (

    boot_data.groupby(
        ['forest_type', 'subplot_id']
    )

    .agg(

        N=('tree_id', 'size'),

        sum_D=('D_term', 'sum'),

        sum_DH=('DH_term', 'sum'),

        sum_DHWD=('DHWD_term', 'sum')

    )

    .reset_index()

)

print('SUBPLOT SUPPORT')

display(

    subplot_stats.groupby(
        'forest_type'
    ).size()

)

assert (

    subplot_stats.groupby(
        'forest_type'
    ).size() == 25

).all()

# -----------------------------------------------------
# 2. Fast exact factor reconstruction
# -----------------------------------------------------

def factors_from_totals(totals):

    n, sd, sdh, sdhwd = totals

    return pd.Series({

        'N': n,

        'Diameter_moment': sd / n,

        'Height_factor': sdh / sd,

        'Wood_density_factor': sdhwd / sdh

    })


# Precompute Shapley subset weights

def shapley_fast(baseline, target):

    m = baseline.reindex(
        factor_names
    ).to_numpy(float)

    s = target.reindex(
        factor_names
    ).to_numpy(float)

    p = len(m)

    result = np.zeros(p)

    for i in range(p):

        others = [

            j for j in range(p)
            if j != i

        ]

        for k in range(p):

            weight = (

                math.factorial(k)

                * math.factorial(p-k-1)

                / math.factorial(p)

            )

            for subset in itertools.combinations(
                others, k
            ):

                x = m.copy()

                for j in subset:

                    x[j] = s[j]

                before = (
                    CHAVE_COEF
                    * np.prod(x)
                    / 1000
                )

                x[i] = s[i]

                after = (
                    CHAVE_COEF
                    * np.prod(x)
                    / 1000
                )

                result[i] += (
                    weight * (after-before)
                )

    return pd.Series(
        result,
        index=factor_names
    )


assert np.allclose(

    shapley_fast(
        mature_factors,
        secondary_factors
    ).values,

    shapley_contributions.reindex(
        factor_names
    ).values

)

# -----------------------------------------------------
# 3. Subplot bootstrap
# -----------------------------------------------------

stat_cols = [
    'N',
    'sum_D',
    'sum_DH',
    'sum_DHWD'
]

subplot_arrays = {

    forest:

        subplot_stats.loc[

            subplot_stats[
                'forest_type'
            ].eq(forest),

            stat_cols

        ].to_numpy(float)

    for forest in forest_order

}

boot_rows = []

for b in tqdm(
    range(BOOT_N),
    desc='Integrated subplot bootstrap'
):

    factors_b = {}

    biomass_b = {}

    for forest in forest_order:

        arr = subplot_arrays[forest]

        indices = rng.integers(
            0, len(arr),
            size=len(arr)
        )

        totals = arr[indices].sum(
            axis=0
        )

        factors_b[forest] = (
            factors_from_totals(totals)
        )

        biomass_b[forest] = (

            CHAVE_COEF
            * totals[3]
            / 1000

        )

    bs = biomass_b['Late_secondary']

    bm = biomass_b['Mature']

    gap = bs - bm

    shp = shapley_fast(

        factors_b['Mature'],

        factors_b['Late_secondary']

    )

    assert np.isclose(
        shp.sum(), gap
    )

    row = {

        'iteration': b,

        'secondary_AGB_Mg': bs,

        'mature_AGB_Mg': bm,

        'biomass_gap_Mg': gap,

        'biomass_ratio': bs / bm

    }

    for component in factor_names:

        row[f'{component}_Mg'] = (
            shp[component]
        )

    boot_rows.append(row)

integrated_boot = pd.DataFrame(
    boot_rows
)

# -----------------------------------------------------
# 4. Conditional bootstrap summary
# -----------------------------------------------------

def summarize_draws(df, columns):

    rows = []

    for col in columns:

        x = df[col].dropna().to_numpy()

        rows.append({

            'metric': col,

            'mean': x.mean(),

            'median': np.median(x),

            'LCL_95': np.quantile(x, 0.025),

            'UCL_95': np.quantile(x, 0.975),

            'SD': x.std(ddof=1),

            'positive_draws_pct':

                100 * np.mean(x > 0)

        })

    return pd.DataFrame(rows)


bootstrap_metrics = [

    'secondary_AGB_Mg',

    'mature_AGB_Mg',

    'biomass_gap_Mg',

    'biomass_ratio'

] + [

    f'{c}_Mg'
    for c in factor_names

]

integrated_boot_summary = summarize_draws(

    integrated_boot,

    bootstrap_metrics

)

print('\nCONDITIONAL SUBPLOT BOOTSTRAP')

display(
    integrated_boot_summary.round(4)
)

print(

    '\nSecondary biomass exceeds mature in',

    round(

        100 * (
            integrated_boot[
                'biomass_gap_Mg'
            ] > 0
        ).mean(),

        2

    ),

    '% of bootstrap draws'

)

# -----------------------------------------------------
# 5. Export
# -----------------------------------------------------

integrated_boot.to_csv(

    functional_dir /

    'BIOMASS_INTEGRATED_SUBPLOT_BOOTSTRAP.csv',

    index=False

)

integrated_boot_summary.to_csv(

    functional_dir /

    'BIOMASS_INTEGRATED_BOOTSTRAP_SUMMARY.csv',

    index=False

)

print('\nCELL 69 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 70: BIOMETRIC MONTE CARLO SENSITIVITY
# =====================================================

MC_N = 2000

rng_mc = np.random.default_rng(
    20261116
)

mc_data = functional.copy().reset_index(
    drop=True
)

error_scenarios = {

    'Low': {

        'dbh_cv': 0.01,

        'height_cv': 0.05,

        'wd_cv': 0.05

    },

    'Moderate': {

        'dbh_cv': 0.02,

        'height_cv': 0.10,

        'wd_cv': 0.10

    },

    'High': {

        'dbh_cv': 0.05,

        'height_cv': 0.20,

        'wd_cv': 0.20

    }

}

# -----------------------------------------------------
# 1. Wood density error groups
# -----------------------------------------------------

# Prefer an explicit assignment identifier if available.

if 'wd_assignment_id' in mc_data.columns:

    wd_group = (

        mc_data['wd_assignment_id']
        .astype(str)

    )

else:

    taxon = (

        mc_data['operational_taxon']

        .fillna('Unresolved')

        .astype(str)

    )

    genus = (

        mc_data['species_accepted']

        .fillna(taxon)

        .astype(str)

        .str.split()

        .str[0]

    )

    if 'family_accepted' in mc_data.columns:

        family = (

            mc_data['family_accepted']

            .fillna('Unknown_family')

            .astype(str)

        )

    else:

        family = pd.Series(

            'Unknown_family',

            index=mc_data.index

        )

    wd_group = taxon.copy()

    genus_mask = mc_data[
        'wd_level'
    ].eq('genus')

    family_mask = mc_data[
        'wd_level'
    ].eq('family')

    pooled_mask = mc_data[
        'wd_level'
    ].eq('community_pooled')

    wd_group.loc[genus_mask] = (

        'GENUS_'

        + genus.loc[genus_mask]

    )

    wd_group.loc[family_mask] = (

        'FAMILY_'

        + family.loc[family_mask]

    )

    wd_group.loc[pooled_mask] = (

        'COMMUNITY_POOLED'

    )

    # Preserve distinct assigned values

    wd_group = (

        wd_group

        + '_'

        + mc_data['wd']
        .round(6)
        .astype(str)

    )

wd_codes, wd_groups = pd.factorize(
    wd_group
)

print(

    'Wood density error groups:',

    len(wd_groups)

)

# -----------------------------------------------------
# 2. Mean-preserving lognormal perturbation
# -----------------------------------------------------

def lognormal_multiplier(
    cv, size, generator
):

    sigma = np.sqrt(
        np.log1p(cv**2)
    )

    return generator.lognormal(

        mean=-0.5*sigma**2,

        sigma=sigma,

        size=size

    )


d0 = mc_data[
    'dbh_cm'
].to_numpy(float)

h0 = mc_data[
    'height_m'
].to_numpy(float)

rho0 = mc_data[
    'wd'
].to_numpy(float)

n_total = len(mc_data)

forest_masks = {

    forest:

        mc_data[
            'forest_type'
        ].eq(forest).to_numpy()

    for forest in forest_order

}

# -----------------------------------------------------
# 3. Monte Carlo simulation
# -----------------------------------------------------

mc_rows = []

for scenario, params in error_scenarios.items():

    for b in tqdm(

        range(MC_N),

        desc=f'Monte Carlo: {scenario}'

    ):

        d = d0 * lognormal_multiplier(

            params['dbh_cv'],

            n_total,

            rng_mc

        )

        h = h0 * lognormal_multiplier(

            params['height_cv'],

            n_total,

            rng_mc

        )

        wd_errors = lognormal_multiplier(

            params['wd_cv'],

            len(wd_groups),

            rng_mc

        )

        rho = rho0 * wd_errors[
            wd_codes
        ]

        dt = d ** EXP_D

        dht = dt * h ** EXP_H

        dhwdt = (

            dht * rho ** EXP_WD

        )

        factors_mc = {}

        biomass_mc = {}

        for forest in forest_order:

            mask = forest_masks[
                forest
            ]

            totals = np.array([

                mask.sum(),

                dt[mask].sum(),

                dht[mask].sum(),

                dhwdt[mask].sum()

            ])

            factors_mc[forest] = (

                factors_from_totals(
                    totals
                )

            )

            biomass_mc[forest] = (

                CHAVE_COEF

                * totals[3]

                / 1000

            )

        bs = biomass_mc[
            'Late_secondary'
        ]

        bm = biomass_mc[
            'Mature'
        ]

        gap = bs - bm

        shp = shapley_fast(

            factors_mc['Mature'],

            factors_mc['Late_secondary']

        )

        assert np.isclose(
            shp.sum(), gap
        )

        row = {

            'scenario': scenario,

            'iteration': b,

            'secondary_AGB_Mg': bs,

            'mature_AGB_Mg': bm,

            'biomass_gap_Mg': gap,

            'biomass_ratio': bs / bm

        }

        for component in factor_names:

            row[f'{component}_Mg'] = (
                shp[component]
            )

        mc_rows.append(row)

biometric_mc = pd.DataFrame(
    mc_rows
)

# -----------------------------------------------------
# 4. Summarize uncertainty scenarios
# -----------------------------------------------------

mc_summary_rows = []

for scenario, group in biometric_mc.groupby(
    'scenario'
):

    summary_mc = summarize_draws(

        group,

        bootstrap_metrics

    )

    summary_mc.insert(

        0,

        'scenario',

        scenario

    )

    mc_summary_rows.append(
        summary_mc
    )

biometric_mc_summary = pd.concat(

    mc_summary_rows,

    ignore_index=True

)

print('\nBIOMETRIC UNCERTAINTY SUMMARY')

display(

    biometric_mc_summary.round(4)

)

print('\nBIOMASS GAP ROBUSTNESS')

display(

    biometric_mc_summary.loc[

        biometric_mc_summary[
            'metric'
        ].eq('biomass_gap_Mg')

    ].round(4)

)

# -----------------------------------------------------
# 5. Export
# -----------------------------------------------------

biometric_mc.to_csv(

    functional_dir /

    'BIOMASS_BIOMETRIC_MONTE_CARLO.csv',

    index=False

)

biometric_mc_summary.to_csv(

    functional_dir /

    'BIOMASS_BIOMETRIC_MC_SUMMARY.csv',

    index=False

)

print('\nCELL 70 COMPLETED SUCCESSFULLY')


In [ ]:

# =====================================================
# CELL 71: SYSTEMATIC HEIGHT BIAS SENSITIVITY
# =====================================================

bs0 = B_secondary

bm0 = B_mature

height_bias_values = np.arange(

    -0.30,

    0.301,

    0.025

)

bias_rows = []

for bias_s in height_bias_values:

    for bias_m in height_bias_values:

        bs = (

            bs0

            * (1+bias_s)**EXP_H

        )

        bm = (

            bm0

            * (1+bias_m)**EXP_H

        )

        bias_rows.append({

            'secondary_height_bias_pct':

                100*bias_s,

            'mature_height_bias_pct':

                100*bias_m,

            'secondary_AGB_Mg': bs,

            'mature_AGB_Mg': bm,

            'biomass_gap_Mg': bs-bm,

            'biomass_ratio': bs/bm

        })

height_bias_table = pd.DataFrame(
    bias_rows
)

# -----------------------------------------------------
# 1. Exact thresholds for equal biomass
# -----------------------------------------------------

secondary_threshold = (

    (bm0/bs0)**(1/EXP_H)

    - 1

)

mature_threshold = (

    (bs0/bm0)**(1/EXP_H)

    - 1

)

# Symmetric opposite biases:
# secondary decreases, mature increases

r = (bm0/bs0)**(1/EXP_H)

symmetric_threshold = (

    (1-r)/(1+r)

)

threshold_table = pd.DataFrame({

    'scenario': [

        'Secondary height reduction only',

        'Mature height increase only',

        'Opposing symmetric height bias'

    ],

    'threshold_pct': [

        100*secondary_threshold,

        100*mature_threshold,

        100*symmetric_threshold

    ]

})

print('HEIGHT BIAS THRESHOLDS')

display(
    threshold_table.round(4)
)

# -----------------------------------------------------
# 2. Plot systematic sensitivity
# -----------------------------------------------------

bias_matrix = height_bias_table.pivot(

    index='mature_height_bias_pct',

    columns='secondary_height_bias_pct',

    values='biomass_gap_Mg'

)

fig, ax = plt.subplots(

    figsize=(8, 6.5)

)

X, Y = np.meshgrid(

    bias_matrix.columns,

    bias_matrix.index

)

Z = bias_matrix.to_numpy()

max_abs = np.abs(Z).max()

contour = ax.contourf(

    X, Y, Z,

    levels=np.linspace(
        -max_abs,
        max_abs,
        25
    ),

    cmap='RdBu_r',

    extend='both'

)

zero_line = ax.contour(

    X, Y, Z,

    levels=[0],

    colors='black',

    linewidths=2

)

ax.clabel(

    zero_line,

    fmt={0: 'Equal biomass'},

    fontsize=10

)

ax.scatter(

    0, 0,

    color='black',

    s=60,

    zorder=5,

    label='Original inventory'

)

ax.axhline(

    0,

    color='gray',

    linestyle='--',

    linewidth=0.7

)

ax.axvline(

    0,

    color='gray',

    linestyle='--',

    linewidth=0.7

)

ax.set_xlabel(

    'Secondary height adjustment (%)'

)

ax.set_ylabel(

    'Mature height adjustment (%)'

)

ax.set_title(

    'Sensitivity to systematic height bias'

)

fig.colorbar(

    contour,

    ax=ax,

    label='Secondary − mature biomass (Mg)'

)

ax.legend(frameon=False)

fig.tight_layout()

for ext in ['png', 'pdf', 'svg']:

    fig.savefig(

        figure_dir /

        f'FIG_SYSTEMATIC_HEIGHT_BIAS.{ext}',

        dpi=600,

        bbox_inches='tight'

    )

plt.show()

# -----------------------------------------------------
# 3. Export
# -----------------------------------------------------

height_bias_table.to_csv(

    functional_dir /

    'BIOMASS_SYSTEMATIC_HEIGHT_BIAS.csv',

    index=False

)

threshold_table.to_csv(

    functional_dir /

    'BIOMASS_HEIGHT_BIAS_THRESHOLDS.csv',

    index=False

)

print('\nCELL 71 COMPLETED SUCCESSFULLY')


In [ ]:
# ============================================================
# CELL 72 — SAFE SCIENTIFIC CHECKPOINT
# ============================================================

!pip -q install dill

import dill
import pandas as pd
import numpy as np
import json
import types
import sys
import hashlib

from pathlib import Path
from datetime import datetime
from importlib.metadata import version, PackageNotFoundError

CHECKPOINT_DIR = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT/"
    "06_CHECKPOINTS"
)

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

OBJECT_DIR = CHECKPOINT_DIR / "python_objects"
OBJECT_DIR.mkdir(exist_ok=True)

TABLE_DIR = CHECKPOINT_DIR / "dataframe_backup"
TABLE_DIR.mkdir(exist_ok=True)

# Capturar variables antes de iniciar el guardado
namespace_snapshot = list(globals().items())

saved_objects = []
failed_objects = []
skipped_objects = []

# Tipos científicos que interesa conservar
allowed_types = (
    pd.DataFrame,
    pd.Series,
    np.ndarray,
    np.generic,
    dict,
    list,
    tuple,
    set,
    str,
    int,
    float,
    bool,
    Path
)

# Excluir objetos internos de Colab/IPython
excluded_names = {
    "In", "Out", "exit", "quit",
    "get_ipython", "open",
    "namespace_snapshot",
    "saved_objects",
    "failed_objects",
    "skipped_objects",
    "allowed_types",
    "excluded_names",
    "ColabKernelApp"
}

print("SAVING SCIENTIFIC OBJECTS")
print("=" * 65)

for name, obj in namespace_snapshot:

    if name.startswith("_") or name in excluded_names:
        continue

    if isinstance(obj, types.ModuleType):
        continue

    if not isinstance(obj, allowed_types):
        skipped_objects.append({
            "name": name,
            "type": type(obj).__name__
        })
        continue

    output_path = OBJECT_DIR / f"{name}.pkl"
    temp_path = OBJECT_DIR / f"{name}.tmp"

    try:
        # Guardar cada objeto independientemente
        with open(temp_path, "wb") as f:
            dill.dump(obj, f, recurse=False)

        # Reemplazo solo después de guardado exitoso
        temp_path.replace(output_path)

        # Verificar integridad mediante SHA256
        checksum = hashlib.sha256(
            output_path.read_bytes()
        ).hexdigest()

        record = {
            "name": name,
            "type": type(obj).__name__,
            "file": output_path.name,
            "size_MB": round(
                output_path.stat().st_size / 1024**2, 4
            ),
            "sha256": checksum
        }

        # Respaldo adicional de DataFrames en CSV
        if isinstance(obj, pd.DataFrame):

            record["rows"] = obj.shape[0]
            record["columns"] = obj.shape[1]

            try:
                obj.to_csv(
                    TABLE_DIR / f"{name}.csv",
                    index=True,
                    encoding="utf-8-sig"
                )
                record["csv_backup"] = True

            except Exception:
                record["csv_backup"] = False

        saved_objects.append(record)

    except Exception as e:

        if temp_path.exists():
            temp_path.unlink()

        failed_objects.append({
            "name": name,
            "type": type(obj).__name__,
            "error": str(e)
        })

# Registrar versiones de dependencias
packages = [
    "dill",
    "numpy",
    "pandas",
    "scipy",
    "matplotlib",
    "scikit-learn",
    "statsmodels",
    "openpyxl",
    "requests",
    "tqdm"
]

package_versions = {}

for package in packages:
    try:
        package_versions[package] = version(package)
    except PackageNotFoundError:
        pass

# Crear manifiesto
metadata = {
    "project": "SATIPO_FOREST_PROJECT",
    "last_analysis_cell": 71,
    "saved_at": datetime.now().isoformat(),
    "python_version": sys.version,
    "package_versions": package_versions,
    "saved_objects": saved_objects,
    "failed_objects": failed_objects,
    "skipped_objects": skipped_objects
}

with open(
    CHECKPOINT_DIR / "CHECKPOINT_MANIFEST.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata,
        f,
        indent=2,
        ensure_ascii=False
    )

# Resumen
checkpoint_summary = pd.DataFrame(saved_objects)

print("\nCHECKPOINT SUMMARY")
print("=" * 65)

print("Saved objects:", len(saved_objects))
print("Failed objects:", len(failed_objects))
print("Skipped objects:", len(skipped_objects))

print(
    "Total checkpoint size (MB):",
    round(
        sum(x["size_MB"] for x in saved_objects),
        2
    )
)

print("\nCHECKPOINT LOCATION:")
print(CHECKPOINT_DIR)

if not checkpoint_summary.empty:
    display(checkpoint_summary)

if failed_objects:
    print("\nFAILED OBJECTS:")
    display(pd.DataFrame(failed_objects))

print("\nCELL 72 COMPLETED")

In [ ]:
# ============================================================
# CELL 73 — COMPLETE PROJECT OUTPUT INVENTORY
# ============================================================

from pathlib import Path
import pandas as pd

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

assert ROOT.exists(), "Project folder not found"

inventory_records = []

valid_extensions = {
    ".csv",
    ".xlsx",
    ".parquet",
    ".pkl",
    ".json",
    ".png",
    ".pdf",
    ".svg",
    ".tif",
    ".tiff",
    ".rds",
    ".rdata",
    ".r",
    ".py",
    ".ipynb"
}

for path in ROOT.rglob("*"):

    if not path.is_file():
        continue

    if path.suffix.lower() not in valid_extensions:
        continue

    inventory_records.append({
        "filename": path.name,
        "extension": path.suffix.lower(),
        "relative_path": str(
            path.relative_to(ROOT)
        ),
        "size_MB": round(
            path.stat().st_size / 1024**2, 4
        ),
        "folder": path.parent.name
    })

project_inventory = pd.DataFrame(
    inventory_records
)

print("PROJECT FILE INVENTORY")
print("=" * 65)

if not project_inventory.empty:

    display(
        project_inventory.groupby("extension")
        .agg(
            n_files=("filename", "count"),
            total_MB=("size_MB", "sum")
        )
        .reset_index()
    )

    print("\nSCIENTIFIC FIGURES")

    figures_inventory = project_inventory[
        project_inventory["extension"].isin(
            [".png", ".pdf", ".svg", ".tif", ".tiff"]
        )
    ].copy()

    display(figures_inventory)

    print("\nSCIENTIFIC TABLES")

    tables_inventory = project_inventory[
        project_inventory["extension"].isin(
            [".csv", ".xlsx", ".parquet"]
        )
    ].copy()

    display(tables_inventory)

    project_inventory.to_csv(
        ROOT / "06_CHECKPOINTS" /
        "PROJECT_FILE_INVENTORY.csv",
        index=False
    )

print("\nCELL 73 COMPLETED")

In [ ]:
# ============================================================
# CELL 74 — MANUSCRIPT-READY FIGURE PACKAGE
# ============================================================

!pip -q install pymupdf pillow

from pathlib import Path
import shutil
import pandas as pd
import fitz
from PIL import Image

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

MANUSCRIPT_DIR = ROOT / "07_MANUSCRIPT_READY"

EDITORIAL_DIRS = {
    "main_figures": MANUSCRIPT_DIR / "MAIN_FIGURES",
    "main_tables": MANUSCRIPT_DIR / "MAIN_TABLES",
    "supp_figures": MANUSCRIPT_DIR / "SUPPLEMENTARY_FIGURES",
    "supp_tables": MANUSCRIPT_DIR / "SUPPLEMENTARY_TABLES",
    "captions": MANUSCRIPT_DIR / "FIGURE_CAPTIONS",
    "results": MANUSCRIPT_DIR / "RESULTS_SUMMARY",
    "metadata": MANUSCRIPT_DIR / "METADATA"
}

for directory in EDITORIAL_DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Editorial figure selection
# ------------------------------------------------------------

FIGURE_PLAN = [

    {
        "figure": "Fig_02",
        "source": "FIG_TAXONOMIC_DIVERSITY",
        "title": "Taxonomic diversity",
        "destination": "main_figures"
    },

    {
        "figure": "Fig_03",
        "source": "Fig_Biomass_Diameter_Classes",
        "title": "Biomass distribution across diameter classes",
        "destination": "main_figures"
    },

    {
        "figure": "Fig_04",
        "source": "FIG_HEIGHT_DIAMETER_ALLOMETRY",
        "title": "Height-diameter allometry",
        "destination": "main_figures"
    },

    {
        "figure": "Fig_05",
        "source": "FIG_ARCHITECTURE_DECOMPOSITION",
        "title": "Counterfactual architecture decomposition",
        "destination": "main_figures"
    },

    {
        "figure": "Fig_06",
        "source": "FIG_TAXONOMIC_BIOMASS_DECOUPLING",
        "title": "Taxonomic abundance-biomass decoupling",
        "destination": "main_figures"
    },

    {
        "figure": "Fig_07",
        "source": "FIG_INTEGRATED_BIOMASS_DECOMPOSITION",
        "title": "Integrated biomass decomposition",
        "destination": "main_figures"
    },

    {
        "figure": "Fig_S01",
        "source": "FIG_INEXT_COVERAGE",
        "title": "Coverage-based diversity assessment",
        "destination": "supp_figures"
    },

    {
        "figure": "Fig_S02",
        "source": "FIG_BETA_DIVERSITY",
        "title": "Taxonomic beta diversity",
        "destination": "supp_figures"
    },

    {
        "figure": "Fig_S03",
        "source": "FIG_ALLOMETRY_BOOTSTRAP",
        "title": "Allometric bootstrap uncertainty",
        "destination": "supp_figures"
    },

    {
        "figure": "Fig_S04",
        "source": "FIG_WOOD_DENSITY_DISTRIBUTION",
        "title": "Weighted wood density distributions",
        "destination": "supp_figures"
    },

    {
        "figure": "Fig_S05",
        "source": "FIG_SYSTEMATIC_HEIGHT_BIAS",
        "title": "Sensitivity to systematic height bias",
        "destination": "supp_figures"
    }
]

# ------------------------------------------------------------
# 2. Locate original figures
# ------------------------------------------------------------

SOURCE_ROOT = ROOT / "05_OUTPUTS"

source_files = list(SOURCE_ROOT.rglob("*"))

figure_records = []
figure_errors = []

# ------------------------------------------------------------
# 3. Copy original files and export high-resolution TIFF
# ------------------------------------------------------------

for item in FIGURE_PLAN:

    figure_id = item["figure"]
    source_stem = item["source"]

    destination = EDITORIAL_DIRS[item["destination"]]

    print(f"\nProcessing {figure_id}: {source_stem}")

    record = {
        "figure_id": figure_id,
        "title": item["title"],
        "original_name": source_stem,
        "destination": item["destination"],
        "pdf_available": False,
        "svg_available": False,
        "png_available": False,
        "tiff_available": False
    }

    for extension in [".pdf", ".svg", ".png"]:

        matches = [
            p for p in source_files
            if p.is_file()
            and p.stem == source_stem
            and p.suffix.lower() == extension
        ]

        if len(matches) == 0:
            continue

        if len(matches) > 1:

            figure_errors.append({
                "figure": figure_id,
                "error": f"Multiple sources for {extension}"
            })

            continue

        original = matches[0]

        output = destination / (
            f"{figure_id}_{source_stem}{extension}"
        )

        shutil.copy2(original, output)

        record[
            f"{extension[1:]}_available"
        ] = True

        print("  Saved:", output.name)

    # --------------------------------------------------------
    # TIFF directly from vector PDF
    # --------------------------------------------------------

    pdf_path = destination / (
        f"{figure_id}_{source_stem}.pdf"
    )

    tiff_path = destination / (
        f"{figure_id}_{source_stem}.tiff"
    )

    if pdf_path.exists():

        try:

            with fitz.open(pdf_path) as pdf:

                if len(pdf) != 1:
                    raise ValueError(
                        "Expected a single-page figure PDF"
                    )

                page = pdf[0]

                pixmap = page.get_pixmap(
                    dpi=600,
                    colorspace=fitz.csRGB,
                    alpha=False
                )

                image = Image.frombytes(
                    "RGB",
                    [pixmap.width, pixmap.height],
                    pixmap.samples
                )

                image.save(
                    tiff_path,
                    format="TIFF",
                    compression="tiff_lzw",
                    dpi=(600, 600)
                )

                record["tiff_available"] = True
                record["width_px"] = pixmap.width
                record["height_px"] = pixmap.height
                record["resolution_dpi"] = 600

                record["width_cm"] = round(
                    pixmap.width / 600 * 2.54, 2
                )

                record["height_cm"] = round(
                    pixmap.height / 600 * 2.54, 2
                )

                record["tiff_size_MB"] = round(
                    tiff_path.stat().st_size / 1024**2,
                    3
                )

                print(
                    "  TIFF:",
                    pixmap.width,
                    "x",
                    pixmap.height,
                    "px"
                )

        except Exception as e:

            figure_errors.append({
                "figure": figure_id,
                "error": str(e)
            })

    else:

        figure_errors.append({
            "figure": figure_id,
            "error": "Vector PDF unavailable"
        })

    figure_records.append(record)

# ------------------------------------------------------------
# 4. Export editorial manifest
# ------------------------------------------------------------

figure_manifest = pd.DataFrame(figure_records)

figure_manifest.to_csv(
    EDITORIAL_DIRS["metadata"] /
    "MANUSCRIPT_FIGURE_MANIFEST.csv",
    index=False
)

print("\nEDITORIAL FIGURE MANIFEST")
display(figure_manifest)

print("\nFIGURE EXPORT SUMMARY")

print(
    "Main figures:",
    sum(
        x["destination"] == "main_figures"
        for x in FIGURE_PLAN
    )
)

print(
    "Supplementary figures:",
    sum(
        x["destination"] == "supp_figures"
        for x in FIGURE_PLAN
    )
)

print(
    "TIFF files generated:",
    figure_manifest["tiff_available"].sum()
)

if figure_errors:

    print("\nFIGURE EXPORT WARNINGS")
    display(pd.DataFrame(figure_errors))

print("\nCELL 74 COMPLETED SUCCESSFULLY")

In [ ]:
# ============================================================
# CELL 75 — SCIENTIFIC TABLE AUDIT
# ============================================================

import pandas as pd
from pathlib import Path

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

MANUSCRIPT_DIR = ROOT / "07_MANUSCRIPT_READY"

TABLE_AUDIT_DIR = MANUSCRIPT_DIR / "METADATA"
TABLE_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# Only scientific analysis folders
SEARCH_FOLDERS = [
    ROOT / "02_DATA_MASTER",
    ROOT / "03_ANALYSIS",
    ROOT / "05_OUTPUTS"
]

table_records = []

def classify_scientific_table(filename):

    name = filename.lower()

    categories = [

        ("uncertainty", [
            "bootstrap", "monte_carlo",
            "uncertainty", "bias"
        ]),

        ("biomass_decomposition", [
            "shapley", "lmdi", "factorization",
            "decomposition", "factor_contrast"
        ]),

        ("architecture", [
            "counterfactual", "architecture",
            "common_support"
        ]),

        ("allometry", [
            "allometr", "weibull",
            "michaelis", "cross_validation",
            "height_diameter"
        ]),

        ("taxonomic_diversity", [
            "diversity", "hill", "inext",
            "rarefaction", "coverage", "richness"
        ]),

        ("wood_density", [
            "wood_density", "wooddensity",
            "functional", "gwdd"
        ]),

        ("species_contribution", [
            "dominant", "taxon_contribution",
            "taxonomic_biomass",
            "species_contribution",
            "concentration"
        ]),

        ("forest_structure", [
            "structure", "diameter",
            "basal_area", "subplot"
        ]),

        ("biomass", [
            "biomass", "agb", "carbon"
        ]),

        ("taxonomy", [
            "taxonomy", "taxonomic",
            "gbif", "accepted"
        ]),

        ("quality_control", [
            "qa", "qc", "audit",
            "validation", "manifest"
        ]),

        ("master_data", [
            "master", "inventory"
        ])
    ]

    for category, keywords in categories:

        if any(k in name for k in keywords):
            return category

    return "other"


# ------------------------------------------------------------
# Inspect CSV and Excel files
# ------------------------------------------------------------

for folder in SEARCH_FOLDERS:

    if not folder.exists():
        continue

    for path in sorted(folder.rglob("*")):

        if not path.is_file():
            continue

        if path.suffix.lower() not in [
            ".csv", ".xlsx", ".parquet"
        ]:
            continue

        record = {
            "filename": path.name,
            "relative_path": str(
                path.relative_to(ROOT)
            ),
            "category": classify_scientific_table(
                path.stem
            ),
            "size_MB": round(
                path.stat().st_size / 1024**2,
                4
            )
        }

        try:

            if path.suffix.lower() == ".csv":

                table = pd.read_csv(
                    path,
                    low_memory=False
                )

            elif path.suffix.lower() == ".parquet":

                table = pd.read_parquet(path)

            else:

                excel = pd.ExcelFile(path)

                record["sheets"] = " | ".join(
                    excel.sheet_names
                )

                table = pd.read_excel(
                    path,
                    sheet_name=0
                )

            record["n_rows"] = len(table)
            record["n_columns"] = len(table.columns)

            record["columns"] = " | ".join(
                map(str, table.columns)
            )

            record["read_status"] = "OK"

        except Exception as e:

            record["read_status"] = "ERROR"
            record["error"] = str(e)

        table_records.append(record)

# ------------------------------------------------------------
# Build table catalogue
# ------------------------------------------------------------

scientific_table_catalogue = pd.DataFrame(
    table_records
)

scientific_table_catalogue = (
    scientific_table_catalogue
    .sort_values(
        ["category", "filename"]
    )
    .reset_index(drop=True)
)

scientific_table_catalogue.to_csv(
    TABLE_AUDIT_DIR /
    "SCIENTIFIC_TABLE_CATALOGUE.csv",
    index=False
)

scientific_table_catalogue.to_excel(
    TABLE_AUDIT_DIR /
    "SCIENTIFIC_TABLE_CATALOGUE.xlsx",
    index=False
)

print("SCIENTIFIC TABLES BY CATEGORY")

display(
    scientific_table_catalogue
    .groupby("category")
    .size()
    .rename("n_tables")
    .reset_index()
)

# ------------------------------------------------------------
# Print complete paths, avoiding notebook truncation
# ------------------------------------------------------------

print("\nCOMPLETE SCIENTIFIC TABLE CATALOGUE")
print("=" * 80)

for category, group in (
    scientific_table_catalogue.groupby("category")
):

    print(f"\n[{category.upper()}]")

    for _, row in group.iterrows():

        print(
            f"{row['filename']} | "
            f"{row.get('n_rows', '?')} rows | "
            f"{row.get('n_columns', '?')} columns"
        )

        print(
            f"    {row['relative_path']}"
        )

# ------------------------------------------------------------
# Identify final manuscript candidates
# ------------------------------------------------------------

editorial_candidates = (
    scientific_table_catalogue.loc[
        (
            scientific_table_catalogue[
                "read_status"
            ] == "OK"
        )
        &
        (
            scientific_table_catalogue[
                "n_rows"
            ] <= 100
        )
        &
        (
            scientific_table_catalogue[
                "n_columns"
            ] >= 3
        )
    ]
    .copy()
)

print("\nPOTENTIAL MANUSCRIPT TABLES")

with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", 120
):

    display(
        editorial_candidates[
            [
                "filename",
                "category",
                "n_rows",
                "n_columns"
            ]
        ]
    )

editorial_candidates.to_csv(
    TABLE_AUDIT_DIR /
    "EDITORIAL_TABLE_CANDIDATES.csv",
    index=False
)

print("\nCELL 75 COMPLETED SUCCESSFULLY")

In [ ]:
# ============================================================
# CELL 76 — MANUSCRIPT-READY SCIENTIFIC TABLES
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import shutil

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

ANALYSIS = ROOT / "03_ANALYSIS"
MASTER = ROOT / "02_DATA_MASTER"

MANUSCRIPT = ROOT / "07_MANUSCRIPT_READY"

MAIN_TABLES = MANUSCRIPT / "MAIN_TABLES"
SUPP_TABLES = MANUSCRIPT / "SUPPLEMENTARY_TABLES"
METADATA = MANUSCRIPT / "METADATA"
RESULTS = MANUSCRIPT / "RESULTS_SUMMARY"

for folder in [
    MAIN_TABLES, SUPP_TABLES, METADATA, RESULTS
]:
    folder.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Safe loading
# ------------------------------------------------------------

def load_table(relative_path):

    path = ROOT / relative_path

    if not path.exists():
        raise FileNotFoundError(path)

    df = pd.read_csv(path, low_memory=False)

    print(
        f"Loaded: {path.name} | "
        f"{df.shape[0]} rows × {df.shape[1]} columns"
    )

    return df


main_tables = {}
supp_tables = {}
table_metadata = []

def register_table(
    table_id,
    title,
    df,
    source,
    destination="main",
    notes=""
):

    if df.empty:
        raise ValueError(
            f"{table_id} is empty"
        )

    df = df.copy()

    target = (
        MAIN_TABLES
        if destination == "main"
        else SUPP_TABLES
    )

    filename = f"{table_id}.csv"

    df.to_csv(
        target / filename,
        index=False,
        encoding="utf-8-sig"
    )

    if destination == "main":
        main_tables[table_id] = df
    else:
        supp_tables[table_id] = df

    table_metadata.append({
        "table_id": table_id,
        "title": title,
        "destination": destination,
        "source": source,
        "n_rows": len(df),
        "n_columns": len(df.columns),
        "notes": notes,
        "file": str(
            (target / filename).relative_to(ROOT)
        )
    })

    print(f"Registered: {table_id}")

    return df


FE = "03_ANALYSIS/FUNCTIONAL_ECOLOGY"
AL = "03_ANALYSIS/ALLOMETRY"
TX = "03_ANALYSIS/TAXONOMIC_DIVERSITY"
IX = "03_ANALYSIS/INEXT_DIVERSITY"
BT = "03_ANALYSIS/BETA_DIVERSITY"
BD = "03_ANALYSIS/BIOMASS_DECOMPOSITION"

# ============================================================
# TABLE 1 — FOREST STRUCTURE AND BIOMASS
# ============================================================

source_t1 = f"{FE}/DOMINANT_SPECIES_EXCLUSION.csv"

full = load_table(source_t1)

full = full.loc[
    full["scenario"].eq("Full_inventory")
].copy()

assert len(full) == 2
assert full["forest_type"].is_unique

full = full.set_index("forest_type")

assert int(
    full.loc["Late_secondary", "n_trees"]
) == 541

assert int(
    full.loc["Mature", "n_trees"]
) == 461

metrics_t1 = [

    (
        "Number of analyzed stems",
        "n_trees", "stems", 0
    ),

    (
        "Stem density",
        "n_trees", "stems ha-1", 0
    ),

    (
        "Operational taxon richness",
        "richness", "taxa", 0
    ),

    (
        "Shannon diversity (q = 1)",
        "abundance_q1", "effective taxa", 2
    ),

    (
        "Simpson diversity (q = 2)",
        "abundance_q2", "effective taxa", 2
    ),

    (
        "Mean diameter at breast height",
        "mean_dbh_cm", "cm", 2
    ),

    (
        "Mean tree height",
        "mean_height_m", "m", 2
    ),

    (
        "Basal area",
        "basal_area_m2", "m2 ha-1", 2
    ),

    (
        "Aboveground biomass",
        "AGB_Mg", "Mg ha-1", 2
    ),

    (
        "Abundance-weighted wood density",
        "CWM_WD_abundance", "g cm-3", 3
    ),

    (
        "Basal-area-weighted wood density",
        "CWM_WD_basal_area", "g cm-3", 3
    ),

    (
        "Biomass-weighted wood density",
        "CWM_WD_biomass", "g cm-3", 3
    ),

    (
        "Biomass-weighted taxonomic diversity (q = 2)",
        "biomass_q2", "effective taxa", 2
    ),

    (
        "Dominant taxon relative abundance",
        "dominant_abundance_pct", "%", 2
    ),

    (
        "Maximum taxon relative biomass",
        "dominant_biomass_pct", "%", 2
    )
]

rows = [{
    "Variable": "Plot area",
    "Unit": "ha",
    "Late-secondary forest": "1.00",
    "Mature forest": "1.00"
}]

for label, column, unit, decimals in metrics_t1:

    secondary = full.loc[
        "Late_secondary", column
    ]

    mature = full.loc[
        "Mature", column
    ]

    rows.append({
        "Variable": label,
        "Unit": unit,
        "Late-secondary forest": (
            f"{secondary:.{decimals}f}"
        ),
        "Mature forest": (
            f"{mature:.{decimals}f}"
        )
    })

table1 = pd.DataFrame(rows)

register_table(
    "Table_01_Forest_Characteristics",
    "Structural, taxonomic and biomass characteristics",
    table1,
    source_t1,
    notes=(
        "Analytical biomass inventory: 541 and 461 stems. "
        "One 1-ha plot per forest condition. "
        "Richness refers to operational taxa. "
        "Maximum taxon biomass and maximum taxon "
        "abundance need not refer to the same taxon."
    )
)

# ============================================================
# TABLE 2 — COVERAGE-STANDARDIZED DIVERSITY
# ============================================================

source_t2 = f"{IX}/INEXT_STANDARDIZED_DIVERSITY.csv"

table2 = load_table(source_t2)

register_table(
    "Table_02_Coverage_Standardized_Diversity",
    "Coverage-standardized Hill diversity",
    table2,
    source_t2,
    notes=(
        "Retain original coverage levels, Hill orders, "
        "taxonomic scenarios and uncertainty estimates. "
        "Final editorial selection requires inspection "
        "of the taxonomic scenario labels."
    )
)

# ============================================================
# TABLE 3 — HEIGHT-DIAMETER MODEL COMPARISON
# ============================================================

source_t3a = f"{AL}/ALLOMETRY_MODEL_COMPARISON.csv"
source_t3b = f"{AL}/ALLOMETRY_CV_SUMMARY.csv"

model_comparison = load_table(source_t3a)
cv_summary = load_table(source_t3b)

# Identify shared model identifiers without
# assuming an undocumented column schema.

common_columns = [
    c for c in model_comparison.columns
    if c in cv_summary.columns
]

candidate_keys = [
    c for c in common_columns
    if (
        "model" in c.lower()
        or "forest" in c.lower()
        or c.lower() in [
            "scenario", "group", "form"
        ]
    )
]

print("\nAllometry merge candidates:")
print(candidate_keys)

can_merge = (
    len(candidate_keys) > 0
    and not model_comparison.duplicated(
        candidate_keys
    ).any()
    and not cv_summary.duplicated(
        candidate_keys
    ).any()
)

if can_merge:

    table3 = model_comparison.merge(
        cv_summary,
        on=candidate_keys,
        how="outer",
        validate="one_to_one",
        suffixes=("_fit", "_cv"),
        indicator=True
    )

    if not table3["_merge"].eq("both").all():

        print(
            "WARNING: model keys do not match. "
            "Keeping separate tables."
        )

        can_merge = False

    else:

        table3 = table3.drop(
            columns="_merge"
        )

if not can_merge:

    table3 = model_comparison.copy()

    register_table(
        "Table_03B_Allometry_Cross_Validation",
        "Cross-validation of height-diameter models",
        cv_summary,
        source_t3b
    )

register_table(
    "Table_03_Allometry_Model_Comparison",
    "Comparison of height-diameter allometric models",
    table3,
    (
        f"{source_t3a}; {source_t3b}"
        if can_merge else source_t3a
    ),
    notes=(
        "Model fit and predictive validation within "
        "the sampled forest plots. "
        "No inference to independently replicated forests."
    )
)

# ============================================================
# TABLE 4 — ARCHITECTURAL COUNTERFACTUAL
# ============================================================

source_t4 = f"{AL}/HEIGHT_COUNTERFACTUAL_SUMMARY.csv"

table4 = load_table(source_t4)

register_table(
    "Table_04_Height_Counterfactual",
    "Counterfactual biomass under alternative height architecture",
    table4,
    source_t4,
    notes=(
        "Model-based counterfactual comparison, "
        "not an experimental or causal estimate. "
        "Preserve the original common-diameter support."
    )
)

# ============================================================
# TABLE 5 — INTEGRATED BIOMASS DECOMPOSITION
# ============================================================

source_t5 = (
    f"{FE}/BIOMASS_INTEGRATED_LMDI_SHAPLEY.csv"
)

table5 = load_table(source_t5)

component_names = {
    "N": "Stem abundance",
    "Diameter_moment": "Diameter distribution",
    "Height_factor": "Height dimension",
    "Wood_density_factor": "Wood density"
}

table5["component"] = (
    table5["component"]
    .replace(component_names)
)

observed_gap = (
    full.loc["Late_secondary", "AGB_Mg"]
    - full.loc["Mature", "AGB_Mg"]
)

assert np.isclose(
    table5["Shapley_Mg"].sum(),
    observed_gap,
    atol=0.01
)

assert np.isclose(
    table5["LMDI_Mg"].sum(),
    observed_gap,
    atol=0.01
)

assert np.isclose(
    table5["Shapley_pct"].sum(),
    100,
    atol=0.05
)

total_row = {
    "component": "Net biomass difference",
    "LMDI_Mg": table5["LMDI_Mg"].sum(),
    "Shapley_Mg": table5["Shapley_Mg"].sum(),
    "LMDI_pct": table5["LMDI_pct"].sum(),
    "Shapley_pct": table5["Shapley_pct"].sum()
}

if "method_difference_Mg" in table5.columns:

    total_row["method_difference_Mg"] = (
        table5["method_difference_Mg"].sum()
    )

table5 = pd.concat(
    [
        table5,
        pd.DataFrame([total_row])
    ],
    ignore_index=True
)

table5 = table5.rename(columns={
    "component": "Component",
    "LMDI_Mg": "LMDI contribution (Mg ha-1)",
    "Shapley_Mg": "Shapley contribution (Mg ha-1)",
    "LMDI_pct": "LMDI contribution (%)",
    "Shapley_pct": "Shapley contribution (%)",
    "method_difference_Mg": "Method difference (Mg ha-1)"
})

register_table(
    "Table_05_Integrated_Biomass_Decomposition",
    "Exact additive decomposition of the biomass difference",
    table5,
    source_t5,
    notes=(
        "Secondary minus mature forest. "
        "LMDI and Shapley provide exact accounting "
        "decompositions. Contributions are not "
        "independent causal effects."
    )
)

# ============================================================
# TABLE 6 — ROBUSTNESS OF BIOMASS CONTRAST
# ============================================================

source_t6a = (
    f"{FE}/BIOMASS_INTEGRATED_SENSITIVITY.csv"
)

source_t6b = (
    f"{FE}/BIOMASS_INTEGRATED_BOOTSTRAP_SUMMARY.csv"
)

source_t6c = (
    f"{FE}/BIOMASS_BIOMETRIC_MC_SUMMARY.csv"
)

sensitivity = load_table(source_t6a)
bootstrap = load_table(source_t6b)
mc_summary = load_table(source_t6c)

# Verify that biomass totals are identical across
# component rows within each sensitivity scenario.

check_columns = [
    "secondary_AGB_Mg",
    "mature_AGB_Mg",
    "total_gap_Mg"
]

for column in check_columns:

    assert (
        sensitivity.groupby("scenario")[column]
        .nunique()
        .le(1)
        .all()
    )

scenario_totals = (
    sensitivity[
        ["scenario"] + check_columns
    ]
    .drop_duplicates()
    .copy()
)

scenario_totals["Biomass ratio"] = (
    scenario_totals["secondary_AGB_Mg"]
    / scenario_totals["mature_AGB_Mg"]
)

scenario_totals["Secondary excess (%)"] = (
    100 * (
        scenario_totals["Biomass ratio"] - 1
    )
)

scenario_names = {
    "Full_inventory": "Full analytical inventory",
    "Without_Pseudosenefeldera":
        "Excluding P. inclinata",
    "Common_diameter_support":
        "Common diameter support",
    "Species_level_WD_only":
        "Species-level wood density only"
}

scenario_totals["scenario"] = (
    scenario_totals["scenario"]
    .replace(scenario_names)
)

table6 = scenario_totals.rename(columns={
    "scenario": "Scenario",
    "secondary_AGB_Mg": "Secondary biomass (Mg)",
    "mature_AGB_Mg": "Mature biomass (Mg)",
    "total_gap_Mg": "Biomass difference (Mg)"
})

register_table(
    "Table_06A_Biomass_Sensitivity",
    "Sensitivity of biomass contrasts to inventory restrictions",
    table6,
    source_t6a,
    notes=(
        "Biomass refers to retained stems within each "
        "original 1-ha plot. Restricted inventories "
        "are not alternative complete forest estimates."
    )
)

# Conditional subplot bootstrap

bootstrap_gap = bootstrap.loc[
    bootstrap["metric"].eq("biomass_gap_Mg")
].copy()

assert len(bootstrap_gap) == 1

# Biometric Monte Carlo uncertainty

mc_gap = mc_summary.loc[
    mc_summary["metric"].eq("biomass_gap_Mg")
].copy()

uncertainty_rows = []

for _, row in bootstrap_gap.iterrows():

    uncertainty_rows.append({
        "Uncertainty source":
            "Conditional subplot bootstrap",
        "Scenario": "Full inventory",
        "Mean difference (Mg)": row["mean"],
        "Median difference (Mg)": row["median"],
        "Lower 95% limit (Mg)": row["LCL_95"],
        "Upper 95% limit (Mg)": row["UCL_95"],
        "Positive draws (%)":
            row["positive_draws_pct"]
    })

for _, row in mc_gap.iterrows():

    uncertainty_rows.append({
        "Uncertainty source":
            "Biometric Monte Carlo",
        "Scenario": row["scenario"],
        "Mean difference (Mg)": row["mean"],
        "Median difference (Mg)": row["median"],
        "Lower 95% limit (Mg)": row["LCL_95"],
        "Upper 95% limit (Mg)": row["UCL_95"],
        "Positive draws (%)":
            row["positive_draws_pct"]
    })

table6b = pd.DataFrame(uncertainty_rows)

register_table(
    "Table_06B_Biomass_Uncertainty",
    "Conditional uncertainty of the biomass difference",
    table6b,
    f"{source_t6b}; {source_t6c}",
    notes=(
        "Bootstrap and biometric Monte Carlo represent "
        "different uncertainty sources and are not pooled. "
        "Intervals are conditional on the sampled plots "
        "and specified uncertainty assumptions. "
        "Positive draws are not population-level p-values."
    )
)

print("\nMAIN TABLES COMPLETED")

In [ ]:
# ============================================================
# CELL 77 — SUPPLEMENTARY SCIENTIFIC TABLES
# ============================================================

SUPPLEMENTARY_PLAN = [

    (
        "Table_S01_Inventory_QA",
        "Inventory quality control",
        "02_DATA_MASTER/SATIPO_QA_QC_V01.csv"
    ),

    (
        "Table_S02_Taxonomic_Validation",
        "Taxonomic harmonization and validation",
        "02_DATA_MASTER/VERSION_03/"
        "SATIPO_TAXONOMY_SUMMARY_V03.csv"
    ),

    (
        "Table_S03_Taxonomic_Checklist",
        "Harmonized taxonomic and trait checklist",
        "02_DATA_MASTER/VERSION_04/"
        "SATIPO_TAXONOMY_MASTER_V04.csv"
    ),

    (
        "Table_S04_Hill_Diversity",
        "Observed Hill diversity across taxonomic scenarios",
        f"{TX}/HILL_DIVERSITY_SUMMARY.csv"
    ),

    (
        "Table_S05_Sample_Completeness",
        "Sample completeness and coverage",
        f"{IX}/INEXT_SAMPLE_INFORMATION.csv"
    ),

    (
        "Table_S06_Beta_Diversity",
        "Taxonomic dissimilarity and turnover",
        f"{BT}/BETA_DIVERSITY_SUMMARY.csv"
    ),

    (
        "Table_S07_Diversity_Comparison",
        "Coverage-standardized diversity contrasts",
        f"{IX}/INEXT_DIVERSITY_COMPARISON.csv"
    ),

    (
        "Table_S08_Allometry_CV",
        "Height-diameter cross-validation results",
        f"{AL}/ALLOMETRY_CV_SUMMARY.csv"
    ),

    (
        "Table_S09_Allometry_Sensitivity",
        "Sensitivity of height-diameter model fitting",
        f"{AL}/ALLOMETRY_MODEL_SENSITIVITY.csv"
    ),

    (
        "Table_S10_Height_Reference_Diameters",
        "Height contrasts at reference diameters",
        f"{AL}/HEIGHT_CONTRAST_REFERENCE_DIAMETERS.csv"
    ),

    (
        "Table_S11_Allometry_Bootstrap",
        "Bootstrap uncertainty in height predictions",
        f"{AL}/ALLOMETRY_BOOTSTRAP_REFERENCE.csv"
    ),

    (
        "Table_S12_Architecture_Sensitivity",
        "Counterfactual architecture model sensitivity",
        f"{AL}/COUNTERFACTUAL_MODEL_SENSITIVITY.csv"
    ),

    (
        "Table_S13_Architecture_Uncertainty",
        "Uncertainty in architectural contributions",
        f"{AL}/ARCHITECTURE_CONTRIBUTION_UNCERTAINTY.csv"
    ),

    (
        "Table_S14_Biomass_Diameter_Classes",
        "Biomass decomposition across diameter classes",
        f"{BD}/DBH_BIOMASS_DECOMPOSITION.csv"
    ),

    (
        "Table_S15_Wood_Density_CWM",
        "Community-weighted wood density",
        f"{FE}/WOOD_DENSITY_CWM.csv"
    ),

    (
        "Table_S16_Wood_Density_Sensitivity",
        "Sensitivity of community-weighted wood density",
        f"{FE}/WOOD_DENSITY_CWM_SENSITIVITY.csv"
    ),

    (
        "Table_S17_Wood_Density_Standardization",
        "Biomass under standardized wood density",
        f"{FE}/WOOD_DENSITY_STANDARDIZED_GAP.csv"
    ),

    (
        "Table_S18_Dominant_Taxon_Exclusion",
        "Inventory characteristics after dominant taxon exclusion",
        f"{FE}/DOMINANT_SPECIES_EXCLUSION.csv"
    ),

    (
        "Table_S19_Taxonomic_Biomass_Concentration",
        "Concentration of biomass among taxa",
        f"{FE}/BIOMASS_TAXONOMIC_CONCENTRATION.csv"
    ),

    (
        "Table_S20_Taxonomic_Biomass_Gap",
        "Taxon-level contributions to biomass differences",
        f"{FE}/TAXONOMIC_BIOMASS_GAP.csv"
    ),

    (
        "Table_S21_Exact_Biomass_Factorization",
        "Exact multiplicative biomass factorization",
        f"{FE}/BIOMASS_EXACT_FACTORIZATION.csv"
    ),

    (
        "Table_S22_Decomposition_Order",
        "Order sensitivity of sequential decomposition",
        f"{FE}/BIOMASS_DECOMPOSITION_ORDER_SENSITIVITY.csv"
    ),

    (
        "Table_S23_Integrated_Sensitivity",
        "Integrated decomposition across inventory scenarios",
        f"{FE}/BIOMASS_INTEGRATED_SENSITIVITY.csv"
    ),

    (
        "Table_S24_Subplot_Bootstrap",
        "Conditional subplot bootstrap uncertainty",
        f"{FE}/BIOMASS_INTEGRATED_BOOTSTRAP_SUMMARY.csv"
    ),

    (
        "Table_S25_Biometric_Monte_Carlo",
        "Biometric Monte Carlo uncertainty",
        f"{FE}/BIOMASS_BIOMETRIC_MC_SUMMARY.csv"
    ),

    (
        "Table_S26_Height_Bias_Thresholds",
        "Systematic height bias thresholds",
        f"{FE}/BIOMASS_HEIGHT_BIAS_THRESHOLDS.csv"
    )
]

supplementary_errors = []

for table_id, title, source in SUPPLEMENTARY_PLAN:

    try:

        df = load_table(source)

        register_table(
            table_id,
            title,
            df,
            source,
            destination="supplementary"
        )

    except Exception as e:

        supplementary_errors.append({
            "table_id": table_id,
            "source": source,
            "error": str(e)
        })

# ------------------------------------------------------------
# Export supplementary table manifest
# ------------------------------------------------------------

table_manifest = pd.DataFrame(table_metadata)

table_manifest.to_csv(
    METADATA / "MANUSCRIPT_TABLE_MANIFEST.csv",
    index=False
)

print("\nSUPPLEMENTARY TABLE SUMMARY")

print(
    "Supplementary tables:",
    len(supp_tables)
)

print(
    "Errors:",
    len(supplementary_errors)
)

if supplementary_errors:

    display(
        pd.DataFrame(supplementary_errors)
    )

display(
    table_manifest[
        [
            "table_id",
            "title",
            "destination",
            "n_rows",
            "n_columns"
        ]
    ]
)

print("\nCELL 77 COMPLETED")

In [ ]:
# ============================================================
# CELL 78 — EDITORIAL EXCEL WORKBOOKS
# ============================================================

!pip -q install openpyxl

from openpyxl.styles import (
    Font, PatternFill, Alignment, Border, Side
)

from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo

# ------------------------------------------------------------
# Excel formatting
# ------------------------------------------------------------

def export_editorial_workbook(
    tables,
    output_path,
    manifest
):

    sheet_map = []

    with pd.ExcelWriter(
        output_path,
        engine="openpyxl"
    ) as writer:

        for i, (table_id, df) in enumerate(
            tables.items(), start=1
        ):

            sheet_name = table_id[:31]

            if sheet_name in writer.book.sheetnames:

                sheet_name = (
                    table_id[:26] + f"_{i:03d}"
                )

            df.to_excel(
                writer,
                sheet_name=sheet_name,
                index=False,
                startrow=3
            )

            ws = writer.sheets[sheet_name]

            meta = manifest.loc[
                manifest["table_id"].eq(table_id)
            ].iloc[0]

            ncols = len(df.columns)

            # Title
            ws.merge_cells(
                start_row=1,
                start_column=1,
                end_row=1,
                end_column=ncols
            )

            ws.cell(1, 1).value = (
                table_id.replace("_", " ")
                + ". "
                + meta["title"]
            )

            ws.cell(1, 1).font = Font(
                name="Calibri",
                size=14,
                bold=True,
                color="173D35"
            )

            ws.cell(1, 1).alignment = Alignment(
                wrap_text=True,
                vertical="center"
            )

            ws.row_dimensions[1].height = 36

            # Notes
            ws.merge_cells(
                start_row=2,
                start_column=1,
                end_row=2,
                end_column=ncols
            )

            ws.cell(2, 1).value = (
                str(meta["notes"])
                if pd.notna(meta["notes"])
                else ""
            )

            ws.cell(2, 1).font = Font(
                size=10,
                italic=True,
                color="666666"
            )

            ws.cell(2, 1).alignment = Alignment(
                wrap_text=True,
                vertical="center"
            )

            ws.row_dimensions[2].height = 42

            # Header
            for cell in ws[4]:

                cell.fill = PatternFill(
                    "solid",
                    fgColor="245B4B"
                )

                cell.font = Font(
                    bold=True,
                    color="FFFFFF",
                    size=11
                )

                cell.alignment = Alignment(
                    wrap_text=True,
                    vertical="center"
                )

            ws.row_dimensions[4].height = 36

            # Body
            for row in ws.iter_rows(
                min_row=5,
                max_row=ws.max_row
            ):

                for cell in row:

                    cell.font = Font(
                        name="Calibri",
                        size=11
                    )

                    cell.alignment = Alignment(
                        vertical="center"
                    )

                    if isinstance(
                        cell.value, float
                    ):

                        cell.number_format = "0.000"

                    if cell.row % 2 == 0:

                        cell.fill = PatternFill(
                            "solid",
                            fgColor="F1F6F3"
                        )

                    cell.border = Border(
                        bottom=Side(
                            style="hair",
                            color="D7E3DD"
                        )
                    )

            # Column widths
            for j, column in enumerate(
                df.columns, start=1
            ):

                lengths = [
                    len(str(column))
                ]

                lengths.extend(
                    df[column]
                    .astype(str)
                    .head(500)
                    .map(len)
                    .tolist()
                )

                width = min(
                    max(max(lengths) + 3, 14),
                    48
                )

                ws.column_dimensions[
                    get_column_letter(j)
                ].width = width

            ws.freeze_panes = "B5"

            ws.auto_filter.ref = (
                f"A4:"
                f"{get_column_letter(ncols)}"
                f"{ws.max_row}"
            )

            ws.sheet_view.showGridLines = False

            ws.print_title_rows = "1:4"

            ws.sheet_properties.pageSetUpPr.fitToPage = True

            ws.page_setup.orientation = "landscape"
            ws.page_setup.fitToWidth = 1
            ws.page_setup.fitToHeight = 0

            sheet_map.append({
                "Table": table_id,
                "Worksheet": sheet_name,
                "Title": meta["title"],
                "Rows": len(df),
                "Columns": ncols,
                "Source": meta["source"]
            })

        # Workbook index
        index_df = pd.DataFrame(sheet_map)

        index_df.to_excel(
            writer,
            sheet_name="INDEX",
            index=False
        )

        index_ws = writer.sheets["INDEX"]

        for cell in index_ws[1]:

            cell.fill = PatternFill(
                "solid",
                fgColor="245B4B"
            )

            cell.font = Font(
                bold=True,
                color="FFFFFF"
            )

        for i, width in enumerate(
            [45, 34, 65, 12, 12, 90],
            start=1
        ):

            index_ws.column_dimensions[
                get_column_letter(i)
            ].width = width

        index_ws.freeze_panes = "A2"
        index_ws.sheet_view.showGridLines = False

        # Move index to first position
        writer.book.move_sheet(
            index_ws,
            offset=-(
                len(writer.book.sheetnames) - 1
            )
        )

    print(
        "Workbook saved:",
        output_path.name
    )

    print(
        "Size:",
        round(
            output_path.stat().st_size / 1024**2,
            3
        ),
        "MB"
    )


# ------------------------------------------------------------
# Export workbooks
# ------------------------------------------------------------

table_manifest = pd.DataFrame(table_metadata)

export_editorial_workbook(
    main_tables,
    MAIN_TABLES /
    "SATIPO_MAIN_TABLES.xlsx",
    table_manifest
)

export_editorial_workbook(
    supp_tables,
    SUPP_TABLES /
    "SATIPO_SUPPLEMENTARY_TABLES.xlsx",
    table_manifest
)

# ------------------------------------------------------------
# Save table captions and notes
# ------------------------------------------------------------

caption_path = (
    METADATA / "TABLE_CAPTIONS_AND_NOTES.txt"
)

with open(
    caption_path,
    "w",
    encoding="utf-8"
) as f:

    for _, row in table_manifest.iterrows():

        f.write(
            row["table_id"].replace("_", " ")
            + ". "
            + row["title"]
            + "\n"
        )

        if row["notes"]:

            f.write(
                "Note: "
                + str(row["notes"])
                + "\n"
            )

        f.write(
            "Source: "
            + row["source"]
            + "\n\n"
        )

# ------------------------------------------------------------
# Final editorial package audit
# ------------------------------------------------------------

package_records = []

for path in sorted(
    MANUSCRIPT.rglob("*")
):

    if path.is_file():

        package_records.append({
            "file": path.name,
            "relative_path": str(
                path.relative_to(MANUSCRIPT)
            ),
            "extension": path.suffix.lower(),
            "size_MB": round(
                path.stat().st_size / 1024**2,
                4
            )
        })

editorial_package = pd.DataFrame(
    package_records
)

editorial_package.to_csv(
    METADATA /
    "EDITORIAL_PACKAGE_INVENTORY.csv",
    index=False
)

print("\n" + "=" * 65)
print("STAGE 1 — EDITORIAL PACKAGE SUMMARY")
print("=" * 65)

print("Main table sheets:", len(main_tables))
print("Supplementary tables:", len(supp_tables))

print(
    "Package files:",
    len(editorial_package)
)

print(
    "Total package size (MB):",
    round(
        editorial_package["size_MB"].sum(),
        2
    )
)

print("\nMAIN TABLES")

display(
    table_manifest.loc[
        table_manifest["destination"].eq("main"),
        [
            "table_id",
            "title",
            "n_rows",
            "n_columns"
        ]
    ]
)

print("\nPACKAGE FILE TYPES")

display(
    editorial_package.groupby("extension")
    .agg(
        n_files=("file", "count"),
        total_MB=("size_MB", "sum")
    )
    .reset_index()
)

print("\nCELL 78 COMPLETED SUCCESSFULLY")

In [ ]:
# ============================================================
# CELL 79 — FINAL MANUSCRIPT TABLE INSPECTION
# ============================================================

from pathlib import Path
import pandas as pd

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

MAIN_TABLES = (
    ROOT / "07_MANUSCRIPT_READY" / "MAIN_TABLES"
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 250)

files_to_review = {

    "TABLE 2 — COVERAGE-STANDARDIZED DIVERSITY":
        "Table_02_Coverage_Standardized_Diversity.csv",

    "TABLE 3 — ALLOMETRIC MODEL COMPARISON":
        "Table_03_Allometry_Model_Comparison.csv",

    "TABLE 4 — HEIGHT COUNTERFACTUAL":
        "Table_04_Height_Counterfactual.csv"

}

review_tables = {}

for title, filename in files_to_review.items():

    path = MAIN_TABLES / filename

    assert path.exists(), f"Missing file: {path}"

    df = pd.read_csv(path)

    review_tables[filename] = df

    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)

    print("\nDIMENSIONS:", df.shape)

    print("\nCOLUMN NAMES:")
    print(df.columns.tolist())

    print("\nCOMPLETE TABLE:")
    display(df)

    print("\nMISSING VALUES:")

    missing = df.isna().sum()

    display(
        missing[missing > 0]
        .rename("missing_values")
        .to_frame()
    )

    print("\nNUMERICAL SUMMARY:")

    display(
        df.select_dtypes(include="number")
        .describe()
        .round(4)
    )

print("\n" + "=" * 90)
print("CELL 79 COMPLETED SUCCESSFULLY")
print("=" * 90)

In [ ]:
# ============================================================
# CELL 80 — SUPPLEMENTARY TABLE FINALIZATION
# ============================================================

from pathlib import Path
import pandas as pd

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

SUPP = (
    ROOT / "07_MANUSCRIPT_READY"
    / "SUPPLEMENTARY_TABLES"
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 300)
pd.set_option("display.max_colwidth", 150)

# Tables requiring exact source inspection

review_plan = {

    "S3 — TAXONOMIC CHECKLIST":
        "Table_S03_Taxonomic_Checklist.csv",

    "S4 — OBSERVED HILL DIVERSITY":
        "Table_S04_Hill_Diversity.csv",

    "S5 — SAMPLE COMPLETENESS":
        "Table_S05_Sample_Completeness.csv",

    "S6 — BETA DIVERSITY":
        "Table_S06_Beta_Diversity.csv",

    "S9 — ALLOMETRIC SENSITIVITY":
        "Table_S09_Allometry_Sensitivity.csv",

    "S10 — REFERENCE DIAMETERS":
        "Table_S10_Height_Reference_Diameters.csv",

    "S11 — ALLOMETRIC BOOTSTRAP":
        "Table_S11_Allometry_Bootstrap.csv",

    "S12 — ARCHITECTURE SENSITIVITY":
        "Table_S12_Architecture_Sensitivity.csv",

    "S13 — ARCHITECTURE UNCERTAINTY":
        "Table_S13_Architecture_Uncertainty.csv",

    "S14 — DIAMETER CLASSES":
        "Table_S14_Biomass_Diameter_Classes.csv",

    "S16 — WOOD DENSITY SENSITIVITY":
        "Table_S16_Wood_Density_Sensitivity.csv",

    "S17 — WOOD DENSITY STANDARDIZATION":
        "Table_S17_Wood_Density_Standardization.csv",

    "S18 — DOMINANT TAXON EXCLUSION":
        "Table_S18_Dominant_Taxon_Exclusion.csv",

    "S19 — TAXONOMIC CONCENTRATION":
        "Table_S19_Taxonomic_Biomass_Concentration.csv",

    "S20 — TAXONOMIC BIOMASS GAP":
        "Table_S20_Taxonomic_Biomass_Gap.csv",

    "S22 — DECOMPOSITION ORDER":
        "Table_S22_Decomposition_Order.csv",

    "S23 — INTEGRATED SENSITIVITY":
        "Table_S23_Integrated_Sensitivity.csv",

    "S24 — SUBPLOT BOOTSTRAP":
        "Table_S24_Subplot_Bootstrap.csv",

    "S25 — BIOMETRIC MONTE CARLO":
        "Table_S25_Biometric_Monte_Carlo.csv"
}

supplementary_review = {}

for title, filename in review_plan.items():

    path = SUPP / filename

    assert path.exists(), f"Missing: {path}"

    df = pd.read_csv(path)

    supplementary_review[filename] = df

    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    print("Dimensions:", df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    # Large tables: inspect schema and sample only
    if len(df) > 100:

        print("\nFirst five records:")
        display(df.head(5))

    else:

        print("\nComplete results:")
        display(df.round(6))

print("\nCELL 80 COMPLETED SUCCESSFULLY")

In [ ]:

# ============================================================
# CELL 81 — COMPLETE SUPPLEMENTARY TAXONOMIC TABLES
# ============================================================

!pip -q install python-docx

from pathlib import Path
import pandas as pd

from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.section import WD_ORIENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

SUPP = (
    ROOT / "07_MANUSCRIPT_READY"
    / "SUPPLEMENTARY_TABLES"
)

# ------------------------------------------------------------
# 1. LOAD EXACT SOURCE TABLES
# ------------------------------------------------------------

taxonomy = pd.read_csv(
    SUPP / "Table_S03_Taxonomic_Checklist.csv"
)

biomass_taxa = pd.read_csv(
    SUPP / "Table_S20_Taxonomic_Biomass_Gap.csv"
)

assert len(taxonomy) == 163
assert len(biomass_taxa) == 161

# ------------------------------------------------------------
# 2. PREPARE TABLE S3
# ------------------------------------------------------------

s3 = taxonomy[[
    "species_clean",
    "final_accepted_name",
    "final_family",
    "resolution_status",
    "operational_taxon",
    "n_records"
]].copy()

s3.columns = [
    "Original identification",
    "Accepted taxonomic name",
    "Family",
    "Resolution status",
    "Operational taxon",
    "Individuals"
]

s3["Resolution status"] = (
    s3["Resolution status"]
    .str.replace("_", " ", regex=False)
    .str.capitalize()
)

s3 = s3.sort_values(
    ["Family", "Operational taxon"],
    na_position="last"
).reset_index(drop=True)

s3 = s3.fillna("—")

# ------------------------------------------------------------
# 3. PREPARE TABLE S20
# ------------------------------------------------------------

s20 = biomass_taxa[[
    "functional_taxon",
    "Late_secondary",
    "Mature",
    "biomass_difference_Mg",
    "net_gap_contribution_pct",
    "absolute_contribution_pct",
    "cumulative_absolute_pct"
]].copy()

s20.columns = [
    "Taxon",
    "Secondary AGB (Mg)",
    "Mature AGB (Mg)",
    "Difference (Mg)",
    "Net contribution (%)",
    "Absolute contribution (%)",
    "Cumulative absolute (%)"
]

# Preserve original ranking
s20 = s20.reset_index(drop=True)

# ------------------------------------------------------------
# 4. WORD DOCUMENT CONFIGURATION
# ------------------------------------------------------------

doc = Document()

section = doc.sections[0]

section.orientation = WD_ORIENT.LANDSCAPE
section.page_width = Inches(11.69)
section.page_height = Inches(8.27)

section.top_margin = Inches(0.65)
section.bottom_margin = Inches(0.65)
section.left_margin = Inches(0.65)
section.right_margin = Inches(0.65)

normal = doc.styles["Normal"]

normal.font.name = "Times New Roman"
normal.font.size = Pt(10)

normal.paragraph_format.space_after = Pt(4)

# ------------------------------------------------------------
# 5. FORMATTING HELPERS
# ------------------------------------------------------------

def set_cell_border(cell, **kwargs):

    tc = cell._tc
    tcPr = tc.get_or_add_tcPr()

    borders = tcPr.first_child_found_in(
        "w:tcBorders"
    )

    if borders is None:
        borders = OxmlElement("w:tcBorders")
        tcPr.append(borders)

    for edge, settings in kwargs.items():

        tag = f"w:{edge}"
        element = borders.find(qn(tag))

        if element is None:
            element = OxmlElement(tag)
            borders.append(element)

        for key, value in settings.items():
            element.set(
                qn(f"w:{key}"),
                str(value)
            )


def repeat_header(row):

    trPr = row._tr.get_or_add_trPr()

    header = OxmlElement("w:tblHeader")
    header.set(qn("w:val"), "true")

    trPr.append(header)


def prevent_row_split(row):

    trPr = row._tr.get_or_add_trPr()

    cant_split = OxmlElement("w:cantSplit")
    trPr.append(cant_split)


def write_cell(
    cell,
    value,
    bold=False,
    italic=False,
    numeric=False
):

    paragraph = cell.paragraphs[0]

    paragraph.paragraph_format.space_after = Pt(2)
    paragraph.paragraph_format.space_before = Pt(2)

    paragraph.alignment = (
        WD_ALIGN_PARAGRAPH.RIGHT
        if numeric
        else WD_ALIGN_PARAGRAPH.LEFT
    )

    run = paragraph.add_run(str(value))

    run.font.name = "Times New Roman"
    run.font.size = Pt(9)

    run.bold = bold
    run.italic = italic


def add_caption(number, title):

    paragraph = doc.add_paragraph()

    paragraph.paragraph_format.keep_with_next = True

    run = paragraph.add_run(
        f"Table {number}. "
    )

    run.bold = True

    paragraph.add_run(title)


def add_note(text):

    paragraph = doc.add_paragraph()

    paragraph.paragraph_format.space_before = Pt(6)

    run = paragraph.add_run("Note: ")
    run.italic = True

    paragraph.add_run(text)


def add_scientific_table(
    df,
    italic_columns=None,
    numeric_columns=None,
    integer_columns=None
):

    italic_columns = italic_columns or []
    numeric_columns = numeric_columns or []
    integer_columns = integer_columns or []

    table = doc.add_table(
        rows=1,
        cols=len(df.columns)
    )

    table.autofit = True

    header = table.rows[0]

    repeat_header(header)
    prevent_row_split(header)

    for j, column in enumerate(df.columns):

        cell = header.cells[j]

        write_cell(
            cell,
            column,
            bold=True
        )

        set_cell_border(
            cell,
            top={
                "val": "single",
                "sz": "8",
                "color": "000000"
            },
            bottom={
                "val": "single",
                "sz": "6",
                "color": "000000"
            }
        )

    for _, record in df.iterrows():

        row = table.add_row()

        prevent_row_split(row)

        for j, column in enumerate(df.columns):

            value = record[column]

            if pd.isna(value):
                value = "—"

            elif column in integer_columns:
                value = f"{int(value)}"

            elif column in numeric_columns:
                value = f"{float(value):.2f}"

            write_cell(
                row.cells[j],
                value,
                italic=(
                    column in italic_columns
                    and value != "—"
                ),
                numeric=(
                    column in numeric_columns
                    or column in integer_columns
                )
            )

    # Bottom rule

    for cell in table.rows[-1].cells:

        set_cell_border(
            cell,
            bottom={
                "val": "single",
                "sz": "8",
                "color": "000000"
            }
        )

    return table

# ------------------------------------------------------------
# 6. TABLE S3
# ------------------------------------------------------------

add_caption(
    "S3",
    "Harmonized taxonomic checklist of the "
    "late-secondary and mature forest inventories "
    "in the central Peruvian Amazon."
)

add_scientific_table(
    s3,
    italic_columns=[
        "Original identification",
        "Accepted taxonomic name",
        "Operational taxon"
    ],
    integer_columns=["Individuals"]
)

add_note(
    "Taxonomic names were harmonized against the "
    "GBIF taxonomic backbone, retaining unresolved "
    "identifications as operational taxa where "
    "appropriate. Individual counts correspond "
    "to the floristic inventory. The checklist "
    "contains 163 original taxonomic entries. "
    "Dashes indicate unavailable information."
)

# ------------------------------------------------------------
# 7. TABLE S20
# ------------------------------------------------------------

doc.add_page_break()

add_caption(
    "S20",
    "Taxon-level contributions to the aboveground "
    "biomass difference between the late-secondary "
    "and mature forest inventories."
)

add_scientific_table(
    s20,
    italic_columns=["Taxon"],
    numeric_columns=[
        "Secondary AGB (Mg)",
        "Mature AGB (Mg)",
        "Difference (Mg)",
        "Net contribution (%)",
        "Absolute contribution (%)",
        "Cumulative absolute (%)"
    ]
)

add_note(
    "AGB, aboveground biomass. Taxon-level "
    "differences were calculated as late-secondary "
    "minus mature forest biomass. Net contributions "
    "were expressed relative to the total biomass "
    "difference. Absolute contributions were "
    "calculated relative to the sum of absolute "
    "taxon-level differences. Taxa were ranked "
    "according to the absolute magnitude of their "
    "biomass differences. The table contains "
    "161 operational taxa."
)

# ------------------------------------------------------------
# 8. SAVE OUTPUTS
# ------------------------------------------------------------

output_docx = (
    SUPP / "SATIPO_COMPLETE_TABLES_S03_S20.docx"
)

doc.save(output_docx)

s3.to_csv(
    SUPP / "Table_S03_Paper_Ready.csv",
    index=False,
    encoding="utf-8-sig"
)

s20.to_csv(
    SUPP / "Table_S20_Paper_Ready.csv",
    index=False,
    encoding="utf-8-sig"
)

print("=" * 70)
print("COMPLETE SUPPLEMENTARY TABLES EXPORTED")
print("=" * 70)

print(f"Table S3: {len(s3)} records")
print(f"Table S20: {len(s20)} records")

print("\nWord document:")
print(output_docx)

print(
    f"\nSize: "
    f"{output_docx.stat().st_size / 1024:.2f} KB"
)

print("\nCELL 81 COMPLETED SUCCESSFULLY")


In [ ]:

# ============================================================
# CELL 82 — AUDIT OF MONTE CARLO UNCERTAINTY FRAMEWORK
# SATIPO FOREST PROJECT
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

ANALYSIS_DIR = ROOT / "03_ANALYSIS"

UNCERTAINTY_DIR = (
    ANALYSIS_DIR / "FUNCTIONAL_ECOLOGY"
)

MC_OUTPUT = (
    ANALYSIS_DIR / "INTEGRATED_UNCERTAINTY"
)

MC_OUTPUT.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 80)
print("SATIPO — MONTE CARLO UNCERTAINTY AUDIT")
print("=" * 80)

# ------------------------------------------------------------
# 1. LOAD FINAL ANALYTICAL INVENTORY
# ------------------------------------------------------------

inventory_path = (
    ANALYSIS_DIR
    / "ALLOMETRY"
    / "SATIPO_ALLOMETRY_INPUT.csv"
)

assert inventory_path.exists(), (
    f"Missing inventory: {inventory_path}"
)

mc_inventory = pd.read_csv(
    inventory_path,
    low_memory=False
)

print("\nFINAL ANALYTICAL INVENTORY")
print("Dimensions:", mc_inventory.shape)

print("\nAvailable columns:")
for i, column in enumerate(
    mc_inventory.columns, start=1
):
    print(f"{i:02d}. {column}")

print("\nForest inventory counts:")

display(
    mc_inventory["forest_type"]
    .value_counts()
    .rename("n_trees")
    .to_frame()
)

assert len(mc_inventory) == 1002

# ------------------------------------------------------------
# 2. IDENTIFY BIOMETRIC AND BIOMASS VARIABLES
# ------------------------------------------------------------

keywords = [
    "dbh",
    "height",
    "density",
    "wood",
    "rho",
    "agb",
    "biomass",
    "subplot",
    "taxon",
    "species",
    "match",
    "source",
    "assignment"
]

relevant_columns = [
    column
    for column in mc_inventory.columns
    if any(
        keyword in column.lower()
        for keyword in keywords
    )
]

print("\nRELEVANT VARIABLES")

for column in relevant_columns:

    series = mc_inventory[column]

    print(
        f"{column:45s} | "
        f"Missing: {series.isna().sum():4d} | "
        f"Unique: {series.nunique():4d}"
    )

# ------------------------------------------------------------
# 3. NUMERICAL INVENTORY AUDIT
# ------------------------------------------------------------

numeric_columns = [
    column
    for column in relevant_columns
    if pd.api.types.is_numeric_dtype(
        mc_inventory[column]
    )
]

print("\nNUMERICAL SUMMARY")

display(
    mc_inventory[numeric_columns]
    .describe()
    .T
    .round(5)
)

print("\nFIRST FIVE INVENTORY RECORDS")

display(
    mc_inventory[relevant_columns]
    .head()
)

# ------------------------------------------------------------
# 4. LOAD PREVIOUS MONTE CARLO RESULTS
# ------------------------------------------------------------

previous_mc_path = (
    UNCERTAINTY_DIR
    / "BIOMASS_BIOMETRIC_MONTE_CARLO.csv"
)

previous_summary_path = (
    UNCERTAINTY_DIR
    / "BIOMASS_BIOMETRIC_MC_SUMMARY.csv"
)

previous_bootstrap_path = (
    UNCERTAINTY_DIR
    / "BIOMASS_INTEGRATED_SUBPLOT_BOOTSTRAP.csv"
)

previous_mc = pd.read_csv(
    previous_mc_path
)

previous_mc_summary = pd.read_csv(
    previous_summary_path
)

previous_bootstrap = pd.read_csv(
    previous_bootstrap_path
)

print("\nPREVIOUS BIOMETRIC MONTE CARLO")
print("Dimensions:", previous_mc.shape)
print("Columns:", previous_mc.columns.tolist())

display(previous_mc.head())

print("\nPREVIOUS MONTE CARLO SUMMARY")

display(previous_mc_summary)

print("\nPREVIOUS SUBPLOT BOOTSTRAP")
print("Dimensions:", previous_bootstrap.shape)
print(
    "Columns:",
    previous_bootstrap.columns.tolist()
)

display(previous_bootstrap.head())

# ------------------------------------------------------------
# 5. RECOVER MONTE CARLO CONFIGURATION FROM MEMORY
# ------------------------------------------------------------

print("\nPOTENTIAL UNCERTAINTY CONFIGURATION OBJECTS")

namespace = list(globals().items())

config_candidates = []

for name, obj in namespace:

    if name.startswith("_"):
        continue

    name_lower = name.lower()

    if not any(
        keyword in name_lower
        for keyword in [
            "scenario",
            "uncertainty",
            "sigma",
            "error",
            "mc_config",
            "mc_params"
        ]
    ):
        continue

    if isinstance(obj, dict):

        representation = repr(obj)

        if len(representation) <= 15000:

            config_candidates.append(name)

            print("\nObject:", name)
            print(representation)

    elif isinstance(
        obj,
        (float, int, np.number)
    ):

        config_candidates.append(name)

        print(
            f"{name} = {obj}"
        )

print(
    "\nConfiguration candidates:",
    config_candidates
)

# ------------------------------------------------------------
# 6. RECOVER RELEVANT CODE FROM NOTEBOOK HISTORY
# ------------------------------------------------------------

print("\nPREVIOUS MONTE CARLO CODE — PARAMETER AUDIT")

try:

    ip = get_ipython()

    history = ip.user_ns.get(
        "In", []
    )

except Exception:

    history = []

search_patterns = [
    r"BIOMASS_BIOMETRIC_MONTE_CARLO",
    r"BIOMASS_BIOMETRIC_MC_SUMMARY",
    r"dbh_cv",
    r"height_cv",
    r"wd_cv",
    r"sigma_dbh",
    r"sigma_height",
    r"sigma_wd",
    r"BIOMETRIC_SCENARIOS"
]

history_matches = []

for cell_number, source in enumerate(history):

    if not isinstance(source, str):
        continue

    if "CELL 82" in source:
        continue

    if any(
        re.search(
            pattern,
            source,
            flags=re.IGNORECASE
        )
        for pattern in search_patterns
    ):

        history_matches.append(
            (cell_number, source)
        )

for cell_number, source in history_matches:

    print("\n" + "-" * 70)
    print(f"Notebook execution: {cell_number}")
    print("-" * 70)

    lines = source.splitlines()

    selected = set()

    for i, line in enumerate(lines):

        if any(
            re.search(
                pattern,
                line,
                flags=re.IGNORECASE
            )
            for pattern in [
                r"scenario",
                r"sigma",
                r"error",
                r"_cv",
                r"normal\(",
                r"lognormal\(",
                r"clip\(",
                r"2000",
                r"seed"
            ]
        ):

            for j in range(
                max(0, i - 2),
                min(len(lines), i + 4)
            ):
                selected.add(j)

    for i in sorted(selected):
        print(f"{i + 1:04d}: {lines[i]}")

# ------------------------------------------------------------
# 7. SAVE AUDIT
# ------------------------------------------------------------

audit_rows = []

for column in mc_inventory.columns:

    audit_rows.append({
        "column": column,
        "dtype": str(
            mc_inventory[column].dtype
        ),
        "missing": int(
            mc_inventory[column].isna().sum()
        ),
        "unique": int(
            mc_inventory[column].nunique()
        )
    })

mc_variable_audit = pd.DataFrame(
    audit_rows
)

mc_variable_audit.to_csv(
    MC_OUTPUT
    / "MC_INVENTORY_VARIABLE_AUDIT.csv",
    index=False
)

with open(
    MC_OUTPUT
    / "PREVIOUS_MC_CODE_AUDIT.txt",
    "w",
    encoding="utf-8"
) as f:

    for cell_number, source in history_matches:

        f.write(
            f"\nEXECUTION {cell_number}\n"
        )

        f.write(source)
        f.write("\n")

# ------------------------------------------------------------
# 8. FINAL VERIFICATION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("AUDIT SUMMARY")
print("=" * 80)

print(
    "Analytical trees:",
    len(mc_inventory)
)

print(
    "Previous MC simulations:",
    len(previous_mc)
)

print(
    "Previous bootstrap simulations:",
    len(previous_bootstrap)
)

print(
    "Recovered configuration objects:",
    len(config_candidates)
)

print(
    "Recovered notebook code cells:",
    len(history_matches)
)

print("\nAudit output folder:")
print(MC_OUTPUT)

print("\nCELL 82 COMPLETED SUCCESSFULLY")


In [ ]:

# ============================================================
# CELL 83 — ALLOMETRIC RESIDUAL UNCERTAINTY
# SATIPO FOREST PROJECT
# Chave et al. (2014), Equation 4
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

MC_OUTPUT = (
    ROOT
    / "03_ANALYSIS"
    / "INTEGRATED_UNCERTAINTY"
)

MC_OUTPUT.mkdir(
    parents=True,
    exist_ok=True
)

FIG_OUTPUT = (
    ROOT / "05_OUTPUTS" / "FIGURES"
)

FIG_OUTPUT.mkdir(
    parents=True,
    exist_ok=True
)

N_SIM = 10000
SEED_ALLOMETRIC = 20261117

# Published residual SD in natural-log scale
SIGMA_ALLOMETRIC = 0.357

rng_allometric = np.random.default_rng(
    SEED_ALLOMETRIC
)

FOREST_ORDER = [
    "Late_secondary",
    "Mature"
]

COLORS = {
    "Late_secondary": "#DB9140",
    "Mature": "#24634F",
    "Gap": "#795C92"
}

# ------------------------------------------------------------
# 2. LOAD FINAL INVENTORY
# ------------------------------------------------------------

inventory_path = (
    ROOT
    / "03_ANALYSIS"
    / "ALLOMETRY"
    / "SATIPO_ALLOMETRY_INPUT.csv"
)

residual_inventory = pd.read_csv(
    inventory_path,
    low_memory=False
)

required_columns = [
    "tree_id",
    "forest_type",
    "subplot_id",
    "dbh_cm",
    "height_m",
    "wood_density_g_cm3",
    "agb_kg"
]

assert all(
    c in residual_inventory.columns
    for c in required_columns
)

assert len(residual_inventory) == 1002

assert residual_inventory[
    "tree_id"
].is_unique

assert residual_inventory[
    required_columns
].notna().all().all()

assert (
    residual_inventory[
        [
            "dbh_cm",
            "height_m",
            "wood_density_g_cm3",
            "agb_kg"
        ]
    ] > 0
).all().all()

counts = (
    residual_inventory["forest_type"]
    .value_counts()
    .to_dict()
)

assert counts == {
    "Late_secondary": 541,
    "Mature": 461
}

# ------------------------------------------------------------
# 3. RECONSTRUCT BASELINE BIOMASS
# ------------------------------------------------------------

D = residual_inventory[
    "dbh_cm"
].to_numpy(dtype=float)

H = residual_inventory[
    "height_m"
].to_numpy(dtype=float)

WD = residual_inventory[
    "wood_density_g_cm3"
].to_numpy(dtype=float)

agb_baseline_kg = (
    0.0673
    * (WD * D**2 * H)**0.976
)

agb_original_kg = residual_inventory[
    "agb_kg"
].to_numpy(dtype=float)

assert np.allclose(
    agb_baseline_kg,
    agb_original_kg,
    rtol=1e-6,
    atol=1e-6
), "Reconstructed biomass differs from inventory"

agb_baseline_Mg = (
    agb_baseline_kg / 1000
)

forest_array = residual_inventory[
    "forest_type"
].to_numpy()

secondary_mask = (
    forest_array == "Late_secondary"
)

mature_mask = (
    forest_array == "Mature"
)

baseline_secondary = (
    agb_baseline_Mg[secondary_mask].sum()
)

baseline_mature = (
    agb_baseline_Mg[mature_mask].sum()
)

baseline_gap = (
    baseline_secondary
    - baseline_mature
)

baseline_ratio = (
    baseline_secondary
    / baseline_mature
)

print("=" * 75)
print("BASELINE INVENTORY")
print("=" * 75)

print(
    f"Secondary AGB: "
    f"{baseline_secondary:.4f} Mg"
)

print(
    f"Mature AGB: "
    f"{baseline_mature:.4f} Mg"
)

print(
    f"Biomass gap: "
    f"{baseline_gap:.4f} Mg"
)

print(
    f"Biomass ratio: "
    f"{baseline_ratio:.4f}"
)

# ------------------------------------------------------------
# 4. RESIDUAL ERROR DISTRIBUTION
# ------------------------------------------------------------

# Chave et al. (2014) Eq. 4 already includes
# the back-transformation correction.
#
# Therefore, use a mean-one multiplicative
# lognormal residual to preserve expected AGB.
#
# E[exp(epsilon)] = 1

sigma = SIGMA_ALLOMETRIC

residual_log_mean = (
    -0.5 * sigma**2
)

residual_cv = np.sqrt(
    np.expm1(sigma**2)
)

individual_multiplier_limits = np.exp(
    residual_log_mean
    + sigma * norm.ppf(
        [0.025, 0.975]
    )
)

print("\nALLOMETRIC RESIDUAL PARAMETERS")

print(
    f"Log-scale SD: {sigma:.3f}"
)

print(
    f"Log-scale mean: "
    f"{residual_log_mean:.6f}"
)

print(
    f"Multiplicative CV: "
    f"{100*residual_cv:.2f}%"
)

print(
    "Individual multiplier 95% interval:",
    np.round(
        individual_multiplier_limits,
        4
    )
)

# ------------------------------------------------------------
# 5. MONTE CARLO SIMULATION
# ------------------------------------------------------------

residual_mc_rows = []

for b in tqdm(
    range(N_SIM),
    desc="Allometric residual Monte Carlo"
):

    # Independent residuals across trees
    # in this initial scenario.

    residual_multiplier = (
        rng_allometric.lognormal(
            mean=residual_log_mean,
            sigma=sigma,
            size=len(residual_inventory)
        )
    )

    agb_simulated = (
        agb_baseline_Mg
        * residual_multiplier
    )

    secondary_agb = (
        agb_simulated[
            secondary_mask
        ].sum()
    )

    mature_agb = (
        agb_simulated[
            mature_mask
        ].sum()
    )

    gap = (
        secondary_agb - mature_agb
    )

    ratio = (
        secondary_agb / mature_agb
    )

    residual_mc_rows.append({
        "iteration": b + 1,
        "scenario": "Independent_allometric_residual",
        "secondary_AGB_Mg": secondary_agb,
        "mature_AGB_Mg": mature_agb,
        "biomass_gap_Mg": gap,
        "biomass_ratio": ratio,
        "secondary_excess_pct": (
            100 * (ratio - 1)
        )
    })

allometric_residual_mc = pd.DataFrame(
    residual_mc_rows
)

assert len(
    allometric_residual_mc
) == N_SIM

assert np.isfinite(
    allometric_residual_mc[
        [
            "secondary_AGB_Mg",
            "mature_AGB_Mg",
            "biomass_gap_Mg",
            "biomass_ratio"
        ]
    ]
).all().all()

# ------------------------------------------------------------
# 6. SUMMARIZE MONTE CARLO DISTRIBUTIONS
# ------------------------------------------------------------

metrics = [
    "secondary_AGB_Mg",
    "mature_AGB_Mg",
    "biomass_gap_Mg",
    "biomass_ratio",
    "secondary_excess_pct"
]

baseline_values = {
    "secondary_AGB_Mg": baseline_secondary,
    "mature_AGB_Mg": baseline_mature,
    "biomass_gap_Mg": baseline_gap,
    "biomass_ratio": baseline_ratio,
    "secondary_excess_pct": (
        100 * (baseline_ratio - 1)
    )
}

summary_rows = []

for metric in metrics:

    values = allometric_residual_mc[
        metric
    ].to_numpy()

    threshold = (
        1.0
        if metric == "biomass_ratio"
        else 0.0
    )

    summary_rows.append({
        "metric": metric,
        "baseline": baseline_values[metric],
        "mean": np.mean(values),
        "median": np.median(values),
        "SD": np.std(values, ddof=1),
        "LCL_95": np.quantile(values, 0.025),
        "UCL_95": np.quantile(values, 0.975),
        "threshold": threshold,
        "above_threshold_pct": (
            100 * np.mean(
                values > threshold
            )
        ),
        "MC_standard_error": (
            np.std(values, ddof=1)
            / np.sqrt(N_SIM)
        )
    })

allometric_residual_summary = (
    pd.DataFrame(summary_rows)
)

print("\nMONTE CARLO SUMMARY")

display(
    allometric_residual_summary.round(4)
)

# ------------------------------------------------------------
# 7. ANALYTICAL VARIANCE VALIDATION
# ------------------------------------------------------------

# For independent mean-one lognormal
# multipliers:
#
# Var(AGB_plot) =
# [exp(sigma^2) - 1] * sum(AGB_i^2)

multiplier_variance = (
    np.expm1(sigma**2)
)

variance_secondary = (
    multiplier_variance
    * np.sum(
        agb_baseline_Mg[
            secondary_mask
        ]**2
    )
)

variance_mature = (
    multiplier_variance
    * np.sum(
        agb_baseline_Mg[
            mature_mask
        ]**2
    )
)

variance_gap = (
    variance_secondary
    + variance_mature
)

analytical_sd = {
    "secondary_AGB_Mg": np.sqrt(
        variance_secondary
    ),
    "mature_AGB_Mg": np.sqrt(
        variance_mature
    ),
    "biomass_gap_Mg": np.sqrt(
        variance_gap
    )
}

validation_rows = []

for metric, sd_theoretical in (
    analytical_sd.items()
):

    values = allometric_residual_mc[
        metric
    ].to_numpy()

    simulated_sd = np.std(
        values,
        ddof=1
    )

    simulated_mean = np.mean(values)

    mean_mcse = (
        simulated_sd / np.sqrt(N_SIM)
    )

    validation_rows.append({
        "metric": metric,
        "expected_mean": baseline_values[metric],
        "simulated_mean": simulated_mean,
        "analytical_SD": sd_theoretical,
        "simulated_SD": simulated_sd,
        "relative_SD_error_pct": (
            100
            * (simulated_sd - sd_theoretical)
            / sd_theoretical
        ),
        "mean_difference_in_MCSE": (
            (
                simulated_mean
                - baseline_values[metric]
            )
            / mean_mcse
        )
    })

allometric_residual_validation = (
    pd.DataFrame(validation_rows)
)

print("\nANALYTICAL VALIDATION")

display(
    allometric_residual_validation.round(5)
)

# ------------------------------------------------------------
# 8. EFFECTIVE NUMBER OF BIOMASS CONTRIBUTORS
# ------------------------------------------------------------

# n_eff = (sum AGB)^2 / sum(AGB^2)
#
# This describes concentration of biomass,
# not independent ecological replication.

effective_rows = []

for forest in FOREST_ORDER:

    values = agb_baseline_Mg[
        forest_array == forest
    ]

    n_eff = (
        values.sum()**2
        / np.sum(values**2)
    )

    effective_rows.append({
        "forest_type": forest,
        "n_trees": len(values),
        "effective_biomass_contributors": n_eff,
        "total_AGB_Mg": values.sum(),
        "relative_residual_SD_pct": (
            100 * residual_cv
            / np.sqrt(n_eff)
        )
    })

effective_biomass_contributors = (
    pd.DataFrame(effective_rows)
)

print("\nBIOMASS CONCENTRATION AND ERROR AGGREGATION")

display(
    effective_biomass_contributors.round(4)
)

# ------------------------------------------------------------
# 9. COMPARE WITH PREVIOUS BIOMETRIC UNCERTAINTY
# ------------------------------------------------------------

previous_summary = pd.read_csv(
    ROOT
    / "03_ANALYSIS"
    / "FUNCTIONAL_ECOLOGY"
    / "BIOMASS_BIOMETRIC_MC_SUMMARY.csv"
)

previous_gap = previous_summary.loc[
    previous_summary["metric"].eq(
        "biomass_gap_Mg"
    )
].copy()

comparison_rows = []

for _, row in previous_gap.iterrows():

    comparison_rows.append({
        "uncertainty_source": (
            f"Biometric_{row['scenario']}"
        ),
        "mean_gap_Mg": row["mean"],
        "SD_gap_Mg": row["SD"],
        "LCL_95_Mg": row["LCL_95"],
        "UCL_95_Mg": row["UCL_95"]
    })

residual_gap_row = (
    allometric_residual_summary.loc[
        allometric_residual_summary[
            "metric"
        ].eq("biomass_gap_Mg")
    ].iloc[0]
)

comparison_rows.append({
    "uncertainty_source": (
        "Independent_allometric_residual"
    ),
    "mean_gap_Mg": residual_gap_row["mean"],
    "SD_gap_Mg": residual_gap_row["SD"],
    "LCL_95_Mg": residual_gap_row["LCL_95"],
    "UCL_95_Mg": residual_gap_row["UCL_95"]
})

allometric_uncertainty_comparison = (
    pd.DataFrame(comparison_rows)
)

print("\nCOMPARISON OF UNCERTAINTY SOURCES")

display(
    allometric_uncertainty_comparison.round(4)
)

# ------------------------------------------------------------
# 10. PUBLICATION-QUALITY FIGURE
# ------------------------------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

fig, axes = plt.subplots(
    1, 3,
    figsize=(15, 4.5),
    constrained_layout=True
)

# Panel A — Plot biomass distributions

ax = axes[0]

for forest, metric, label in [
    (
        "Late_secondary",
        "secondary_AGB_Mg",
        "Late-secondary"
    ),
    (
        "Mature",
        "mature_AGB_Mg",
        "Mature"
    )
]:

    values = allometric_residual_mc[
        metric
    ]

    ax.hist(
        values,
        bins=55,
        density=True,
        alpha=0.65,
        color=COLORS[forest],
        label=label
    )

    ax.axvline(
        baseline_values[metric],
        color=COLORS[forest],
        linewidth=1.6,
        linestyle="--"
    )

ax.set_xlabel(
    "Aboveground biomass (Mg)"
)

ax.set_ylabel(
    "Probability density"
)

ax.set_title(
    "(a) Plot-level biomass",
    loc="left",
    fontweight="bold"
)

ax.legend(
    frameon=False
)

# Panel B — Biomass gap

ax = axes[1]

gap_values = allometric_residual_mc[
    "biomass_gap_Mg"
]

ax.hist(
    gap_values,
    bins=55,
    density=True,
    color=COLORS["Gap"],
    alpha=0.75
)

ax.axvline(
    baseline_gap,
    color="black",
    linestyle="--",
    linewidth=1.5,
    label="Baseline"
)

ax.axvspan(
    residual_gap_row["LCL_95"],
    residual_gap_row["UCL_95"],
    color=COLORS["Gap"],
    alpha=0.13,
    label="95% simulation interval"
)

ax.set_xlabel(
    "Secondary − mature biomass (Mg)"
)

ax.set_ylabel(
    "Probability density"
)

ax.set_title(
    "(b) Biomass difference",
    loc="left",
    fontweight="bold"
)

ax.legend(
    frameon=False,
    fontsize=8
)

# Panel C — Uncertainty source comparison

ax = axes[2]

plot_comparison = (
    allometric_uncertainty_comparison
    .set_index("uncertainty_source")
    .loc[
        [
            "Biometric_Low",
            "Biometric_Moderate",
            "Biometric_High",
            "Independent_allometric_residual"
        ]
    ]
    .reset_index()
)

plot_labels = [
    "Biometric: low",
    "Biometric: moderate",
    "Biometric: high",
    "Allometric residual"
]

for i, row in plot_comparison.iterrows():

    center = row["mean_gap_Mg"]

    lower_error = (
        center - row["LCL_95_Mg"]
    )

    upper_error = (
        row["UCL_95_Mg"] - center
    )

    ax.errorbar(
        center,
        i,
        xerr=[
            [lower_error],
            [upper_error]
        ],
        fmt="o",
        markersize=6,
        capsize=4,
        linewidth=1.6,
        color=(
            COLORS["Gap"]
            if i == 3
            else "#527B98"
        )
    )

ax.axvline(
    baseline_gap,
    color="black",
    linestyle="--",
    linewidth=1
)

ax.axvline(
    0,
    color="grey",
    linewidth=0.8
)

ax.set_yticks(
    range(len(plot_labels)),
    plot_labels
)

ax.invert_yaxis()

ax.set_xlabel(
    "Biomass difference (Mg)"
)

ax.set_title(
    "(c) Uncertainty comparison",
    loc="left",
    fontweight="bold"
)

figure_stem = (
    FIG_OUTPUT
    / "FIG_ALLOMETRIC_RESIDUAL_UNCERTAINTY"
)

fig.savefig(
    figure_stem.with_suffix(".pdf"),
    bbox_inches="tight"
)

fig.savefig(
    figure_stem.with_suffix(".svg"),
    bbox_inches="tight"
)

fig.savefig(
    figure_stem.with_suffix(".png"),
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# ------------------------------------------------------------
# 11. EXPORT ALL RESULTS
# ------------------------------------------------------------

export_objects = {
    "ALLOMETRIC_RESIDUAL_MC_DRAWS.csv":
        allometric_residual_mc,

    "ALLOMETRIC_RESIDUAL_MC_SUMMARY.csv":
        allometric_residual_summary,

    "ALLOMETRIC_RESIDUAL_VALIDATION.csv":
        allometric_residual_validation,

    "EFFECTIVE_BIOMASS_CONTRIBUTORS.csv":
        effective_biomass_contributors,

    "ALLOMETRIC_UNCERTAINTY_COMPARISON.csv":
        allometric_uncertainty_comparison
}

for filename, dataframe in (
    export_objects.items()
):

    dataframe.to_csv(
        MC_OUTPUT / filename,
        index=False
    )

    print(
        f"Saved: {filename}"
    )

# Save reproducible model configuration

import json

residual_config = {
    "equation": (
        "AGB_kg = 0.0673 * "
        "(WD * DBH_cm**2 * H_m)**0.976"
    ),
    "reference": "Chave et al. (2014), Eq. 4",
    "doi": "10.1111/gcb.12629",
    "sigma_log_AGB": sigma,
    "residual_log_mean": residual_log_mean,
    "residual_CV": residual_cv,
    "residual_dependence": "Independent trees",
    "n_simulations": N_SIM,
    "seed": SEED_ALLOMETRIC,
    "inventory_size": len(residual_inventory),
    "fixed_inventory": True,
    "biometric_uncertainty_included": False,
    "subplot_bootstrap_included": False,
    "parameter_uncertainty_included": False
}

with open(
    MC_OUTPUT
    / "ALLOMETRIC_RESIDUAL_CONFIG.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        residual_config,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 12. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("ALLOMETRIC RESIDUAL UNCERTAINTY")
print("=" * 75)

print(
    f"Simulations: {N_SIM:,}"
)

print(
    f"Published residual SD: {sigma:.3f}"
)

print(
    f"Baseline biomass gap: "
    f"{baseline_gap:.3f} Mg"
)

print(
    f"Mean simulated gap: "
    f"{residual_gap_row['mean']:.3f} Mg"
)

print(
    f"95% simulation interval: "
    f"{residual_gap_row['LCL_95']:.3f} "
    f"to "
    f"{residual_gap_row['UCL_95']:.3f} Mg"
)

print(
    "Positive biomass gaps: "
    f"{residual_gap_row['above_threshold_pct']:.2f}%"
)

print("\nOutputs:")
print(MC_OUTPUT)

print(
    "\nCELL 83 COMPLETED SUCCESSFULLY"
)


In [ ]:

# ============================================================
# CELL 84 — INTEGRATED MONTE CARLO UNCERTAINTY
# SATIPO FOREST PROJECT
# Biometric + allometric + subplot uncertainty
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

from tqdm.auto import tqdm
from itertools import product

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

MC_OUTPUT = (
    ROOT
    / "03_ANALYSIS"
    / "INTEGRATED_UNCERTAINTY"
)

FIG_OUTPUT = (
    ROOT / "05_OUTPUTS" / "FIGURES"
)

MC_OUTPUT.mkdir(parents=True, exist_ok=True)
FIG_OUTPUT.mkdir(parents=True, exist_ok=True)

N_INTEGRATED = 10000
SEED_INTEGRATED = 20261118

SIGMA_ALLOMETRIC = 0.357

RESIDUAL_CORRELATIONS = [
    0.00,
    0.10,
    0.25
]

INVENTORY_MODES = [
    "Fixed_inventory",
    "Subplot_bootstrap"
]

ERROR_SCENARIOS = {
    "Low": {
        "dbh_cv": 0.01,
        "height_cv": 0.05,
        "wd_cv": 0.05
    },
    "Moderate": {
        "dbh_cv": 0.02,
        "height_cv": 0.10,
        "wd_cv": 0.10
    },
    "High": {
        "dbh_cv": 0.05,
        "height_cv": 0.20,
        "wd_cv": 0.20
    }
}

rng_integrated = np.random.default_rng(
    SEED_INTEGRATED
)

# ------------------------------------------------------------
# 2. RECOVER VALIDATED ANALYTICAL INVENTORY
# ------------------------------------------------------------

integrated_inventory = (
    residual_inventory.copy()
    .reset_index(drop=True)
)

required_columns = [
    "tree_id",
    "forest_type",
    "subplot_id",
    "dbh_cm",
    "height_m",
    "wood_density_g_cm3",
    "wd_reference_taxon",
    "agb_kg"
]

assert all(
    c in integrated_inventory.columns
    for c in required_columns
)

assert len(integrated_inventory) == 1002

assert integrated_inventory[
    required_columns
].notna().all().all()

assert integrated_inventory[
    "tree_id"
].is_unique

d0 = integrated_inventory[
    "dbh_cm"
].to_numpy(dtype=float)

h0 = integrated_inventory[
    "height_m"
].to_numpy(dtype=float)

rho0 = integrated_inventory[
    "wood_density_g_cm3"
].to_numpy(dtype=float)

forest = integrated_inventory[
    "forest_type"
].to_numpy()

sec_mask = (
    forest == "Late_secondary"
)

mat_mask = (
    forest == "Mature"
)

assert sec_mask.sum() == 541
assert mat_mask.sum() == 461

n_total = len(integrated_inventory)

# Forest codes:
# 0 = late-secondary
# 1 = mature

forest_codes = np.where(
    sec_mask,
    0,
    1
)

# ------------------------------------------------------------
# 3. WOOD DENSITY ERROR GROUPS
# ------------------------------------------------------------

# Preserve shared WD uncertainty among trees
# assigned the same reference taxon.

wd_groups, wd_codes = np.unique(
    integrated_inventory[
        "wd_reference_taxon"
    ].astype(str),
    return_inverse=True
)

n_wd_groups = len(wd_groups)

assert n_wd_groups == 137, (
    f"Expected 137 WD groups, "
    f"found {n_wd_groups}"
)

# ------------------------------------------------------------
# 4. SUBPLOT IDENTIFIERS
# ------------------------------------------------------------

subplot_labels, subplot_codes = np.unique(
    integrated_inventory[
        "subplot_id"
    ].astype(str),
    return_inverse=True
)

n_subplots = len(subplot_labels)

assert n_subplots == 50

sec_subplot_codes = np.unique(
    subplot_codes[sec_mask]
)

mat_subplot_codes = np.unique(
    subplot_codes[mat_mask]
)

assert len(sec_subplot_codes) == 25
assert len(mat_subplot_codes) == 25

assert len(
    np.intersect1d(
        sec_subplot_codes,
        mat_subplot_codes
    )
) == 0

# ------------------------------------------------------------
# 5. BASELINE BIOMASS VALIDATION
# ------------------------------------------------------------

baseline_tree_agb = (
    0.0673
    * (rho0 * d0**2 * h0)**0.976
    / 1000
)

assert np.allclose(
    baseline_tree_agb,
    integrated_inventory[
        "agb_kg"
    ].to_numpy() / 1000,
    rtol=1e-6,
    atol=1e-9
)

baseline_sec = (
    baseline_tree_agb[sec_mask].sum()
)

baseline_mat = (
    baseline_tree_agb[mat_mask].sum()
)

baseline_gap = baseline_sec - baseline_mat
baseline_ratio = baseline_sec / baseline_mat

print("=" * 75)
print("INTEGRATED MONTE CARLO CONFIGURATION")
print("=" * 75)

print(f"Analytical trees: {n_total}")
print(f"Wood density groups: {n_wd_groups}")
print(f"Subplots: {n_subplots}")
print(f"Secondary AGB: {baseline_sec:.4f} Mg")
print(f"Mature AGB: {baseline_mat:.4f} Mg")
print(f"Baseline gap: {baseline_gap:.4f} Mg")

# ------------------------------------------------------------
# 6. HELPER FUNCTIONS
# ------------------------------------------------------------

def mean_one_lognormal(cv, size, rng):

    sigma = np.sqrt(
        np.log1p(cv**2)
    )

    return rng.lognormal(
        mean=-0.5 * sigma**2,
        sigma=sigma,
        size=size
    )


def correlated_allometric_multiplier(
    correlation,
    rng
):

    # Independent tree-level standard normals

    z_tree = rng.standard_normal(
        n_total
    )

    # Independent plot-level standard normals.
    # Shared within each plot.

    z_plot = rng.standard_normal(2)

    log_error = (
        SIGMA_ALLOMETRIC
        * (
            np.sqrt(correlation)
            * z_plot[forest_codes]
            +
            np.sqrt(1 - correlation)
            * z_tree
        )
        -
        0.5 * SIGMA_ALLOMETRIC**2
    )

    return np.exp(log_error)


def bootstrap_subplot_weights(rng):

    # Resample 25 subplots independently
    # within each of the two observed plots.

    sec_draw = rng.choice(
        sec_subplot_codes,
        size=25,
        replace=True
    )

    mat_draw = rng.choice(
        mat_subplot_codes,
        size=25,
        replace=True
    )

    sampled_codes = np.concatenate([
        sec_draw,
        mat_draw
    ])

    subplot_weights = np.bincount(
        sampled_codes,
        minlength=n_subplots
    )

    assert subplot_weights.sum() == 50

    assert (
        subplot_weights[
            sec_subplot_codes
        ].sum() == 25
    )

    assert (
        subplot_weights[
            mat_subplot_codes
        ].sum() == 25
    )

    # Repeated subplots retain the same
    # simulated tree-level measurement errors.

    return subplot_weights[
        subplot_codes
    ]


# ------------------------------------------------------------
# 7. SIMULATION DESIGN
# ------------------------------------------------------------

simulation_design = list(
    product(
        ERROR_SCENARIOS.keys(),
        RESIDUAL_CORRELATIONS,
        INVENTORY_MODES
    )
)

n_combinations = len(
    simulation_design
)

print(
    f"\nScenario combinations: "
    f"{n_combinations}"
)

print(
    f"Total simulations: "
    f"{n_combinations*N_INTEGRATED:,}"
)

# ------------------------------------------------------------
# 8. INTEGRATED MONTE CARLO
# ------------------------------------------------------------

integrated_frames = []

for (
    scenario,
    correlation,
    inventory_mode
) in simulation_design:

    params = ERROR_SCENARIOS[
        scenario
    ]

    results = np.empty(
        (N_INTEGRATED, 6),
        dtype=float
    )

    description = (
        f"{scenario} | "
        f"r={correlation:.2f} | "
        f"{inventory_mode}"
    )

    for b in tqdm(
        range(N_INTEGRATED),
        desc=description,
        leave=False
    ):

        # A. Independent DBH measurement error

        d_sim = (
            d0
            * mean_one_lognormal(
                params["dbh_cv"],
                n_total,
                rng_integrated
            )
        )

        # B. Independent height measurement error

        h_sim = (
            h0
            * mean_one_lognormal(
                params["height_cv"],
                n_total,
                rng_integrated
            )
        )

        # C. Shared wood density assignment error

        wd_group_multiplier = (
            mean_one_lognormal(
                params["wd_cv"],
                n_wd_groups,
                rng_integrated
            )
        )

        rho_sim = (
            rho0
            * wd_group_multiplier[
                wd_codes
            ]
        )

        # D. Biomass equation

        agb_sim = (
            0.0673
            * (
                rho_sim
                * d_sim**2
                * h_sim
            )**0.976
            / 1000
        )

        # E. Allometric residual uncertainty

        agb_sim *= (
            correlated_allometric_multiplier(
                correlation,
                rng_integrated
            )
        )

        # F. Conditional subplot resampling

        if inventory_mode == "Subplot_bootstrap":

            tree_weights = (
                bootstrap_subplot_weights(
                    rng_integrated
                )
            )

            agb_sim *= tree_weights

        # G. Plot-level aggregation

        sec_agb = agb_sim[
            sec_mask
        ].sum()

        mat_agb = agb_sim[
            mat_mask
        ].sum()

        gap = sec_agb - mat_agb
        ratio = sec_agb / mat_agb

        results[b] = [
            b + 1,
            sec_agb,
            mat_agb,
            gap,
            ratio,
            100 * (ratio - 1)
        ]

    scenario_df = pd.DataFrame(
        results,
        columns=[
            "iteration",
            "secondary_AGB_Mg",
            "mature_AGB_Mg",
            "biomass_gap_Mg",
            "biomass_ratio",
            "secondary_excess_pct"
        ]
    )

    scenario_df["iteration"] = (
        scenario_df["iteration"]
        .astype(int)
    )

    scenario_df.insert(
        0,
        "biometric_scenario",
        scenario
    )

    scenario_df.insert(
        1,
        "residual_correlation",
        correlation
    )

    scenario_df.insert(
        2,
        "inventory_mode",
        inventory_mode
    )

    integrated_frames.append(
        scenario_df
    )

    print(
        f"Completed: {description} | "
        f"Positive gaps: "
        f"{100*(scenario_df['biomass_gap_Mg'] > 0).mean():.2f}%"
    )

integrated_mc = pd.concat(
    integrated_frames,
    ignore_index=True
)

assert len(integrated_mc) == (
    N_INTEGRATED
    * n_combinations
)

assert np.isfinite(
    integrated_mc.select_dtypes(
        include=np.number
    )
).all().all()

# ------------------------------------------------------------
# 9. SUMMARIZE ALL SIMULATIONS
# ------------------------------------------------------------

summary_metrics = [
    "secondary_AGB_Mg",
    "mature_AGB_Mg",
    "biomass_gap_Mg",
    "biomass_ratio",
    "secondary_excess_pct"
]

group_columns = [
    "biometric_scenario",
    "residual_correlation",
    "inventory_mode"
]

integrated_summary_rows = []

for keys, group in integrated_mc.groupby(
    group_columns,
    sort=False
):

    scenario, correlation, inventory_mode = keys

    for metric in summary_metrics:

        values = group[
            metric
        ].to_numpy()

        threshold = (
            1.0
            if metric == "biomass_ratio"
            else 0.0
        )

        sd = np.std(
            values,
            ddof=1
        )

        integrated_summary_rows.append({
            "biometric_scenario": scenario,
            "residual_correlation": correlation,
            "inventory_mode": inventory_mode,
            "metric": metric,
            "n_simulations": len(values),
            "mean": np.mean(values),
            "median": np.median(values),
            "SD": sd,
            "LCL_95": np.quantile(
                values, 0.025
            ),
            "UCL_95": np.quantile(
                values, 0.975
            ),
            "above_threshold_pct": (
                100 * np.mean(
                    values > threshold
                )
            ),
            "MC_standard_error": (
                sd / np.sqrt(len(values))
            )
        })

integrated_mc_summary = pd.DataFrame(
    integrated_summary_rows
)

# ------------------------------------------------------------
# 10. PAPER-READY BIOMASS GAP SUMMARY
# ------------------------------------------------------------

integrated_gap_summary = (
    integrated_mc_summary.loc[
        integrated_mc_summary[
            "metric"
        ].eq("biomass_gap_Mg")
    ].copy()
)

integrated_gap_summary[
    "interval_width_Mg"
] = (
    integrated_gap_summary["UCL_95"]
    -
    integrated_gap_summary["LCL_95"]
)

integrated_gap_summary[
    "interval_excludes_zero"
] = (
    integrated_gap_summary[
        "LCL_95"
    ] > 0
)

integrated_gap_summary[
    "nonpositive_draws_pct"
] = (
    100
    -
    integrated_gap_summary[
        "above_threshold_pct"
    ]
)

print("\n" + "=" * 75)
print("INTEGRATED BIOMASS GAP SUMMARY")
print("=" * 75)

display(
    integrated_gap_summary[
        [
            "biometric_scenario",
            "residual_correlation",
            "inventory_mode",
            "mean",
            "median",
            "SD",
            "LCL_95",
            "UCL_95",
            "above_threshold_pct",
            "interval_excludes_zero"
        ]
    ].round(4)
)

# ------------------------------------------------------------
# 11. SELECT PRINCIPAL SCENARIOS
# ------------------------------------------------------------

# Primary scenario:
# Moderate biometric uncertainty,
# independent residuals, fixed inventory.

primary_scenario = (
    integrated_gap_summary.loc[
        (
            integrated_gap_summary[
                "biometric_scenario"
            ].eq("Moderate")
        )
        &
        (
            integrated_gap_summary[
                "residual_correlation"
            ].eq(0.0)
        )
        &
        (
            integrated_gap_summary[
                "inventory_mode"
            ].eq("Fixed_inventory")
        )
    ]
)

# Stress-test scenario:
# High biometric uncertainty,
# correlated residuals, subplot bootstrap.

stress_scenario = (
    integrated_gap_summary.loc[
        (
            integrated_gap_summary[
                "biometric_scenario"
            ].eq("High")
        )
        &
        (
            integrated_gap_summary[
                "residual_correlation"
            ].eq(0.25)
        )
        &
        (
            integrated_gap_summary[
                "inventory_mode"
            ].eq("Subplot_bootstrap")
        )
    ]
)

assert len(primary_scenario) == 1
assert len(stress_scenario) == 1

print("\nPRIMARY INTEGRATED SCENARIO")

display(
    primary_scenario.round(4)
)

print("\nCOMBINED STRESS-TEST SCENARIO")

display(
    stress_scenario.round(4)
)

# ------------------------------------------------------------
# 12. ANALYTICAL CORRELATION SENSITIVITY
# ------------------------------------------------------------

# Exact residual-only variance for correlated
# lognormal errors within each plot.
#
# Cov(M_i, M_j) =
# exp(r*sigma^2) - 1

analytical_correlation_rows = []

sigma2 = SIGMA_ALLOMETRIC**2

v_individual = np.expm1(
    sigma2
)

for correlation in RESIDUAL_CORRELATIONS:

    covariance = np.expm1(
        correlation * sigma2
    )

    plot_variances = {}

    for forest_name, mask in [
        ("Late_secondary", sec_mask),
        ("Mature", mat_mask)
    ]:

        values = baseline_tree_agb[
            mask
        ]

        sum_squares = np.sum(
            values**2
        )

        total = values.sum()

        variance = (
            v_individual * sum_squares
            +
            covariance
            * (
                total**2
                - sum_squares
            )
        )

        plot_variances[
            forest_name
        ] = variance

        analytical_correlation_rows.append({
            "residual_correlation": correlation,
            "metric": forest_name,
            "baseline_Mg": total,
            "analytical_SD_Mg": np.sqrt(
                variance
            ),
            "relative_SD_pct": (
                100 * np.sqrt(variance)
                / total
            )
        })

    # Plot-level shared errors are independent
    # between the two forests.

    gap_variance = (
        plot_variances[
            "Late_secondary"
        ]
        +
        plot_variances[
            "Mature"
        ]
    )

    analytical_correlation_rows.append({
        "residual_correlation": correlation,
        "metric": "Biomass_gap",
        "baseline_Mg": baseline_gap,
        "analytical_SD_Mg": np.sqrt(
            gap_variance
        ),
        "relative_SD_pct": (
            100 * np.sqrt(gap_variance)
            / baseline_gap
        )
    })

analytical_correlation_summary = (
    pd.DataFrame(
        analytical_correlation_rows
    )
)

print("\nANALYTICAL RESIDUAL CORRELATION SENSITIVITY")

display(
    analytical_correlation_summary.round(4)
)

# ------------------------------------------------------------
# 13. PUBLICATION FIGURE
# ------------------------------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

scenario_colors = {
    "Low": "#527B98",
    "Moderate": "#795C92",
    "High": "#DB9140"
}

fig, axes = plt.subplots(
    1, 3,
    figsize=(16, 5.4),
    constrained_layout=True
)

# Panels A and B:
# Integrated biomass difference intervals

for ax, mode, panel_title in [
    (
        axes[0],
        "Fixed_inventory",
        "(a) Fixed inventory"
    ),
    (
        axes[1],
        "Subplot_bootstrap",
        "(b) Subplot resampling"
    )
]:

    plot_data = (
        integrated_gap_summary.loc[
            integrated_gap_summary[
                "inventory_mode"
            ].eq(mode)
        ]
    )

    labels = []
    y = 0

    for correlation in (
        RESIDUAL_CORRELATIONS
    ):

        for scenario in [
            "Low",
            "Moderate",
            "High"
        ]:

            row = plot_data.loc[
                (
                    plot_data[
                        "residual_correlation"
                    ].eq(correlation)
                )
                &
                (
                    plot_data[
                        "biometric_scenario"
                    ].eq(scenario)
                )
            ].iloc[0]

            center = row["median"]

            ax.errorbar(
                center,
                y,
                xerr=[
                    [
                        center
                        - row["LCL_95"]
                    ],
                    [
                        row["UCL_95"]
                        - center
                    ]
                ],
                fmt="o",
                color=scenario_colors[
                    scenario
                ],
                markersize=6,
                capsize=3,
                linewidth=1.7
            )

            labels.append(
                f"{scenario}, r={correlation:.2f}"
            )

            y += 1

    ax.axvline(
        0,
        color="#777777",
        linewidth=1
    )

    ax.axvline(
        baseline_gap,
        color="black",
        linestyle="--",
        linewidth=1.1
    )

    ax.set_yticks(
        range(len(labels)),
        labels
    )

    ax.invert_yaxis()

    ax.set_xlabel(
        "Secondary − mature AGB (Mg)"
    )

    ax.set_title(
        panel_title,
        loc="left",
        fontweight="bold"
    )

# Common horizontal scale

x_min = min(
    0,
    integrated_gap_summary[
        "LCL_95"
    ].min()
)

x_max = (
    integrated_gap_summary[
        "UCL_95"
    ].max()
)

padding = (
    0.05 * (x_max - x_min)
)

for ax in axes[:2]:

    ax.set_xlim(
        x_min - padding,
        x_max + padding
    )

# Panel C:
# Analytical sensitivity to residual correlation

ax = axes[2]

for metric, label, color in [
    (
        "Late_secondary",
        "Late-secondary",
        "#DB9140"
    ),
    (
        "Mature",
        "Mature",
        "#24634F"
    ),
    (
        "Biomass_gap",
        "Biomass difference",
        "#795C92"
    )
]:

    data = (
        analytical_correlation_summary.loc[
            analytical_correlation_summary[
                "metric"
            ].eq(metric)
        ]
    )

    ax.plot(
        data["residual_correlation"],
        data["analytical_SD_Mg"],
        marker="o",
        linewidth=2,
        markersize=6,
        color=color,
        label=label
    )

ax.set_xlabel(
    "Within-plot residual correlation"
)

ax.set_ylabel(
    "Analytical residual SD (Mg)"
)

ax.set_xticks(
    RESIDUAL_CORRELATIONS
)

ax.set_title(
    "(c) Correlated residual uncertainty",
    loc="left",
    fontweight="bold"
)

ax.legend(
    frameon=False,
    fontsize=9
)

figure_stem = (
    FIG_OUTPUT
    / "FIG_INTEGRATED_MONTE_CARLO"
)

for extension in [
    ".pdf",
    ".svg",
    ".png"
]:

    fig.savefig(
        figure_stem.with_suffix(
            extension
        ),
        dpi=600,
        bbox_inches="tight"
    )

plt.show()
plt.close(fig)

# ------------------------------------------------------------
# 14. EXPORT RESULTS
# ------------------------------------------------------------

export_tables = {
    "INTEGRATED_MC_DRAWS.csv":
        integrated_mc,

    "INTEGRATED_MC_SUMMARY.csv":
        integrated_mc_summary,

    "INTEGRATED_MC_GAP_SUMMARY.csv":
        integrated_gap_summary,

    "INTEGRATED_MC_PRIMARY_SCENARIO.csv":
        primary_scenario,

    "INTEGRATED_MC_STRESS_SCENARIO.csv":
        stress_scenario,

    "ALLOMETRIC_CORRELATION_ANALYTICAL.csv":
        analytical_correlation_summary
}

for filename, dataframe in (
    export_tables.items()
):

    dataframe.to_csv(
        MC_OUTPUT / filename,
        index=False
    )

    print(f"Saved: {filename}")

# Excel workbook

excel_path = (
    MC_OUTPUT
    / "SATIPO_INTEGRATED_UNCERTAINTY.xlsx"
)

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    integrated_gap_summary.to_excel(
        writer,
        sheet_name="Biomass_gap",
        index=False
    )

    integrated_mc_summary.to_excel(
        writer,
        sheet_name="All_metrics",
        index=False
    )

    analytical_correlation_summary.to_excel(
        writer,
        sheet_name="Residual_correlation",
        index=False
    )

    primary_scenario.to_excel(
        writer,
        sheet_name="Primary_scenario",
        index=False
    )

    stress_scenario.to_excel(
        writer,
        sheet_name="Stress_test",
        index=False
    )

# ------------------------------------------------------------
# 15. SAVE REPRODUCIBLE CONFIGURATION
# ------------------------------------------------------------

integrated_config = {
    "n_trees": n_total,
    "n_subplots": n_subplots,
    "n_wood_density_groups": n_wd_groups,
    "seed": SEED_INTEGRATED,
    "n_simulations_per_combination": (
        N_INTEGRATED
    ),
    "total_simulations": len(
        integrated_mc
    ),
    "biometric_scenarios": ERROR_SCENARIOS,
    "allometric_sigma_log": SIGMA_ALLOMETRIC,
    "residual_correlations": (
        RESIDUAL_CORRELATIONS
    ),
    "inventory_modes": INVENTORY_MODES,
    "wood_density_dependence": (
        "Shared by reference taxon"
    ),
    "dbh_height_dependence": (
        "Independent errors between trees "
        "and between biometric variables"
    ),
    "allometric_dependence": (
        "Exchangeable within-plot correlation; "
        "independent shared errors between plots"
    ),
    "bootstrap_design": (
        "25 subplots resampled with replacement "
        "within each observed 1-ha plot"
    ),
    "bootstrap_interpretation": (
        "Conditional spatial heterogeneity; "
        "not independent forest-type replication"
    ),
    "correlation_interpretation": (
        "Hypothetical sensitivity scenarios; "
        "not locally estimated"
    ),
    "parameter_covariance_included": False,
    "primary_scenario": {
        "biometric": "Moderate",
        "residual_correlation": 0.0,
        "inventory": "Fixed_inventory"
    }
}

with open(
    MC_OUTPUT / "INTEGRATED_MC_CONFIG.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        integrated_config,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 16. FINAL REPORT
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("INTEGRATED MONTE CARLO — FINAL REPORT")
print("=" * 75)

print(
    f"Total simulations: "
    f"{len(integrated_mc):,}"
)

print(
    f"Scenario combinations: "
    f"{n_combinations}"
)

print(
    "Scenarios with 95% interval "
    "entirely above zero:",
    integrated_gap_summary[
        "interval_excludes_zero"
    ].sum(),
    "/",
    n_combinations
)

print(
    "\nMinimum positive-gap frequency:",
    round(
        integrated_gap_summary[
            "above_threshold_pct"
        ].min(),
        3
    ),
    "%"
)

print(
    "\nPrimary scenario:"
)

print(
    primary_scenario[
        [
            "median",
            "LCL_95",
            "UCL_95",
            "above_threshold_pct"
        ]
    ].to_string(index=False)
)

print(
    "\nCombined stress-test scenario:"
)

print(
    stress_scenario[
        [
            "median",
            "LCL_95",
            "UCL_95",
            "above_threshold_pct"
        ]
    ].to_string(index=False)
)

print("\nOutputs:")
print(MC_OUTPUT)

print(
    "\nCELL 84 COMPLETED SUCCESSFULLY"
)


In [ ]:

from pathlib import Path
import json
import pandas as pd

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

CHECKPOINT_DIR = ROOT / "06_CHECKPOINTS"

print("=" * 70)
print("SATIPO — CHECKPOINT DIAGNOSTIC")
print("=" * 70)

print("Project exists:", ROOT.exists())
print("Checkpoint directory exists:", CHECKPOINT_DIR.exists())

# Buscar todos los manifiestos disponibles

manifests = sorted(
    CHECKPOINT_DIR.rglob("CHECKPOINT_MANIFEST.json"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

print("\nCHECKPOINTS FOUND:", len(manifests))

for i, path in enumerate(manifests, 1):

    print(f"\nCheckpoint {i}")
    print("Path:", path)

    try:

        with open(path, encoding="utf-8") as f:
            manifest = json.load(f)

        print(
            "Last analysis cell:",
            manifest.get("last_analysis_cell", "Unknown")
        )

        print(
            "Saved objects:",
            len(manifest.get("saved_objects", []))
        )

        print(
            "Failed objects:",
            len(manifest.get("failed_objects", []))
        )

        print(
            "Verification failures:",
            len(manifest.get("verification_failures", []))
        )

    except Exception as e:

        print("Manifest error:", e)

# Verificar los resultados científicos persistentes

MC_DIR = (
    ROOT / "03_ANALYSIS" / "INTEGRATED_UNCERTAINTY"
)

critical_outputs = [
    "ALLOMETRIC_RESIDUAL_MC_SUMMARY.csv",
    "ALLOMETRIC_RESIDUAL_MC_DRAWS.csv",
    "INTEGRATED_MC_DRAWS.csv",
    "INTEGRATED_MC_SUMMARY.csv",
    "INTEGRATED_MC_GAP_SUMMARY.csv",
    "ALLOMETRIC_CORRELATION_ANALYTICAL.csv"
]

print("\n" + "=" * 70)
print("SCIENTIFIC OUTPUTS")
print("=" * 70)

for filename in critical_outputs:

    path = MC_DIR / filename

    if path.exists():

        print(
            "OK:",
            filename,
            "|",
            round(path.stat().st_size / 1024**2, 3),
            "MB"
        )

    else:

        print("MISSING:", filename)

# Verificar las simulaciones

draws_path = MC_DIR / "INTEGRATED_MC_DRAWS.csv"

if draws_path.exists():

    draws_check = pd.read_csv(draws_path)

    print("\nMonte Carlo realizations:", len(draws_check))

    if len(draws_check) == 180000:

        print("ALL 180,000 SIMULATIONS PRESERVED")

# Verificar los objetos todavía activos en RAM

print("\n" + "=" * 70)
print("ACTIVE PYTHON SESSION")
print("=" * 70)

active_dataframes = [
    name
    for name, obj in list(globals().items())
    if isinstance(obj, pd.DataFrame)
    and not name.startswith("_")
]

print("Active DataFrames:", len(active_dataframes))

print("\nExamples:")
print(active_dataframes[:30])

print("\nDIAGNOSTIC COMPLETED")


In [ ]:

# ============================================================
# CELL 85 — FINAL CHECKPOINT (CORRECTED VERSION)
# SATIPO FOREST PROJECT
# ============================================================

!pip -q install dill

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import dill
import json
import hashlib
import types
import sys
import importlib.metadata

from tqdm.auto import tqdm

# ============================================================
# 1. CONFIGURATION
# ============================================================

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

assert ROOT.exists(), "Project directory not found"

CHECKPOINT_DIR = ROOT / "06_CHECKPOINTS"

MC_DIR = (
    ROOT / "03_ANALYSIS"
    / "INTEGRATED_UNCERTAINTY"
)

RUN_ID = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_UTC")

FINAL_CHECKPOINT = (
    CHECKPOINT_DIR
    / f"STAGE_01_FINAL_{RUN_ID}"
)

OBJECT_DIR = (
    FINAL_CHECKPOINT / "python_objects"
)

TABLE_DIR = (
    FINAL_CHECKPOINT / "dataframe_backup"
)

for folder in [
    FINAL_CHECKPOINT,
    OBJECT_DIR,
    TABLE_DIR
]:

    folder.mkdir(
        parents=True,
        exist_ok=True
    )

print("=" * 75)
print("SATIPO — FINAL SCIENTIFIC CHECKPOINT")
print("=" * 75)

print("Checkpoint:", FINAL_CHECKPOINT)

# ============================================================
# 2. VERIFY CONFIRMED SCIENTIFIC FILES
# ============================================================

critical_files = [

    MC_DIR / "ALLOMETRIC_RESIDUAL_MC_SUMMARY.csv",

    MC_DIR / "ALLOMETRIC_RESIDUAL_MC_DRAWS.csv",

    MC_DIR / "INTEGRATED_MC_DRAWS.csv",

    MC_DIR / "INTEGRATED_MC_SUMMARY.csv",

    MC_DIR / "INTEGRATED_MC_GAP_SUMMARY.csv",

    MC_DIR / "ALLOMETRIC_CORRELATION_ANALYTICAL.csv"
]

missing_files = [
    str(path)
    for path in critical_files
    if not path.exists()
]

assert not missing_files, missing_files

final_mc_draws = pd.read_csv(
    MC_DIR / "INTEGRATED_MC_DRAWS.csv"
)

final_mc_summary = pd.read_csv(
    MC_DIR / "INTEGRATED_MC_GAP_SUMMARY.csv"
)

assert len(final_mc_draws) == 180000

assert len(final_mc_summary) == 18

assert np.isclose(
    final_mc_summary[
        "above_threshold_pct"
    ].min(),
    91.85,
    atol=0.01
)

print("\nSCIENTIFIC OUTPUTS VERIFIED")

print("Monte Carlo draws:", len(final_mc_draws))

print("Scenarios:", len(final_mc_summary))

# ============================================================
# 3. HASH FUNCTION
# ============================================================

def checkpoint_sha256(path):

    sha = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            sha.update(chunk)

    return sha.hexdigest()

# ============================================================
# 4. CAPTURE SCIENTIFIC OBJECTS
# ============================================================

checkpoint_namespace = list(
    globals().items()
)

checkpoint_allowed_types = (

    pd.DataFrame,
    pd.Series,
    np.ndarray,
    np.generic,
    dict,
    list,
    tuple,
    set,
    str,
    int,
    float,
    bool,
    Path
)

checkpoint_excluded_names = {

    "In",
    "Out",
    "exit",
    "quit",
    "get_ipython",
    "open",
    "ColabKernelApp",

    "checkpoint_namespace",
    "checkpoint_allowed_types",
    "checkpoint_excluded_names",

    "archive_namespace",
    "namespace_snapshot",

    "saved_objects",
    "failed_objects",
    "skipped_objects",

    "archive_saved",
    "archive_failed",
    "archive_skipped",

    "restored_objects",
    "restore_errors",

    "checkpoint_saved",
    "checkpoint_failed",
    "checkpoint_skipped",

    # Already saved as persistent CSV files
    "final_mc_draws",
    "draws_check"
}

checkpoint_saved = []
checkpoint_failed = []
checkpoint_skipped = []

# ============================================================
# 5. SAVE INDIVIDUAL OBJECTS
# ============================================================

for name, obj in tqdm(

    checkpoint_namespace,

    desc="Saving scientific objects"

):

    if (

        name.startswith("_")

        or name.startswith("checkpoint_")

        or name in checkpoint_excluded_names

        or isinstance(obj, types.ModuleType)

        or not isinstance(
            obj,
            checkpoint_allowed_types
        )

    ):

        checkpoint_skipped.append({
            "name": name,
            "type": type(obj).__name__
        })

        continue

    object_path = (
        OBJECT_DIR / f"{name}.pkl"
    )

    temporary_path = (
        OBJECT_DIR / f"{name}.tmp"
    )

    try:

        # Save object independently
        with open(
            temporary_path,
            "wb"
        ) as f:

            dill.dump(
                obj,
                f,
                recurse=False
            )

        temporary_path.replace(
            object_path
        )

        record = {

            "name": name,

            "type": type(obj).__name__,

            "file": str(
                object_path.relative_to(
                    FINAL_CHECKPOINT
                )
            ),

            "size_MB": round(
                object_path.stat().st_size
                / 1024**2,
                6
            ),

            "sha256": checkpoint_sha256(
                object_path
            )
        }

        # Additional portable backup
        if isinstance(obj, pd.DataFrame):

            csv_path = (
                TABLE_DIR / f"{name}.csv"
            )

            try:

                obj.to_csv(
                    csv_path,
                    index=True,
                    encoding="utf-8-sig"
                )

                record["csv_backup"] = str(
                    csv_path.relative_to(
                        FINAL_CHECKPOINT
                    )
                )

            except Exception as csv_error:

                record["csv_backup_error"] = (
                    str(csv_error)
                )

            record["rows"] = len(obj)

            record["columns"] = len(
                obj.columns
            )

        checkpoint_saved.append(record)

    except Exception as error:

        checkpoint_failed.append({

            "name": name,

            "type": type(obj).__name__,

            "error": str(error)
        })

        temporary_path.unlink(
            missing_ok=True
        )

print("\nOBJECT SAVE SUMMARY")

print("Saved:", len(checkpoint_saved))

print("Failed:", len(checkpoint_failed))

print("Skipped:", len(checkpoint_skipped))

# ============================================================
# 6. VERIFY ACTUAL DESERIALIZATION
# ============================================================

checkpoint_verification = []

for record in tqdm(

    checkpoint_saved,

    desc="Testing checkpoint recovery"

):

    object_path = (

        FINAL_CHECKPOINT
        / record["file"]

    )

    try:

        assert checkpoint_sha256(
            object_path
        ) == record["sha256"]

        with open(
            object_path,
            "rb"
        ) as f:

            test_obj = dill.load(f)

        assert type(
            test_obj
        ).__name__ == record["type"]

        if isinstance(
            test_obj,
            pd.DataFrame
        ):

            assert test_obj.shape == (

                record["rows"],

                record["columns"]
            )

        checkpoint_verification.append({

            "name": record["name"],

            "status": "OK"
        })

        del test_obj

    except Exception as error:

        checkpoint_verification.append({

            "name": record["name"],

            "status": "FAILED",

            "error": str(error)
        })

checkpoint_verification_df = pd.DataFrame(
    checkpoint_verification
)

checkpoint_verification_df.to_csv(

    FINAL_CHECKPOINT
    / "CHECKPOINT_VERIFICATION.csv",

    index=False
)

checkpoint_verification_failures = [

    record

    for record in checkpoint_verification

    if record["status"] != "OK"
]

# ============================================================
# 7. VERIFY ESSENTIAL DATAFRAMES
# ============================================================

saved_names = {

    record["name"]

    for record in checkpoint_saved
}

verified_names = {

    record["name"]

    for record in checkpoint_verification

    if record["status"] == "OK"
}

essential_objects = [

    "master",

    "inventory",

    "floristic",

    "subplot_master",

    "taxonomy_master",

    "master_v03",

    "master_v04",

    "tax_v04",

    "final_mc_summary"
]

essential_status = []

for name in essential_objects:

    essential_status.append({

        "object": name,

        "saved": name in saved_names,

        "verified": name in verified_names
    })

essential_status_df = pd.DataFrame(
    essential_status
)

print("\nESSENTIAL SCIENTIFIC OBJECTS")

display(essential_status_df)

essential_ok = bool(

    essential_status_df[
        "verified"
    ].all()
)

# ============================================================
# 8. SAVE SOFTWARE VERSIONS
# ============================================================

checkpoint_packages = [

    "dill",
    "numpy",
    "pandas",
    "scipy",
    "matplotlib",
    "scikit-learn",
    "statsmodels",
    "openpyxl",
    "requests",
    "tqdm",
    "pymupdf",
    "pillow"
]

checkpoint_versions = {}

for package in checkpoint_packages:

    try:

        checkpoint_versions[package] = (

            importlib.metadata.version(
                package
            )
        )

    except Exception:

        checkpoint_versions[package] = None

(
    FINAL_CHECKPOINT
    / "requirements_checkpoint.txt"
).write_text(

    "\n".join(

        f"{package}=={version}"

        for package, version
        in checkpoint_versions.items()

        if version is not None
    ),

    encoding="utf-8"
)

# ============================================================
# 9. CREATE FINAL MANIFEST
# ============================================================

checkpoint_manifest = {

    "project": "SATIPO_FOREST_PROJECT",

    "stage": "Stage 1 - Field analysis",

    "last_analysis_cell": 84,

    "archive_cell": 85,

    "next_cell": 86,

    "saved_at_utc": RUN_ID,

    "python_version": sys.version,

    "package_versions": checkpoint_versions,

    "checkpoint_directory": str(
        FINAL_CHECKPOINT
    ),

    "saved_objects": checkpoint_saved,

    "failed_objects": checkpoint_failed,

    "skipped_objects": checkpoint_skipped,

    "verification_failures":
        checkpoint_verification_failures,

    "essential_objects":
        essential_status,

    "critical_files": [

        {

            "path": str(
                path.relative_to(ROOT)
            ),

            "sha256": checkpoint_sha256(
                path
            ),

            "size_MB": round(
                path.stat().st_size
                / 1024**2,
                6
            )
        }

        for path in critical_files
    ],

    "scientific_results": {

        "biomass_inventory_trees": 1002,

        "secondary_AGB_Mg_ha": 355.226643,

        "mature_AGB_Mg_ha": 234.36256,

        "baseline_gap_Mg_ha": 120.86408,

        "integrated_MC_draws": 180000,

        "integrated_MC_scenarios": 18,

        "minimum_positive_gap_pct": 91.85
    }
}

manifest_path = (

    FINAL_CHECKPOINT
    / "CHECKPOINT_MANIFEST.json"

)

with open(

    manifest_path,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        checkpoint_manifest,

        f,

        indent=4,

        ensure_ascii=False
    )

# ============================================================
# 10. PUBLISH VERIFIED CHECKPOINT POINTER
# ============================================================

checkpoint_success = (

    len(checkpoint_saved) > 0

    and essential_ok

    and len(checkpoint_failed) == 0

    and len(
        checkpoint_verification_failures
    ) == 0
)

pointer_path = (

    CHECKPOINT_DIR
    / "LATEST_FINAL_CHECKPOINT.json"
)

if checkpoint_success:

    latest_pointer = {

        "checkpoint": str(
            FINAL_CHECKPOINT
        ),

        "manifest": str(
            manifest_path
        ),

        "stage": 1,

        "last_analysis_cell": 84,

        "archive_cell": 85,

        "next_cell": 86,

        "verified": True,

        "saved_objects": len(
            checkpoint_saved
        ),

        "failed_objects": 0,

        "verification_failures": 0
    }

    temporary_pointer = (

        CHECKPOINT_DIR
        / "LATEST_FINAL_CHECKPOINT.tmp"
    )

    with open(

        temporary_pointer,

        "w",

        encoding="utf-8"

    ) as f:

        json.dump(

            latest_pointer,

            f,

            indent=4
        )

    temporary_pointer.replace(
        pointer_path
    )

    # Verify the pointer itself
    with open(
        pointer_path,
        encoding="utf-8"
    ) as f:

        pointer_test = json.load(f)

    assert pointer_test["verified"]

    assert Path(
        pointer_test["manifest"]
    ).exists()

# ============================================================
# 11. PREPARE REMOTE SENSING FOLDERS
# ============================================================

REMOTE_DIR = ROOT / "04_REMOTE_SENSING"

for folder_name in [

    "01_VECTOR_INPUTS",

    "02_GEOMETRY_VALIDATION",

    "03_LANDSAT",

    "04_SENTINEL2",

    "05_GEDI",

    "06_TOPOGRAPHY",

    "07_FIELD_SATELLITE_INTEGRATION",

    "08_OUTPUTS"

]:

    (
        REMOTE_DIR / folder_name
    ).mkdir(

        parents=True,

        exist_ok=True
    )

# ============================================================
# 12. FINAL REPORT
# ============================================================

print("\n" + "=" * 75)

print("SATIPO — FINAL CHECKPOINT REPORT")

print("=" * 75)

print(
    "Objects saved:",
    len(checkpoint_saved)
)

print(
    "Save failures:",
    len(checkpoint_failed)
)

print(
    "Recovery failures:",
    len(
        checkpoint_verification_failures
    )
)

print(
    "Essential objects verified:",
    essential_ok
)

print(
    "Monte Carlo draws preserved:",
    len(final_mc_draws)
)

print(
    "Checkpoint manifest:",
    manifest_path
)

print(
    "Final pointer exists:",
    pointer_path.exists()
)

if checkpoint_success:

    print("\n" + "=" * 75)

    print("STAGE 1 ARCHIVED SUCCESSFULLY")

    print("CHECKPOINT VERIFIED")

    print("SAFE TO DISCONNECT COLAB")

    print("NEXT: CELL 86 — REMOTE SENSING")

    print("=" * 75)

else:

    print("\nCHECKPOINT REQUIRES REVIEW")

    print(
        "The previous checkpoint remains intact."
    )

    if checkpoint_failed:

        display(
            pd.DataFrame(
                checkpoint_failed
            )
        )

    if checkpoint_verification_failures:

        display(
            pd.DataFrame(
                checkpoint_verification_failures
            )
        )

    if not essential_ok:

        display(
            essential_status_df[
                ~essential_status_df["verified"]
            ]
        )


In [ ]:

# ============================================================
# CELL 85B — FINALIZE VERIFIED CHECKPOINT
# SATIPO FOREST PROJECT
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import json
import hashlib
import pandas as pd

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

CHECKPOINT_DIR = ROOT / "06_CHECKPOINTS"

FINAL_CHECKPOINT = (
    CHECKPOINT_DIR
    / "STAGE_01_FINAL_20261115_053524_UTC"
)

manifest_path = (
    FINAL_CHECKPOINT
    / "CHECKPOINT_MANIFEST.json"
)

assert manifest_path.exists(), (
    "Checkpoint manifest not found."
)

# ------------------------------------------------------------
# 1. LOAD EXISTING CHECKPOINT
# ------------------------------------------------------------

with open(
    manifest_path,
    "r",
    encoding="utf-8"
) as f:

    manifest = json.load(f)

saved_objects = manifest["saved_objects"]

failed_objects = manifest.get(
    "failed_objects", []
)

verification_failures = manifest.get(
    "verification_failures", []
)

essential_objects = manifest.get(
    "essential_objects", []
)

print("=" * 70)
print("SATIPO — FINAL CHECKPOINT VALIDATION")
print("=" * 70)

print("Saved objects:", len(saved_objects))
print("Original save failures:", len(failed_objects))
print("Recovery failures:", len(verification_failures))

# ------------------------------------------------------------
# 2. CLASSIFY NON-SCIENTIFIC FAILURE
# ------------------------------------------------------------

# The namespace list contains references to the
# interactive Colab environment, not scientific data.

allowed_auxiliary_failures = {
    "namespace"
}

auxiliary_failures = [

    item for item in failed_objects

    if item["name"] in allowed_auxiliary_failures
]

scientific_failures = [

    item for item in failed_objects

    if item["name"] not in allowed_auxiliary_failures
]

assert len(scientific_failures) == 0, (
    f"Unexpected scientific failures: "
    f"{scientific_failures}"
)

assert len(verification_failures) == 0, (
    "Checkpoint recovery verification failed."
)

assert len(saved_objects) == 890

assert len(essential_objects) == 9

assert all(

    item["saved"] and item["verified"]

    for item in essential_objects

), "Essential scientific objects are missing."

print("\nScientific objects: ALL VERIFIED")

print(
    "Excluded auxiliary objects:",
    [item["name"] for item in auxiliary_failures]
)

# ------------------------------------------------------------
# 3. VERIFY SAVED FILE INTEGRITY
# ------------------------------------------------------------

def file_sha256(path):

    sha = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            sha.update(chunk)

    return sha.hexdigest()

integrity_errors = []

for item in saved_objects:

    path = (
        FINAL_CHECKPOINT
        / item["file"]
    )

    if not path.exists():

        integrity_errors.append({
            "name": item["name"],
            "error": "File missing"
        })

        continue

    if file_sha256(path) != item["sha256"]:

        integrity_errors.append({
            "name": item["name"],
            "error": "SHA256 mismatch"
        })

assert not integrity_errors, integrity_errors

print(
    "\nObject file integrity verified:",
    len(saved_objects)
)

# ------------------------------------------------------------
# 4. VERIFY CRITICAL SCIENTIFIC OUTPUTS
# ------------------------------------------------------------

for item in manifest["critical_files"]:

    path = ROOT / item["path"]

    assert path.exists(), (
        f"Missing scientific file: {path}"
    )

    assert file_sha256(path) == item["sha256"], (
        f"Scientific file checksum mismatch: {path}"
    )

MC_DIR = (
    ROOT
    / "03_ANALYSIS"
    / "INTEGRATED_UNCERTAINTY"
)

final_mc_draws = pd.read_csv(
    MC_DIR / "INTEGRATED_MC_DRAWS.csv"
)

final_mc_summary = pd.read_csv(
    MC_DIR / "INTEGRATED_MC_GAP_SUMMARY.csv"
)

assert len(final_mc_draws) == 180000

assert len(final_mc_summary) == 18

print(
    "Monte Carlo draws:",
    len(final_mc_draws)
)

print(
    "Monte Carlo scenarios:",
    len(final_mc_summary)
)

# ------------------------------------------------------------
# 5. PRESERVE ORIGINAL MANIFEST
# ------------------------------------------------------------

original_manifest_path = (
    FINAL_CHECKPOINT
    / "CHECKPOINT_MANIFEST_ORIGINAL.json"
)

if not original_manifest_path.exists():

    original_manifest_path.write_bytes(
        manifest_path.read_bytes()
    )

# ------------------------------------------------------------
# 6. UPDATE MANIFEST
# ------------------------------------------------------------

manifest["stage"] = (
    "Stage 1 completed and verified"
)

manifest["verified"] = True

manifest["last_analysis_cell"] = 84

manifest["archive_cell"] = 85

manifest["next_cell"] = 86

manifest["finalized_at_utc"] = (
    datetime.now(timezone.utc).isoformat()
)

# Preserve the original failure for transparency.

manifest["original_failed_objects"] = (
    failed_objects
)

manifest["failed_objects"] = (
    scientific_failures
)

manifest["excluded_auxiliary_objects"] = (
    auxiliary_failures
)

manifest["finalization_note"] = (
    "All 890 saved scientific and compatible Python "
    "objects passed recovery verification. "
    "The only serialization failure was the auxiliary "
    "namespace list containing interactive Colab "
    "environment references. This object is not "
    "required for scientific recovery."
)

manifest["verification_summary"] = {

    "saved_objects": len(saved_objects),

    "verified_objects": len(saved_objects),

    "scientific_save_failures": 0,

    "recovery_failures": 0,

    "excluded_auxiliary_failures":
        len(auxiliary_failures),

    "essential_objects_verified":
        len(essential_objects),

    "monte_carlo_draws": 180000,

    "monte_carlo_scenarios": 18
}

temporary_manifest = (
    FINAL_CHECKPOINT
    / "CHECKPOINT_MANIFEST.tmp"
)

with open(

    temporary_manifest,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        manifest,

        f,

        indent=4,

        ensure_ascii=False
    )

temporary_manifest.replace(
    manifest_path
)

# ------------------------------------------------------------
# 7. PUBLISH FINAL CHECKPOINT POINTER
# ------------------------------------------------------------

latest_pointer = {

    "checkpoint": str(
        FINAL_CHECKPOINT
    ),

    "manifest": str(
        manifest_path
    ),

    "stage": 1,

    "last_analysis_cell": 84,

    "archive_cell": 85,

    "next_cell": 86,

    "verified": True,

    "saved_objects": len(saved_objects),

    "failed_objects": 0,

    "verification_failures": 0,

    "excluded_auxiliary_failures":
        len(auxiliary_failures)
}

pointer_path = (
    CHECKPOINT_DIR
    / "LATEST_FINAL_CHECKPOINT.json"
)

temporary_pointer = (
    CHECKPOINT_DIR
    / "LATEST_FINAL_CHECKPOINT.tmp"
)

with open(

    temporary_pointer,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        latest_pointer,

        f,

        indent=4,

        ensure_ascii=False
    )

temporary_pointer.replace(
    pointer_path
)

# ------------------------------------------------------------
# 8. VERIFY FINAL POINTER
# ------------------------------------------------------------

with open(

    pointer_path,

    "r",

    encoding="utf-8"

) as f:

    pointer_test = json.load(f)

assert pointer_test["verified"] is True

assert Path(
    pointer_test["manifest"]
).exists()

assert pointer_test["saved_objects"] == 890

assert pointer_test["next_cell"] == 86

# ------------------------------------------------------------
# 9. FINAL REPORT
# ------------------------------------------------------------

print("\n" + "=" * 70)

print("SATIPO — STAGE 1 FINALIZED")

print("=" * 70)

print("Verified objects: 890")

print("Scientific save failures: 0")

print("Recovery failures: 0")

print("Excluded auxiliary objects: 1")

print("Essential scientific objects: 9/9")

print("Monte Carlo draws: 180,000")

print("Monte Carlo scenarios: 18")

print("\nFinal checkpoint pointer:")

print(pointer_path)

print("\nFinal pointer exists:", pointer_path.exists())

print("\n" + "=" * 70)

print("STAGE 1 ARCHIVED SUCCESSFULLY")

print("CHECKPOINT VERIFIED")

print("SAFE TO DISCONNECT COLAB")

print("NEXT: CELL 86 — REMOTE SENSING")

print("=" * 70)


In [ ]:

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

VECTOR_DIR = (
    ROOT
    / "04_REMOTE_SENSING"
    / "01_VECTOR_INPUTS"
)

VECTOR_DIR.mkdir(
    parents=True,
    exist_ok=True
)

VECTOR_ZIP = (
    VECTOR_DIR
    / "SATIPO_VECTOR_INPUTS.zip"
)

print("Project exists:", ROOT.exists())
print("Vector directory:", VECTOR_DIR)
print("ZIP exists:", VECTOR_ZIP.exists())

print("\nFiles in vector directory:")

for file in VECTOR_DIR.iterdir():
    print(file.name)

print("\nField data active in RAM:",
      "master_v04" in globals())

print("Final checkpoint available:",
      (ROOT / "06_CHECKPOINTS" /
       "LATEST_FINAL_CHECKPOINT.json").exists())


In [ ]:

from google.colab import files
import shutil

uploaded = files.upload()

for filename in uploaded:

    if filename == "SATIPO_VECTOR_INPUTS.zip":

        shutil.copy2(
            filename,
            VECTOR_DIR / filename
        )

        print("ZIP saved successfully")

assert VECTOR_ZIP.exists()

print("Ready for Cell 86")


In [ ]:

# ============================================================
# CELL 86 — GEOSPATIAL DATA PREPARATION & VALIDATION
# SATIPO FOREST PROJECT
# STAGE 2 — REMOTE SENSING
# ============================================================

!pip -q install geopandas pyogrio shapely pyproj

from google.colab import drive
from pathlib import Path

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

import zipfile
import json
import hashlib

from shapely.geometry import Point
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from pyproj import CRS

drive.mount("/content/drive")

# ============================================================
# 1. PROJECT DIRECTORIES
# ============================================================

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

REMOTE_DIR = ROOT / "04_REMOTE_SENSING"

VECTOR_DIR = REMOTE_DIR / "01_VECTOR_INPUTS"

GEOMETRY_DIR = REMOTE_DIR / "02_GEOMETRY_VALIDATION"

REMOTE_OUTPUTS = REMOTE_DIR / "08_OUTPUTS"

FIGURE_DIR = REMOTE_OUTPUTS / "FIGURES"

TABLE_DIR = REMOTE_OUTPUTS / "TABLES"

EXTRACT_DIR = VECTOR_DIR / "EXTRACTED_SHAPES"

for folder in [
    VECTOR_DIR,
    GEOMETRY_DIR,
    REMOTE_OUTPUTS,
    FIGURE_DIR,
    TABLE_DIR,
    EXTRACT_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

VECTOR_ZIP = (
    VECTOR_DIR / "SATIPO_VECTOR_INPUTS.zip"
)

assert VECTOR_ZIP.exists(), (
    f"ZIP not found: {VECTOR_ZIP}"
)

print("=" * 75)
print("SATIPO — GEOSPATIAL PREPARATION")
print("=" * 75)

print("Input:", VECTOR_ZIP)

# ============================================================
# 2. EXTRACT ORIGINAL VECTOR FILES
# ============================================================

with zipfile.ZipFile(
    VECTOR_ZIP,
    "r"
) as archive:

    assert archive.testzip() is None, (
        "ZIP integrity verification failed."
    )

    # Prevent extraction outside target directory
    for member in archive.infolist():

        target = (
            EXTRACT_DIR / member.filename
        ).resolve()

        assert target.is_relative_to(
            EXTRACT_DIR.resolve()
        ), "Unsafe ZIP path."

    archive.extractall(
        EXTRACT_DIR
    )

    extracted_files = archive.namelist()

print("\nExtracted files:", len(extracted_files))

# ============================================================
# 3. LOCATE VECTOR LAYERS
# ============================================================

def find_vector_layer(filename):

    matches = list(
        EXTRACT_DIR.rglob(filename)
    )

    assert len(matches) == 1, (
        f"Expected one {filename}, "
        f"found {len(matches)}"
    )

    return matches[0]

fragment_path = find_vector_layer(
    "ACP_EL_TORNILLAL.shp"
)

conservation_path = find_vector_layer(
    "ACP_EL_TORNILLAL_V2.shp"
)

plot_path = find_vector_layer(
    "PPM_POLIGONO.shp"
)

vertex_path = find_vector_layer(
    "VERTICES_PPM.shp"
)

vector_paths = {

    "forest_fragments": fragment_path,

    "conservation_area": conservation_path,

    "permanent_plots": plot_path,

    "plot_vertices": vertex_path
}

# ============================================================
# 4. READ ORIGINAL GEOSPATIAL DATA
# ============================================================

original_vectors = {

    name: gpd.read_file(path)

    for name, path in vector_paths.items()
}

for name, layer in original_vectors.items():

    print("\n" + "-" * 65)

    print("Layer:", name)

    print("Features:", len(layer))

    print("CRS:", layer.crs)

    print("Geometry types:")

    print(
        layer.geom_type.value_counts()
    )

    print("Attributes:")

    display(
        layer.drop(
            columns="geometry"
        )
    )

# ============================================================
# 5. STANDARDIZE COORDINATE REFERENCE SYSTEM
# ============================================================

TARGET_CRS = "EPSG:32718"

WGS84_CRS = "EPSG:4326"

def validate_vector(
    layer,
    layer_name,
    expected_geometry,
    expected_count=None
):

    assert layer.crs is not None, (
        f"{layer_name}: CRS is missing."
    )

    assert not layer.empty, (
        f"{layer_name}: empty layer."
    )

    if expected_count is not None:

        assert len(layer) == expected_count, (
            f"{layer_name}: unexpected feature count."
        )

    assert layer.geometry.notna().all(), (
        f"{layer_name}: missing geometries."
    )

    assert not layer.geometry.is_empty.any(), (
        f"{layer_name}: empty geometries."
    )

    assert layer.geometry.is_valid.all(), (
        f"{layer_name}: invalid geometries."
    )

    assert layer.geom_type.isin(
        expected_geometry
    ).all(), (
        f"{layer_name}: unexpected geometry type."
    )

    result = layer.to_crs(
        TARGET_CRS
    ).copy()

    return result

fragments = validate_vector(

    original_vectors["forest_fragments"],

    "Forest fragments",

    ["Polygon", "MultiPolygon"],

    expected_count=2
)

conservation = validate_vector(

    original_vectors["conservation_area"],

    "Conservation area",

    ["Polygon", "MultiPolygon"]
)

plots = validate_vector(

    original_vectors["permanent_plots"],

    "Permanent plots",

    ["Polygon", "MultiPolygon"],

    expected_count=2
)

vertices = validate_vector(

    original_vectors["plot_vertices"],

    "Plot vertices",

    ["Point", "MultiPoint"],

    expected_count=8
)

print("\nAll four vector layers validated.")

# ============================================================
# 6. IDENTIFY FOREST FRAGMENTS
# ============================================================

# North-south identification follows the original
# study map and permanent-plot correspondence.
#
# Forest-type labels remain provisional until
# independently confirmed by the field researcher.

fragments["centroid_easting"] = (
    fragments.geometry.centroid.x
)

fragments["centroid_northing"] = (
    fragments.geometry.centroid.y
)

fragments = fragments.sort_values(

    "centroid_northing",

    ascending=True

).reset_index(drop=True)

fragments["fragment_id"] = [

    "FRAG_01",

    "FRAG_02"
]

fragments["forest_type"] = [

    "Late_secondary",

    "Mature"
]

fragments["spatial_position"] = [

    "South",

    "North"
]

fragments["label_status"] = (
    "Provisional_map_based"
)

fragments["area_ha"] = (

    fragments.geometry.area / 10000
)

fragments["perimeter_m"] = (

    fragments.geometry.length
)

fragments["historical_area_ha"] = [

    23.0,

    35.0
]

fragments["area_difference_ha"] = (

    fragments["area_ha"]

    - fragments["historical_area_ha"]
)

fragments["area_difference_pct"] = (

    100

    * fragments["area_difference_ha"]

    / fragments["historical_area_ha"]
)

# Geographic coordinates of projected centroids

fragment_centroids_wgs84 = (

    fragments.geometry.centroid

    .to_crs(WGS84_CRS)
)

fragments["longitude"] = (

    fragment_centroids_wgs84.x
)

fragments["latitude"] = (

    fragment_centroids_wgs84.y
)

print("\nFOREST FRAGMENT SUMMARY")

display(

    fragments[[
        "fragment_id",
        "forest_type",
        "spatial_position",
        "area_ha",
        "historical_area_ha",
        "area_difference_ha",
        "area_difference_pct",
        "longitude",
        "latitude"
    ]].round(5)
)

# ============================================================
# 7. IDENTIFY PERMANENT FIELD PLOTS
# ============================================================

plots["centroid_easting"] = (

    plots.geometry.centroid.x
)

plots["centroid_northing"] = (

    plots.geometry.centroid.y
)

plots = plots.sort_values(

    "centroid_northing",

    ascending=True

).reset_index(drop=True)

plots["plot_id"] = [

    "P01",

    "P02"
]

plots["original_plot_label"] = [

    "PPM_1",

    "PPM_2"
]

plots["forest_type"] = [

    "Late_secondary",

    "Mature"
]

plots["label_status"] = (
    "Provisional_map_based"
)

plots["area_ha"] = (

    plots.geometry.area / 10000
)

plots["perimeter_m"] = (

    plots.geometry.length
)

plots["census_year"] = 2019

plots["n_subplots"] = 25

plots["subplot_area_m2"] = 400

plot_centroids_wgs84 = (

    plots.geometry.centroid

    .to_crs(WGS84_CRS)
)

plots["longitude"] = (

    plot_centroids_wgs84.x
)

plots["latitude"] = (

    plot_centroids_wgs84.y
)

# Validate approximately one hectare per plot

assert np.allclose(

    plots["area_ha"],

    1.0,

    atol=0.01

), "Plot area differs from one hectare."

print("\nPERMANENT PLOT SUMMARY")

display(

    plots[[
        "plot_id",
        "original_plot_label",
        "forest_type",
        "area_ha",
        "centroid_easting",
        "centroid_northing",
        "longitude",
        "latitude"
    ]].round(6)
)

# ============================================================
# 8. VERIFY PLOT–FRAGMENT CORRESPONDENCE
# ============================================================

containment_records = []

for _, plot_row in plots.iterrows():

    for _, fragment_row in fragments.iterrows():

        intersection_area = (

            plot_row.geometry.intersection(

                fragment_row.geometry

            ).area
        )

        overlap_pct = (

            100

            * intersection_area

            / plot_row.geometry.area
        )

        containment_records.append({

            "plot_id": plot_row["plot_id"],

            "plot_forest_type":
                plot_row["forest_type"],

            "fragment_id":
                fragment_row["fragment_id"],

            "fragment_forest_type":
                fragment_row["forest_type"],

            "intersection_area_ha":
                intersection_area / 10000,

            "plot_containment_pct":
                overlap_pct,

            "expected_match": (

                plot_row["forest_type"]

                == fragment_row["forest_type"]
            )
        })

containment_summary = pd.DataFrame(
    containment_records
)

expected_containment = (

    containment_summary.loc[

        containment_summary["expected_match"]

    ].copy()
)

assert (

    expected_containment[
        "plot_containment_pct"
    ] >= 99.99

).all(), (

    "A permanent plot is not fully contained "
    "within its expected fragment."
)

print("\nPLOT–FRAGMENT CONTAINMENT")

display(
    containment_summary.round(6)
)

# ============================================================
# 9. VALIDATE PLOT VERTICES
# ============================================================

vertex_records = []

for _, plot_row in plots.iterrows():

    plot_boundary = (
        plot_row.geometry.boundary
    )

    vertex_distances = (

        vertices.geometry.distance(
            plot_boundary
        )
    )

    # A one-metre tolerance accommodates
    # small coordinate-rounding differences.

    matching_vertices = (
        vertex_distances <= 1.0
    )

    vertex_records.append({

        "plot_id": plot_row["plot_id"],

        "vertices_within_1m":

            int(matching_vertices.sum()),

        "expected_vertices": 4,

        "maximum_matching_distance_m":

            float(

                vertex_distances.loc[
                    matching_vertices
                ].max()
            )

            if matching_vertices.any()

            else np.nan
    })

vertex_validation = pd.DataFrame(
    vertex_records
)

print("\nPLOT VERTEX VALIDATION")

display(
    vertex_validation.round(6)
)

if not (

    vertex_validation[
        "vertices_within_1m"
    ] == 4

).all():

    print(
        "WARNING: Vertex correspondence "
        "requires manual review."
    )

# ============================================================
# 10. SPATIAL RELATIONSHIPS
# ============================================================

secondary_fragment = (

    fragments.loc[

        fragments["forest_type"]

        .eq("Late_secondary")

    ].geometry.iloc[0]
)

mature_fragment = (

    fragments.loc[

        fragments["forest_type"]

        .eq("Mature")

    ].geometry.iloc[0]
)

secondary_plot = (

    plots.loc[

        plots["forest_type"]

        .eq("Late_secondary")

    ].geometry.iloc[0]
)

mature_plot = (

    plots.loc[

        plots["forest_type"]

        .eq("Mature")

    ].geometry.iloc[0]
)

fragment_overlap_ha = (

    secondary_fragment.intersection(

        mature_fragment

    ).area / 10000
)

fragment_union = (

    secondary_fragment.union(

        mature_fragment
    )
)

plot_centroid_distance_m = (

    secondary_plot.centroid.distance(

        mature_plot.centroid
    )
)

plot_edge_distance_m = (

    secondary_plot.distance(

        mature_plot
    )
)

fragment_shared_boundary_m = (

    secondary_fragment.boundary.intersection(

        mature_fragment.boundary

    ).length
)

conservation_union = (

    conservation.geometry.union_all()
)

conservation_area_ha = (

    conservation_union.area / 10000
)

union_difference_ha = (

    fragment_union.symmetric_difference(

        conservation_union

    ).area / 10000
)

assert fragment_overlap_ha < 0.0001, (

    "Forest fragment polygons overlap."
)

spatial_summary = pd.DataFrame({

    "metric": [

        "Secondary fragment area (ha)",

        "Mature fragment area (ha)",

        "Total fragment area (ha)",

        "Conservation polygon area (ha)",

        "Fragment overlap (ha)",

        "Shared fragment boundary (m)",

        "Plot centroid distance (m)",

        "Plot edge distance (m)",

        "Fragment-conservation difference (ha)"
    ],

    "value": [

        secondary_fragment.area / 10000,

        mature_fragment.area / 10000,

        fragment_union.area / 10000,

        conservation_area_ha,

        fragment_overlap_ha,

        fragment_shared_boundary_m,

        plot_centroid_distance_m,

        plot_edge_distance_m,

        union_difference_ha
    ]
})

print("\nSPATIAL RELATIONSHIPS")

display(
    spatial_summary.round(6)
)

# ============================================================
# 11. ESTIMATE NOMINAL PIXEL SUPPORT
# ============================================================

# Area-equivalent pixels, NOT actual independent
# observations or exact raster intersection counts.

pixel_records = []

for spatial_scale, layer, id_column in [

    ("Fragment", fragments, "fragment_id"),

    ("Plot", plots, "plot_id")
]:

    for _, row in layer.iterrows():

        area_m2 = row.geometry.area

        for sensor, resolution in [

            ("Landsat", 30),

            ("Sentinel2_10m", 10),

            ("Sentinel2_20m", 20),

            ("DEM_30m", 30)
        ]:

            pixel_records.append({

                "spatial_scale": spatial_scale,

                "unit_id": row[id_column],

                "forest_type": row["forest_type"],

                "area_ha": area_m2 / 10000,

                "sensor": sensor,

                "resolution_m": resolution,

                "nominal_area_equivalent_pixels":

                    area_m2 / resolution**2
            })

pixel_support = pd.DataFrame(
    pixel_records
)

print("\nNOMINAL SATELLITE PIXEL SUPPORT")

display(
    pixel_support.round(2)
)

# ============================================================
# 12. PREPARE INTERIOR BUFFER GEOMETRIES
# ============================================================

# Interior buffers reduce potential edge mixing.
# They are sensitivity-analysis units, not new
# independent forest replicates.

buffer_records = []

for spatial_scale, layer, id_column in [

    ("Fragment", fragments, "fragment_id"),

    ("Plot", plots, "plot_id")
]:

    for _, row in layer.iterrows():

        for buffer_m in [

            0,

            10,

            15,

            30
        ]:

            if buffer_m == 0:

                interior_geometry = row.geometry

            else:

                interior_geometry = (

                    row.geometry.buffer(
                        -buffer_m
                    )
                )

            if interior_geometry.is_empty:

                continue

            assert interior_geometry.is_valid

            buffer_records.append({

                "spatial_scale": spatial_scale,

                "unit_id": row[id_column],

                "forest_type": row["forest_type"],

                "buffer_m": buffer_m,

                "area_ha":

                    interior_geometry.area / 10000,

                "retained_area_pct":

                    100

                    * interior_geometry.area

                    / row.geometry.area,

                "geometry": interior_geometry
            })

interior_buffers = gpd.GeoDataFrame(

    buffer_records,

    geometry="geometry",

    crs=TARGET_CRS
)

print("\nINTERIOR BUFFER SENSITIVITY")

display(

    interior_buffers.drop(

        columns="geometry"

    ).round(3)
)

# ============================================================
# 13. PREPARE CLEAN ANALYTICAL VECTOR LAYERS
# ============================================================

fragment_columns = [

    "fragment_id",

    "forest_type",

    "spatial_position",

    "label_status",

    "area_ha",

    "perimeter_m",

    "historical_area_ha",

    "area_difference_ha",

    "area_difference_pct",

    "centroid_easting",

    "centroid_northing",

    "longitude",

    "latitude",

    "geometry"
]

plot_columns = [

    "plot_id",

    "original_plot_label",

    "forest_type",

    "label_status",

    "area_ha",

    "perimeter_m",

    "census_year",

    "n_subplots",

    "subplot_area_m2",

    "centroid_easting",

    "centroid_northing",

    "longitude",

    "latitude",

    "geometry"
]

satipo_fragments = (

    fragments[fragment_columns].copy()
)

satipo_plots = (

    plots[plot_columns].copy()
)

satipo_conservation = gpd.GeoDataFrame(

    {

        "site_id": ["EL_TORNILLAL"],

        "area_ha": [conservation_area_ha]
    },

    geometry=[conservation_union],

    crs=TARGET_CRS
)

satipo_vertices = vertices.copy()

# ============================================================
# 14. EXPORT MASTER GEOPACKAGE
# ============================================================

GPKG_PATH = (

    GEOMETRY_DIR

    / "SATIPO_GEOSPATIAL_MASTER.gpkg"
)

# Replace only this generated analytical file.
# Original shapefiles remain untouched.

if GPKG_PATH.exists():

    GPKG_PATH.unlink()

master_vector_layers = {

    "forest_fragments": satipo_fragments,

    "permanent_plots": satipo_plots,

    "conservation_area": satipo_conservation,

    "plot_vertices": satipo_vertices,

    "interior_buffers": interior_buffers
}

for layer_name, layer in (

    master_vector_layers.items()
):

    layer.to_file(

        GPKG_PATH,

        layer=layer_name,

        driver="GPKG",

        engine="pyogrio",

        index=False
    )

    print(
        "GeoPackage layer saved:",
        layer_name
    )

# Verify the exported layers

for layer_name, original_layer in (

    master_vector_layers.items()
):

    restored_layer = gpd.read_file(

        GPKG_PATH,

        layer=layer_name
    )

    assert len(restored_layer) == len(
        original_layer
    )

    assert restored_layer.crs.to_epsg() == 32718

    assert restored_layer.geometry.is_valid.all()

print("\nGeoPackage verified.")

# ============================================================
# 15. EXPORT GOOGLE EARTH ENGINE GEOJSON
# ============================================================

GEE_VECTOR_DIR = (

    GEOMETRY_DIR / "GEE_GEOJSON"
)

GEE_VECTOR_DIR.mkdir(

    parents=True,

    exist_ok=True
)

geojson_paths = {}

for layer_name, layer in [

    ("forest_fragments", satipo_fragments),

    ("permanent_plots", satipo_plots),

    ("conservation_area", satipo_conservation),

    ("interior_buffers", interior_buffers)
]:

    output_path = (

        GEE_VECTOR_DIR

        / f"SATIPO_{layer_name.upper()}.geojson"
    )

    layer_wgs84 = layer.to_crs(
        WGS84_CRS
    )

    layer_wgs84.to_file(

        output_path,

        driver="GeoJSON",

        engine="pyogrio",

        index=False
    )

    geojson_paths[layer_name] = str(
        output_path
    )

    print("GeoJSON saved:", output_path.name)

# ============================================================
# 16. EXPORT GEOSPATIAL QA TABLES
# ============================================================

geospatial_tables = {

    "FRAGMENT_GEOMETRY_SUMMARY":

        satipo_fragments.drop(
            columns="geometry"
        ),

    "PERMANENT_PLOT_GEOMETRY_SUMMARY":

        satipo_plots.drop(
            columns="geometry"
        ),

    "PLOT_FRAGMENT_CONTAINMENT":

        containment_summary,

    "PLOT_VERTEX_VALIDATION":

        vertex_validation,

    "SPATIAL_RELATIONSHIPS":

        spatial_summary,

    "NOMINAL_PIXEL_SUPPORT":

        pixel_support,

    "INTERIOR_BUFFER_SENSITIVITY":

        interior_buffers.drop(
            columns="geometry"
        )
}

for table_name, dataframe in (

    geospatial_tables.items()
):

    dataframe.to_csv(

        TABLE_DIR / f"{table_name}.csv",

        index=False,

        encoding="utf-8-sig"
    )

geospatial_workbook = (

    TABLE_DIR

    / "SATIPO_GEOSPATIAL_VALIDATION.xlsx"
)

with pd.ExcelWriter(

    geospatial_workbook,

    engine="openpyxl"

) as writer:

    for table_name, dataframe in (

        geospatial_tables.items()
    ):

        dataframe.to_excel(

            writer,

            sheet_name=table_name[:31],

            index=False
        )

print("\nGeospatial QA tables saved.")

# ============================================================
# 17. GENERATE GEOMETRY VALIDATION FIGURE
# ============================================================

FOREST_COLORS = {

    "Late_secondary": "#DB9140",

    "Mature": "#24634F"
}

fig, ax = plt.subplots(

    figsize=(8, 9),

    dpi=180
)

for _, row in satipo_fragments.iterrows():

    gpd.GeoSeries(

        [row.geometry],

        crs=TARGET_CRS

    ).plot(

        ax=ax,

        facecolor=FOREST_COLORS[
            row["forest_type"]
        ],

        edgecolor="#263238",

        linewidth=1.4,

        alpha=0.72
    )

    label_point = (

        row.geometry.representative_point()
    )

    forest_label = (

        "Late-secondary forest"

        if row["forest_type"] == "Late_secondary"

        else "Mature forest"
    )

    ax.annotate(

        f"{forest_label}\n"
        f"{row['area_ha']:.2f} ha",

        xy=(label_point.x, label_point.y),

        xytext=(0, 25),

        textcoords="offset points",

        ha="center",

        fontsize=10,

        fontweight="bold",

        color="#202020",

        path_effects=[

            pe.withStroke(

                linewidth=3,

                foreground="white"
            )
        ]
    )

# Plot the two permanent field plots

satipo_plots.plot(

    ax=ax,

    facecolor="white",

    edgecolor="#222222",

    linewidth=2.0,

    alpha=0.95
)

for _, row in satipo_plots.iterrows():

    centroid = row.geometry.centroid

    ax.text(

        centroid.x,

        centroid.y,

        f"{row['plot_id']}\n1 ha",

        ha="center",

        va="center",

        fontsize=9,

        fontweight="bold",

        color="#202020"
    )

# Original surveyed vertices

satipo_vertices.plot(

    ax=ax,

    color="#1A1A1A",

    markersize=15,

    zorder=5
)

# Coordinate limits

minx, miny, maxx, maxy = (

    satipo_fragments.total_bounds
)

x_range = maxx - minx

y_range = maxy - miny

ax.set_xlim(

    minx - 0.12 * x_range,

    maxx + 0.12 * x_range
)

ax.set_ylim(

    miny - 0.12 * y_range,

    maxy + 0.12 * y_range
)

# North arrow

ax.annotate(

    "N",

    xy=(0.94, 0.96),

    xytext=(0.94, 0.86),

    xycoords="axes fraction",

    textcoords="axes fraction",

    ha="center",

    va="center",

    fontsize=12,

    fontweight="bold",

    arrowprops=dict(

        arrowstyle="-|>",

        linewidth=1.5,

        color="black"
    )
)

# Scale bar — projected coordinates in metres

scale_length = 200

scale_x = minx

scale_y = miny - 0.06 * y_range

ax.plot(

    [scale_x, scale_x + scale_length],

    [scale_y, scale_y],

    color="black",

    linewidth=3,

    solid_capstyle="butt"
)

ax.text(

    scale_x + scale_length / 2,

    scale_y + 0.018 * y_range,

    "200 m",

    ha="center",

    fontsize=9
)

legend_elements = [

    Patch(

        facecolor=FOREST_COLORS["Late_secondary"],

        edgecolor="#263238",

        label="Late-secondary forest"
    ),

    Patch(

        facecolor=FOREST_COLORS["Mature"],

        edgecolor="#263238",

        label="Mature forest"
    ),

    Patch(

        facecolor="white",

        edgecolor="black",

        label="Permanent field plot"
    ),

    Line2D(

        [0],

        [0],

        marker="o",

        linestyle="none",

        color="#1A1A1A",

        markersize=5,

        label="Surveyed vertices"
    )
]

ax.legend(

    handles=legend_elements,

    loc="lower right",

    fontsize=9,

    frameon=True
)

ax.set_title(

    "El Tornillal forest fragments\n"
    "Permanent plot geometry validation",

    fontsize=14,

    fontweight="bold",

    pad=14
)

ax.set_xlabel(

    "Easting (m) — UTM zone 18S"
)

ax.set_ylabel(

    "Northing (m) — UTM zone 18S"
)

ax.ticklabel_format(

    style="plain",

    axis="both",

    useOffset=False
)

ax.tick_params(

    axis="x",

    labelrotation=30
)

ax.set_aspect("equal")

ax.grid(

    linestyle=":",

    alpha=0.25
)

fig.tight_layout()

for extension in [

    "png",

    "pdf",

    "svg"
]:

    fig.savefig(

        FIGURE_DIR

        / f"FIG_GEOSPATIAL_VALIDATION.{extension}",

        dpi=600,

        bbox_inches="tight",

        facecolor="white"
    )

plt.show()

# ============================================================
# 18. SAVE VECTOR INPUT MANIFEST
# ============================================================

def vector_sha256(path):

    sha = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(

            lambda: f.read(1024 * 1024),

            b""
        ):

            sha.update(chunk)

    return sha.hexdigest()

geospatial_manifest = {

    "project": "SATIPO_FOREST_PROJECT",

    "stage": "Stage 2 - Remote sensing",

    "cell": 86,

    "source_zip": str(VECTOR_ZIP),

    "source_zip_sha256":

        vector_sha256(VECTOR_ZIP),

    "projected_crs": TARGET_CRS,

    "gee_crs": WGS84_CRS,

    "forest_labels_status":

        "Provisional map-based identification",

    "field_census_year": 2019,

    "n_fragments": len(satipo_fragments),

    "n_permanent_plots": len(satipo_plots),

    "n_vertices": len(satipo_vertices),

    "fragment_areas_ha":

        satipo_fragments.set_index(

            "forest_type"

        )["area_ha"].to_dict(),

    "plot_areas_ha":

        satipo_plots.set_index(

            "plot_id"

        )["area_ha"].to_dict(),

    "fragment_overlap_ha":

        float(fragment_overlap_ha),

    "plot_centroid_distance_m":

        float(plot_centroid_distance_m),

    "geopackage": str(GPKG_PATH),

    "gee_geojson": geojson_paths,

    "qa_workbook": str(geospatial_workbook),

    "source_layers": {

        name: str(path)

        for name, path in vector_paths.items()
    },

    "limitations": [

        "Historical and mapped fragment areas differ.",

        "Forest-type assignments require confirmation.",

        "One independent field plot per forest type.",

        "Satellite pixels are not independent forest replicates.",

        "Nominal pixel counts are area-equivalent estimates.",

        "Exact subplot geometry is not available in these layers."
    ]
}

geospatial_manifest_path = (

    GEOMETRY_DIR

    / "SATIPO_GEOSPATIAL_MANIFEST.json"
)

with open(

    geospatial_manifest_path,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        geospatial_manifest,

        f,

        indent=4,

        ensure_ascii=False
    )

# ============================================================
# 19. FINAL REPORT
# ============================================================

print("\n" + "=" * 75)

print("SATIPO — GEOSPATIAL VALIDATION REPORT")

print("=" * 75)

print("Forest fragments:", len(satipo_fragments))

print("Permanent plots:", len(satipo_plots))

print("Surveyed vertices:", len(satipo_vertices))

print("Projected CRS:", TARGET_CRS)

print(

    "Total forest area (ha):",

    round(fragment_union.area / 10000, 4)
)

print(

    "Plot centroid distance (m):",

    round(plot_centroid_distance_m, 2)
)

print(

    "Fragment overlap (ha):",

    round(fragment_overlap_ha, 8)
)

print(

    "Minimum plot containment (%):",

    round(

        expected_containment[

            "plot_containment_pct"

        ].min(),

        4
    )
)

print("\nMaster GeoPackage:")

print(GPKG_PATH)

print("\nEarth Engine inputs:")

print(GEE_VECTOR_DIR)

print("\nGeospatial QA workbook:")

print(geospatial_workbook)

print("\nManifest:")

print(geospatial_manifest_path)

print("\n" + "=" * 75)

print("GEOSPATIAL PREPARATION COMPLETED")

print("FOREST LABELS REMAIN PROVISIONAL")

print("NEXT: CELL 87 — GOOGLE EARTH ENGINE INITIALIZATION")

print("=" * 75)


In [ ]:

# ============================================================
# CELL 87 — GOOGLE EARTH ENGINE INITIALIZATION
# SATIPO FOREST PROJECT
# STAGE 2 — REMOTE SENSING
# ============================================================

!pip -q install earthengine-api geemap

import ee
import geemap
import json
import pandas as pd
import geopandas as gpd

from pathlib import Path
from google.colab import drive
from IPython.display import display

drive.mount("/content/drive")

# ============================================================
# 1. PROJECT DIRECTORIES
# ============================================================

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

REMOTE_DIR = ROOT / "04_REMOTE_SENSING"

GEOMETRY_DIR = (
    REMOTE_DIR / "02_GEOMETRY_VALIDATION"
)

GEE_VECTOR_DIR = (
    GEOMETRY_DIR / "GEE_GEOJSON"
)

LANDSAT_DIR = REMOTE_DIR / "03_LANDSAT"

SENTINEL_DIR = REMOTE_DIR / "04_SENTINEL2"

GEDI_DIR = REMOTE_DIR / "05_GEDI"

TOPOGRAPHY_DIR = REMOTE_DIR / "06_TOPOGRAPHY"

INTEGRATION_DIR = (
    REMOTE_DIR / "07_FIELD_SATELLITE_INTEGRATION"
)

REMOTE_OUTPUTS = REMOTE_DIR / "08_OUTPUTS"

for folder in [
    LANDSAT_DIR,
    SENTINEL_DIR,
    GEDI_DIR,
    TOPOGRAPHY_DIR,
    INTEGRATION_DIR,
    REMOTE_OUTPUTS
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

# ============================================================
# 2. GOOGLE EARTH ENGINE AUTHENTICATION
# ============================================================

# IMPORTANT:
# Replace this value with your registered
# Google Cloud project ID.


GEE_PROJECT = "brunei159"

print("=" * 70)
print("GOOGLE EARTH ENGINE INITIALIZATION")
print("=" * 70)

ee.Initialize(project=GEE_PROJECT)

assert ee.Number(1).getInfo() == 1

print("Earth Engine initialized successfully.")
print("Earth Engine server connection: OK")
print("Google Cloud project:", GEE_PROJECT)


print("=" * 70)
print("GOOGLE EARTH ENGINE INITIALIZATION")
print("=" * 70)

try:

    ee.Initialize(
        project=GEE_PROJECT
    )

    print("Earth Engine initialized successfully.")

except Exception:

    print("Authentication required.")

    ee.Authenticate()

    ee.Initialize(
        project=GEE_PROJECT
    )

    print("Authentication successful.")

# Server-side connectivity test

gee_test = ee.Number(1).getInfo()

assert gee_test == 1

print("Earth Engine server connection: OK")
print("Google Cloud project:", GEE_PROJECT)

# ============================================================
# 3. LOAD VALIDATED GEOJSON FILES
# ============================================================

geojson_files = {

    "fragments":

        GEE_VECTOR_DIR
        / "SATIPO_FOREST_FRAGMENTS.geojson",

    "plots":

        GEE_VECTOR_DIR
        / "SATIPO_PERMANENT_PLOTS.geojson",

    "conservation":

        GEE_VECTOR_DIR
        / "SATIPO_CONSERVATION_AREA.geojson",

    "buffers":

        GEE_VECTOR_DIR
        / "SATIPO_INTERIOR_BUFFERS.geojson"
}

for name, path in geojson_files.items():

    assert path.exists(), (
        f"Missing GeoJSON: {path}"
    )

    print("Verified:", name)

# ============================================================
# 4. CONVERT GEOJSON TO EE FEATURE COLLECTION
# ============================================================

def load_ee_feature_collection(path):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as file:

        geojson = json.load(file)

    assert geojson["type"] == "FeatureCollection"

    features = []

    for item in geojson["features"]:

        geometry = ee.Geometry(
            item["geometry"],
            proj="EPSG:4326",
            geodesic=False
        )

        properties = item.get(
            "properties",
            {}
        )

        properties = {

            key: value

            for key, value in properties.items()

            if value is not None
        }

        feature = ee.Feature(
            geometry,
            properties
        )

        features.append(feature)

    return ee.FeatureCollection(
        features
    )

ee_fragments = load_ee_feature_collection(

    geojson_files["fragments"]
)

ee_plots = load_ee_feature_collection(

    geojson_files["plots"]
)

ee_conservation = load_ee_feature_collection(

    geojson_files["conservation"]
)

ee_buffers = load_ee_feature_collection(

    geojson_files["buffers"]
)

# ============================================================
# 5. VALIDATE EARTH ENGINE FEATURE COLLECTIONS
# ============================================================

gee_layers = {

    "Forest fragments": ee_fragments,

    "Permanent plots": ee_plots,

    "Conservation area": ee_conservation,

    "Interior buffers": ee_buffers
}

gee_layer_records = []

for name, collection in gee_layers.items():

    feature_count = collection.size().getInfo()

    gee_layer_records.append({

        "layer": name,

        "features": feature_count
    })

    print(

        f"{name}: {feature_count} features"
    )

gee_layer_summary = pd.DataFrame(

    gee_layer_records
)

assert ee_fragments.size().getInfo() == 2

assert ee_plots.size().getInfo() == 2

assert ee_conservation.size().getInfo() == 1

assert ee_buffers.size().getInfo() == 16

display(gee_layer_summary)

# ============================================================
# 6. DEFINE SPATIAL ANALYSIS UNITS
# ============================================================

ee_secondary_fragment = ee.Feature(

    ee_fragments.filter(

        ee.Filter.eq(

            "fragment_id",

            "FRAG_01"
        )

    ).first()
)

ee_mature_fragment = ee.Feature(

    ee_fragments.filter(

        ee.Filter.eq(

            "fragment_id",

            "FRAG_02"
        )

    ).first()
)

ee_secondary_plot = ee.Feature(

    ee_plots.filter(

        ee.Filter.eq(

            "plot_id",

            "P01"
        )

    ).first()
)

ee_mature_plot = ee.Feature(

    ee_plots.filter(

        ee.Filter.eq(

            "plot_id",

            "P02"
        )

    ).first()
)

# Fragment-level analysis geometries

geom_secondary = (

    ee_secondary_fragment.geometry()
)

geom_mature = (

    ee_mature_fragment.geometry()
)

# Plot-level analysis geometries

geom_p01 = (

    ee_secondary_plot.geometry()
)

geom_p02 = (

    ee_mature_plot.geometry()
)

# Complete study area

SATIPO_AOI = (

    ee_conservation.geometry()
)

# Contextual region for image filtering

SATIPO_CONTEXT = (

    SATIPO_AOI.buffer(

        distance=2000,

        maxError=1
    )
)

# ============================================================
# 7. VERIFY EARTH ENGINE AREA ESTIMATES
# ============================================================

def add_gee_area(feature):

    area_ha = (

        feature.geometry()

        .area(maxError=1)

        .divide(10000)
    )

    return feature.set(

        "gee_area_ha",

        area_ha
    )

gee_fragments_area = (

    ee_fragments.map(add_gee_area)
)

gee_plots_area = (

    ee_plots.map(add_gee_area)
)

gee_area_records = []

for scale, collection, id_field in [

    ("Fragment", gee_fragments_area, "fragment_id"),

    ("Plot", gee_plots_area, "plot_id")
]:

    features = collection.getInfo()["features"]

    for feature in features:

        p = feature["properties"]

        gee_area_records.append({

            "spatial_scale": scale,

            "unit_id": p[id_field],

            "forest_type": p["forest_type"],

            "local_utm_area_ha": p["area_ha"],

            "gee_area_ha": p["gee_area_ha"]
        })

gee_area_validation = pd.DataFrame(

    gee_area_records
)

gee_area_validation["difference_pct"] = (

    100

    * (

        gee_area_validation["gee_area_ha"]

        - gee_area_validation["local_utm_area_ha"]
    )

    / gee_area_validation["local_utm_area_ha"]
)

print("\nEARTH ENGINE AREA VALIDATION")

display(

    gee_area_validation.round(6)
)

assert (

    gee_area_validation["difference_pct"]

    .abs()

    .lt(1.0)

).all(), (

    "Unexpected area discrepancy between "
    "local and Earth Engine geometries."
)

# ============================================================
# 8. DEFINE SATELLITE CATALOGUE
# ============================================================

SATELLITE_CATALOGUE = {

    "Landsat_5": {

        "collection":

            "LANDSAT/LT05/C02/T1_L2",

        "start": "1985-01-01",

        "end": "2013-01-01",

        "resolution_m": 30
    },

    "Landsat_7": {

        "collection":

            "LANDSAT/LE07/C02/T1_L2",

        "start": "1999-01-01",

        "end": "2026-01-01",

        "resolution_m": 30
    },

    "Landsat_8": {

        "collection":

            "LANDSAT/LC08/C02/T1_L2",

        "start": "2013-01-01",

        "end": "2026-01-01",

        "resolution_m": 30
    },

    "Landsat_9": {

        "collection":

            "LANDSAT/LC09/C02/T1_L2",

        "start": "2021-01-01",

        "end": "2026-01-01",

        "resolution_m": 30
    },

    "Sentinel_2": {

        "collection":

            "COPERNICUS/S2_SR_HARMONIZED",

        "start": "2017-01-01",

        "end": "2026-01-01",

        "resolution_m": 10
    },

    "GEDI_L2A_monthly": {

        "collection":

            "LARSE/GEDI/GEDI02_A_002_MONTHLY",

        "start": "2019-01-01",

        "end": "2026-01-01",

        "resolution_m": 25
    },

    "GEDI_L2B_monthly": {

        "collection":

            "LARSE/GEDI/GEDI02_B_002_MONTHLY",

        "start": "2019-01-01",

        "end": "2026-01-01",

        "resolution_m": 25
    }
}

# ============================================================
# 9. INITIAL SATELLITE AVAILABILITY TEST
# ============================================================

catalogue_records = []

for sensor, config in (

    SATELLITE_CATALOGUE.items()
):

    print("\nChecking:", sensor)

    try:

        collection = (

            ee.ImageCollection(

                config["collection"]
            )

            .filterBounds(SATIPO_AOI)

            .filterDate(

                config["start"],

                config["end"]
            )
        )

        image_count = collection.size().getInfo()

        if image_count > 0:

            first_image = ee.Image(

                collection.sort(

                    "system:time_start"

                ).first()
            )

            last_image = ee.Image(

                collection.sort(

                    "system:time_start",

                    False

                ).first()
            )

            first_date = (

                first_image.date()

                .format("YYYY-MM-dd")

                .getInfo()
            )

            last_date = (

                last_image.date()

                .format("YYYY-MM-dd")

                .getInfo()
            )

        else:

            first_date = None

            last_date = None

        catalogue_records.append({

            "sensor": sensor,

            "collection_id":

                config["collection"],

            "nominal_resolution_m":

                config["resolution_m"],

            "candidate_images": image_count,

            "first_date": first_date,

            "last_date": last_date,

            "status": "OK"

                if image_count > 0

                else "No images"
        })

        print(

            "Candidate images:",

            image_count
        )

    except Exception as error:

        catalogue_records.append({

            "sensor": sensor,

            "collection_id":

                config["collection"],

            "nominal_resolution_m":

                config["resolution_m"],

            "candidate_images": None,

            "first_date": None,

            "last_date": None,

            "status": str(error)
        })

        print("Catalogue error:", error)

satellite_catalogue_audit = pd.DataFrame(

    catalogue_records
)

print("\n" + "=" * 70)
print("SATELLITE CATALOGUE AUDIT")
print("=" * 70)

display(satellite_catalogue_audit)

# ============================================================
# 10. EXPORT INITIALIZATION REPORT
# ============================================================

GEE_QA_DIR = (

    REMOTE_OUTPUTS / "TABLES"
)

GEE_QA_DIR.mkdir(

    parents=True,

    exist_ok=True
)

satellite_catalogue_audit.to_csv(

    GEE_QA_DIR

    / "SATELLITE_CATALOGUE_AUDIT.csv",

    index=False,

    encoding="utf-8-sig"
)

gee_area_validation.to_csv(

    GEE_QA_DIR

    / "GEE_AREA_VALIDATION.csv",

    index=False,

    encoding="utf-8-sig"
)

gee_layer_summary.to_csv(

    GEE_QA_DIR

    / "GEE_LAYER_SUMMARY.csv",

    index=False,

    encoding="utf-8-sig"
)

gee_initialization_manifest = {

    "project": "SATIPO_FOREST_PROJECT",

    "cell": 87,

    "gee_project": GEE_PROJECT,

    "earthengine_api_version":

        ee.__version__,

    "analysis_start": "1985-01-01",

    "analysis_end_exclusive": "2026-01-01",

    "field_census_year": 2019,

    "forest_labels_status":

        "Provisional map-based identification",

    "satellite_catalogue":

        SATELLITE_CATALOGUE,

    "limitations": [

        "Candidate image counts do not imply "

        "cloud-free observations.",

        "GEDI monthly image counts do not "

        "demonstrate valid footprint coverage.",

        "Satellite observations do not "

        "constitute independent forest replicates."
    ]
}

with open(

    GEOMETRY_DIR

    / "GEE_INITIALIZATION_MANIFEST.json",

    "w",

    encoding="utf-8"

) as file:

    json.dump(

        gee_initialization_manifest,

        file,

        indent=4
    )

# ============================================================
# 11. INTERACTIVE EARTH ENGINE MAP
# ============================================================

Map = geemap.Map()

Map.centerObject(

    SATIPO_AOI,

    zoom=15
)

Map.add_basemap("SATELLITE")

secondary_style = {

    "color": "DB9140",

    "fillColor": "DB914040",

    "width": 2
}

mature_style = {

    "color": "24634F",

    "fillColor": "24634F40",

    "width": 2
}

plot_style = {

    "color": "FFFFFF",

    "fillColor": "FFFFFF20",

    "width": 3
}

Map.addLayer(

    ee.FeatureCollection([

        ee_secondary_fragment

    ]).style(**secondary_style),

    {},

    "Late-secondary forest"
)

Map.addLayer(

    ee.FeatureCollection([

        ee_mature_fragment

    ]).style(**mature_style),

    {},

    "Mature forest"
)

Map.addLayer(

    ee_plots.style(**plot_style),

    {},

    "Permanent field plots"
)

Map.addLayerControl()

display(Map)

# ============================================================
# 12. FINAL REPORT
# ============================================================

print("\n" + "=" * 70)

print("EARTH ENGINE INITIALIZATION COMPLETED")

print("=" * 70)

print("Cloud project:", GEE_PROJECT)

print("Forest fragments: 2")

print("Permanent plots: 2")

print("Interior buffer units: 16")

print("Field census: 2019")

print("Historical analysis: 1985–2025")

print("Satellite catalogue audit saved.")

print("\nNEXT: CELL 88 — LANDSAT DATA AVAILABILITY")

print("=" * 70)


In [ ]:

!pip -q install earthengine-api

import ee

GEE_PROJECT = "brunei159"

ee.Authenticate()

ee.Initialize(project=GEE_PROJECT)

print("Earth Engine conectado correctamente")
print("Proyecto:", GEE_PROJECT)
print("Test:", ee.Number(1).getInfo())


In [ ]:

# ============================================================
# CELL 88 — LANDSAT OBSERVATION AVAILABILITY & QUALITY AUDIT
# SATIPO FOREST PROJECT
# STAGE 2 — REMOTE SENSING
# ============================================================

import ee
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm.auto import tqdm

import json
import time

# ============================================================
# 1. VERIFY EARTH ENGINE CONNECTION
# ============================================================

GEE_PROJECT = "brunei159"

ee.Initialize(project=GEE_PROJECT)

assert ee.Number(1).getInfo() == 1

print("Earth Engine connection: OK")

# ============================================================
# 2. PROJECT DIRECTORIES
# ============================================================

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

REMOTE_DIR = ROOT / "04_REMOTE_SENSING"

LANDSAT_DIR = REMOTE_DIR / "03_LANDSAT"

AUDIT_DIR = LANDSAT_DIR / "01_AVAILABILITY_AUDIT"

CACHE_DIR = AUDIT_DIR / "ANNUAL_CACHE"

FIGURE_DIR = (
    REMOTE_DIR / "08_OUTPUTS" / "FIGURES"
)

TABLE_DIR = (
    REMOTE_DIR / "08_OUTPUTS" / "TABLES"
)

for folder in [
    AUDIT_DIR,
    CACHE_DIR,
    FIGURE_DIR,
    TABLE_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

# ============================================================
# 3. ANALYSIS CONFIGURATION
# ============================================================

START_YEAR = 1985
END_YEAR = 2025

TARGET_CRS = "EPSG:32718"
TARGET_SCALE = 30

# Coverage thresholds for observation auditing.
# These are provisional quality criteria, not
# ecological significance thresholds.

COVERAGE_THRESHOLDS = [
    0.50,
    0.70,
    0.90
]

PRIMARY_COVERAGE = 0.70

AUDIT_VERSION = "V01"

YEARS = list(
    range(START_YEAR, END_YEAR + 1)
)

SENSORS = {

    "Landsat_5": {
        "collection": "LANDSAT/LT05/C02/T1_L2",
        "bands": [
            "SR_B1", "SR_B2", "SR_B3",
            "SR_B4", "SR_B5", "SR_B7"
        ]
    },

    "Landsat_7": {
        "collection": "LANDSAT/LE07/C02/T1_L2",
        "bands": [
            "SR_B1", "SR_B2", "SR_B3",
            "SR_B4", "SR_B5", "SR_B7"
        ]
    },

    "Landsat_8": {
        "collection": "LANDSAT/LC08/C02/T1_L2",
        "bands": [
            "SR_B2", "SR_B3", "SR_B4",
            "SR_B5", "SR_B6", "SR_B7"
        ]
    },

    "Landsat_9": {
        "collection": "LANDSAT/LC09/C02/T1_L2",
        "bands": [
            "SR_B2", "SR_B3", "SR_B4",
            "SR_B5", "SR_B6", "SR_B7"
        ]
    }
}

# ============================================================
# 4. RELOAD VALIDATED GEOMETRIES IF NECESSARY
# ============================================================

GEE_VECTOR_DIR = (

    REMOTE_DIR

    / "02_GEOMETRY_VALIDATION"

    / "GEE_GEOJSON"
)

def load_ee_geojson(path):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as file:

        data = json.load(file)

    return ee.FeatureCollection([

        ee.Feature(

            ee.Geometry(
                feature["geometry"],
                proj="EPSG:4326",
                geodesic=False
            ),

            {
                k: v
                for k, v in
                feature.get("properties", {}).items()
                if v is not None
            }
        )

        for feature in data["features"]
    ])

ee_fragments = load_ee_geojson(

    GEE_VECTOR_DIR

    / "SATIPO_FOREST_FRAGMENTS.geojson"
)

ee_plots = load_ee_geojson(

    GEE_VECTOR_DIR

    / "SATIPO_PERMANENT_PLOTS.geojson"
)

ee_conservation = load_ee_geojson(

    GEE_VECTOR_DIR

    / "SATIPO_CONSERVATION_AREA.geojson"
)

ee_buffers = load_ee_geojson(

    GEE_VECTOR_DIR

    / "SATIPO_INTERIOR_BUFFERS.geojson"
)

SATIPO_AOI = ee_conservation.geometry()

# ============================================================
# 5. DEFINE ANALYSIS UNITS
# ============================================================

def prepare_fragment(feature):

    return ee.Feature(

        feature.geometry(),

        {
            "unit_id": feature.get("fragment_id"),

            "forest_type": feature.get("forest_type"),

            "spatial_scale": "Fragment",

            "buffer_m": 0
        }
    )

def prepare_plot(feature):

    return ee.Feature(

        feature.geometry(),

        {
            "unit_id": feature.get("plot_id"),

            "forest_type": feature.get("forest_type"),

            "spatial_scale": "Plot",

            "buffer_m": 0
        }
    )

def prepare_core(feature):

    return ee.Feature(

        feature.geometry(),

        {
            "unit_id": ee.String(
                feature.get("unit_id")
            ).cat("_CORE30"),

            "forest_type": feature.get("forest_type"),

            "spatial_scale": "Fragment_core",

            "buffer_m": 30
        }
    )

ee_fragment_units = ee_fragments.map(
    prepare_fragment
)

ee_plot_units = ee_plots.map(
    prepare_plot
)

ee_core_units = (

    ee_buffers

    .filter(
        ee.Filter.eq(
            "spatial_scale",
            "Fragment"
        )
    )

    .filter(
        ee.Filter.eq(
            "buffer_m",
            30
        )
    )

    .map(prepare_core)
)

ee_audit_units = (

    ee_fragment_units

    .merge(ee_plot_units)

    .merge(ee_core_units)
)

assert ee_audit_units.size().getInfo() == 6

print("\nSpatial analysis units: 6")

# ============================================================
# 6. BUILD LANDSAT COLLECTION
# ============================================================

def get_sensor_collection(
    sensor,
    start_date,
    end_date
):

    config = SENSORS[sensor]

    return (

        ee.ImageCollection(
            config["collection"]
        )

        .filterBounds(SATIPO_AOI)

        .filterDate(
            start_date,
            end_date
        )

        .map(
            lambda image: image.set(
                "sensor_name",
                sensor
            )
        )
    )

def get_annual_collection(year):

    start = f"{year}-01-01"

    end = f"{year + 1}-01-01"

    collection = ee.ImageCollection([])

    for sensor in SENSORS:

        collection = collection.merge(

            get_sensor_collection(
                sensor,
                start,
                end
            )
        )

    return collection.sort(
        "system:time_start"
    )

# ============================================================
# 7. LANDSAT COLLECTION 2 QUALITY MASK
# ============================================================

# QA_PIXEL:
# Bit 0 = Fill
# Bit 1 = Dilated cloud
# Bit 2 = Cirrus (Landsat 8/9)
# Bit 3 = Cloud
# Bit 4 = Cloud shadow
# Bit 5 = Snow
#
# QA_RADSAT = 0 excludes radiometric saturation
# and other sensor-specific flagged conditions.
#
# Water is not masked at this stage because
# doing so could bias disturbance assessment.

def get_landsat_valid_mask(image):

    image = ee.Image(image)

    qa = image.select("QA_PIXEL")

    clear = (

        qa.bitwiseAnd(63).eq(0)
    )

    unsaturated = (

        image.select("QA_RADSAT").eq(0)
    )

    sensor = ee.String(
        image.get("sensor_name")
    )

    is_oli = (

        sensor.compareTo("Landsat_8").eq(0)

        .Or(
            sensor.compareTo("Landsat_9").eq(0)
        )
    )

    optical_bands = ee.List(

        ee.Algorithms.If(

            is_oli,

            [
                "SR_B2", "SR_B3", "SR_B4",
                "SR_B5", "SR_B6", "SR_B7"
            ],

            [
                "SR_B1", "SR_B2", "SR_B3",
                "SR_B4", "SR_B5", "SR_B7"
            ]
        )
    )

    optical_valid = (

        image.select(optical_bands)

        .mask()

        .reduce(ee.Reducer.min())
    )

    valid = (

        clear

        .And(unsaturated)

        .And(optical_valid)

        .rename("valid")

        .unmask(
            value=0,
            sameFootprint=False
        )

        .toFloat()
    )

    return valid

# ============================================================
# 8. AUDIT INDIVIDUAL SCENE COVERAGE
# ============================================================

def audit_landsat_image(image):

    image = ee.Image(image)

    valid = get_landsat_valid_mask(image)

    # Binary coverage raster:
    # 1 = valid observation
    # 0 = invalid or outside scene coverage.
    #
    # Mean reduction provides the valid fraction
    # on the common 30 m analysis grid.

    statistics = valid.reduceRegions(

        collection=ee_audit_units,

        reducer=ee.Reducer.mean(),

        scale=TARGET_SCALE,

        crs=TARGET_CRS,

        tileScale=2
    )

    acquisition_date = image.date()

    scene_id = image.get(
        "LANDSAT_PRODUCT_ID"
    )

    def add_metadata(feature):

        return ee.Feature(

            None,

            {
                "unit_id":
                    feature.get("unit_id"),

                "forest_type":
                    feature.get("forest_type"),

                "spatial_scale":
                    feature.get("spatial_scale"),

                "buffer_m":
                    feature.get("buffer_m"),

                "sensor":
                    image.get("sensor_name"),

                "scene_id":
                    scene_id,

                "date":
                    acquisition_date.format(
                        "YYYY-MM-dd"
                    ),

                "year":
                    acquisition_date.get("year"),

                "month":
                    acquisition_date.get("month"),

                "scene_cloud_cover_pct":
                    image.get("CLOUD_COVER"),

                "valid_fraction":
                    feature.get("mean")
            }
        )

    return statistics.map(
        add_metadata
    )

# ============================================================
# 9. TEST ON ONE IMAGE BEFORE FULL AUDIT
# ============================================================

test_collection = get_annual_collection(2019)

test_count = test_collection.size().getInfo()

assert test_count > 0

test_image = ee.Image(
    test_collection.first()
)

test_result = audit_landsat_image(
    test_image
).getInfo()

test_df = pd.DataFrame([

    feature["properties"]

    for feature in test_result["features"]
])

assert len(test_df) == 6

assert test_df["valid_fraction"].notna().all()

assert test_df["valid_fraction"].between(
    0, 1
).all()

print("\n2019 TEST SCENE")

display(test_df)

# ============================================================
# 10. DOWNLOAD ANNUAL AUDIT WITH CHECKPOINTS
# ============================================================

def fetch_audit_batch(
    image_list,
    offset,
    batch_size,
    retries=3
):

    subset = image_list.slice(

        offset,

        offset + batch_size
    )

    result = ee.FeatureCollection(

        subset.map(
            audit_landsat_image
        )

    ).flatten()

    for attempt in range(retries):

        try:

            response = result.getInfo()

            return [

                feature["properties"]

                for feature in response["features"]
            ]

        except Exception as error:

            if attempt == retries - 1:

                raise error

            time.sleep(
                5 * (attempt + 1)
            )

annual_frames = []

failed_years = []

BATCH_SIZE = 10

print("\n" + "=" * 70)

print("LANDSAT QUALITY AUDIT — 1985–2025")

print("=" * 70)

for year in tqdm(
    YEARS,
    desc="Auditing Landsat years"
):

    cache_path = (

        CACHE_DIR

        / f"LANDSAT_QA_{year}_{AUDIT_VERSION}.csv"
    )

    # Resume completed annual downloads.

    if cache_path.exists():

        cached = pd.read_csv(cache_path)

        required = {
            "unit_id",
            "sensor",
            "scene_id",
            "year",
            "valid_fraction"
        }

        if (

            required.issubset(cached.columns)

            and len(cached) > 0

            and cached["year"].eq(year).all()

            and len(cached) % 6 == 0

            and cached["valid_fraction"]
                .between(0, 1).all()
        ):

            annual_frames.append(cached)

            continue

    try:

        collection = get_annual_collection(year)

        n_images = collection.size().getInfo()

        if n_images == 0:

            print(f"{year}: no candidate scenes")

            continue

        image_list = collection.toList(
            n_images
        )

        year_records = []

        for offset in range(

            0,

            n_images,

            BATCH_SIZE
        ):

            records = fetch_audit_batch(

                image_list,

                offset,

                BATCH_SIZE
            )

            year_records.extend(records)

        year_df = pd.DataFrame(
            year_records
        )

        expected_rows = n_images * 6

        assert len(year_df) == expected_rows, (

            f"{year}: expected {expected_rows} rows, "

            f"received {len(year_df)}."
        )

        assert year_df["valid_fraction"].notna().all()

        assert year_df["valid_fraction"].between(
            0, 1
        ).all()

        assert not year_df.duplicated(

            subset=[
                "sensor",
                "scene_id",
                "unit_id"
            ]

        ).any()

        # Atomic annual checkpoint

        temporary_path = cache_path.with_suffix(
            ".tmp"
        )

        year_df.to_csv(

            temporary_path,

            index=False,

            encoding="utf-8-sig"
        )

        temporary_path.replace(
            cache_path
        )

        annual_frames.append(year_df)

        print(

            f"{year}: {n_images} scenes, "

            f"{len(year_df)} unit observations saved"
        )

    except Exception as error:

        failed_years.append({

            "year": year,

            "error": str(error)
        })

        print(
            f"WARNING — {year}: {error}"
        )

# ============================================================
# 11. CONSOLIDATE SCENE-LEVEL AUDIT
# ============================================================

assert len(annual_frames) > 0, (

    "No annual audit data available."
)

landsat_scene_audit = pd.concat(

    annual_frames,

    ignore_index=True
)

landsat_scene_audit["date"] = pd.to_datetime(

    landsat_scene_audit["date"]
)

landsat_scene_audit["valid_pct"] = (

    100

    * landsat_scene_audit["valid_fraction"]
)

landsat_scene_audit = (

    landsat_scene_audit

    .sort_values([
        "date",
        "sensor",
        "unit_id"
    ])

    .reset_index(drop=True)
)

assert not landsat_scene_audit.duplicated(

    subset=[
        "sensor",
        "scene_id",
        "unit_id"
    ]

).any()

for threshold in COVERAGE_THRESHOLDS:

    column = f"valid_ge_{int(threshold * 100)}"

    landsat_scene_audit[column] = (

        landsat_scene_audit["valid_fraction"]

        .ge(threshold)
    )

landsat_scene_audit.to_csv(

    AUDIT_DIR / "LANDSAT_SCENE_QUALITY_AUDIT.csv",

    index=False,

    encoding="utf-8-sig"
)

failed_years_df = pd.DataFrame(

    failed_years,

    columns=["year", "error"]
)

failed_years_df.to_csv(

    AUDIT_DIR / "LANDSAT_AUDIT_FAILED_YEARS.csv",

    index=False
)

if failed_years:

    display(failed_years_df)

    raise RuntimeError(

        "Some years failed. Rerun Cell 88. "

        "Completed annual checkpoints will be reused."
    )

# ============================================================
# 12. SENSOR-LEVEL QUALITY SUMMARY
# ============================================================

landsat_sensor_quality = (

    landsat_scene_audit

    .groupby(
        ["sensor", "unit_id"],
        as_index=False
    )

    .agg(

        candidate_scenes=(
            "scene_id",
            "count"
        ),

        mean_valid_pct=(
            "valid_pct",
            "mean"
        ),

        median_valid_pct=(
            "valid_pct",
            "median"
        ),

        scenes_ge_50=(
            "valid_ge_50",
            "sum"
        ),

        scenes_ge_70=(
            "valid_ge_70",
            "sum"
        ),

        scenes_ge_90=(
            "valid_ge_90",
            "sum"
        )
    )
)

landsat_sensor_quality["usable_pct_70"] = (

    100

    * landsat_sensor_quality["scenes_ge_70"]

    / landsat_sensor_quality["candidate_scenes"]
)

print("\nSENSOR-LEVEL QUALITY SUMMARY")

display(

    landsat_sensor_quality.round(3)
)

# ============================================================
# 13. ANNUAL QUALITY SUMMARY
# ============================================================

landsat_annual_quality = (

    landsat_scene_audit

    .groupby(

        [
            "year",
            "unit_id",
            "forest_type",
            "spatial_scale"
        ],

        as_index=False
    )

    .agg(

        candidate_scenes=(
            "scene_id",
            "count"
        ),

        mean_valid_pct=(
            "valid_pct",
            "mean"
        ),

        median_valid_pct=(
            "valid_pct",
            "median"
        ),

        scenes_ge_50=(
            "valid_ge_50",
            "sum"
        ),

        scenes_ge_70=(
            "valid_ge_70",
            "sum"
        ),

        scenes_ge_90=(
            "valid_ge_90",
            "sum"
        )
    )
)

# Include all calendar years explicitly.

unit_metadata = (

    landsat_scene_audit[

        [
            "unit_id",
            "forest_type",
            "spatial_scale"
        ]
    ]

    .drop_duplicates()
)

annual_grid = pd.DataFrame({

    "year": YEARS
}).merge(

    unit_metadata,

    how="cross"
)

landsat_annual_quality = annual_grid.merge(

    landsat_annual_quality,

    on=[
        "year",
        "unit_id",
        "forest_type",
        "spatial_scale"
    ],

    how="left"
)

count_columns = [

    "candidate_scenes",

    "scenes_ge_50",

    "scenes_ge_70",

    "scenes_ge_90"
]

landsat_annual_quality[count_columns] = (

    landsat_annual_quality[count_columns]

    .fillna(0)

    .astype(int)
)

landsat_annual_quality["quality_class"] = (

    np.select(

        [
            landsat_annual_quality[
                "scenes_ge_70"
            ].ge(5),

            landsat_annual_quality[
                "scenes_ge_70"
            ].ge(3),

            landsat_annual_quality[
                "scenes_ge_70"
            ].ge(1)
        ],

        [
            "High_support",

            "Moderate_support",

            "Low_support"
        ],

        default="No_usable_scene"
    )
)

print("\nANNUAL FRAGMENT COVERAGE")

display(

    landsat_annual_quality.loc[

        landsat_annual_quality[
            "spatial_scale"
        ].eq("Fragment")

    ].round(3)
)

# ============================================================
# 14. PAIRED SCENE AVAILABILITY
# ============================================================

# Both fragments must meet the coverage threshold
# within the same scene for paired comparisons.

fragment_audit = landsat_scene_audit.loc[

    landsat_scene_audit[
        "spatial_scale"
    ].eq("Fragment")

].copy()

landsat_paired_scenes = (

    fragment_audit

    .pivot(

        index=[
            "sensor",
            "scene_id",
            "date",
            "year",
            "month"
        ],

        columns="unit_id",

        values="valid_fraction"
    )

    .reset_index()
)

landsat_paired_scenes.columns.name = None

assert {

    "FRAG_01",
    "FRAG_02"

}.issubset(

    landsat_paired_scenes.columns
)

assert landsat_paired_scenes[

    ["FRAG_01", "FRAG_02"]

].notna().all().all()

landsat_paired_scenes["minimum_valid_fraction"] = (

    landsat_paired_scenes[

        ["FRAG_01", "FRAG_02"]

    ].min(axis=1)
)

for threshold in COVERAGE_THRESHOLDS:

    column = f"paired_ge_{int(threshold * 100)}"

    landsat_paired_scenes[column] = (

        landsat_paired_scenes[
            "minimum_valid_fraction"
        ].ge(threshold)
    )

landsat_paired_annual = (

    landsat_paired_scenes

    .groupby(
        "year",
        as_index=False
    )

    .agg(

        candidate_scenes=(
            "scene_id",
            "count"
        ),

        paired_ge_50=(
            "paired_ge_50",
            "sum"
        ),

        paired_ge_70=(
            "paired_ge_70",
            "sum"
        ),

        paired_ge_90=(
            "paired_ge_90",
            "sum"
        )
    )

    .set_index("year")

    .reindex(YEARS, fill_value=0)

    .reset_index()
)

landsat_paired_annual["adequate_support"] = (

    landsat_paired_annual[
        "paired_ge_70"
    ].ge(3)
)

print("\nPAIRED ANNUAL SCENE AVAILABILITY")

display(landsat_paired_annual)

# ============================================================
# 15. MONTHLY OBSERVATION AVAILABILITY
# ============================================================

landsat_monthly_quality = (

    landsat_paired_scenes

    .groupby(
        "month",
        as_index=False
    )

    .agg(

        candidate_scenes=(
            "scene_id",
            "count"
        ),

        paired_ge_50=(
            "paired_ge_50",
            "sum"
        ),

        paired_ge_70=(
            "paired_ge_70",
            "sum"
        ),

        paired_ge_90=(
            "paired_ge_90",
            "sum"
        )
    )

    .set_index("month")

    .reindex(
        range(1, 13),
        fill_value=0
    )

    .reset_index()
)

landsat_monthly_quality["usable_pct_70"] = (

    100

    * landsat_monthly_quality["paired_ge_70"]

    / landsat_monthly_quality[
        "candidate_scenes"
    ].replace(0, np.nan)
)

print("\nMONTHLY PAIRED COVERAGE")

display(

    landsat_monthly_quality.round(3)
)

# ============================================================
# 16. YEAR-MONTH QUALITY MATRIX
# ============================================================

landsat_year_month = (

    landsat_paired_scenes

    .groupby(
        ["year", "month"],
        as_index=False
    )

    .agg(

        candidate_scenes=(
            "scene_id",
            "count"
        ),

        paired_ge_70=(
            "paired_ge_70",
            "sum"
        )
    )
)

monthly_matrix = (

    landsat_year_month

    .pivot(

        index="year",

        columns="month",

        values="paired_ge_70"
    )

    .reindex(

        index=YEARS,

        columns=range(1, 13)
    )

    .fillna(0)

    .astype(int)
)

# ============================================================
# 17. EVALUATE POTENTIAL SEASONAL WINDOWS
# ============================================================

# These windows are candidate acquisition windows.
# Final season selection must also consider local
# climatology and phenology, not only image counts.

SEASON_WINDOWS = {

    "Annual": list(range(1, 13)),

    "May_September": [5, 6, 7, 8, 9],

    "June_August": [6, 7, 8],

    "June_September": [6, 7, 8, 9],

    "July_September": [7, 8, 9]
}

window_records = []

for window_name, months in (

    SEASON_WINDOWS.items()
):

    subset = landsat_paired_scenes.loc[

        landsat_paired_scenes["month"]

        .isin(months)
    ]

    annual_counts = (

        subset.groupby("year")[
            "paired_ge_70"
        ]

        .sum()

        .reindex(
            YEARS,
            fill_value=0
        )
    )

    window_records.append({

        "window": window_name,

        "months": ",".join(
            map(str, months)
        ),

        "total_paired_scenes":
            int(annual_counts.sum()),

        "years_with_ge_1_scene":
            int(annual_counts.ge(1).sum()),

        "years_with_ge_3_scenes":
            int(annual_counts.ge(3).sum()),

        "years_with_ge_5_scenes":
            int(annual_counts.ge(5).sum()),

        "median_scenes_per_year":
            float(annual_counts.median()),

        "minimum_scenes_per_year":
            int(annual_counts.min()),

        "scenes_in_2019":
            int(annual_counts.loc[2019])
    })

landsat_season_windows = pd.DataFrame(

    window_records
)

print("\nPOTENTIAL SEASONAL WINDOWS")

display(landsat_season_windows)

# ============================================================
# 18. FIELD CENSUS YEAR AUDIT
# ============================================================

landsat_2019_quality = (

    landsat_paired_scenes.loc[

        landsat_paired_scenes["year"].eq(2019)

    ].copy()
)

landsat_2019_quality = (

    landsat_2019_quality

    .sort_values(

        "minimum_valid_fraction",

        ascending=False
    )
)

print("\n2019 — BEST PAIRED LANDSAT OBSERVATIONS")

display(

    landsat_2019_quality.head(15).round(4)
)

# ============================================================
# 19. COVERAGE AUDIT FIGURE
# ============================================================

plt.rcParams.update({

    "font.family": "DejaVu Sans",

    "font.size": 10,

    "axes.spines.top": False,

    "axes.spines.right": False,

    "axes.labelsize": 11
})

fig, axes = plt.subplots(

    3,

    1,

    figsize=(15, 12),

    gridspec_kw={

        "height_ratios": [1, 1, 1.8]
    }
)

# Panel A — Annual candidate and usable scenes

ax = axes[0]

ax.bar(

    landsat_paired_annual["year"],

    landsat_paired_annual["candidate_scenes"],

    color="#D5D9DC",

    label="Candidate scenes"
)

ax.bar(

    landsat_paired_annual["year"],

    landsat_paired_annual["paired_ge_70"],

    color="#24634F",

    label="Paired valid coverage ≥70%"
)

ax.axvline(

    2019,

    color="#B64B40",

    linestyle="--",

    linewidth=1.5,

    label="Field census"
)

ax.set_ylabel("Number of scenes")

ax.set_title(

    "(a) Annual Landsat observation availability",

    loc="left",

    fontweight="bold"
)

ax.legend(

    frameon=False,

    ncol=3,

    loc="upper left"
)

ax.set_xlim(1984.4, 2025.6)

ax.grid(

    axis="y",

    alpha=0.2
)

# Panel B — Monthly usability

ax = axes[1]

ax.bar(

    landsat_monthly_quality["month"],

    landsat_monthly_quality["usable_pct_70"],

    color="#527B98",

    edgecolor="white"
)

ax.set_xticks(range(1, 13))

ax.set_xticklabels([

    "Jan", "Feb", "Mar", "Apr",

    "May", "Jun", "Jul", "Aug",

    "Sep", "Oct", "Nov", "Dec"
])

ax.set_ylabel(

    "Usable paired scenes (%)"
)

ax.set_ylim(0, 100)

ax.set_title(

    "(b) Monthly observation usability",

    loc="left",

    fontweight="bold"
)

ax.grid(

    axis="y",

    alpha=0.2
)

# Panel C — Temporal coverage matrix

ax = axes[2]

heatmap = ax.imshow(

    monthly_matrix.values,

    aspect="auto",

    cmap="YlGnBu",

    interpolation="nearest",

    vmin=0
)

ax.set_xticks(range(12))

ax.set_xticklabels([

    "Jan", "Feb", "Mar", "Apr",

    "May", "Jun", "Jul", "Aug",

    "Sep", "Oct", "Nov", "Dec"
])

year_ticks = [

    i

    for i, year in enumerate(YEARS)

    if year % 5 == 0
]

ax.set_yticks(year_ticks)

ax.set_yticklabels([

    YEARS[i]

    for i in year_ticks
])

ax.set_ylabel("Year")

ax.set_title(

    "(c) Paired observations with ≥70% valid coverage",

    loc="left",

    fontweight="bold"
)

cbar = fig.colorbar(

    heatmap,

    ax=ax,

    pad=0.015
)

cbar.set_label(

    "Number of paired scenes"
)

# Highlight census year

census_index = YEARS.index(2019)

ax.axhline(

    census_index - 0.5,

    color="#B64B40",

    linewidth=1
)

ax.axhline(

    census_index + 0.5,

    color="#B64B40",

    linewidth=1
)

fig.tight_layout(pad=2)

for extension in [

    "png",

    "pdf",

    "svg"
]:

    fig.savefig(

        FIGURE_DIR

        / f"FIG_LANDSAT_AVAILABILITY.{extension}",

        dpi=600,

        bbox_inches="tight",

        facecolor="white"
    )

plt.show()

# ============================================================
# 20. EXPORT QUALITY AUDIT TABLES
# ============================================================

landsat_audit_tables = {

    "LANDSAT_SENSOR_QUALITY":

        landsat_sensor_quality,

    "LANDSAT_ANNUAL_QUALITY":

        landsat_annual_quality,

    "LANDSAT_PAIRED_SCENES":

        landsat_paired_scenes,

    "LANDSAT_PAIRED_ANNUAL":

        landsat_paired_annual,

    "LANDSAT_MONTHLY_QUALITY":

        landsat_monthly_quality,

    "LANDSAT_YEAR_MONTH":

        landsat_year_month,

    "LANDSAT_SEASON_WINDOWS":

        landsat_season_windows,

    "LANDSAT_2019_QUALITY":

        landsat_2019_quality
}

for table_name, dataframe in (

    landsat_audit_tables.items()
):

    dataframe.to_csv(

        AUDIT_DIR / f"{table_name}.csv",

        index=False,

        encoding="utf-8-sig"
    )

audit_workbook = (

    TABLE_DIR

    / "SATIPO_LANDSAT_AVAILABILITY.xlsx"
)

with pd.ExcelWriter(

    audit_workbook,

    engine="openpyxl"

) as writer:

    for table_name, dataframe in (

        landsat_audit_tables.items()
    ):

        dataframe.to_excel(

            writer,

            sheet_name=table_name[:31],

            index=False
        )

# ============================================================
# 21. FINAL AUDIT SUMMARY
# ============================================================

n_candidate_scenes = (

    len(landsat_paired_scenes)
)

n_paired_50 = int(

    landsat_paired_scenes[
        "paired_ge_50"
    ].sum()
)

n_paired_70 = int(

    landsat_paired_scenes[
        "paired_ge_70"
    ].sum()
)

n_paired_90 = int(

    landsat_paired_scenes[
        "paired_ge_90"
    ].sum()
)

years_ge_3 = int(

    landsat_paired_annual[
        "paired_ge_70"
    ].ge(3).sum()
)

years_ge_5 = int(

    landsat_paired_annual[
        "paired_ge_70"
    ].ge(5).sum()
)

missing_years = (

    landsat_paired_annual.loc[

        landsat_paired_annual[
            "paired_ge_70"
        ].eq(0),

        "year"
    ]

    .astype(int)

    .tolist()
)

audit_manifest = {

    "project": "SATIPO_FOREST_PROJECT",

    "cell": 88,

    "audit_version": AUDIT_VERSION,

    "start_year": START_YEAR,

    "end_year": END_YEAR,

    "target_crs": TARGET_CRS,

    "target_scale_m": TARGET_SCALE,

    "candidate_scenes": n_candidate_scenes,

    "paired_scenes_ge_50": n_paired_50,

    "paired_scenes_ge_70": n_paired_70,

    "paired_scenes_ge_90": n_paired_90,

    "years_with_ge_3_paired_scenes": years_ge_3,

    "years_with_ge_5_paired_scenes": years_ge_5,

    "years_without_paired_scenes_ge_70":
        missing_years,

    "analysis_units": 6,

    "qa_pixel_excluded_bits": [0, 1, 2, 3, 4, 5],

    "qa_radsat_required_value": 0,

    "water_masked": False,

    "failed_years": failed_years,

    "limitations": [

        "Scene-level valid fractions are grid-based "
        "coverage estimates.",

        "Multiple scenes may contain observations "
        "from the same acquisition date.",

        "QA masks do not guarantee complete removal "
        "of haze or residual cloud contamination.",

        "Seasonal windows remain provisional.",

        "Scene availability does not establish "
        "cross-sensor spectral comparability.",

        "Satellite pixels are not independent "
        "forest replicates."
    ]
}

with open(

    AUDIT_DIR

    / "LANDSAT_AVAILABILITY_MANIFEST.json",

    "w",

    encoding="utf-8"

) as file:

    json.dump(

        audit_manifest,

        file,

        indent=4
    )

print("\n" + "=" * 75)

print("SATIPO — LANDSAT AVAILABILITY REPORT")

print("=" * 75)

print(

    "Candidate scenes:",

    n_candidate_scenes
)

print(

    "Scene-unit observations:",

    len(landsat_scene_audit)
)

print(

    "Paired scenes ≥50% coverage:",

    n_paired_50
)

print(

    "Paired scenes ≥70% coverage:",

    n_paired_70
)

print(

    "Paired scenes ≥90% coverage:",

    n_paired_90
)

print(

    "Years with ≥3 paired scenes:",

    years_ge_3,

    "/ 41"
)

print(

    "Years with ≥5 paired scenes:",

    years_ge_5,

    "/ 41"
)

print(

    "Years without paired usable scenes:",

    missing_years
)

print(

    "Usable paired scenes in 2019:",

    int(

        landsat_2019_quality[
            "paired_ge_70"
        ].sum()
    )
)

print("\nAudit workbook:")

print(audit_workbook)

print("\nAnnual checkpoints:")

print(CACHE_DIR)

print("\n" + "=" * 75)

print("LANDSAT AVAILABILITY AUDIT COMPLETED")

print("NEXT: CELL 89 — LANDSAT PREPROCESSING")

print("=" * 75)


In [ ]:

# ============================================================
# CELL 89 — LANDSAT SPECTRAL PREPROCESSING
# SATIPO FOREST PROJECT
# STAGE 2 — REMOTE SENSING
# ============================================================

import ee
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm.auto import tqdm

import json
import time

# ============================================================
# 1. INITIALIZATION
# ============================================================

GEE_PROJECT = "brunei159"

ee.Initialize(project=GEE_PROJECT)

assert ee.Number(1).getInfo() == 1

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

REMOTE_DIR = ROOT / "04_REMOTE_SENSING"

LANDSAT_DIR = REMOTE_DIR / "03_LANDSAT"

AUDIT_DIR = LANDSAT_DIR / "01_AVAILABILITY_AUDIT"

SPECTRAL_DIR = LANDSAT_DIR / "02_SPECTRAL_PREPROCESSING"

CACHE_DIR_89 = SPECTRAL_DIR / "ANNUAL_CACHE"

FIGURE_DIR = REMOTE_DIR / "08_OUTPUTS" / "FIGURES"

TABLE_DIR = REMOTE_DIR / "08_OUTPUTS" / "TABLES"

for folder in [
    SPECTRAL_DIR,
    CACHE_DIR_89,
    FIGURE_DIR,
    TABLE_DIR
]:

    folder.mkdir(
        parents=True,
        exist_ok=True
    )

print("Earth Engine connection: OK")

# ============================================================
# 2. CONFIGURATION
# ============================================================

START_YEAR = 1985
END_YEAR = 2025

YEARS = list(
    range(START_YEAR, END_YEAR + 1)
)

TARGET_CRS = "EPSG:32718"

TARGET_SCALE = 30

MIN_PAIRED_COVERAGE = 0.50

PRIMARY_COVERAGE = 0.70

STRICT_COVERAGE = 0.90

PRIMARY_MONTHS = [5, 6, 7, 8, 9]

PROCESSING_VERSION = "V01"

REFLECTANCE_SCALE = 0.0000275

REFLECTANCE_OFFSET = -0.2

DENOMINATOR_EPS = 1e-6

BAND_NAMES = [

    "Blue",
    "Green",
    "Red",
    "NIR",
    "SWIR1",
    "SWIR2"
]

INDEX_NAMES = [

    "NDVI",
    "EVI",
    "NDMI",
    "NBR",
    "NBR2"
]

SPECTRAL_BANDS = BAND_NAMES + INDEX_NAMES

SENSORS = {

    "Landsat_5": {

        "collection": "LANDSAT/LT05/C02/T1_L2",

        "bands": [
            "SR_B1", "SR_B2", "SR_B3",
            "SR_B4", "SR_B5", "SR_B7"
        ]
    },

    "Landsat_7": {

        "collection": "LANDSAT/LE07/C02/T1_L2",

        "bands": [
            "SR_B1", "SR_B2", "SR_B3",
            "SR_B4", "SR_B5", "SR_B7"
        ]
    },

    "Landsat_8": {

        "collection": "LANDSAT/LC08/C02/T1_L2",

        "bands": [
            "SR_B2", "SR_B3", "SR_B4",
            "SR_B5", "SR_B6", "SR_B7"
        ]
    },

    "Landsat_9": {

        "collection": "LANDSAT/LC09/C02/T1_L2",

        "bands": [
            "SR_B2", "SR_B3", "SR_B4",
            "SR_B5", "SR_B6", "SR_B7"
        ]
    }
}

# ============================================================
# 3. LOAD QUALITY AUDIT FROM CELL 88
# ============================================================

paired_path = (

    AUDIT_DIR

    / "LANDSAT_PAIRED_SCENES.csv"
)

scene_quality_path = (

    AUDIT_DIR

    / "LANDSAT_SCENE_QUALITY_AUDIT.csv"
)

assert paired_path.exists()

assert scene_quality_path.exists()

paired_quality = pd.read_csv(

    paired_path,

    parse_dates=["date"]
)

scene_quality = pd.read_csv(

    scene_quality_path,

    parse_dates=["date"]
)

assert len(paired_quality) == 1039

assert len(scene_quality) == 6234

selected_scenes = (

    paired_quality.loc[

        paired_quality[
            "minimum_valid_fraction"
        ].ge(MIN_PAIRED_COVERAGE)

    ].copy()
)

selected_scenes = selected_scenes.sort_values(

    ["date", "sensor", "scene_id"]
)

assert len(selected_scenes) == 388

assert not selected_scenes.duplicated(

    ["sensor", "scene_id"]

).any()

print("\nSELECTED LANDSAT SCENES")

print("Candidate scenes:", len(paired_quality))

print("Selected scenes:", len(selected_scenes))

print(

    "Paired ≥70%:",

    selected_scenes[
        "minimum_valid_fraction"
    ].ge(0.70).sum()
)

print(

    "Paired ≥90%:",

    selected_scenes[
        "minimum_valid_fraction"
    ].ge(0.90).sum()
)

display(

    selected_scenes.groupby(

        "sensor"

    ).size().rename("selected_scenes")
)

# ============================================================
# 4. RELOAD VALIDATED GEOMETRIES
# ============================================================

GEE_VECTOR_DIR = (

    REMOTE_DIR

    / "02_GEOMETRY_VALIDATION"

    / "GEE_GEOJSON"
)

def load_geojson_89(path):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as file:

        data = json.load(file)

    features = []

    for feature in data["features"]:

        geometry = ee.Geometry(

            feature["geometry"],

            proj="EPSG:4326",

            geodesic=False
        )

        properties = {

            k: v

            for k, v in

            feature.get("properties", {}).items()

            if v is not None
        }

        features.append(

            ee.Feature(
                geometry,
                properties
            )
        )

    return ee.FeatureCollection(features)

ee_fragments = load_geojson_89(

    GEE_VECTOR_DIR

    / "SATIPO_FOREST_FRAGMENTS.geojson"
)

ee_plots = load_geojson_89(

    GEE_VECTOR_DIR

    / "SATIPO_PERMANENT_PLOTS.geojson"
)

ee_conservation = load_geojson_89(

    GEE_VECTOR_DIR

    / "SATIPO_CONSERVATION_AREA.geojson"
)

ee_buffers = load_geojson_89(

    GEE_VECTOR_DIR

    / "SATIPO_INTERIOR_BUFFERS.geojson"
)

SATIPO_AOI = ee_conservation.geometry()

# ============================================================
# 5. DEFINE SIX SPATIAL UNITS
# ============================================================

def fragment_unit_89(feature):

    return ee.Feature(

        feature.geometry(),

        {
            "unit_id":
                feature.get("fragment_id"),

            "forest_type":
                feature.get("forest_type"),

            "spatial_scale": "Fragment",

            "buffer_m": 0
        }
    )

def plot_unit_89(feature):

    return ee.Feature(

        feature.geometry(),

        {
            "unit_id":
                feature.get("plot_id"),

            "forest_type":
                feature.get("forest_type"),

            "spatial_scale": "Plot",

            "buffer_m": 0
        }
    )

def core_unit_89(feature):

    return ee.Feature(

        feature.geometry(),

        {
            "unit_id":

                ee.String(
                    feature.get("unit_id")
                ).cat("_CORE30"),

            "forest_type":
                feature.get("forest_type"),

            "spatial_scale": "Fragment_core",

            "buffer_m": 30
        }
    )

ee_spectral_units = (

    ee_fragments.map(fragment_unit_89)

    .merge(
        ee_plots.map(plot_unit_89)
    )

    .merge(

        ee_buffers

        .filter(

            ee.Filter.eq(
                "spatial_scale",
                "Fragment"
            )
        )

        .filter(

            ee.Filter.eq(
                "buffer_m",
                30
            )
        )

        .map(core_unit_89)
    )
)

assert ee_spectral_units.size().getInfo() == 6

print("Spatial analysis units: 6")

# ============================================================
# 6. LANDSAT QUALITY MASK AND REFLECTANCE
# ============================================================

def preprocess_landsat_89(image, sensor):

    image = ee.Image(image)

    config = SENSORS[sensor]

    raw_optical = image.select(

        config["bands"],

        BAND_NAMES
    )

    # Same QA logic as Cell 88.

    clear = (

        image.select("QA_PIXEL")

        .bitwiseAnd(63)

        .eq(0)
    )

    unsaturated = (

        image.select("QA_RADSAT")

        .eq(0)
    )

    optical_valid = (

        raw_optical.mask()

        .reduce(ee.Reducer.min())
    )

    valid_mask = (

        clear

        .And(unsaturated)

        .And(optical_valid)
    )

    # Collection 2 Level 2 surface reflectance.

    reflectance = (

        raw_optical

        .multiply(REFLECTANCE_SCALE)

        .add(REFLECTANCE_OFFSET)

        .updateMask(valid_mask)

        .toFloat()
    )

    # Do not clamp reflectance to [0,1].
    # Negative values can occur after atmospheric
    # correction. They are flagged for auditing.

    return reflectance.copyProperties(

        image,

        [
            "system:time_start",
            "LANDSAT_PRODUCT_ID",
            "SPACECRAFT_ID",
            "WRS_PATH",
            "WRS_ROW",
            "CLOUD_COVER",
            "SUN_ELEVATION",
            "SUN_AZIMUTH"
        ]
    ).set(

        "sensor_name",
        sensor
    )

# ============================================================
# 7. SPECTRAL INDEX CALCULATION
# ============================================================

def safe_normalized_difference(
    image,
    band_a,
    band_b,
    output_name
):

    a = image.select(band_a)

    b = image.select(band_b)

    denominator = a.add(b)

    # Explicit calculation avoids automatic masking
    # of negative inputs by normalizedDifference().

    index = (

        a.subtract(b)

        .divide(denominator)

        .updateMask(

            denominator.abs().gt(
                DENOMINATOR_EPS
            )
        )

        .rename(output_name)
    )

    return index

def add_spectral_indices_89(image):

    image = ee.Image(image)

    ndvi = safe_normalized_difference(

        image,
        "NIR",
        "Red",
        "NDVI"
    )

    ndmi = safe_normalized_difference(

        image,
        "NIR",
        "SWIR1",
        "NDMI"
    )

    nbr = safe_normalized_difference(

        image,
        "NIR",
        "SWIR2",
        "NBR"
    )

    nbr2 = safe_normalized_difference(

        image,
        "SWIR1",
        "SWIR2",
        "NBR2"
    )

    nir = image.select("NIR")

    red = image.select("Red")

    blue = image.select("Blue")

    evi_denominator = (

        nir

        .add(red.multiply(6))

        .subtract(blue.multiply(7.5))

        .add(1)
    )

    evi = (

        nir.subtract(red)

        .multiply(2.5)

        .divide(evi_denominator)

        .updateMask(

            evi_denominator.abs().gt(

                DENOMINATOR_EPS
            )
        )

        .rename("EVI")
    )

    return image.addBands([

        ndvi,
        evi,
        ndmi,
        nbr,
        nbr2
    ])

# ============================================================
# 8. SPECTRAL QUALITY FLAGS
# ============================================================

def add_spectral_flags_89(image):

    image = ee.Image(image)

    optical = image.select(BAND_NAMES)

    negative_reflectance = (

        optical.lt(0)

        .reduce(ee.Reducer.max())

        .rename("negative_reflectance")
    )

    high_reflectance = (

        optical.gt(1)

        .reduce(ee.Reducer.max())

        .rename("reflectance_gt_one")
    )

    normalized_indices = image.select([

        "NDVI",
        "NDMI",
        "NBR",
        "NBR2"
    ])

    index_outside_range = (

        normalized_indices.abs().gt(1)

        .reduce(ee.Reducer.max())

        .rename("normalized_index_outside_range")
    )

    evi_extreme = (

        image.select("EVI")

        .abs().gt(1.5)

        .rename("extreme_evi")
    )

    # Flags are retained as diagnostic variables.
    # No automatic clipping of index values.

    return image.addBands([

        negative_reflectance,

        high_reflectance,

        index_outside_range,

        evi_extreme
    ])

# ============================================================
# 9. SPATIAL STATISTICS
# ============================================================

REDUCER_89 = (

    ee.Reducer.mean()

    .combine(

        reducer2=ee.Reducer.median(),

        sharedInputs=True
    )

    .combine(

        reducer2=ee.Reducer.stdDev(),

        sharedInputs=True
    )

    .combine(

        reducer2=ee.Reducer.count(),

        sharedInputs=True
    )
)

FLAG_NAMES = [

    "negative_reflectance",

    "reflectance_gt_one",

    "normalized_index_outside_range",

    "extreme_evi"
]

EXTRACTION_BANDS = (

    SPECTRAL_BANDS + FLAG_NAMES
)

def extract_scene_89(image):

    image = ee.Image(image)

    stats = (

        image.select(EXTRACTION_BANDS)

        .reduceRegions(

            collection=ee_spectral_units,

            reducer=REDUCER_89,

            scale=TARGET_SCALE,

            crs=TARGET_CRS,

            tileScale=2
        )
    )

    def attach_metadata(feature):

        return ee.Feature(

            None,

            feature.toDictionary()
        ).set({

            "scene_id":

                image.get("LANDSAT_PRODUCT_ID"),

            "sensor":

                image.get("sensor_name"),

            "date":

                image.date().format("YYYY-MM-dd"),

            "year":

                image.date().get("year"),

            "month":

                image.date().get("month"),

            "wrs_path":

                image.get("WRS_PATH"),

            "wrs_row":

                image.get("WRS_ROW"),

            "sun_elevation":

                image.get("SUN_ELEVATION"),

            "sun_azimuth":

                image.get("SUN_AZIMUTH"),

            "scene_cloud_cover_pct":

                image.get("CLOUD_COVER")
        })

    return stats.map(attach_metadata)

# ============================================================
# 10. BUILD COLLECTION OF SELECTED SCENES
# ============================================================

def build_selected_collection_89(
    selected_dataframe
):

    merged = ee.ImageCollection([])

    for sensor, config in SENSORS.items():

        sensor_ids = (

            selected_dataframe.loc[

                selected_dataframe[
                    "sensor"
                ].eq(sensor),

                "scene_id"
            ].tolist()
        )

        if not sensor_ids:

            continue

        collection = (

            ee.ImageCollection(

                config["collection"]
            )

            .filterBounds(SATIPO_AOI)

            .filter(

                ee.Filter.inList(

                    "LANDSAT_PRODUCT_ID",

                    sensor_ids
                )
            )

            .map(

                lambda image, s=sensor:

                    preprocess_landsat_89(
                        image,
                        s
                    )
            )

            .map(add_spectral_indices_89)

            .map(add_spectral_flags_89)
        )

        merged = merged.merge(collection)

    return merged.sort("system:time_start")

# ============================================================
# 11. SINGLE-SCENE VALIDATION
# ============================================================

test_selection = (

    selected_scenes.loc[

        selected_scenes["year"].eq(2019)

        & selected_scenes["sensor"].eq("Landsat_8")

        & selected_scenes[
            "minimum_valid_fraction"
        ].ge(0.90)

    ].head(1)
)

assert len(test_selection) == 1

test_collection_89 = (

    build_selected_collection_89(

        test_selection
    )
)

assert test_collection_89.size().getInfo() == 1

test_image_89 = ee.Image(

    test_collection_89.first()
)

test_response_89 = (

    extract_scene_89(

        test_image_89

    ).getInfo()
)

test_spectral_df = pd.DataFrame([

    feature["properties"]

    for feature in

    test_response_89["features"]
])

assert len(test_spectral_df) == 6

assert {

    "NDVI_mean",
    "EVI_mean",
    "NDMI_mean",
    "NBR_mean",
    "NBR2_mean"

}.issubset(test_spectral_df.columns)

print("\n2019 SPECTRAL VALIDATION")

display(

    test_spectral_df[[

        "unit_id",

        "forest_type",

        "date",

        "NDVI_mean",

        "EVI_mean",

        "NDMI_mean",

        "NBR_mean",

        "NBR2_mean",

        "NDVI_count"

    ]].round(4)
)

# ============================================================
# 12. BATCH EXTRACTION WITH RETRIES
# ============================================================

def fetch_spectral_batch_89(

    image_list,

    offset,

    batch_size,

    retries=3
):

    subset = image_list.slice(

        offset,

        offset + batch_size
    )

    result = ee.FeatureCollection(

        subset.map(extract_scene_89)

    ).flatten()

    for attempt in range(retries):

        try:

            response = result.getInfo()

            return [

                feature["properties"]

                for feature in

                response["features"]
            ]

        except Exception as error:

            if attempt == retries - 1:

                raise error

            time.sleep(

                5 * (attempt + 1)
            )

# ============================================================
# 13. ANNUAL EXTRACTION AND CHECKPOINTS
# ============================================================

annual_spectral_frames = []

spectral_failed_years = []

BATCH_SIZE_89 = 5

EXPECTED_UNITS = {

    "FRAG_01",

    "FRAG_02",

    "P01",

    "P02",

    "FRAG_01_CORE30",

    "FRAG_02_CORE30"
}

def validate_spectral_cache_89(
    dataframe,
    expected_selection
):

    required = {

        "scene_id",

        "sensor",

        "unit_id",

        "year",

        "NDVI_mean",

        "NBR_mean",

        "NDMI_mean"
    }

    if not required.issubset(

        dataframe.columns
    ):

        return False

    if len(dataframe) != (

        len(expected_selection) * 6
    ):

        return False

    if dataframe.duplicated([

        "sensor",

        "scene_id",

        "unit_id"

    ]).any():

        return False

    actual_keys = set(

        zip(

            dataframe["sensor"],

            dataframe["scene_id"],

            dataframe["unit_id"]
        )
    )

    expected_keys = {

        (row.sensor, row.scene_id, unit)

        for row in

        expected_selection.itertuples()

        for unit in EXPECTED_UNITS
    }

    return actual_keys == expected_keys

print("\n" + "=" * 70)

print("LANDSAT SPECTRAL EXTRACTION — 1985–2025")

print("=" * 70)

for year in tqdm(

    YEARS,

    desc="Extracting Landsat spectra"
):

    year_selection = (

        selected_scenes.loc[

            selected_scenes["year"].eq(year)

        ].copy()
    )

    expected_scenes = len(year_selection)

    if expected_scenes == 0:

        continue

    cache_path = (

        CACHE_DIR_89

        / f"LANDSAT_SPECTRAL_{year}_{PROCESSING_VERSION}.csv"
    )

    if cache_path.exists():

        cached = pd.read_csv(cache_path)

        if validate_spectral_cache_89(

            cached,

            year_selection
        ):

            annual_spectral_frames.append(

                cached
            )

            continue

    try:

        annual_collection = (

            build_selected_collection_89(

                year_selection
            )
        )

        actual_scenes = (

            annual_collection.size().getInfo()
        )

        assert actual_scenes == expected_scenes, (

            f"{year}: expected {expected_scenes}, "

            f"found {actual_scenes}"
        )

        image_list = annual_collection.toList(

            actual_scenes
        )

        records = []

        for offset in range(

            0,

            actual_scenes,

            BATCH_SIZE_89
        ):

            records.extend(

                fetch_spectral_batch_89(

                    image_list,

                    offset,

                    BATCH_SIZE_89
                )
            )

        year_df = pd.DataFrame(records)

        assert validate_spectral_cache_89(

            year_df,

            year_selection
        ), f"Invalid annual results: {year}"

        temporary_path = (

            cache_path.with_suffix(".tmp")
        )

        year_df.to_csv(

            temporary_path,

            index=False,

            encoding="utf-8-sig"
        )

        temporary_path.replace(cache_path)

        annual_spectral_frames.append(year_df)

        print(

            f"{year}: {actual_scenes} scenes, "

            f"{len(year_df)} spatial observations"
        )

    except Exception as error:

        spectral_failed_years.append({

            "year": year,

            "error": str(error)
        })

        print(

            f"WARNING — {year}: {error}"
        )

# ============================================================
# 14. CHECK FOR INCOMPLETE YEARS
# ============================================================

failed_df_89 = pd.DataFrame(

    spectral_failed_years,

    columns=["year", "error"]
)

failed_df_89.to_csv(

    SPECTRAL_DIR

    / "LANDSAT_SPECTRAL_FAILED_YEARS.csv",

    index=False
)

if spectral_failed_years:

    display(failed_df_89)

    raise RuntimeError(

        "Incomplete extraction. Rerun Cell 89. "

        "Valid annual checkpoints will be reused."
    )

# ============================================================
# 15. CONSOLIDATE SPECTRAL DATABASE
# ============================================================

landsat_spectral_master = pd.concat(

    annual_spectral_frames,

    ignore_index=True
)

landsat_spectral_master["date"] = (

    pd.to_datetime(

        landsat_spectral_master["date"]
    )
)

assert len(landsat_spectral_master) == (

    len(selected_scenes) * 6
)

assert len(landsat_spectral_master) == 2328

assert validate_spectral_cache_89(

    landsat_spectral_master,

    selected_scenes
)

# ============================================================
# 16. MERGE LOCAL AND PAIRED COVERAGE
# ============================================================

local_quality = scene_quality[[

    "sensor",

    "scene_id",

    "unit_id",

    "valid_fraction"
]].copy()

landsat_spectral_master = (

    landsat_spectral_master.merge(

        local_quality,

        on=[

            "sensor",

            "scene_id",

            "unit_id"
        ],

        how="left",

        validate="one_to_one"
    )
)

paired_metadata = selected_scenes[[

    "sensor",

    "scene_id",

    "minimum_valid_fraction"
]].copy()

landsat_spectral_master = (

    landsat_spectral_master.merge(

        paired_metadata,

        on=[

            "sensor",

            "scene_id"
        ],

        how="left",

        validate="many_to_one"
    )
)

assert landsat_spectral_master[

    "valid_fraction"

].notna().all()

assert landsat_spectral_master[

    "minimum_valid_fraction"

].notna().all()

landsat_spectral_master["paired_ge_70"] = (

    landsat_spectral_master[

        "minimum_valid_fraction"

    ].ge(0.70)
)

landsat_spectral_master["paired_ge_90"] = (

    landsat_spectral_master[

        "minimum_valid_fraction"

    ].ge(0.90)
)

landsat_spectral_master["local_ge_70"] = (

    landsat_spectral_master[

        "valid_fraction"

    ].ge(0.70)
)

landsat_spectral_master["primary_season"] = (

    landsat_spectral_master["month"]

    .isin(PRIMARY_MONTHS)
)

landsat_spectral_master["sensor_family"] = (

    landsat_spectral_master["sensor"]

    .map({

        "Landsat_5": "TM_ETM",

        "Landsat_7": "TM_ETM",

        "Landsat_8": "OLI",

        "Landsat_9": "OLI"
    })
)

landsat_spectral_master["slc_off"] = (

    landsat_spectral_master["sensor"]

    .eq("Landsat_7")

    & landsat_spectral_master["date"].ge(

        pd.Timestamp("2003-05-31")
    )
)

landsat_spectral_master = (

    landsat_spectral_master

    .sort_values([

        "date",

        "sensor",

        "unit_id"
    ])

    .reset_index(drop=True)
)

# ============================================================
# 17. SPECTRAL QUALITY CONTROL
# ============================================================

spectral_qa_records = []

for index in INDEX_NAMES:

    column = f"{index}_mean"

    values = pd.to_numeric(

        landsat_spectral_master[column],

        errors="coerce"
    )

    limit = 1.5 if index == "EVI" else 1.0

    spectral_qa_records.append({

        "index": index,

        "total_rows": len(values),

        "nonmissing": int(

            values.notna().sum()
        ),

        "missing": int(

            values.isna().sum()
        ),

        "minimum": values.min(),

        "maximum": values.max(),

        "mean": values.mean(),

        "median": values.median(),

        "outside_diagnostic_range": int(

            values.abs().gt(limit).sum()
        )
    })

landsat_spectral_qa = pd.DataFrame(

    spectral_qa_records
)

print("\nSPECTRAL INDEX QUALITY CONTROL")

display(

    landsat_spectral_qa.round(5)
)

# ============================================================
# 18. AUDIT REFLECTANCE AND INDEX FLAGS
# ============================================================

flag_summary_records = []

for sensor in SENSORS:

    subset = landsat_spectral_master.loc[

        landsat_spectral_master["sensor"].eq(sensor)

        & landsat_spectral_master[

            "spatial_scale"

        ].eq("Fragment")
    ]

    for flag in FLAG_NAMES:

        column = f"{flag}_mean"

        values = pd.to_numeric(

            subset[column],

            errors="coerce"
        )

        flag_summary_records.append({

            "sensor": sensor,

            "flag": flag,

            "scene_unit_rows": len(subset),

            "mean_flagged_fraction":

                values.mean(),

            "maximum_flagged_fraction":

                values.max(),

            "rows_with_any_flag": int(

                values.gt(0).sum()
            )
        })

landsat_flag_summary = pd.DataFrame(

    flag_summary_records
)

print("\nSPECTRAL FLAG SUMMARY")

display(

    landsat_flag_summary.round(5)
)

# ============================================================
# 19. CREATE PRELIMINARY PAIRED DATASET
# ============================================================

# These are native-sensor index values.
# They are not yet cross-sensor harmonized.

fragment_spectra = (

    landsat_spectral_master.loc[

        landsat_spectral_master[

            "spatial_scale"

        ].eq("Fragment")

    ].copy()
)

paired_index_columns = [

    f"{index}_median"

    for index in INDEX_NAMES
]

paired_spectral_native = (

    fragment_spectra.pivot(

        index=[

            "sensor",

            "scene_id",

            "date",

            "year",

            "month",

            "minimum_valid_fraction"
        ],

        columns="unit_id",

        values=paired_index_columns
    )
)

paired_spectral_native.columns = [

    f"{index}_{unit}"

    for index, unit in

    paired_spectral_native.columns
]

paired_spectral_native = (

    paired_spectral_native.reset_index()
)

assert len(paired_spectral_native) == 388

for index in INDEX_NAMES:

    secondary_column = (

        f"{index}_median_FRAG_01"
    )

    mature_column = (

        f"{index}_median_FRAG_02"
    )

    paired_spectral_native[

        f"delta_{index}"

    ] = (

        paired_spectral_native[

            secondary_column
        ]

        - paired_spectral_native[

            mature_column
        ]
    )

paired_spectral_native["primary_season"] = (

    paired_spectral_native["month"]

    .isin(PRIMARY_MONTHS)
)

paired_spectral_native["paired_ge_70"] = (

    paired_spectral_native[

        "minimum_valid_fraction"

    ].ge(0.70)
)

paired_spectral_native["paired_ge_90"] = (

    paired_spectral_native[

        "minimum_valid_fraction"

    ].ge(0.90)
)

# ============================================================
# 20. SENSOR-SPECIFIC SPECTRAL SUMMARY
# ============================================================

primary_fragment_spectra = (

    fragment_spectra.loc[

        fragment_spectra["paired_ge_70"]

        & fragment_spectra["primary_season"]

    ].copy()
)

sensor_spectral_summary = (

    primary_fragment_spectra

    .groupby(

        ["sensor", "forest_type"],

        as_index=False
    )

    .agg(

        n_scenes=("scene_id", "nunique"),

        first_year=("year", "min"),

        last_year=("year", "max"),

        NDVI=("NDVI_median", "median"),

        EVI=("EVI_median", "median"),

        NDMI=("NDMI_median", "median"),

        NBR=("NBR_median", "median"),

        NBR2=("NBR2_median", "median")
    )
)

print("\nNATIVE-SENSOR SPECTRAL SUMMARY")

display(

    sensor_spectral_summary.round(4)
)

# ============================================================
# 21. PRELIMINARY 2019 SPECTRAL CONDITIONS
# ============================================================

spectral_2019 = (

    landsat_spectral_master.loc[

        landsat_spectral_master["year"].eq(2019)

        & landsat_spectral_master["paired_ge_70"]

        & landsat_spectral_master["local_ge_70"]

    ].copy()
)

spectral_2019_summary = (

    spectral_2019

    .groupby(

        [

            "unit_id",

            "forest_type",

            "spatial_scale",

            "sensor"
        ],

        as_index=False
    )

    .agg(

        n_scenes=("scene_id", "nunique"),

        NDVI=("NDVI_median", "median"),

        EVI=("EVI_median", "median"),

        NDMI=("NDMI_median", "median"),

        NBR=("NBR_median", "median"),

        NBR2=("NBR2_median", "median")
    )
)

print("\n2019 SPECTRAL CONDITIONS — BY SENSOR")

display(

    spectral_2019_summary.round(4)
)

# ============================================================
# 22. PRELIMINARY NATIVE-SENSOR FIGURE
# ============================================================

# Diagnostic only: do not interpret changes
# across sensor transitions as ecological trends.

sensor_colors = {

    "Landsat_5": "#527B98",

    "Landsat_7": "#DB9140",

    "Landsat_8": "#24634F",

    "Landsat_9": "#795C92"
}

forest_markers = {

    "Late_secondary": "o",

    "Mature": "^"
}

fig, axes = plt.subplots(

    5,

    1,

    figsize=(15, 16),

    sharex=True
)

for ax, index in zip(

    axes,

    INDEX_NAMES
):

    for sensor, color in (

        sensor_colors.items()
    ):

        for forest, marker in (

            forest_markers.items()
        ):

            subset = primary_fragment_spectra.loc[

                primary_fragment_spectra[

                    "sensor"

                ].eq(sensor)

                & primary_fragment_spectra[

                    "forest_type"

                ].eq(forest)
            ]

            ax.scatter(

                subset["date"],

                subset[f"{index}_median"],

                color=color,

                marker=marker,

                s=22,

                alpha=0.65,

                edgecolors="none",

                label=f"{sensor} — {forest}"
            )

    ax.axvline(

        pd.Timestamp("2019-01-01"),

        color="#B64B40",

        linestyle="--",

        linewidth=1.2
    )

    ax.set_ylabel(index)

    ax.grid(alpha=0.2)

    ax.spines["top"].set_visible(False)

    ax.spines["right"].set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(

    handles,

    labels,

    loc="upper center",

    ncol=4,

    frameon=False,

    fontsize=9,

    bbox_to_anchor=(0.5, 0.965)
)

fig.suptitle(

    "Native-sensor Landsat spectral observations\n"

    "May–September | Paired valid coverage ≥70%",

    fontsize=16,

    fontweight="bold",

    y=0.995
)

axes[-1].set_xlabel("Acquisition date")

fig.tight_layout(

    rect=[0, 0, 1, 0.92]
)

for extension in [

    "png",

    "pdf",

    "svg"
]:

    fig.savefig(

        FIGURE_DIR

        / f"FIG_LANDSAT_NATIVE_SPECTRAL_QA.{extension}",

        dpi=600,

        bbox_inches="tight",

        facecolor="white"
    )

plt.show()

# ============================================================
# 23. EXPORT SCIENTIFIC TABLES
# ============================================================

spectral_tables = {

    "LANDSAT_SPECTRAL_MASTER":

        landsat_spectral_master,

    "LANDSAT_SPECTRAL_QA":

        landsat_spectral_qa,

    "LANDSAT_SPECTRAL_FLAGS":

        landsat_flag_summary,

    "LANDSAT_PAIRED_NATIVE":

        paired_spectral_native,

    "LANDSAT_SENSOR_SPECTRAL":

        sensor_spectral_summary,

    "LANDSAT_2019_SPECTRAL":

        spectral_2019_summary
}

for name, dataframe in (

    spectral_tables.items()
):

    dataframe.to_csv(

        SPECTRAL_DIR / f"{name}.csv",

        index=False,

        encoding="utf-8-sig"
    )

spectral_workbook = (

    TABLE_DIR

    / "SATIPO_LANDSAT_SPECTRAL_PREPROCESSING.xlsx"
)

with pd.ExcelWriter(

    spectral_workbook,

    engine="openpyxl"

) as writer:

    for name, dataframe in (

        spectral_tables.items()
    ):

        dataframe.to_excel(

            writer,

            sheet_name=name[:31],

            index=False
        )

# ============================================================
# 24. SAVE PROCESSING MANIFEST
# ============================================================

spectral_manifest = {

    "project": "SATIPO_FOREST_PROJECT",

    "cell": 89,

    "processing_version": PROCESSING_VERSION,

    "start_year": START_YEAR,

    "end_year": END_YEAR,

    "target_crs": TARGET_CRS,

    "target_scale_m": TARGET_SCALE,

    "reflectance_scale": REFLECTANCE_SCALE,

    "reflectance_offset": REFLECTANCE_OFFSET,

    "minimum_paired_coverage":

        MIN_PAIRED_COVERAGE,

    "primary_paired_coverage":

        PRIMARY_COVERAGE,

    "strict_paired_coverage":

        STRICT_COVERAGE,

    "provisional_primary_months":

        PRIMARY_MONTHS,

    "selected_scenes":

        len(selected_scenes),

    "scene_unit_rows":

        len(landsat_spectral_master),

    "spatial_units": 6,

    "reflectance_bands": BAND_NAMES,

    "spectral_indices": INDEX_NAMES,

    "diagnostic_flags": FLAG_NAMES,

    "denominator_epsilon":

        DENOMINATOR_EPS,

    "cross_sensor_harmonized": False,

    "reflectance_clipped": False,

    "index_values_clipped": False,

    "failed_years": spectral_failed_years,

    "limitations": [

        "Native-sensor reflectance is not yet "

        "radiometrically harmonized.",

        "Spatial index statistics are calculated "

        "after pixel-level index computation.",

        "Spatial standard deviations describe "

        "within-unit variability, not independent "

        "forest replication.",

        "Index values outside diagnostic ranges "

        "are retained and flagged for review.",

        "The 2019 summaries describe satellite "

        "observations during the inventory year, "

        "not necessarily the exact census dates.",

        "Landsat 7 SLC-off gaps remain masked.",

        "Forest labels remain provisional until "

        "confirmed by the original investigators."
    ]
}

with open(

    SPECTRAL_DIR

    / "LANDSAT_SPECTRAL_MANIFEST.json",

    "w",

    encoding="utf-8"

) as file:

    json.dump(

        spectral_manifest,

        file,

        indent=4
    )

# ============================================================
# 25. FINAL REPORT
# ============================================================

print("\n" + "=" * 75)

print("SATIPO — LANDSAT SPECTRAL PREPROCESSING REPORT")

print("=" * 75)

print(

    "Selected scenes:",

    len(selected_scenes)
)

print(

    "Scene-unit observations:",

    len(landsat_spectral_master)
)

print(

    "Spectral indices:",

    len(INDEX_NAMES)
)

print(

    "Reflectance bands:",

    len(BAND_NAMES)
)

print(

    "Paired scenes ≥70%:",

    paired_spectral_native[

        "paired_ge_70"

    ].sum()
)

print(

    "Primary-season paired scenes:",

    (

        paired_spectral_native["paired_ge_70"]

        & paired_spectral_native["primary_season"]

    ).sum()
)

print(

    "Paired scenes in 2019 ≥70%:",

    (

        paired_spectral_native["year"].eq(2019)

        & paired_spectral_native["paired_ge_70"]

    ).sum()
)

print(

    "Failed years:",

    len(spectral_failed_years)
)

print(

    "Cross-sensor harmonization:",

    "PENDING"
)

print("\nSpectral master:")

print(

    SPECTRAL_DIR

    / "LANDSAT_SPECTRAL_MASTER.csv"
)

print("\nWorkbook:")

print(spectral_workbook)

print("\nAnnual checkpoints:")

print(CACHE_DIR_89)

print("\n" + "=" * 75)

print("LANDSAT SPECTRAL PREPROCESSING COMPLETED")

print("NEXT: CELL 90 — CROSS-SENSOR HARMONIZATION")

print("=" * 75)


In [ ]:

# ============================================================
# CELL 90A — RADIOMETRIC QUALITY AUDIT
# SATIPO FOREST PROJECT
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
import json

# ============================================================
# 1. PATHS
# ============================================================

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

REMOTE_DIR = ROOT / "04_REMOTE_SENSING"

LANDSAT_DIR = REMOTE_DIR / "03_LANDSAT"

SPECTRAL_DIR = (
    LANDSAT_DIR / "02_SPECTRAL_PREPROCESSING"
)

RADIOMETRIC_DIR = (
    LANDSAT_DIR / "03_RADIOMETRIC_QA"
)

TABLE_DIR = REMOTE_DIR / "08_OUTPUTS" / "TABLES"

FIGURE_DIR = REMOTE_DIR / "08_OUTPUTS" / "FIGURES"

for folder in [
    RADIOMETRIC_DIR,
    TABLE_DIR,
    FIGURE_DIR
]:

    folder.mkdir(
        parents=True,
        exist_ok=True
    )

# ============================================================
# 2. LOAD NATIVE SPECTRAL MASTER
# ============================================================

source_path = (
    SPECTRAL_DIR
    / "LANDSAT_SPECTRAL_MASTER.csv"
)

assert source_path.exists(), (
    "Cell 89 spectral master not found."
)

radiometric_master = pd.read_csv(

    source_path,

    parse_dates=["date"]
)

assert len(radiometric_master) == 2328

assert radiometric_master[
    "scene_id"
].nunique() == 388

assert not radiometric_master.duplicated([

    "sensor",
    "scene_id",
    "unit_id"

]).any()

INDEX_NAMES = [
    "NDVI",
    "EVI",
    "NDMI",
    "NBR",
    "NBR2"
]

BAND_NAMES = [
    "Blue",
    "Green",
    "Red",
    "NIR",
    "SWIR1",
    "SWIR2"
]

NORMALIZED_INDICES = [
    "NDVI",
    "NDMI",
    "NBR",
    "NBR2"
]

FLAG_NAMES = [

    "negative_reflectance_mean",

    "reflectance_gt_one_mean",

    "normalized_index_outside_range_mean",

    "extreme_evi_mean"
]

print("=" * 75)

print("SATIPO — RADIOMETRIC QUALITY AUDIT")

print("=" * 75)

print("Scene-unit rows:", len(radiometric_master))

print(
    "Unique scenes:",
    radiometric_master["scene_id"].nunique()
)

# ============================================================
# 3. IDENTIFY MISSING OBSERVATIONS
# ============================================================

index_mean_cols = [
    f"{index}_mean"
    for index in INDEX_NAMES
]

index_median_cols = [
    f"{index}_median"
    for index in INDEX_NAMES
]

radiometric_master["missing_any_index"] = (

    radiometric_master[
        index_mean_cols + index_median_cols
    ]

    .isna()

    .any(axis=1)
)

missing_spectral_rows = (

    radiometric_master.loc[

        radiometric_master[
            "missing_any_index"
        ]

    ].copy()
)

print("\nMISSING SPECTRAL OBSERVATIONS")

display(

    missing_spectral_rows[[

        "sensor",
        "scene_id",
        "date",
        "unit_id",
        "spatial_scale",
        "valid_fraction",
        "NDVI_count"

    ]]
)

# ============================================================
# 4. INDEX-SPECIFIC RANGE AUDIT
# ============================================================

for index in NORMALIZED_INDICES:

    for statistic in ["mean", "median"]:

        column = f"{index}_{statistic}"

        radiometric_master[
            f"{column}_outside_range"
        ] = (

            radiometric_master[column]

            .abs()

            .gt(1)
        )

range_flag_columns = [

    f"{index}_{statistic}_outside_range"

    for index in NORMALIZED_INDICES

    for statistic in ["mean", "median"]
]

radiometric_master["any_index_outside_range"] = (

    radiometric_master[
        range_flag_columns
    ].any(axis=1)
)

index_range_audit = []

for index in INDEX_NAMES:

    for statistic in ["mean", "median"]:

        column = f"{index}_{statistic}"

        values = radiometric_master[column]

        limit = (
            1.5 if index == "EVI" else 1.0
        )

        index_range_audit.append({

            "index": index,

            "statistic": statistic,

            "nonmissing": values.notna().sum(),

            "missing": values.isna().sum(),

            "minimum": values.min(),

            "maximum": values.max(),

            "outside_range": (
                values.abs().gt(limit).sum()
            ),

            "outside_range_pct": (

                100

                * values.abs().gt(limit).sum()

                / max(values.notna().sum(), 1)
            )
        })

index_range_audit = pd.DataFrame(
    index_range_audit
)

print("\nINDEX RANGE AUDIT")

display(index_range_audit.round(5))

# ============================================================
# 5. REFLECTANCE BAND AUDIT
# ============================================================

band_audit_records = []

for sensor, group in (

    radiometric_master.groupby("sensor")
):

    for band in BAND_NAMES:

        for statistic in ["mean", "median"]:

            column = f"{band}_{statistic}"

            values = group[column]

            band_audit_records.append({

                "sensor": sensor,

                "band": band,

                "statistic": statistic,

                "n_nonmissing": values.notna().sum(),

                "minimum": values.min(),

                "q01": values.quantile(0.01),

                "q05": values.quantile(0.05),

                "median": values.median(),

                "q95": values.quantile(0.95),

                "maximum": values.max(),

                "negative_values": values.lt(0).sum(),

                "values_gt_one": values.gt(1).sum()
            })

band_radiometric_audit = pd.DataFrame(

    band_audit_records
)

print("\nREFLECTANCE BAND AUDIT")

display(

    band_radiometric_audit.round(5)
)

# ============================================================
# 6. DIAGNOSTIC FLAG THRESHOLDS
# ============================================================

# These thresholds are exploratory.
# They are NOT final exclusion criteria.

FLAG_THRESHOLDS = [

    0.00,
    0.01,
    0.05,
    0.10,
    0.25,
    0.50
]

threshold_records = []

fragment_qa = radiometric_master.loc[

    radiometric_master[
        "spatial_scale"
    ].eq("Fragment")

].copy()

for sensor, group in (

    fragment_qa.groupby("sensor")
):

    for flag in FLAG_NAMES:

        values = group[flag]

        for threshold in FLAG_THRESHOLDS:

            n_flagged = values.gt(
                threshold
            ).sum()

            threshold_records.append({

                "sensor": sensor,

                "flag": flag,

                "threshold_fraction": threshold,

                "n_observations": len(group),

                "n_flagged": n_flagged,

                "flagged_pct": (

                    100 * n_flagged / len(group)
                )
            })

radiometric_threshold_audit = pd.DataFrame(

    threshold_records
)

print("\nRADIOMETRIC THRESHOLD AUDIT")

display(

    radiometric_threshold_audit.loc[

        radiometric_threshold_audit[
            "threshold_fraction"
        ].isin([0.01, 0.05, 0.10, 0.25])

        & radiometric_threshold_audit[
            "flag"
        ].isin([

            "negative_reflectance_mean",

            "normalized_index_outside_range_mean"
        ])

    ].round(4)
)

# ============================================================
# 7. CLASSIFY RADIOMETRIC SEVERITY
# ============================================================

severity_columns = [

    "negative_reflectance_mean",

    "reflectance_gt_one_mean",

    "normalized_index_outside_range_mean",

    "extreme_evi_mean"
]

radiometric_master["maximum_flag_fraction"] = (

    radiometric_master[
        severity_columns
    ].max(axis=1)
)

conditions = [

    radiometric_master["missing_any_index"],

    radiometric_master[
        "maximum_flag_fraction"
    ].gt(0.25),

    radiometric_master[
        "maximum_flag_fraction"
    ].gt(0.10),

    radiometric_master[
        "maximum_flag_fraction"
    ].gt(0.05),

    radiometric_master[
        "maximum_flag_fraction"
    ].gt(0.01),

    radiometric_master[
        "maximum_flag_fraction"
    ].gt(0)
]

severity_labels = [

    "Missing",

    "Critical_gt25pct",

    "High_10to25pct",

    "Moderate_5to10pct",

    "Low_1to5pct",

    "Trace_le1pct"
]

radiometric_master["radiometric_severity"] = (

    np.select(

        conditions,

        severity_labels,

        default="No_flags"
    )
)

severity_summary = (

    radiometric_master

    .groupby([

        "sensor",

        "spatial_scale",

        "radiometric_severity"

    ])

    .size()

    .rename("n_observations")

    .reset_index()
)

print("\nRADIOMETRIC SEVERITY SUMMARY")

display(severity_summary)

# ============================================================
# 8. IDENTIFY MOST PROBLEMATIC SCENES
# ============================================================

critical_radiometric_rows = (

    radiometric_master.loc[

        radiometric_master[
            "maximum_flag_fraction"
        ].gt(0.10)

        | radiometric_master[
            "any_index_outside_range"
        ]

        | radiometric_master[
            "missing_any_index"
        ]

    ]

    .sort_values(

        "maximum_flag_fraction",

        ascending=False
    )

    .copy()
)

critical_display_cols = [

    "sensor",
    "scene_id",
    "date",
    "unit_id",
    "forest_type",
    "valid_fraction",
    "negative_reflectance_mean",
    "normalized_index_outside_range_mean",
    "NDVI_mean",
    "NDVI_median",
    "Red_mean",
    "Red_median",
    "NIR_mean",
    "NIR_median",
    "radiometric_severity"
]

print("\nMOST PROBLEMATIC OBSERVATIONS")

display(

    critical_radiometric_rows[
        critical_display_cols
    ].head(30).round(5)
)

# ============================================================
# 9. SCENE-LEVEL PAIRED QUALITY
# ============================================================

fragment_qa = radiometric_master.loc[

    radiometric_master[
        "spatial_scale"
    ].eq("Fragment")

].copy()

assert len(fragment_qa) == 776

paired_radiometric_qa = (

    fragment_qa

    .groupby([

        "sensor",
        "scene_id",
        "date",
        "year",
        "month"

    ], as_index=False)

    .agg(

        n_fragments=(
            "unit_id",
            "nunique"
        ),

        minimum_valid_fraction=(
            "valid_fraction",
            "min"
        ),

        maximum_negative_fraction=(
            "negative_reflectance_mean",
            "max"
        ),

        maximum_high_reflectance_fraction=(
            "reflectance_gt_one_mean",
            "max"
        ),

        maximum_invalid_index_fraction=(
            "normalized_index_outside_range_mean",
            "max"
        ),

        maximum_extreme_evi_fraction=(
            "extreme_evi_mean",
            "max"
        ),

        any_index_outside_range=(
            "any_index_outside_range",
            "max"
        ),

        any_missing=(
            "missing_any_index",
            "max"
        )
    )
)

assert len(paired_radiometric_qa) == 388

assert paired_radiometric_qa[
    "n_fragments"
].eq(2).all()

paired_radiometric_qa[
    "maximum_radiometric_flag"
] = (

    paired_radiometric_qa[[

        "maximum_negative_fraction",

        "maximum_high_reflectance_fraction",

        "maximum_invalid_index_fraction",

        "maximum_extreme_evi_fraction"

    ]].max(axis=1)
)

paired_radiometric_qa["primary_season"] = (

    paired_radiometric_qa["month"]

    .isin([5, 6, 7, 8, 9])
)

# ============================================================
# 10. SCENE RETENTION SENSITIVITY
# ============================================================

# Assess the cost of excluding whole scenes.
# The next cell will instead test pixel-level masks.

retention_records = []

for coverage in [0.50, 0.70, 0.90]:

    for max_flag in [

        1.00,
        0.25,
        0.10,
        0.05,
        0.01,
        0.00
    ]:

        for seasonal in [False, True]:

            selected = (

                paired_radiometric_qa.loc[

                    paired_radiometric_qa[
                        "minimum_valid_fraction"
                    ].ge(coverage)

                    & paired_radiometric_qa[
                        "maximum_radiometric_flag"
                    ].le(max_flag)

                    & ~paired_radiometric_qa[
                        "any_missing"
                    ]

                    & (

                        paired_radiometric_qa[
                            "primary_season"
                        ]

                        if seasonal

                        else True
                    )

                ].copy()
            )

            annual_counts = (

                selected.groupby("year")

                .size()

                .reindex(

                    range(1985, 2026),

                    fill_value=0
                )
            )

            retention_records.append({

                "minimum_coverage": coverage,

                "maximum_flag_fraction": max_flag,

                "window": (

                    "May_September"

                    if seasonal

                    else "Annual"
                ),

                "retained_scenes": len(selected),

                "years_ge_1": (
                    annual_counts.ge(1).sum()
                ),

                "years_ge_3": (
                    annual_counts.ge(3).sum()
                ),

                "years_ge_5": (
                    annual_counts.ge(5).sum()
                ),

                "scenes_2019": (

                    selected["year"]

                    .eq(2019)

                    .sum()
                ),

                "median_scenes_per_year": (

                    annual_counts.median()
                )
            })

radiometric_retention = pd.DataFrame(

    retention_records
)

print("\nSCENE RETENTION SENSITIVITY")

display(

    radiometric_retention.loc[

        radiometric_retention[
            "minimum_coverage"
        ].eq(0.70)

    ].round(4)
)

# ============================================================
# 11. SENSOR-SPECIFIC RETENTION
# ============================================================

sensor_retention_records = []

primary_pair_qa = paired_radiometric_qa.loc[

    paired_radiometric_qa[
        "minimum_valid_fraction"
    ].ge(0.70)

    & paired_radiometric_qa[
        "primary_season"
    ]

].copy()

for sensor, group in (

    primary_pair_qa.groupby("sensor")
):

    for threshold in [

        1.00,
        0.25,
        0.10,
        0.05,
        0.01
    ]:

        retained = group.loc[

            group[
                "maximum_radiometric_flag"
            ].le(threshold)

            & ~group["any_missing"]
        ]

        sensor_retention_records.append({

            "sensor": sensor,

            "maximum_flag_fraction": threshold,

            "initial_scenes": len(group),

            "retained_scenes": len(retained),

            "retention_pct": (

                100

                * len(retained)

                / len(group)
            )
        })

sensor_radiometric_retention = pd.DataFrame(

    sensor_retention_records
)

print("\nSENSOR-SPECIFIC RETENTION")

display(

    sensor_radiometric_retention.round(4)
)

# ============================================================
# 12. RED REFLECTANCE AND NDVI DIAGNOSTIC
# ============================================================

# NDVI > 1 can result from negative red
# reflectance when NIR remains positive.

red_ndvi_audit = fragment_qa[[

    "sensor",
    "scene_id",
    "date",
    "unit_id",
    "forest_type",
    "Red_mean",
    "Red_median",
    "NIR_mean",
    "NIR_median",
    "NDVI_mean",
    "NDVI_median",
    "negative_reflectance_mean",
    "normalized_index_outside_range_mean"

]].copy()

red_ndvi_audit["negative_red_median"] = (

    red_ndvi_audit["Red_median"].lt(0)
)

red_ndvi_audit["ndvi_median_gt_one"] = (

    red_ndvi_audit["NDVI_median"].gt(1)
)

red_ndvi_summary = (

    red_ndvi_audit

    .groupby("sensor", as_index=False)

    .agg(

        n_observations=(
            "scene_id",
            "size"
        ),

        negative_red_medians=(
            "negative_red_median",
            "sum"
        ),

        ndvi_medians_gt_one=(
            "ndvi_median_gt_one",
            "sum"
        ),

        minimum_red_median=(
            "Red_median",
            "min"
        ),

        maximum_ndvi_median=(
            "NDVI_median",
            "max"
        )
    )
)

print("\nRED REFLECTANCE AND NDVI DIAGNOSTIC")

display(red_ndvi_summary.round(5))

# ============================================================
# 13. PIXEL COUNT DIAGNOSTIC
# ============================================================

# Pixel counts are not equivalent to
# independent observations or exact area.

pixel_count_audit = (

    radiometric_master

    .groupby([

        "unit_id",

        "spatial_scale",

        "sensor"

    ], as_index=False)

    .agg(

        minimum_count=(
            "NDVI_count",
            "min"
        ),

        median_count=(
            "NDVI_count",
            "median"
        ),

        maximum_count=(
            "NDVI_count",
            "max"
        ),

        median_valid_fraction=(
            "valid_fraction",
            "median"
        )
    )
)

print("\nPIXEL COUNT DIAGNOSTIC")

display(pixel_count_audit.round(4))

# ============================================================
# 14. RADIOMETRIC DIAGNOSTIC FIGURE
# ============================================================

sensor_order = [

    "Landsat_5",

    "Landsat_7",

    "Landsat_8",

    "Landsat_9"
]

sensor_colors = {

    "Landsat_5": "#527B98",

    "Landsat_7": "#DB9140",

    "Landsat_8": "#24634F",

    "Landsat_9": "#795C92"
}

fig, axes = plt.subplots(

    2,

    2,

    figsize=(15, 10)
)

# Panel A: negative reflectance

ax = axes[0, 0]

for sensor in sensor_order:

    group = fragment_qa.loc[

        fragment_qa["sensor"].eq(sensor)
    ]

    ax.scatter(

        group["date"],

        group[
            "negative_reflectance_mean"
        ] * 100,

        s=20,

        alpha=0.65,

        color=sensor_colors[sensor],

        label=sensor
    )

ax.axhline(

    5,

    color="gray",

    linestyle="--",

    linewidth=1
)

ax.set_ylabel(
    "Pixels with negative reflectance (%)"
)

ax.set_title(

    "(a) Negative surface reflectance",

    loc="left",

    fontweight="bold"
)

ax.legend(

    frameon=False,

    fontsize=8
)

# Panel B: normalized index anomalies

ax = axes[0, 1]

for sensor in sensor_order:

    group = fragment_qa.loc[

        fragment_qa["sensor"].eq(sensor)
    ]

    ax.scatter(

        group["date"],

        group[
            "normalized_index_outside_range_mean"
        ] * 100,

        s=20,

        alpha=0.65,

        color=sensor_colors[sensor]
    )

ax.axhline(

    5,

    color="gray",

    linestyle="--",

    linewidth=1
)

ax.set_ylabel(
    "Pixels with out-of-range indices (%)"
)

ax.set_title(

    "(b) Normalized-index anomalies",

    loc="left",

    fontweight="bold"
)

# Panel C: red reflectance vs NDVI

ax = axes[1, 0]

for sensor in sensor_order:

    group = red_ndvi_audit.loc[

        red_ndvi_audit["sensor"].eq(sensor)
    ]

    ax.scatter(

        group["Red_median"],

        group["NDVI_median"],

        s=23,

        alpha=0.65,

        color=sensor_colors[sensor]
    )

ax.axvline(

    0,

    color="gray",

    linestyle="--"
)

ax.axhline(

    1,

    color="#B64B40",

    linestyle="--"
)

ax.set_xlabel(
    "Median red surface reflectance"
)

ax.set_ylabel(
    "Median NDVI"
)

ax.set_title(

    "(c) Red reflectance and NDVI",

    loc="left",

    fontweight="bold"
)

# Panel D: retention by sensor

ax = axes[1, 1]

for sensor in sensor_order:

    group = (

        sensor_radiometric_retention.loc[

            sensor_radiometric_retention[
                "sensor"
            ].eq(sensor)

        ]

        .sort_values(
            "maximum_flag_fraction"
        )
    )

    ax.plot(

        group[
            "maximum_flag_fraction"
        ] * 100,

        group["retention_pct"],

        marker="o",

        linewidth=1.8,

        color=sensor_colors[sensor],

        label=sensor
    )

ax.set_xscale("log")

ax.set_xticks([1, 5, 10, 25, 100])

ax.set_xticklabels([

    "1",
    "5",
    "10",
    "25",
    "100"
])

ax.set_ylim(0, 105)

ax.set_xlabel(

    "Maximum flagged fraction allowed (%)"
)

ax.set_ylabel(
    "Retained paired scenes (%)"
)

ax.set_title(

    "(d) Scene-retention sensitivity",

    loc="left",

    fontweight="bold"
)

for ax in axes.flat:

    ax.grid(alpha=0.2)

    ax.spines["top"].set_visible(False)

    ax.spines["right"].set_visible(False)

fig.suptitle(

    "Landsat radiometric quality diagnostics",

    fontsize=16,

    fontweight="bold"
)

fig.tight_layout(

    rect=[0, 0, 1, 0.96]
)

for extension in [

    "png",

    "pdf",

    "svg"
]:

    fig.savefig(

        FIGURE_DIR

        / f"FIG_LANDSAT_RADIOMETRIC_QA.{extension}",

        dpi=600,

        bbox_inches="tight",

        facecolor="white"
    )

plt.show()

# ============================================================
# 15. EXPORT ALL DIAGNOSTIC TABLES
# ============================================================

radiometric_tables = {

    "RADIOMETRIC_MASTER":

        radiometric_master,

    "MISSING_OBSERVATIONS":

        missing_spectral_rows,

    "INDEX_RANGE_AUDIT":

        index_range_audit,

    "REFLECTANCE_BAND_AUDIT":

        band_radiometric_audit,

    "FLAG_THRESHOLD_AUDIT":

        radiometric_threshold_audit,

    "SEVERITY_SUMMARY":

        severity_summary,

    "CRITICAL_OBSERVATIONS":

        critical_radiometric_rows,

    "PAIRED_RADIOMETRIC_QA":

        paired_radiometric_qa,

    "SCENE_RETENTION":

        radiometric_retention,

    "SENSOR_RETENTION":

        sensor_radiometric_retention,

    "RED_NDVI_AUDIT":

        red_ndvi_audit,

    "RED_NDVI_SUMMARY":

        red_ndvi_summary,

    "PIXEL_COUNT_AUDIT":

        pixel_count_audit
}

for name, dataframe in (

    radiometric_tables.items()
):

    dataframe.to_csv(

        RADIOMETRIC_DIR

        / f"LANDSAT_{name}.csv",

        index=False,

        encoding="utf-8-sig"
    )

radiometric_workbook = (

    TABLE_DIR

    / "SATIPO_LANDSAT_RADIOMETRIC_QA.xlsx"
)

with pd.ExcelWriter(

    radiometric_workbook,

    engine="openpyxl"

) as writer:

    for name, dataframe in (

        radiometric_tables.items()
    ):

        dataframe.to_excel(

            writer,

            sheet_name=name[:31],

            index=False
        )

# ============================================================
# 16. SAVE MANIFEST
# ============================================================

radiometric_manifest = {

    "project": "SATIPO_FOREST_PROJECT",

    "cell": "90A",

    "source_file": str(source_path),

    "scene_unit_observations":

        len(radiometric_master),

    "unique_scenes":

        int(

            radiometric_master[
                "scene_id"
            ].nunique()
        ),

    "missing_observations":

        int(len(missing_spectral_rows)),

    "critical_observations":

        int(len(critical_radiometric_rows)),

    "index_range_flags":

        int(

            radiometric_master[
                "any_index_outside_range"
            ].sum()
        ),

    "diagnostic_thresholds":

        FLAG_THRESHOLDS,

    "native_data_modified": False,

    "scenes_excluded": False,

    "cross_sensor_harmonization": False,

    "next_step":

        "90B — Pixel-level radiometric screening",

    "notes": [

        "Thresholds are diagnostic and provisional.",

        "Negative reflectance is not automatically "

        "equivalent to cloud contamination.",

        "Index-specific physical screening must be "

        "performed before temporal aggregation.",

        "Scene-level flag fractions describe "

        "different potentially overlapping conditions.",

        "The maximum flag fraction is not the union "

        "of all anomalous pixels.",

        "Original Cell 89 outputs remain unchanged."
    ]
}

with open(

    RADIOMETRIC_DIR

    / "LANDSAT_RADIOMETRIC_MANIFEST.json",

    "w",

    encoding="utf-8"

) as file:

    json.dump(

        radiometric_manifest,

        file,

        indent=4
    )

# ============================================================
# 17. FINAL REPORT
# ============================================================

print("\n" + "=" * 75)

print("SATIPO — RADIOMETRIC AUDIT REPORT")

print("=" * 75)

print(

    "Native spectral observations:",

    len(radiometric_master)
)

print(

    "Missing spectral observations:",

    len(missing_spectral_rows)
)

print(

    "Rows with out-of-range normalized indices:",

    radiometric_master[
        "any_index_outside_range"
    ].sum()
)

print(

    "Problematic rows requiring review:",

    len(critical_radiometric_rows)
)

print(

    "Paired scenes:",

    len(paired_radiometric_qa)
)

print(

    "Primary-season paired scenes:",

    len(primary_pair_qa)
)

print(

    "Original spectral master:",

    "PRESERVED"
)

print(

    "Diagnostic tables exported:",

    len(radiometric_tables)
)

print("\nWorkbook:")

print(radiometric_workbook)

print("\n" + "=" * 75)

print("RADIOMETRIC QUALITY AUDIT COMPLETED")

print("NEXT: CELL 90B — PIXEL-LEVEL RADIOMETRIC SCREENING")

print("=" * 75)


In [ ]:

# ============================================================
# CELL 90B-FIX — RECOVER VALIDATED SPATIAL GEOMETRIES
# SATIPO FOREST PROJECT
# ============================================================

import ee
import pandas as pd
import geopandas as gpd

from pathlib import Path
from shapely.geometry import shape

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

REMOTE_DIR = ROOT / "04_REMOTE_SENSING"

EXPECTED_IDS = {
    "FRAG_01",
    "FRAG_02",
    "P01",
    "P02",
    "FRAG_01_CORE30",
    "FRAG_02_CORE30"
}

print("=" * 70)
print("SATIPO — SPATIAL GEOMETRY RECOVERY")
print("=" * 70)

# ============================================================
# 1. INVENTORY OF EXISTING SPATIAL OBJECTS
# ============================================================

spatial_objects = []

# Snapshot avoids changing globals during iteration.

global_snapshot = list(globals().items())

for name, obj in global_snapshot:

    if name.startswith("_"):
        continue

    if isinstance(obj, gpd.GeoDataFrame):

        spatial_objects.append({
            "variable": name,
            "type": "GeoDataFrame",
            "rows": len(obj),
            "columns": ", ".join(
                map(str, obj.columns)
            ),
            "CRS": str(obj.crs)
        })

    elif isinstance(obj, ee.FeatureCollection):

        spatial_objects.append({
            "variable": name,
            "type": "EE FeatureCollection",
            "rows": None,
            "columns": None,
            "CRS": None
        })

    elif isinstance(obj, ee.Feature):

        spatial_objects.append({
            "variable": name,
            "type": "EE Feature",
            "rows": 1,
            "columns": None,
            "CRS": None
        })

    elif isinstance(obj, ee.Geometry):

        spatial_objects.append({
            "variable": name,
            "type": "EE Geometry",
            "rows": 1,
            "columns": None,
            "CRS": None
        })

spatial_inventory = pd.DataFrame(
    spatial_objects
)

print("\nSPATIAL OBJECTS IN MEMORY")

display(spatial_inventory)

# ============================================================
# 2. SEARCH EARTH ENGINE FEATURE COLLECTIONS
# ============================================================

recovered_units = None

recovery_source = None

ee_diagnostics = []

for name, obj in global_snapshot:

    if not isinstance(
        obj,
        ee.FeatureCollection
    ):
        continue

    try:

        # Inspect only a few features.
        # Avoid downloading large collections.

        sample = obj.limit(7).getInfo()

        features = sample.get(
            "features",
            []
        )

        if not features:
            continue

        properties = [
            feature.get("properties", {})
            for feature in features
        ]

        property_names = sorted(
            set().union(
                *[set(p.keys()) for p in properties]
            )
        )

        ee_diagnostics.append({
            "variable": name,
            "sample_features": len(features),
            "properties": ", ".join(
                property_names
            )
        })

        # Find the property containing our six IDs.
        # It does not need to be named unit_id.

        for property_name in property_names:

            observed_ids = {
                str(p.get(property_name))
                for p in properties
            }

            if not EXPECTED_IDS.issubset(
                observed_ids
            ):
                continue

            selected_fc = obj.filter(
                ee.Filter.inList(
                    property_name,
                    sorted(EXPECTED_IDS)
                )
            )

            selected = selected_fc.getInfo()[
                "features"
            ]

            if len(selected) != 6:
                continue

            records = []

            for feature in selected:

                props = dict(
                    feature["properties"]
                )

                props["unit_id"] = str(
                    props[property_name]
                )

                props["geometry"] = shape(
                    feature["geometry"]
                )

                records.append(props)

            recovered_units = gpd.GeoDataFrame(
                records,
                geometry="geometry",
                crs="EPSG:4326"
            )

            recovery_source = (
                f"Earth Engine: {name}"
            )

            break

        if recovered_units is not None:
            break

    except Exception as error:

        ee_diagnostics.append({
            "variable": name,
            "sample_features": None,
            "properties": str(error)[:200]
        })

print("\nEARTH ENGINE COLLECTION DIAGNOSTICS")

display(
    pd.DataFrame(ee_diagnostics)
)

# ============================================================
# 3. SEARCH GEODATAFRAMES WITH ALTERNATIVE ID COLUMNS
# ============================================================

if recovered_units is None:

    for name, obj in global_snapshot:

        if not isinstance(
            obj,
            gpd.GeoDataFrame
        ):
            continue

        for column in obj.columns:

            if column == obj.geometry.name:
                continue

            try:

                observed_ids = set(
                    obj[column]
                    .dropna()
                    .astype(str)
                )

            except Exception:
                continue

            if EXPECTED_IDS.issubset(
                observed_ids
            ):

                candidate = obj.loc[
                    obj[column]
                    .astype(str)
                    .isin(EXPECTED_IDS)
                ].copy()

                if len(candidate) != 6:
                    continue

                candidate["unit_id"] = (
                    candidate[column]
                    .astype(str)
                )

                recovered_units = candidate

                recovery_source = (
                    f"GeoDataFrame: {name}"
                    f" | ID column: {column}"
                )

                break

        if recovered_units is not None:
            break

# ============================================================
# 4. SEARCH ALL REMOTE SENSING VECTOR FILES
# ============================================================

vector_inventory = []

vector_extensions = {
    ".gpkg",
    ".geojson",
    ".shp",
    ".kml",
    ".kmz",
    ".json",
    ".parquet"
}

vector_paths = sorted([
    path
    for path in REMOTE_DIR.rglob("*")
    if path.is_file()
    and path.suffix.lower()
    in vector_extensions
])

for path in vector_paths:

    vector_inventory.append({
        "filename": path.name,
        "extension": path.suffix,
        "relative_path": str(
            path.relative_to(ROOT)
        ),
        "size_KB": round(
            path.stat().st_size / 1024,
            2
        )
    })

print("\nVECTOR FILES IN GOOGLE DRIVE")

display(
    pd.DataFrame(vector_inventory)
)

# ============================================================
# 5. RECOVER FROM VECTOR FILES IF NECESSARY
# ============================================================

if recovered_units is None:

    for path in vector_paths:

        try:

            if path.suffix.lower() == ".gpkg":

                layers = gpd.list_layers(
                    path
                )["name"].tolist()

            else:

                layers = [None]

            for layer in layers:

                try:

                    if path.suffix.lower() == ".parquet":

                        candidate = (
                            gpd.read_parquet(path)
                        )

                    elif layer is not None:

                        candidate = gpd.read_file(
                            path,
                            layer=layer
                        )

                    else:

                        candidate = gpd.read_file(
                            path
                        )

                except Exception:
                    continue

                if candidate.crs is None:
                    continue

                for column in candidate.columns:

                    if column == candidate.geometry.name:
                        continue

                    try:

                        observed_ids = set(
                            candidate[column]
                            .dropna()
                            .astype(str)
                        )

                    except Exception:
                        continue

                    if not EXPECTED_IDS.issubset(
                        observed_ids
                    ):
                        continue

                    selected = candidate.loc[
                        candidate[column]
                        .astype(str)
                        .isin(EXPECTED_IDS)
                    ].copy()

                    if len(selected) != 6:
                        continue

                    selected["unit_id"] = (
                        selected[column]
                        .astype(str)
                    )

                    recovered_units = selected

                    recovery_source = (
                        f"File: {path}"
                        f" | Layer: {layer}"
                    )

                    break

                if recovered_units is not None:
                    break

        except Exception:
            continue

        if recovered_units is not None:
            break

# ============================================================
# 6. VALIDATE RECOVERED GEOMETRIES
# ============================================================

if recovered_units is not None:

    assert len(recovered_units) == 6

    assert recovered_units[
        "unit_id"
    ].is_unique

    assert set(
        recovered_units["unit_id"]
    ) == EXPECTED_IDS

    assert recovered_units.crs is not None

    assert (
        recovered_units.geometry.is_valid.all()
    )

    assert not (
        recovered_units.geometry.is_empty.any()
    )

    assert recovered_units.geometry.geom_type.isin([
        "Polygon",
        "MultiPolygon"
    ]).all()

    # Recover metadata from the original spectral master.

    spectral_path = (
        REMOTE_DIR
        / "03_LANDSAT"
        / "02_SPECTRAL_PREPROCESSING"
        / "LANDSAT_SPECTRAL_MASTER.csv"
    )

    spectral_metadata = pd.read_csv(
        spectral_path,
        usecols=[
            "unit_id",
            "forest_type",
            "spatial_scale",
            "buffer_m"
        ]
    ).drop_duplicates()

    assert len(spectral_metadata) == 6

    original_crs = recovered_units.crs

    screening_units = (
        recovered_units[
            ["unit_id", "geometry"]
        ]
        .merge(
            spectral_metadata,
            on="unit_id",
            validate="one_to_one"
        )
    )

    screening_units = gpd.GeoDataFrame(
        screening_units,
        geometry="geometry",
        crs=original_crs
    )

    screening_units["area_ha"] = (
        screening_units
        .to_crs("EPSG:32718")
        .geometry.area
        / 10000
    )

    # ========================================================
    # 7. EXPORT RECOVERED SIX-UNIT GEOMETRY MASTER
    # ========================================================

    recovery_dir = (
        REMOTE_DIR
        / "02_GEOMETRY_VALIDATION"
    )

    recovery_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    recovery_path = (
        recovery_dir
        / "SATIPO_SIX_SPATIAL_UNITS.gpkg"
    )

    screening_units.to_file(
        recovery_path,
        layer="spatial_units",
        driver="GPKG"
    )

    print("\n" + "=" * 70)

    print("GEOMETRIES RECOVERED SUCCESSFULLY")

    print("=" * 70)

    print("Source:", recovery_source)

    print("CRS:", screening_units.crs)

    print("\nVALIDATED SPATIAL UNITS")

    display(
        screening_units.drop(
            columns="geometry"
        ).round(5)
    )

    print("\nSaved geometry master:")

    print(recovery_path)

    print("\nVariable ready: screening_units")

    print("\nNEXT: RERUN CELL 90B")

else:

    print("\n" + "=" * 70)

    print("AUTOMATIC RECOVERY NOT COMPLETED")

    print("=" * 70)

    print(
        "The original geometries may be stored "
        "as separate polygons or collections."
    )

    print(
        "Review the spatial object inventory "
        "and vector file inventory above."
    )

    print(
        "No geometries were invented or modified."
    )

    print(
        "Send the diagnostic tables to identify "
        "the exact source."
    )


In [ ]:

# ============================================================
# CELL 90B-FIX2 — GRID AREA DIAGNOSTICS
# ============================================================

import sys
import pandas as pd
import traceback

print("=" * 70)
print("SATIPO — GRID AREA DIAGNOSTICS")
print("=" * 70)

# 1. Recover the exact traceback

tb = getattr(sys, "last_traceback", None)

if tb is not None:

    print("\nLAST ERROR LOCATION\n")

    traceback.print_tb(tb)

    print("\nDATAFRAMES IN ERROR CONTEXT")

    while tb is not None:

        frame = tb.tb_frame

        filename = frame.f_code.co_filename

        if "ipykernel" in filename:

            print("\nFile:", filename)
            print("Line:", tb.tb_lineno)

            for name, obj in list(
                frame.f_locals.items()
            ):

                if not isinstance(
                    obj, pd.DataFrame
                ):
                    continue

                columns = [
                    str(c) for c in obj.columns
                ]

                relevant = any(
                    any(
                        term in c.lower()
                        for term in [
                            "grid",
                            "area",
                            "unit_id",
                            "pixel"
                        ]
                    )
                    for c in columns
                )

                if relevant:

                    print("\nVariable:", name)
                    print("Shape:", obj.shape)
                    print("Columns:", columns)

                    display(obj.head(6))

        tb = tb.tb_next

else:

    print("Previous traceback unavailable.")

# 2. Identify spatial and grid-area tables

print("\n" + "=" * 70)
print("GRID-RELATED DATAFRAMES IN MEMORY")
print("=" * 70)

grid_diagnostics = []

for name, obj in list(globals().items()):

    if name.startswith("_"):
        continue

    if not isinstance(obj, pd.DataFrame):
        continue

    area_columns = [
        str(c)
        for c in obj.columns
        if any(
            term in str(c).lower()
            for term in [
                "area",
                "grid",
                "pixel",
                "sum"
            ]
        )
    ]

    if area_columns:

        grid_diagnostics.append({
            "variable": name,
            "rows": len(obj),
            "columns": ", ".join(
                area_columns
            ),
            "has_grid_area_m2": (
                "grid_area_m2" in obj.columns
            )
        })

grid_diagnostics_df = pd.DataFrame(
    grid_diagnostics
)

display(grid_diagnostics_df)

# 3. Inspect small spatial tables

print("\nSMALL SPATIAL TABLES")

for name, obj in list(globals().items()):

    if not isinstance(obj, pd.DataFrame):
        continue

    if not 1 <= len(obj) <= 16:
        continue

    if "unit_id" not in obj.columns:
        continue

    print("\nVariable:", name)
    print("Columns:", obj.columns.tolist())

    display(obj.head(6))

print("\n" + "=" * 70)
print("DIAGNOSTICS COMPLETED")
print("=" * 70)


In [ ]:

# ============================================================
# CELL 90B — PIXEL-LEVEL RADIOMETRIC SCREENING
# SATIPO FOREST PROJECT
#
# Landsat 5 / 7 / 8 / 9
# Period: 1985–2025
#
# Corrected version:
#   - Explicit spatial-unit recovery
#   - Explicit grid-area reducer output
#   - Fixed UTM analysis grid
#   - Pixel-level radiometric screening
#   - Recalculation of spectral indices
#   - Annual checkpoints
#   - Paired coverage recalculation
#   - Preservation of original spectral master
# ============================================================

!pip -q install earthengine-api geopandas openpyxl tqdm

import ee
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

import json
import time
import hashlib

from pathlib import Path
from tqdm.auto import tqdm
from google.colab import drive
from IPython.display import display

# ============================================================
# 1. INITIALIZATION
# ============================================================

drive.mount("/content/drive")

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

REMOTE = ROOT / "04_REMOTE_SENSING"

CLOUD_PROJECT = "brunei159"

try:
    ee.Initialize(project=CLOUD_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=CLOUD_PROJECT)

assert ee.Number(1).getInfo() == 1

print("=" * 75)
print("SATIPO — PIXEL-LEVEL RADIOMETRIC SCREENING")
print("=" * 75)
print("Earth Engine connection: OK")

# ============================================================
# 2. OUTPUT DIRECTORIES
# ============================================================

SCREEN_DIR = (
    REMOTE
    / "03_LANDSAT"
    / "03_RADIOMETRIC_SCREENING"
)

SCREEN_CACHE = SCREEN_DIR / "ANNUAL_CACHE_V03"

SCREEN_TABLES = REMOTE / "08_OUTPUTS" / "TABLES"

SCREEN_FIGURES = REMOTE / "08_OUTPUTS" / "FIGURES"

for folder in [
    SCREEN_DIR,
    SCREEN_CACHE,
    SCREEN_TABLES,
    SCREEN_FIGURES
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

# ============================================================
# 3. ANALYSIS CONFIGURATION
# ============================================================

SCREEN_VERSION = "V03_FIXED_UTM_GRID"

START_YEAR = 1985
END_YEAR = 2025

PRIMARY_MONTHS = [5, 6, 7, 8, 9]

MIN_PAIRED_COVERAGE = 0.70

MIN_LOCAL_COVERAGE = 0.70

REFLECTANCE_MIN = 0.0
REFLECTANCE_MAX = 1.0

DENOMINATOR_EPSILON = 1e-6

# Broad EVI diagnostic bounds.
# These are screening choices, not universal physical bounds.

EVI_MIN = -1.0
EVI_MAX = 1.5

# Fixed 30-m grid in UTM zone 18S.
# All area and spectral reducers use this same grid.

ANALYSIS_CRS = "EPSG:32718"

ANALYSIS_TRANSFORM = [
    30, 0, 0,
    0, -30, 10000000
]

EXPECTED_UNITS = [
    "FRAG_01",
    "FRAG_02",
    "P01",
    "P02",
    "FRAG_01_CORE30",
    "FRAG_02_CORE30"
]

EXPECTED_FOREST = {
    "FRAG_01": "Late_secondary",
    "FRAG_02": "Mature",
    "P01": "Late_secondary",
    "P02": "Mature",
    "FRAG_01_CORE30": "Late_secondary",
    "FRAG_02_CORE30": "Mature"
}

SPECTRAL_BANDS = [
    "Blue",
    "Green",
    "Red",
    "NIR",
    "SWIR1",
    "SWIR2"
]

SPECTRAL_INDICES = [
    "NDVI",
    "EVI",
    "NDMI",
    "NBR",
    "NBR2"
]

ALL_SPECTRAL = (
    SPECTRAL_BANDS
    + SPECTRAL_INDICES
)

LANDSAT_COLLECTIONS = {
    "Landsat_5": "LANDSAT/LT05/C02/T1_L2",
    "Landsat_7": "LANDSAT/LE07/C02/T1_L2",
    "Landsat_8": "LANDSAT/LC08/C02/T1_L2",
    "Landsat_9": "LANDSAT/LC09/C02/T1_L2"
}

SCREEN_CONFIG = {
    "version": SCREEN_VERSION,
    "start_year": START_YEAR,
    "end_year": END_YEAR,
    "primary_months": PRIMARY_MONTHS,
    "minimum_paired_coverage": MIN_PAIRED_COVERAGE,
    "minimum_local_coverage": MIN_LOCAL_COVERAGE,
    "reflectance_min": REFLECTANCE_MIN,
    "reflectance_max": REFLECTANCE_MAX,
    "denominator_epsilon": DENOMINATOR_EPSILON,
    "evi_min": EVI_MIN,
    "evi_max": EVI_MAX,
    "analysis_crs": ANALYSIS_CRS,
    "analysis_transform": ANALYSIS_TRANSFORM,
    "qa_pixel_excluded_bits": [0, 1, 2, 3, 4, 5],
    "qa_radsat_rule": "QA_RADSAT == 0",
    "common_spectral_mask": True,
    "cross_sensor_harmonization": False
}

# ============================================================
# 4. LOAD ORIGINAL SPECTRAL MASTER
# ============================================================

NATIVE_MASTER_PATH = (
    REMOTE
    / "03_LANDSAT"
    / "02_SPECTRAL_PREPROCESSING"
    / "LANDSAT_SPECTRAL_MASTER.csv"
)

assert NATIVE_MASTER_PATH.exists(), (
    f"Original spectral master not found:\n"
    f"{NATIVE_MASTER_PATH}"
)

native_spectral = pd.read_csv(
    NATIVE_MASTER_PATH
)

required_native_columns = [
    "scene_id",
    "sensor",
    "date",
    "year",
    "month",
    "unit_id",
    "forest_type",
    "spatial_scale"
]

assert set(
    required_native_columns
).issubset(native_spectral.columns)

native_spectral["date"] = pd.to_datetime(
    native_spectral["date"]
).dt.strftime("%Y-%m-%d")

native_spectral["year"] = (
    native_spectral["year"].astype(int)
)

native_spectral["month"] = (
    native_spectral["month"].astype(int)
)

assert not native_spectral.duplicated(
    ["sensor", "scene_id", "unit_id"]
).any()

assert len(native_spectral) == 2328, (
    f"Unexpected original spectral master size: "
    f"{len(native_spectral)}"
)

selected_scenes = (
    native_spectral[
        [
            "sensor",
            "scene_id",
            "date",
            "year",
            "month"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["year", "date", "sensor"]
    )
    .reset_index(drop=True)
)

assert len(selected_scenes) == 388

print("\nSelected Landsat scenes:", len(selected_scenes))

display(
    selected_scenes
    .groupby("sensor")
    .size()
    .rename("n_scenes")
)

# ============================================================
# 5. RECOVER THE SIX VALIDATED SPATIAL UNITS
# ============================================================

# Use the already recovered GeoDataFrame whenever available.
# Otherwise reconstruct it from the validated fragment,
# permanent-plot and interior-buffer layers.

if (
    "recovered_units" in globals()
    and isinstance(
        recovered_units,
        gpd.GeoDataFrame
    )
):

    screen_units = recovered_units.copy()

    geometry_source = "memory:recovered_units"

else:

    required_objects = [
        "satipo_fragments",
        "satipo_plots",
        "interior_buffers"
    ]

    missing_objects = [
        name
        for name in required_objects
        if name not in globals()
    ]

    assert not missing_objects, (
        "Required spatial layers are unavailable: "
        f"{missing_objects}. "
        "Run the spatial recovery cell first."
    )

    # Fragments

    frag_units = satipo_fragments[
        [
            "fragment_id",
            "forest_type",
            "geometry"
        ]
    ].copy()

    frag_units = frag_units.rename(
        columns={
            "fragment_id": "unit_id"
        }
    )

    frag_units["spatial_scale"] = "Fragment"
    frag_units["buffer_m"] = 0

    # Permanent plots

    plot_units = satipo_plots[
        [
            "plot_id",
            "forest_type",
            "geometry"
        ]
    ].copy()

    plot_units = plot_units.rename(
        columns={
            "plot_id": "unit_id"
        }
    )

    plot_units["spatial_scale"] = "Plot"
    plot_units["buffer_m"] = 0

    # Fragment interior buffers

    core_units = interior_buffers.loc[
        (
            interior_buffers["spatial_scale"]
            == "Fragment"
        )
        &
        (
            interior_buffers["buffer_m"] == 30
        ),
        [
            "unit_id",
            "forest_type",
            "spatial_scale",
            "buffer_m",
            "geometry"
        ]
    ].copy()

    core_units["unit_id"] = (
        core_units["unit_id"]
        .astype(str)
        .str.replace(
            "_CORE30",
            "",
            regex=False
        )
        + "_CORE30"
    )

    core_units["spatial_scale"] = (
        "Fragment_core"
    )

    assert frag_units.crs is not None
    assert plot_units.crs is not None
    assert core_units.crs is not None

    frag_units = frag_units.to_crs(
        ANALYSIS_CRS
    )

    plot_units = plot_units.to_crs(
        ANALYSIS_CRS
    )

    core_units = core_units.to_crs(
        ANALYSIS_CRS
    )

    screen_units = gpd.GeoDataFrame(
        pd.concat(
            [
                frag_units,
                plot_units,
                core_units
            ],
            ignore_index=True
        ),
        geometry="geometry",
        crs=ANALYSIS_CRS
    )

    geometry_source = (
        "reconstructed_validated_layers"
    )

# ============================================================
# 6. VALIDATE GEOMETRIES
# ============================================================

assert screen_units.crs is not None

screen_units = screen_units.to_crs(
    ANALYSIS_CRS
)

required_spatial_columns = [
    "unit_id",
    "forest_type",
    "spatial_scale",
    "buffer_m",
    "geometry"
]

assert set(
    required_spatial_columns
).issubset(screen_units.columns)

screen_units = screen_units[
    required_spatial_columns
].copy()

screen_units["unit_id"] = (
    screen_units["unit_id"].astype(str)
)

screen_units = screen_units.loc[
    screen_units["unit_id"].isin(
        EXPECTED_UNITS
    )
].copy()

assert len(screen_units) == 6

assert screen_units["unit_id"].is_unique

assert set(
    screen_units["unit_id"]
) == set(EXPECTED_UNITS)

assert screen_units.geometry.notna().all()

assert (
    ~screen_units.geometry.is_empty
).all()

assert screen_units.geometry.is_valid.all()

assert screen_units.geometry.geom_type.isin(
    ["Polygon", "MultiPolygon"]
).all()

for _, row in screen_units.iterrows():

    assert row["forest_type"] == (
        EXPECTED_FOREST[row["unit_id"]]
    )

screen_units["geometry_area_m2"] = (
    screen_units.geometry.area
)

assert (
    screen_units["geometry_area_m2"] > 0
).all()

screen_units = (
    screen_units
    .set_index("unit_id")
    .loc[EXPECTED_UNITS]
    .reset_index()
)

print("\nVALIDATED SPATIAL UNITS")
print("Geometry source:", geometry_source)

display(
    screen_units.drop(
        columns="geometry"
    ).round(4)
)

# Save the exact geometry used by this cell.

screen_units.to_file(
    SCREEN_DIR / "SCREENING_SPATIAL_UNITS.gpkg",
    layer="analysis_units",
    driver="GPKG"
)

# ============================================================
# 7. CONVERT SPATIAL UNITS TO EARTH ENGINE
# ============================================================

screen_units_wgs84 = screen_units.to_crs(
    "EPSG:4326"
)

screen_features = []

for _, row in screen_units_wgs84.iterrows():

    geometry_json = (
        row.geometry.__geo_interface__
    )

    ee_geometry = ee.Geometry(
        geometry_json,
        proj="EPSG:4326",
        geodesic=False
    )

    properties = {
        "unit_id": str(row["unit_id"]),
        "forest_type": str(row["forest_type"]),
        "spatial_scale": str(row["spatial_scale"]),
        "buffer_m": int(row["buffer_m"]),
        "geometry_area_m2": float(
            row["geometry_area_m2"]
        )
    }

    screen_features.append(
        ee.Feature(
            ee_geometry,
            properties
        )
    )

screen_fc = ee.FeatureCollection(
    screen_features
)

assert screen_fc.size().getInfo() == 6

# ============================================================
# 8. EARTH ENGINE GRID AREA
# ============================================================

# Explicit reducer output name.
# This directly addresses the previous KeyError.

grid_area_image = (
    ee.Image.pixelArea()
    .rename("grid_area_m2")
)

grid_area_reducer = (
    ee.Reducer.sum()
    .setOutputs(["grid_area_m2"])
)

grid_area_fc = (
    grid_area_image.reduceRegions(
        collection=screen_fc,
        reducer=grid_area_reducer,
        crs=ANALYSIS_CRS,
        crsTransform=ANALYSIS_TRANSFORM,
        tileScale=4
    )
)

grid_area_result = grid_area_fc.getInfo()

grid_reference = pd.DataFrame(
    [
        feature["properties"]
        for feature in grid_area_result[
            "features"
        ]
    ]
)

# Defensive correction in case Earth Engine returns
# the default reducer output name instead.

if "grid_area_m2" not in grid_reference.columns:

    if "sum" in grid_reference.columns:

        grid_reference = (
            grid_reference.rename(
                columns={
                    "sum": "grid_area_m2"
                }
            )
        )

    else:

        raise KeyError(
            "Grid area column unavailable. "
            f"Returned columns: "
            f"{grid_reference.columns.tolist()}"
        )

grid_reference["grid_area_m2"] = (
    pd.to_numeric(
        grid_reference["grid_area_m2"],
        errors="raise"
    )
)

grid_reference["geometry_area_m2"] = (
    pd.to_numeric(
        grid_reference["geometry_area_m2"],
        errors="raise"
    )
)

assert len(grid_reference) == 6

assert grid_reference["unit_id"].is_unique

assert set(
    grid_reference["unit_id"]
) == set(EXPECTED_UNITS)

assert grid_reference[
    "grid_area_m2"
].notna().all()

assert (
    grid_reference["grid_area_m2"] > 0
).all()

grid_reference["grid_area_difference_pct"] = (
    (
        grid_reference["grid_area_m2"]
        -
        grid_reference["geometry_area_m2"]
    )
    /
    grid_reference["geometry_area_m2"]
    * 100
)

# Allow small differences caused by projection and
# raster boundary integration, but reject major errors.

assert (
    grid_reference[
        "grid_area_difference_pct"
    ].abs() < 2.0
).all(), (
    "Grid-to-geometry area discrepancy exceeds 2%."
)

grid_reference = (
    grid_reference
    .set_index("unit_id")
    .loc[EXPECTED_UNITS]
    .reset_index()
)

print("\nGRID AREA VALIDATION — PASSED")

display(
    grid_reference[
        [
            "unit_id",
            "forest_type",
            "geometry_area_m2",
            "grid_area_m2",
            "grid_area_difference_pct"
        ]
    ].round(4)
)

grid_reference.to_csv(
    SCREEN_DIR / "LANDSAT_GRID_REFERENCE.csv",
    index=False
)

# Attach reference areas to the EE features.

grid_area_dictionary = ee.Dictionary(
    dict(
        zip(
            grid_reference["unit_id"],
            grid_reference["grid_area_m2"]
        )
    )
)

def attach_grid_area(feature):

    feature = ee.Feature(feature)

    return feature.set(
        "grid_area_m2",
        grid_area_dictionary.get(
            feature.get("unit_id")
        )
    )

screen_fc = screen_fc.map(
    attach_grid_area
)

# ============================================================
# 9. CACHE CONFIGURATION AND GEOMETRY FINGERPRINT
# ============================================================

geometry_fingerprint = hashlib.sha256()

for _, row in screen_units.iterrows():

    geometry_fingerprint.update(
        row["unit_id"].encode("utf-8")
    )

    geometry_fingerprint.update(
        row.geometry.wkb
    )

scene_fingerprint = hashlib.sha256(
    selected_scenes[
        ["sensor", "scene_id"]
    ].to_csv(
        index=False
    ).encode("utf-8")
).hexdigest()

SCREEN_CONFIG["geometry_sha256"] = (
    geometry_fingerprint.hexdigest()
)

SCREEN_CONFIG["scene_sha256"] = (
    scene_fingerprint
)

config_hash = hashlib.sha256(
    json.dumps(
        SCREEN_CONFIG,
        sort_keys=True
    ).encode("utf-8")
).hexdigest()

SCREEN_CONFIG["configuration_sha256"] = (
    config_hash
)

with open(
    SCREEN_DIR / "SCREENING_CONFIG.json",
    "w"
) as file:

    json.dump(
        SCREEN_CONFIG,
        file,
        indent=2
    )

print("\nScreening configuration:", SCREEN_VERSION)
print("Configuration hash:", config_hash[:16])

# ============================================================
# 10. LANDSAT QA MASK
# ============================================================

def landsat_base_mask(image):

    image = ee.Image(image)

    qa = image.select("QA_PIXEL")

    # Collection 2 QA_PIXEL:
    # bit 0: fill
    # bit 1: dilated cloud
    # bit 2: cirrus (used by OLI)
    # bit 3: cloud
    # bit 4: cloud shadow
    # bit 5: snow

    qa_mask = qa.bitwiseAnd(
        63
    ).eq(0)

    saturation_mask = (
        image.select("QA_RADSAT")
        .eq(0)
    )

    return qa_mask.And(
        saturation_mask
    )

# ============================================================
# 11. SURFACE REFLECTANCE SCALING
# ============================================================

def get_scaled_reflectance(
    image,
    sensor
):

    image = ee.Image(image)

    if sensor in [
        "Landsat_5",
        "Landsat_7"
    ]:

        input_bands = [
            "SR_B1",
            "SR_B2",
            "SR_B3",
            "SR_B4",
            "SR_B5",
            "SR_B7"
        ]

    elif sensor in [
        "Landsat_8",
        "Landsat_9"
    ]:

        input_bands = [
            "SR_B2",
            "SR_B3",
            "SR_B4",
            "SR_B5",
            "SR_B6",
            "SR_B7"
        ]

    else:

        raise ValueError(
            f"Unknown Landsat sensor: {sensor}"
        )

    reflectance = (
        image.select(
            input_bands,
            SPECTRAL_BANDS
        )
        .multiply(0.0000275)
        .add(-0.2)
        .toFloat()
    )

    return reflectance

# ============================================================
# 12. SAFE SPECTRAL INDEX CALCULATION
# ============================================================

def safe_normalized_difference(
    band_a,
    band_b,
    output_name
):

    denominator = band_a.add(
        band_b
    )

    valid_denominator = (
        denominator.abs()
        .gt(DENOMINATOR_EPSILON)
    )

    index = (
        band_a.subtract(band_b)
        .divide(denominator)
        .updateMask(valid_denominator)
        .rename(output_name)
    )

    return index


def calculate_spectral_indices(
    reflectance
):

    blue = reflectance.select("Blue")
    red = reflectance.select("Red")
    nir = reflectance.select("NIR")
    swir1 = reflectance.select("SWIR1")
    swir2 = reflectance.select("SWIR2")

    ndvi = safe_normalized_difference(
        nir,
        red,
        "NDVI"
    )

    ndmi = safe_normalized_difference(
        nir,
        swir1,
        "NDMI"
    )

    nbr = safe_normalized_difference(
        nir,
        swir2,
        "NBR"
    )

    nbr2 = safe_normalized_difference(
        swir1,
        swir2,
        "NBR2"
    )

    evi_denominator = (
        nir
        .add(red.multiply(6))
        .subtract(blue.multiply(7.5))
        .add(1)
    )

    evi = (
        nir.subtract(red)
        .multiply(2.5)
        .divide(evi_denominator)
        .updateMask(
            evi_denominator.abs().gt(
                DENOMINATOR_EPSILON
            )
        )
        .rename("EVI")
    )

    return ee.Image.cat(
        [
            ndvi,
            evi,
            ndmi,
            nbr,
            nbr2
        ]
    ).toFloat()

# ============================================================
# 13. PIXEL-LEVEL RADIOMETRIC SCREENING
# ============================================================

def prepare_screened_image(
    image,
    sensor
):

    image = ee.Image(image)

    reflectance = get_scaled_reflectance(
        image,
        sensor
    )

    qa_mask = landsat_base_mask(
        image
    )

    # All six reflectance bands must be available.

    native_band_mask = (
        reflectance.mask()
        .reduce(ee.Reducer.min())
        .gt(0)
    )

    base_mask = (
        qa_mask
        .And(native_band_mask)
    )

    base_reflectance = (
        reflectance.updateMask(
            base_mask
        )
    )

    # Radiometric diagnostic flags.

    negative_flag = (
        base_reflectance
        .lt(REFLECTANCE_MIN)
        .reduce(ee.Reducer.max())
        .rename("negative_reflectance")
    )

    above_one_flag = (
        base_reflectance
        .gt(REFLECTANCE_MAX)
        .reduce(ee.Reducer.max())
        .rename("reflectance_gt_one")
    )

    reflectance_valid = (
        negative_flag.eq(0)
        .And(above_one_flag.eq(0))
    )

    # Indices are calculated after reflectance screening.

    clean_reflectance = (
        base_reflectance.updateMask(
            reflectance_valid
        )
    )

    clean_indices = (
        calculate_spectral_indices(
            clean_reflectance
        )
    )

    normalized_indices = (
        clean_indices.select(
            [
                "NDVI",
                "NDMI",
                "NBR",
                "NBR2"
            ]
        )
    )

    normalized_valid = (
        normalized_indices
        .gte(-1)
        .And(
            normalized_indices.lte(1)
        )
        .reduce(ee.Reducer.min())
        .eq(1)
    )

    evi_valid = (
        clean_indices.select("EVI")
        .gte(EVI_MIN)
        .And(
            clean_indices.select("EVI")
            .lte(EVI_MAX)
        )
    )

    index_availability = (
        clean_indices.mask()
        .reduce(ee.Reducer.min())
        .gt(0)
    )

    final_mask = (
        base_mask
        .And(reflectance_valid)
        .And(normalized_valid)
        .And(evi_valid)
        .And(index_availability)
    )

    screened_spectral = (
        clean_reflectance
        .addBands(clean_indices)
        .updateMask(final_mask)
        .select(ALL_SPECTRAL)
    )

    # Area images use the exact same fixed analysis grid.
    # Unmasked zero values ensure missing pixels contribute
    # zero valid area rather than disappearing from totals.

    pixel_area = ee.Image.pixelArea()

    base_area = (
        pixel_area.multiply(
            base_mask.unmask(
                0,
                False
            )
        )
        .rename("base_valid_area_m2")
    )

    reflectance_area = (
        pixel_area.multiply(
            base_mask
            .And(reflectance_valid)
            .unmask(0, False)
        )
        .rename(
            "reflectance_valid_area_m2"
        )
    )

    screened_area = (
        pixel_area.multiply(
            final_mask.unmask(
                0,
                False
            )
        )
        .rename(
            "screened_valid_area_m2"
        )
    )

    negative_area = (
        pixel_area.multiply(
            negative_flag.unmask(
                0,
                False
            )
        )
        .rename(
            "negative_reflectance_area_m2"
        )
    )

    above_one_area = (
        pixel_area.multiply(
            above_one_flag.unmask(
                0,
                False
            )
        )
        .rename(
            "reflectance_gt_one_area_m2"
        )
    )

    area_image = ee.Image.cat(
        [
            base_area,
            reflectance_area,
            screened_area,
            negative_area,
            above_one_area
        ]
    )

    return (
        screened_spectral,
        area_image
    )

# ============================================================
# 14. SPATIAL REDUCERS
# ============================================================

spectral_reducer = (
    ee.Reducer.mean()
    .combine(
        reducer2=ee.Reducer.median(),
        sharedInputs=True
    )
    .combine(
        reducer2=ee.Reducer.stdDev(),
        sharedInputs=True
    )
    .combine(
        reducer2=ee.Reducer.count(),
        sharedInputs=True
    )
)

AREA_BANDS = [
    "base_valid_area_m2",
    "reflectance_valid_area_m2",
    "screened_valid_area_m2",
    "negative_reflectance_area_m2",
    "reflectance_gt_one_area_m2"
]


# ============================================================
# 14B. LANDSAT ASSET RESOLUTION
# ============================================================

def resolve_landsat_image(sensor, scene_id):

    collection_id = LANDSAT_COLLECTIONS[sensor]

    # Resolve the image using its official product ID,
    # rather than treating that ID as an EE asset name.

    collection = ee.ImageCollection(
        collection_id
    )

    matched = collection.filter(
        ee.Filter.eq(
            "LANDSAT_PRODUCT_ID",
            str(scene_id)
        )
    )

    return ee.Image(matched.first())

# ============================================================
# 15. EXTRACT ONE SCENE
# ============================================================

def extract_screened_scene(
    scene_record
):

    sensor = str(
        scene_record["sensor"]
    )

    scene_id = str(
        scene_record["scene_id"]
    )

    image = resolve_landsat_image(
    sensor,
    scene_id
)
    spectral_image, area_image = (
        prepare_screened_image(
            image,
            sensor
        )
    )

    spectral_fc = (
        spectral_image.reduceRegions(
            collection=screen_fc,
            reducer=spectral_reducer,
            crs=ANALYSIS_CRS,
            crsTransform=ANALYSIS_TRANSFORM,
            tileScale=4
        )
    )

    area_fc = (
        area_image.reduceRegions(
            collection=screen_fc,
            reducer=ee.Reducer.sum(),
            crs=ANALYSIS_CRS,
            crsTransform=ANALYSIS_TRANSFORM,
            tileScale=4
        )
    )

    # Join spectral and area statistics by unit_id.

    joined_fc = ee.FeatureCollection(
        ee.Join.saveFirst(
            "area_record"
        ).apply(
            primary=spectral_fc,
            secondary=area_fc,
            condition=ee.Filter.equals(
                leftField="unit_id",
                rightField="unit_id"
            )
        )
    )

    def finalize_feature(feature):

        feature = ee.Feature(feature)

        area_feature = ee.Feature(
            feature.get("area_record")
        )

        area_properties = (
            area_feature.toDictionary(
                AREA_BANDS
            )
        )

        metadata = {
            "scene_id": scene_id,
            "sensor": sensor,
            "date": str(
                scene_record["date"]
            ),
            "year": int(
                scene_record["year"]
            ),
            "month": int(
                scene_record["month"]
            ),
            "scene_cloud_cover_pct": (
                image.get("CLOUD_COVER")
            ),
            "sun_elevation": (
                image.get("SUN_ELEVATION")
            ),
            "sun_azimuth": (
                image.get("SUN_AZIMUTH")
            ),
            "wrs_path": (
                image.get("WRS_PATH")
            ),
            "wrs_row": (
                image.get("WRS_ROW")
            )
        }

        # Return properties only.
        # Avoid exporting repeated geometries and
        # nested joined features.

        properties = (
            feature.toDictionary(
                feature.propertyNames()
                .remove("area_record")
            )
            .combine(
                area_properties,
                overwrite=True
            )
            .combine(
                ee.Dictionary(metadata),
                overwrite=True
            )
        )

        return ee.Feature(
            None,
            properties
        )

    return joined_fc.map(
        finalize_feature
    )

# ============================================================
# 16. EARTH ENGINE REQUEST WITH RETRIES
# ============================================================

def get_ee_info_retry(
    ee_object,
    attempts=4,
    initial_wait=3
):

    last_error = None

    for attempt in range(attempts):

        try:

            return ee_object.getInfo()

        except Exception as error:

            last_error = error

            wait_time = (
                initial_wait * (2 ** attempt)
            )

            print(
                f"EE request failed "
                f"({attempt + 1}/{attempts}): "
                f"{str(error)[:200]}"
            )

            if attempt < attempts - 1:

                time.sleep(wait_time)

    raise RuntimeError(
        f"Earth Engine request failed: "
        f"{last_error}"
    )

# ============================================================
# 17. CONVERT EARTH ENGINE RESULTS TO DATAFRAME
# ============================================================

def ee_features_to_dataframe(
    result
):

    rows = [
        feature["properties"]
        for feature in result["features"]
    ]

    return pd.DataFrame(rows)

# ============================================================
# 18. VALIDATE AND CALCULATE COVERAGE
# ============================================================

def validate_screened_dataframe(
    dataframe
):

    dataframe = dataframe.copy()

    required = [
        "scene_id",
        "sensor",
        "unit_id",
        "forest_type",
        "grid_area_m2",
        "base_valid_area_m2",
        "reflectance_valid_area_m2",
        "screened_valid_area_m2"
    ]

    assert set(required).issubset(
        dataframe.columns
    ), (
        f"Missing columns: "
        f"{set(required) - set(dataframe.columns)}"
    )

    for column in (
        ["grid_area_m2"] + AREA_BANDS
    ):

        if column not in dataframe.columns:

            raise KeyError(
                f"Missing area output: {column}"
            )

        dataframe[column] = (
            pd.to_numeric(
                dataframe[column],
                errors="raise"
            )
        )

    assert dataframe[
        ["grid_area_m2"] + AREA_BANDS
    ].notna().all().all()

    assert (
        dataframe["grid_area_m2"] > 0
    ).all()

    assert not dataframe.duplicated(
        [
            "sensor",
            "scene_id",
            "unit_id"
        ]
    ).any()

    assert dataframe[
        "unit_id"
    ].isin(EXPECTED_UNITS).all()

    for column in AREA_BANDS:

        assert (
            dataframe[column] >= -1e-6
        ).all()

    # Check nesting of the valid areas.

    tolerance = (
        dataframe["grid_area_m2"]
        * 1e-6
        + 1e-6
    )

    assert (
        dataframe["base_valid_area_m2"]
        <=
        dataframe["grid_area_m2"]
        + tolerance
    ).all()

    assert (
        dataframe["reflectance_valid_area_m2"]
        <=
        dataframe["base_valid_area_m2"]
        + tolerance
    ).all()

    assert (
        dataframe["screened_valid_area_m2"]
        <=
        dataframe["reflectance_valid_area_m2"]
        + tolerance
    ).all()

    dataframe["base_valid_fraction"] = (
        dataframe["base_valid_area_m2"]
        /
        dataframe["grid_area_m2"]
    ).clip(0, 1)

    dataframe["reflectance_valid_fraction"] = (
        dataframe["reflectance_valid_area_m2"]
        /
        dataframe["grid_area_m2"]
    ).clip(0, 1)

    dataframe["screened_valid_fraction"] = (
        dataframe["screened_valid_area_m2"]
        /
        dataframe["grid_area_m2"]
    ).clip(0, 1)

    dataframe["radiometric_loss_fraction"] = (
        dataframe["base_valid_fraction"]
        -
        dataframe["screened_valid_fraction"]
    ).clip(0, 1)

    # Fraction of initially valid area removed
    # by radiometric and index screening.

    dataframe["radiometric_loss_of_base"] = (
        np.where(
            dataframe["base_valid_area_m2"] > 0,
            (
                dataframe["base_valid_area_m2"]
                -
                dataframe["screened_valid_area_m2"]
            )
            /
            dataframe["base_valid_area_m2"],
            np.nan
        )
    )

    dataframe["negative_fraction_of_base"] = (
        np.where(
            dataframe["base_valid_area_m2"] > 0,
            dataframe[
                "negative_reflectance_area_m2"
            ]
            /
            dataframe["base_valid_area_m2"],
            np.nan
        )
    )

    dataframe["above_one_fraction_of_base"] = (
        np.where(
            dataframe["base_valid_area_m2"] > 0,
            dataframe[
                "reflectance_gt_one_area_m2"
            ]
            /
            dataframe["base_valid_area_m2"],
            np.nan
        )
    )

    # Guarantee a consistent schema even when a
    # scene has no remaining valid spectral pixels.

    for band in ALL_SPECTRAL:

        for statistic in [
            "mean",
            "median",
            "stdDev",
            "count"
        ]:

            column = (
                f"{band}_{statistic}"
            )

            if column not in dataframe.columns:

                dataframe[column] = (
                    np.nan
                )

            dataframe[column] = (
                pd.to_numeric(
                    dataframe[column],
                    errors="coerce"
                )
            )

            if statistic == "count":

                dataframe[column] = (
                    dataframe[column]
                    .fillna(0)
                    .astype(int)
                )

    dataframe["screened_local_ge_50"] = (
        dataframe["screened_valid_fraction"]
        >= 0.50
    )

    dataframe["screened_local_ge_70"] = (
        dataframe["screened_valid_fraction"]
        >= 0.70
    )

    dataframe["screened_local_ge_90"] = (
        dataframe["screened_valid_fraction"]
        >= 0.90
    )

    dataframe["screened_missing_index"] = (
        dataframe[
            [
                f"{index}_mean"
                for index in SPECTRAL_INDICES
            ]
        ]
        .isna()
        .any(axis=1)
    )

    dataframe["screen_version"] = (
        SCREEN_VERSION
    )

    dataframe["configuration_sha256"] = (
        config_hash
    )

    return dataframe

# ============================================================
# 19. TEST SCENE — 2019
# ============================================================

test_candidates = selected_scenes.loc[
    (
        selected_scenes["year"] == 2019
    )
    &
    (
        selected_scenes["sensor"] == "Landsat_8"
    )
    &
    (
        selected_scenes["date"] == "2019-05-09"
    )
]

if test_candidates.empty:

    test_candidates = selected_scenes.loc[
        selected_scenes["year"] == 2019
    ]

assert not test_candidates.empty

test_record = (
    test_candidates.iloc[0].to_dict()
)

print("\n" + "=" * 75)
print("2019 PIXEL-SCREENING VALIDATION")
print("=" * 75)

test_screened_fc = (
    extract_screened_scene(
        test_record
    )
)

test_screened_result = (
    get_ee_info_retry(
        test_screened_fc
    )
)

test_screened_df = (
    ee_features_to_dataframe(
        test_screened_result
    )
)

test_screened_df = (
    validate_screened_dataframe(
        test_screened_df
    )
)

assert len(test_screened_df) == 6

assert set(
    test_screened_df["unit_id"]
) == set(EXPECTED_UNITS)

display(
    test_screened_df[
        [
            "unit_id",
            "base_valid_fraction",
            "screened_valid_fraction",
            "radiometric_loss_of_base",
            "NDVI_mean",
            "EVI_mean",
            "NDMI_mean",
            "NBR_mean",
            "NBR2_mean"
        ]
    ].round(4)
)

test_screened_df.to_csv(
    SCREEN_DIR / "SCREENING_TEST_2019.csv",
    index=False
)

print("Test scene: PASSED")

# ============================================================
# 20. ANNUAL SCREENING WITH CHECKPOINTS
# ============================================================

print("\n" + "=" * 75)
print("LANDSAT PIXEL SCREENING — 1985–2025")
print("=" * 75)

annual_screened_frames = []

screening_failed_years = []

# Limit request size to reduce Earth Engine
# concurrent-aggregation and memory problems.

SCENE_BATCH_SIZE = 5

for year in tqdm(
    range(
        START_YEAR,
        END_YEAR + 1
    ),
    desc="Screening Landsat years"
):

    annual_scenes = selected_scenes.loc[
        selected_scenes["year"] == year
    ].copy()

    expected_rows = (
        len(annual_scenes) * 6
    )

    if annual_scenes.empty:

        continue

    annual_path = (
        SCREEN_CACHE
        / f"LANDSAT_SCREENED_{year}.csv"
    )

    cache_valid = False

    # --------------------------------------------------------
    # Recover annual checkpoint if configuration matches.
    # --------------------------------------------------------

    if annual_path.exists():

        try:

            cached_year = pd.read_csv(
                annual_path
            )

            expected_keys = set(
                zip(
                    annual_scenes["sensor"],
                    annual_scenes["scene_id"]
                )
            )

            cached_keys = set(
                zip(
                    cached_year["sensor"],
                    cached_year["scene_id"]
                )
            )

            cache_valid = (
                len(cached_year)
                == expected_rows
                and
                cached_keys == expected_keys
                and
                "configuration_sha256"
                in cached_year.columns
                and
                cached_year[
                    "configuration_sha256"
                ].eq(config_hash).all()
                and
                not cached_year.duplicated(
                    [
                        "sensor",
                        "scene_id",
                        "unit_id"
                    ]
                ).any()
            )

            if cache_valid:

                cached_year = (
                    validate_screened_dataframe(
                        cached_year
                    )
                )

                annual_screened_frames.append(
                    cached_year
                )

                print(
                    f"{year}: "
                    f"{len(annual_scenes)} scenes, "
                    f"{len(cached_year)} rows "
                    f"— restored from checkpoint"
                )

                continue

        except Exception as error:

            print(
                f"{year}: cache rejected — "
                f"{str(error)[:150]}"
            )

    # --------------------------------------------------------
    # Process annual scenes.
    # --------------------------------------------------------

    try:

        annual_records = (
            annual_scenes.to_dict(
                orient="records"
            )
        )

        batch_frames = []

        for start in range(
            0,
            len(annual_records),
            SCENE_BATCH_SIZE
        ):

            batch_records = annual_records[
                start:
                start + SCENE_BATCH_SIZE
            ]

            batch_fc_list = [
                extract_screened_scene(
                    record
                )
                for record in batch_records
            ]

            batch_fc = (
                ee.FeatureCollection(
                    batch_fc_list
                ).flatten()
            )

            batch_result = (
                get_ee_info_retry(
                    batch_fc
                )
            )

            batch_df = (
                ee_features_to_dataframe(
                    batch_result
                )
            )

            assert len(batch_df) == (
                len(batch_records) * 6
            ), (
                f"Unexpected batch size "
                f"in {year}: {len(batch_df)}"
            )

            batch_frames.append(
                batch_df
            )

        screened_year = pd.concat(
            batch_frames,
            ignore_index=True
        )

        screened_year = (
            validate_screened_dataframe(
                screened_year
            )
        )

        assert len(screened_year) == (
            expected_rows
        )

        unit_counts = (
            screened_year.groupby(
                ["sensor", "scene_id"]
            )["unit_id"]
            .nunique()
        )

        assert unit_counts.eq(6).all()

        # Atomic annual checkpoint.

        temporary_path = (
            annual_path.with_suffix(
                ".tmp"
            )
        )

        screened_year.to_csv(
            temporary_path,
            index=False
        )

        temporary_path.replace(
            annual_path
        )

        annual_screened_frames.append(
            screened_year
        )

        print(
            f"{year}: "
            f"{len(annual_scenes)} scenes, "
            f"{len(screened_year)} rows saved"
        )

    except Exception as error:

        screening_failed_years.append(
            {
                "year": year,
                "expected_scenes": len(
                    annual_scenes
                ),
                "error": str(error)
            }
        )

        print(
            f"FAILED YEAR {year}: "
            f"{str(error)[:300]}"
        )

# ============================================================
# 21. CHECK PROCESSING COMPLETENESS
# ============================================================

failed_years_df = pd.DataFrame(
    screening_failed_years,
    columns=[
        "year",
        "expected_scenes",
        "error"
    ]
)

failed_years_df.to_csv(
    SCREEN_DIR / "SCREENING_FAILED_YEARS.csv",
    index=False
)

assert len(screening_failed_years) == 0, (
    f"Some years failed: "
    f"{failed_years_df['year'].tolist()}. "
    "Successful checkpoints were preserved. "
    "Rerun Cell 90B to recover missing years."
)

assert annual_screened_frames

landsat_screened_master = pd.concat(
    annual_screened_frames,
    ignore_index=True
)

landsat_screened_master = (
    validate_screened_dataframe(
        landsat_screened_master
    )
)

landsat_screened_master = (
    landsat_screened_master
    .sort_values(
        [
            "year",
            "date",
            "sensor",
            "unit_id"
        ]
    )
    .reset_index(drop=True)
)

assert len(landsat_screened_master) == 2328

assert (
    landsat_screened_master[
        ["sensor", "scene_id"]
    ]
    .drop_duplicates()
    .shape[0]
    == 388
)

assert (
    landsat_screened_master
    .groupby(
        ["sensor", "scene_id"]
    )["unit_id"]
    .nunique()
    .eq(6)
    .all()
)

print("\nAll annual records validated.")

# ============================================================
# 22. RECOMPUTE PAIRED FRAGMENT COVERAGE
# ============================================================

screened_fragments = (
    landsat_screened_master.loc[
        landsat_screened_master[
            "spatial_scale"
        ] == "Fragment"
    ].copy()
)

screened_pairs = (
    screened_fragments.pivot(
        index=[
            "sensor",
            "scene_id",
            "date",
            "year",
            "month"
        ],
        columns="unit_id",
        values="screened_valid_fraction"
    )
    .reset_index()
)

screened_pairs.columns.name = None

assert len(screened_pairs) == 388

assert {
    "FRAG_01",
    "FRAG_02"
}.issubset(
    screened_pairs.columns
)

assert screened_pairs[
    ["FRAG_01", "FRAG_02"]
].notna().all().all()

screened_pairs[
    "screened_minimum_valid_fraction"
] = screened_pairs[
    ["FRAG_01", "FRAG_02"]
].min(axis=1)

for threshold in [50, 70, 90]:

    screened_pairs[
        f"screened_paired_ge_{threshold}"
    ] = (
        screened_pairs[
            "screened_minimum_valid_fraction"
        ]
        >= threshold / 100
    )

screened_pairs["primary_season"] = (
    screened_pairs["month"]
    .isin(PRIMARY_MONTHS)
)

pair_columns = [
    "sensor",
    "scene_id",
    "screened_minimum_valid_fraction",
    "screened_paired_ge_50",
    "screened_paired_ge_70",
    "screened_paired_ge_90",
    "primary_season"
]

landsat_screened_master = (
    landsat_screened_master.merge(
        screened_pairs[pair_columns],
        on=["sensor", "scene_id"],
        how="left",
        validate="many_to_one"
    )
)

landsat_screened_master[
    "screened_primary_eligible"
] = (
    landsat_screened_master[
        "screened_paired_ge_70"
    ]
    &
    landsat_screened_master[
        "screened_local_ge_70"
    ]
    &
    landsat_screened_master[
        "primary_season"
    ]
    &
    ~landsat_screened_master[
        "screened_missing_index"
    ]
)

# ============================================================
# 23. COMPARE NATIVE AND SCREENED SPECTRA
# ============================================================

comparison_columns = [
    "sensor",
    "scene_id",
    "unit_id"
]

for index in SPECTRAL_INDICES:

    comparison_columns.extend(
        [
            f"{index}_mean",
            f"{index}_median"
        ]
    )

native_comparison = native_spectral[
    comparison_columns
].copy()

native_comparison = (
    native_comparison.rename(
        columns={
            column: f"{column}_native"
            for column in comparison_columns
            if column not in [
                "sensor",
                "scene_id",
                "unit_id"
            ]
        }
    )
)

screened_comparison = (
    landsat_screened_master.merge(
        native_comparison,
        on=[
            "sensor",
            "scene_id",
            "unit_id"
        ],
        how="left",
        validate="one_to_one"
    )
)

for index in SPECTRAL_INDICES:

    screened_comparison[
        f"{index}_mean_change"
    ] = (
        screened_comparison[
            f"{index}_mean"
        ]
        -
        screened_comparison[
            f"{index}_mean_native"
        ]
    )

    screened_comparison[
        f"{index}_median_change"
    ] = (
        screened_comparison[
            f"{index}_median"
        ]
        -
        screened_comparison[
            f"{index}_median_native"
        ]
    )

# ============================================================
# 24. FINAL SPECTRAL QUALITY CONTROL
# ============================================================

screening_qc_rows = []

for index in SPECTRAL_INDICES:

    values = (
        landsat_screened_master[
            f"{index}_mean"
        ]
    )

    medians = (
        landsat_screened_master[
            f"{index}_median"
        ]
    )

    if index == "EVI":

        lower_bound = EVI_MIN
        upper_bound = EVI_MAX

    else:

        lower_bound = -1
        upper_bound = 1

    outside_mean = (
        (
            values
            < lower_bound - 1e-6
        )
        |
        (
            values
            > upper_bound + 1e-6
        )
    )

    outside_median = (
        (
            medians
            < lower_bound - 1e-6
        )
        |
        (
            medians
            > upper_bound + 1e-6
        )
    )

    screening_qc_rows.append(
        {
            "index": index,
            "total_rows": len(values),
            "nonmissing": values.notna().sum(),
            "missing": values.isna().sum(),
            "minimum": values.min(),
            "maximum": values.max(),
            "mean": values.mean(),
            "median": values.median(),
            "outside_mean_range": (
                outside_mean.sum()
            ),
            "outside_median_range": (
                outside_median.sum()
            )
        }
    )

screened_index_qc = pd.DataFrame(
    screening_qc_rows
)

assert (
    screened_index_qc[
        "outside_mean_range"
    ].sum() == 0
)

assert (
    screened_index_qc[
        "outside_median_range"
    ].sum() == 0
)

print("\nSCREENED SPECTRAL INDEX QC")

display(
    screened_index_qc.round(5)
)

# ============================================================
# 25. SENSOR-LEVEL SCREENING SUMMARY
# ============================================================

screened_sensor_summary = (
    landsat_screened_master
    .groupby(
        [
            "sensor",
            "unit_id",
            "forest_type",
            "spatial_scale"
        ],
        as_index=False
    )
    .agg(
        n_scenes=(
            "scene_id",
            "nunique"
        ),
        mean_base_coverage=(
            "base_valid_fraction",
            "mean"
        ),
        mean_screened_coverage=(
            "screened_valid_fraction",
            "mean"
        ),
        median_screened_coverage=(
            "screened_valid_fraction",
            "median"
        ),
        mean_radiometric_loss=(
            "radiometric_loss_fraction",
            "mean"
        ),
        mean_negative_fraction=(
            "negative_fraction_of_base",
            "mean"
        ),
        scenes_ge_70=(
            "screened_local_ge_70",
            "sum"
        ),
        scenes_ge_90=(
            "screened_local_ge_90",
            "sum"
        )
    )
)

for column in [
    "mean_base_coverage",
    "mean_screened_coverage",
    "median_screened_coverage",
    "mean_radiometric_loss",
    "mean_negative_fraction"
]:

    screened_sensor_summary[
        column + "_pct"
    ] = (
        screened_sensor_summary[column]
        * 100
    )

print("\nSENSOR-LEVEL SCREENING SUMMARY")

display(
    screened_sensor_summary[
        [
            "sensor",
            "unit_id",
            "n_scenes",
            "mean_base_coverage_pct",
            "mean_screened_coverage_pct",
            "mean_radiometric_loss_pct",
            "scenes_ge_70"
        ]
    ].round(3)
)

# ============================================================
# 26. PAIRED SCENE RETENTION
# ============================================================

native_pair_columns = [
    "sensor",
    "scene_id"
]

if "paired_ge_70" in native_spectral.columns:

    native_pairs = (
        native_spectral[
            native_pair_columns
            + ["paired_ge_70"]
        ]
        .drop_duplicates()
        .rename(
            columns={
                "paired_ge_70":
                "native_paired_ge_70"
            }
        )
    )

    assert not native_pairs.duplicated(
        native_pair_columns
    ).any()

    native_pairs[
        "native_paired_ge_70"
    ] = (
        native_pairs[
            "native_paired_ge_70"
        ]
        .astype(str)
        .str.lower()
        .isin(["true", "1"])
    )

    screened_pairs = (
        screened_pairs.merge(
            native_pairs,
            on=native_pair_columns,
            how="left",
            validate="one_to_one"
        )
    )

else:

    screened_pairs[
        "native_paired_ge_70"
    ] = np.nan

screened_pairs["lost_native_ge_70"] = (
    screened_pairs[
        "native_paired_ge_70"
    ].eq(True)
    &
    ~screened_pairs[
        "screened_paired_ge_70"
    ]
)

screened_pairs["gained_grid_ge_70"] = (
    screened_pairs[
        "native_paired_ge_70"
    ].eq(False)
    &
    screened_pairs[
        "screened_paired_ge_70"
    ]
)

screened_retention = (
    screened_pairs
    .groupby(
        "sensor",
        as_index=False
    )
    .agg(
        selected_scenes=(
            "scene_id",
            "nunique"
        ),
        native_paired_ge_70=(
            "native_paired_ge_70",
            "sum"
        ),
        screened_paired_ge_50=(
            "screened_paired_ge_50",
            "sum"
        ),
        screened_paired_ge_70=(
            "screened_paired_ge_70",
            "sum"
        ),
        screened_paired_ge_90=(
            "screened_paired_ge_90",
            "sum"
        ),
        lost_native_ge_70=(
            "lost_native_ge_70",
            "sum"
        ),
        gained_grid_ge_70=(
            "gained_grid_ge_70",
            "sum"
        )
    )
)

screened_retention[
    "screened_retention_pct"
] = (
    screened_retention[
        "screened_paired_ge_70"
    ]
    /
    screened_retention[
        "selected_scenes"
    ]
    * 100
)

print("\nPAIRED SCENE RETENTION")

display(
    screened_retention.round(3)
)

# ============================================================
# 27. ANNUAL PAIRED AVAILABILITY
# ============================================================

screened_annual = (
    screened_pairs
    .groupby(
        "year",
        as_index=False
    )
    .agg(
        selected_scenes=(
            "scene_id",
            "nunique"
        ),
        native_paired_ge_70=(
            "native_paired_ge_70",
            "sum"
        ),
        screened_paired_ge_50=(
            "screened_paired_ge_50",
            "sum"
        ),
        screened_paired_ge_70=(
            "screened_paired_ge_70",
            "sum"
        ),
        screened_paired_ge_90=(
            "screened_paired_ge_90",
            "sum"
        )
    )
)

primary_annual = (
    screened_pairs.loc[
        screened_pairs["primary_season"]
    ]
    .groupby("year")[
        "screened_paired_ge_70"
    ]
    .sum()
    .rename(
        "primary_season_paired_ge_70"
    )
    .reset_index()
)

screened_annual = (
    screened_annual.merge(
        primary_annual,
        on="year",
        how="left"
    )
)

screened_annual[
    "primary_season_paired_ge_70"
] = (
    screened_annual[
        "primary_season_paired_ge_70"
    ]
    .fillna(0)
    .astype(int)
)

screened_annual[
    "adequate_annual_support"
] = (
    screened_annual[
        "screened_paired_ge_70"
    ] >= 3
)

screened_annual[
    "adequate_primary_support"
] = (
    screened_annual[
        "primary_season_paired_ge_70"
    ] >= 3
)

print("\nANNUAL SCREENED AVAILABILITY")

display(
    screened_annual
)

# ============================================================
# 28. SEASONAL WINDOW SENSITIVITY
# ============================================================

season_windows = {
    "Annual": list(range(1, 13)),
    "May_September": [5, 6, 7, 8, 9],
    "June_August": [6, 7, 8],
    "June_September": [6, 7, 8, 9],
    "July_September": [7, 8, 9]
}

season_screen_rows = []

for window, months in season_windows.items():

    window_data = screened_pairs.loc[
        screened_pairs["month"].isin(
            months
        )
        &
        screened_pairs[
            "screened_paired_ge_70"
        ]
    ]

    annual_counts = (
        window_data.groupby("year")
        .size()
        .reindex(
            range(
                START_YEAR,
                END_YEAR + 1
            ),
            fill_value=0
        )
    )

    season_screen_rows.append(
        {
            "window": window,
            "months": ",".join(
                map(str, months)
            ),
            "total_paired_scenes": len(
                window_data
            ),
            "years_ge_1": (
                annual_counts >= 1
            ).sum(),
            "years_ge_3": (
                annual_counts >= 3
            ).sum(),
            "years_ge_5": (
                annual_counts >= 5
            ).sum(),
            "median_scenes_per_year": (
                annual_counts.median()
            ),
            "minimum_scenes_per_year": (
                annual_counts.min()
            ),
            "scenes_in_2019": (
                annual_counts.loc[2019]
            )
        }
    )

screened_season_summary = pd.DataFrame(
    season_screen_rows
)

print("\nSCREENED SEASONAL WINDOWS")

display(
    screened_season_summary
)

# ============================================================
# 29. PRIMARY-SEASON SPECTRAL SUMMARY
# ============================================================

primary_screened = (
    landsat_screened_master.loc[
        landsat_screened_master[
            "screened_primary_eligible"
        ]
    ].copy()
)

primary_screened_fragments = (
    primary_screened.loc[
        primary_screened[
            "spatial_scale"
        ] == "Fragment"
    ].copy()
)

aggregation = {
    "n_scenes": (
        "scene_id",
        "nunique"
    ),
    "first_year": (
        "year",
        "min"
    ),
    "last_year": (
        "year",
        "max"
    )
}

for index in SPECTRAL_INDICES:

    aggregation[index] = (
        f"{index}_mean",
        "median"
    )

screened_native_sensor_summary = (
    primary_screened_fragments
    .groupby(
        [
            "sensor",
            "forest_type"
        ],
        as_index=False
    )
    .agg(**aggregation)
)

print("\nSCREENED PRIMARY-SEASON SPECTRA")

display(
    screened_native_sensor_summary.round(4)
)

# ============================================================
# 30. FIELD CENSUS YEAR — 2019
# ============================================================

screened_2019 = (
    landsat_screened_master.loc[
        (
            landsat_screened_master["year"]
            == 2019
        )
        &
        landsat_screened_master[
            "screened_paired_ge_70"
        ]
        &
        landsat_screened_master[
            "screened_local_ge_70"
        ]
        &
        ~landsat_screened_master[
            "screened_missing_index"
        ]
    ].copy()
)

aggregation_2019 = {
    "n_scenes": (
        "scene_id",
        "nunique"
    ),
    "median_valid_coverage": (
        "screened_valid_fraction",
        "median"
    )
}

for index in SPECTRAL_INDICES:

    aggregation_2019[index] = (
        f"{index}_mean",
        "median"
    )

screened_2019_summary = (
    screened_2019
    .groupby(
        [
            "unit_id",
            "forest_type",
            "spatial_scale",
            "sensor"
        ],
        as_index=False
    )
    .agg(**aggregation_2019)
)

print("\n2019 SCREENED SPECTRAL CONDITIONS")

display(
    screened_2019_summary.round(4)
)

# ============================================================
# 31. NATIVE VERSUS SCREENED INDEX CHANGES
# ============================================================

change_rows = []

for (
    sensor,
    forest_type
), group in screened_comparison.loc[
    screened_comparison[
        "spatial_scale"
    ] == "Fragment"
].groupby(
    ["sensor", "forest_type"]
):

    for index in SPECTRAL_INDICES:

        changes = group[
            f"{index}_mean_change"
        ].dropna()

        change_rows.append(
            {
                "sensor": sensor,
                "forest_type": forest_type,
                "index": index,
                "n_comparisons": len(changes),
                "mean_change": changes.mean(),
                "median_change": changes.median(),
                "median_absolute_change": (
                    changes.abs().median()
                ),
                "maximum_absolute_change": (
                    changes.abs().max()
                )
            }
        )

screened_change_summary = pd.DataFrame(
    change_rows
)

print("\nNATIVE–SCREENED SPECTRAL DIFFERENCES")

display(
    screened_change_summary.round(5)
)

# ============================================================
# 32. FIGURE — RADIOMETRIC SCREENING EFFECT
# ============================================================

plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titleweight": "bold"
    }
)

sensor_colors = {
    "Landsat_5": "#527B98",
    "Landsat_7": "#DB9140",
    "Landsat_8": "#24634F",
    "Landsat_9": "#795C92"
}

fig, axes = plt.subplots(
    2,
    2,
    figsize=(15, 10)
)

# ------------------------------------------------------------
# Panel A: annual paired scene retention
# ------------------------------------------------------------

ax = axes[0, 0]

ax.bar(
    screened_annual["year"],
    screened_annual["native_paired_ge_70"],
    color="#D1D5D8",
    width=0.85,
    label="Native QA"
)

ax.bar(
    screened_annual["year"],
    screened_annual["screened_paired_ge_70"],
    color="#24634F",
    width=0.65,
    label="Radiometrically screened"
)

ax.axvline(
    2019,
    color="#B64135",
    linestyle="--",
    linewidth=1.2,
    label="Field census"
)

ax.set_title(
    "(a) Annual paired scene availability",
    loc="left"
)

ax.set_ylabel(
    "Number of scenes"
)

ax.set_xlabel("Year")

ax.legend(
    frameon=False,
    fontsize=8
)

ax.grid(
    axis="y",
    alpha=0.2
)

# ------------------------------------------------------------
# Panel B: paired valid coverage before and after screening
# ------------------------------------------------------------

ax = axes[0, 1]

for sensor, group in (
    screened_fragments.groupby("sensor")
):

    ax.scatter(
        group["base_valid_fraction"] * 100,
        group["screened_valid_fraction"] * 100,
        s=23,
        alpha=0.65,
        color=sensor_colors[sensor],
        label=sensor
    )

ax.plot(
    [0, 100],
    [0, 100],
    color="gray",
    linestyle="--",
    linewidth=1
)

ax.axhline(
    70,
    color="#B64135",
    linestyle=":",
    linewidth=1
)

ax.set_xlim(0, 103)
ax.set_ylim(0, 103)

ax.set_xlabel(
    "Base QA valid coverage (%)"
)

ax.set_ylabel(
    "Screened valid coverage (%)"
)

ax.set_title(
    "(b) Coverage loss after screening",
    loc="left"
)

ax.legend(
    frameon=False,
    fontsize=8
)

ax.grid(alpha=0.15)

# ------------------------------------------------------------
# Panel C: NDVI before and after screening
# ------------------------------------------------------------

ax = axes[1, 0]

comparison_fragments = (
    screened_comparison.loc[
        screened_comparison[
            "spatial_scale"
        ] == "Fragment"
    ]
)

for sensor, group in (
    comparison_fragments.groupby("sensor")
):

    ax.scatter(
        group["NDVI_mean_native"],
        group["NDVI_mean"],
        s=24,
        alpha=0.65,
        color=sensor_colors[sensor],
        label=sensor
    )

ax.plot(
    [0, 1.6],
    [0, 1.6],
    color="gray",
    linestyle="--",
    linewidth=1
)

ax.axvline(
    1,
    color="#B64135",
    linestyle=":",
    linewidth=1
)

ax.axhline(
    1,
    color="#B64135",
    linestyle=":",
    linewidth=1
)

ax.set_xlabel(
    "Native mean NDVI"
)

ax.set_ylabel(
    "Screened mean NDVI"
)

ax.set_title(
    "(c) Correction of NDVI anomalies",
    loc="left"
)

ax.grid(alpha=0.15)

# ------------------------------------------------------------
# Panel D: sensor-level scene retention
# ------------------------------------------------------------

ax = axes[1, 1]

sensor_order = [
    "Landsat_5",
    "Landsat_7",
    "Landsat_8",
    "Landsat_9"
]

retention_plot = (
    screened_retention
    .set_index("sensor")
    .reindex(sensor_order)
)

positions = np.arange(
    len(sensor_order)
)

native_pct = (
    retention_plot[
        "native_paired_ge_70"
    ]
    /
    retention_plot[
        "selected_scenes"
    ]
    * 100
)

screened_pct = (
    retention_plot[
        "screened_paired_ge_70"
    ]
    /
    retention_plot[
        "selected_scenes"
    ]
    * 100
)

ax.bar(
    positions - 0.18,
    native_pct,
    width=0.36,
    color="#D1D5D8",
    label="Native QA"
)

ax.bar(
    positions + 0.18,
    screened_pct,
    width=0.36,
    color=[
        sensor_colors[sensor]
        for sensor in sensor_order
    ],
    label="Screened"
)

ax.set_xticks(
    positions,
    [
        "Landsat 5",
        "Landsat 7",
        "Landsat 8",
        "Landsat 9"
    ]
)

ax.set_ylim(0, 105)

ax.set_ylabel(
    "Paired scenes retained (%)"
)

ax.set_title(
    "(d) Sensor-specific retention",
    loc="left"
)

ax.legend(
    frameon=False,
    fontsize=8
)

ax.grid(
    axis="y",
    alpha=0.2
)

fig.suptitle(
    "Landsat pixel-level radiometric screening",
    fontsize=17,
    fontweight="bold"
)

fig.tight_layout(
    rect=[0, 0, 1, 0.96]
)

figure_stem = (
    SCREEN_FIGURES
    / "FIG_LANDSAT_PIXEL_SCREENING"
)

for extension in [
    "png",
    "pdf",
    "svg"
]:

    fig.savefig(
        figure_stem.with_suffix(
            f".{extension}"
        ),
        dpi=600,
        bbox_inches="tight"
    )

plt.show()

# ============================================================
# 33. FIGURE — SCREENED NATIVE-SENSOR TIME SERIES
# ============================================================

fig, axes = plt.subplots(
    len(SPECTRAL_INDICES),
    1,
    figsize=(15, 14),
    sharex=True
)

forest_markers = {
    "Late_secondary": "o",
    "Mature": "^"
}

plot_data = (
    primary_screened_fragments.copy()
)

plot_data["date_plot"] = pd.to_datetime(
    plot_data["date"]
)

for ax, index in zip(
    axes,
    SPECTRAL_INDICES
):

    for (
        sensor,
        forest_type
    ), group in plot_data.groupby(
        [
            "sensor",
            "forest_type"
        ]
    ):

        ax.scatter(
            group["date_plot"],
            group[f"{index}_mean"],
            color=sensor_colors[sensor],
            marker=forest_markers[
                forest_type
            ],
            alpha=0.70,
            s=25,
            label=(
                f"{sensor} — "
                f"{forest_type}"
            )
        )

    ax.axvline(
        pd.Timestamp("2019-01-01"),
        color="#B64135",
        linestyle="--",
        linewidth=1.1
    )

    ax.set_ylabel(index)

    ax.grid(alpha=0.18)

axes[-1].set_xlabel(
    "Acquisition date"
)

handles, labels = (
    axes[0].get_legend_handles_labels()
)

fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.955),
    ncol=4,
    frameon=False,
    fontsize=9
)

fig.suptitle(
    "Radiometrically screened Landsat observations\n"
    "May–September | Paired valid coverage ≥70%",
    fontsize=16,
    fontweight="bold",
    y=0.99
)

fig.tight_layout(
    rect=[0, 0, 1, 0.91]
)

series_stem = (
    SCREEN_FIGURES
    / "FIG_LANDSAT_SCREENED_NATIVE_SERIES"
)

for extension in [
    "png",
    "pdf",
    "svg"
]:

    fig.savefig(
        series_stem.with_suffix(
            f".{extension}"
        ),
        dpi=600,
        bbox_inches="tight"
    )

plt.show()

# ============================================================
# 34. EXPORT ALL SCIENTIFIC TABLES
# ============================================================

screened_exports = {
    "LANDSAT_SCREENED_MASTER":
        landsat_screened_master,

    "LANDSAT_SCREENED_PAIRED_SCENES":
        screened_pairs,

    "LANDSAT_SCREENED_INDEX_QC":
        screened_index_qc,

    "LANDSAT_SCREENED_SENSOR_SUMMARY":
        screened_sensor_summary,

    "LANDSAT_SCREENED_RETENTION":
        screened_retention,

    "LANDSAT_SCREENED_ANNUAL":
        screened_annual,

    "LANDSAT_SCREENED_SEASON_WINDOWS":
        screened_season_summary,

    "LANDSAT_SCREENED_PRIMARY_SPECTRA":
        primary_screened,

    "LANDSAT_SCREENED_NATIVE_SENSOR_SUMMARY":
        screened_native_sensor_summary,

    "LANDSAT_SCREENED_2019":
        screened_2019,

    "LANDSAT_SCREENED_2019_SUMMARY":
        screened_2019_summary,

    "LANDSAT_NATIVE_SCREENED_COMPARISON":
        screened_comparison,

    "LANDSAT_SCREENED_INDEX_CHANGES":
        screened_change_summary,

    "LANDSAT_GRID_REFERENCE":
        grid_reference
}

for filename, dataframe in (
    screened_exports.items()
):

    output_path = (
        SCREEN_DIR / f"{filename}.csv"
    )

    temporary_path = (
        output_path.with_suffix(".tmp")
    )

    dataframe.to_csv(
        temporary_path,
        index=False
    )

    temporary_path.replace(
        output_path
    )

# ============================================================
# 35. EXPORT EXCEL WORKBOOK
# ============================================================

screening_workbook = (
    SCREEN_TABLES
    / "SATIPO_LANDSAT_PIXEL_SCREENING.xlsx"
)

with pd.ExcelWriter(
    screening_workbook,
    engine="openpyxl"
) as writer:

    grid_reference.to_excel(
        writer,
        sheet_name="Grid_reference",
        index=False
    )

    test_screened_df.to_excel(
        writer,
        sheet_name="Test_2019",
        index=False
    )

    landsat_screened_master.to_excel(
        writer,
        sheet_name="Screened_master",
        index=False
    )

    screened_pairs.to_excel(
        writer,
        sheet_name="Paired_scenes",
        index=False
    )

    screened_index_qc.to_excel(
        writer,
        sheet_name="Index_QC",
        index=False
    )

    screened_sensor_summary.to_excel(
        writer,
        sheet_name="Sensor_summary",
        index=False
    )

    screened_retention.to_excel(
        writer,
        sheet_name="Scene_retention",
        index=False
    )

    screened_annual.to_excel(
        writer,
        sheet_name="Annual_support",
        index=False
    )

    screened_season_summary.to_excel(
        writer,
        sheet_name="Season_windows",
        index=False
    )

    screened_native_sensor_summary.to_excel(
        writer,
        sheet_name="Native_sensor_summary",
        index=False
    )

    screened_2019_summary.to_excel(
        writer,
        sheet_name="Census_2019",
        index=False
    )

    screened_change_summary.to_excel(
        writer,
        sheet_name="Index_changes",
        index=False
    )

# ============================================================
# 36. FINAL VALIDATION AND REPORT
# ============================================================

n_screened_50 = int(
    screened_pairs[
        "screened_paired_ge_50"
    ].sum()
)

n_screened_70 = int(
    screened_pairs[
        "screened_paired_ge_70"
    ].sum()
)

n_screened_90 = int(
    screened_pairs[
        "screened_paired_ge_90"
    ].sum()
)

n_primary_screened = int(
    (
        screened_pairs[
            "screened_paired_ge_70"
        ]
        &
        screened_pairs[
            "primary_season"
        ]
    ).sum()
)

n_2019_screened = int(
    (
        screened_pairs[
            "screened_paired_ge_70"
        ]
        &
        (
            screened_pairs["year"]
            == 2019
        )
    ).sum()
)

n_years_adequate = int(
    screened_annual[
        "adequate_annual_support"
    ].sum()
)

n_primary_years_adequate = int(
    screened_annual[
        "adequate_primary_support"
    ].sum()
)

n_invalid_indices = int(
    screened_index_qc[
        "outside_mean_range"
    ].sum()
    +
    screened_index_qc[
        "outside_median_range"
    ].sum()
)

screening_report = {
    "status": "COMPLETED",
    "version": SCREEN_VERSION,
    "selected_scenes": len(
        selected_scenes
    ),
    "scene_unit_observations": len(
        landsat_screened_master
    ),
    "spatial_units": len(
        screen_units
    ),
    "screened_paired_ge_50": (
        n_screened_50
    ),
    "screened_paired_ge_70": (
        n_screened_70
    ),
    "screened_paired_ge_90": (
        n_screened_90
    ),
    "primary_season_paired_ge_70": (
        n_primary_screened
    ),
    "census_2019_paired_ge_70": (
        n_2019_screened
    ),
    "years_with_ge_3_paired_scenes": (
        n_years_adequate
    ),
    "primary_years_with_ge_3_paired_scenes": (
        n_primary_years_adequate
    ),
    "missing_spectral_rows": int(
        landsat_screened_master[
            "screened_missing_index"
        ].sum()
    ),
    "out_of_range_index_statistics": (
        n_invalid_indices
    ),
    "failed_years": len(
        screening_failed_years
    ),
    "native_master_preserved": True,
    "cross_sensor_harmonization": (
        "PENDING"
    ),
    "configuration_sha256": config_hash
}

with open(
    SCREEN_DIR / "SCREENING_REPORT.json",
    "w"
) as file:

    json.dump(
        screening_report,
        file,
        indent=2
    )

assert NATIVE_MASTER_PATH.exists()

assert n_invalid_indices == 0

assert len(
    screening_failed_years
) == 0

assert len(
    landsat_screened_master
) == 2328

print("\n" + "=" * 75)
print("SATIPO — PIXEL-LEVEL SCREENING REPORT")
print("=" * 75)

print(
    "Selected scenes:",
    len(selected_scenes)
)

print(
    "Scene-unit observations:",
    len(landsat_screened_master)
)

print(
    "Validated spatial units:",
    len(screen_units)
)

print(
    "Paired scenes ≥50%:",
    n_screened_50
)

print(
    "Paired scenes ≥70%:",
    n_screened_70
)

print(
    "Paired scenes ≥90%:",
    n_screened_90
)

print(
    "Primary-season paired scenes:",
    n_primary_screened
)

print(
    "2019 paired scenes:",
    n_2019_screened
)

print(
    "Years with ≥3 paired scenes:",
    f"{n_years_adequate}/41"
)

print(
    "Primary-season years with ≥3 scenes:",
    f"{n_primary_years_adequate}/41"
)

print(
    "Out-of-range index statistics:",
    n_invalid_indices
)

print(
    "Failed years:",
    len(screening_failed_years)
)

print(
    "Original spectral master:",
    "PRESERVED"
)

print(
    "Cross-sensor harmonization:",
    "PENDING"
)

print("\nScreened spectral master:")

print(
    SCREEN_DIR
    / "LANDSAT_SCREENED_MASTER.csv"
)

print("\nWorkbook:")
print(screening_workbook)

print("\nAnnual checkpoints:")
print(SCREEN_CACHE)

print("\nFigures:")
print(figure_stem)
print(series_stem)

print("\n" + "=" * 75)
print("PIXEL-LEVEL RADIOMETRIC SCREENING COMPLETED")
print("NEXT: CELL 91 — CROSS-SENSOR HARMONIZATION")
print("=" * 75)


In [ ]:

# ============================================================================
# CELL 91A — CROSS-SENSOR TEMPORAL OVERLAP AND CALIBRATION FEASIBILITY
# SATIPO FOREST PROJECT
#
# Objectives:
#   1. Identify near-coincident Landsat acquisitions.
#   2. Construct one-to-one temporal matches.
#   3. Quantify cross-sensor spectral discrepancies.
#   4. Assess calibration feasibility.
#   5. Evaluate stability of paired forest contrasts.
#
# IMPORTANT:
#   This cell does NOT apply harmonization coefficients.
#   It evaluates whether local calibration is supported.
# ============================================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from scipy.optimize import linear_sum_assignment
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

# ============================================================================
# 1. PROJECT CONFIGURATION
# ============================================================================

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

REMOTE_DIR = ROOT / "04_REMOTE_SENSING"

LANDSAT_DIR = REMOTE_DIR / "03_LANDSAT"

SCREEN_DIR = (
    LANDSAT_DIR / "03_RADIOMETRIC_SCREENING"
)

HARM_DIR = (
    LANDSAT_DIR / "04_CROSS_SENSOR_HARMONIZATION"
)

TABLE_DIR = (
    REMOTE_DIR / "08_OUTPUTS" / "TABLES"
)

FIGURE_DIR = (
    REMOTE_DIR / "08_OUTPUTS" / "FIGURES"
)

for directory in [
    HARM_DIR,
    TABLE_DIR,
    FIGURE_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

SCREENED_PATH = (
    SCREEN_DIR / "LANDSAT_SCREENED_MASTER.csv"
)

assert SCREENED_PATH.exists(), (
    f"Screened master not found: {SCREENED_PATH}"
)

INDICES = [
    "NDVI",
    "EVI",
    "NDMI",
    "NBR",
    "NBR2"
]

SENSORS = [
    "Landsat_5",
    "Landsat_7",
    "Landsat_8",
    "Landsat_9"
]

SENSOR_PAIRS = [
    ("Landsat_5", "Landsat_7"),
    ("Landsat_7", "Landsat_8"),
    ("Landsat_8", "Landsat_9")
]

WINDOWS = [1, 4, 8, 16]

PRIMARY_MONTHS = [5, 6, 7, 8, 9]

PRIMARY_COVERAGE = 0.70

SENSITIVITY_COVERAGE = 0.90

PRIMARY_WINDOW = 8

# Exploratory feasibility thresholds.
# These are screening rules, not formal statistical guarantees.

MIN_PAIRS_OFFSET = 10
MIN_YEARS_OFFSET = 3

MIN_PAIRS_REGRESSION = 20
MIN_YEARS_REGRESSION = 4

SEED = 20261115

SENSOR_COLORS = {
    "Landsat_5": "#527B98",
    "Landsat_7": "#DB9140",
    "Landsat_8": "#24634F",
    "Landsat_9": "#795C92"
}

FOREST_COLORS = {
    "Late_secondary": "#DB9140",
    "Mature": "#24634F"
}

print("=" * 76)
print("SATIPO — CROSS-SENSOR TEMPORAL OVERLAP")
print("=" * 76)

# ============================================================================
# 2. LOAD SCREENED SPECTRAL MASTER
# ============================================================================

screened = pd.read_csv(
    SCREENED_PATH
)

required_columns = [
    "sensor",
    "scene_id",
    "date",
    "unit_id",
    "forest_type",
    "spatial_scale",
    "screened_valid_fraction"
] + [
    f"{index}_mean"
    for index in INDICES
]

missing_columns = [
    col
    for col in required_columns
    if col not in screened.columns
]

assert not missing_columns, (
    f"Missing required columns: {missing_columns}\n"
    f"Available columns: {screened.columns.tolist()}"
)

screened["date"] = pd.to_datetime(
    screened["date"],
    errors="raise",
    utc=True
).dt.tz_convert(None).dt.normalize()

screened["year"] = (
    screened["date"].dt.year
)

screened["month"] = (
    screened["date"].dt.month
)

screened["sensor"] = (
    screened["sensor"].astype(str)
)

screened["scene_id"] = (
    screened["scene_id"].astype(str)
)

screened["screened_valid_fraction"] = (
    pd.to_numeric(
        screened["screened_valid_fraction"],
        errors="coerce"
    )
)

for index in INDICES:

    screened[f"{index}_mean"] = (
        pd.to_numeric(
            screened[f"{index}_mean"],
            errors="coerce"
        )
    )

assert len(screened) == 2328, (
    f"Unexpected number of rows: {len(screened)}"
)

assert screened["scene_id"].nunique() == 388

assert not screened.duplicated(
    ["sensor", "scene_id", "unit_id"]
).any()

assert set(
    screened["sensor"].unique()
) == set(SENSORS)

valid_coverage = (
    screened["screened_valid_fraction"]
    .dropna()
)

assert valid_coverage.between(
    0, 1.001
).all()

print("\nSCREENED MASTER LOADED")
print("Rows:", len(screened))
print(
    "Scenes:",
    screened["scene_id"].nunique()
)

print(
    "Period:",
    screened["date"].min().date(),
    "to",
    screened["date"].max().date()
)

# ============================================================================
# 3. PREPARE FRAGMENT-LEVEL SCENE TABLE
#
# Calibration uses the two complete fragments.
#
# Plot and core observations are reserved for sensitivity analysis,
# because they are spatially nested within the fragments.
# ============================================================================

FRAGMENT_IDS = {
    "FRAG_01",
    "FRAG_02"
}

fragments = screened.loc[
    screened["unit_id"].isin(
        FRAGMENT_IDS
    )
].copy()

assert len(fragments) == 776

assert set(
    fragments["forest_type"].unique()
) == {
    "Late_secondary",
    "Mature"
}

scene_keys = [
    "sensor",
    "scene_id",
    "date",
    "year",
    "month"
]

value_columns = [
    "screened_valid_fraction"
] + [
    f"{index}_mean"
    for index in INDICES
]

fragment_wide = (
    fragments.pivot(
        index=scene_keys,
        columns="forest_type",
        values=value_columns
    )
)

fragment_wide.columns = [
    f"{variable}__{forest}"
    for variable, forest
    in fragment_wide.columns
]

fragment_wide = (
    fragment_wide.reset_index()
)

assert len(fragment_wide) == 388

coverage_columns = [
    "screened_valid_fraction__Late_secondary",
    "screened_valid_fraction__Mature"
]

fragment_wide["min_paired_coverage"] = (
    fragment_wide[coverage_columns]
    .min(axis=1, skipna=False)
)

spectral_columns = [
    f"{index}_mean__{forest}"
    for index in INDICES
    for forest in [
        "Late_secondary",
        "Mature"
    ]
]

fragment_wide["complete_spectra"] = (
    fragment_wide[spectral_columns]
    .notna()
    .all(axis=1)
)

fragment_wide["primary_season"] = (
    fragment_wide["month"]
    .isin(PRIMARY_MONTHS)
)

for index in INDICES:

    fragment_wide[
        f"{index}_forest_contrast"
    ] = (
        fragment_wide[
            f"{index}_mean__Late_secondary"
        ]
        -
        fragment_wide[
            f"{index}_mean__Mature"
        ]
    )

fragment_wide = (
    fragment_wide.sort_values(
        ["date", "sensor", "scene_id"]
    )
    .reset_index(drop=True)
)

print("\nFRAGMENT-LEVEL SCENE TABLE")

display(
    fragment_wide.groupby("sensor")
    .agg(
        n_scenes=("scene_id", "size"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        median_paired_coverage=(
            "min_paired_coverage",
            "median"
        )
    )
)

# ============================================================================
# 4. ELIGIBILITY SCENARIOS
# ============================================================================

SCENARIOS = {
    "Annual_ge70": {
        "coverage": 0.70,
        "months": list(range(1, 13)),
        "exclude_l7_after_2017": False
    },
    "MaySep_ge70": {
        "coverage": 0.70,
        "months": PRIMARY_MONTHS,
        "exclude_l7_after_2017": False
    },
    "Annual_ge90": {
        "coverage": 0.90,
        "months": list(range(1, 13)),
        "exclude_l7_after_2017": False
    },
    "MaySep_ge90": {
        "coverage": 0.90,
        "months": PRIMARY_MONTHS,
        "exclude_l7_after_2017": False
    },
    "JunSep_ge70": {
        "coverage": 0.70,
        "months": [6, 7, 8, 9],
        "exclude_l7_after_2017": False
    },
    "Annual_ge70_L7_until2017": {
        "coverage": 0.70,
        "months": list(range(1, 13)),
        "exclude_l7_after_2017": True
    }
}


def get_eligible_scenes(
    data,
    scenario
):

    config = SCENARIOS[scenario]

    eligible = data.loc[
        (
            data["min_paired_coverage"]
            >= config["coverage"]
        )
        &
        data["complete_spectra"]
        &
        data["month"].isin(
            config["months"]
        )
    ].copy()

    if config["exclude_l7_after_2017"]:

        eligible = eligible.loc[
            ~(
                (
                    eligible["sensor"]
                    == "Landsat_7"
                )
                &
                (
                    eligible["year"] > 2017
                )
            )
        ].copy()

    return eligible


eligibility_records = []

for scenario in SCENARIOS:

    eligible = get_eligible_scenes(
        fragment_wide,
        scenario
    )

    for sensor in SENSORS:

        subset = eligible.loc[
            eligible["sensor"] == sensor
        ]

        eligibility_records.append({
            "scenario": scenario,
            "sensor": sensor,
            "n_scenes": len(subset),
            "n_years": subset["year"].nunique(),
            "median_coverage": (
                subset["min_paired_coverage"]
                .median()
            )
        })

eligibility_summary = pd.DataFrame(
    eligibility_records
)

print("\nSCENE ELIGIBILITY")

display(
    eligibility_summary.pivot(
        index="scenario",
        columns="sensor",
        values="n_scenes"
    )
)

# ============================================================================
# 5. OPTIMAL ONE-TO-ONE TEMPORAL MATCHING
#
# The assignment algorithm:
#   - Maximizes the number of eligible matches.
#   - Minimizes total temporal separation among those matches.
#   - Does not reuse a scene within a sensor pair and scenario.
#
# Unmatched scenes are represented by dummy assignments.
# ============================================================================

def optimal_temporal_matching(
    data,
    sensor_a,
    sensor_b,
    max_days
):

    a = (
        data.loc[
            data["sensor"] == sensor_a
        ]
        .sort_values(
            ["date", "scene_id"]
        )
        .reset_index(drop=True)
    )

    b = (
        data.loc[
            data["sensor"] == sensor_b
        ]
        .sort_values(
            ["date", "scene_id"]
        )
        .reset_index(drop=True)
    )

    n_a = len(a)
    n_b = len(b)

    if n_a == 0 or n_b == 0:
        return []

    dates_a = (
        a["date"]
        .to_numpy(dtype="datetime64[D]")
    )

    dates_b = (
        b["date"]
        .to_numpy(dtype="datetime64[D]")
    )

    signed_days = (
        dates_b[None, :]
        -
        dates_a[:, None]
    ).astype(
        "timedelta64[D]"
    ).astype(int)

    distance = np.abs(
        signed_days
    )

    allowed = (
        distance <= max_days
    )

    # A penalty large enough to prioritize the number
    # of matches over differences in total date separation.

    unmatched_penalty = (
        (n_a + n_b + 1)
        *
        (max_days + 1)
    )

    forbidden_penalty = (
        unmatched_penalty * 100
    )

    cost = np.full(
        (n_a, n_b + n_a),
        float(unmatched_penalty)
    )

    cost[:, :n_b] = np.where(
        allowed,
        distance.astype(float),
        float(forbidden_penalty)
    )

    rows, cols = (
        linear_sum_assignment(cost)
    )

    matches = []

    for i, j in zip(rows, cols):

        if j >= n_b:
            continue

        if not allowed[i, j]:
            continue

        matches.append({
            "a": a.iloc[i],
            "b": b.iloc[j],
            "signed_days": int(
                signed_days[i, j]
            ),
            "absolute_days": int(
                distance[i, j]
            )
        })

    return matches


# ============================================================================
# 6. RUN ALL MATCHING SCENARIOS
# ============================================================================

pair_records = []
support_records = []

for scenario in SCENARIOS:

    eligible = get_eligible_scenes(
        fragment_wide,
        scenario
    )

    for sensor_a, sensor_b in SENSOR_PAIRS:

        pair_name = (
            f"{sensor_a}__{sensor_b}"
        )

        a = eligible.loc[
            eligible["sensor"] == sensor_a
        ]

        b = eligible.loc[
            eligible["sensor"] == sensor_b
        ]

        if len(a) and len(b):

            overlap_start = max(
                a["date"].min(),
                b["date"].min()
            )

            overlap_end = min(
                a["date"].max(),
                b["date"].max()
            )

            if overlap_start <= overlap_end:

                n_a_overlap = int(
                    a["date"].between(
                        overlap_start,
                        overlap_end
                    ).sum()
                )

                n_b_overlap = int(
                    b["date"].between(
                        overlap_start,
                        overlap_end
                    ).sum()
                )

            else:

                n_a_overlap = 0
                n_b_overlap = 0

        else:

            overlap_start = pd.NaT
            overlap_end = pd.NaT

            n_a_overlap = 0
            n_b_overlap = 0

        for max_days in WINDOWS:

            matches = (
                optimal_temporal_matching(
                    eligible,
                    sensor_a,
                    sensor_b,
                    max_days
                )
            )

            matched_years = set()
            matched_dates = []

            for number, match in enumerate(
                matches,
                start=1
            ):

                row_a = match["a"]
                row_b = match["b"]

                pair_date = (
                    row_a["date"]
                    +
                    (
                        row_b["date"]
                        -
                        row_a["date"]
                    ) / 2
                )

                matched_years.add(
                    pair_date.year
                )

                matched_dates.append(
                    match["absolute_days"]
                )

                pair_id = (
                    f"{scenario}__"
                    f"{pair_name}__"
                    f"W{max_days:02d}__"
                    f"{number:04d}"
                )

                record = {
                    "pair_id": pair_id,
                    "scenario": scenario,
                    "sensor_pair": pair_name,
                    "sensor_a": sensor_a,
                    "sensor_b": sensor_b,
                    "max_days": max_days,
                    "scene_a": row_a["scene_id"],
                    "scene_b": row_b["scene_id"],
                    "date_a": row_a["date"],
                    "date_b": row_b["date"],
                    "pair_date": pair_date,
                    "pair_year": pair_date.year,
                    "signed_days": (
                        match["signed_days"]
                    ),
                    "absolute_days": (
                        match["absolute_days"]
                    ),
                    "coverage_a": (
                        row_a[
                            "min_paired_coverage"
                        ]
                    ),
                    "coverage_b": (
                        row_b[
                            "min_paired_coverage"
                        ]
                    )
                }

                for index in INDICES:

                    for forest in [
                        "Late_secondary",
                        "Mature"
                    ]:

                        col = (
                            f"{index}_mean__"
                            f"{forest}"
                        )

                        value_a = float(
                            row_a[col]
                        )

                        value_b = float(
                            row_b[col]
                        )

                        record[
                            f"{index}_a__{forest}"
                        ] = value_a

                        record[
                            f"{index}_b__{forest}"
                        ] = value_b

                        record[
                            f"{index}_bias__{forest}"
                        ] = (
                            value_b - value_a
                        )

                    contrast_a = float(
                        row_a[
                            f"{index}_forest_contrast"
                        ]
                    )

                    contrast_b = float(
                        row_b[
                            f"{index}_forest_contrast"
                        ]
                    )

                    record[
                        f"{index}_contrast_a"
                    ] = contrast_a

                    record[
                        f"{index}_contrast_b"
                    ] = contrast_b

                    record[
                        f"{index}_contrast_change"
                    ] = (
                        contrast_b
                        -
                        contrast_a
                    )

                pair_records.append(
                    record
                )

            support_records.append({
                "scenario": scenario,
                "sensor_pair": pair_name,
                "sensor_a": sensor_a,
                "sensor_b": sensor_b,
                "max_days": max_days,
                "n_available_a": len(a),
                "n_available_b": len(b),
                "overlap_start": overlap_start,
                "overlap_end": overlap_end,
                "n_overlap_a": n_a_overlap,
                "n_overlap_b": n_b_overlap,
                "n_pairs": len(matches),
                "n_pair_years": len(
                    matched_years
                ),
                "median_separation_days": (
                    np.median(matched_dates)
                    if matched_dates
                    else np.nan
                ),
                "max_separation_days": (
                    np.max(matched_dates)
                    if matched_dates
                    else np.nan
                ),
                "n_same_day": (
                    int(
                        np.sum(
                            np.array(
                                matched_dates
                            ) == 0
                        )
                    )
                    if matched_dates
                    else 0
                )
            })

matched_pairs = pd.DataFrame(
    pair_records
)

overlap_support = pd.DataFrame(
    support_records
)

print("\n" + "=" * 76)
print("TEMPORAL MATCHING SUPPORT — ALL WINDOWS")
print("=" * 76)

for scenario in [
    "Annual_ge70",
    "MaySep_ge70",
    "Annual_ge90",
    "MaySep_ge90"
]:

    print(f"\n{scenario}")

    display(
        overlap_support.loc[
            overlap_support["scenario"]
            == scenario
        ]
        .pivot(
            index="sensor_pair",
            columns="max_days",
            values="n_pairs"
        )
    )

# ============================================================================
# 7. MATCHING QUALITY CONTROL
# ============================================================================

if not matched_pairs.empty:

    assert (
        matched_pairs["absolute_days"]
        <= matched_pairs["max_days"]
    ).all()

    for _, group in matched_pairs.groupby(
        [
            "scenario",
            "sensor_pair",
            "max_days"
        ]
    ):

        assert not group[
            "scene_a"
        ].duplicated().any()

        assert not group[
            "scene_b"
        ].duplicated().any()

    for scenario, group in (
        matched_pairs.groupby("scenario")
    ):

        threshold = (
            SCENARIOS[scenario]["coverage"]
        )

        assert (
            group["coverage_a"]
            >= threshold
        ).all()

        assert (
            group["coverage_b"]
            >= threshold
        ).all()

print("\nMATCHING QUALITY CONTROL: PASSED")

# ============================================================================
# 8. CROSS-SENSOR SPECTRAL DISCREPANCIES
#
# Bias convention:
#   sensor B minus sensor A
#
# Forest contrast:
#   Late-secondary minus mature
#
# IMPORTANT:
#   These differences combine instrumental effects,
#   temporal variability, atmospheric residuals and
#   differences in valid pixel support.
# ============================================================================

def summarize_difference(
    values
):

    values = np.asarray(
        values,
        dtype=float
    )

    values = values[
        np.isfinite(values)
    ]

    if len(values) == 0:

        return {
            "n": 0,
            "mean": np.nan,
            "median": np.nan,
            "sd": np.nan,
            "mae": np.nan,
            "rmse": np.nan,
            "mad": np.nan,
            "q025": np.nan,
            "q975": np.nan
        }

    median = np.median(
        values
    )

    return {
        "n": len(values),
        "mean": np.mean(values),
        "median": median,
        "sd": (
            np.std(values, ddof=1)
            if len(values) > 1
            else np.nan
        ),
        "mae": np.mean(
            np.abs(values)
        ),
        "rmse": np.sqrt(
            np.mean(values ** 2)
        ),
        "mad": np.median(
            np.abs(values - median)
        ),
        "q025": np.quantile(
            values, 0.025
        ),
        "q975": np.quantile(
            values, 0.975
        )
    }


bias_records = []
contrast_records = []

if not matched_pairs.empty:

    grouping = [
        "scenario",
        "sensor_pair",
        "max_days"
    ]

    for keys, group in (
        matched_pairs.groupby(grouping)
    ):

        scenario, pair_name, max_days = (
            keys
        )

        for index in INDICES:

            for forest in [
                "Late_secondary",
                "Mature"
            ]:

                values_a = group[
                    f"{index}_a__{forest}"
                ].to_numpy()

                values_b = group[
                    f"{index}_b__{forest}"
                ].to_numpy()

                difference = (
                    values_b - values_a
                )

                stats = (
                    summarize_difference(
                        difference
                    )
                )

                if (
                    len(values_a) >= 3
                    and np.std(values_a) > 0
                    and np.std(values_b) > 0
                ):

                    correlation = (
                        np.corrcoef(
                            values_a,
                            values_b
                        )[0, 1]
                    )

                else:

                    correlation = np.nan

                bias_records.append({
                    "scenario": scenario,
                    "sensor_pair": pair_name,
                    "max_days": max_days,
                    "index": index,
                    "forest_type": forest,
                    "n_pair_years": (
                        group["pair_year"]
                        .nunique()
                    ),
                    "median_separation_days": (
                        group["absolute_days"]
                        .median()
                    ),
                    "median_sensor_a": (
                        np.median(values_a)
                    ),
                    "median_sensor_b": (
                        np.median(values_b)
                    ),
                    "sensor_a_range": (
                        np.ptp(values_a)
                    ),
                    "sensor_b_range": (
                        np.ptp(values_b)
                    ),
                    "sensor_a_iqr": (
                        np.quantile(
                            values_a, 0.75
                        )
                        -
                        np.quantile(
                            values_a, 0.25
                        )
                    ),
                    "pearson_r": correlation,
                    **stats
                })

            contrast_a = group[
                f"{index}_contrast_a"
            ].to_numpy()

            contrast_b = group[
                f"{index}_contrast_b"
            ].to_numpy()

            contrast_change = (
                contrast_b
                -
                contrast_a
            )

            stats = summarize_difference(
                contrast_change
            )

            absolute_discrepancies = (
                np.concatenate([
                    group[
                        f"{index}_bias__Late_secondary"
                    ].to_numpy(),
                    group[
                        f"{index}_bias__Mature"
                    ].to_numpy()
                ])
            )

            absolute_mae = np.mean(
                np.abs(
                    absolute_discrepancies
                )
            )

            contrast_mae = stats["mae"]

            # A descriptive measure of contrast stability.
            # Not a test of statistical equivalence.

            cancellation_ratio = (
                contrast_mae / absolute_mae
                if absolute_mae > 0
                else np.nan
            )

            contrast_records.append({
                "scenario": scenario,
                "sensor_pair": pair_name,
                "max_days": max_days,
                "index": index,
                "n_pair_years": (
                    group["pair_year"]
                    .nunique()
                ),
                "median_contrast_a": (
                    np.median(contrast_a)
                ),
                "median_contrast_b": (
                    np.median(contrast_b)
                ),
                "sign_agreement_pct": (
                    100
                    *
                    np.mean(
                        np.sign(contrast_a)
                        ==
                        np.sign(contrast_b)
                    )
                ),
                "absolute_index_mae": (
                    absolute_mae
                ),
                "contrast_mae": (
                    contrast_mae
                ),
                "contrast_to_absolute_mae_ratio": (
                    cancellation_ratio
                ),
                **stats
            })

bias_summary = pd.DataFrame(
    bias_records
)

contrast_stability = pd.DataFrame(
    contrast_records
)

print("\n" + "=" * 76)
print("CROSS-SENSOR DISCREPANCIES — PRIMARY 8-DAY WINDOW")
print("=" * 76)

if not bias_summary.empty:

    primary_bias = bias_summary.loc[
        (
            bias_summary["scenario"]
            == "Annual_ge70"
        )
        &
        (
            bias_summary["max_days"]
            == PRIMARY_WINDOW
        )
    ].copy()

    display(
        primary_bias[[
            "sensor_pair",
            "index",
            "forest_type",
            "n",
            "n_pair_years",
            "median",
            "mae",
            "rmse",
            "pearson_r",
            "sensor_a_iqr"
        ]].round(5)
    )

else:

    print(
        "No matched observations available."
    )

print("\nFOREST-CONTRAST STABILITY")

if not contrast_stability.empty:

    primary_contrast = (
        contrast_stability.loc[
            (
                contrast_stability["scenario"]
                == "Annual_ge70"
            )
            &
            (
                contrast_stability["max_days"]
                == PRIMARY_WINDOW
            )
        ].copy()
    )

    display(
        primary_contrast[[
            "sensor_pair",
            "index",
            "n",
            "median_contrast_a",
            "median_contrast_b",
            "median",
            "absolute_index_mae",
            "contrast_mae",
            "contrast_to_absolute_mae_ratio",
            "sign_agreement_pct"
        ]].round(5)
    )

# ============================================================================
# 9. CALIBRATION FEASIBILITY
#
# Pair counts are counted at acquisition-pair level,
# NOT at forest-observation level.
#
# Thresholds are deliberately conservative exploratory
# criteria and do not authorize automatic correction.
# ============================================================================

feasibility_records = []

for _, row in overlap_support.iterrows():

    n_pairs = int(
        row["n_pairs"]
    )

    n_years = int(
        row["n_pair_years"]
    )

    if n_pairs == 0:

        status = (
            "NO_TEMPORAL_SUPPORT"
        )

    elif (
        n_pairs >= MIN_PAIRS_REGRESSION
        and n_years >= MIN_YEARS_REGRESSION
    ):

        status = (
            "REGRESSION_CANDIDATE"
        )

    elif (
        n_pairs >= MIN_PAIRS_OFFSET
        and n_years >= MIN_YEARS_OFFSET
    ):

        status = (
            "OFFSET_CANDIDATE"
        )

    else:

        status = (
            "DESCRIPTIVE_ONLY"
        )

    feasibility_records.append({
        **row.to_dict(),
        "calibration_status": status,
        "requires_dynamic_range_check": (
            status == "REGRESSION_CANDIDATE"
        ),
        "requires_out_of_year_validation": (
            status in [
                "REGRESSION_CANDIDATE",
                "OFFSET_CANDIDATE"
            ]
        )
    })

calibration_feasibility = pd.DataFrame(
    feasibility_records
)

print("\n" + "=" * 76)
print("CALIBRATION FEASIBILITY — 8 AND 16 DAYS")
print("=" * 76)

display(
    calibration_feasibility.loc[
        calibration_feasibility[
            "scenario"
        ].isin([
            "Annual_ge70",
            "MaySep_ge70",
            "Annual_ge90",
            "Annual_ge70_L7_until2017"
        ])
        &
        calibration_feasibility[
            "max_days"
        ].isin([8, 16]),
        [
            "scenario",
            "sensor_pair",
            "max_days",
            "n_pairs",
            "n_pair_years",
            "median_separation_days",
            "calibration_status"
        ]
    ]
)

# ============================================================================
# 10. DATE-LAG SENSITIVITY
#
# Examine whether the estimated median discrepancy
# changes when the temporal window is expanded.
#
# Each window is matched independently; therefore
# these samples are not necessarily nested.
# ============================================================================

if not bias_summary.empty:

    lag_sensitivity = (
        bias_summary.loc[
            bias_summary["scenario"]
            == "Annual_ge70"
        ]
        .pivot_table(
            index=[
                "sensor_pair",
                "index",
                "forest_type"
            ],
            columns="max_days",
            values="median"
        )
        .reset_index()
    )

    lag_sensitivity.columns = [
        (
            f"median_bias_{col}d"
            if isinstance(
                col,
                (int, np.integer)
            )
            else col
        )
        for col in lag_sensitivity.columns
    ]

    for window in WINDOWS:

        col = (
            f"median_bias_{window}d"
        )

        if col not in (
            lag_sensitivity.columns
        ):

            lag_sensitivity[col] = (
                np.nan
            )

    lag_sensitivity[
        "bias_change_16d_minus_8d"
    ] = (
        lag_sensitivity[
            "median_bias_16d"
        ]
        -
        lag_sensitivity[
            "median_bias_8d"
        ]
    )

else:

    lag_sensitivity = pd.DataFrame()

print("\nTEMPORAL-WINDOW SENSITIVITY")

display(
    lag_sensitivity.round(5)
)

# ============================================================================
# 11. SPATIAL-SCALE SENSITIVITY
#
# Apply the SAME acquisition pairs to:
#   - Complete fragments
#   - 30-m interior cores
#   - 1-ha field plots
#
# Nested spatial units are not treated as independent
# calibration replicates.
# ============================================================================

spatial_records = []

scale_mapping = {
    "Fragment": (
        "FRAG_01",
        "FRAG_02"
    ),
    "Fragment_core": (
        "FRAG_01_CORE30",
        "FRAG_02_CORE30"
    ),
    "Plot": (
        "P01",
        "P02"
    )
}

spectral_lookup = (
    screened.set_index(
        ["sensor", "scene_id", "unit_id"]
    )
)

if not matched_pairs.empty:

    spatial_pairs = (
        matched_pairs.loc[
            matched_pairs[
                "scenario"
            ].isin([
                "Annual_ge70",
                "MaySep_ge70"
            ])
            &
            (
                matched_pairs["max_days"]
                == PRIMARY_WINDOW
            )
        ].copy()
    )

    for _, pair in (
        spatial_pairs.iterrows()
    ):

        for scale, units in (
            scale_mapping.items()
        ):

            unit_secondary, unit_mature = (
                units
            )

            keys = [
                (
                    pair["sensor_a"],
                    pair["scene_a"],
                    unit_secondary
                ),
                (
                    pair["sensor_a"],
                    pair["scene_a"],
                    unit_mature
                ),
                (
                    pair["sensor_b"],
                    pair["scene_b"],
                    unit_secondary
                ),
                (
                    pair["sensor_b"],
                    pair["scene_b"],
                    unit_mature
                )
            ]

            rows = [
                spectral_lookup.loc[key]
                for key in keys
            ]

            coverage = np.array([
                row[
                    "screened_valid_fraction"
                ]
                for row in rows
            ])

            if (
                not np.isfinite(
                    coverage
                ).all()
                or np.min(coverage)
                < PRIMARY_COVERAGE
            ):

                continue

            for index in INDICES:

                values = np.array([
                    row[f"{index}_mean"]
                    for row in rows
                ])

                if not np.isfinite(
                    values
                ).all():

                    continue

                a_secondary = values[0]
                a_mature = values[1]

                b_secondary = values[2]
                b_mature = values[3]

                contrast_a = (
                    a_secondary
                    -
                    a_mature
                )

                contrast_b = (
                    b_secondary
                    -
                    b_mature
                )

                spatial_records.append({
                    "pair_id": pair["pair_id"],
                    "scenario": pair["scenario"],
                    "sensor_pair": (
                        pair["sensor_pair"]
                    ),
                    "pair_year": (
                        pair["pair_year"]
                    ),
                    "spatial_scale": scale,
                    "index": index,
                    "min_coverage": (
                        np.min(coverage)
                    ),
                    "bias_secondary": (
                        b_secondary
                        -
                        a_secondary
                    ),
                    "bias_mature": (
                        b_mature
                        -
                        a_mature
                    ),
                    "contrast_a": contrast_a,
                    "contrast_b": contrast_b,
                    "contrast_change": (
                        contrast_b
                        -
                        contrast_a
                    )
                })

spatial_sensitivity = pd.DataFrame(
    spatial_records
)

spatial_summary_records = []

if not spatial_sensitivity.empty:

    for keys, group in (
        spatial_sensitivity.groupby([
            "scenario",
            "sensor_pair",
            "spatial_scale",
            "index"
        ])
    ):

        scenario, pair_name, scale, index = (
            keys
        )

        stats = summarize_difference(
            group["contrast_change"]
        )

        spatial_summary_records.append({
            "scenario": scenario,
            "sensor_pair": pair_name,
            "spatial_scale": scale,
            "index": index,
            "n_pair_years": (
                group["pair_year"].nunique()
            ),
            "median_contrast_a": (
                group["contrast_a"].median()
            ),
            "median_contrast_b": (
                group["contrast_b"].median()
            ),
            **stats
        })

spatial_summary = pd.DataFrame(
    spatial_summary_records
)

print("\nSPATIAL-SCALE SENSITIVITY")

if not spatial_summary.empty:

    display(
        spatial_summary.loc[
            (
                spatial_summary["scenario"]
                == "Annual_ge70"
            )
            &
            spatial_summary["index"].isin([
                "NDVI",
                "EVI",
                "NDMI",
                "NBR"
            ]),
            [
                "sensor_pair",
                "spatial_scale",
                "index",
                "n",
                "median_contrast_a",
                "median_contrast_b",
                "median",
                "mae"
            ]
        ].round(5)
    )

# ============================================================================
# 12. 2019 FIELD-CENSUS CROSS-SENSOR MATCHES
# ============================================================================

if not matched_pairs.empty:

    census_matches = (
        matched_pairs.loc[
            (
                matched_pairs["sensor_pair"]
                == "Landsat_7__Landsat_8"
            )
            &
            (
                matched_pairs["scenario"]
                == "Annual_ge70"
            )
            &
            (
                matched_pairs["max_days"]
                == PRIMARY_WINDOW
            )
            &
            (
                matched_pairs["date_a"].dt.year
                == 2019
            )
            &
            (
                matched_pairs["date_b"].dt.year
                == 2019
            )
        ].copy()
    )

else:

    census_matches = pd.DataFrame()

print("\n" + "=" * 76)
print("2019 FIELD-CENSUS CROSS-SENSOR MATCHES")
print("=" * 76)

print(
    "Matched L7–L8 acquisitions:",
    len(census_matches)
)

if not census_matches.empty:

    display(
        census_matches[[
            "date_a",
            "date_b",
            "absolute_days",
            "coverage_a",
            "coverage_b",
            "NDVI_contrast_a",
            "NDVI_contrast_b",
            "EVI_contrast_a",
            "EVI_contrast_b",
            "NDMI_contrast_a",
            "NDMI_contrast_b",
            "NBR_contrast_a",
            "NBR_contrast_b"
        ]].round(5)
    )

# ============================================================================
# 13. FIGURE — TEMPORAL MATCHING AND CALIBRATION FEASIBILITY
# ============================================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "savefig.dpi": 600,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

pair_labels = {
    "Landsat_5__Landsat_7": "L5–L7",
    "Landsat_7__Landsat_8": "L7–L8",
    "Landsat_8__Landsat_9": "L8–L9"
}

pair_colors = {
    "Landsat_5__Landsat_7": "#527B98",
    "Landsat_7__Landsat_8": "#DB9140",
    "Landsat_8__Landsat_9": "#24634F"
}

fig, axes = plt.subplots(
    2,
    2,
    figsize=(14, 10),
    constrained_layout=True
)

# ---------------------------------------------------------------------------
# Panel A — Pair availability
# ---------------------------------------------------------------------------

ax = axes[0, 0]

for pair_name, label in (
    pair_labels.items()
):

    subset = overlap_support.loc[
        (
            overlap_support["scenario"]
            == "Annual_ge70"
        )
        &
        (
            overlap_support["sensor_pair"]
            == pair_name
        )
    ].sort_values("max_days")

    ax.plot(
        subset["max_days"],
        subset["n_pairs"],
        marker="o",
        linewidth=2,
        markersize=6,
        color=pair_colors[pair_name],
        label=label
    )

ax.axhline(
    MIN_PAIRS_OFFSET,
    color="#888888",
    linestyle=":",
    linewidth=1
)

ax.axhline(
    MIN_PAIRS_REGRESSION,
    color="#888888",
    linestyle="--",
    linewidth=1
)

ax.set_xticks(WINDOWS)

ax.set_xlabel(
    "Maximum acquisition separation (days)"
)

ax.set_ylabel(
    "One-to-one acquisition pairs"
)

ax.set_title(
    "(a) Temporal matching support",
    loc="left",
    fontweight="bold"
)

ax.legend(
    frameon=False
)

ax.grid(
    alpha=0.15
)

# ---------------------------------------------------------------------------
# Panel B — Temporal distribution of matches
# ---------------------------------------------------------------------------

ax = axes[0, 1]

if not matched_pairs.empty:

    plot_pairs = matched_pairs.loc[
        (
            matched_pairs["scenario"]
            == "Annual_ge70"
        )
        &
        (
            matched_pairs["max_days"]
            == PRIMARY_WINDOW
        )
    ]

    for pair_name, label in (
        pair_labels.items()
    ):

        subset = plot_pairs.loc[
            plot_pairs["sensor_pair"]
            == pair_name
        ]

        ax.scatter(
            subset["pair_date"],
            subset["absolute_days"],
            color=pair_colors[pair_name],
            s=35,
            alpha=0.8,
            label=label
        )

    if plot_pairs.empty:

        ax.text(
            0.5,
            0.5,
            "No matches within 8 days",
            transform=ax.transAxes,
            ha="center",
            va="center"
        )

else:

    ax.text(
        0.5,
        0.5,
        "No eligible matches",
        transform=ax.transAxes,
        ha="center",
        va="center"
    )

ax.axvline(
    pd.Timestamp("2019-01-01"),
    color="#C84C4C",
    linestyle="--",
    linewidth=1
)

ax.set_ylabel(
    "Acquisition separation (days)"
)

ax.set_xlabel(
    "Matched acquisition date"
)

ax.set_ylim(
    -0.5,
    PRIMARY_WINDOW + 0.5
)

ax.xaxis.set_major_locator(
    mdates.YearLocator(4)
)

ax.xaxis.set_major_formatter(
    mdates.DateFormatter("%Y")
)

ax.set_title(
    "(b) Distribution of matched acquisitions",
    loc="left",
    fontweight="bold"
)

ax.grid(
    alpha=0.15
)

# ---------------------------------------------------------------------------
# Panel C — Median cross-sensor discrepancy
# ---------------------------------------------------------------------------

ax = axes[1, 0]

positions = np.arange(
    len(INDICES)
)

if not bias_summary.empty:

    for p, (
        pair_name,
        label
    ) in enumerate(
        pair_labels.items()
    ):

        for forest, marker, offset in [
            (
                "Late_secondary",
                "o",
                -0.035
            ),
            (
                "Mature",
                "^",
                0.035
            )
        ]:

            subset = (
                bias_summary.loc[
                    (
                        bias_summary["scenario"]
                        == "Annual_ge70"
                    )
                    &
                    (
                        bias_summary["max_days"]
                        == PRIMARY_WINDOW
                    )
                    &
                    (
                        bias_summary["sensor_pair"]
                        == pair_name
                    )
                    &
                    (
                        bias_summary["forest_type"]
                        == forest
                    )
                ]
                .set_index("index")
                .reindex(INDICES)
            )

            ax.scatter(
                positions
                +
                (p - 1) * 0.22
                +
                offset,
                subset["median"],
                marker=marker,
                s=50,
                color=pair_colors[pair_name],
                label=(
                    f"{label} — "
                    f"{forest.replace('_', ' ')}"
                )
            )

ax.axhline(
    0,
    color="#666666",
    linestyle="--",
    linewidth=1
)

ax.set_xticks(
    positions,
    INDICES
)

ax.set_ylabel(
    "Median difference (sensor B − sensor A)"
)

ax.set_title(
    "(c) Cross-sensor spectral discrepancies",
    loc="left",
    fontweight="bold"
)

if not bias_summary.empty:

    ax.legend(
        frameon=False,
        fontsize=7,
        ncol=2
    )

ax.grid(
    alpha=0.15,
    axis="y"
)

# ---------------------------------------------------------------------------
# Panel D — Absolute-index versus forest-contrast MAE
# ---------------------------------------------------------------------------

ax = axes[1, 1]

if not contrast_stability.empty:

    subset = contrast_stability.loc[
        (
            contrast_stability["scenario"]
            == "Annual_ge70"
        )
        &
        (
            contrast_stability["max_days"]
            == PRIMARY_WINDOW
        )
    ].copy()

    for pair_name, label in (
        pair_labels.items()
    ):

        group = subset.loc[
            subset["sensor_pair"]
            == pair_name
        ]

        ax.scatter(
            group["absolute_index_mae"],
            group["contrast_mae"],
            s=55,
            color=pair_colors[pair_name],
            label=label,
            alpha=0.85
        )

        for _, row in group.iterrows():

            ax.annotate(
                row["index"],
                (
                    row["absolute_index_mae"],
                    row["contrast_mae"]
                ),
                xytext=(4, 4),
                textcoords="offset points",
                fontsize=8
            )

    if not subset.empty:

        limit = max(
            subset[
                "absolute_index_mae"
            ].max(),
            subset[
                "contrast_mae"
            ].max()
        ) * 1.15

        limit = max(
            limit,
            0.01
        )

        ax.plot(
            [0, limit],
            [0, limit],
            color="#888888",
            linestyle="--",
            linewidth=1
        )

        ax.set_xlim(
            0, limit
        )

        ax.set_ylim(
            0, limit
        )

        ax.legend(
            frameon=False
        )

ax.set_xlabel(
    "Absolute-index cross-sensor MAE"
)

ax.set_ylabel(
    "Forest-contrast cross-sensor MAE"
)

ax.set_title(
    "(d) Stability of forest contrasts",
    loc="left",
    fontweight="bold"
)

ax.grid(
    alpha=0.15
)

fig.suptitle(
    "Landsat cross-sensor temporal overlap "
    "and calibration feasibility",
    fontsize=16,
    fontweight="bold"
)

FIGURE_STEM = (
    FIGURE_DIR
    /
    "FIG_LANDSAT_CROSS_SENSOR_OVERLAP"
)

for extension in [
    "png",
    "pdf",
    "svg"
]:

    fig.savefig(
        FIGURE_STEM.with_suffix(
            f".{extension}"
        ),
        bbox_inches="tight",
        facecolor="white"
    )

plt.show()

# ============================================================================
# 14. FIGURE — NEAR-COINCIDENT SPECTRAL COMPARISONS
# ============================================================================

fig, axes = plt.subplots(
    3,
    5,
    figsize=(19, 11),
    constrained_layout=True
)

for r, (
    sensor_a,
    sensor_b
) in enumerate(SENSOR_PAIRS):

    pair_name = (
        f"{sensor_a}__{sensor_b}"
    )

    if not matched_pairs.empty:

        subset = matched_pairs.loc[
            (
                matched_pairs["scenario"]
                == "Annual_ge70"
            )
            &
            (
                matched_pairs["sensor_pair"]
                == pair_name
            )
            &
            (
                matched_pairs["max_days"]
                == PRIMARY_WINDOW
            )
        ]

    else:

        subset = pd.DataFrame()

    for c, index in enumerate(
        INDICES
    ):

        ax = axes[r, c]

        if not subset.empty:

            all_values = []

            for forest, marker in [
                ("Late_secondary", "o"),
                ("Mature", "^")
            ]:

                x = subset[
                    f"{index}_a__{forest}"
                ].to_numpy()

                y = subset[
                    f"{index}_b__{forest}"
                ].to_numpy()

                all_values.extend(x)
                all_values.extend(y)

                ax.scatter(
                    x,
                    y,
                    s=32,
                    alpha=0.8,
                    marker=marker,
                    color=FOREST_COLORS[
                        forest
                    ],
                    label=forest.replace(
                        "_", " "
                    )
                )

            lower = min(all_values)
            upper = max(all_values)

            padding = max(
                (upper - lower) * 0.08,
                0.015
            )

            lower -= padding
            upper += padding

            ax.plot(
                [lower, upper],
                [lower, upper],
                linestyle="--",
                color="#888888",
                linewidth=1
            )

            ax.set_xlim(
                lower, upper
            )

            ax.set_ylim(
                lower, upper
            )

            ax.set_aspect(
                "equal",
                adjustable="box"
            )

        else:

            ax.text(
                0.5,
                0.5,
                "No matched pairs",
                transform=ax.transAxes,
                ha="center",
                va="center",
                fontsize=9
            )

        ax.set_title(
            f"{pair_labels[pair_name]} | {index}",
            fontweight="bold",
            fontsize=10
        )

        ax.text(
            0.04,
            0.94,
            f"n = {len(subset)} pairs",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=8
        )

        ax.set_xlabel(
            sensor_a.replace("_", " ")
        )

        if c == 0:

            ax.set_ylabel(
                sensor_b.replace("_", " ")
            )

        ax.grid(
            alpha=0.12
        )

from matplotlib.lines import Line2D

legend_handles = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="None",
        color=FOREST_COLORS[
            "Late_secondary"
        ],
        label="Late-secondary forest"
    ),
    Line2D(
        [0], [0],
        marker="^",
        linestyle="None",
        color=FOREST_COLORS[
            "Mature"
        ],
        label="Mature forest"
    )
]

fig.legend(
    handles=legend_handles,
    loc="outside lower center",
    ncol=2,
    frameon=False
)

fig.suptitle(
    "Near-coincident Landsat spectral observations\n"
    "Maximum separation: 8 days | "
    "Paired coverage ≥70%",
    fontsize=15,
    fontweight="bold"
)

FIGURE_STEM_2 = (
    FIGURE_DIR
    /
    "FIG_LANDSAT_NEAR_COINCIDENT_SPECTRA"
)

for extension in [
    "png",
    "pdf",
    "svg"
]:

    fig.savefig(
        FIGURE_STEM_2.with_suffix(
            f".{extension}"
        ),
        bbox_inches="tight",
        facecolor="white"
    )

plt.show()

# ============================================================================
# 15. EXPORT ALL RESULTS
# ============================================================================

outputs = {
    "LANDSAT_FRAGMENT_SCENE_TABLE.csv": (
        fragment_wide
    ),
    "LANDSAT_CALIBRATION_ELIGIBILITY.csv": (
        eligibility_summary
    ),
    "LANDSAT_TEMPORAL_MATCHED_PAIRS.csv": (
        matched_pairs
    ),
    "LANDSAT_TEMPORAL_OVERLAP_SUPPORT.csv": (
        overlap_support
    ),
    "LANDSAT_CROSS_SENSOR_BIAS.csv": (
        bias_summary
    ),
    "LANDSAT_FOREST_CONTRAST_STABILITY.csv": (
        contrast_stability
    ),
    "LANDSAT_CALIBRATION_FEASIBILITY.csv": (
        calibration_feasibility
    ),
    "LANDSAT_TEMPORAL_LAG_SENSITIVITY.csv": (
        lag_sensitivity
    ),
    "LANDSAT_SPATIAL_MATCH_SENSITIVITY.csv": (
        spatial_sensitivity
    ),
    "LANDSAT_SPATIAL_MATCH_SUMMARY.csv": (
        spatial_summary
    ),
    "LANDSAT_2019_CROSS_SENSOR_MATCHES.csv": (
        census_matches
    )
}

for filename, dataframe in (
    outputs.items()
):

    dataframe.to_csv(
        HARM_DIR / filename,
        index=False,
        encoding="utf-8-sig"
    )

workbook_path = (
    TABLE_DIR
    /
    "SATIPO_LANDSAT_CROSS_SENSOR_OVERLAP.xlsx"
)

with pd.ExcelWriter(
    workbook_path,
    engine="openpyxl"
) as writer:

    fragment_wide.to_excel(
        writer,
        sheet_name="Fragment_scenes",
        index=False
    )

    eligibility_summary.to_excel(
        writer,
        sheet_name="Eligibility",
        index=False
    )

    matched_pairs.to_excel(
        writer,
        sheet_name="Matched_pairs",
        index=False
    )

    overlap_support.to_excel(
        writer,
        sheet_name="Temporal_support",
        index=False
    )

    bias_summary.to_excel(
        writer,
        sheet_name="Spectral_discrepancy",
        index=False
    )

    contrast_stability.to_excel(
        writer,
        sheet_name="Contrast_stability",
        index=False
    )

    calibration_feasibility.to_excel(
        writer,
        sheet_name="Calibration_feasibility",
        index=False
    )

    lag_sensitivity.to_excel(
        writer,
        sheet_name="Lag_sensitivity",
        index=False
    )

    spatial_summary.to_excel(
        writer,
        sheet_name="Spatial_sensitivity",
        index=False
    )

    census_matches.to_excel(
        writer,
        sheet_name="Census_2019",
        index=False
    )

# ============================================================================
# 16. CONFIGURATION MANIFEST
# ============================================================================

manifest = {
    "project": "SATIPO_FOREST_PROJECT",
    "cell": "91A",
    "analysis": (
        "Cross-sensor temporal overlap "
        "and calibration feasibility"
    ),
    "input": str(SCREENED_PATH),
    "n_input_rows": int(
        len(screened)
    ),
    "n_input_scenes": int(
        screened["scene_id"].nunique()
    ),
    "sensor_pairs": SENSOR_PAIRS,
    "indices": INDICES,
    "temporal_windows_days": WINDOWS,
    "primary_window_days": PRIMARY_WINDOW,
    "scenarios": SCENARIOS,
    "matching_method": (
        "Maximum-cardinality minimum-total-distance "
        "one-to-one assignment"
    ),
    "calibration_spatial_units": [
        "FRAG_01",
        "FRAG_02"
    ],
    "nested_spatial_units_used_as_replicates": False,
    "offset_candidate_min_pairs": (
        MIN_PAIRS_OFFSET
    ),
    "offset_candidate_min_years": (
        MIN_YEARS_OFFSET
    ),
    "regression_candidate_min_pairs": (
        MIN_PAIRS_REGRESSION
    ),
    "regression_candidate_min_years": (
        MIN_YEARS_REGRESSION
    ),
    "bias_definition": (
        "Sensor B minus sensor A"
    ),
    "forest_contrast_definition": (
        "Late-secondary minus mature"
    ),
    "harmonization_applied": False,
    "original_screened_master_preserved": True,
    "outputs": list(outputs.keys()),
    "workbook": str(workbook_path),
    "limitations": [
        "Near-coincident differences are not pure sensor effects.",
        "One-to-one matching does not guarantee temporal independence.",
        "Calibration thresholds are exploratory.",
        "Fragment, core and plot observations are spatially nested.",
        "Local calibration requires out-of-year validation.",
        "Valid pixel support can differ between acquisitions.",
        "Sixteen-day matches are sensitivity analyses."
    ]
}

manifest_path = (
    HARM_DIR
    /
    "LANDSAT_OVERLAP_MANIFEST.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

# ============================================================================
# 17. FINAL REPORT
# ============================================================================

print("\n" + "=" * 76)
print("SATIPO — CROSS-SENSOR OVERLAP REPORT")
print("=" * 76)

print(
    "Screened scenes:",
    screened["scene_id"].nunique()
)

print(
    "Scene-unit observations:",
    len(screened)
)

print(
    "Matching scenarios:",
    len(SCENARIOS)
)

print(
    "Temporal windows:",
    WINDOWS
)

print(
    "Sensor connections:",
    len(SENSOR_PAIRS)
)

print(
    "Matching configurations:",
    len(overlap_support)
)

print(
    "Total matched-pair records:",
    len(matched_pairs)
)

print(
    "2019 matched L7–L8 acquisitions:",
    len(census_matches)
)

print("\nPRIMARY MATCHING SUPPORT")

final_support = (
    calibration_feasibility.loc[
        (
            calibration_feasibility["scenario"]
            == "Annual_ge70"
        )
        &
        (
            calibration_feasibility["max_days"]
            == PRIMARY_WINDOW
        ),
        [
            "sensor_pair",
            "n_pairs",
            "n_pair_years",
            "median_separation_days",
            "calibration_status"
        ]
    ]
)

display(final_support)

print("\nSEASONAL MATCHING SUPPORT")

display(
    calibration_feasibility.loc[
        (
            calibration_feasibility["scenario"]
            == "MaySep_ge70"
        )
        &
        (
            calibration_feasibility["max_days"]
            == PRIMARY_WINDOW
        ),
        [
            "sensor_pair",
            "n_pairs",
            "n_pair_years",
            "median_separation_days",
            "calibration_status"
        ]
    ]
)

print("\nOriginal screened master: PRESERVED")
print("Harmonization coefficients: NOT APPLIED")
print("Local calibration: PENDING VALIDATION")

print("\nOutput directory:")
print(HARM_DIR)

print("\nWorkbook:")
print(workbook_path)

print("\nManifest:")
print(manifest_path)

print("\nFigures:")
print(FIGURE_STEM)
print(FIGURE_STEM_2)

print("\n" + "=" * 76)
print("CELL 91A COMPLETED")
print("NEXT: CELL 91B — VALIDATED CROSS-SENSOR CALIBRATION")
print("=" * 76)


In [ ]:

# ============================================================================
# CELL 91B — VALIDATED CROSS-SENSOR CALIBRATION
# SATIPO FOREST PROJECT
#
# Model candidates:
#   1. Identity
#   2. Robust additive offset
#   3. Robust Theil-Sen regression
#
# Validation:
#   Leave-one-year-out cross-validation
#
# Important:
#   One common transformation for both forest fragments.
#   No automatic modification of historical observations.
# ============================================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import theilslopes
from IPython.display import display

# ============================================================================
# 1. CONFIGURATION
# ============================================================================

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

REMOTE_DIR = ROOT / "04_REMOTE_SENSING"

HARM_DIR = (
    REMOTE_DIR
    / "03_LANDSAT"
    / "04_CROSS_SENSOR_HARMONIZATION"
)

CAL_DIR = HARM_DIR / "VALIDATED_CALIBRATION"

TABLE_DIR = (
    REMOTE_DIR / "08_OUTPUTS" / "TABLES"
)

FIGURE_DIR = (
    REMOTE_DIR / "08_OUTPUTS" / "FIGURES"
)

for directory in [
    CAL_DIR,
    TABLE_DIR,
    FIGURE_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

PAIR_PATH = (
    HARM_DIR
    / "LANDSAT_TEMPORAL_MATCHED_PAIRS.csv"
)

assert PAIR_PATH.exists(), (
    f"Missing matched-pair table: {PAIR_PATH}"
)

INDICES = [
    "NDVI",
    "EVI",
    "NDMI",
    "NBR",
    "NBR2"
]

FORESTS = [
    "Late_secondary",
    "Mature"
]

SENSOR_PAIRS = [
    "Landsat_5__Landsat_7",
    "Landsat_7__Landsat_8",
    "Landsat_8__Landsat_9"
]

SCENARIOS = [
    "Annual_ge70",
    "MaySep_ge70",
    "Annual_ge90",
    "MaySep_ge90",
    "JunSep_ge70",
    "Annual_ge70_L7_until2017"
]

MODELS = [
    "Identity",
    "Median_offset",
    "Theil_Sen"
]

PRIMARY_SCENARIO = "Annual_ge70"

PRIMARY_WINDOW = 8

MIN_PAIRS_OFFSET = 10
MIN_YEARS_OFFSET = 3

MIN_PAIRS_REGRESSION = 20
MIN_YEARS_REGRESSION = 4

# Minimum support within each training fold.
MIN_TRAIN_PAIRS_OFFSET = 6
MIN_TRAIN_YEARS_OFFSET = 2

MIN_TRAIN_PAIRS_REGRESSION = 12
MIN_TRAIN_YEARS_REGRESSION = 3

# Predefined conservative model-selection criteria.
MIN_MAE_IMPROVEMENT_PCT = 10.0
MIN_RMSE_IMPROVEMENT_PCT = 0.0

# Regression must not distort forest contrasts excessively.
MAX_CONTRAST_MAE_INCREASE_PCT = 10.0

# Exploratory plausible slope interval.
MIN_SLOPE = 0.5
MAX_SLOPE = 1.5

# Avoid fitting slopes from almost invariant observations.
MIN_SOURCE_IQR = 0.03

# A correction must improve at least this proportion
# of held-out acquisition years.
MIN_YEAR_WIN_FRACTION = 0.60

# Require sensitivity support before recommending
# a historical correction.
MIN_SENSITIVITY_SCENARIOS = 2
MIN_SENSITIVITY_PASS_FRACTION = 0.60

# Offset stability across alternative scenarios.
MAX_OFFSET_DRIFT = 0.03

SEED = 20261115

print("=" * 76)
print("SATIPO — VALIDATED CROSS-SENSOR CALIBRATION")
print("=" * 76)

# ============================================================================
# 2. LOAD MATCHED ACQUISITIONS
# ============================================================================

pairs = pd.read_csv(
    PAIR_PATH
)

required_columns = [
    "pair_id",
    "scenario",
    "sensor_pair",
    "sensor_a",
    "sensor_b",
    "max_days",
    "pair_year",
    "date_a",
    "date_b",
    "absolute_days",
    "coverage_a",
    "coverage_b"
]

required_columns += [
    f"{index}_{side}__{forest}"
    for index in INDICES
    for side in ["a", "b"]
    for forest in FORESTS
]

missing = [
    col
    for col in required_columns
    if col not in pairs.columns
]

assert not missing, (
    f"Missing columns: {missing}"
)

pairs["date_a"] = pd.to_datetime(
    pairs["date_a"]
)

pairs["date_b"] = pd.to_datetime(
    pairs["date_b"]
)

pairs["pair_year"] = (
    pairs["pair_year"].astype(int)
)

pairs["max_days"] = (
    pairs["max_days"].astype(int)
)

pairs = pairs.loc[
    (
        pairs["max_days"]
        == PRIMARY_WINDOW
    )
    &
    pairs["scenario"].isin(
        SCENARIOS
    )
].copy()

assert not pairs.empty

assert not pairs[
    "pair_id"
].duplicated().any()

assert (
    pairs["absolute_days"]
    <= PRIMARY_WINDOW
).all()

print("\nMATCHED DATA LOADED")

display(
    pairs.groupby([
        "scenario",
        "sensor_pair"
    ])
    .agg(
        n_pairs=("pair_id", "size"),
        n_years=("pair_year", "nunique"),
        median_lag=("absolute_days", "median")
    )
    .reset_index()
)

# ============================================================================
# 3. PREPARE MODEL DATA
#
# Two observations per acquisition pair:
#   secondary and mature.
#
# Both observations always belong to the same
# training or validation fold.
#
# Calibration direction:
#   L5 -> L7
#   L7 -> L8
#   L9 -> L8
#
# L9 is reversed because Landsat 8 is the
# intended modern reference.
# ============================================================================

def get_direction(pair_name):

    if pair_name == "Landsat_8__Landsat_9":

        return {
            "source_sensor": "Landsat_9",
            "target_sensor": "Landsat_8",
            "source_side": "b",
            "target_side": "a"
        }

    sensor_a, sensor_b = (
        pair_name.split("__")
    )

    return {
        "source_sensor": sensor_a,
        "target_sensor": sensor_b,
        "source_side": "a",
        "target_side": "b"
    }


def prepare_calibration_data(
    group,
    index,
    pair_name
):

    direction = get_direction(
        pair_name
    )

    source_side = (
        direction["source_side"]
    )

    target_side = (
        direction["target_side"]
    )

    data = group[[
        "pair_id",
        "pair_year",
        "date_a",
        "date_b",
        "absolute_days",
        "coverage_a",
        "coverage_b"
    ]].copy()

    for forest in FORESTS:

        data[f"x_{forest}"] = (
            pd.to_numeric(
                group[
                    f"{index}_{source_side}__{forest}"
                ],
                errors="coerce"
            ).to_numpy()
        )

        data[f"y_{forest}"] = (
            pd.to_numeric(
                group[
                    f"{index}_{target_side}__{forest}"
                ],
                errors="coerce"
            ).to_numpy()
        )

    value_cols = [
        f"{side}_{forest}"
        for side in ["x", "y"]
        for forest in FORESTS
    ]

    finite = np.isfinite(
        data[value_cols].to_numpy(
            dtype=float
        )
    ).all(axis=1)

    data = data.loc[
        finite
    ].reset_index(drop=True)

    return data


def stack_xy(data):

    x = data[[
        f"x_{forest}"
        for forest in FORESTS
    ]].to_numpy(
        dtype=float
    ).ravel()

    y = data[[
        f"y_{forest}"
        for forest in FORESTS
    ]].to_numpy(
        dtype=float
    ).ravel()

    return x, y


def source_iqr(data):

    x, _ = stack_xy(data)

    if len(x) == 0:

        return np.nan

    return float(
        np.quantile(x, 0.75)
        -
        np.quantile(x, 0.25)
    )


# ============================================================================
# 4. MODEL FITTING
#
# All coefficients are shared by both forests.
#
# Theil-Sen is robust to some outliers but is not
# assumed to recover a pure instrumental effect.
# ============================================================================

def fit_calibration(
    train,
    model,
    training_fold=True
):

    n_pairs = len(train)

    n_years = (
        train["pair_year"]
        .nunique()
    )

    x, y = stack_xy(train)

    if model == "Identity":

        return {
            "valid": True,
            "intercept": 0.0,
            "slope": 1.0,
            "reason": "Identity_reference"
        }

    if model == "Median_offset":

        min_pairs = (
            MIN_TRAIN_PAIRS_OFFSET
            if training_fold
            else MIN_PAIRS_OFFSET
        )

        min_years = (
            MIN_TRAIN_YEARS_OFFSET
            if training_fold
            else MIN_YEARS_OFFSET
        )

        if (
            n_pairs < min_pairs
            or n_years < min_years
        ):

            return {
                "valid": False,
                "reason": "Insufficient_offset_support"
            }

        # Each acquisition contributes equally:
        # average its two forest discrepancies,
        # then take the median across acquisitions.

        pair_offsets = (
            (
                train["y_Late_secondary"]
                -
                train["x_Late_secondary"]
            )
            +
            (
                train["y_Mature"]
                -
                train["x_Mature"]
            )
        ) / 2

        intercept = float(
            np.median(pair_offsets)
        )

        return {
            "valid": True,
            "intercept": intercept,
            "slope": 1.0,
            "reason": "OK"
        }

    if model == "Theil_Sen":

        min_pairs = (
            MIN_TRAIN_PAIRS_REGRESSION
            if training_fold
            else MIN_PAIRS_REGRESSION
        )

        min_years = (
            MIN_TRAIN_YEARS_REGRESSION
            if training_fold
            else MIN_YEARS_REGRESSION
        )

        if (
            n_pairs < min_pairs
            or n_years < min_years
        ):

            return {
                "valid": False,
                "reason": "Insufficient_regression_support"
            }

        if (
            source_iqr(train)
            < MIN_SOURCE_IQR
        ):

            return {
                "valid": False,
                "reason": "Insufficient_dynamic_range"
            }

        if len(np.unique(x)) < 3:

            return {
                "valid": False,
                "reason": "Insufficient_unique_values"
            }

        try:

            fit = theilslopes(
                y,
                x,
                method="joint"
            )

            slope = float(
                fit.slope
            )

            intercept = float(
                fit.intercept
            )

        except Exception as exc:

            return {
                "valid": False,
                "reason": (
                    f"Fit_error: {exc}"
                )
            }

        if not np.isfinite(
            [intercept, slope]
        ).all():

            return {
                "valid": False,
                "reason": "Nonfinite_coefficients"
            }

        if not (
            MIN_SLOPE
            <= slope
            <= MAX_SLOPE
        ):

            return {
                "valid": False,
                "reason": "Implausible_slope",
                "intercept": intercept,
                "slope": slope
            }

        return {
            "valid": True,
            "intercept": intercept,
            "slope": slope,
            "reason": "OK"
        }

    raise ValueError(
        f"Unknown model: {model}"
    )


# ============================================================================
# 5. VALIDATION METRICS
# ============================================================================

def calculate_metrics(
    observed,
    predicted
):

    observed = np.asarray(
        observed,
        dtype=float
    )

    predicted = np.asarray(
        predicted,
        dtype=float
    )

    error = (
        predicted - observed
    )

    return {
        "mae": float(
            np.mean(
                np.abs(error)
            )
        ),
        "rmse": float(
            np.sqrt(
                np.mean(error ** 2)
            )
        ),
        "median_error": float(
            np.median(error)
        ),
        "mean_error": float(
            np.mean(error)
        )
    }


def improvement_pct(
    baseline,
    candidate
):

    if (
        not np.isfinite(baseline)
        or baseline <= 1e-12
    ):

        return np.nan

    return float(
        100
        *
        (
            baseline - candidate
        )
        / baseline
    )


def normalized_range_violations(
    values,
    index
):

    # EVI is not constrained to [-1, 1]
    # by its mathematical definition.

    if index == "EVI":

        return 0

    values = np.asarray(
        values,
        dtype=float
    )

    return int(
        np.sum(
            (values < -1)
            |
            (values > 1)
        )
    )


# ============================================================================
# 6. LEAVE-ONE-YEAR-OUT VALIDATION
#
# Each year is held out in its entirety.
# No scene or forest observation from that year
# is used to fit the candidate coefficients.
#
# If a candidate fails in ANY fold, it is not
# accepted on a favorable subset of years.
# ============================================================================

def validate_model(
    data,
    model,
    index
):

    full_fit = fit_calibration(
        data,
        model,
        training_fold=False
    )

    if not full_fit["valid"]:

        return {
            "status": "NOT_ELIGIBLE",
            "reason": full_fit["reason"],
            "full_fit": full_fit,
            "predictions": pd.DataFrame(),
            "folds": pd.DataFrame()
        }

    years = sorted(
        data["pair_year"].unique()
    )

    if len(years) < 2:

        return {
            "status": "NOT_ELIGIBLE",
            "reason": "Fewer_than_two_years",
            "full_fit": full_fit,
            "predictions": pd.DataFrame(),
            "folds": pd.DataFrame()
        }

    prediction_records = []
    fold_records = []

    for test_year in years:

        train = data.loc[
            data["pair_year"]
            != test_year
        ].copy()

        test = data.loc[
            data["pair_year"]
            == test_year
        ].copy()

        fit = fit_calibration(
            train,
            model,
            training_fold=True
        )

        fold_record = {
            "test_year": int(test_year),
            "n_train_pairs": len(train),
            "n_test_pairs": len(test),
            "n_train_years": (
                train["pair_year"].nunique()
            ),
            "source_iqr": source_iqr(train),
            "valid": fit["valid"],
            "reason": fit["reason"],
            "intercept": fit.get(
                "intercept",
                np.nan
            ),
            "slope": fit.get(
                "slope",
                np.nan
            )
        }

        if not fit["valid"]:

            fold_records.append(
                fold_record
            )

            continue

        intercept = fit[
            "intercept"
        ]

        slope = fit[
            "slope"
        ]

        x, y = stack_xy(test)

        yhat = (
            intercept + slope * x
        )

        metrics = calculate_metrics(
            y,
            yhat
        )

        identity_metrics = (
            calculate_metrics(
                y,
                x
            )
        )

        fold_record.update({
            "mae": metrics["mae"],
            "rmse": metrics["rmse"],
            "identity_mae": (
                identity_metrics["mae"]
            ),
            "mae_improvement_pct": (
                improvement_pct(
                    identity_metrics["mae"],
                    metrics["mae"]
                )
            )
        })

        fold_records.append(
            fold_record
        )

        for _, row in test.iterrows():

            for forest in FORESTS:

                x_value = float(
                    row[f"x_{forest}"]
                )

                y_value = float(
                    row[f"y_{forest}"]
                )

                prediction = (
                    intercept
                    +
                    slope * x_value
                )

                prediction_records.append({
                    "pair_id": row["pair_id"],
                    "pair_year": int(
                        row["pair_year"]
                    ),
                    "forest_type": forest,
                    "observed_source": x_value,
                    "observed_target": y_value,
                    "predicted_target": prediction,
                    "identity_error": (
                        x_value - y_value
                    ),
                    "prediction_error": (
                        prediction - y_value
                    ),
                    "intercept": intercept,
                    "slope": slope,
                    "absolute_days": (
                        row["absolute_days"]
                    )
                })

    folds = pd.DataFrame(
        fold_records
    )

    predictions = pd.DataFrame(
        prediction_records
    )

    all_folds_valid = bool(
        folds["valid"].all()
    )

    expected_predictions = (
        len(data) * len(FORESTS)
    )

    complete_predictions = (
        len(predictions)
        == expected_predictions
    )

    if not (
        all_folds_valid
        and complete_predictions
    ):

        return {
            "status": "FAILED_FOLD",
            "reason": (
                "At_least_one_invalid_training_fold"
            ),
            "full_fit": full_fit,
            "predictions": predictions,
            "folds": folds
        }

    assert not predictions.duplicated(
        ["pair_id", "forest_type"]
    ).any()

    observed = predictions[
        "observed_target"
    ].to_numpy()

    predicted = predictions[
        "predicted_target"
    ].to_numpy()

    source = predictions[
        "observed_source"
    ].to_numpy()

    metrics = calculate_metrics(
        observed,
        predicted
    )

    identity_metrics = (
        calculate_metrics(
            observed,
            source
        )
    )

    # Forest contrasts calculated on the same
    # held-out acquisition pairs.

    wide = predictions.pivot(
        index="pair_id",
        columns="forest_type",
        values=[
            "observed_source",
            "observed_target",
            "predicted_target"
        ]
    )

    source_contrast = (
        wide[
            (
                "observed_source",
                "Late_secondary"
            )
        ]
        -
        wide[
            (
                "observed_source",
                "Mature"
            )
        ]
    ).to_numpy()

    target_contrast = (
        wide[
            (
                "observed_target",
                "Late_secondary"
            )
        ]
        -
        wide[
            (
                "observed_target",
                "Mature"
            )
        ]
    ).to_numpy()

    predicted_contrast = (
        wide[
            (
                "predicted_target",
                "Late_secondary"
            )
        ]
        -
        wide[
            (
                "predicted_target",
                "Mature"
            )
        ]
    ).to_numpy()

    contrast_metrics = (
        calculate_metrics(
            target_contrast,
            predicted_contrast
        )
    )

    identity_contrast_metrics = (
        calculate_metrics(
            target_contrast,
            source_contrast
        )
    )

    # Equal-year metrics prevent years with more
    # acquisitions from dominating model selection.

    year_mae = float(
        folds["mae"].mean()
    )

    identity_year_mae = float(
        folds["identity_mae"].mean()
    )

    year_win_fraction = float(
        np.mean(
            folds["mae"]
            <
            folds["identity_mae"]
        )
    )

    range_violations = (
        normalized_range_violations(
            predicted,
            index
        )
    )

    return {
        "status": "VALIDATED",
        "reason": "OK",
        "full_fit": full_fit,
        "predictions": predictions,
        "folds": folds,
        "metrics": {
            "n_predictions": len(predictions),
            "n_valid_years": len(folds),
            "cv_mae": metrics["mae"],
            "cv_rmse": metrics["rmse"],
            "cv_median_error": (
                metrics["median_error"]
            ),
            "identity_mae": (
                identity_metrics["mae"]
            ),
            "identity_rmse": (
                identity_metrics["rmse"]
            ),
            "mae_improvement_pct": (
                improvement_pct(
                    identity_metrics["mae"],
                    metrics["mae"]
                )
            ),
            "rmse_improvement_pct": (
                improvement_pct(
                    identity_metrics["rmse"],
                    metrics["rmse"]
                )
            ),
            "equal_year_mae": year_mae,
            "identity_equal_year_mae": (
                identity_year_mae
            ),
            "equal_year_improvement_pct": (
                improvement_pct(
                    identity_year_mae,
                    year_mae
                )
            ),
            "year_win_fraction": (
                year_win_fraction
            ),
            "contrast_mae": (
                contrast_metrics["mae"]
            ),
            "identity_contrast_mae": (
                identity_contrast_metrics["mae"]
            ),
            "contrast_improvement_pct": (
                improvement_pct(
                    identity_contrast_metrics["mae"],
                    contrast_metrics["mae"]
                )
            ),
            "contrast_sign_agreement_pct": (
                100
                *
                np.mean(
                    np.sign(predicted_contrast)
                    ==
                    np.sign(target_contrast)
                )
            ),
            "range_violations": (
                range_violations
            ),
            "fold_intercept_min": (
                folds["intercept"].min()
            ),
            "fold_intercept_max": (
                folds["intercept"].max()
            ),
            "fold_slope_min": (
                folds["slope"].min()
            ),
            "fold_slope_max": (
                folds["slope"].max()
            )
        }
    }


# ============================================================================
# 7. RUN ALL CANDIDATES
# ============================================================================

model_records = []
all_predictions = []
all_folds = []

for scenario in SCENARIOS:

    for pair_name in SENSOR_PAIRS:

        group = pairs.loc[
            (
                pairs["scenario"]
                == scenario
            )
            &
            (
                pairs["sensor_pair"]
                == pair_name
            )
        ].copy()

        if group.empty:

            continue

        direction = get_direction(
            pair_name
        )

        for index in INDICES:

            data = prepare_calibration_data(
                group,
                index,
                pair_name
            )

            if data.empty:

                continue

            for model in MODELS:

                result = validate_model(
                    data,
                    model,
                    index
                )

                full_fit = result[
                    "full_fit"
                ]

                record = {
                    "scenario": scenario,
                    "sensor_pair": pair_name,
                    "source_sensor": (
                        direction["source_sensor"]
                    ),
                    "target_sensor": (
                        direction["target_sensor"]
                    ),
                    "index": index,
                    "model": model,
                    "n_pairs": len(data),
                    "n_years": (
                        data["pair_year"].nunique()
                    ),
                    "source_iqr": source_iqr(data),
                    "validation_status": (
                        result["status"]
                    ),
                    "reason": result["reason"],
                    "full_intercept": (
                        full_fit.get(
                            "intercept",
                            np.nan
                        )
                    ),
                    "full_slope": (
                        full_fit.get(
                            "slope",
                            np.nan
                        )
                    )
                }

                record.update(
                    result.get(
                        "metrics",
                        {}
                    )
                )

                model_records.append(
                    record
                )

                metadata = {
                    "scenario": scenario,
                    "sensor_pair": pair_name,
                    "index": index,
                    "model": model
                }

                if not result[
                    "predictions"
                ].empty:

                    predictions = (
                        result["predictions"]
                        .assign(**metadata)
                    )

                    all_predictions.append(
                        predictions
                    )

                if not result[
                    "folds"
                ].empty:

                    folds = (
                        result["folds"]
                        .assign(**metadata)
                    )

                    all_folds.append(
                        folds
                    )

model_summary = pd.DataFrame(
    model_records
)

cv_predictions = (
    pd.concat(
        all_predictions,
        ignore_index=True
    )
    if all_predictions
    else pd.DataFrame()
)

cv_folds = (
    pd.concat(
        all_folds,
        ignore_index=True
    )
    if all_folds
    else pd.DataFrame()
)

assert not model_summary.empty

print("\nMODEL VALIDATION STATUS")

display(
    model_summary.groupby([
        "model",
        "validation_status"
    ])
    .size()
    .unstack(fill_value=0)
)

# ============================================================================
# 8. PREDEFINED ACCEPTANCE RULES
#
# These are conservative decision criteria,
# not formal significance tests.
# ============================================================================

def evaluate_acceptance(row):

    if row["model"] == "Identity":

        return (
            False,
            "Reference_model"
        )

    if (
        row["validation_status"]
        != "VALIDATED"
    ):

        return (
            False,
            row["reason"]
        )

    reasons = []

    if (
        not np.isfinite(
            row["mae_improvement_pct"]
        )
        or row["mae_improvement_pct"]
        < MIN_MAE_IMPROVEMENT_PCT
    ):

        reasons.append(
            "Insufficient_MAE_improvement"
        )

    if (
        not np.isfinite(
            row["rmse_improvement_pct"]
        )
        or row["rmse_improvement_pct"]
        < MIN_RMSE_IMPROVEMENT_PCT
    ):

        reasons.append(
            "RMSE_not_improved"
        )

    if (
        not np.isfinite(
            row["equal_year_improvement_pct"]
        )
        or row["equal_year_improvement_pct"]
        < MIN_MAE_IMPROVEMENT_PCT
    ):

        reasons.append(
            "Insufficient_equal_year_improvement"
        )

    if (
        row["year_win_fraction"]
        < MIN_YEAR_WIN_FRACTION
    ):

        reasons.append(
            "Insufficient_year_consistency"
        )

    if (
        not np.isfinite(
            row["contrast_improvement_pct"]
        )
        or row["contrast_improvement_pct"]
        <
        -MAX_CONTRAST_MAE_INCREASE_PCT
    ):

        reasons.append(
            "Forest_contrast_degraded"
        )

    if row["range_violations"] > 0:

        reasons.append(
            "Out_of_range_predictions"
        )

    accepted = (
        len(reasons) == 0
    )

    return (
        accepted,
        "PASS"
        if accepted
        else "; ".join(reasons)
    )


acceptance = model_summary.apply(
    evaluate_acceptance,
    axis=1,
    result_type="expand"
)

model_summary[
    "passes_validation_rules"
] = acceptance[0].astype(bool)

model_summary[
    "acceptance_reason"
] = acceptance[1]

print("\n" + "=" * 76)
print("PRIMARY CROSS-VALIDATION RESULTS")
print("=" * 76)

primary_models = model_summary.loc[
    model_summary["scenario"]
    == PRIMARY_SCENARIO
].copy()

display(
    primary_models[[
        "sensor_pair",
        "index",
        "model",
        "n_pairs",
        "full_intercept",
        "full_slope",
        "validation_status",
        "cv_mae",
        "cv_rmse",
        "mae_improvement_pct",
        "equal_year_improvement_pct",
        "year_win_fraction",
        "contrast_improvement_pct",
        "passes_validation_rules"
    ]].round(5)
)

# ============================================================================
# 9. SELECT BEST MODEL WITHIN EACH SCENARIO
#
# Identity remains the fallback.
#
# If two correction models perform similarly,
# favor the simpler additive offset.
# ============================================================================

selection_records = []

for keys, group in model_summary.groupby([
    "scenario",
    "sensor_pair",
    "index"
]):

    scenario, pair_name, index = keys

    identity = group.loc[
        group["model"] == "Identity"
    ].iloc[0]

    candidates = group.loc[
        group["passes_validation_rules"]
    ].copy()

    if candidates.empty:

        chosen = identity

        selection_reason = (
            "No_correction_passed_validation"
        )

    else:

        candidates = candidates.sort_values(
            "equal_year_mae"
        )

        chosen = candidates.iloc[0]

        offset = candidates.loc[
            candidates["model"]
            == "Median_offset"
        ]

        if not offset.empty:

            offset_row = offset.iloc[0]

            # Require at least 5% additional
            # equal-year MAE improvement before
            # preferring regression over offset.

            if (
                offset_row["equal_year_mae"]
                <= chosen["equal_year_mae"] * 1.05
            ):

                chosen = offset_row

        selection_reason = (
            "Correction_passed_validation"
        )

    selection_records.append({
        "scenario": scenario,
        "sensor_pair": pair_name,
        "index": index,
        "selected_model": chosen["model"],
        "n_pairs": chosen["n_pairs"],
        "n_years": chosen["n_years"],
        "intercept": chosen["full_intercept"],
        "slope": chosen["full_slope"],
        "cv_mae": chosen.get(
            "cv_mae",
            np.nan
        ),
        "cv_rmse": chosen.get(
            "cv_rmse",
            np.nan
        ),
        "mae_improvement_pct": chosen.get(
            "mae_improvement_pct",
            np.nan
        ),
        "equal_year_improvement_pct": (
            chosen.get(
                "equal_year_improvement_pct",
                np.nan
            )
        ),
        "contrast_improvement_pct": (
            chosen.get(
                "contrast_improvement_pct",
                np.nan
            )
        ),
        "selection_reason": selection_reason
    })

scenario_selection = pd.DataFrame(
    selection_records
)

print("\n" + "=" * 76)
print("MODEL SELECTION BY SCENARIO")
print("=" * 76)

display(
    scenario_selection.pivot_table(
        index=[
            "sensor_pair",
            "index"
        ],
        columns="scenario",
        values="selected_model",
        aggfunc="first"
    )
)

# ============================================================================
# 10. CROSS-SCENARIO ROBUSTNESS
#
# A correction selected in the primary scenario
# must also be evaluated under alternative
# eligibility criteria.
#
# Sensitivity scenarios are overlapping datasets,
# NOT independent external validations.
# ============================================================================

robustness_records = []

primary_selection = (
    scenario_selection.loc[
        scenario_selection["scenario"]
        == PRIMARY_SCENARIO
    ].copy()
)

for _, primary in (
    primary_selection.iterrows()
):

    pair_name = primary[
        "sensor_pair"
    ]

    index = primary[
        "index"
    ]

    selected_model = primary[
        "selected_model"
    ]

    if selected_model == "Identity":

        robustness_records.append({
            "sensor_pair": pair_name,
            "index": index,
            "primary_model": selected_model,
            "n_evaluable_sensitivity": 0,
            "n_passing_sensitivity": 0,
            "sensitivity_pass_fraction": (
                np.nan
            ),
            "max_offset_drift": np.nan,
            "robustness_status": (
                "IDENTITY_RETAINED"
            )
        })

        continue

    alternatives = model_summary.loc[
        (
            model_summary["sensor_pair"]
            == pair_name
        )
        &
        (
            model_summary["index"]
            == index
        )
        &
        (
            model_summary["model"]
            == selected_model
        )
        &
        (
            model_summary["scenario"]
            != PRIMARY_SCENARIO
        )
        &
        (
            model_summary["validation_status"]
            == "VALIDATED"
        )
    ].copy()

    n_evaluable = len(
        alternatives
    )

    n_passing = int(
        alternatives[
            "passes_validation_rules"
        ].sum()
    )

    pass_fraction = (
        n_passing / n_evaluable
        if n_evaluable
        else np.nan
    )

    if selected_model == "Median_offset":

        offset_drift = (
            float(
                np.max(
                    np.abs(
                        alternatives[
                            "full_intercept"
                        ]
                        -
                        primary["intercept"]
                    )
                )
            )
            if n_evaluable
            else np.nan
        )

    else:

        offset_drift = np.nan

    reasons = []

    if (
        n_evaluable
        < MIN_SENSITIVITY_SCENARIOS
    ):

        reasons.append(
            "Insufficient_sensitivity_support"
        )

    if (
        not np.isfinite(pass_fraction)
        or pass_fraction
        < MIN_SENSITIVITY_PASS_FRACTION
    ):

        reasons.append(
            "Sensitivity_validation_failed"
        )

    if (
        selected_model == "Median_offset"
        and (
            not np.isfinite(offset_drift)
            or offset_drift > MAX_OFFSET_DRIFT
        )
    ):

        reasons.append(
            "Unstable_offset"
        )

    # Require validation in the main seasonal
    # window, since this is the intended
    # historical analysis period.

    seasonal = alternatives.loc[
        alternatives["scenario"]
        == "MaySep_ge70"
    ]

    if (
        seasonal.empty
        or not seasonal[
            "passes_validation_rules"
        ].all()
    ):

        reasons.append(
            "Main_season_not_validated"
        )

    # Explicit L7 orbital-drift sensitivity.
    # This is a conservative cutoff scenario,
    # not an exact physical onset date.

    if pair_name == "Landsat_7__Landsat_8":

        early_l7 = alternatives.loc[
            alternatives["scenario"]
            == "Annual_ge70_L7_until2017"
        ]

        if (
            early_l7.empty
            or not early_l7[
                "passes_validation_rules"
            ].all()
        ):

            reasons.append(
                "Early_L7_not_validated"
            )

    robustness_records.append({
        "sensor_pair": pair_name,
        "index": index,
        "primary_model": selected_model,
        "n_evaluable_sensitivity": (
            n_evaluable
        ),
        "n_passing_sensitivity": (
            n_passing
        ),
        "sensitivity_pass_fraction": (
            pass_fraction
        ),
        "max_offset_drift": (
            offset_drift
        ),
        "robustness_status": (
            "ROBUST_CANDIDATE"
            if not reasons
            else "; ".join(reasons)
        )
    })

robustness_summary = pd.DataFrame(
    robustness_records
)

print("\n" + "=" * 76)
print("CROSS-SCENARIO ROBUSTNESS")
print("=" * 76)

display(
    robustness_summary.round(5)
)

# ============================================================================
# 11. FINAL CALIBRATION RECOMMENDATIONS
#
# No historical values are modified.
#
# L8-L9 has only five annual pairs and cannot
# receive a locally validated correction.
#
# Identity is not evidence of equivalence.
# It means no correction was justified.
# ============================================================================

recommendations = (
    primary_selection.merge(
        robustness_summary,
        on=[
            "sensor_pair",
            "index"
        ],
        validate="one_to_one"
    )
)

final_records = []

for _, row in (
    recommendations.iterrows()
):

    pair_name = row[
        "sensor_pair"
    ]

    direction = get_direction(
        pair_name
    )

    robust = (
        row["robustness_status"]
        == "ROBUST_CANDIDATE"
    )

    if robust:

        final_model = row[
            "selected_model"
        ]

        intercept = float(
            row["intercept"]
        )

        slope = float(
            row["slope"]
        )

        status = (
            "LOCALLY_VALIDATED_CANDIDATE"
        )

    else:

        final_model = "Identity"
        intercept = 0.0
        slope = 1.0

        status = (
            "NO_JUSTIFIED_LOCAL_CORRECTION"
        )

    if pair_name == "Landsat_8__Landsat_9":

        final_model = "Identity"
        intercept = 0.0
        slope = 1.0

        status = (
            "INSUFFICIENT_LOCAL_SUPPORT"
        )

    final_records.append({
        **row.to_dict(),
        "source_sensor": (
            direction["source_sensor"]
        ),
        "target_sensor": (
            direction["target_sensor"]
        ),
        "recommended_model": final_model,
        "recommended_intercept": intercept,
        "recommended_slope": slope,
        "recommendation_status": status,
        "applied_to_historical_series": False
    })

final_recommendations = pd.DataFrame(
    final_records
)

print("\n" + "=" * 76)
print("FINAL LOCAL CALIBRATION RECOMMENDATIONS")
print("=" * 76)

display(
    final_recommendations[[
        "source_sensor",
        "target_sensor",
        "index",
        "selected_model",
        "recommended_model",
        "recommended_intercept",
        "recommended_slope",
        "recommendation_status"
    ]].round(5)
)

# ============================================================================
# 12. CALIBRATION NETWORK AUDIT
#
# A chain is complete only if each required
# connection has a validated correction.
#
# Do not silently assume that identity means
# cross-sensor equivalence.
# ============================================================================

network_records = []

for index in INDICES:

    subset = final_recommendations.loc[
        final_recommendations["index"]
        == index
    ]

    connections = (
        subset.set_index(
            "sensor_pair"
        )["recommendation_status"]
        .to_dict()
    )

    l57_valid = (
        connections.get(
            "Landsat_5__Landsat_7"
        )
        == "LOCALLY_VALIDATED_CANDIDATE"
    )

    l78_valid = (
        connections.get(
            "Landsat_7__Landsat_8"
        )
        == "LOCALLY_VALIDATED_CANDIDATE"
    )

    l89_valid = (
        connections.get(
            "Landsat_8__Landsat_9"
        )
        == "LOCALLY_VALIDATED_CANDIDATE"
    )

    network_records.append({
        "index": index,
        "L5_to_L7_validated": l57_valid,
        "L7_to_L8_validated": l78_valid,
        "L9_to_L8_validated": l89_valid,
        "L5_to_L8_chain_complete": (
            l57_valid and l78_valid
        ),
        "all_sensor_network_complete": (
            l57_valid
            and l78_valid
            and l89_valid
        ),
        "historical_series_harmonized": False
    })

network_audit = pd.DataFrame(
    network_records
)

print("\nCALIBRATION NETWORK AUDIT")

display(network_audit)

# ============================================================================
# 13. OUT-OF-SAMPLE PREDICTION DIAGNOSTICS
# ============================================================================

diagnostic_records = []

if not cv_predictions.empty:

    valid_models = model_summary.loc[
        model_summary["validation_status"]
        == "VALIDATED",
        [
            "scenario",
            "sensor_pair",
            "index",
            "model"
        ]
    ]

    validated_predictions = (
        cv_predictions.merge(
            valid_models,
            on=[
                "scenario",
                "sensor_pair",
                "index",
                "model"
            ],
            how="inner",
            validate="many_to_one"
        )
    )

    for keys, group in (
        validated_predictions.groupby([
            "scenario",
            "sensor_pair",
            "index",
            "model",
            "forest_type"
        ])
    ):

        (
            scenario,
            pair_name,
            index,
            model,
            forest
        ) = keys

        metrics = calculate_metrics(
            group["observed_target"],
            group["predicted_target"]
        )

        identity = calculate_metrics(
            group["observed_target"],
            group["observed_source"]
        )

        diagnostic_records.append({
            "scenario": scenario,
            "sensor_pair": pair_name,
            "index": index,
            "model": model,
            "forest_type": forest,
            "n_pairs": len(group),
            "cv_mae": metrics["mae"],
            "cv_rmse": metrics["rmse"],
            "median_error": (
                metrics["median_error"]
            ),
            "identity_mae": identity["mae"],
            "mae_improvement_pct": (
                improvement_pct(
                    identity["mae"],
                    metrics["mae"]
                )
            )
        })

else:

    validated_predictions = (
        pd.DataFrame()
    )

forest_diagnostics = pd.DataFrame(
    diagnostic_records
)

print("\nFOREST-SPECIFIC VALIDATION DIAGNOSTICS")

if not forest_diagnostics.empty:

    display(
        forest_diagnostics.loc[
            (
                forest_diagnostics["scenario"]
                == PRIMARY_SCENARIO
            )
            &
            (
                forest_diagnostics["model"]
                == "Median_offset"
            )
        ].round(5)
    )

# ============================================================================
# 14. FIGURE — VALIDATION AND MODEL SELECTION
# ============================================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "savefig.dpi": 600,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

PAIR_LABELS = {
    "Landsat_5__Landsat_7": "L5–L7",
    "Landsat_7__Landsat_8": "L7–L8",
    "Landsat_8__Landsat_9": "L9–L8"
}

MODEL_COLORS = {
    "Identity": "#A6ADB2",
    "Median_offset": "#DB9140",
    "Theil_Sen": "#24634F"
}

fig, axes = plt.subplots(
    2,
    2,
    figsize=(15, 10),
    constrained_layout=True
)

# ---------------------------------------------------------------------------
# Panel A: MAE improvement
# ---------------------------------------------------------------------------

ax = axes[0, 0]

labels = [
    f"{PAIR_LABELS[pair]}\n{index}"
    for pair in SENSOR_PAIRS
    for index in INDICES
]

positions = np.arange(
    len(labels)
)

for model, offset in [
    ("Median_offset", -0.17),
    ("Theil_Sen", 0.17)
]:

    values = []

    for pair in SENSOR_PAIRS:

        for index in INDICES:

            row = primary_models.loc[
                (
                    primary_models["sensor_pair"]
                    == pair
                )
                &
                (
                    primary_models["index"]
                    == index
                )
                &
                (
                    primary_models["model"]
                    == model
                )
                &
                (
                    primary_models["validation_status"]
                    == "VALIDATED"
                )
            ]

            values.append(
                row[
                    "mae_improvement_pct"
                ].iloc[0]
                if len(row)
                else np.nan
            )

    ax.bar(
        positions + offset,
        values,
        width=0.32,
        color=MODEL_COLORS[model],
        label=model.replace("_", " ")
    )

ax.axhline(
    0,
    color="#555555",
    linewidth=1
)

ax.axhline(
    MIN_MAE_IMPROVEMENT_PCT,
    color="#888888",
    linestyle="--",
    linewidth=1
)

ax.set_xticks(
    positions,
    labels,
    rotation=45,
    ha="right",
    fontsize=8
)

ax.set_ylabel(
    "Out-of-year MAE improvement (%)"
)

ax.set_title(
    "(a) Predictive improvement over identity",
    loc="left",
    fontweight="bold"
)

ax.legend(frameon=False)
ax.grid(alpha=0.15, axis="y")

# ---------------------------------------------------------------------------
# Panel B: Contrast preservation
# ---------------------------------------------------------------------------

ax = axes[0, 1]

for model, marker in [
    ("Median_offset", "o"),
    ("Theil_Sen", "^")
]:

    subset = primary_models.loc[
        (
            primary_models["model"]
            == model
        )
        &
        (
            primary_models["validation_status"]
            == "VALIDATED"
        )
    ]

    ax.scatter(
        subset["identity_contrast_mae"],
        subset["contrast_mae"],
        color=MODEL_COLORS[model],
        marker=marker,
        s=65,
        alpha=0.85,
        label=model.replace("_", " ")
    )

limit = max(
    0.10,
    primary_models[
        "identity_contrast_mae"
    ].max() * 1.15,
    primary_models[
        "contrast_mae"
    ].max() * 1.15
)

ax.plot(
    [0, limit],
    [0, limit],
    linestyle="--",
    color="#888888"
)

ax.set_xlim(0, limit)
ax.set_ylim(0, limit)

ax.set_xlabel(
    "Identity contrast MAE"
)

ax.set_ylabel(
    "Calibrated contrast MAE"
)

ax.set_title(
    "(b) Preservation of forest contrasts",
    loc="left",
    fontweight="bold"
)

ax.legend(frameon=False)
ax.grid(alpha=0.15)

# ---------------------------------------------------------------------------
# Panel C: Robustness across scenarios
# ---------------------------------------------------------------------------

ax = axes[1, 0]

matrix = np.full(
    (
        len(SENSOR_PAIRS) * len(INDICES),
        len(SCENARIOS)
    ),
    np.nan
)

for i, pair in enumerate(
    SENSOR_PAIRS
):

    for j, index in enumerate(
        INDICES
    ):

        row_number = (
            i * len(INDICES) + j
        )

        for k, scenario in enumerate(
            SCENARIOS
        ):

            subset = (
                scenario_selection.loc[
                    (
                        scenario_selection["scenario"]
                        == scenario
                    )
                    &
                    (
                        scenario_selection["sensor_pair"]
                        == pair
                    )
                    &
                    (
                        scenario_selection["index"]
                        == index
                    )
                ]
            )

            if len(subset):

                selected = subset[
                    "selected_model"
                ].iloc[0]

                matrix[row_number, k] = {
                    "Identity": 0,
                    "Median_offset": 1,
                    "Theil_Sen": 2
                }[selected]

from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

cmap = ListedColormap([
    MODEL_COLORS["Identity"],
    MODEL_COLORS["Median_offset"],
    MODEL_COLORS["Theil_Sen"]
])

ax.imshow(
    matrix,
    cmap=cmap,
    vmin=-0.5,
    vmax=2.5,
    aspect="auto"
)

ax.set_yticks(
    positions,
    [
        f"{PAIR_LABELS[pair]} | {index}"
        for pair in SENSOR_PAIRS
        for index in INDICES
    ],
    fontsize=8
)

ax.set_xticks(
    np.arange(len(SCENARIOS)),
    [
        "Annual 70%",
        "May–Sep 70%",
        "Annual 90%",
        "May–Sep 90%",
        "Jun–Sep 70%",
        "L7 ≤2017"
    ],
    rotation=35,
    ha="right",
    fontsize=8
)

ax.set_title(
    "(c) Scenario-specific model selection",
    loc="left",
    fontweight="bold"
)

ax.legend(
    handles=[
        Patch(
            color=MODEL_COLORS[model],
            label=model.replace("_", " ")
        )
        for model in MODELS
    ],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.30),
    ncol=3,
    frameon=False,
    fontsize=8
)

# ---------------------------------------------------------------------------
# Panel D: Final recommendations
# ---------------------------------------------------------------------------

ax = axes[1, 1]

final_matrix = np.zeros(
    (
        len(SENSOR_PAIRS),
        len(INDICES)
    )
)

for i, pair in enumerate(
    SENSOR_PAIRS
):

    for j, index in enumerate(
        INDICES
    ):

        subset = (
            final_recommendations.loc[
                (
                    final_recommendations["sensor_pair"]
                    == pair
                )
                &
                (
                    final_recommendations["index"]
                    == index
                )
            ]
        )

        if len(subset):

            model = subset[
                "recommended_model"
            ].iloc[0]

            final_matrix[i, j] = {
                "Identity": 0,
                "Median_offset": 1,
                "Theil_Sen": 2
            }[model]

ax.imshow(
    final_matrix,
    cmap=cmap,
    vmin=-0.5,
    vmax=2.5,
    aspect="auto"
)

ax.set_xticks(
    np.arange(len(INDICES)),
    INDICES
)

ax.set_yticks(
    np.arange(len(SENSOR_PAIRS)),
    [
        PAIR_LABELS[pair]
        for pair in SENSOR_PAIRS
    ]
)

for i in range(
    len(SENSOR_PAIRS)
):

    for j in range(
        len(INDICES)
    ):

        code = int(
            final_matrix[i, j]
        )

        text = {
            0: "No\ncorrection",
            1: "Offset",
            2: "Regression"
        }[code]

        ax.text(
            j,
            i,
            text,
            ha="center",
            va="center",
            fontsize=9,
            color=(
                "white"
                if code == 2
                else "black"
            )
        )

ax.set_title(
    "(d) Final local calibration recommendations",
    loc="left",
    fontweight="bold"
)

fig.suptitle(
    "Cross-sensor calibration: "
    "out-of-year validation and robustness",
    fontsize=16,
    fontweight="bold"
)

FIGURE_STEM = (
    FIGURE_DIR
    /
    "FIG_LANDSAT_VALIDATED_CALIBRATION"
)

for extension in [
    "png",
    "pdf",
    "svg"
]:

    fig.savefig(
        FIGURE_STEM.with_suffix(
            f".{extension}"
        ),
        bbox_inches="tight",
        facecolor="white"
    )

plt.show()

# ============================================================================
# 15. FIGURE — OUT-OF-YEAR ERROR DISTRIBUTIONS
# ============================================================================

fig, axes = plt.subplots(
    3,
    5,
    figsize=(19, 11),
    constrained_layout=True
)

for i, pair in enumerate(
    SENSOR_PAIRS
):

    for j, index in enumerate(
        INDICES
    ):

        ax = axes[i, j]

        subset = validated_predictions.loc[
            (
                validated_predictions["scenario"]
                == PRIMARY_SCENARIO
            )
            &
            (
                validated_predictions["sensor_pair"]
                == pair
            )
            &
            (
                validated_predictions["index"]
                == index
            )
        ]

        box_values = []
        box_labels = []
        box_colors = []

        for model in MODELS:

            model_data = subset.loc[
                subset["model"]
                == model
            ]

            if model_data.empty:

                continue

            # Mean absolute error per acquisition pair,
            # rather than treating two forests as
            # independent validation replicates.

            pair_error = (
                model_data.assign(
                    absolute_error=np.abs(
                        model_data[
                            "prediction_error"
                        ]
                    )
                )
                .groupby("pair_id")[
                    "absolute_error"
                ]
                .mean()
            )

            box_values.append(
                pair_error.to_numpy()
            )

            box_labels.append({
                "Identity": "ID",
                "Median_offset": "Offset",
                "Theil_Sen": "TS"
            }[model])

            box_colors.append(
                MODEL_COLORS[model]
            )

        if box_values:

            bp = ax.boxplot(
                box_values,
                patch_artist=True,
                showfliers=True,
                widths=0.55
            )

            for patch, color in zip(
                bp["boxes"],
                box_colors
            ):

                patch.set_facecolor(
                    color
                )

                patch.set_alpha(
                    0.75
                )

            ax.set_xticks(
                np.arange(
                    1,
                    len(box_labels) + 1
                ),
                box_labels
            )

        else:

            ax.text(
                0.5,
                0.5,
                "No validation support",
                transform=ax.transAxes,
                ha="center",
                va="center",
                fontsize=9
            )

        ax.set_title(
            f"{PAIR_LABELS[pair]} | {index}",
            fontweight="bold",
            fontsize=10
        )

        if j == 0:

            ax.set_ylabel(
                "Absolute prediction error"
            )

        ax.grid(
            alpha=0.15,
            axis="y"
        )

fig.suptitle(
    "Held-out acquisition-pair errors\n"
    "Leave-one-year-out validation",
    fontsize=15,
    fontweight="bold"
)

FIGURE_STEM_2 = (
    FIGURE_DIR
    /
    "FIG_LANDSAT_CALIBRATION_CV_ERRORS"
)

for extension in [
    "png",
    "pdf",
    "svg"
]:

    fig.savefig(
        FIGURE_STEM_2.with_suffix(
            f".{extension}"
        ),
        bbox_inches="tight",
        facecolor="white"
    )

plt.show()

# ============================================================================
# 16. EXPORT RESULTS
# ============================================================================

outputs = {
    "LANDSAT_CALIBRATION_MODEL_SUMMARY.csv": (
        model_summary
    ),
    "LANDSAT_CALIBRATION_CV_PREDICTIONS.csv": (
        cv_predictions
    ),
    "LANDSAT_CALIBRATION_CV_FOLDS.csv": (
        cv_folds
    ),
    "LANDSAT_CALIBRATION_SCENARIO_SELECTION.csv": (
        scenario_selection
    ),
    "LANDSAT_CALIBRATION_ROBUSTNESS.csv": (
        robustness_summary
    ),
    "LANDSAT_CALIBRATION_RECOMMENDATIONS.csv": (
        final_recommendations
    ),
    "LANDSAT_CALIBRATION_NETWORK_AUDIT.csv": (
        network_audit
    ),
    "LANDSAT_CALIBRATION_FOREST_DIAGNOSTICS.csv": (
        forest_diagnostics
    )
}

for filename, dataframe in (
    outputs.items()
):

    dataframe.to_csv(
        CAL_DIR / filename,
        index=False,
        encoding="utf-8-sig"
    )

workbook_path = (
    TABLE_DIR
    /
    "SATIPO_LANDSAT_VALIDATED_CALIBRATION.xlsx"
)

with pd.ExcelWriter(
    workbook_path,
    engine="openpyxl"
) as writer:

    model_summary.to_excel(
        writer,
        sheet_name="Model_summary",
        index=False
    )

    cv_predictions.to_excel(
        writer,
        sheet_name="CV_predictions",
        index=False
    )

    cv_folds.to_excel(
        writer,
        sheet_name="CV_folds",
        index=False
    )

    scenario_selection.to_excel(
        writer,
        sheet_name="Scenario_selection",
        index=False
    )

    robustness_summary.to_excel(
        writer,
        sheet_name="Robustness",
        index=False
    )

    final_recommendations.to_excel(
        writer,
        sheet_name="Recommendations",
        index=False
    )

    network_audit.to_excel(
        writer,
        sheet_name="Network_audit",
        index=False
    )

    forest_diagnostics.to_excel(
        writer,
        sheet_name="Forest_diagnostics",
        index=False
    )

# ============================================================================
# 17. CONFIGURATION AND PROVENANCE MANIFEST
# ============================================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):

            digest.update(chunk)

    return digest.hexdigest()


manifest = {
    "project": "SATIPO_FOREST_PROJECT",
    "cell": "91B",
    "analysis": (
        "Validated local cross-sensor calibration"
    ),
    "input": str(PAIR_PATH),
    "input_sha256": sha256_file(
        PAIR_PATH
    ),
    "primary_scenario": PRIMARY_SCENARIO,
    "primary_window_days": PRIMARY_WINDOW,
    "scenarios": SCENARIOS,
    "indices": INDICES,
    "models": MODELS,
    "validation": (
        "Leave-one-acquisition-year-out"
    ),
    "shared_transformation_between_forests": True,
    "calibration_direction": {
        pair: get_direction(pair)
        for pair in SENSOR_PAIRS
    },
    "thresholds": {
        "min_pairs_offset": MIN_PAIRS_OFFSET,
        "min_years_offset": MIN_YEARS_OFFSET,
        "min_pairs_regression": (
            MIN_PAIRS_REGRESSION
        ),
        "min_years_regression": (
            MIN_YEARS_REGRESSION
        ),
        "min_train_pairs_offset": (
            MIN_TRAIN_PAIRS_OFFSET
        ),
        "min_train_years_offset": (
            MIN_TRAIN_YEARS_OFFSET
        ),
        "min_train_pairs_regression": (
            MIN_TRAIN_PAIRS_REGRESSION
        ),
        "min_train_years_regression": (
            MIN_TRAIN_YEARS_REGRESSION
        ),
        "min_mae_improvement_pct": (
            MIN_MAE_IMPROVEMENT_PCT
        ),
        "min_rmse_improvement_pct": (
            MIN_RMSE_IMPROVEMENT_PCT
        ),
        "max_contrast_mae_increase_pct": (
            MAX_CONTRAST_MAE_INCREASE_PCT
        ),
        "min_slope": MIN_SLOPE,
        "max_slope": MAX_SLOPE,
        "min_source_iqr": MIN_SOURCE_IQR,
        "min_year_win_fraction": (
            MIN_YEAR_WIN_FRACTION
        ),
        "min_sensitivity_scenarios": (
            MIN_SENSITIVITY_SCENARIOS
        ),
        "min_sensitivity_pass_fraction": (
            MIN_SENSITIVITY_PASS_FRACTION
        ),
        "max_offset_drift": MAX_OFFSET_DRIFT
    },
    "n_candidate_models": int(
        len(model_summary)
    ),
    "n_validated_models": int(
        (
            model_summary["validation_status"]
            == "VALIDATED"
        ).sum()
    ),
    "n_robust_local_candidates": int(
        (
            final_recommendations[
                "recommendation_status"
            ]
            == "LOCALLY_VALIDATED_CANDIDATE"
        ).sum()
    ),
    "historical_harmonization_applied": False,
    "original_screened_master_preserved": True,
    "outputs": list(outputs.keys()),
    "limitations": [
        "Near-coincident acquisitions are separated by up to eight days.",
        "Calibration cannot isolate pure instrumental effects.",
        "Leave-one-year-out validation is internal temporal validation.",
        "Sensitivity scenarios are overlapping, not independent datasets.",
        "Identity does not establish cross-sensor equivalence.",
        "Model-selection thresholds are exploratory decision rules.",
        "Theil-Sen fitting uses two nested forest observations per pair.",
        "Regression candidates require extrapolation checks before use.",
        "Landsat 8-9 local calibration support is insufficient.",
        "Cross-sensor coefficient uncertainty is not yet propagated."
    ]
}

manifest_path = (
    CAL_DIR
    /
    "LANDSAT_VALIDATED_CALIBRATION_MANIFEST.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        manifest,
        file,
        indent=2,
        ensure_ascii=False
    )

# ============================================================================
# 18. FINAL SCIENTIFIC REPORT
# ============================================================================

print("\n" + "=" * 76)
print("SATIPO — VALIDATED CALIBRATION REPORT")
print("=" * 76)

print(
    "Calibration scenarios:",
    len(SCENARIOS)
)

print(
    "Sensor connections:",
    len(SENSOR_PAIRS)
)

print(
    "Spectral indices:",
    len(INDICES)
)

print(
    "Candidate model configurations:",
    len(model_summary)
)

print(
    "Complete out-of-year validations:",
    int(
        (
            model_summary["validation_status"]
            == "VALIDATED"
        ).sum()
    )
)

print(
    "Failed validation folds:",
    int(
        (
            ~cv_folds["valid"].astype(bool)
        ).sum()
    )
)

print(
    "Primary corrections passing validation:",
    int(
        primary_models[
            "passes_validation_rules"
        ].sum()
    )
)

print(
    "Robust local calibration candidates:",
    int(
        (
            final_recommendations[
                "recommendation_status"
            ]
            == "LOCALLY_VALIDATED_CANDIDATE"
        ).sum()
    )
)

print(
    "Complete L5-to-L8 calibration chains:",
    int(
        network_audit[
            "L5_to_L8_chain_complete"
        ].sum()
    )
)

print(
    "Complete four-sensor networks:",
    int(
        network_audit[
            "all_sensor_network_complete"
        ].sum()
    )
)

print("\nFINAL MODEL RECOMMENDATIONS")

display(
    final_recommendations[[
        "sensor_pair",
        "index",
        "recommended_model",
        "recommended_intercept",
        "recommended_slope",
        "recommendation_status"
    ]].round(5)
)

print("\nCALIBRATION NETWORK")

display(network_audit)

print("\nOriginal screened master: PRESERVED")
print("Historical harmonization: NOT APPLIED")
print("Coefficient uncertainty: PENDING")
print("Extrapolation assessment: PENDING")

print("\nOutput directory:")
print(CAL_DIR)

print("\nWorkbook:")
print(workbook_path)

print("\nManifest:")
print(manifest_path)

print("\nFigures:")
print(FIGURE_STEM)
print(FIGURE_STEM_2)

print("\n" + "=" * 76)
print("CELL 91B COMPLETED")
print("NEXT: CELL 91C — HARMONIZATION SENSITIVITY")
print("=" * 76)


In [ ]:

# ============================================================================
# CELL 91C — FINAL HARMONIZATION SENSITIVITY
# SATIPO FOREST PROJECT
#
# Objectives:
#   1. Quantify uncertainty of the validated L7 -> L8 EVI offset.
#   2. Evaluate calibration-domain extrapolation.
#   3. Compare native and locally adjusted EVI trajectories.
#   4. Preserve native spectral observations.
#   5. Prepare historical analysis-ready datasets.
#
# IMPORTANT:
#   The local EVI adjustment is a sensitivity scenario,
#   NOT a complete four-sensor harmonization.
# ============================================================================

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

# ============================================================================
# 1. CONFIGURATION
# ============================================================================

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

REMOTE_DIR = ROOT / "04_REMOTE_SENSING"

LANDSAT_DIR = REMOTE_DIR / "03_LANDSAT"

SCREEN_DIR = (
    LANDSAT_DIR / "03_RADIOMETRIC_SCREENING"
)

HARM_DIR = (
    LANDSAT_DIR / "04_CROSS_SENSOR_HARMONIZATION"
)

CAL_DIR = HARM_DIR / "VALIDATED_CALIBRATION"

SENS_DIR = HARM_DIR / "FINAL_SENSITIVITY"

TABLE_DIR = REMOTE_DIR / "08_OUTPUTS" / "TABLES"

FIGURE_DIR = REMOTE_DIR / "08_OUTPUTS" / "FIGURES"

for directory in [
    SENS_DIR,
    TABLE_DIR,
    FIGURE_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

SCREEN_PATH = (
    SCREEN_DIR / "LANDSAT_SCREENED_MASTER.csv"
)

PAIR_PATH = (
    HARM_DIR / "LANDSAT_TEMPORAL_MATCHED_PAIRS.csv"
)

REC_PATH = (
    CAL_DIR
    / "LANDSAT_CALIBRATION_RECOMMENDATIONS.csv"
)

MODEL_PATH = (
    CAL_DIR
    / "LANDSAT_CALIBRATION_MODEL_SUMMARY.csv"
)

for path in [
    SCREEN_PATH,
    PAIR_PATH,
    REC_PATH,
    MODEL_PATH
]:
    assert path.exists(), f"Missing file: {path}"

INDICES = [
    "NDVI",
    "EVI",
    "NDMI",
    "NBR",
    "NBR2"
]

FORESTS = [
    "Late_secondary",
    "Mature"
]

SENSORS = [
    "Landsat_5",
    "Landsat_7",
    "Landsat_8",
    "Landsat_9"
]

PRIMARY_MONTHS = [5, 6, 7, 8, 9]

PRIMARY_COVERAGE = 0.70

MIN_ANNUAL_SCENES = 3

N_BOOT = 10000

SEED = 20261115

rng = np.random.default_rng(SEED)

print("=" * 76)
print("SATIPO — FINAL HARMONIZATION SENSITIVITY")
print("=" * 76)

# ============================================================================
# 2. LOAD AND VERIFY INPUTS
# ============================================================================

screened = pd.read_csv(SCREEN_PATH)

matched = pd.read_csv(PAIR_PATH)

recommendations = pd.read_csv(REC_PATH)

models = pd.read_csv(MODEL_PATH)

assert len(screened) == 2328

assert screened["scene_id"].nunique() == 388

assert screened["unit_id"].nunique() == 6

screened["date"] = pd.to_datetime(
    screened["date"]
)

screened["year"] = (
    screened["date"].dt.year
)

screened["month"] = (
    screened["date"].dt.month
)

assert not screened.duplicated([
    "scene_id",
    "unit_id"
]).any()

accepted = recommendations.loc[
    recommendations["recommendation_status"]
    == "LOCALLY_VALIDATED_CANDIDATE"
].copy()

assert len(accepted) == 1

accepted = accepted.iloc[0]

assert accepted["source_sensor"] == "Landsat_7"

assert accepted["target_sensor"] == "Landsat_8"

assert accepted["index"] == "EVI"

assert accepted["recommended_model"] == "Median_offset"

assert np.isclose(
    accepted["recommended_slope"],
    1.0
)

EVI_OFFSET = float(
    accepted["recommended_intercept"]
)

print("\nValidated EVI offset:", EVI_OFFSET)

print("\nSCREENED MASTER COLUMNS")

print(screened.columns.tolist())

# ============================================================================
# 3. AUDIT REJECTED NDVI CORRECTION
# ============================================================================

ndvi_rejection = models.loc[
    (
        models["scenario"]
        == "Annual_ge70"
    )
    &
    (
        models["sensor_pair"]
        == "Landsat_7__Landsat_8"
    )
    &
    (
        models["index"] == "NDVI"
    )
    &
    (
        models["model"] == "Median_offset"
    )
].copy()

print("\nNDVI CORRECTION REJECTION AUDIT")

audit_cols = [
    "sensor_pair",
    "index",
    "full_intercept",
    "mae_improvement_pct",
    "equal_year_improvement_pct",
    "year_win_fraction",
    "range_violations",
    "acceptance_reason"
]

display(
    ndvi_rejection[
        audit_cols
    ]
)

# ============================================================================
# 4. PREPARE EVI CALIBRATION PAIRS
#
# Each acquisition pair contributes one average
# discrepancy across the two forests.
#
# The two forests are not independent replicates.
# ============================================================================

evi_pairs = matched.loc[
    (
        matched["scenario"]
        == "Annual_ge70"
    )
    &
    (
        matched["sensor_pair"]
        == "Landsat_7__Landsat_8"
    )
    &
    (
        matched["max_days"] == 8
    )
].copy()

required_pair_cols = [
    "pair_id",
    "pair_year",
    "date_a",
    "date_b",
    "EVI_a__Late_secondary",
    "EVI_a__Mature",
    "EVI_b__Late_secondary",
    "EVI_b__Mature"
]

assert all(
    col in evi_pairs.columns
    for col in required_pair_cols
)

assert len(evi_pairs) == 25

assert evi_pairs["pair_year"].nunique() == 10

assert not evi_pairs["pair_id"].duplicated().any()

evi_pairs["pair_year"] = (
    evi_pairs["pair_year"].astype(int)
)

evi_pairs["offset_secondary"] = (
    evi_pairs["EVI_b__Late_secondary"]
    -
    evi_pairs["EVI_a__Late_secondary"]
)

evi_pairs["offset_mature"] = (
    evi_pairs["EVI_b__Mature"]
    -
    evi_pairs["EVI_a__Mature"]
)

evi_pairs["pair_offset"] = (
    evi_pairs["offset_secondary"]
    +
    evi_pairs["offset_mature"]
) / 2

assert np.isfinite(
    evi_pairs["pair_offset"]
).all()

observed_offset = float(
    evi_pairs["pair_offset"].median()
)

assert np.isclose(
    observed_offset,
    EVI_OFFSET,
    atol=1e-5
)

year_groups = {
    int(year): group[
        "pair_offset"
    ].to_numpy(dtype=float)
    for year, group in
    evi_pairs.groupby("pair_year")
}

calibration_years = np.array(
    sorted(year_groups.keys())
)

print("\nCALIBRATION SUPPORT")

print("Acquisition pairs:", len(evi_pairs))

print("Years:", len(calibration_years))

print("Calibration years:", calibration_years)

print("Observed offset:", observed_offset)

# ============================================================================
# 5. YEAR-CLUSTER BOOTSTRAP
#
# Resample acquisition years with replacement.
# Keep all acquisition pairs within each sampled year.
#
# This quantifies conditional uncertainty in the
# empirical offset, not pure instrumental uncertainty.
# ============================================================================

bootstrap_offsets = np.empty(
    N_BOOT,
    dtype=float
)

bootstrap_equal_year = np.empty(
    N_BOOT,
    dtype=float
)

for b in range(N_BOOT):

    sampled_years = rng.choice(
        calibration_years,
        size=len(calibration_years),
        replace=True
    )

    sampled_offsets = np.concatenate([
        year_groups[int(year)]
        for year in sampled_years
    ])

    bootstrap_offsets[b] = (
        np.median(sampled_offsets)
    )

    # Additional sensitivity:
    # equal weighting of year-specific medians.

    year_medians = [
        np.median(year_groups[int(year)])
        for year in sampled_years
    ]

    bootstrap_equal_year[b] = (
        np.median(year_medians)
    )

offset_lcl, offset_ucl = np.quantile(
    bootstrap_offsets,
    [0.025, 0.975]
)

equal_year_offset = float(
    np.median([
        np.median(values)
        for values in year_groups.values()
    ])
)

offset_bootstrap = pd.DataFrame({
    "draw": np.arange(1, N_BOOT + 1),
    "offset_pair_weighted": bootstrap_offsets,
    "offset_equal_year": bootstrap_equal_year
})

offset_summary = pd.DataFrame([
    {
        "estimator": "Pair_weighted_median",
        "estimate": observed_offset,
        "bootstrap_mean": (
            bootstrap_offsets.mean()
        ),
        "bootstrap_median": (
            np.median(bootstrap_offsets)
        ),
        "bootstrap_sd": (
            bootstrap_offsets.std(ddof=1)
        ),
        "lcl_95": offset_lcl,
        "ucl_95": offset_ucl,
        "positive_draws_pct": (
            100 * np.mean(
                bootstrap_offsets > 0
            )
        ),
        "n_pairs": len(evi_pairs),
        "n_years": len(calibration_years),
        "n_bootstrap": N_BOOT
    },
    {
        "estimator": "Equal_year_median",
        "estimate": equal_year_offset,
        "bootstrap_mean": (
            bootstrap_equal_year.mean()
        ),
        "bootstrap_median": (
            np.median(bootstrap_equal_year)
        ),
        "bootstrap_sd": (
            bootstrap_equal_year.std(ddof=1)
        ),
        "lcl_95": np.quantile(
            bootstrap_equal_year,
            0.025
        ),
        "ucl_95": np.quantile(
            bootstrap_equal_year,
            0.975
        ),
        "positive_draws_pct": (
            100 * np.mean(
                bootstrap_equal_year > 0
            )
        ),
        "n_pairs": len(evi_pairs),
        "n_years": len(calibration_years),
        "n_bootstrap": N_BOOT
    }
])

print("\n" + "=" * 76)
print("EVI OFFSET UNCERTAINTY")
print("=" * 76)

display(
    offset_summary.round(5)
)

# ============================================================================
# 6. LEAVE-ONE-YEAR-OUT COEFFICIENT STABILITY
# ============================================================================

year_stability_records = []

for year in calibration_years:

    training = evi_pairs.loc[
        evi_pairs["pair_year"] != year
    ]

    testing = evi_pairs.loc[
        evi_pairs["pair_year"] == year
    ]

    fold_offset = float(
        training["pair_offset"].median()
    )

    year_stability_records.append({
        "excluded_year": int(year),
        "n_train_pairs": len(training),
        "n_test_pairs": len(testing),
        "fold_offset": fold_offset,
        "full_offset": observed_offset,
        "offset_difference": (
            fold_offset - observed_offset
        ),
        "heldout_median_discrepancy": (
            testing["pair_offset"].median()
        )
    })

year_stability = pd.DataFrame(
    year_stability_records
)

print("\nYEAR-SPECIFIC COEFFICIENT STABILITY")

display(
    year_stability.round(5)
)

# ============================================================================
# 7. CROSS-SCENARIO COEFFICIENT STABILITY
# ============================================================================

scenario_offsets = models.loc[
    (
        models["sensor_pair"]
        == "Landsat_7__Landsat_8"
    )
    &
    (
        models["index"] == "EVI"
    )
    &
    (
        models["model"] == "Median_offset"
    )
].copy()

scenario_offsets = scenario_offsets[[
    "scenario",
    "n_pairs",
    "n_years",
    "full_intercept",
    "cv_mae",
    "cv_rmse",
    "mae_improvement_pct",
    "equal_year_improvement_pct",
    "year_win_fraction",
    "passes_validation_rules"
]]

scenario_offsets["difference_from_primary"] = (
    scenario_offsets["full_intercept"]
    -
    EVI_OFFSET
)

print("\nCROSS-SCENARIO EVI OFFSETS")

display(
    scenario_offsets.round(5)
)

# ============================================================================
# 8. RESOLVE SCREENED SPECTRAL COLUMN NAMES
#
# Support common naming conventions used in the
# previous screening cells.
#
# Do not silently substitute native unscreened
# columns if screened columns cannot be identified.
# ============================================================================

def normalize_column_name(name):

    return (
        str(name)
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )


column_lookup = {
    normalize_column_name(col): col
    for col in screened.columns
}


def resolve_column(candidates):

    for candidate in candidates:

        key = normalize_column_name(
            candidate
        )

        if key in column_lookup:

            return column_lookup[key]

    return None


def resolve_index_column(index):

    candidates = [
        f"{index}_screened_mean",
        f"screened_{index}_mean",
        f"{index}_mean_screened",
        f"{index}_Index_specific_mean",
        f"Index_specific_{index}_mean",
        f"{index}_mean_Index_specific",
        f"{index}_screen_mean",
        f"screen_{index}_mean",
        f"{index}_physical_mean",
        f"{index}_radiometric_mean",
        f"{index}_mean_filtered",
        f"{index}_mean"
    ]

    return resolve_column(candidates)


spectral_columns = {
    index: resolve_index_column(index)
    for index in INDICES
}

coverage_column = resolve_column([
    "screened_valid_fraction",
    "valid_fraction_screened",
    "screened_coverage",
    "screened_coverage_fraction",
    "screened_valid_coverage",
    "radiometric_valid_fraction",
    "valid_fraction_radiometric",
    "valid_fraction_physical",
    "physical_valid_fraction",
    "screened_fraction",
    "valid_fraction_screen",
    "coverage_screened",
    "valid_fraction"
])

print("\nRESOLVED COLUMN MAPPING")

print("Indices:", spectral_columns)

print("Coverage:", coverage_column)

missing_indices = [
    index
    for index, column in
    spectral_columns.items()
    if column is None
]

assert not missing_indices, (
    "Unresolved spectral columns: "
    f"{missing_indices}. "
    "Inspect the printed screened-master columns."
)

assert coverage_column is not None, (
    "Could not identify screened coverage column."
)

# If the screened master uses generic names,
# verify against the known cell 90B retention counts.

# ============================================================================
# 9. BUILD NATIVE SCREENED ANALYSIS MASTER
#
# Retain all six spatial units.
# Use explicit standardized output columns.
# ============================================================================

base_cols = [
    "sensor",
    "scene_id",
    "date",
    "year",
    "month",
    "unit_id",
    "forest_type",
    "spatial_scale"
]

assert all(
    col in screened.columns
    for col in base_cols
)

historical_master = screened[
    base_cols
].copy()

historical_master[
    "screened_coverage"
] = pd.to_numeric(
    screened[coverage_column],
    errors="coerce"
)

for index, column in (
    spectral_columns.items()
):

    historical_master[
        f"{index}_native"
    ] = pd.to_numeric(
        screened[column],
        errors="coerce"
    )

coverage_max = historical_master[
    "screened_coverage"
].max()

# Support coverage expressed either as
# fractions or percentages.

if coverage_max > 1.001:

    assert coverage_max <= 100.001

    historical_master[
        "screened_coverage"
    ] /= 100

assert historical_master[
    "screened_coverage"
].dropna().between(
    0,
    1.001
).all()

historical_master[
    "screened_coverage"
] = historical_master[
    "screened_coverage"
].clip(0, 1)

historical_master[
    "primary_season"
] = historical_master[
    "month"
].isin(PRIMARY_MONTHS)

historical_master[
    "coverage_ge70"
] = (
    historical_master[
        "screened_coverage"
    ]
    >= PRIMARY_COVERAGE
)

historical_master[
    "coverage_ge90"
] = (
    historical_master[
        "screened_coverage"
    ]
    >= 0.90
)

historical_master[
    "L7_after2017"
] = (
    (
        historical_master["sensor"]
        == "Landsat_7"
    )
    &
    (
        historical_master["year"] > 2017
    )
)

# Verify normalized-index statistics.
# EVI is not mathematically restricted to [-1, 1].

for index in [
    "NDVI",
    "NDMI",
    "NBR",
    "NBR2"
]:

    values = historical_master[
        f"{index}_native"
    ].dropna()

    assert values.between(
        -1.000001,
        1.000001
    ).all(), (
        f"Out-of-range {index} values. "
        "Check screened column mapping."
    )

# ============================================================================
# 10. VERIFY FRAGMENT-LEVEL SCENE RETENTION
#
# These are known results from cell 90B.
# Stop if the inferred coverage column does not
# reproduce the previously validated counts.
# ============================================================================

fragment_master = historical_master.loc[
    historical_master["unit_id"].isin([
        "FRAG_01",
        "FRAG_02"
    ])
].copy()

assert len(fragment_master) == 388 * 2

fragment_coverage = (
    fragment_master.pivot(
        index="scene_id",
        columns="forest_type",
        values="screened_coverage"
    )
)

fragment_coverage[
    "paired_coverage"
] = fragment_coverage[
    FORESTS
].min(axis=1)

scene_metadata = (
    fragment_master[[
        "scene_id",
        "sensor",
        "date",
        "year",
        "month",
        "primary_season"
    ]]
    .drop_duplicates("scene_id")
    .set_index("scene_id")
)

fragment_coverage = (
    fragment_coverage.join(
        scene_metadata,
        validate="one_to_one"
    )
)

retention_50 = int(
    (
        fragment_coverage[
            "paired_coverage"
        ] >= 0.50
    ).sum()
)

retention_70 = int(
    (
        fragment_coverage[
            "paired_coverage"
        ] >= 0.70
    ).sum()
)

retention_90 = int(
    (
        fragment_coverage[
            "paired_coverage"
        ] >= 0.90
    ).sum()
)

retention_season = int(
    (
        (
            fragment_coverage[
                "paired_coverage"
            ] >= 0.70
        )
        &
        fragment_coverage[
            "primary_season"
        ]
    ).sum()
)

print("\nSCREENED RETENTION REPRODUCTION")

print("Paired >=50%:", retention_50)

print("Paired >=70%:", retention_70)

print("Paired >=90%:", retention_90)

print(
    "May–September >=70%:",
    retention_season
)

assert retention_50 == 382, (
    "Coverage mapping does not reproduce "
    "cell 90B at the 50% threshold."
)

assert retention_70 == 345, (
    "Coverage mapping does not reproduce "
    "cell 90B at the 70% threshold."
)

assert retention_90 == 269, (
    "Coverage mapping does not reproduce "
    "cell 90B at the 90% threshold."
)

assert retention_season == 219, (
    "Seasonal retention differs from cell 90B."
)

print("Screened coverage mapping: VERIFIED")

# ============================================================================
# 11. VERIFY EVI VALUES AGAINST CALIBRATION PAIRS
#
# Compare the mapped screened EVI values against
# the exact data used in cell 91A/91B.
#
# This guards against accidentally selecting
# native unscreened EVI columns.
# ============================================================================

fragment_evi_lookup = (
    fragment_master.pivot_table(
        index=[
            "sensor",
            "date"
        ],
        columns="forest_type",
        values="EVI_native",
        aggfunc="first"
    )
)

mapping_records = []

for _, row in evi_pairs.iterrows():

    for sensor, side in [
        ("Landsat_7", "a"),
        ("Landsat_8", "b")
    ]:

        date = pd.Timestamp(
            row[f"date_{side}"]
        )

        key = (sensor, date)

        assert key in (
            fragment_evi_lookup.index
        ), f"Missing calibration scene: {key}"

        for forest in FORESTS:

            master_value = float(
                fragment_evi_lookup.loc[
                    key,
                    forest
                ]
            )

            pair_value = float(
                row[f"EVI_{side}__{forest}"]
            )

            mapping_records.append({
                "pair_id": row["pair_id"],
                "sensor": sensor,
                "date": date,
                "forest_type": forest,
                "master_value": master_value,
                "pair_value": pair_value,
                "absolute_difference": abs(
                    master_value - pair_value
                )
            })

mapping_audit = pd.DataFrame(
    mapping_records
)

assert len(mapping_audit) == 100

max_mapping_difference = (
    mapping_audit[
        "absolute_difference"
    ].max()
)

print(
    "\nMaximum EVI mapping difference:",
    max_mapping_difference
)

assert max_mapping_difference < 1e-5, (
    "Selected EVI column differs from the "
    "screened values used in calibration."
)

print("EVI mapping: VERIFIED")

# ============================================================================
# 12. CALIBRATION DOMAIN
#
# Calibration support is defined using source
# observations from Landsat 7 fragments.
#
# The empirical range is not a universal
# physical validity interval.
# ============================================================================

source_evi = evi_pairs[[
    "EVI_a__Late_secondary",
    "EVI_a__Mature"
]].to_numpy(
    dtype=float
).ravel()

target_evi = evi_pairs[[
    "EVI_b__Late_secondary",
    "EVI_b__Mature"
]].to_numpy(
    dtype=float
).ravel()

SOURCE_MIN = float(
    np.min(source_evi)
)

SOURCE_MAX = float(
    np.max(source_evi)
)

SOURCE_Q05 = float(
    np.quantile(source_evi, 0.05)
)

SOURCE_Q95 = float(
    np.quantile(source_evi, 0.95)
)

TARGET_MIN = float(
    np.min(target_evi)
)

TARGET_MAX = float(
    np.max(target_evi)
)

calibration_domain = pd.DataFrame([
    {
        "sensor": "Landsat_7",
        "role": "Source",
        "n_observations": len(source_evi),
        "minimum": SOURCE_MIN,
        "q05": SOURCE_Q05,
        "median": np.median(source_evi),
        "q95": SOURCE_Q95,
        "maximum": SOURCE_MAX
    },
    {
        "sensor": "Landsat_8",
        "role": "Target",
        "n_observations": len(target_evi),
        "minimum": TARGET_MIN,
        "q05": np.quantile(
            target_evi, 0.05
        ),
        "median": np.median(target_evi),
        "q95": np.quantile(
            target_evi, 0.95
        ),
        "maximum": TARGET_MAX
    }
])

print("\n" + "=" * 76)
print("CALIBRATION DOMAIN")
print("=" * 76)

display(
    calibration_domain.round(5)
)

# ============================================================================
# 13. APPLY LOCAL EVI SENSITIVITY
#
# Only Landsat 7 fragment-level observations
# are eligible for the adjustment.
#
# Plot and core observations are not assumed
# to share a validated fragment-level offset.
#
# No corrections are applied to NDVI, NDMI,
# NBR or NBR2.
# ============================================================================

historical_master[
    "EVI_local_sensitivity"
] = historical_master[
    "EVI_native"
]

historical_master[
    "EVI_offset_lcl"
] = historical_master[
    "EVI_native"
]

historical_master[
    "EVI_offset_ucl"
] = historical_master[
    "EVI_native"
]

historical_master[
    "EVI_adjustment_status"
] = "Native_retained"

historical_master[
    "EVI_calibration_domain"
] = "Not_applicable"

eligible_l7 = (
    (
        historical_master["sensor"]
        == "Landsat_7"
    )
    &
    historical_master["unit_id"].isin([
        "FRAG_01",
        "FRAG_02"
    ])
    &
    historical_master[
        "EVI_native"
    ].notna()
)

within_domain = (
    historical_master[
        "EVI_native"
    ].between(
        SOURCE_MIN,
        SOURCE_MAX
    )
)

within_central_domain = (
    historical_master[
        "EVI_native"
    ].between(
        SOURCE_Q05,
        SOURCE_Q95
    )
)

historical_master.loc[
    eligible_l7,
    "EVI_calibration_domain"
] = "Outside_observed_range"

historical_master.loc[
    eligible_l7 & within_domain,
    "EVI_calibration_domain"
] = "Within_observed_range"

historical_master.loc[
    eligible_l7 & within_central_domain,
    "EVI_calibration_domain"
] = "Within_central_90pct"

# Broad sensitivity:
# apply the constant offset, retaining explicit
# extrapolation flags.

historical_master.loc[
    eligible_l7,
    "EVI_local_sensitivity"
] = (
    historical_master.loc[
        eligible_l7,
        "EVI_native"
    ]
    +
    EVI_OFFSET
)

historical_master.loc[
    eligible_l7,
    "EVI_offset_lcl"
] = (
    historical_master.loc[
        eligible_l7,
        "EVI_native"
    ]
    +
    offset_lcl
)

historical_master.loc[
    eligible_l7,
    "EVI_offset_ucl"
] = (
    historical_master.loc[
        eligible_l7,
        "EVI_native"
    ]
    +
    offset_ucl
)

historical_master.loc[
    eligible_l7 & within_domain,
    "EVI_adjustment_status"
] = "Local_offset_within_domain"

historical_master.loc[
    eligible_l7 & ~within_domain,
    "EVI_adjustment_status"
] = "Local_offset_extrapolated"

# A domain-restricted version is kept separately.
# Outside-domain L7 values become missing in this
# sensitivity scenario, not silently uncorrected.

historical_master[
    "EVI_domain_restricted"
] = historical_master[
    "EVI_local_sensitivity"
]

historical_master.loc[
    eligible_l7 & ~within_domain,
    "EVI_domain_restricted"
] = np.nan

historical_master[
    "fully_harmonized"
] = False

# ============================================================================
# 14. EXTRAPOLATION AUDIT
# ============================================================================

extrapolation_records = []

for season_label, season_mask in [
    (
        "Annual",
        pd.Series(
            True,
            index=historical_master.index
        )
    ),
    (
        "MaySep",
        historical_master[
            "primary_season"
        ]
    )
]:

    for threshold in [
        0.50,
        0.70,
        0.90
    ]:

        eligible = historical_master.loc[
            eligible_l7
            &
            season_mask
            &
            (
                historical_master[
                    "screened_coverage"
                ] >= threshold
            )
        ].copy()

        for forest in FORESTS:

            subset = eligible.loc[
                eligible["forest_type"]
                == forest
            ]

            n = len(subset)

            n_outside = int(
                (
                    subset[
                        "EVI_calibration_domain"
                    ]
                    == "Outside_observed_range"
                ).sum()
            )

            n_central = int(
                (
                    subset[
                        "EVI_calibration_domain"
                    ]
                    == "Within_central_90pct"
                ).sum()
            )

            n_target_outside = int(
                (
                    ~subset[
                        "EVI_local_sensitivity"
                    ].between(
                        TARGET_MIN,
                        TARGET_MAX
                    )
                ).sum()
            )

            extrapolation_records.append({
                "season": season_label,
                "coverage_threshold": threshold,
                "forest_type": forest,
                "n_observations": n,
                "n_outside_source_range": (
                    n_outside
                ),
                "outside_source_pct": (
                    100 * n_outside / n
                    if n else np.nan
                ),
                "n_within_central_domain": (
                    n_central
                ),
                "within_central_pct": (
                    100 * n_central / n
                    if n else np.nan
                ),
                "n_adjusted_outside_target_range": (
                    n_target_outside
                ),
                "adjusted_outside_target_pct": (
                    100 * n_target_outside / n
                    if n else np.nan
                )
            })

extrapolation_summary = pd.DataFrame(
    extrapolation_records
)

print("\n" + "=" * 76)
print("EVI EXTRAPOLATION AUDIT")
print("=" * 76)

display(
    extrapolation_summary.round(4)
)

# ============================================================================
# 15. BUILD PAIRED HISTORICAL OBSERVATIONS
#
# All comparisons use both forests from
# the same acquisition and spatial scale.
#
# The main series uses native screened indices.
# ============================================================================

paired_records = []

group_cols = [
    "sensor",
    "scene_id",
    "date",
    "year",
    "month",
    "spatial_scale"
]

for keys, group in (
    historical_master.groupby(
        group_cols,
        dropna=False
    )
):

    (
        sensor,
        scene_id,
        date,
        year,
        month,
        spatial_scale
    ) = keys

    if len(group) != 2:

        raise ValueError(
            f"Expected two forest units: {keys}"
        )

    assert set(
        group["forest_type"]
    ) == set(FORESTS)

    wide = group.set_index(
        "forest_type"
    )

    coverage_s = float(
        wide.loc[
            "Late_secondary",
            "screened_coverage"
        ]
    )

    coverage_m = float(
        wide.loc[
            "Mature",
            "screened_coverage"
        ]
    )

    paired_coverage = (
        min(coverage_s, coverage_m)
        if np.isfinite([
            coverage_s,
            coverage_m
        ]).all()
        else np.nan
    )

    record = {
        "sensor": sensor,
        "scene_id": scene_id,
        "date": date,
        "year": int(year),
        "month": int(month),
        "spatial_scale": spatial_scale,
        "coverage_secondary": coverage_s,
        "coverage_mature": coverage_m,
        "paired_coverage": paired_coverage,
        "paired_ge70": (
            paired_coverage >= 0.70
        ),
        "paired_ge90": (
            paired_coverage >= 0.90
        ),
        "primary_season": (
            month in PRIMARY_MONTHS
        )
    }

    for index in INDICES:

        secondary = float(
            wide.loc[
                "Late_secondary",
                f"{index}_native"
            ]
        )

        mature = float(
            wide.loc[
                "Mature",
                f"{index}_native"
            ]
        )

        record.update({
            f"{index}_secondary": secondary,
            f"{index}_mature": mature,
            f"{index}_contrast": (
                secondary - mature
            )
        })

    adjusted_s = float(
        wide.loc[
            "Late_secondary",
            "EVI_local_sensitivity"
        ]
    )

    adjusted_m = float(
        wide.loc[
            "Mature",
            "EVI_local_sensitivity"
        ]
    )

    record.update({
        "EVI_adjusted_secondary": adjusted_s,
        "EVI_adjusted_mature": adjusted_m,
        "EVI_adjusted_contrast": (
            adjusted_s - adjusted_m
        ),
        "EVI_domain_secondary": (
            wide.loc[
                "Late_secondary",
                "EVI_calibration_domain"
            ]
        ),
        "EVI_domain_mature": (
            wide.loc[
                "Mature",
                "EVI_calibration_domain"
            ]
        )
    })

    paired_records.append(record)

historical_pairs = pd.DataFrame(
    paired_records
)

assert len(historical_pairs) == 388 * 3

assert not historical_pairs.duplicated([
    "scene_id",
    "spatial_scale"
]).any()

contrast_check = historical_pairs[[
    "EVI_contrast",
    "EVI_adjusted_contrast"
]].dropna()

max_contrast_change = float(
    np.max(
        np.abs(
            contrast_check["EVI_contrast"]
            -
            contrast_check[
                "EVI_adjusted_contrast"
            ]
        )
    )
)

assert max_contrast_change < 1e-10

print("\nPAIRED HISTORICAL MASTER")

print("Rows:", len(historical_pairs))

print(
    "Maximum EVI contrast change:",
    max_contrast_change
)

print("Contrast preservation: VERIFIED")

# ============================================================================
# 16. PRIMARY HISTORICAL DATASET
#
# May–September.
# Paired screened coverage >=70%.
# Sensor identity retained.
# ============================================================================

historical_primary = historical_pairs.loc[
    historical_pairs["primary_season"]
    &
    historical_pairs["paired_ge70"]
].copy()

primary_fragments = (
    historical_primary.loc[
        historical_primary[
            "spatial_scale"
        ] == "Fragment"
    ].copy()
)

assert len(primary_fragments) == 219

print("\nPRIMARY HISTORICAL DATASET")

display(
    historical_primary.groupby([
        "spatial_scale",
        "sensor"
    ])
    .agg(
        n_scenes=("scene_id", "nunique"),
        n_years=("year", "nunique"),
        first_year=("year", "min"),
        last_year=("year", "max"),
        median_coverage=(
            "paired_coverage",
            "median"
        )
    )
    .reset_index()
)

# ============================================================================
# 17. ANNUAL SENSOR-SPECIFIC SUMMARIES
#
# Compute annual medians separately by sensor.
#
# Do not pool sensors into a supposedly
# harmonized annual trajectory.
#
# Retain low-support years but flag them.
# ============================================================================

annual_records = []

for keys, group in (
    historical_primary.groupby([
        "spatial_scale",
        "sensor",
        "year"
    ])
):

    spatial_scale, sensor, year = keys

    for index in INDICES:

        columns = [
            f"{index}_secondary",
            f"{index}_mature",
            f"{index}_contrast"
        ]

        valid = group.dropna(
            subset=columns
        )

        n = len(valid)

        if n == 0:

            continue

        record = {
            "spatial_scale": spatial_scale,
            "sensor": sensor,
            "year": int(year),
            "index": index,
            "n_scenes": n,
            "adequate_support": (
                n >= MIN_ANNUAL_SCENES
            ),
            "median_coverage": (
                valid[
                    "paired_coverage"
                ].median()
            ),
            "secondary_median": (
                valid[
                    f"{index}_secondary"
                ].median()
            ),
            "mature_median": (
                valid[
                    f"{index}_mature"
                ].median()
            ),
            "contrast_median": (
                valid[
                    f"{index}_contrast"
                ].median()
            ),
            "contrast_q25": (
                valid[
                    f"{index}_contrast"
                ].quantile(0.25)
            ),
            "contrast_q75": (
                valid[
                    f"{index}_contrast"
                ].quantile(0.75)
            ),
            "secondary_q25": (
                valid[
                    f"{index}_secondary"
                ].quantile(0.25)
            ),
            "secondary_q75": (
                valid[
                    f"{index}_secondary"
                ].quantile(0.75)
            ),
            "mature_q25": (
                valid[
                    f"{index}_mature"
                ].quantile(0.25)
            ),
            "mature_q75": (
                valid[
                    f"{index}_mature"
                ].quantile(0.75)
            )
        }

        if index == "EVI":

            record[
                "secondary_adjusted_median"
            ] = valid[
                "EVI_adjusted_secondary"
            ].median()

            record[
                "mature_adjusted_median"
            ] = valid[
                "EVI_adjusted_mature"
            ].median()

            record[
                "adjusted_contrast_median"
            ] = valid[
                "EVI_adjusted_contrast"
            ].median()

        annual_records.append(record)

annual_sensor_summary = pd.DataFrame(
    annual_records
)

print("\nANNUAL SENSOR-SPECIFIC SUPPORT")

display(
    annual_sensor_summary.loc[
        (
            annual_sensor_summary[
                "spatial_scale"
            ] == "Fragment"
        )
        &
        (
            annual_sensor_summary[
                "index"
            ] == "EVI"
        )
    ]
    .groupby("sensor")
    .agg(
        n_years=("year", "nunique"),
        adequate_years=(
            "adequate_support",
            "sum"
        ),
        median_scenes=("n_scenes", "median")
    )
    .reset_index()
)

# ============================================================================
# 18. EVI SENSITIVITY AT SENSOR OVERLAP
#
# Compare annual medians in years with both
# Landsat 7 and Landsat 8 observations.
#
# Annual medians need not correspond to
# simultaneous acquisitions.
#
# This is a descriptive sensitivity analysis,
# not an independent validation dataset.
# ============================================================================

evi_annual = annual_sensor_summary.loc[
    (
        annual_sensor_summary[
            "spatial_scale"
        ] == "Fragment"
    )
    &
    (
        annual_sensor_summary[
            "index"
        ] == "EVI"
    )
].copy()

l7_annual = evi_annual.loc[
    evi_annual["sensor"] == "Landsat_7"
].copy()

l8_annual = evi_annual.loc[
    evi_annual["sensor"] == "Landsat_8"
].copy()

annual_overlap = l7_annual.merge(
    l8_annual,
    on="year",
    suffixes=("_L7", "_L8"),
    validate="one_to_one"
)

overlap_records = []

for _, row in annual_overlap.iterrows():

    for forest in [
        "secondary",
        "mature"
    ]:

        native = row[
            f"{forest}_median_L7"
        ]

        adjusted = row[
            f"{forest}_adjusted_median_L7"
        ]

        reference = row[
            f"{forest}_median_L8"
        ]

        overlap_records.append({
            "year": int(row["year"]),
            "forest_type": forest,
            "n_scenes_L7": (
                row["n_scenes_L7"]
            ),
            "n_scenes_L8": (
                row["n_scenes_L8"]
            ),
            "adequate_support_both": bool(
                row["adequate_support_L7"]
                and
                row["adequate_support_L8"]
            ),
            "L7_native": native,
            "L7_adjusted": adjusted,
            "L8_native": reference,
            "native_absolute_difference": (
                abs(native - reference)
            ),
            "adjusted_absolute_difference": (
                abs(adjusted - reference)
            )
        })

annual_overlap_sensitivity = pd.DataFrame(
    overlap_records
)

print("\n" + "=" * 76)
print("ANNUAL EVI OVERLAP SENSITIVITY")
print("=" * 76)

display(
    annual_overlap_sensitivity.round(5)
)

# ============================================================================
# 19. 2019 FIELD-CENSUS CONDITIONS
#
# Sensor-specific summaries.
# Same seasonal and paired coverage criteria.
#
# Do not pool observations from different sensors.
# ============================================================================

field_2019 = (
    annual_sensor_summary.loc[
        annual_sensor_summary["year"]
        == 2019
    ].copy()
)

print("\n2019 FIELD-CENSUS SPECTRAL CONDITIONS")

display(
    field_2019[[
        "spatial_scale",
        "sensor",
        "index",
        "n_scenes",
        "adequate_support",
        "secondary_median",
        "mature_median",
        "contrast_median"
    ]].round(5)
)

# ============================================================================
# 20. FIGURE — FINAL EVI HARMONIZATION SENSITIVITY
# ============================================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

FOREST_COLORS = {
    "Late_secondary": "#DB9140",
    "Mature": "#24634F"
}

SENSOR_COLORS = {
    "Landsat_5": "#527B98",
    "Landsat_7": "#DB9140",
    "Landsat_8": "#24634F",
    "Landsat_9": "#795C92"
}

fig, axes = plt.subplots(
    2,
    2,
    figsize=(15, 10),
    constrained_layout=True
)

# ---------------------------------------------------------------------------
# Panel A — bootstrap uncertainty
# ---------------------------------------------------------------------------

ax = axes[0, 0]

ax.hist(
    bootstrap_offsets,
    bins=45,
    color="#527B98",
    alpha=0.80,
    edgecolor="white"
)

ax.axvline(
    observed_offset,
    color="#222222",
    linewidth=2,
    label=f"Offset = {observed_offset:.3f}"
)

ax.axvline(
    offset_lcl,
    color="#DB9140",
    linestyle="--",
    linewidth=1.5,
    label="95% bootstrap interval"
)

ax.axvline(
    offset_ucl,
    color="#DB9140",
    linestyle="--",
    linewidth=1.5
)

ax.set_xlabel(
    "Landsat 7 → Landsat 8 EVI offset"
)

ax.set_ylabel(
    "Bootstrap frequency"
)

ax.set_title(
    "(a) Year-cluster offset uncertainty",
    loc="left",
    fontweight="bold"
)

ax.legend(frameon=False)

# ---------------------------------------------------------------------------
# Panel B — scenario coefficient stability
# ---------------------------------------------------------------------------

ax = axes[0, 1]

plot_offsets = scenario_offsets.sort_values(
    "full_intercept"
)

y = np.arange(
    len(plot_offsets)
)

ax.axvspan(
    offset_lcl,
    offset_ucl,
    color="#527B98",
    alpha=0.15,
    label="Primary bootstrap interval"
)

ax.scatter(
    plot_offsets["full_intercept"],
    y,
    color="#24634F",
    s=75,
    zorder=3
)

ax.axvline(
    EVI_OFFSET,
    color="#222222",
    linestyle="--"
)

ax.set_yticks(
    y,
    plot_offsets["scenario"]
)

ax.set_xlabel(
    "Estimated additive offset"
)

ax.set_title(
    "(b) Sensitivity to eligibility criteria",
    loc="left",
    fontweight="bold"
)

ax.legend(
    frameon=False,
    fontsize=8
)

ax.grid(
    axis="x",
    alpha=0.15
)

# ---------------------------------------------------------------------------
# Panel C — annual native vs adjusted EVI
#
# Plot points instead of a continuous joined
# four-sensor line.
# ---------------------------------------------------------------------------

ax = axes[1, 0]

for forest, label in [
    ("secondary", "Late-secondary"),
    ("mature", "Mature")
]:

    color = (
        "#DB9140"
        if forest == "secondary"
        else "#24634F"
    )

    ax.scatter(
        l7_annual["year"],
        l7_annual[
            f"{forest}_median"
        ],
        facecolors="none",
        edgecolors=color,
        marker="o",
        s=40,
        alpha=0.8,
        label=f"L7 native — {label}"
    )

    ax.scatter(
        l7_annual["year"],
        l7_annual[
            f"{forest}_adjusted_median"
        ],
        color=color,
        marker="o",
        s=35,
        alpha=0.8,
        label=f"L7 adjusted — {label}"
    )

    ax.scatter(
        l8_annual["year"],
        l8_annual[
            f"{forest}_median"
        ],
        color=color,
        marker="^",
        s=45,
        alpha=0.8,
        label=f"L8 native — {label}"
    )

ax.axvline(
    2019,
    color="#C74D40",
    linestyle="--",
    linewidth=1
)

ax.set_xlabel("Year")

ax.set_ylabel("Annual median EVI")

ax.set_title(
    "(c) Native and locally adjusted EVI",
    loc="left",
    fontweight="bold"
)

ax.legend(
    frameon=False,
    fontsize=7,
    ncol=2
)

ax.grid(alpha=0.15)

# ---------------------------------------------------------------------------
# Panel D — forest contrast preservation
# ---------------------------------------------------------------------------

ax = axes[1, 1]

for sensor in SENSORS:

    subset = evi_annual.loc[
        evi_annual["sensor"] == sensor
    ]

    ax.scatter(
        subset["year"],
        subset["contrast_median"],
        color=SENSOR_COLORS[sensor],
        s=40,
        alpha=0.85,
        label=sensor.replace("_", " ")
    )

ax.axhline(
    0,
    color="#777777",
    linestyle="--"
)

ax.axvline(
    2019,
    color="#C74D40",
    linestyle="--",
    linewidth=1
)

ax.set_xlabel("Year")

ax.set_ylabel(
    "Median EVI contrast\n"
    "(secondary − mature)"
)

ax.set_title(
    "(d) Native within-scene forest contrasts",
    loc="left",
    fontweight="bold"
)

ax.legend(
    frameon=False,
    fontsize=8
)

ax.grid(alpha=0.15)

fig.suptitle(
    "Final sensitivity of local Landsat EVI calibration",
    fontsize=16,
    fontweight="bold"
)

FIGURE_STEM = (
    FIGURE_DIR
    / "FIG_LANDSAT_FINAL_HARMONIZATION_SENSITIVITY"
)

for extension in [
    "png",
    "pdf",
    "svg"
]:

    fig.savefig(
        FIGURE_STEM.with_suffix(
            f".{extension}"
        ),
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )

plt.show()

# ============================================================================
# 21. FIGURE — HISTORICAL NATIVE SENSOR-SPECIFIC
#      FOREST CONTRASTS
#
# Preliminary historical visualization.
# No trend significance tests at this stage.
# ============================================================================

fig, axes = plt.subplots(
    5,
    1,
    figsize=(14, 13),
    sharex=True,
    constrained_layout=True
)

for ax, index in zip(
    axes,
    INDICES
):

    subset = annual_sensor_summary.loc[
        (
            annual_sensor_summary[
                "spatial_scale"
            ] == "Fragment"
        )
        &
        (
            annual_sensor_summary[
                "index"
            ] == index
        )
    ].copy()

    for sensor in SENSORS:

        data = subset.loc[
            subset["sensor"] == sensor
        ]

        supported = data.loc[
            data["adequate_support"]
        ]

        sparse = data.loc[
            ~data["adequate_support"]
        ]

        ax.scatter(
            supported["year"],
            supported["contrast_median"],
            color=SENSOR_COLORS[sensor],
            s=45,
            label=sensor.replace("_", " "),
            zorder=3
        )

        ax.scatter(
            sparse["year"],
            sparse["contrast_median"],
            facecolors="none",
            edgecolors=SENSOR_COLORS[sensor],
            s=40,
            alpha=0.75,
            zorder=2
        )

        if not supported.empty:

            ax.vlines(
                supported["year"],
                supported["contrast_q25"],
                supported["contrast_q75"],
                color=SENSOR_COLORS[sensor],
                alpha=0.45,
                linewidth=1.1
            )

    ax.axhline(
        0,
        color="#777777",
        linestyle="--",
        linewidth=0.8
    )

    ax.axvline(
        2019,
        color="#C74D40",
        linestyle="--",
        linewidth=1
    )

    ax.set_ylabel(
        f"Δ{index}",
        fontweight="bold"
    )

    ax.grid(
        alpha=0.15
    )

axes[0].legend(
    frameon=False,
    ncol=4,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.30)
)

axes[-1].set_xlabel("Year")

fig.suptitle(
    "Historical within-scene spectral contrasts\n"
    "May–September | Paired coverage ≥70%",
    fontsize=15,
    fontweight="bold"
)

FIGURE_STEM_2 = (
    FIGURE_DIR
    / "FIG_LANDSAT_NATIVE_HISTORICAL_CONTRASTS"
)

for extension in [
    "png",
    "pdf",
    "svg"
]:

    fig.savefig(
        FIGURE_STEM_2.with_suffix(
            f".{extension}"
        ),
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )

plt.show()

# ============================================================================
# 22. EXPORT ALL RESULTS
# ============================================================================

outputs = {
    "EVI_OFFSET_BOOTSTRAP_DRAWS.csv": (
        offset_bootstrap
    ),
    "EVI_OFFSET_UNCERTAINTY.csv": (
        offset_summary
    ),
    "EVI_OFFSET_YEAR_STABILITY.csv": (
        year_stability
    ),
    "EVI_OFFSET_SCENARIO_STABILITY.csv": (
        scenario_offsets
    ),
    "EVI_CALIBRATION_DOMAIN.csv": (
        calibration_domain
    ),
    "EVI_EXTRAPOLATION_SUMMARY.csv": (
        extrapolation_summary
    ),
    "EVI_SCREENED_MAPPING_AUDIT.csv": (
        mapping_audit
    ),
    "NDVI_REJECTION_AUDIT.csv": (
        ndvi_rejection
    ),
    "LANDSAT_HISTORICAL_MASTER.csv": (
        historical_master
    ),
    "LANDSAT_HISTORICAL_PAIRED_MASTER.csv": (
        historical_pairs
    ),
    "LANDSAT_HISTORICAL_PRIMARY.csv": (
        historical_primary
    ),
    "LANDSAT_ANNUAL_SENSOR_SUMMARY.csv": (
        annual_sensor_summary
    ),
    "EVI_ANNUAL_OVERLAP_SENSITIVITY.csv": (
        annual_overlap_sensitivity
    ),
    "LANDSAT_FIELD_CENSUS_2019.csv": (
        field_2019
    )
}

for filename, dataframe in (
    outputs.items()
):

    dataframe.to_csv(
        SENS_DIR / filename,
        index=False,
        encoding="utf-8-sig"
    )

workbook_path = (
    TABLE_DIR
    / "SATIPO_LANDSAT_FINAL_HARMONIZATION.xlsx"
)

with pd.ExcelWriter(
    workbook_path,
    engine="openpyxl"
) as writer:

    offset_summary.to_excel(
        writer,
        sheet_name="Offset_uncertainty",
        index=False
    )

    year_stability.to_excel(
        writer,
        sheet_name="Year_stability",
        index=False
    )

    scenario_offsets.to_excel(
        writer,
        sheet_name="Scenario_stability",
        index=False
    )

    calibration_domain.to_excel(
        writer,
        sheet_name="Calibration_domain",
        index=False
    )

    extrapolation_summary.to_excel(
        writer,
        sheet_name="Extrapolation",
        index=False
    )

    mapping_audit.to_excel(
        writer,
        sheet_name="Mapping_audit",
        index=False
    )

    ndvi_rejection.to_excel(
        writer,
        sheet_name="NDVI_rejection",
        index=False
    )

    historical_master.to_excel(
        writer,
        sheet_name="Historical_master",
        index=False
    )

    historical_pairs.to_excel(
        writer,
        sheet_name="Paired_master",
        index=False
    )

    historical_primary.to_excel(
        writer,
        sheet_name="Primary_observations",
        index=False
    )

    annual_sensor_summary.to_excel(
        writer,
        sheet_name="Annual_sensor_summary",
        index=False
    )

    annual_overlap_sensitivity.to_excel(
        writer,
        sheet_name="EVI_overlap",
        index=False
    )

    field_2019.to_excel(
        writer,
        sheet_name="Field_census_2019",
        index=False
    )

# ============================================================================
# 23. FINAL MANIFEST
# ============================================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as file:

        for chunk in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b""
        ):

            digest.update(chunk)

    return digest.hexdigest()


manifest = {
    "project": "SATIPO_FOREST_PROJECT",
    "cell": "91C",
    "analysis": (
        "Final local harmonization sensitivity"
    ),
    "input_files": {
        str(path): sha256_file(path)
        for path in [
            SCREEN_PATH,
            PAIR_PATH,
            REC_PATH,
            MODEL_PATH
        ]
    },
    "primary_analysis": (
        "Native radiometrically screened "
        "sensor-specific observations"
    ),
    "primary_months": PRIMARY_MONTHS,
    "primary_paired_coverage": (
        PRIMARY_COVERAGE
    ),
    "minimum_annual_scenes": (
        MIN_ANNUAL_SCENES
    ),
    "spectral_column_mapping": (
        spectral_columns
    ),
    "coverage_column": coverage_column,
    "coverage_mapping_verified": True,
    "EVI_mapping_verified": True,
    "local_EVI_adjustment": {
        "source": "Landsat_7",
        "target": "Landsat_8",
        "offset": EVI_OFFSET,
        "bootstrap_lcl_95": float(
            offset_lcl
        ),
        "bootstrap_ucl_95": float(
            offset_ucl
        ),
        "bootstrap_draws": N_BOOT,
        "bootstrap_unit": "Acquisition_year",
        "n_pairs": len(evi_pairs),
        "n_years": len(calibration_years),
        "spatial_scope": "Fragment",
        "purpose": "Sensitivity_only",
        "source_min": SOURCE_MIN,
        "source_max": SOURCE_MAX,
        "source_q05": SOURCE_Q05,
        "source_q95": SOURCE_Q95
    },
    "contrast_preservation_max_error": (
        max_contrast_change
    ),
    "historical_master_rows": (
        len(historical_master)
    ),
    "paired_master_rows": (
        len(historical_pairs)
    ),
    "primary_fragment_scenes": (
        len(primary_fragments)
    ),
    "original_screened_master_preserved": True,
    "fully_harmonized_series_created": False,
    "outputs": list(outputs.keys()),
    "limitations": [
        "Calibration pairs are separated by up to eight days.",
        "The empirical offset does not isolate pure instrumental effects.",
        "Bootstrap uncertainty is conditional on ten calibration years.",
        "Offset uncertainty is not total spectral measurement uncertainty.",
        "The offset is validated locally at fragment scale only.",
        "Outside-domain adjustments are explicitly flagged.",
        "Landsat 5 and Landsat 9 are not transformed to Landsat 8.",
        "Annual sensor-specific medians are not a fully harmonized series.",
        "Nested plots and fragment cores are not independent replicates.",
        "Annual IQRs describe scene variability, not confidence intervals."
    ],
    "next_cell": 92
}

manifest_path = (
    SENS_DIR
    / "LANDSAT_FINAL_HARMONIZATION_MANIFEST.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        manifest,
        file,
        indent=2,
        ensure_ascii=False
    )

# ============================================================================
# 24. FINAL REPORT
# ============================================================================

print("\n" + "=" * 76)
print("SATIPO — FINAL HARMONIZATION REPORT")
print("=" * 76)

print(
    "Original screened observations:",
    len(screened)
)

print(
    "Historical master observations:",
    len(historical_master)
)

print(
    "Historical paired observations:",
    len(historical_pairs)
)

print(
    "Primary fragment scenes:",
    len(primary_fragments)
)

print(
    "Primary historical years:",
    primary_fragments["year"].nunique()
)

print(
    "Validated local corrections:",
    len(
        recommendations.loc[
            recommendations[
                "recommendation_status"
            ]
            == "LOCALLY_VALIDATED_CANDIDATE"
        ]
    )
)

print(
    "EVI offset:",
    round(EVI_OFFSET, 6)
)

print(
    "95% year-bootstrap interval:",
    (
        round(offset_lcl, 6),
        round(offset_ucl, 6)
    )
)

print(
    "Bootstrap draws:",
    N_BOOT
)

print(
    "Maximum contrast change:",
    max_contrast_change
)

print(
    "Original screened master: PRESERVED"
)

print(
    "Complete historical harmonization: NOT CLAIMED"
)

print(
    "Primary analysis: NATIVE SENSOR-SPECIFIC"
)

print(
    "Local EVI adjustment: SENSITIVITY ONLY"
)

print("\nPRIMARY EXTRAPOLATION SUMMARY")

display(
    extrapolation_summary.loc[
        (
            extrapolation_summary["season"]
            == "MaySep"
        )
        &
        (
            extrapolation_summary[
                "coverage_threshold"
            ] == 0.70
        )
    ].round(5)
)

print("\nOutput directory:")
print(SENS_DIR)

print("\nHistorical master:")
print(
    SENS_DIR
    / "LANDSAT_HISTORICAL_MASTER.csv"
)

print("\nPrimary historical dataset:")
print(
    SENS_DIR
    / "LANDSAT_HISTORICAL_PRIMARY.csv"
)

print("\nWorkbook:")
print(workbook_path)

print("\nManifest:")
print(manifest_path)

print("\nFigures:")
print(FIGURE_STEM)
print(FIGURE_STEM_2)

print("\n" + "=" * 76)
print("CELL 91C COMPLETED")
print("LANDSAT CALIBRATION STAGE CLOSED")
print("NEXT: CELL 92 — HISTORICAL FOREST TRAJECTORIES")
print("=" * 76)


In [ ]:

# =============================================================================
# SATIPO FOREST PROJECT
# CELL 92 — HISTORICAL FOREST TRAJECTORIES
# =============================================================================
#
# Scientific objectives:
#
# 1. Characterize historical spectral trajectories from 1985 to 2025.
# 2. Quantify within-scene contrasts between the two forest remnants.
# 3. Estimate sensor-specific temporal trends.
# 4. Evaluate temporal persistence and spectral convergence.
# 5. Test sensitivity to spatial scale, coverage and Landsat 7 orbit drift.
# 6. Contextualize the 2019 field census.
#
# IMPORTANT:
# - Native radiometrically screened data are the primary input.
# - No complete historical cross-sensor harmonization is assumed.
# - Forest contrasts are paired within acquisitions.
# - The two forests are individual study sites, not replicated treatments.
# - Temporal trends do not establish causal recovery trajectories.
#
# =============================================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone

import json
import hashlib
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from scipy.stats import theilslopes, kendalltau

from IPython.display import display

warnings.filterwarnings(
    'ignore',
    category=FutureWarning
)

# =============================================================================
# 1. CONFIGURATION
# =============================================================================

ROOT = Path(
    '/content/drive/MyDrive/SATIPO_FOREST_PROJECT'
)

REMOTE_DIR = ROOT / '04_REMOTE_SENSING'

LANDSAT_DIR = REMOTE_DIR / '03_LANDSAT'

SCREEN_DIR = (
    LANDSAT_DIR
    / '03_RADIOMETRIC_SCREENING'
)

MASTER_PATH = (
    SCREEN_DIR
    / 'LANDSAT_SCREENED_MASTER.csv'
)

OUTPUT_DIR = (
    LANDSAT_DIR
    / '05_HISTORICAL_TRAJECTORIES'
)

TABLE_DIR = (
    REMOTE_DIR
    / '08_OUTPUTS'
    / 'TABLES'
)

FIGURE_DIR = (
    REMOTE_DIR
    / '08_OUTPUTS'
    / 'FIGURES'
)

for directory in [
    OUTPUT_DIR,
    TABLE_DIR,
    FIGURE_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

INDICES = [
    'NDVI',
    'EVI',
    'NDMI',
    'NBR',
    'NBR2'
]

INDEX_COLUMNS = {
    index: f'{index}_mean'
    for index in INDICES
}

SENSORS = [
    'Landsat_5',
    'Landsat_7',
    'Landsat_8',
    'Landsat_9'
]

SENSOR_LABELS = {
    'Landsat_5': 'Landsat 5',
    'Landsat_7': 'Landsat 7',
    'Landsat_8': 'Landsat 8',
    'Landsat_9': 'Landsat 9'
}

SENSOR_COLORS = {
    'Landsat_5': '#527B98',
    'Landsat_7': '#DB9140',
    'Landsat_8': '#24634F',
    'Landsat_9': '#795C92'
}

FOREST_COLORS = {
    'Late_secondary': '#DB9140',
    'Mature': '#24634F'
}

SPATIAL_SCALES = [
    'Fragment',
    'Fragment_core',
    'Plot'
]

FOREST_TYPES = [
    'Late_secondary',
    'Mature'
]

UNIT_MAP = {
    'Fragment': {
        'Late_secondary': 'FRAG_01',
        'Mature': 'FRAG_02'
    },
    'Fragment_core': {
        'Late_secondary': 'FRAG_01_CORE30',
        'Mature': 'FRAG_02_CORE30'
    },
    'Plot': {
        'Late_secondary': 'P01',
        'Mature': 'P02'
    }
}

START_YEAR = 1985
END_YEAR = 2025
CENSUS_YEAR = 2019

PRIMARY_MONTHS = [5, 6, 7, 8, 9]

MIN_SCENES_YEAR = 3
MIN_TREND_YEARS = 8
MIN_TREND_SPAN = 8

N_BOOTSTRAP = 2000

RANDOM_SEED = 20261119

rng = np.random.default_rng(
    RANDOM_SEED
)

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'figure.dpi': 120,
    'savefig.dpi': 600,
    'pdf.fonttype': 42,
    'ps.fonttype': 42
})

print('=' * 78)
print('SATIPO — HISTORICAL FOREST TRAJECTORIES')
print('=' * 78)

# =============================================================================
# 2. VERIFIED INPUT FILE
# =============================================================================

# Exact path confirmed by Cell 90B.
# Do not use the renamed historical master from Cell 91C.

assert MASTER_PATH.exists(), (
    f'Screened master not found: {MASTER_PATH}'
)

native = pd.read_csv(
    MASTER_PATH
)

required_columns = [
    'sensor',
    'scene_id',
    'date',
    'year',
    'month',
    'unit_id',
    'forest_type',
    'spatial_scale',
    'screened_valid_fraction',
    'NDVI_mean',
    'EVI_mean',
    'NDMI_mean',
    'NBR_mean',
    'NBR2_mean'
]

missing = [
    column
    for column in required_columns
    if column not in native.columns
]

assert not missing, (
    f'Missing columns: {missing}'
)

native['date'] = pd.to_datetime(
    native['date'],
    errors='raise'
)

native['year'] = pd.to_numeric(
    native['year'],
    errors='raise'
).astype(int)

native['month'] = pd.to_numeric(
    native['month'],
    errors='raise'
).astype(int)

native['screened_valid_fraction'] = (
    pd.to_numeric(
        native['screened_valid_fraction'],
        errors='coerce'
    )
)

for column in INDEX_COLUMNS.values():

    native[column] = pd.to_numeric(
        native[column],
        errors='coerce'
    )

assert len(native) == 2328, (
    f'Unexpected observation count: {len(native)}'
)

assert native['scene_id'].nunique() == 388

assert native['unit_id'].nunique() == 6

assert not native.duplicated([
    'sensor',
    'scene_id',
    'unit_id'
]).any()

assert (
    native['date'].dt.year
    == native['year']
).all()

assert (
    native['date'].dt.month
    == native['month']
).all()

assert native['year'].min() == START_YEAR

assert native['year'].max() == END_YEAR

assert set(native['sensor']) == set(SENSORS)

assert set(native['forest_type']) == set(
    FOREST_TYPES
)

assert set(native['spatial_scale']) == set(
    SPATIAL_SCALES
)

coverage_values = (
    native['screened_valid_fraction']
    .dropna()
)

assert coverage_values.between(
    -1e-8,
    1 + 1e-6
).all(), 'Invalid coverage fractions'

native['screened_valid_fraction'] = (
    native['screened_valid_fraction']
    .clip(0, 1)
)

for index in [
    'NDVI',
    'NDMI',
    'NBR',
    'NBR2'
]:

    values = (
        native[INDEX_COLUMNS[index]]
        .dropna()
    )

    assert values.between(
        -1 - 1e-6,
        1 + 1e-6
    ).all(), (
        f'Out-of-range screened values: {index}'
    )

# Verify exact spatial identities.

for scale, mapping in UNIT_MAP.items():

    for forest, expected_unit in mapping.items():

        observed_units = native.loc[
            (
                native['spatial_scale'] == scale
            )
            & (
                native['forest_type'] == forest
            ),
            'unit_id'
        ].unique()

        assert set(observed_units) == {
            expected_unit
        }, (
            scale,
            forest,
            observed_units
        )

print('\nINPUT DATA VALIDATED')
print('Input:', MASTER_PATH)
print('Native observations:', len(native))
print('Unique scenes:', native['scene_id'].nunique())
print('Spatial units:', native['unit_id'].nunique())
print(
    'Years:',
    native['year'].min(),
    native['year'].max()
)

# =============================================================================
# 3. CONSTRUCT WITHIN-SCENE FOREST PAIRS
# =============================================================================

pair_keys = [
    'sensor',
    'scene_id',
    'date',
    'year',
    'month',
    'spatial_scale'
]

pair_value_columns = [
    'screened_valid_fraction'
] + list(
    INDEX_COLUMNS.values()
)

secondary = native.loc[
    native['forest_type'] == 'Late_secondary',
    pair_keys + pair_value_columns
].copy()

mature = native.loc[
    native['forest_type'] == 'Mature',
    pair_keys + pair_value_columns
].copy()

secondary = secondary.rename(
    columns={
        column: f'{column}_secondary'
        for column in pair_value_columns
    }
)

mature = mature.rename(
    columns={
        column: f'{column}_mature'
        for column in pair_value_columns
    }
)

historical_pairs = secondary.merge(
    mature,
    on=pair_keys,
    how='inner',
    validate='one_to_one'
)

assert len(historical_pairs) == 1164

historical_pairs = (
    historical_pairs
    .sort_values([
        'spatial_scale',
        'date',
        'sensor'
    ])
    .reset_index(drop=True)
)

historical_pairs['paired_minimum_coverage'] = (
    historical_pairs[
        [
            'screened_valid_fraction_secondary',
            'screened_valid_fraction_mature'
        ]
    ].min(
        axis=1,
        skipna=False
    )
)

historical_pairs['primary_season'] = (
    historical_pairs['month']
    .isin(PRIMARY_MONTHS)
)

historical_pairs['paired_ge50'] = (
    historical_pairs['paired_minimum_coverage']
    >= 0.50
)

historical_pairs['paired_ge70'] = (
    historical_pairs['paired_minimum_coverage']
    >= 0.70
)

historical_pairs['paired_ge90'] = (
    historical_pairs['paired_minimum_coverage']
    >= 0.90
)

for index in INDICES:

    secondary_col = (
        f'{index}_mean_secondary'
    )

    mature_col = (
        f'{index}_mean_mature'
    )

    historical_pairs[
        f'{index}_contrast'
    ] = (
        historical_pairs[secondary_col]
        - historical_pairs[mature_col]
    )

    historical_pairs[
        f'{index}_absolute_contrast'
    ] = (
        historical_pairs[
            f'{index}_contrast'
        ].abs()
    )

print('\nPAIRED DATA')
print('Rows:', len(historical_pairs))

# =============================================================================
# 4. REPRODUCE CELL 90B / 91C RETENTION
# =============================================================================

fragment_pairs = historical_pairs.loc[
    historical_pairs['spatial_scale']
    == 'Fragment'
].copy()

assert len(fragment_pairs) == 388

retention_reproduction = pd.DataFrame([
    {
        'criterion': 'Annual_ge50',
        'n_scenes': int(
            fragment_pairs['paired_ge50'].sum()
        ),
        'expected': 382
    },
    {
        'criterion': 'Annual_ge70',
        'n_scenes': int(
            fragment_pairs['paired_ge70'].sum()
        ),
        'expected': 345
    },
    {
        'criterion': 'Annual_ge90',
        'n_scenes': int(
            fragment_pairs['paired_ge90'].sum()
        ),
        'expected': 269
    },
    {
        'criterion': 'MaySep_ge70',
        'n_scenes': int(
            (
                fragment_pairs['paired_ge70']
                & fragment_pairs['primary_season']
            ).sum()
        ),
        'expected': 219
    }
])

retention_reproduction['verified'] = (
    retention_reproduction['n_scenes']
    == retention_reproduction['expected']
)

assert retention_reproduction[
    'verified'
].all(), (
    'Retention differs from Cell 91C'
)

print('\nRETENTION REPRODUCTION')
display(retention_reproduction)

# =============================================================================
# 5. DEFINE ANALYTICAL SCENARIOS
# =============================================================================

SCENARIOS = [
    {
        'scenario': 'Primary_fragment',
        'scale': 'Fragment',
        'months': [5, 6, 7, 8, 9],
        'coverage': 0.70,
        'exclude_l7_after_2017': False
    },
    {
        'scenario': 'Fragment_ge90',
        'scale': 'Fragment',
        'months': [5, 6, 7, 8, 9],
        'coverage': 0.90,
        'exclude_l7_after_2017': False
    },
    {
        'scenario': 'Fragment_JunSep',
        'scale': 'Fragment',
        'months': [6, 7, 8, 9],
        'coverage': 0.70,
        'exclude_l7_after_2017': False
    },
    {
        'scenario': 'Fragment_annual',
        'scale': 'Fragment',
        'months': list(range(1, 13)),
        'coverage': 0.70,
        'exclude_l7_after_2017': False
    },
    {
        'scenario': 'Fragment_L7_until2017',
        'scale': 'Fragment',
        'months': [5, 6, 7, 8, 9],
        'coverage': 0.70,
        'exclude_l7_after_2017': True
    },
    {
        'scenario': 'Core_ge70',
        'scale': 'Fragment_core',
        'months': [5, 6, 7, 8, 9],
        'coverage': 0.70,
        'exclude_l7_after_2017': False
    },
    {
        'scenario': 'Core_ge90',
        'scale': 'Fragment_core',
        'months': [5, 6, 7, 8, 9],
        'coverage': 0.90,
        'exclude_l7_after_2017': False
    },
    {
        'scenario': 'Plot_ge70',
        'scale': 'Plot',
        'months': [5, 6, 7, 8, 9],
        'coverage': 0.70,
        'exclude_l7_after_2017': False
    },
    {
        'scenario': 'Plot_ge90',
        'scale': 'Plot',
        'months': [5, 6, 7, 8, 9],
        'coverage': 0.90,
        'exclude_l7_after_2017': False
    }
]

scenario_datasets = {}

scenario_support_rows = []

for config in SCENARIOS:

    selected = historical_pairs.loc[
        (
            historical_pairs['spatial_scale']
            == config['scale']
        )
        & (
            historical_pairs['month']
            .isin(config['months'])
        )
        & (
            historical_pairs[
                'paired_minimum_coverage'
            ] >= config['coverage']
        )
    ].copy()

    if config['exclude_l7_after_2017']:

        selected = selected.loc[
            ~(
                (
                    selected['sensor']
                    == 'Landsat_7'
                )
                & (
                    selected['year'] > 2017
                )
            )
        ].copy()

    selected['scenario'] = (
        config['scenario']
    )

    scenario_datasets[
        config['scenario']
    ] = selected

    yearly_counts = (
        selected
        .groupby('year')
        .size()
    )

    scenario_support_rows.append({
        'scenario': config['scenario'],
        'spatial_scale': config['scale'],
        'coverage_threshold': config['coverage'],
        'n_scenes': len(selected),
        'n_years': selected['year'].nunique(),
        'first_year': selected['year'].min(),
        'last_year': selected['year'].max(),
        'years_ge3_scenes': int(
            (yearly_counts >= 3).sum()
        ),
        'median_coverage': selected[
            'paired_minimum_coverage'
        ].median(),
        'exclude_l7_after_2017': (
            config['exclude_l7_after_2017']
        )
    })

scenario_support = pd.DataFrame(
    scenario_support_rows
)

primary = scenario_datasets[
    'Primary_fragment'
].copy()

assert len(primary) == 219

print('\nSCENARIO SUPPORT')
display(
    scenario_support.round(4)
)

print('\nPRIMARY DATASET')
print('Fragment scenes:', len(primary))
print('Historical years:', primary['year'].nunique())

# =============================================================================
# 6. ANNUAL SENSOR-SPECIFIC AGGREGATION
# =============================================================================

# Important:
# Annual forest contrast is the median of within-scene differences.
# It is NOT calculated by subtracting separate annual forest medians.

def summarize_annual(
    dataset,
    scenario_name
):

    rows = []

    grouping = dataset.groupby(
        ['sensor', 'year'],
        sort=True
    )

    for (sensor, year), group in grouping:

        for index in INDICES:

            secondary_col = (
                f'{index}_mean_secondary'
            )

            mature_col = (
                f'{index}_mean_mature'
            )

            contrast_col = (
                f'{index}_contrast'
            )

            valid = group.dropna(
                subset=[
                    secondary_col,
                    mature_col,
                    contrast_col
                ]
            ).copy()

            n = len(valid)

            if n == 0:
                continue

            contrast = valid[
                contrast_col
            ]

            rows.append({
                'scenario': scenario_name,
                'spatial_scale': group[
                    'spatial_scale'
                ].iloc[0],
                'sensor': sensor,
                'year': int(year),
                'index': index,
                'n_scenes': n,
                'adequate_support': (
                    n >= MIN_SCENES_YEAR
                ),
                'secondary_median': valid[
                    secondary_col
                ].median(),
                'mature_median': valid[
                    mature_col
                ].median(),
                'contrast_median': (
                    contrast.median()
                ),
                'contrast_mean': (
                    contrast.mean()
                ),
                'contrast_q25': (
                    contrast.quantile(0.25)
                ),
                'contrast_q75': (
                    contrast.quantile(0.75)
                ),
                'absolute_contrast_median': (
                    contrast.abs().median()
                ),
                'positive_scene_fraction': (
                    (contrast > 0).mean()
                ),
                'negative_scene_fraction': (
                    (contrast < 0).mean()
                ),
                'median_coverage': valid[
                    'paired_minimum_coverage'
                ].median()
            })

    return pd.DataFrame(rows)

annual_frames = []

for name, dataset in scenario_datasets.items():

    annual_frames.append(
        summarize_annual(
            dataset,
            name
        )
    )

annual_sensor = pd.concat(
    annual_frames,
    ignore_index=True
)

annual_primary = annual_sensor.loc[
    annual_sensor['scenario']
    == 'Primary_fragment'
].copy()

annual_support = (
    annual_primary.loc[
        annual_primary['index'] == 'NDVI'
    ]
    .groupby('sensor')
    .agg(
        n_years=('year', 'nunique'),
        adequate_years=(
            'adequate_support',
            'sum'
        ),
        median_scenes=('n_scenes', 'median'),
        first_year=('year', 'min'),
        last_year=('year', 'max')
    )
    .reset_index()
)

print('\nANNUAL SENSOR-SPECIFIC SUPPORT')
display(annual_support)

# =============================================================================
# 7. SENSOR-SPECIFIC TREND ESTIMATION
# =============================================================================

# Theil-Sen slopes use actual calendar years.
#
# Kendall tau is reported as a descriptive measure.
# No ordinary independent-observation p-values are used.
#
# Uncertainty:
# Residual moving-block bootstrap conditional on observed years.
# Blocks are constructed only from consecutive calendar years.
# Missing years are never treated as adjacent calendar years.
#
# The resulting intervals are conditional temporal sensitivity
# intervals, not evidence of replicated forest-type effects.

def make_residual_blocks(
    years,
    residuals,
    block_length
):

    years = np.asarray(
        years,
        dtype=int
    )

    residuals = np.asarray(
        residuals,
        dtype=float
    )

    blocks = []

    n = len(years)

    for start in range(
        n - block_length + 1
    ):

        end = start + block_length

        block_years = years[
            start:end
        ]

        if (
            block_length == 1
            or np.all(
                np.diff(block_years) == 1
            )
        ):

            blocks.append(
                residuals[start:end]
            )

    return blocks


def estimate_temporal_trend(
    years,
    values,
    n_bootstrap=N_BOOTSTRAP,
    seed=RANDOM_SEED
):

    x = np.asarray(
        years,
        dtype=float
    )

    y = np.asarray(
        values,
        dtype=float
    )

    valid = (
        np.isfinite(x)
        & np.isfinite(y)
    )

    x = x[valid]
    y = y[valid]

    order = np.argsort(x)

    x = x[order]
    y = y[order]

    n = len(x)

    span = (
        float(x.max() - x.min())
        if n > 0
        else np.nan
    )

    result = {
        'n_years': n,
        'first_year': (
            int(x.min())
            if n > 0
            else np.nan
        ),
        'last_year': (
            int(x.max())
            if n > 0
            else np.nan
        ),
        'span_years': span,
        'sen_slope_per_year': np.nan,
        'sen_slope_per_decade': np.nan,
        'kendall_tau': np.nan,
        'slope_lcl_95': np.nan,
        'slope_ucl_95': np.nan,
        'bootstrap_positive_pct': np.nan,
        'bootstrap_negative_pct': np.nan,
        'bootstrap_block_length': np.nan,
        'n_residual_blocks': 0,
        'residual_lag1': np.nan,
        'bootstrap_status': 'NOT_RUN',
        'trend_status': 'INSUFFICIENT_SUPPORT'
    }

    if (
        n < MIN_TREND_YEARS
        or span < MIN_TREND_SPAN
    ):

        return result

    assert len(np.unique(x)) == n

    fit = theilslopes(
        y,
        x
    )

    slope = float(fit.slope)

    intercept = float(
        np.median(
            y - slope * x
        )
    )

    tau = float(
        kendalltau(
            x,
            y
        ).statistic
    )

    fitted = (
        intercept + slope * x
    )

    residuals = y - fitted

    residuals = (
        residuals
        - np.mean(residuals)
    )

    adjacent = (
        np.diff(x) == 1
    )

    residual_lag1 = np.nan

    if adjacent.sum() >= 3:

        left = residuals[:-1][
            adjacent
        ]

        right = residuals[1:][
            adjacent
        ]

        if (
            np.std(left) > 0
            and np.std(right) > 0
        ):

            residual_lag1 = float(
                np.corrcoef(
                    left,
                    right
                )[0, 1]
            )

    result.update({
        'sen_slope_per_year': slope,
        'sen_slope_per_decade': (
            slope * 10
        ),
        'kendall_tau': tau,
        'residual_lag1': residual_lag1,
        'trend_status': (
            'DESCRIPTIVE_TREND_ESTIMATED'
        )
    })

    # Prespecified short-block sensitivity.
    # No independent residual bootstrap fallback is used.

    blocks = []

    selected_length = None

    for candidate_length in [3, 2]:

        candidate_blocks = (
            make_residual_blocks(
                x.astype(int),
                residuals,
                candidate_length
            )
        )

        if len(candidate_blocks) >= 4:

            blocks = candidate_blocks

            selected_length = (
                candidate_length
            )

            break

    if selected_length is None:

        result['bootstrap_status'] = (
            'INSUFFICIENT_CONSECUTIVE_BLOCKS'
        )

        return result

    local_rng = np.random.default_rng(
        seed
    )

    bootstrap_slopes = np.empty(
        n_bootstrap,
        dtype=float
    )

    blocks = np.asarray(
        blocks,
        dtype=float
    )

    blocks_per_draw = int(
        np.ceil(
            n / selected_length
        )
    )

    # Vectorized Theil-Sen slope calculation.
    # For distinct x values, the slope is the median
    # of all pairwise slopes.

    ii, jj = np.triu_indices(
        n,
        k=1
    )

    dx = x[jj] - x[ii]

    for b in range(n_bootstrap):

        chosen = local_rng.integers(
            0,
            len(blocks),
            size=blocks_per_draw
        )

        sampled_residuals = (
            blocks[chosen]
            .reshape(-1)[:n]
        )

        y_boot = (
            fitted
            + sampled_residuals
        )

        pairwise_slopes = (
            y_boot[jj]
            - y_boot[ii]
        ) / dx

        bootstrap_slopes[b] = (
            np.median(
                pairwise_slopes
            )
        )

    lcl, ucl = np.quantile(
        bootstrap_slopes,
        [0.025, 0.975]
    )

    result.update({
        'slope_lcl_95': float(lcl),
        'slope_ucl_95': float(ucl),
        'bootstrap_positive_pct': float(
            100 * np.mean(
                bootstrap_slopes > 0
            )
        ),
        'bootstrap_negative_pct': float(
            100 * np.mean(
                bootstrap_slopes < 0
            )
        ),
        'bootstrap_block_length': (
            selected_length
        ),
        'n_residual_blocks': len(blocks),
        'bootstrap_status': (
            'CONDITIONAL_BLOCK_BOOTSTRAP'
        )
    })

    return result

# =============================================================================
# 8. ESTIMATE TRENDS ACROSS ALL SCENARIOS
# =============================================================================

# Signed contrast:
# positive = secondary forest has higher spectral index.
#
# Absolute contrast:
# negative temporal slope = decreasing spectral separation.
# This is a spectral convergence diagnostic, not ecological
# recovery or biomass convergence.

TREND_VARIABLES = {
    'Secondary_native': 'secondary_median',
    'Mature_native': 'mature_median',
    'Signed_contrast': 'contrast_median',
    'Absolute_contrast': (
        'absolute_contrast_median'
    )
}

trend_rows = []

group_columns = [
    'scenario',
    'spatial_scale',
    'sensor',
    'index'
]

trend_counter = 0

for keys, group in annual_sensor.groupby(
    group_columns,
    sort=True
):

    scenario, scale, sensor, index = keys

    adequate = group.loc[
        group['adequate_support']
    ].sort_values('year')

    for variable, column in (
        TREND_VARIABLES.items()
    ):

        trend_counter += 1

        # Bootstrap contrast trends for the main scenario.
        # Other scenarios are descriptive slope sensitivity.
        # This avoids unnecessary computational duplication.

        use_bootstrap = (
            scenario == 'Primary_fragment'
            and variable in [
                'Signed_contrast',
                'Absolute_contrast'
            ]
        )

        if use_bootstrap:

            result = estimate_temporal_trend(
                adequate['year'],
                adequate[column],
                n_bootstrap=N_BOOTSTRAP,
                seed=(
                    RANDOM_SEED
                    + trend_counter
                )
            )

        else:

            # Point estimates only.

            x = adequate[
                'year'
            ].to_numpy(dtype=float)

            y = adequate[
                column
            ].to_numpy(dtype=float)

            finite = (
                np.isfinite(x)
                & np.isfinite(y)
            )

            x = x[finite]
            y = y[finite]

            n = len(x)

            span = (
                float(
                    x.max() - x.min()
                )
                if n > 0
                else np.nan
            )

            result = {
                'n_years': n,
                'first_year': (
                    int(x.min())
                    if n > 0
                    else np.nan
                ),
                'last_year': (
                    int(x.max())
                    if n > 0
                    else np.nan
                ),
                'span_years': span,
                'sen_slope_per_year': np.nan,
                'sen_slope_per_decade': np.nan,
                'kendall_tau': np.nan,
                'slope_lcl_95': np.nan,
                'slope_ucl_95': np.nan,
                'bootstrap_positive_pct': np.nan,
                'bootstrap_negative_pct': np.nan,
                'bootstrap_block_length': np.nan,
                'n_residual_blocks': 0,
                'residual_lag1': np.nan,
                'bootstrap_status': (
                    'POINT_ESTIMATE_ONLY'
                ),
                'trend_status': (
                    'INSUFFICIENT_SUPPORT'
                )
            }

            if (
                n >= MIN_TREND_YEARS
                and span >= MIN_TREND_SPAN
            ):

                slope = float(
                    theilslopes(y, x).slope
                )

                result.update({
                    'sen_slope_per_year': (
                        slope
                    ),
                    'sen_slope_per_decade': (
                        slope * 10
                    ),
                    'kendall_tau': float(
                        kendalltau(
                            x,
                            y
                        ).statistic
                    ),
                    'trend_status': (
                        'DESCRIPTIVE_TREND_ESTIMATED'
                    )
                })

        trend_rows.append({
            'scenario': scenario,
            'spatial_scale': scale,
            'sensor': sensor,
            'index': index,
            'variable': variable,
            **result
        })

historical_trends = pd.DataFrame(
    trend_rows
)

primary_trends = historical_trends.loc[
    historical_trends['scenario']
    == 'Primary_fragment'
].copy()

primary_contrast_trends = (
    primary_trends.loc[
        primary_trends['variable'].isin([
            'Signed_contrast',
            'Absolute_contrast'
        ])
    ].copy()
)

print('\n' + '=' * 78)
print('PRIMARY SENSOR-SPECIFIC CONTRAST TRENDS')
print('=' * 78)

display(
    primary_contrast_trends[
        [
            'sensor',
            'index',
            'variable',
            'n_years',
            'sen_slope_per_decade',
            'kendall_tau',
            'slope_lcl_95',
            'slope_ucl_95',
            'bootstrap_status'
        ]
    ].round(5)
)

# =============================================================================
# 9. HISTORICAL CONTRAST PERSISTENCE
# =============================================================================

# Equal-year summaries prevent years with many scenes
# from dominating the historical description.

persistence_rows = []

for (
    scenario,
    scale,
    sensor,
    index
), group in annual_sensor.groupby(
    group_columns
):

    adequate = group.loc[
        group['adequate_support']
    ].copy()

    if len(adequate) == 0:
        continue

    values = adequate[
        'contrast_median'
    ]

    abs_values = adequate[
        'absolute_contrast_median'
    ]

    persistence_rows.append({
        'scenario': scenario,
        'spatial_scale': scale,
        'sensor': sensor,
        'index': index,
        'n_adequate_years': len(adequate),
        'first_year': adequate['year'].min(),
        'last_year': adequate['year'].max(),
        'median_signed_contrast': (
            values.median()
        ),
        'signed_contrast_q25': (
            values.quantile(0.25)
        ),
        'signed_contrast_q75': (
            values.quantile(0.75)
        ),
        'median_absolute_contrast': (
            abs_values.median()
        ),
        'positive_years': int(
            (values > 0).sum()
        ),
        'negative_years': int(
            (values < 0).sum()
        ),
        'zero_years': int(
            (values == 0).sum()
        ),
        'positive_year_pct': (
            100 * (values > 0).mean()
        ),
        'negative_year_pct': (
            100 * (values < 0).mean()
        ),
        'median_positive_scene_fraction': (
            adequate[
                'positive_scene_fraction'
            ].median()
        )
    })

historical_persistence = pd.DataFrame(
    persistence_rows
)

primary_persistence = (
    historical_persistence.loc[
        historical_persistence['scenario']
        == 'Primary_fragment'
    ].copy()
)

print('\n' + '=' * 78)
print('HISTORICAL CONTRAST PERSISTENCE')
print('=' * 78)

display(
    primary_persistence[
        [
            'sensor',
            'index',
            'n_adequate_years',
            'median_signed_contrast',
            'median_absolute_contrast',
            'positive_year_pct'
        ]
    ].round(5)
)

# =============================================================================
# 10. CROSS-SENSOR ANNUAL CONTRAST SYNTHESIS
# =============================================================================

# Secondary descriptive analysis only.
#
# We combine within-scene contrasts, not absolute index values.
# Each eligible sensor-year receives equal weight.
#
# This does not imply full cross-sensor harmonization.
# Sensor-specific contrasts may still differ due to spectral
# response, observation geometry, masks and acquisition timing.

consensus_rows = []

for (
    scenario,
    scale,
    year,
    index
), group in annual_sensor.groupby([
    'scenario',
    'spatial_scale',
    'year',
    'index'
]):

    adequate = group.loc[
        group['adequate_support']
    ].copy()

    if len(adequate) == 0:
        continue

    signed = adequate[
        'contrast_median'
    ]

    absolute = adequate[
        'absolute_contrast_median'
    ]

    consensus_rows.append({
        'scenario': scenario,
        'spatial_scale': scale,
        'year': int(year),
        'index': index,
        'n_sensors': (
            adequate['sensor'].nunique()
        ),
        'n_total_scenes': int(
            adequate['n_scenes'].sum()
        ),
        'sensors': ';'.join(
            sorted(
                adequate['sensor'].unique()
            )
        ),
        'equal_sensor_signed_contrast': (
            signed.median()
        ),
        'equal_sensor_absolute_contrast': (
            absolute.median()
        ),
        'minimum_sensor_contrast': (
            signed.min()
        ),
        'maximum_sensor_contrast': (
            signed.max()
        ),
        'between_sensor_range': (
            signed.max() - signed.min()
        ),
        'all_sensors_positive': bool(
            (signed > 0).all()
        ),
        'all_sensors_negative': bool(
            (signed < 0).all()
        )
    })

annual_consensus = pd.DataFrame(
    consensus_rows
)

primary_consensus = (
    annual_consensus.loc[
        annual_consensus['scenario']
        == 'Primary_fragment'
    ].copy()
)

# Descriptive historical slope of paired contrasts.
# No bootstrap or p-value is attached to this cross-sensor
# synthesis because changing sensor composition may confound
# long-term trends.

consensus_trend_rows = []

for (
    scenario,
    scale,
    index
), group in annual_consensus.groupby([
    'scenario',
    'spatial_scale',
    'index'
]):

    group = group.sort_values('year')

    for variable, column in [
        (
            'Signed_contrast',
            'equal_sensor_signed_contrast'
        ),
        (
            'Absolute_contrast',
            'equal_sensor_absolute_contrast'
        )
    ]:

        x = group[
            'year'
        ].to_numpy(dtype=float)

        y = group[
            column
        ].to_numpy(dtype=float)

        valid = (
            np.isfinite(x)
            & np.isfinite(y)
        )

        x = x[valid]
        y = y[valid]

        n = len(x)

        slope = np.nan
        tau = np.nan

        if (
            n >= MIN_TREND_YEARS
            and (
                x.max() - x.min()
            ) >= MIN_TREND_SPAN
        ):

            slope = float(
                theilslopes(
                    y,
                    x
                ).slope
            )

            tau = float(
                kendalltau(
                    x,
                    y
                ).statistic
            )

        consensus_trend_rows.append({
            'scenario': scenario,
            'spatial_scale': scale,
            'index': index,
            'variable': variable,
            'n_years': n,
            'first_year': (
                int(x.min())
                if n > 0
                else np.nan
            ),
            'last_year': (
                int(x.max())
                if n > 0
                else np.nan
            ),
            'sen_slope_per_year': slope,
            'sen_slope_per_decade': (
                slope * 10
            ),
            'kendall_tau': tau,
            'interpretation': (
                'DESCRIPTIVE_SENSOR_MIXTURE'
            )
        })

consensus_trends = pd.DataFrame(
    consensus_trend_rows
)

print('\nCROSS-SENSOR CONTRAST SYNTHESIS')
print(
    'Adequately supported primary years:',
    primary_consensus['year'].nunique()
)

display(
    consensus_trends.loc[
        consensus_trends['scenario']
        == 'Primary_fragment'
    ].round(5)
)

# =============================================================================
# 11. EARLY VS LATE PERIOD CONTRASTS
# =============================================================================

# Within-sensor comparison only.
# Compare the first and last five adequately supported years.
# Require at least ten adequately supported years.
# No arbitrary pre/post disturbance classification.

period_rows = []

for (
    scenario,
    scale,
    sensor,
    index
), group in annual_sensor.groupby(
    group_columns
):

    adequate = group.loc[
        group['adequate_support']
    ].sort_values('year')

    if len(adequate) < 10:
        continue

    early = adequate.head(5)

    late = adequate.tail(5)

    assert set(
        early['year']
    ).isdisjoint(
        set(late['year'])
    )

    early_signed = early[
        'contrast_median'
    ].median()

    late_signed = late[
        'contrast_median'
    ].median()

    early_abs = early[
        'absolute_contrast_median'
    ].median()

    late_abs = late[
        'absolute_contrast_median'
    ].median()

    period_rows.append({
        'scenario': scenario,
        'spatial_scale': scale,
        'sensor': sensor,
        'index': index,
        'early_years': ';'.join(
            early['year'].astype(str)
        ),
        'late_years': ';'.join(
            late['year'].astype(str)
        ),
        'early_signed_contrast': (
            early_signed
        ),
        'late_signed_contrast': (
            late_signed
        ),
        'signed_contrast_change': (
            late_signed - early_signed
        ),
        'early_absolute_contrast': (
            early_abs
        ),
        'late_absolute_contrast': (
            late_abs
        ),
        'absolute_contrast_change': (
            late_abs - early_abs
        ),
        'absolute_contrast_change_pct': (
            100 * (
                late_abs / early_abs - 1
            )
            if early_abs > 1e-8
            else np.nan
        ),
        'spectral_separation_direction': (
            'Decreased'
            if late_abs < early_abs
            else 'Increased'
            if late_abs > early_abs
            else 'Unchanged'
        )
    })

period_comparison = pd.DataFrame(
    period_rows
)

print('\n' + '=' * 78)
print('EARLY VS LATE WITHIN-SENSOR COMPARISON')
print('=' * 78)

display(
    period_comparison.loc[
        period_comparison['scenario']
        == 'Primary_fragment'
    ].round(5)
)

# =============================================================================
# 12. MATCHED SPATIAL-SCALE SENSITIVITY
# =============================================================================

# Compare fragment, core and field-plot contrasts using
# exactly the same acquisitions.
#
# These are nested spatial units, not independent replicates.

spatial_sensitivity_rows = []

spatial_match_frames = []

for threshold in [0.70, 0.90]:

    selected = historical_pairs.loc[
        historical_pairs['primary_season']
        & (
            historical_pairs[
                'paired_minimum_coverage'
            ] >= threshold
        )
    ].copy()

    for index in INDICES:

        wide = selected.pivot(
            index=[
                'sensor',
                'scene_id',
                'date',
                'year'
            ],
            columns='spatial_scale',
            values=f'{index}_contrast'
        )

        for comparison in [
            'Fragment_core',
            'Plot'
        ]:

            matched = wide[
                [
                    'Fragment',
                    comparison
                ]
            ].dropna().reset_index()

            if len(matched) == 0:
                continue

            matched['index'] = index

            matched[
                'coverage_threshold'
            ] = threshold

            matched[
                'comparison_scale'
            ] = comparison

            matched[
                'fragment_contrast'
            ] = matched['Fragment']

            matched[
                'comparison_contrast'
            ] = matched[comparison]

            matched[
                'contrast_difference'
            ] = (
                matched[comparison]
                - matched['Fragment']
            )

            matched[
                'same_sign'
            ] = (
                np.sign(
                    matched[comparison]
                )
                == np.sign(
                    matched['Fragment']
                )
            )

            spatial_match_frames.append(
                matched[
                    [
                        'sensor',
                        'scene_id',
                        'date',
                        'year',
                        'index',
                        'coverage_threshold',
                        'comparison_scale',
                        'fragment_contrast',
                        'comparison_contrast',
                        'contrast_difference',
                        'same_sign'
                    ]
                ]
            )

            sensor_groups = [
                ('All_sensors', matched)
            ] + list(
                matched.groupby('sensor')
            )

            for sensor, group in (
                sensor_groups
            ):

                a = group[
                    'fragment_contrast'
                ]

                b = group[
                    'comparison_contrast'
                ]

                correlation = (
                    a.corr(
                        b,
                        method='spearman'
                    )
                    if (
                        len(group) >= 3
                        and a.nunique() > 1
                        and b.nunique() > 1
                    )
                    else np.nan
                )

                spatial_sensitivity_rows.append({
                    'coverage_threshold': (
                        threshold
                    ),
                    'comparison_scale': (
                        comparison
                    ),
                    'sensor': sensor,
                    'index': index,
                    'n_matched_scenes': (
                        len(group)
                    ),
                    'fragment_median': (
                        a.median()
                    ),
                    'comparison_median': (
                        b.median()
                    ),
                    'median_difference': (
                        (b - a).median()
                    ),
                    'mae': (
                        (b - a).abs().mean()
                    ),
                    'spearman_rho': (
                        correlation
                    ),
                    'same_sign_pct': (
                        100 * group[
                            'same_sign'
                        ].mean()
                    )
                })

spatial_sensitivity = pd.DataFrame(
    spatial_sensitivity_rows
)

spatial_matched = pd.concat(
    spatial_match_frames,
    ignore_index=True
)

print('\nMATCHED SPATIAL SENSITIVITY')

display(
    spatial_sensitivity.loc[
        (
            spatial_sensitivity['sensor']
            == 'All_sensors'
        )
        & (
            spatial_sensitivity[
                'coverage_threshold'
            ] == 0.70
        )
    ].round(5)
)

# =============================================================================
# 13. TREND ROBUSTNESS ACROSS SCENARIOS
# =============================================================================

primary_reference = (
    primary_contrast_trends[
        [
            'sensor',
            'index',
            'variable',
            'sen_slope_per_year',
            'n_years'
        ]
    ]
    .rename(columns={
        'sen_slope_per_year': (
            'primary_slope'
        ),
        'n_years': (
            'primary_n_years'
        )
    })
)

trend_sensitivity = (
    historical_trends.loc[
        historical_trends['variable'].isin([
            'Signed_contrast',
            'Absolute_contrast'
        ])
    ]
    .merge(
        primary_reference,
        on=[
            'sensor',
            'index',
            'variable'
        ],
        how='left',
        validate='many_to_one'
    )
)

trend_sensitivity['slope_difference'] = (
    trend_sensitivity[
        'sen_slope_per_year'
    ]
    - trend_sensitivity[
        'primary_slope'
    ]
)

trend_sensitivity[
    'evaluable_comparison'
] = (
    trend_sensitivity[
        'sen_slope_per_year'
    ].notna()
    & trend_sensitivity[
        'primary_slope'
    ].notna()
)

trend_sensitivity[
    'same_direction'
] = np.where(
    trend_sensitivity[
        'evaluable_comparison'
    ],
    (
        np.sign(
            trend_sensitivity[
                'sen_slope_per_year'
            ]
        )
        == np.sign(
            trend_sensitivity[
                'primary_slope'
            ]
        )
    ).astype(float),
    np.nan
)

robustness_rows = []

for (
    sensor,
    index,
    variable
), group in trend_sensitivity.groupby([
    'sensor',
    'index',
    'variable'
]):

    sensitivity = group.loc[
        (
            group['scenario']
            != 'Primary_fragment'
        )
        & group[
            'evaluable_comparison'
        ]
    ]

    if len(sensitivity) == 0:
        continue

    robustness_rows.append({
        'sensor': sensor,
        'index': index,
        'variable': variable,
        'primary_slope_per_decade': (
            group['primary_slope']
            .iloc[0] * 10
        ),
        'n_evaluable_scenarios': (
            len(sensitivity)
        ),
        'n_same_direction': int(
            sensitivity[
                'same_direction'
            ].sum()
        ),
        'same_direction_pct': (
            100 * sensitivity[
                'same_direction'
            ].mean()
        ),
        'minimum_sensitivity_slope_decade': (
            sensitivity[
                'sen_slope_per_year'
            ].min() * 10
        ),
        'maximum_sensitivity_slope_decade': (
            sensitivity[
                'sen_slope_per_year'
            ].max() * 10
        )
    })

trend_robustness = pd.DataFrame(
    robustness_rows
)

print('\n' + '=' * 78)
print('TREND DIRECTION ROBUSTNESS')
print('=' * 78)

display(
    trend_robustness.round(5)
)

# =============================================================================
# 14. FIELD-CENSUS YEAR CONTEXT — 2019
# =============================================================================

field_2019 = annual_sensor.loc[
    (
        annual_sensor['year']
        == CENSUS_YEAR
    )
    & annual_sensor['scenario'].isin([
        'Primary_fragment',
        'Core_ge70',
        'Plot_ge70'
    ])
].copy()

field_2019 = field_2019.sort_values([
    'spatial_scale',
    'sensor',
    'index'
])

field_scene_2019 = (
    historical_pairs.loc[
        (
            historical_pairs['year']
            == CENSUS_YEAR
        )
        & historical_pairs[
            'primary_season'
        ]
        & historical_pairs[
            'paired_ge70'
        ]
    ].copy()
)

print('\n' + '=' * 78)
print('2019 FIELD-CENSUS SPECTRAL CONTEXT')
print('=' * 78)

display(
    field_2019[
        [
            'spatial_scale',
            'sensor',
            'index',
            'n_scenes',
            'secondary_median',
            'mature_median',
            'contrast_median',
            'absolute_contrast_median'
        ]
    ].round(5)
)

# =============================================================================
# 15. CENSUS-YEAR HISTORICAL POSITION
# =============================================================================

# Descriptive historical percentile of the 2019 contrast
# relative to adequately supported years from the same sensor.
#
# This is not a statistical test of an anomalous census year.

census_position_rows = []

for (
    scenario,
    scale,
    sensor,
    index
), group in annual_sensor.groupby(
    group_columns
):

    if scenario not in [
        'Primary_fragment',
        'Core_ge70',
        'Plot_ge70'
    ]:
        continue

    adequate = group.loc[
        group['adequate_support']
    ].copy()

    current = adequate.loc[
        adequate['year']
        == CENSUS_YEAR
    ]

    reference = adequate.loc[
        adequate['year']
        != CENSUS_YEAR
    ]

    if (
        len(current) != 1
        or len(reference) < 5
    ):
        continue

    value = float(
        current[
            'contrast_median'
        ].iloc[0]
    )

    ref = reference[
        'contrast_median'
    ].to_numpy(dtype=float)

    percentile = (
        100 * (
            np.sum(ref < value)
            + 0.5 * np.sum(ref == value)
        ) / len(ref)
    )

    census_position_rows.append({
        'scenario': scenario,
        'spatial_scale': scale,
        'sensor': sensor,
        'index': index,
        'census_year': CENSUS_YEAR,
        'census_contrast': value,
        'n_reference_years': len(ref),
        'historical_median_contrast': (
            np.median(ref)
        ),
        'historical_q25': (
            np.quantile(ref, 0.25)
        ),
        'historical_q75': (
            np.quantile(ref, 0.75)
        ),
        'census_percentile': percentile,
        'difference_from_historical_median': (
            value - np.median(ref)
        )
    })

census_historical_position = (
    pd.DataFrame(
        census_position_rows
    )
)

print('\n2019 HISTORICAL POSITION')

display(
    census_historical_position.round(5)
)

# =============================================================================
# 16. HISTORICAL EXTREME-OBSERVATION AUDIT
# =============================================================================

# Identify unusual low index observations using sensor-
# and forest-specific robust standardized residuals.
#
# These are diagnostic flags only.
# They are NOT automatically removed and are NOT interpreted
# as confirmed forest disturbances.

extreme_rows = []

for sensor, group in primary.groupby(
    'sensor'
):

    for forest_suffix in [
        'secondary',
        'mature'
    ]:

        for index in INDICES:

            column = (
                f'{index}_mean_{forest_suffix}'
            )

            values = group[
                column
            ].dropna()

            if len(values) < 10:
                continue

            median = float(
                values.median()
            )

            mad = float(
                np.median(
                    np.abs(
                        values - median
                    )
                )
            )

            if mad <= 1e-10:
                continue

            robust_scale = (
                1.4826 * mad
            )

            for _, row in group.iterrows():

                value = row[column]

                if not np.isfinite(value):
                    continue

                robust_z = (
                    value - median
                ) / robust_scale

                if robust_z <= -3.5:

                    extreme_rows.append({
                        'sensor': sensor,
                        'scene_id': row[
                            'scene_id'
                        ],
                        'date': row['date'],
                        'year': row['year'],
                        'forest': forest_suffix,
                        'index': index,
                        'value': value,
                        'sensor_forest_median': (
                            median
                        ),
                        'robust_z': robust_z,
                        'paired_coverage': row[
                            'paired_minimum_coverage'
                        ],
                        'status': (
                            'REVIEW_ONLY_NOT_REMOVED'
                        )
                    })

extreme_observations = pd.DataFrame(
    extreme_rows,
    columns=[
        'sensor',
        'scene_id',
        'date',
        'year',
        'forest',
        'index',
        'value',
        'sensor_forest_median',
        'robust_z',
        'paired_coverage',
        'status'
    ]
)

print('\nEXTREME OBSERVATION AUDIT')
print(
    'Flagged index observations:',
    len(extreme_observations)
)

print(
    'Unique flagged acquisitions:',
    extreme_observations[
        'scene_id'
    ].nunique()
)

print('Observations removed: 0')

if len(extreme_observations) > 0:

    display(
        extreme_observations
        .sort_values('robust_z')
        .head(15)
        .round(5)
    )

# =============================================================================
# 17. FIGURE EXPORT FUNCTION
# =============================================================================

def save_figure(
    fig,
    stem
):

    for extension in [
        'png',
        'pdf',
        'svg'
    ]:

        fig.savefig(
            FIGURE_DIR / (
                f'{stem}.{extension}'
            ),
            bbox_inches='tight',
            dpi=600,
            facecolor='white'
        )

    plt.show()

    plt.close(fig)

# =============================================================================
# 18. FIGURE 1 — NATIVE SENSOR-SPECIFIC TRAJECTORIES
# =============================================================================

fig, axes = plt.subplots(
    5,
    2,
    figsize=(16, 15),
    sharex=True
)

for row_number, index in enumerate(
    INDICES
):

    for column_number, forest in enumerate([
        'secondary',
        'mature'
    ]):

        ax = axes[
            row_number,
            column_number
        ]

        value_column = (
            f'{forest}_median'
        )

        for sensor in SENSORS:

            data = annual_primary.loc[
                (
                    annual_primary['index']
                    == index
                )
                & (
                    annual_primary['sensor']
                    == sensor
                )
            ].sort_values('year')

            adequate = data.loc[
                data['adequate_support']
            ]

            sparse = data.loc[
                ~data['adequate_support']
            ]

            ax.scatter(
                adequate['year'],
                adequate[value_column],
                s=35,
                color=SENSOR_COLORS[sensor],
                label=SENSOR_LABELS[sensor],
                zorder=3
            )

            ax.scatter(
                sparse['year'],
                sparse[value_column],
                s=30,
                facecolors='none',
                edgecolors=SENSOR_COLORS[sensor],
                linewidths=0.9,
                alpha=0.75,
                zorder=2
            )

            # Connect consecutive adequately supported
            # years only, within the same sensor.

            for k in range(
                len(adequate) - 1
            ):

                pair = adequate.iloc[
                    k:k + 2
                ]

                if (
                    pair['year'].iloc[1]
                    - pair['year'].iloc[0]
                ) == 1:

                    ax.plot(
                        pair['year'],
                        pair[value_column],
                        color=SENSOR_COLORS[sensor],
                        alpha=0.45,
                        linewidth=1
                    )

        ax.axvline(
            CENSUS_YEAR,
            color='#D94B45',
            linestyle='--',
            linewidth=1
        )

        ax.grid(
            alpha=0.15
        )

        ax.set_ylabel(index)

        ax.set_xlim(
            START_YEAR - 0.7,
            END_YEAR + 0.7
        )

        if row_number == 0:

            ax.set_title(
                'Late-secondary forest'
                if forest == 'secondary'
                else 'Mature forest'
            )

        if row_number == 4:

            ax.set_xlabel('Year')

handles, labels = axes[
    0, 0
].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.958),
    ncol=4,
    frameon=False
)

fig.suptitle(
    'Historical native Landsat spectral trajectories\n'
    'May–September | Paired valid coverage ≥70%',
    fontsize=17,
    fontweight='bold',
    y=0.995
)

fig.text(
    0.5,
    0.012,
    'Filled symbols: ≥3 acquisitions per sensor-year; '
    'open symbols: 1–2 acquisitions. '
    'Dashed red line: 2019 field census.',
    ha='center',
    fontsize=10
)

fig.tight_layout(
    rect=[0, 0.035, 1, 0.93]
)

save_figure(
    fig,
    'FIG_HISTORICAL_NATIVE_TRAJECTORIES'
)

# =============================================================================
# 19. FIGURE 2 — HISTORICAL SPECTRAL CONTRASTS
# =============================================================================

fig, axes = plt.subplots(
    5,
    2,
    figsize=(16, 15),
    sharex=True
)

for row_number, index in enumerate(
    INDICES
):

    for column_number, variable in enumerate([
        'Signed_contrast',
        'Absolute_contrast'
    ]):

        ax = axes[
            row_number,
            column_number
        ]

        value_column = (
            'contrast_median'
            if variable == 'Signed_contrast'
            else 'absolute_contrast_median'
        )

        for sensor in SENSORS:

            data = annual_primary.loc[
                (
                    annual_primary['index']
                    == index
                )
                & (
                    annual_primary['sensor']
                    == sensor
                )
            ].sort_values('year')

            adequate = data.loc[
                data['adequate_support']
            ]

            sparse = data.loc[
                ~data['adequate_support']
            ]

            ax.scatter(
                adequate['year'],
                adequate[value_column],
                color=SENSOR_COLORS[sensor],
                s=35,
                label=SENSOR_LABELS[sensor],
                zorder=3
            )

            ax.scatter(
                sparse['year'],
                sparse[value_column],
                facecolors='none',
                edgecolors=SENSOR_COLORS[sensor],
                s=28,
                alpha=0.75,
                linewidths=0.9
            )

            if variable == 'Signed_contrast':

                lower = adequate[
                    'contrast_median'
                ] - adequate[
                    'contrast_q25'
                ]

                upper = adequate[
                    'contrast_q75'
                ] - adequate[
                    'contrast_median'
                ]

                ax.errorbar(
                    adequate['year'],
                    adequate[
                        'contrast_median'
                    ],
                    yerr=[
                        lower.clip(lower=0),
                        upper.clip(lower=0)
                    ],
                    fmt='none',
                    ecolor=SENSOR_COLORS[sensor],
                    alpha=0.35,
                    linewidth=0.9,
                    zorder=1
                )

            trend = (
                primary_contrast_trends.loc[
                    (
                        primary_contrast_trends[
                            'sensor'
                        ] == sensor
                    )
                    & (
                        primary_contrast_trends[
                            'index'
                        ] == index
                    )
                    & (
                        primary_contrast_trends[
                            'variable'
                        ] == variable
                    )
                ]
            )

            if (
                len(trend) == 1
                and pd.notna(
                    trend[
                        'sen_slope_per_year'
                    ].iloc[0]
                )
            ):

                slope = float(
                    trend[
                        'sen_slope_per_year'
                    ].iloc[0]
                )

                x = adequate[
                    'year'
                ].to_numpy()

                y = adequate[
                    value_column
                ].to_numpy()

                intercept = np.median(
                    y - slope * x
                )

                line_x = np.array([
                    x.min(),
                    x.max()
                ])

                ax.plot(
                    line_x,
                    intercept + slope * line_x,
                    color=SENSOR_COLORS[sensor],
                    linewidth=1.5,
                    alpha=0.8
                )

        ax.axhline(
            0,
            color='gray',
            linestyle='--',
            linewidth=0.8
        )

        ax.axvline(
            CENSUS_YEAR,
            color='#D94B45',
            linestyle='--',
            linewidth=1
        )

        ax.grid(alpha=0.15)

        ax.set_ylabel(
            f'Δ{index}'
            if variable == 'Signed_contrast'
            else f'|Δ{index}|'
        )

        if row_number == 0:

            ax.set_title(
                'Signed forest contrast'
                if column_number == 0
                else 'Absolute spectral separation'
            )

        if row_number == 4:

            ax.set_xlabel('Year')

handles, labels = axes[
    0, 0
].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.958),
    ncol=4,
    frameon=False
)

fig.suptitle(
    'Historical within-scene forest contrasts\n'
    'Sensor-specific trajectories and spectral separation',
    fontsize=17,
    fontweight='bold',
    y=0.995
)

fig.text(
    0.5,
    0.012,
    'Δ = late-secondary minus mature forest. '
    'Error bars show within-year interquartile ranges, '
    'not confidence intervals.',
    ha='center',
    fontsize=10
)

fig.tight_layout(
    rect=[0, 0.035, 1, 0.93]
)

save_figure(
    fig,
    'FIG_HISTORICAL_CONTRAST_TRAJECTORIES'
)

# =============================================================================
# 20. FIGURE 3 — TREND MAGNITUDES AND UNCERTAINTY
# =============================================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(15, 7),
    sharey=True
)

trend_plot_sensors = [
    'Landsat_5',
    'Landsat_7',
    'Landsat_8'
]

offsets = {
    'Landsat_5': -0.22,
    'Landsat_7': 0.0,
    'Landsat_8': 0.22
}

for ax, variable in zip(
    axes,
    [
        'Signed_contrast',
        'Absolute_contrast'
    ]
):

    for i, index in enumerate(
        INDICES
    ):

        for sensor in trend_plot_sensors:

            row = (
                primary_contrast_trends.loc[
                    (
                        primary_contrast_trends[
                            'sensor'
                        ] == sensor
                    )
                    & (
                        primary_contrast_trends[
                            'index'
                        ] == index
                    )
                    & (
                        primary_contrast_trends[
                            'variable'
                        ] == variable
                    )
                ]
            )

            if len(row) != 1:
                continue

            row = row.iloc[0]

            slope = row[
                'sen_slope_per_decade'
            ]

            if not np.isfinite(slope):
                continue

            y_position = (
                i + offsets[sensor]
            )

            lower = (
                row['slope_lcl_95'] * 10
            )

            upper = (
                row['slope_ucl_95'] * 10
            )

            if (
                np.isfinite(lower)
                and np.isfinite(upper)
            ):

                ax.hlines(
                    y_position,
                    lower,
                    upper,
                    color=SENSOR_COLORS[sensor],
                    linewidth=1.8
                )

            ax.scatter(
                slope,
                y_position,
                color=SENSOR_COLORS[sensor],
                s=55,
                zorder=3
            )

    ax.axvline(
        0,
        color='gray',
        linestyle='--',
        linewidth=1
    )

    ax.set_yticks(
        range(len(INDICES))
    )

    ax.set_yticklabels(
        INDICES
    )

    ax.set_xlabel(
        'Theil–Sen slope per decade'
    )

    ax.set_title(
        'Signed contrast'
        if variable == 'Signed_contrast'
        else 'Absolute spectral separation'
    )

    ax.grid(
        axis='x',
        alpha=0.2
    )

axes[0].invert_yaxis()

legend_handles = [
    plt.Line2D(
        [0],
        [0],
        marker='o',
        linestyle='none',
        color=SENSOR_COLORS[sensor],
        label=SENSOR_LABELS[sensor],
        markersize=7
    )
    for sensor in trend_plot_sensors
]

fig.legend(
    handles=legend_handles,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.91),
    ncol=3,
    frameon=False
)

fig.suptitle(
    'Sensor-specific temporal trends in forest contrasts',
    fontsize=16,
    fontweight='bold'
)

fig.text(
    0.5,
    0.015,
    'Intervals: conditional 95% residual-block bootstrap '
    'where consecutive-year support permits. '
    'Landsat 9 has insufficient trend support.',
    ha='center',
    fontsize=9
)

fig.tight_layout(
    rect=[0, 0.06, 1, 0.86]
)

save_figure(
    fig,
    'FIG_HISTORICAL_CONTRAST_TRENDS'
)

# =============================================================================
# 21. FIGURE 4 — MATCHED SPATIAL SENSITIVITY
# =============================================================================

fig, axes = plt.subplots(
    2,
    5,
    figsize=(17, 7)
)

for row_number, comparison in enumerate([
    'Fragment_core',
    'Plot'
]):

    for col_number, index in enumerate(
        INDICES
    ):

        ax = axes[
            row_number,
            col_number
        ]

        data = spatial_matched.loc[
            (
                spatial_matched[
                    'comparison_scale'
                ] == comparison
            )
            & (
                spatial_matched['index']
                == index
            )
            & (
                spatial_matched[
                    'coverage_threshold'
                ] == 0.70
            )
        ]

        for sensor in SENSORS:

            group = data.loc[
                data['sensor'] == sensor
            ]

            ax.scatter(
                group['fragment_contrast'],
                group['comparison_contrast'],
                s=18,
                color=SENSOR_COLORS[sensor],
                alpha=0.65
            )

        if len(data) > 0:

            limits = np.array([
                data[
                    [
                        'fragment_contrast',
                        'comparison_contrast'
                    ]
                ].min().min(),
                data[
                    [
                        'fragment_contrast',
                        'comparison_contrast'
                    ]
                ].max().max()
            ])

            padding = max(
                0.005,
                0.05 * (
                    limits[1] - limits[0]
                )
            )

            limits = limits + np.array([
                -padding,
                padding
            ])

            ax.plot(
                limits,
                limits,
                color='gray',
                linestyle='--',
                linewidth=0.9
            )

            ax.set_xlim(limits)
            ax.set_ylim(limits)

        ax.axhline(
            0,
            color='gray',
            linewidth=0.5,
            alpha=0.5
        )

        ax.axvline(
            0,
            color='gray',
            linewidth=0.5,
            alpha=0.5
        )

        ax.set_title(
            f'{index} | n = {len(data)}',
            fontsize=11
        )

        ax.set_xlabel(
            'Fragment contrast'
        )

        if col_number == 0:

            ax.set_ylabel(
                'Core contrast'
                if comparison == 'Fragment_core'
                else 'Field-plot contrast'
            )

        ax.grid(alpha=0.15)

legend_handles = [
    plt.Line2D(
        [0],
        [0],
        marker='o',
        linestyle='none',
        color=SENSOR_COLORS[sensor],
        label=SENSOR_LABELS[sensor],
        markersize=6
    )
    for sensor in SENSORS
]

fig.legend(
    handles=legend_handles,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.92),
    ncol=4,
    frameon=False
)

fig.suptitle(
    'Spatial-scale sensitivity of historical forest contrasts\n'
    'Matched acquisitions | May–September | Coverage ≥70%',
    fontsize=16,
    fontweight='bold'
)

fig.tight_layout(
    rect=[0, 0, 1, 0.86]
)

save_figure(
    fig,
    'FIG_HISTORICAL_SPATIAL_SENSITIVITY'
)

# =============================================================================
# 22. FIGURE 5 — CENSUS-YEAR SPECTRAL CONTRASTS
# =============================================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(16, 5.5),
    sharey=True
)

for ax, scale in zip(
    axes,
    SPATIAL_SCALES
):

    data = field_2019.loc[
        field_2019['spatial_scale']
        == scale
    ]

    for i, index in enumerate(
        INDICES
    ):

        for sensor, offset in [
            ('Landsat_7', -0.13),
            ('Landsat_8', 0.13)
        ]:

            row = data.loc[
                (
                    data['sensor']
                    == sensor
                )
                & (
                    data['index']
                    == index
                )
            ]

            if len(row) != 1:
                continue

            row = row.iloc[0]

            y_position = i + offset

            ax.hlines(
                y_position,
                row['contrast_q25'],
                row['contrast_q75'],
                color=SENSOR_COLORS[sensor],
                linewidth=1.5
            )

            ax.scatter(
                row['contrast_median'],
                y_position,
                color=SENSOR_COLORS[sensor],
                s=55,
                zorder=3
            )

    ax.axvline(
        0,
        color='gray',
        linestyle='--',
        linewidth=1
    )

    ax.set_yticks(
        range(len(INDICES))
    )

    ax.set_yticklabels(
        INDICES
    )

    ax.set_xlabel(
        'Secondary − mature'
    )

    ax.set_title({
        'Fragment': 'Forest fragments',
        'Fragment_core': 'Fragment cores',
        'Plot': 'Field plots'
    }[scale])

    ax.grid(
        axis='x',
        alpha=0.2
    )

axes[0].invert_yaxis()

legend_handles = [
    plt.Line2D(
        [0],
        [0],
        marker='o',
        linestyle='none',
        color=SENSOR_COLORS[sensor],
        label=SENSOR_LABELS[sensor],
        markersize=7
    )
    for sensor in [
        'Landsat_7',
        'Landsat_8'
    ]
]

fig.legend(
    handles=legend_handles,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.9),
    ncol=2,
    frameon=False
)

fig.suptitle(
    'Spectral context of the 2019 field census',
    fontsize=16,
    fontweight='bold'
)

fig.text(
    0.5,
    0.015,
    'Points: median within-scene contrasts. '
    'Horizontal bars: interquartile ranges. '
    'Three acquisitions per sensor and spatial scale '
    'in the primary seasonal dataset.',
    ha='center',
    fontsize=9
)

fig.tight_layout(
    rect=[0, 0.07, 1, 0.85]
)

save_figure(
    fig,
    'FIG_HISTORICAL_2019_FIELD_CONTEXT'
)

# =============================================================================
# 23. EXPORT SCIENTIFIC TABLES
# =============================================================================

export_tables = {
    'LANDSAT_HISTORICAL_PAIRED_MASTER': (
        historical_pairs
    ),
    'LANDSAT_HISTORICAL_PRIMARY_PAIRS': (
        primary
    ),
    'LANDSAT_HISTORICAL_RETENTION_CHECK': (
        retention_reproduction
    ),
    'LANDSAT_HISTORICAL_SCENARIO_SUPPORT': (
        scenario_support
    ),
    'LANDSAT_ANNUAL_SENSOR_TRAJECTORIES': (
        annual_sensor
    ),
    'LANDSAT_ANNUAL_PRIMARY_TRAJECTORIES': (
        annual_primary
    ),
    'LANDSAT_ANNUAL_SENSOR_SUPPORT': (
        annual_support
    ),
    'LANDSAT_SENSOR_SPECIFIC_TRENDS': (
        historical_trends
    ),
    'LANDSAT_PRIMARY_CONTRAST_TRENDS': (
        primary_contrast_trends
    ),
    'LANDSAT_HISTORICAL_PERSISTENCE': (
        historical_persistence
    ),
    'LANDSAT_PRIMARY_PERSISTENCE': (
        primary_persistence
    ),
    'LANDSAT_ANNUAL_CONTRAST_CONSENSUS': (
        annual_consensus
    ),
    'LANDSAT_CONSENSUS_TRENDS': (
        consensus_trends
    ),
    'LANDSAT_EARLY_LATE_COMPARISON': (
        period_comparison
    ),
    'LANDSAT_SPATIAL_MATCHED_CONTRASTS': (
        spatial_matched
    ),
    'LANDSAT_SPATIAL_SENSITIVITY': (
        spatial_sensitivity
    ),
    'LANDSAT_TREND_SENSITIVITY': (
        trend_sensitivity
    ),
    'LANDSAT_TREND_ROBUSTNESS': (
        trend_robustness
    ),
    'LANDSAT_2019_FIELD_CONTEXT': (
        field_2019
    ),
    'LANDSAT_2019_SCENE_CONTRASTS': (
        field_scene_2019
    ),
    'LANDSAT_2019_HISTORICAL_POSITION': (
        census_historical_position
    ),
    'LANDSAT_EXTREME_OBSERVATION_AUDIT': (
        extreme_observations
    )
}

for name, dataframe in (
    export_tables.items()
):

    dataframe.to_csv(
        OUTPUT_DIR / f'{name}.csv',
        index=False,
        encoding='utf-8-sig'
    )

# =============================================================================
# 24. EXPORT EXCEL WORKBOOK
# =============================================================================

workbook_path = (
    TABLE_DIR
    / 'SATIPO_LANDSAT_HISTORICAL_TRAJECTORIES.xlsx'
)

excel_tables = {
    'Scenario_support': scenario_support,
    'Annual_primary': annual_primary,
    'Annual_all': annual_sensor,
    'Sensor_support': annual_support,
    'Primary_trends': primary_trends,
    'Contrast_trends': primary_contrast_trends,
    'All_trends': historical_trends,
    'Persistence': primary_persistence,
    'Consensus_annual': annual_consensus,
    'Consensus_trends': consensus_trends,
    'Early_late': period_comparison,
    'Spatial_sensitivity': spatial_sensitivity,
    'Trend_robustness': trend_robustness,
    'Trend_sensitivity': trend_sensitivity,
    'Field_2019': field_2019,
    'Census_position': census_historical_position,
    'Extreme_audit': extreme_observations,
    'Retention_check': retention_reproduction
}

with pd.ExcelWriter(
    workbook_path,
    engine='openpyxl'
) as writer:

    for sheet_name, dataframe in (
        excel_tables.items()
    ):

        dataframe.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False
        )

        worksheet = writer.sheets[
            sheet_name
        ]

        worksheet.freeze_panes = 'A2'

        worksheet.auto_filter.ref = (
            worksheet.dimensions
        )

        for column in worksheet.columns:

            column_letter = (
                column[0].column_letter
            )

            maximum_length = max(
                len(str(cell.value))
                if cell.value is not None
                else 0
                for cell in column
            )

            worksheet.column_dimensions[
                column_letter
            ].width = min(
                maximum_length + 2,
                45
            )

# =============================================================================
# 25. REPRODUCIBILITY MANIFEST
# =============================================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, 'rb') as file:

        for chunk in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b''
        ):

            digest.update(chunk)

    return digest.hexdigest()

estimated_primary = (
    primary_contrast_trends[
        'sen_slope_per_year'
    ].notna()
)

bootstrap_primary = (
    primary_contrast_trends[
        'bootstrap_status'
    ]
    == 'CONDITIONAL_BLOCK_BOOTSTRAP'
)

manifest = {
    'project': 'SATIPO_FOREST_PROJECT',
    'cell': 92,
    'stage': 'Historical forest trajectories',
    'version': 'V01_NATIVE_SENSOR_SPECIFIC',
    'created_utc': datetime.now(
        timezone.utc
    ).isoformat(),
    'input_file': str(MASTER_PATH),
    'input_sha256': sha256_file(
        MASTER_PATH
    ),
    'n_native_observations': len(native),
    'n_scenes': int(
        native['scene_id'].nunique()
    ),
    'n_spatial_units': int(
        native['unit_id'].nunique()
    ),
    'n_paired_observations': len(
        historical_pairs
    ),
    'n_primary_fragment_scenes': len(
        primary
    ),
    'n_primary_historical_years': int(
        primary['year'].nunique()
    ),
    'primary_months': PRIMARY_MONTHS,
    'primary_coverage_threshold': 0.70,
    'minimum_scenes_per_sensor_year': (
        MIN_SCENES_YEAR
    ),
    'minimum_trend_years': (
        MIN_TREND_YEARS
    ),
    'minimum_trend_span_years': (
        MIN_TREND_SPAN
    ),
    'n_primary_contrast_trends': int(
        estimated_primary.sum()
    ),
    'n_primary_bootstrap_intervals': int(
        bootstrap_primary.sum()
    ),
    'bootstrap_draws': N_BOOTSTRAP,
    'bootstrap_method': (
        'Residual moving blocks of 3 or 2 '
        'consecutive calendar years'
    ),
    'random_seed': RANDOM_SEED,
    'indices': INDICES,
    'scenarios': SCENARIOS,
    'contrast_definition': (
        'Late_secondary minus Mature '
        'within each acquisition'
    ),
    'annual_aggregation': (
        'Median of paired scene contrasts '
        'within sensor and calendar year'
    ),
    'primary_analysis': (
        'Native sensor-specific trajectories'
    ),
    'cross_sensor_consensus': (
        'Secondary descriptive analysis only'
    ),
    'historical_harmonization_claimed': False,
    'local_evi_offset_applied': False,
    'causal_recovery_claimed': False,
    'forest_type_replication_claimed': False,
    'ordinary_kendall_pvalues_reported': False,
    'extreme_observations_removed': 0,
    'limitations': [
        'One forest remnant per disturbance history',
        'Temporal observations are not forest-type replicates',
        'Cross-sensor spectral discrepancies remain',
        'Changing sensor composition may affect consensus trends',
        'Sparse annual observations in early years',
        'Conditional bootstrap does not include all measurement errors',
        'Spectral convergence does not establish biomass convergence',
        'Historical anomalies require independent disturbance validation'
    ],
    'output_directory': str(OUTPUT_DIR),
    'workbook': str(workbook_path),
    'exports': {
        name: {
            'rows': int(len(dataframe)),
            'columns': int(
                len(dataframe.columns)
            ),
            'file': str(
                OUTPUT_DIR / f'{name}.csv'
            )
        }
        for name, dataframe in (
            export_tables.items()
        )
    }
}

manifest_path = (
    OUTPUT_DIR
    / 'LANDSAT_HISTORICAL_TRAJECTORIES_MANIFEST.json'
)

with open(
    manifest_path,
    'w',
    encoding='utf-8'
) as file:

    json.dump(
        manifest,
        file,
        indent=2,
        ensure_ascii=False
    )

# =============================================================================
# 26. FINAL QUALITY CONTROL
# =============================================================================

assert len(historical_pairs) == 1164

assert len(primary) == 219

assert len(native) == 2328

assert retention_reproduction[
    'verified'
].all()

assert not historical_pairs.duplicated(
    pair_keys
).any()

assert (
    primary[
        'paired_minimum_coverage'
    ] >= 0.70
).all()

assert primary[
    'month'
].isin(
    PRIMARY_MONTHS
).all()

assert workbook_path.exists()

assert manifest_path.exists()

for name in export_tables:

    assert (
        OUTPUT_DIR / f'{name}.csv'
    ).exists()

for index in INDICES:

    assert np.allclose(
        historical_pairs[
            f'{index}_contrast'
        ],
        (
            historical_pairs[
                f'{index}_mean_secondary'
            ]
            - historical_pairs[
                f'{index}_mean_mature'
            ]
        ),
        equal_nan=True
    )

# =============================================================================
# 27. FINAL REPORT
# =============================================================================

print('\n' + '=' * 78)
print('SATIPO — HISTORICAL TRAJECTORIES REPORT')
print('=' * 78)

print('Original observations:', len(native))

print(
    'Unique Landsat scenes:',
    native['scene_id'].nunique()
)

print(
    'Paired historical observations:',
    len(historical_pairs)
)

print(
    'Primary fragment scenes:',
    len(primary)
)

print(
    'Primary historical years:',
    primary['year'].nunique()
)

print(
    'Adequately supported consensus years:',
    primary_consensus['year'].nunique()
)

print(
    'Analytical scenarios:',
    len(SCENARIOS)
)

print(
    'Sensor-specific trend configurations:',
    len(historical_trends)
)

print(
    'Estimated primary contrast trends:',
    int(estimated_primary.sum())
)

print(
    'Primary conditional bootstrap intervals:',
    int(bootstrap_primary.sum())
)

print(
    'Flagged unusual index observations:',
    len(extreme_observations)
)

print('Automatically excluded anomalies: 0')

print('Original screened master: PRESERVED')

print(
    'Complete historical harmonization: '
    'NOT CLAIMED'
)

print(
    'Primary analysis: '
    'NATIVE SENSOR-SPECIFIC'
)

print(
    'Cross-sensor synthesis: '
    'DESCRIPTIVE ONLY'
)

print('\nPRIMARY SIGNED CONTRAST TRENDS')

display(
    primary_contrast_trends.loc[
        primary_contrast_trends['variable']
        == 'Signed_contrast',
        [
            'sensor',
            'index',
            'n_years',
            'sen_slope_per_decade',
            'kendall_tau',
            'slope_lcl_95',
            'slope_ucl_95'
        ]
    ].round(5)
)

print('\nPRIMARY ABSOLUTE CONTRAST TRENDS')

display(
    primary_contrast_trends.loc[
        primary_contrast_trends['variable']
        == 'Absolute_contrast',
        [
            'sensor',
            'index',
            'n_years',
            'sen_slope_per_decade',
            'kendall_tau',
            'slope_lcl_95',
            'slope_ucl_95'
        ]
    ].round(5)
)

print('\nPRIMARY HISTORICAL PERSISTENCE')

display(
    primary_persistence[
        [
            'sensor',
            'index',
            'n_adequate_years',
            'median_signed_contrast',
            'median_absolute_contrast',
            'positive_year_pct'
        ]
    ].round(5)
)

print('\nOutput directory:')
print(OUTPUT_DIR)

print('\nWorkbook:')
print(workbook_path)

print('\nManifest:')
print(manifest_path)

print('\nFigures:')

for stem in [
    'FIG_HISTORICAL_NATIVE_TRAJECTORIES',
    'FIG_HISTORICAL_CONTRAST_TRAJECTORIES',
    'FIG_HISTORICAL_CONTRAST_TRENDS',
    'FIG_HISTORICAL_SPATIAL_SENSITIVITY',
    'FIG_HISTORICAL_2019_FIELD_CONTEXT'
]:

    print(FIGURE_DIR / stem)

print('\n' + '=' * 78)
print('CELL 92 COMPLETED')
print('NEXT: CELL 93 — HISTORICAL DISTURBANCE ASSESSMENT')
print('=' * 78)


In [ ]:
# ============================================================
# CELL 92B
# FINAL TREND AUDIT AND ANOMALY SENSITIVITY
# SATIPO FOREST PROJECT
# ============================================================

from pathlib import Path
from google.colab import drive

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import hashlib

from scipy.stats import theilslopes, kendalltau
from datetime import datetime, timezone

drive.mount('/content/drive')

# ============================================================
# 1. PROJECT PATHS
# ============================================================

ROOT = Path(
    '/content/drive/MyDrive/SATIPO_FOREST_PROJECT'
)

REMOTE = ROOT / '04_REMOTE_SENSING'
LANDSAT = REMOTE / '03_LANDSAT'

TRAJECTORY_DIR = (
    LANDSAT / '05_HISTORICAL_TRAJECTORIES'
)

SCREENED_PATH = (
    LANDSAT
    / '03_RADIOMETRIC_SCREENING'
    / 'LANDSAT_SCREENED_MASTER.csv'
)

OUTPUT_DIR = (
    TRAJECTORY_DIR / 'FINAL_TREND_AUDIT'
)

TABLE_DIR = REMOTE / '08_OUTPUTS' / 'TABLES'
FIGURE_DIR = REMOTE / '08_OUTPUTS' / 'FIGURES'

for directory in [
    OUTPUT_DIR,
    TABLE_DIR,
    FIGURE_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

assert SCREENED_PATH.exists(), SCREENED_PATH
assert TRAJECTORY_DIR.exists(), TRAJECTORY_DIR

print('=' * 78)
print('SATIPO — FINAL HISTORICAL TREND AUDIT')
print('=' * 78)

# ============================================================
# 2. LOCATE ORIGINAL TREND TABLE
# ============================================================

required_trend_columns = {
    'scenario',
    'spatial_scale',
    'sensor',
    'index',
    'variable',
    'n_years',
    'sen_slope_per_decade',
    'slope_lcl_95',
    'slope_ucl_95'
}

trend_candidates = []

for path in TRAJECTORY_DIR.rglob('*.csv'):

    if OUTPUT_DIR in path.parents:
        continue

    try:
        header = pd.read_csv(
            path,
            nrows=0
        )

        if required_trend_columns.issubset(
            header.columns
        ):

            temp = pd.read_csv(path)

            trend_candidates.append(
                (path, len(temp))
            )

    except Exception:
        continue

assert trend_candidates, (
    'No compatible historical trend table found.'
)

trend_candidates.sort(
    key=lambda x: x[1],
    reverse=True
)

TREND_PATH = trend_candidates[0][0]

trends_original = pd.read_csv(TREND_PATH)

print('\nORIGINAL TREND TABLE')
print(TREND_PATH)
print('Rows:', len(trends_original))

assert len(trends_original) == 720, (
    'Unexpected number of trend configurations.'
)

# ============================================================
# 3. CORRECT INTERVAL UNITS
# ============================================================

trends_corrected = trends_original.copy()

trends_corrected[
    'slope_lcl_95_per_year'
] = trends_original['slope_lcl_95']

trends_corrected[
    'slope_ucl_95_per_year'
] = trends_original['slope_ucl_95']

trends_corrected[
    'slope_lcl_95_per_decade'
] = (
    trends_corrected[
        'slope_lcl_95_per_year'
    ] * 10
)

trends_corrected[
    'slope_ucl_95_per_decade'
] = (
    trends_corrected[
        'slope_ucl_95_per_year'
    ] * 10
)

# Preserve the original interval columns,
# but explicitly identify their units.

trends_corrected = trends_corrected.rename(
    columns={
        'slope_lcl_95':
            'original_lcl_95_annual',
        'slope_ucl_95':
            'original_ucl_95_annual'
    }
)

trends_corrected[
    'interval_units'
] = 'Index units per decade'

trends_corrected[
    'interval_excludes_zero'
] = (
    (
        trends_corrected[
            'slope_lcl_95_per_decade'
        ] > 0
    )
    |
    (
        trends_corrected[
            'slope_ucl_95_per_decade'
        ] < 0
    )
)

# ============================================================
# 4. VERIFY CONVERSION AGAINST CELL 92
# ============================================================

check = trends_corrected.loc[
    (
        trends_corrected['scenario']
        == 'Primary_fragment'
    )
    &
    (
        trends_corrected['sensor']
        == 'Landsat_8'
    )
    &
    (
        trends_corrected['index']
        == 'EVI'
    )
    &
    (
        trends_corrected['variable']
        == 'Absolute_contrast'
    )
]

assert len(check) == 1

check = check.iloc[0]

assert np.isclose(
    check['sen_slope_per_decade'],
    -0.01067,
    atol=0.00002
)

assert np.isclose(
    check['slope_lcl_95_per_decade'],
    -0.0163,
    atol=0.00015
)

assert np.isclose(
    check['slope_ucl_95_per_decade'],
    -0.0057,
    atol=0.00015
)

print('\nINTERVAL UNIT CONVERSION VERIFIED')

primary_trends = trends_corrected.loc[
    (
        trends_corrected['scenario']
        == 'Primary_fragment'
    )
    &
    (
        trends_corrected['variable'].isin([
            'Signed_contrast',
            'Absolute_contrast'
        ])
    )
].copy()

primary_trends = primary_trends.loc[
    primary_trends[
        'sen_slope_per_decade'
    ].notna()
]

assert len(primary_trends) == 30

trend_display_columns = [
    'sensor',
    'index',
    'variable',
    'n_years',
    'sen_slope_per_decade',
    'slope_lcl_95_per_decade',
    'slope_ucl_95_per_decade',
    'interval_excludes_zero'
]

print('\nCORRECTED PRIMARY TRENDS')

display(
    primary_trends[
        trend_display_columns
    ].round(6)
)

# ============================================================
# 5. LOAD SCREENED NATIVE OBSERVATIONS
# ============================================================

native = pd.read_csv(
    SCREENED_PATH,
    low_memory=False
)

native['date'] = pd.to_datetime(
    native['date']
)

INDICES = [
    'NDVI',
    'EVI',
    'NDMI',
    'NBR',
    'NBR2'
]

SENSORS = [
    'Landsat_5',
    'Landsat_7',
    'Landsat_8',
    'Landsat_9'
]

index_columns = [
    f'{index}_mean'
    for index in INDICES
]

required_native = {
    'scene_id',
    'sensor',
    'date',
    'year',
    'month',
    'forest_type',
    'spatial_scale',
    'screened_valid_fraction',
    *index_columns
}

assert required_native.issubset(
    native.columns
), (
    required_native - set(native.columns)
)

assert len(native) == 2328
assert native['scene_id'].nunique() == 388

fragment = native.loc[
    native['spatial_scale'] == 'Fragment'
].copy()

assert len(fragment) == 776

# ============================================================
# 6. RECONSTRUCT PRIMARY PAIRED DATASET
# ============================================================

secondary = fragment.loc[
    fragment['forest_type']
    == 'Late_secondary'
].copy()

mature = fragment.loc[
    fragment['forest_type']
    == 'Mature'
].copy()

pair_keys = [
    'scene_id',
    'sensor',
    'date',
    'year',
    'month'
]

paired = secondary[
    pair_keys
    + ['screened_valid_fraction']
    + index_columns
].merge(
    mature[
        pair_keys
        + ['screened_valid_fraction']
        + index_columns
    ],
    on=pair_keys,
    suffixes=('_secondary', '_mature'),
    validate='one_to_one'
)

assert len(paired) == 388

paired['paired_coverage'] = paired[
    [
        'screened_valid_fraction_secondary',
        'screened_valid_fraction_mature'
    ]
].min(axis=1)

for index in INDICES:

    paired[f'{index}_contrast'] = (
        paired[f'{index}_mean_secondary']
        -
        paired[f'{index}_mean_mature']
    )

    paired[f'{index}_absolute'] = (
        paired[f'{index}_contrast'].abs()
    )

primary = paired.loc[
    paired['month'].between(5, 9)
    &
    (paired['paired_coverage'] >= 0.70)
].copy()

assert len(primary) == 219

print('\nPRIMARY DATASET REPRODUCED')
print('Scenes:', len(primary))
print('Years:', primary['year'].nunique())

# ============================================================
# 7. RECOVER ORIGINAL EXTREME-OBSERVATION AUDIT
# ============================================================

required_audit = {
    'scene_id',
    'sensor',
    'index',
    'robust_z'
}

audit_candidates = []

for path in TRAJECTORY_DIR.rglob('*.csv'):

    if OUTPUT_DIR in path.parents:
        continue

    try:
        header = pd.read_csv(
            path,
            nrows=0
        )

        if required_audit.issubset(
            header.columns
        ):

            candidate = pd.read_csv(path)

            audit_candidates.append(
                (path, len(candidate))
            )

    except Exception:
        continue

assert audit_candidates, (
    'Original extreme-observation audit not found.'
)

# Select the original audit with 206 records.

matching_audits = [
    path
    for path, rows in audit_candidates
    if rows == 206
]

assert matching_audits, (
    'No audit with the expected 206 records.'
)

AUDIT_PATH = matching_audits[0]

extreme_audit = pd.read_csv(
    AUDIT_PATH
)

flagged_scenes = set(
    extreme_audit['scene_id']
)

assert len(flagged_scenes) == 48

primary['flagged_scene'] = (
    primary['scene_id'].isin(
        flagged_scenes
    )
)

print('\nORIGINAL ANOMALY AUDIT RECOVERED')
print(AUDIT_PATH)

print(
    'Flagged index observations:',
    len(extreme_audit)
)

print(
    'Unique flagged acquisitions:',
    len(flagged_scenes)
)

print(
    'Flagged acquisitions in primary dataset:',
    primary['flagged_scene'].sum()
)

# ============================================================
# 8. DEFINE SENSITIVITY SCENARIOS
# ============================================================

# No permanent deletion of observations.
# Exclusions below are diagnostic only.

scenarios = {
    'Primary_all': primary.copy(),

    'Exclude_flagged_acquisitions':
        primary.loc[
            ~primary['flagged_scene']
        ].copy(),

    'Coverage_ge90':
        primary.loc[
            primary['paired_coverage'] >= 0.90
        ].copy(),

    'June_September':
        primary.loc[
            primary['month'].between(6, 9)
        ].copy(),

    'Exclude_L7_after2017':
        primary.loc[
            ~(
                (
                    primary['sensor']
                    == 'Landsat_7'
                )
                &
                (
                    primary['year'] > 2017
                )
            )
        ].copy(),

    'Coverage_ge90_no_flagged':
        primary.loc[
            (
                primary['paired_coverage']
                >= 0.90
            )
            &
            ~primary['flagged_scene']
        ].copy()
}

scenario_support = []

for scenario, frame in scenarios.items():

    scenario_support.append({
        'scenario': scenario,
        'n_scenes': len(frame),
        'n_years': frame['year'].nunique(),
        'flagged_scenes_retained':
            int(frame['flagged_scene'].sum()),
        'median_coverage':
            frame['paired_coverage'].median()
    })

scenario_support = pd.DataFrame(
    scenario_support
)

print('\nSENSITIVITY SCENARIO SUPPORT')

display(
    scenario_support.round(4)
)

# ============================================================
# 9. ANNUAL AGGREGATION
# ============================================================

MIN_SCENES_PER_YEAR = 3
MIN_TREND_YEARS = 8

annual_rows = []

for scenario, frame in scenarios.items():

    for (sensor, year), group in frame.groupby(
        ['sensor', 'year']
    ):

        for index in INDICES:

            signed = group[
                f'{index}_contrast'
            ].dropna()

            absolute = group[
                f'{index}_absolute'
            ].dropna()

            annual_rows.append({

                'scenario': scenario,
                'sensor': sensor,
                'year': int(year),
                'index': index,

                'n_scenes': len(signed),

                'adequate_support':
                    len(signed)
                    >= MIN_SCENES_PER_YEAR,

                'signed_contrast':
                    signed.median(),

                'absolute_contrast':
                    absolute.median(),

                'signed_q25':
                    signed.quantile(0.25),

                'signed_q75':
                    signed.quantile(0.75)
            })

annual_sensitivity = pd.DataFrame(
    annual_rows
)

# ============================================================
# 10. SENSOR-SPECIFIC TREND SENSITIVITY
# ============================================================

trend_rows = []

for scenario in scenarios:

    for sensor in SENSORS:

        for index in INDICES:

            subset = annual_sensitivity.loc[
                (
                    annual_sensitivity['scenario']
                    == scenario
                )
                &
                (
                    annual_sensitivity['sensor']
                    == sensor
                )
                &
                (
                    annual_sensitivity['index']
                    == index
                )
                &
                annual_sensitivity[
                    'adequate_support'
                ]
            ].sort_values('year')

            for variable in [
                'signed_contrast',
                'absolute_contrast'
            ]:

                valid = subset[
                    ['year', variable]
                ].dropna()

                x = valid['year'].to_numpy(
                    dtype=float
                )

                y = valid[variable].to_numpy(
                    dtype=float
                )

                result = {
                    'scenario': scenario,
                    'sensor': sensor,
                    'index': index,
                    'variable': variable,
                    'n_years': len(valid),
                    'first_year':
                        x.min() if len(x) else np.nan,
                    'last_year':
                        x.max() if len(x) else np.nan,
                    'sen_slope_per_decade': np.nan,
                    'kendall_tau': np.nan,
                    'status':
                        'INSUFFICIENT_SUPPORT'
                }

                if len(valid) >= MIN_TREND_YEARS:

                    slope = theilslopes(
                        y,
                        x
                    ).slope

                    tau = kendalltau(
                        x,
                        y
                    ).statistic

                    result.update({
                        'sen_slope_per_decade':
                            slope * 10,

                        'kendall_tau': tau,

                        'status': 'ESTIMATED'
                    })

                trend_rows.append(result)

sensitivity_trends = pd.DataFrame(
    trend_rows
)

# ============================================================
# 11. VERIFY PRIMARY TREND REPRODUCTION
# ============================================================

reference = primary_trends[
    [
        'sensor',
        'index',
        'variable',
        'sen_slope_per_decade'
    ]
].copy()

reference['variable'] = (
    reference['variable'].str.lower()
)

reference = reference.rename(
    columns={
        'sen_slope_per_decade':
            'original_slope_per_decade'
    }
)

reproduced = sensitivity_trends.loc[
    sensitivity_trends['scenario']
    == 'Primary_all'
].merge(
    reference,
    on=[
        'sensor',
        'index',
        'variable'
    ],
    validate='one_to_one'
)

reproduced['absolute_difference'] = (
    reproduced['sen_slope_per_decade']
    -
    reproduced['original_slope_per_decade']
).abs()

max_difference = reproduced[
    'absolute_difference'
].max()

print('\nPRIMARY TREND REPRODUCTION')

print(
    'Maximum slope difference:',
    max_difference
)

assert len(reproduced) == 30

assert max_difference < 0.00002, (
    'Trend reproduction failed. '
    'Check annual aggregation before interpretation.'
)

# ============================================================
# 12. LANDSAT 8 CONVERGENCE ROBUSTNESS
# ============================================================

l8_absolute = sensitivity_trends.loc[
    (
        sensitivity_trends['sensor']
        == 'Landsat_8'
    )
    &
    (
        sensitivity_trends['variable']
        == 'absolute_contrast'
    )
].copy()

l8_summary = []

for index in INDICES:

    subset = l8_absolute.loc[
        l8_absolute['index'] == index
    ]

    evaluated = subset.loc[
        subset['status'] == 'ESTIMATED'
    ]

    baseline = evaluated.loc[
        evaluated['scenario'] == 'Primary_all',
        'sen_slope_per_decade'
    ].iloc[0]

    n_negative = (
        evaluated['sen_slope_per_decade'] < 0
    ).sum()

    l8_summary.append({

        'index': index,

        'primary_slope_per_decade':
            baseline,

        'n_evaluable_scenarios':
            len(evaluated),

        'n_negative_slopes':
            int(n_negative),

        'negative_slope_pct':
            100 * n_negative / len(evaluated),

        'minimum_slope':
            evaluated[
                'sen_slope_per_decade'
            ].min(),

        'maximum_slope':
            evaluated[
                'sen_slope_per_decade'
            ].max()
    })

l8_robustness = pd.DataFrame(
    l8_summary
)

print('\nLANDSAT 8 CONVERGENCE ROBUSTNESS')

display(
    l8_robustness.round(6)
)

print('\nLANDSAT 8 — ALL SENSITIVITY SLOPES')

display(
    l8_absolute[
        [
            'scenario',
            'index',
            'n_years',
            'sen_slope_per_decade',
            'status'
        ]
    ].round(6)
)

# ============================================================
# 13. INDEX-SPECIFIC ANOMALY SENSITIVITY
# ============================================================

# Exclude a scene only for the index flagged
# in the original anomaly audit.

index_specific_rows = []

for index in INDICES:

    index_flagged_scenes = set(
        extreme_audit.loc[
            extreme_audit['index'] == index,
            'scene_id'
        ]
    )

    frame = primary.loc[
        ~primary['scene_id'].isin(
            index_flagged_scenes
        )
    ].copy()

    for sensor in SENSORS:

        sensor_frame = frame.loc[
            frame['sensor'] == sensor
        ]

        annual = (
            sensor_frame
            .groupby('year')[
                f'{index}_absolute'
            ]
            .agg(
                median='median',
                n_scenes='count'
            )
            .reset_index()
        )

        annual = annual.loc[
            annual['n_scenes']
            >= MIN_SCENES_PER_YEAR
        ]

        slope = np.nan
        tau = np.nan

        if len(annual) >= MIN_TREND_YEARS:

            slope = theilslopes(
                annual['median'],
                annual['year']
            ).slope * 10

            tau = kendalltau(
                annual['year'],
                annual['median']
            ).statistic

        index_specific_rows.append({

            'sensor': sensor,
            'index': index,

            'n_scenes_retained':
                len(sensor_frame),

            'n_adequate_years':
                len(annual),

            'sen_slope_per_decade':
                slope,

            'kendall_tau':
                tau
        })

index_specific_sensitivity = pd.DataFrame(
    index_specific_rows
)

print('\nINDEX-SPECIFIC ANOMALY SENSITIVITY')

display(
    index_specific_sensitivity.round(6)
)

# ============================================================
# 14. LEAVE-ONE-YEAR-OUT SENSITIVITY
# ============================================================

# Identify whether a single year drives
# the Landsat 8 convergence result.

loyo_rows = []

for index in INDICES:

    subset = annual_sensitivity.loc[
        (
            annual_sensitivity['scenario']
            == 'Primary_all'
        )
        &
        (
            annual_sensitivity['sensor']
            == 'Landsat_8'
        )
        &
        (
            annual_sensitivity['index']
            == index
        )
        &
        annual_sensitivity['adequate_support']
    ].copy()

    for excluded_year in subset['year']:

        retained = subset.loc[
            subset['year'] != excluded_year
        ]

        slope = theilslopes(
            retained['absolute_contrast'],
            retained['year']
        ).slope * 10

        loyo_rows.append({

            'sensor': 'Landsat_8',
            'index': index,

            'excluded_year':
                int(excluded_year),

            'n_years':
                len(retained),

            'sen_slope_per_decade':
                slope
        })

loyo = pd.DataFrame(loyo_rows)

loyo_summary = (
    loyo.groupby('index')
    .agg(
        n_folds=(
            'excluded_year',
            'count'
        ),

        minimum_slope=(
            'sen_slope_per_decade',
            'min'
        ),

        maximum_slope=(
            'sen_slope_per_decade',
            'max'
        ),

        negative_fold_pct=(
            'sen_slope_per_decade',
            lambda x: 100 * (x < 0).mean()
        )
    )
    .reset_index()
)

print('\nLEAVE-ONE-YEAR-OUT ROBUSTNESS')

display(
    loyo_summary.round(6)
)

# ============================================================
# 15. FIGURE — FINAL TREND SENSITIVITY
# ============================================================

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42
})

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 6.5)
)

colors = {
    'NDVI': '#527B98',
    'EVI': '#DB9140',
    'NDMI': '#24634F',
    'NBR': '#795C92',
    'NBR2': '#A86D58'
}

# ------------------------------------------------------------
# Panel A — corrected primary intervals
# ------------------------------------------------------------

ax = axes[0]

l8_primary = primary_trends.loc[
    (
        primary_trends['sensor']
        == 'Landsat_8'
    )
    &
    (
        primary_trends['variable']
        == 'Absolute_contrast'
    )
].set_index('index').loc[INDICES]

for position, index in enumerate(INDICES):

    row = l8_primary.loc[index]

    estimate = row[
        'sen_slope_per_decade'
    ]

    lower = row[
        'slope_lcl_95_per_decade'
    ]

    upper = row[
        'slope_ucl_95_per_decade'
    ]

    ax.hlines(
        position,
        lower,
        upper,
        color=colors[index],
        linewidth=2.5
    )

    ax.scatter(
        estimate,
        position,
        color=colors[index],
        s=65,
        zorder=3
    )

ax.axvline(
    0,
    color='gray',
    linestyle='--'
)

ax.set_yticks(
    range(len(INDICES)),
    INDICES
)

ax.invert_yaxis()

ax.set_xlabel(
    'Theil–Sen slope per decade'
)

ax.set_title(
    '(a) Primary trends and corrected intervals',
    fontweight='bold'
)

ax.grid(
    axis='x',
    alpha=0.2
)

# ------------------------------------------------------------
# Panel B — scenario sensitivity
# ------------------------------------------------------------

ax = axes[1]

scenario_order = list(scenarios.keys())

scenario_labels = [
    'Primary',
    'No flagged scenes',
    'Coverage ≥90%',
    'June–September',
    'L7 ≤2017',
    'Coverage ≥90%, no flags'
]

for index in INDICES:

    subset = (
        l8_absolute.loc[
            l8_absolute['index'] == index
        ]
        .set_index('scenario')
        .reindex(scenario_order)
    )

    ax.plot(
        subset['sen_slope_per_decade'],
        np.arange(len(scenario_order)),
        marker='o',
        color=colors[index],
        label=index,
        linewidth=1.5,
        markersize=5
    )

ax.axvline(
    0,
    color='gray',
    linestyle='--'
)

ax.set_yticks(
    np.arange(len(scenario_order)),
    scenario_labels
)

ax.invert_yaxis()

ax.set_xlabel(
    'Theil–Sen slope per decade'
)

ax.set_title(
    '(b) Eligibility and anomaly sensitivity',
    fontweight='bold'
)

ax.grid(
    axis='x',
    alpha=0.2
)

# ------------------------------------------------------------
# Panel C — leave-one-year-out
# ------------------------------------------------------------

ax = axes[2]

for position, index in enumerate(INDICES):

    subset = loyo.loc[
        loyo['index'] == index
    ]

    values = subset[
        'sen_slope_per_decade'
    ].to_numpy()

    ax.scatter(
        values,
        np.full(len(values), position),
        color=colors[index],
        alpha=0.45,
        s=30
    )

    ax.scatter(
        np.median(values),
        position,
        color=colors[index],
        edgecolor='black',
        s=75,
        zorder=4
    )

ax.axvline(
    0,
    color='gray',
    linestyle='--'
)

ax.set_yticks(
    range(len(INDICES)),
    INDICES
)

ax.invert_yaxis()

ax.set_xlabel(
    'Theil–Sen slope per decade'
)

ax.set_title(
    '(c) Leave-one-year-out sensitivity',
    fontweight='bold'
)

ax.grid(
    axis='x',
    alpha=0.2
)

fig.suptitle(
    'Robustness of recent spectral convergence '
    'between forest fragments',
    fontsize=17,
    fontweight='bold',
    y=0.98
)

handles, labels = axes[1].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.92),
    ncol=5,
    frameon=False
)

fig.text(
    0.5,
    0.015,
    'Landsat 8 | Native within-scene absolute contrasts. '
    'Intervals in panel (a) are conditional 95% '
    'residual-block bootstrap intervals.',
    ha='center',
    fontsize=9
)

fig.tight_layout(
    rect=[0, 0.055, 1, 0.84]
)

FIGURE_STEM = (
    FIGURE_DIR
    / 'FIG_HISTORICAL_FINAL_TREND_ROBUSTNESS'
)

for extension in ['png', 'pdf', 'svg']:

    fig.savefig(
        f'{FIGURE_STEM}.{extension}',
        dpi=600,
        bbox_inches='tight'
    )

plt.show()

# ============================================================
# 16. EXPORT CORRECTED RESULTS
# ============================================================

exports = {

    'LANDSAT_TRENDS_CORRECTED':
        trends_corrected,

    'LANDSAT_PRIMARY_TRENDS_CORRECTED':
        primary_trends,

    'LANDSAT_TREND_SCENARIO_SUPPORT':
        scenario_support,

    'LANDSAT_ANNUAL_ANOMALY_SENSITIVITY':
        annual_sensitivity,

    'LANDSAT_TREND_ANOMALY_SENSITIVITY':
        sensitivity_trends,

    'LANDSAT_TREND_REPRODUCTION':
        reproduced,

    'LANDSAT8_CONVERGENCE_ROBUSTNESS':
        l8_robustness,

    'LANDSAT_INDEX_SPECIFIC_ANOMALY_SENSITIVITY':
        index_specific_sensitivity,

    'LANDSAT8_LEAVE_ONE_YEAR_OUT':
        loyo,

    'LANDSAT8_LEAVE_ONE_YEAR_OUT_SUMMARY':
        loyo_summary
}

for filename, frame in exports.items():

    frame.to_csv(
        OUTPUT_DIR / f'{filename}.csv',
        index=False,
        encoding='utf-8-sig'
    )

WORKBOOK = (
    TABLE_DIR
    / 'SATIPO_LANDSAT_FINAL_TREND_AUDIT.xlsx'
)

with pd.ExcelWriter(
    WORKBOOK,
    engine='openpyxl'
) as writer:

    sheets = {

        'Corrected_trends':
            trends_corrected,

        'Primary_trends':
            primary_trends,

        'Scenario_support':
            scenario_support,

        'Annual_sensitivity':
            annual_sensitivity,

        'Trend_sensitivity':
            sensitivity_trends,

        'Trend_reproduction':
            reproduced,

        'L8_robustness':
            l8_robustness,

        'Index_specific_QA':
            index_specific_sensitivity,

        'L8_LOYO':
            loyo,

        'L8_LOYO_summary':
            loyo_summary
    }

    for sheet, frame in sheets.items():

        frame.to_excel(
            writer,
            sheet_name=sheet,
            index=False
        )

# ============================================================
# 17. MANIFEST
# ============================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, 'rb') as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b''
        ):
            digest.update(chunk)

    return digest.hexdigest()

manifest = {

    'project':
        'SATIPO_FOREST_PROJECT',

    'cell':
        '92B',

    'created_utc':
        datetime.now(
            timezone.utc
        ).isoformat(),

    'original_trend_table':
        str(TREND_PATH),

    'original_trend_sha256':
        sha256_file(TREND_PATH),

    'original_anomaly_audit':
        str(AUDIT_PATH),

    'screened_master':
        str(SCREENED_PATH),

    'screened_master_sha256':
        sha256_file(SCREENED_PATH),

    'trend_configurations':
        int(len(trends_corrected)),

    'primary_estimated_trends':
        int(len(primary_trends)),

    'primary_scenes':
        int(len(primary)),

    'flagged_acquisitions':
        int(len(flagged_scenes)),

    'permanently_removed_acquisitions':
        0,

    'interval_conversion':
        'Annual bounds multiplied by 10',

    'trend_reproduction_max_difference':
        float(max_difference),

    'sensitivity_scenarios':
        list(scenarios.keys()),

    'minimum_scenes_per_year':
        MIN_SCENES_PER_YEAR,

    'minimum_trend_years':
        MIN_TREND_YEARS,

    'primary_analysis':
        'Native sensor-specific',

    'sensitivity_intervals':
        'Not estimated; sensitivity uses point slopes',

    'interpretation':
        (
            'Conditional comparison of two forest '
            'fragments; spectral convergence does not '
            'demonstrate ecological recovery.'
        ),

    'workbook':
        str(WORKBOOK),

    'figure':
        str(FIGURE_STEM)
}

MANIFEST_PATH = (
    OUTPUT_DIR
    / 'LANDSAT_FINAL_TREND_AUDIT_MANIFEST.json'
)

with open(
    MANIFEST_PATH,
    'w',
    encoding='utf-8'
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

# ============================================================
# 18. FINAL REPORT
# ============================================================

print('\n' + '=' * 78)
print('SATIPO — FINAL TREND AUDIT REPORT')
print('=' * 78)

print(
    'Corrected trend configurations:',
    len(trends_corrected)
)

print(
    'Corrected primary trends:',
    len(primary_trends)
)

print(
    'Primary scene count:',
    len(primary)
)

print(
    'Primary slope reproduction:',
    'VERIFIED'
)

print(
    'Maximum reproduction difference:',
    max_difference
)

print(
    'Original flagged acquisitions:',
    len(flagged_scenes)
)

print(
    'Permanently removed acquisitions:',
    0
)

print(
    'Original screened master:',
    'PRESERVED'
)

print(
    'Original trend table:',
    'PRESERVED'
)

print('\nLANDSAT 8 — FINAL ROBUSTNESS')

display(
    l8_robustness.round(6)
)

print('\nLANDSAT 8 — LEAVE-ONE-YEAR-OUT')

display(
    loyo_summary.round(6)
)

print('\nOUTPUT DIRECTORY:')
print(OUTPUT_DIR)

print('\nWORKBOOK:')
print(WORKBOOK)

print('\nMANIFEST:')
print(MANIFEST_PATH)

print('\nFIGURE:')
print(FIGURE_STEM)

print('\n' + '=' * 78)
print('CELL 92B COMPLETED')
print('NEXT: CELL 93 — HISTORICAL DISTURBANCE ASSESSMENT')
print('=' * 78)

In [ ]:
# ============================================================
# CELL 93
# HISTORICAL DISTURBANCE ASSESSMENT
# SATIPO FOREST PROJECT
# ============================================================

from pathlib import Path
from google.colab import drive

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

from scipy.stats import spearmanr
from datetime import datetime, timezone

drive.mount('/content/drive')

# ============================================================
# 1. PATHS AND CONFIGURATION
# ============================================================

ROOT = Path(
    '/content/drive/MyDrive/SATIPO_FOREST_PROJECT'
)

REMOTE = ROOT / '04_REMOTE_SENSING'
LANDSAT = REMOTE / '03_LANDSAT'

TRAJECTORY_DIR = (
    LANDSAT / '05_HISTORICAL_TRAJECTORIES'
)

SCREENED_PATH = (
    LANDSAT
    / '03_RADIOMETRIC_SCREENING'
    / 'LANDSAT_SCREENED_MASTER.csv'
)

AUDIT_PATH = (
    TRAJECTORY_DIR
    / 'LANDSAT_EXTREME_OBSERVATION_AUDIT.csv'
)

OUT = (
    LANDSAT / '06_HISTORICAL_DISTURBANCE'
)

TABLE_DIR = REMOTE / '08_OUTPUTS' / 'TABLES'
FIGURE_DIR = REMOTE / '08_OUTPUTS' / 'FIGURES'

for directory in [OUT, TABLE_DIR, FIGURE_DIR]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

assert SCREENED_PATH.exists(), SCREENED_PATH
assert AUDIT_PATH.exists(), AUDIT_PATH

INDICES = [
    'NDVI', 'EVI', 'NDMI', 'NBR', 'NBR2'
]

SENSORS = [
    'Landsat_5',
    'Landsat_7',
    'Landsat_8',
    'Landsat_9'
]

# Primary disturbance indicators.
# NDVI and EVI are retained as supporting indices.

DISTURBANCE_INDICES = [
    'NBR', 'NDMI'
]

MIN_SCENES = 3

# Robust anomaly thresholds.
# These are screening thresholds, not calibrated
# disturbance probabilities.

Z_THRESHOLD = 2.5
Z_SENSITIVITY = [2.0, 2.5, 3.0]

MIN_BASELINE_YEARS = 8
MIN_LOCAL_REFERENCE_YEARS = 2

# ============================================================
# 2. LOAD VALIDATED INPUTS
# ============================================================

native = pd.read_csv(
    SCREENED_PATH,
    low_memory=False
)

native['date'] = pd.to_datetime(
    native['date']
)

audit = pd.read_csv(AUDIT_PATH)

flagged_scenes = set(
    audit['scene_id']
)

required = {
    'scene_id',
    'sensor',
    'date',
    'year',
    'month',
    'forest_type',
    'spatial_scale',
    'screened_valid_fraction',
    *[f'{i}_mean' for i in INDICES]
}

missing = required - set(native.columns)

assert not missing, (
    f'Missing columns: {missing}'
)

assert len(native) == 2328
assert native['scene_id'].nunique() == 388
assert len(flagged_scenes) == 48

print('=' * 78)
print('SATIPO — HISTORICAL DISTURBANCE ASSESSMENT')
print('=' * 78)

print('Input observations:', len(native))
print('Unique scenes:', native['scene_id'].nunique())

# ============================================================
# 3. RECONSTRUCT PAIRED DATA
# ============================================================

keys = [
    'scene_id',
    'sensor',
    'date',
    'year',
    'month',
    'spatial_scale'
]

value_columns = [
    'screened_valid_fraction',
    *[f'{i}_mean' for i in INDICES]
]

secondary = native.loc[
    native['forest_type'] == 'Late_secondary',
    keys + value_columns
]

mature = native.loc[
    native['forest_type'] == 'Mature',
    keys + value_columns
]

paired = secondary.merge(
    mature,
    on=keys,
    suffixes=('_secondary', '_mature'),
    validate='one_to_one'
)

assert len(paired) == 1164

paired['paired_coverage'] = paired[
    [
        'screened_valid_fraction_secondary',
        'screened_valid_fraction_mature'
    ]
].min(axis=1)

paired['flagged_scene'] = (
    paired['scene_id'].isin(flagged_scenes)
)

for index in INDICES:

    paired[f'{index}_contrast'] = (
        paired[f'{index}_mean_secondary']
        -
        paired[f'{index}_mean_mature']
    )

    paired[f'{index}_common_signal'] = (
        paired[f'{index}_mean_secondary']
        +
        paired[f'{index}_mean_mature']
    ) / 2

primary = paired.loc[
    (paired['spatial_scale'] == 'Fragment')
    &
    paired['month'].between(5, 9)
    &
    (paired['paired_coverage'] >= 0.70)
].copy()

assert len(primary) == 219

print('\nPRIMARY DATASET')
print('Scenes:', len(primary))
print('Years:', primary['year'].nunique())

# ============================================================
# 4. DEFINE SENSITIVITY SCENARIOS
# ============================================================

seasonal = paired.loc[
    paired['month'].between(5, 9)
].copy()

scenarios = {

    'Primary_fragment':
        primary.copy(),

    'Fragment_ge90':
        primary.loc[
            primary['paired_coverage'] >= 0.90
        ].copy(),

    'Fragment_no_flagged':
        primary.loc[
            ~primary['flagged_scene']
        ].copy(),

    'Fragment_ge90_no_flagged':
        primary.loc[
            (primary['paired_coverage'] >= 0.90)
            &
            ~primary['flagged_scene']
        ].copy(),

    'Fragment_JunSep':
        primary.loc[
            primary['month'].between(6, 9)
        ].copy(),

    'Fragment_L7_until2017':
        primary.loc[
            ~(
                (primary['sensor'] == 'Landsat_7')
                &
                (primary['year'] > 2017)
            )
        ].copy(),

    'Core_ge70':
        seasonal.loc[
            (
                seasonal['spatial_scale']
                == 'Fragment_core'
            )
            &
            (seasonal['paired_coverage'] >= 0.70)
        ].copy(),

    'Plot_ge70':
        seasonal.loc[
            seasonal['spatial_scale'] == 'Plot'
        ].loc[
            lambda x: x['paired_coverage'] >= 0.70
        ].copy()
}

scenario_support = pd.DataFrame([

    {
        'scenario': name,
        'n_scenes': len(frame),
        'n_years': frame['year'].nunique(),
        'n_flagged_scenes': int(
            frame['flagged_scene'].sum()
        )
    }

    for name, frame in scenarios.items()
])

print('\nSCENARIO SUPPORT')
display(scenario_support)

# ============================================================
# 5. ANNUAL SENSOR-SPECIFIC MEDIANS
# ============================================================

annual_rows = []

for scenario, frame in scenarios.items():

    for (sensor, year), group in frame.groupby(
        ['sensor', 'year']
    ):

        for index in INDICES:

            s_col = f'{index}_mean_secondary'
            m_col = f'{index}_mean_mature'

            valid = group.dropna(
                subset=[s_col, m_col]
            )

            if valid.empty:
                continue

            secondary_values = valid[s_col]
            mature_values = valid[m_col]

            contrast = (
                secondary_values
                - mature_values
            )

            common = (
                secondary_values
                + mature_values
            ) / 2

            annual_rows.append({

                'scenario': scenario,
                'sensor': sensor,
                'year': int(year),
                'index': index,

                'n_scenes': len(valid),

                'adequate_support':
                    len(valid) >= MIN_SCENES,

                'secondary':
                    secondary_values.median(),

                'mature':
                    mature_values.median(),

                'contrast':
                    contrast.median(),

                'common_signal':
                    common.median(),

                'absolute_contrast':
                    contrast.abs().median(),

                'secondary_iqr':
                    secondary_values.quantile(0.75)
                    -
                    secondary_values.quantile(0.25),

                'mature_iqr':
                    mature_values.quantile(0.75)
                    -
                    mature_values.quantile(0.25)
            })

annual = pd.DataFrame(annual_rows)

print('\nANNUAL DATA')
print('Rows:', len(annual))

# ============================================================
# 6. ROBUST STANDARDIZATION
# ============================================================

def robust_center_scale(values):

    values = np.asarray(
        values,
        dtype=float
    )

    values = values[
        np.isfinite(values)
    ]

    if len(values) == 0:
        return np.nan, np.nan

    center = np.median(values)

    mad = np.median(
        np.abs(values - center)
    )

    scale = 1.4826 * mad

    if not np.isfinite(scale) or scale <= 1e-10:

        q25, q75 = np.quantile(
            values,
            [0.25, 0.75]
        )

        scale = (
            q75 - q25
        ) / 1.349

    if not np.isfinite(scale) or scale <= 1e-10:
        scale = np.nan

    return center, scale


# ============================================================
# 7. HISTORICAL ANOMALY SCREENING
# ============================================================

# Each sensor and scenario receives its own
# robust reference distribution.
#
# Each target year is excluded from its own
# reference calculation.
#
# This detects unusual annual spectral states,
# not necessarily disturbance events.

anomaly_rows = []

for (
    scenario,
    sensor,
    index
), group in annual.groupby(
    ['scenario', 'sensor', 'index']
):

    group = group.loc[
        group['adequate_support']
    ].sort_values('year')

    if len(group) < MIN_BASELINE_YEARS:
        continue

    for _, row in group.iterrows():

        reference = group.loc[
            group['year'] != row['year']
        ]

        result = {

            'scenario': scenario,
            'sensor': sensor,
            'index': index,
            'year': int(row['year']),
            'n_scenes': int(row['n_scenes']),
            'n_reference_years': len(reference)
        }

        for variable in [
            'secondary',
            'mature',
            'contrast',
            'common_signal'
        ]:

            center, scale = robust_center_scale(
                reference[variable]
            )

            result[f'{variable}_value'] = (
                row[variable]
            )

            result[f'{variable}_reference'] = center

            result[f'{variable}_robust_scale'] = scale

            result[f'{variable}_z'] = (
                (row[variable] - center) / scale
                if np.isfinite(scale)
                else np.nan
            )

        anomaly_rows.append(result)

annual_anomalies = pd.DataFrame(
    anomaly_rows
)

assert not annual_anomalies.empty

# ============================================================
# 8. CLASSIFY SPECTRAL ANOMALIES
# ============================================================

def classify_anomaly(row, threshold):

    zs = row['secondary_z']
    zm = row['mature_z']
    zd = row['contrast_z']

    if not all(
        np.isfinite([zs, zm, zd])
    ):
        return 'Insufficient_reference'

    secondary_low = zs <= -threshold
    mature_low = zm <= -threshold

    if secondary_low and mature_low:

        if abs(zd) >= threshold:
            return 'Shared_decline_asymmetric'

        return 'Shared_decline'

    if secondary_low and zd <= -threshold:
        return 'Secondary_specific_decline'

    if mature_low and zd >= threshold:
        return 'Mature_specific_decline'

    if secondary_low:
        return 'Secondary_low_unresolved'

    if mature_low:
        return 'Mature_low_unresolved'

    if abs(zd) >= threshold:
        return 'Differential_anomaly'

    return 'No_major_anomaly'


classification_frames = []

for threshold in Z_SENSITIVITY:

    frame = annual_anomalies.copy()

    frame['z_threshold'] = threshold

    frame['anomaly_class'] = frame.apply(
        classify_anomaly,
        axis=1,
        threshold=threshold
    )

    classification_frames.append(frame)

classified = pd.concat(
    classification_frames,
    ignore_index=True
)

primary_anomalies = classified.loc[
    (classified['scenario'] == 'Primary_fragment')
    &
    (classified['z_threshold'] == Z_THRESHOLD)
].copy()

primary_flagged = primary_anomalies.loc[
    ~primary_anomalies['anomaly_class'].isin([
        'No_major_anomaly',
        'Insufficient_reference'
    ])
].copy()

print('\nPRIMARY ANNUAL ANOMALIES')

display(
    primary_flagged[
        [
            'sensor',
            'year',
            'index',
            'n_scenes',
            'secondary_z',
            'mature_z',
            'contrast_z',
            'anomaly_class'
        ]
    ].sort_values(
        ['year', 'sensor', 'index']
    ).round(4)
)

# ============================================================
# 9. CONSECUTIVE-YEAR CHANGE DETECTION
# ============================================================

# A large negative year-to-year change provides
# evidence of a possible abrupt spectral decline.
#
# Only consecutive adequately supported years
# from the same sensor are compared.

change_rows = []

for (
    scenario,
    sensor,
    index
), group in annual.groupby(
    ['scenario', 'sensor', 'index']
):

    group = group.loc[
        group['adequate_support']
    ].sort_values('year')

    if len(group) < MIN_BASELINE_YEARS:
        continue

    group = group.copy()

    group['previous_year'] = (
        group['year'].shift(1)
    )

    group['year_gap'] = (
        group['year']
        - group['previous_year']
    )

    for variable in [
        'secondary',
        'mature',
        'contrast',
        'common_signal'
    ]:

        group[f'{variable}_change'] = (
            group[variable].diff()
        )

    group = group.loc[
        group['year_gap'] == 1
    ].copy()

    # Require sufficient consecutive transitions
    # for robust change standardization.

    if len(group) < 6:
        continue

    for _, row in group.iterrows():

        reference = group.loc[
            group['year'] != row['year']
        ]

        result = {

            'scenario': scenario,
            'sensor': sensor,
            'index': index,
            'year': int(row['year']),
            'previous_year': int(
                row['previous_year']
            ),
            'n_reference_transitions':
                len(reference)
        }

        for variable in [
            'secondary',
            'mature',
            'contrast',
            'common_signal'
        ]:

            column = f'{variable}_change'

            center, scale = robust_center_scale(
                reference[column]
            )

            result[column] = row[column]

            result[f'{variable}_change_z'] = (
                (row[column] - center) / scale
                if np.isfinite(scale)
                else np.nan
            )

        change_rows.append(result)

annual_changes = pd.DataFrame(change_rows)

assert not annual_changes.empty

def classify_change(row, threshold=2.5):

    zs = row['secondary_change_z']
    zm = row['mature_change_z']
    zd = row['contrast_change_z']

    if not all(np.isfinite([zs, zm, zd])):
        return 'Insufficient_reference'

    secondary_drop = (
        zs <= -threshold
        and row['secondary_change'] < 0
    )

    mature_drop = (
        zm <= -threshold
        and row['mature_change'] < 0
    )

    if secondary_drop and mature_drop:
        return 'Shared_abrupt_decline'

    if secondary_drop and zd <= -threshold:
        return 'Secondary_specific_abrupt_decline'

    if mature_drop and zd >= threshold:
        return 'Mature_specific_abrupt_decline'

    if secondary_drop:
        return 'Secondary_decline_unresolved'

    if mature_drop:
        return 'Mature_decline_unresolved'

    return 'No_major_decline'


annual_changes['change_class'] = (
    annual_changes.apply(
        classify_change,
        axis=1
    )
)

primary_changes = annual_changes.loc[
    annual_changes['scenario'] == 'Primary_fragment'
].copy()

abrupt_candidates = primary_changes.loc[
    ~primary_changes['change_class'].isin([
        'No_major_decline',
        'Insufficient_reference'
    ])
].copy()

print('\nABRUPT CHANGE CANDIDATES')

display(
    abrupt_candidates[
        [
            'sensor',
            'year',
            'index',
            'secondary_change',
            'mature_change',
            'contrast_change',
            'change_class'
        ]
    ].sort_values(
        ['year', 'sensor', 'index']
    ).round(5)
)

# ============================================================
# 10. MULTI-INDEX DISTURBANCE CANDIDATES
# ============================================================

# Require agreement between NBR and NDMI
# for stronger disturbance candidates.
#
# NDVI and EVI provide supporting evidence
# but are not required for this classification.

event_rows = []

for (sensor, year), group in primary_changes.groupby(
    ['sensor', 'year']
):

    index_group = group.set_index('index')

    if not set(DISTURBANCE_INDICES).issubset(
        index_group.index
    ):
        continue

    secondary_support = []
    mature_support = []
    shared_support = []

    for index in DISTURBANCE_INDICES:

        row = index_group.loc[index]

        if (
            row['secondary_change_z']
            <= -Z_THRESHOLD
            and row['secondary_change'] < 0
        ):
            secondary_support.append(index)

        if (
            row['mature_change_z']
            <= -Z_THRESHOLD
            and row['mature_change'] < 0
        ):
            mature_support.append(index)

        if row['change_class'] == (
            'Shared_abrupt_decline'
        ):
            shared_support.append(index)

    if len(shared_support) == 2:

        event_class = 'Shared_multi_index_decline'

    elif (
        len(secondary_support) == 2
        and len(mature_support) == 0
        and all(
            index_group.loc[i, 'change_class']
            == 'Secondary_specific_abrupt_decline'
            for i in DISTURBANCE_INDICES
        )
    ):

        event_class = (
            'Secondary_specific_multi_index_decline'
        )

    elif (
        len(mature_support) == 2
        and len(secondary_support) == 0
        and all(
            index_group.loc[i, 'change_class']
            == 'Mature_specific_abrupt_decline'
            for i in DISTURBANCE_INDICES
        )
    ):

        event_class = (
            'Mature_specific_multi_index_decline'
        )

    elif (
        len(secondary_support)
        + len(mature_support)
    ) > 0:

        event_class = 'Partial_index_agreement'

    else:

        event_class = 'No_multi_index_decline'

    event_rows.append({

        'sensor': sensor,
        'year': int(year),

        'secondary_support':
            ';'.join(secondary_support),

        'mature_support':
            ';'.join(mature_support),

        'shared_support':
            ';'.join(shared_support),

        'event_class': event_class
    })

event_columns = [
    'sensor',
    'year',
    'secondary_support',
    'mature_support',
    'shared_support',
    'event_class'
]

events = pd.DataFrame(
    event_rows,
    columns=event_columns
)

candidate_events = events.loc[
    events['event_class']
    != 'No_multi_index_decline'
].copy()

print('\nMULTI-INDEX DISTURBANCE CANDIDATES')

display(
    candidate_events.sort_values(
        ['year', 'sensor']
    )
)

# ============================================================
# 11. LOCAL PRE/POST PERSISTENCE
# ============================================================

# For each candidate:
# compare the three calendar years preceding
# the candidate with the candidate year and
# two subsequent calendar years.
#
# Require at least two adequately supported
# years in each window.
#
# Persistence is descriptive and conditional
# on available annual observations.

persistence_rows = []

primary_annual = annual.loc[
    (
        annual['scenario']
        == 'Primary_fragment'
    )
    &
    annual['adequate_support']
].copy()

for _, event in candidate_events.iterrows():

    sensor = event['sensor']
    year = int(event['year'])

    for index in INDICES:

        series = primary_annual.loc[
            (primary_annual['sensor'] == sensor)
            &
            (primary_annual['index'] == index)
        ]

        before = series.loc[
            series['year'].between(
                year - 3,
                year - 1
            )
        ]

        after = series.loc[
            series['year'].between(
                year,
                year + 2
            )
        ]

        future_only = after.loc[
            after['year'] > year
        ]

        for forest in ['secondary', 'mature']:

            result = {

                'sensor': sensor,
                'candidate_year': year,
                'index': index,
                'forest': forest,

                'n_pre_years': len(before),
                'n_post_years': len(after),

                'pre_median': np.nan,
                'post_median': np.nan,
                'future_only_median': np.nan,
                'post_minus_pre': np.nan,
                'future_minus_pre': np.nan,

                'persistence_status':
                    'INSUFFICIENT_SUPPORT'
            }

            if (
                len(before)
                >= MIN_LOCAL_REFERENCE_YEARS
                and len(after)
                >= MIN_LOCAL_REFERENCE_YEARS
                and len(future_only) >= 1
            ):

                pre = before[forest].median()
                post = after[forest].median()

                future = (
                    future_only[forest].median()
                )

                result.update({

                    'pre_median': pre,
                    'post_median': post,
                    'future_only_median': future,

                    'post_minus_pre':
                        post - pre,

                    'future_minus_pre':
                        future - pre,

                    'persistence_status': (
                        'Lower_post_event_state'
                        if (
                            post < pre
                            and future < pre
                        )
                        else
                        'No_persistent_lower_state'
                    )
                })

            persistence_rows.append(result)

persistence_columns = [
    'sensor',
    'candidate_year',
    'index',
    'forest',
    'n_pre_years',
    'n_post_years',
    'pre_median',
    'post_median',
    'future_only_median',
    'post_minus_pre',
    'future_minus_pre',
    'persistence_status'
]

persistence = pd.DataFrame(
    persistence_rows,
    columns=persistence_columns
)

print('\nPRE/POST PERSISTENCE')

display(
    persistence.loc[
        persistence['index'].isin(
            DISTURBANCE_INDICES
        )
    ].round(5)
)

# ============================================================
# 12. EVENT SENSITIVITY ACROSS SCENARIOS
# ============================================================

event_sensitivity_rows = []

for _, event in candidate_events.iterrows():

    for scenario in scenarios:

        for index in DISTURBANCE_INDICES:

            match = annual_changes.loc[
                (annual_changes['scenario'] == scenario)
                &
                (annual_changes['sensor'] == event['sensor'])
                &
                (annual_changes['year'] == event['year'])
                &
                (annual_changes['index'] == index)
            ]

            result = {

                'sensor': event['sensor'],
                'year': int(event['year']),
                'primary_event_class':
                    event['event_class'],

                'scenario': scenario,
                'index': index,

                'secondary_change': np.nan,
                'mature_change': np.nan,
                'contrast_change': np.nan,

                'change_class':
                    'INSUFFICIENT_SUPPORT'
            }

            if len(match) == 1:

                row = match.iloc[0]

                for column in [
                    'secondary_change',
                    'mature_change',
                    'contrast_change',
                    'change_class'
                ]:

                    result[column] = row[column]

            event_sensitivity_rows.append(result)

event_sensitivity = pd.DataFrame(
    event_sensitivity_rows,
    columns=[
        'sensor',
        'year',
        'primary_event_class',
        'scenario',
        'index',
        'secondary_change',
        'mature_change',
        'contrast_change',
        'change_class'
    ]
)

print('\nEVENT SENSITIVITY')

display(
    event_sensitivity.head(30).round(5)
)

# ============================================================
# 13. SHARED VS DIFFERENTIAL VARIABILITY
# ============================================================

# Descriptive decomposition of paired annual
# variability into common and differential
# components, within each sensor.

covariance_rows = []

for (
    scenario,
    sensor,
    index
), group in annual.groupby(
    ['scenario', 'sensor', 'index']
):

    group = group.loc[
        group['adequate_support']
    ]

    if len(group) < MIN_BASELINE_YEARS:
        continue

    s = group['secondary'].to_numpy()
    m = group['mature'].to_numpy()

    common = (s + m) / 2
    differential = (s - m) / 2

    var_common = np.var(
        common,
        ddof=1
    )

    var_differential = np.var(
        differential,
        ddof=1
    )

    denominator = (
        var_common
        + var_differential
    )

    covariance_rows.append({

        'scenario': scenario,
        'sensor': sensor,
        'index': index,
        'n_years': len(group),

        'spearman_rho':
            spearmanr(s, m).statistic,

        'common_variance':
            var_common,

        'differential_variance':
            var_differential,

        'common_variance_fraction': (
            var_common / denominator
            if denominator > 0
            else np.nan
        ),

        'differential_variance_fraction': (
            var_differential / denominator
            if denominator > 0
            else np.nan
        )
    })

covariance = pd.DataFrame(
    covariance_rows
)

primary_covariance = covariance.loc[
    covariance['scenario']
    == 'Primary_fragment'
].copy()

print('\nSHARED VS DIFFERENTIAL VARIABILITY')

display(
    primary_covariance.round(5)
)

# ============================================================
# 14. SCENE-LEVEL DIAGNOSTIC AUDIT
# ============================================================

# Identify extreme scenes associated with
# candidate years.
#
# No observations are permanently removed.

candidate_year_keys = set(
    zip(
        candidate_events['sensor'],
        candidate_events['year']
    )
)

primary['candidate_event_year'] = [

    (sensor, year) in candidate_year_keys

    for sensor, year in zip(
        primary['sensor'],
        primary['year']
    )
]

event_scene_audit = primary.loc[
    primary['candidate_event_year']
    |
    primary['flagged_scene']
].copy()

scene_columns = [

    'scene_id',
    'sensor',
    'date',
    'year',
    'paired_coverage',
    'flagged_scene',
    'candidate_event_year',

    *[
        f'{index}_mean_{forest}'
        for index in INDICES
        for forest in ['secondary', 'mature']
    ],

    *[
        f'{index}_contrast'
        for index in INDICES
    ]
]

event_scene_audit = event_scene_audit[
    scene_columns
].sort_values(
    ['year', 'date', 'sensor']
)

print('\nSCENE-LEVEL AUDIT')
print('Rows:', len(event_scene_audit))

# ============================================================
# 15. FIGURE — ANNUAL DISTURBANCE DIAGNOSTICS
# ============================================================

plt.rcParams.update({

    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42
})

sensor_colors = {

    'Landsat_5': '#527B98',
    'Landsat_7': '#DB9140',
    'Landsat_8': '#24634F',
    'Landsat_9': '#795C92'
}

forest_colors = {

    'secondary': '#DB9140',
    'mature': '#24634F'
}

fig, axes = plt.subplots(
    2,
    3,
    figsize=(17, 10),
    sharex=True
)

for row_number, index in enumerate(
    DISTURBANCE_INDICES
):

    for sensor in SENSORS:

        group = primary_annual.loc[
            (primary_annual['index'] == index)
            &
            (primary_annual['sensor'] == sensor)
        ].sort_values('year')

        if group.empty:
            continue

        # Panel 1: secondary forest

        axes[row_number, 0].scatter(

            group['year'],
            group['secondary'],

            color=sensor_colors[sensor],
            s=35,

            label=sensor.replace('_', ' ')
            if row_number == 0
            else None
        )

        # Panel 2: mature forest

        axes[row_number, 1].scatter(

            group['year'],
            group['mature'],

            color=sensor_colors[sensor],
            s=35
        )

        # Panel 3: paired contrast

        axes[row_number, 2].scatter(

            group['year'],
            group['contrast'],

            color=sensor_colors[sensor],
            s=35
        )

    for col in range(3):

        ax = axes[row_number, col]

        ax.axvline(
            2019,
            color='#D95F59',
            linestyle='--',
            linewidth=1
        )

        ax.grid(alpha=0.15)

        ax.set_ylabel(
            index
            if col < 2
            else f'Δ{index}'
        )

        ax.set_xlim(1984, 2026)

    axes[row_number, 2].axhline(
        0,
        color='gray',
        linestyle=':',
        linewidth=1
    )

    # Highlight abrupt change candidates.

    subset = abrupt_candidates.loc[
        abrupt_candidates['index'] == index
    ]

    for _, candidate in subset.iterrows():

        sensor = candidate['sensor']
        year = candidate['year']

        observed = primary_annual.loc[
            (primary_annual['sensor'] == sensor)
            &
            (primary_annual['year'] == year)
            &
            (primary_annual['index'] == index)
        ]

        if len(observed) != 1:
            continue

        observed = observed.iloc[0]

        for col, variable in enumerate([
            'secondary',
            'mature',
            'contrast'
        ]):

            axes[row_number, col].scatter(

                year,
                observed[variable],

                facecolors='none',
                edgecolors='#B3261E',
                linewidths=1.6,
                s=125,
                zorder=5
            )

titles = [

    'Late-secondary forest',
    'Mature forest',
    'Within-scene contrast'
]

for col, title in enumerate(titles):

    axes[0, col].set_title(
        title,
        fontweight='bold'
    )

    axes[1, col].set_xlabel('Year')

fig.suptitle(

    'Historical spectral disturbance diagnostics',
    fontsize=17,
    fontweight='bold',
    y=0.98
)

handles, labels = (
    axes[0, 0].get_legend_handles_labels()
)

fig.legend(

    handles,
    labels,

    loc='upper center',
    bbox_to_anchor=(0.5, 0.93),
    ncol=4,
    frameon=False
)

fig.text(

    0.5,
    0.015,

    'May–September | Paired coverage ≥70% | '
    'At least three acquisitions per sensor-year. '
    'Red circles indicate candidate abrupt changes, '
    'not confirmed disturbances.',

    ha='center',
    fontsize=9
)

fig.tight_layout(
    rect=[0, 0.05, 1, 0.88]
)

FIGURE_1 = (
    FIGURE_DIR
    / 'FIG_HISTORICAL_DISTURBANCE_DIAGNOSTICS'
)

for extension in ['png', 'pdf', 'svg']:

    fig.savefig(

        f'{FIGURE_1}.{extension}',
        dpi=600,
        bbox_inches='tight'
    )

plt.show()

# ============================================================
# 16. FIGURE — SHARED SPECTRAL VARIABILITY
# ============================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(16, 5.5),
    sharey=True
)

for ax, sensor in zip(
    axes,
    SENSORS[:3]
):

    group = primary_covariance.loc[
        primary_covariance['sensor'] == sensor
    ].set_index('index').reindex(INDICES)

    positions = np.arange(len(INDICES))

    common_pct = (
        group['common_variance_fraction'] * 100
    )

    differential_pct = (
        group['differential_variance_fraction']
        * 100
    )

    ax.barh(

        positions,
        common_pct,

        color='#527B98',
        label='Common component'
    )

    ax.barh(

        positions,
        differential_pct,

        left=common_pct,

        color='#DB9140',
        label='Differential component'
    )

    ax.set_yticks(
        positions,
        INDICES
    )

    ax.set_xlim(0, 100)

    ax.set_xlabel(
        'Variance fraction (%)'
    )

    ax.set_title(

        sensor.replace('_', ' '),
        fontweight='bold'
    )

    ax.grid(
        axis='x',
        alpha=0.15
    )

axes[0].invert_yaxis()

fig.suptitle(

    'Shared and differential spectral variability '
    'between forest fragments',

    fontsize=16,
    fontweight='bold',
    y=0.98
)

handles, labels = (
    axes[0].get_legend_handles_labels()
)

fig.legend(

    handles,
    labels,

    loc='upper center',
    bbox_to_anchor=(0.5, 0.90),
    ncol=2,
    frameon=False
)

fig.text(

    0.5,
    0.015,

    'Descriptive decomposition of within-sensor annual '
    'median variability. Shared variability does not '
    'identify a common causal driver.',

    ha='center',
    fontsize=9
)

fig.tight_layout(
    rect=[0, 0.06, 1, 0.82]
)

FIGURE_2 = (
    FIGURE_DIR
    / 'FIG_HISTORICAL_SHARED_VARIABILITY'
)

for extension in ['png', 'pdf', 'svg']:

    fig.savefig(

        f'{FIGURE_2}.{extension}',
        dpi=600,
        bbox_inches='tight'
    )

plt.show()

# ============================================================
# 17. EXPORT RESULTS
# ============================================================

exports = {

    'DISTURBANCE_SCENARIO_SUPPORT':
        scenario_support,

    'DISTURBANCE_ANNUAL_SERIES':
        annual,

    'DISTURBANCE_ANNUAL_ANOMALIES':
        annual_anomalies,

    'DISTURBANCE_THRESHOLD_SENSITIVITY':
        classified,

    'DISTURBANCE_PRIMARY_ANOMALIES':
        primary_flagged,

    'DISTURBANCE_ANNUAL_CHANGES':
        annual_changes,

    'DISTURBANCE_ABRUPT_CANDIDATES':
        abrupt_candidates,

    'DISTURBANCE_MULTI_INDEX_EVENTS':
        events,

    'DISTURBANCE_CANDIDATE_EVENTS':
        candidate_events,

    'DISTURBANCE_PRE_POST_PERSISTENCE':
        persistence,

    'DISTURBANCE_EVENT_SENSITIVITY':
        event_sensitivity,

    'DISTURBANCE_SHARED_VARIABILITY':
        covariance,

    'DISTURBANCE_PRIMARY_VARIABILITY':
        primary_covariance,

    'DISTURBANCE_SCENE_AUDIT':
        event_scene_audit
}

for name, frame in exports.items():

    frame.to_csv(

        OUT / f'{name}.csv',

        index=False,
        encoding='utf-8-sig'
    )

WORKBOOK = (

    TABLE_DIR
    / 'SATIPO_HISTORICAL_DISTURBANCE.xlsx'
)

with pd.ExcelWriter(

    WORKBOOK,
    engine='openpyxl'

) as writer:

    sheets = {

        'Scenario_support':
            scenario_support,

        'Annual_series':
            annual,

        'Annual_anomalies':
            annual_anomalies,

        'Threshold_sensitivity':
            classified,

        'Primary_anomalies':
            primary_flagged,

        'Annual_changes':
            annual_changes,

        'Abrupt_candidates':
            abrupt_candidates,

        'Multi_index_events':
            events,

        'Candidate_events':
            candidate_events,

        'Pre_post_persistence':
            persistence,

        'Event_sensitivity':
            event_sensitivity,

        'Shared_variability':
            covariance,

        'Primary_variability':
            primary_covariance,

        'Scene_audit':
            event_scene_audit
    }

    for name, frame in sheets.items():

        frame.to_excel(

            writer,
            sheet_name=name,
            index=False
        )

# ============================================================
# 18. MANIFEST
# ============================================================

manifest = {

    'project': 'SATIPO_FOREST_PROJECT',

    'cell': '93',

    'created_utc':
        datetime.now(
            timezone.utc
        ).isoformat(),

    'input': str(SCREENED_PATH),

    'primary_scenes': int(len(primary)),

    'primary_years':
        int(primary['year'].nunique()),

    'minimum_scenes_per_year':
        MIN_SCENES,

    'minimum_baseline_years':
        MIN_BASELINE_YEARS,

    'robust_z_threshold':
        Z_THRESHOLD,

    'threshold_sensitivity':
        Z_SENSITIVITY,

    'disturbance_indices':
        DISTURBANCE_INDICES,

    'candidate_events':
        int(len(candidate_events)),

    'permanently_removed_scenes': 0,

    'sensor_harmonization':
        'None; within-sensor analysis',

    'event_confirmation':
        'Not established',

    'methodological_limits': [

        'Annual spectral anomalies are not '
        'confirmed ecological disturbances.',

        'Only adequately supported sensor-years '
        'are evaluated.',

        'Consecutive-year changes require '
        'consecutive calendar years.',

        'Persistence is descriptive and conditional '
        'on available observations.',

        'Robust thresholds are diagnostic and '
        'not calibrated false-positive probabilities.',

        'Shared variability may reflect climate, '
        'atmosphere, acquisition geometry or '
        'shared ecological processes.',

        'Two forest fragments do not provide '
        'independent replication of forest types.'
    ],

    'workbook': str(WORKBOOK),

    'figures': [

        str(FIGURE_1),
        str(FIGURE_2)
    ]
}

MANIFEST_PATH = (

    OUT / 'DISTURBANCE_MANIFEST.json'
)

with open(

    MANIFEST_PATH,
    'w',
    encoding='utf-8'

) as f:

    json.dump(

        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

# ============================================================
# 19. FINAL REPORT
# ============================================================

print('\n' + '=' * 78)
print('SATIPO — HISTORICAL DISTURBANCE REPORT')
print('=' * 78)

print('Primary scenes:', len(primary))

print(
    'Primary annual anomalies:',
    len(primary_flagged)
)

print(
    'Abrupt index-change candidates:',
    len(abrupt_candidates)
)

print(
    'Multi-index candidate events:',
    len(candidate_events)
)

print(
    'Permanently removed observations:',
    0
)

print('\nCANDIDATE EVENT CLASSIFICATION')

display(

    candidate_events.sort_values(
        ['year', 'sensor']
    )
)

print('\nSHARED VARIABILITY — PRIMARY DATASET')

display(

    primary_covariance[
        [
            'sensor',
            'index',
            'n_years',
            'spearman_rho',
            'common_variance_fraction',
            'differential_variance_fraction'
        ]
    ].round(5)
)

print('\nOUTPUT DIRECTORY:')
print(OUT)

print('\nWORKBOOK:')
print(WORKBOOK)

print('\nMANIFEST:')
print(MANIFEST_PATH)

print('\n' + '=' * 78)
print('CELL 93 COMPLETED')
print('NEXT: CELL 94 — SPATIAL DISTURBANCE VALIDATION')
print('=' * 78)

In [ ]:
# ============================================================
# CELL 94
# SPATIAL DISTURBANCE VALIDATION
# SATIPO FOREST PROJECT
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import ee
import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import json
import requests
import zipfile
import io
import rasterio

from rasterio.features import geometry_mask
from matplotlib.colors import TwoSlopeNorm
from shapely.geometry import mapping

# ============================================================
# 1. CONFIGURATION
# ============================================================

ROOT = Path(
    '/content/drive/MyDrive/SATIPO_FOREST_PROJECT'
)

REMOTE = ROOT / '04_REMOTE_SENSING'

LANDSAT = REMOTE / '03_LANDSAT'

OUT = LANDSAT / '07_SPATIAL_DISTURBANCE'

RASTER_DIR = OUT / 'GEOTIFF'
TABLE_DIR = REMOTE / '08_OUTPUTS' / 'TABLES'
FIGURE_DIR = REMOTE / '08_OUTPUTS' / 'FIGURES'

for directory in [
    OUT, RASTER_DIR, TABLE_DIR, FIGURE_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

CRS = 'EPSG:32718'

TRANSFORM = [
    30, 0, 0,
    0, -30, 10000000
]

MIN_PIXEL_OBSERVATIONS = 2

# Diagnostic thresholds, not calibrated
# ecological disturbance thresholds.

CHANGE_THRESHOLDS = [
    -0.025,
    -0.05,
    -0.10,
    -0.20
]

PRIMARY_THRESHOLD = -0.05

EVENTS = {

    'Event_1991': {

        'sensor': 'Landsat_5',

        'pre_years': [
            1988, 1989, 1990
        ],

        'event_year': 1991,

        'post_years': [
            1992, 1993
        ]
    },

    'Event_2021': {

        'sensor': 'Landsat_8',

        'pre_years': [
            2018, 2019, 2020
        ],

        'event_year': 2021,

        'post_years': [
            2022, 2023
        ]
    }
}

COLLECTIONS = {

    'Landsat_5':
        'LANDSAT/LT05/C02/T1_L2',

    'Landsat_7':
        'LANDSAT/LE07/C02/T1_L2',

    'Landsat_8':
        'LANDSAT/LC08/C02/T1_L2',

    'Landsat_9':
        'LANDSAT/LC09/C02/T1_L2'
}

# ============================================================
# 2. EARTH ENGINE CONNECTION
# ============================================================

try:

    ee.Number(1).getInfo()

    print('Earth Engine connection: ACTIVE')

except Exception:

    ee.Authenticate()
    ee.Initialize()

    print('Earth Engine initialized')

# ============================================================
# 3. RECOVER VALIDATED GEOMETRIES
# ============================================================

REQUIRED_UNITS = {

    'FRAG_01',
    'FRAG_02',
    'FRAG_01_CORE30',
    'FRAG_02_CORE30',
    'P01',
    'P02'
}

geometry_candidates = []

# Search previously created GeoDataFrames.

for variable_name, obj in list(globals().items()):

    if isinstance(obj, gpd.GeoDataFrame):

        if 'unit_id' in obj.columns:

            ids = set(
                obj['unit_id'].astype(str)
            )

            if REQUIRED_UNITS.issubset(ids):

                geometry_candidates.append(
                    (
                        variable_name,
                        obj.copy()
                    )
                )

# Search validated vector files if necessary.

if not geometry_candidates:

    search_directories = [

        REMOTE / '02_GEOMETRY_VALIDATION',

        REMOTE / '01_VECTOR_INPUTS'
    ]

    for directory in search_directories:

        if not directory.exists():
            continue

        for path in directory.rglob('*'):

            if path.suffix.lower() not in [
                '.gpkg',
                '.geojson',
                '.shp'
            ]:
                continue

            try:

                if path.suffix.lower() == '.gpkg':

                    layers = gpd.list_layers(
                        path
                    )['name'].tolist()

                else:

                    layers = [None]

                for layer in layers:

                    frame = gpd.read_file(
                        path,
                        layer=layer
                    )

                    if 'unit_id' not in frame:
                        continue

                    ids = set(
                        frame['unit_id'].astype(str)
                    )

                    if REQUIRED_UNITS.issubset(ids):

                        geometry_candidates.append(
                            (
                                str(path),
                                frame
                            )
                        )

            except Exception as error:

                print(
                    'Vector skipped:',
                    path.name,
                    str(error)[:100]
                )

assert geometry_candidates, (
    'Validated six-unit geometry not found.'
)

geometry_source, units = geometry_candidates[0]

units = units.loc[
    units['unit_id'].isin(REQUIRED_UNITS)
].copy()

assert units['unit_id'].is_unique
assert len(units) == 6
assert units.crs is not None
assert units.geometry.is_valid.all()
assert (~units.geometry.is_empty).all()

units = units.to_crs(CRS)

units['area_ha'] = (
    units.geometry.area / 10000
)

print('\nGEOMETRY SOURCE')
print(geometry_source)

display(
    units[
        ['unit_id', 'area_ha']
    ]
)

# ============================================================
# 4. DEFINE EXPORT REGION
# ============================================================

fragments = units.loc[
    units['unit_id'].isin([
        'FRAG_01',
        'FRAG_02'
    ])
].copy()

# Include nearby landscape for visual context.

region_utm = (
    fragments.geometry.union_all()
    .buffer(300)
    .envelope
)

region_wgs = gpd.GeoSeries(
    [region_utm],
    crs=CRS
).to_crs('EPSG:4326').iloc[0]

region = ee.Geometry(
    mapping(region_wgs),
    proj='EPSG:4326',
    geodesic=False
)

# ============================================================
# 5. RECOVER PRIMARY SCENE SELECTION
# ============================================================

SCREENED_PATH = (

    LANDSAT
    / '03_RADIOMETRIC_SCREENING'
    / 'LANDSAT_SCREENED_MASTER.csv'
)

# Fallback for alternative screening directory.

if not SCREENED_PATH.exists():

    matches = list(
        LANDSAT.rglob(
            'LANDSAT_SCREENED_MASTER.csv'
        )
    )

    assert len(matches) == 1, matches

    SCREENED_PATH = matches[0]

screened = pd.read_csv(
    SCREENED_PATH,
    low_memory=False
)

screened['date'] = pd.to_datetime(
    screened['date']
)

fragment_data = screened.loc[
    screened['spatial_scale'] == 'Fragment'
].copy()

scene_keys = [

    'scene_id',
    'sensor',
    'date',
    'year',
    'month'
]

scene_coverage = fragment_data.pivot(

    index=scene_keys,

    columns='forest_type',

    values='screened_valid_fraction'

).reset_index()

scene_coverage['paired_coverage'] = (

    scene_coverage[
        ['Late_secondary', 'Mature']
    ].min(axis=1)
)

primary_scenes = scene_coverage.loc[

    scene_coverage['month'].between(5, 9)

    &

    (
        scene_coverage['paired_coverage']
        >= 0.70
    )

].copy()

assert len(primary_scenes) == 219

print('\nPRIMARY SCENES:', len(primary_scenes))

# ============================================================
# 6. LOAD ANOMALY FLAGS
# ============================================================

AUDIT_PATH = (

    LANDSAT
    / '05_HISTORICAL_TRAJECTORIES'
    / 'LANDSAT_EXTREME_OBSERVATION_AUDIT.csv'
)

anomaly_audit = pd.read_csv(
    AUDIT_PATH
)

flagged_ids = set(
    anomaly_audit['scene_id']
)

primary_scenes['flagged_scene'] = (

    primary_scenes['scene_id']
    .isin(flagged_ids)
)

# ============================================================
# 7. EVENT-SPECIFIC SCENE SUPPORT
# ============================================================

support_rows = []

for event_name, config in EVENTS.items():

    years = (

        config['pre_years']

        + [config['event_year']]

        + config['post_years']
    )

    for year in years:

        selected = primary_scenes.loc[

            (
                primary_scenes['sensor']
                == config['sensor']
            )

            &

            (
                primary_scenes['year']
                == year
            )
        ]

        support_rows.append({

            'event': event_name,

            'sensor': config['sensor'],

            'year': year,

            'n_scenes': len(selected),

            'n_flagged': int(
                selected['flagged_scene'].sum()
            ),

            'n_ge90': int(
                (
                    selected['paired_coverage']
                    >= 0.90
                ).sum()
            ),

            'n_no_flagged': int(
                (~selected['flagged_scene']).sum()
            ),

            'adequate_year':
                len(selected) >= 3
        })

event_support = pd.DataFrame(
    support_rows
)

print('\nEVENT SCENE SUPPORT')

display(event_support)

event_support.to_csv(

    OUT / 'SPATIAL_EVENT_SCENE_SUPPORT.csv',

    index=False
)

# ============================================================
# 8. LANDSAT IMAGE RESOLUTION
# ============================================================

def resolve_landsat_image(
    scene_id,
    sensor
):

    collection = ee.ImageCollection(
        COLLECTIONS[sensor]
    )

    scene_id = str(scene_id)

    # Support both LANDSAT_PRODUCT_ID
    # and Earth Engine system:index.

    matches = collection.filter(

        ee.Filter.Or(

            ee.Filter.eq(
                'LANDSAT_PRODUCT_ID',
                scene_id
            ),

            ee.Filter.eq(
                'system:index',
                scene_id
            )
        )
    )

    n_matches = matches.size().getInfo()

    assert n_matches == 1, (
        sensor,
        scene_id,
        n_matches
    )

    return ee.Image(matches.first())


# ============================================================
# 9. PIXEL-LEVEL SPECTRAL PREPROCESSING
# ============================================================

def prepare_disturbance_image(
    image,
    sensor
):

    qa = image.select('QA_PIXEL')

    # Fill, dilated cloud, cloud,
    # cloud shadow and snow.

    bits = [0, 1, 3, 4, 5]

    if sensor in [
        'Landsat_8',
        'Landsat_9'
    ]:

        bits.append(2)

    mask = ee.Image.constant(1)

    for bit in bits:

        mask = mask.And(

            qa.bitwiseAnd(
                1 << bit
            ).eq(0)
        )

    mask = mask.And(

        image.select(
            'QA_RADSAT'
        ).eq(0)
    )

    if sensor in [
        'Landsat_5',
        'Landsat_7'
    ]:

        source_bands = [

            'SR_B1',
            'SR_B2',
            'SR_B3',
            'SR_B4',
            'SR_B5',
            'SR_B7'
        ]

    else:

        source_bands = [

            'SR_B2',
            'SR_B3',
            'SR_B4',
            'SR_B5',
            'SR_B6',
            'SR_B7'
        ]

    reflectance = image.select(

        source_bands,

        [
            'Blue',
            'Green',
            'Red',
            'NIR',
            'SWIR1',
            'SWIR2'
        ]

    ).multiply(

        0.0000275

    ).add(

        -0.2

    ).updateMask(mask)

    nir = reflectance.select('NIR')

    swir1 = reflectance.select('SWIR1')

    swir2 = reflectance.select('SWIR2')

    def normalized_index(
        a,
        b,
        name
    ):

        denominator = a.add(b)

        physical = (

            a.gte(0)

            .And(a.lte(1))

            .And(b.gte(0))

            .And(b.lte(1))

            .And(
                denominator.gt(1e-6)
            )
        )

        result = (

            a.subtract(b)

            .divide(denominator)

            .updateMask(physical)

            .rename(name)
        )

        return result.updateMask(

            result.gte(-1)

            .And(result.lte(1))
        )

    nbr = normalized_index(

        nir,
        swir2,
        'NBR'
    )

    ndmi = normalized_index(

        nir,
        swir1,
        'NDMI'
    )

    return nbr.addBands(

        ndmi

    ).copyProperties(

        image,

        ['system:time_start']
    )


# ============================================================
# 10. CACHE PREPROCESSED SCENES
# ============================================================

image_cache = {}

def get_prepared_image(
    scene_id,
    sensor
):

    key = (
        sensor,
        scene_id
    )

    if key not in image_cache:

        raw = resolve_landsat_image(

            scene_id,
            sensor
        )

        image_cache[key] = (

            prepare_disturbance_image(

                raw,
                sensor
            )
        )

    return image_cache[key]


# ============================================================
# 11. ANNUAL COMPOSITES
# ============================================================

def build_annual_composite(
    sensor,
    year,
    exclude_flagged=False,
    minimum_coverage=0.70
):

    selected = primary_scenes.loc[

        (
            primary_scenes['sensor']
            == sensor
        )

        &

        (
            primary_scenes['year']
            == year
        )

        &

        (
            primary_scenes['paired_coverage']
            >= minimum_coverage
        )

    ].copy()

    if exclude_flagged:

        selected = selected.loc[
            ~selected['flagged_scene']
        ]

    if len(selected) < 3:

        return None, len(selected)

    images = [

        get_prepared_image(

            row.scene_id,
            row.sensor
        )

        for row in selected.itertuples()
    ]

    collection = ee.ImageCollection.fromImages(
        images
    )

    count = collection.count()

    composite = collection.median()

    composite = composite.updateMask(

        count.gte(
            MIN_PIXEL_OBSERVATIONS
        )
    )

    return composite, len(selected)


# ============================================================
# 12. MULTI-YEAR REFERENCE COMPOSITES
# ============================================================

def build_period_composite(
    sensor,
    years,
    exclude_flagged=False,
    minimum_coverage=0.70
):

    images = []
    supported_years = []

    for year in years:

        composite, n_scenes = (

            build_annual_composite(

                sensor,
                year,
                exclude_flagged,
                minimum_coverage
            )
        )

        if composite is not None:

            images.append(composite)

            supported_years.append(year)

    if not images:

        return None, []

    collection = ee.ImageCollection.fromImages(
        images
    )

    period = collection.median()

    # Require valid observations in at least
    # two annual composites when available.

    min_years = min(
        2,
        len(images)
    )

    period = period.updateMask(

        collection.count().gte(
            min_years
        )
    )

    return period, supported_years


# ============================================================
# 13. BUILD SPATIAL EVENT PRODUCTS
# ============================================================

def build_event_product(
    event_name,
    config,
    exclude_flagged=False,
    minimum_coverage=0.70
):

    sensor = config['sensor']

    pre, pre_years = (

        build_period_composite(

            sensor,

            config['pre_years'],

            exclude_flagged,

            minimum_coverage
        )
    )

    event, n_event_scenes = (

        build_annual_composite(

            sensor,

            config['event_year'],

            exclude_flagged,

            minimum_coverage
        )
    )

    post, post_years = (

        build_period_composite(

            sensor,

            config['post_years'],

            exclude_flagged,

            minimum_coverage
        )
    )

    metadata = {

        'event': event_name,

        'sensor': sensor,

        'event_year':
            config['event_year'],

        'pre_years':
            pre_years,

        'post_years':
            post_years,

        'event_scenes':
            n_event_scenes,

        'exclude_flagged':
            exclude_flagged,

        'minimum_coverage':
            minimum_coverage
    }

    if (

        pre is None

        or event is None

        or post is None

        or len(pre_years) < 2

    ):

        metadata['status'] = (
            'INSUFFICIENT_SUPPORT'
        )

        return None, metadata

    metadata['status'] = 'ESTIMATED'

    # Event minus pre:
    # negative = spectral decline.

    event_change = event.subtract(
        pre
    )

    # Post minus event:
    # positive = spectral rebound.

    rebound = post.subtract(
        event
    )

    # Post minus pre:
    # negative = remaining spectral deficit.

    residual = post.subtract(
        pre
    )

    product = (

        pre.rename([
            'NBR_pre',
            'NDMI_pre'
        ])

        .addBands(

            event.rename([
                'NBR_event',
                'NDMI_event'
            ])
        )

        .addBands(

            post.rename([
                'NBR_post',
                'NDMI_post'
            ])
        )

        .addBands(

            event_change.rename([
                'NBR_event_change',
                'NDMI_event_change'
            ])
        )

        .addBands(

            rebound.rename([
                'NBR_rebound',
                'NDMI_rebound'
            ])
        )

        .addBands(

            residual.rename([
                'NBR_residual',
                'NDMI_residual'
            ])
        )
    )

    return product.toFloat(), metadata


BAND_NAMES = [

    'NBR_pre',
    'NDMI_pre',

    'NBR_event',
    'NDMI_event',

    'NBR_post',
    'NDMI_post',

    'NBR_event_change',
    'NDMI_event_change',

    'NBR_rebound',
    'NDMI_rebound',

    'NBR_residual',
    'NDMI_residual'
]

# ============================================================
# 14. EXPORT GEOTIFFS
# ============================================================

def download_event_raster(
    image,
    output_path
):

    if output_path.exists():

        with rasterio.open(
            output_path
        ) as src:

            if (

                src.count == len(BAND_NAMES)

                and src.crs.to_string() == CRS

            ):

                print(
                    'Cached:',
                    output_path.name
                )

                return

    # Explicit NoData preserves invalid pixels.

    export_image = image.unmask(
        -9999,
        False
    )

    url = export_image.getDownloadURL({

        'name': output_path.stem,

        'region': region,

        'crs': CRS,

        'crs_transform': TRANSFORM,

        'format': 'GEO_TIFF',

        'filePerBand': False
    })

    response = requests.get(

        url,

        timeout=180
    )

    response.raise_for_status()

    temporary = output_path.with_suffix(
        '.tmp'
    )

    temporary.write_bytes(
        response.content
    )

    # Earth Engine may return a ZIP archive.

    if zipfile.is_zipfile(temporary):

        with zipfile.ZipFile(
            temporary
        ) as archive:

            tif_names = [

                name

                for name in archive.namelist()

                if name.lower().endswith(
                    '.tif'
                )
            ]

            assert len(tif_names) == 1

            output_path.write_bytes(

                archive.read(
                    tif_names[0]
                )
            )

        temporary.unlink()

    else:

        temporary.replace(
            output_path
        )

    with rasterio.open(

        output_path,

        'r+'

    ) as src:

        assert src.count == len(BAND_NAMES)

        assert src.crs.to_string() == CRS

        src.nodata = -9999

        for number, name in enumerate(

            BAND_NAMES,

            start=1
        ):

            src.set_band_description(

                number,
                name
            )

    print(
        'Exported:',
        output_path.name
    )


# ============================================================
# 15. PRIMARY AND SENSITIVITY SCENARIOS
# ============================================================

SPATIAL_SCENARIOS = {

    'Primary': {

        'exclude_flagged': False,

        'minimum_coverage': 0.70
    },

    'No_flagged': {

        'exclude_flagged': True,

        'minimum_coverage': 0.70
    },

    'Coverage_ge90': {

        'exclude_flagged': False,

        'minimum_coverage': 0.90
    },

    'Coverage_ge90_no_flagged': {

        'exclude_flagged': True,

        'minimum_coverage': 0.90
    }
}

product_metadata = []

raster_products = {}

for event_name, config in EVENTS.items():

    for scenario, options in (
        SPATIAL_SCENARIOS.items()
    ):

        print(
            '\nPROCESSING:',
            event_name,
            scenario
        )

        product, metadata = (

            build_event_product(

                event_name,

                config,

                **options
            )
        )

        metadata['scenario'] = scenario

        product_metadata.append(metadata)

        if product is None:

            print(
                'INSUFFICIENT SUPPORT'
            )

            continue

        path = (

            RASTER_DIR

            / (
                f'{event_name}_'
                f'{scenario}.tif'
            )
        )

        download_event_raster(

            product,
            path
        )

        raster_products[
            (event_name, scenario)
        ] = path

product_support = pd.DataFrame(
    product_metadata
)

print('\nSPATIAL PRODUCT SUPPORT')

display(product_support)

# ============================================================
# 16. PIXEL-LEVEL STATISTICS
# ============================================================

summary_rows = []

threshold_rows = []

for (
    event_name,
    scenario
), path in raster_products.items():

    with rasterio.open(path) as src:

        data = src.read().astype(float)

        data[
            data == -9999
        ] = np.nan

        data[
            ~np.isfinite(data)
        ] = np.nan

        bands = {

            name: data[i]

            for i, name in enumerate(
                BAND_NAMES
            )
        }

        pixel_area_ha = abs(

            src.transform.a

            * src.transform.e

        ) / 10000

        for unit in units.itertuples():

            inside = geometry_mask(

                [mapping(unit.geometry)],

                out_shape=(
                    src.height,
                    src.width
                ),

                transform=src.transform,

                invert=True,

                all_touched=False
            )

            n_grid_pixels = int(
                inside.sum()
            )

            for index in ['NBR', 'NDMI']:

                pre = bands[
                    f'{index}_pre'
                ]

                event = bands[
                    f'{index}_event'
                ]

                post = bands[
                    f'{index}_post'
                ]

                change = bands[
                    f'{index}_event_change'
                ]

                rebound = bands[
                    f'{index}_rebound'
                ]

                residual = bands[
                    f'{index}_residual'
                ]

                # Use common valid spatial
                # support for all comparisons.

                valid = (

                    inside

                    & np.isfinite(pre)

                    & np.isfinite(event)

                    & np.isfinite(post)
                )

                n_valid = int(
                    valid.sum()
                )

                if n_valid == 0:
                    continue

                decline = (

                    valid

                    & (
                        change
                        <= PRIMARY_THRESHOLD
                    )
                )

                n_decline = int(
                    decline.sum()
                )

                rebound_pixels = (

                    decline

                    & (rebound > 0)
                )

                residual_pixels = (

                    decline

                    & (
                        residual
                        <= PRIMARY_THRESHOLD
                    )
                )

                summary_rows.append({

                    'event': event_name,

                    'scenario': scenario,

                    'unit_id': unit.unit_id,

                    'index': index,

                    'polygon_area_ha':
                        unit.area_ha,

                    'grid_pixels':
                        n_grid_pixels,

                    'valid_pixels':
                        n_valid,

                    'valid_grid_fraction':
                        n_valid / n_grid_pixels,

                    'valid_pixel_area_ha':
                        n_valid * pixel_area_ha,

                    'pre_median':
                        np.nanmedian(pre[valid]),

                    'event_median':
                        np.nanmedian(event[valid]),

                    'post_median':
                        np.nanmedian(post[valid]),

                    'event_change_mean':
                        np.nanmean(change[valid]),

                    'event_change_median':
                        np.nanmedian(change[valid]),

                    'event_change_q25':
                        np.nanquantile(
                            change[valid],
                            0.25
                        ),

                    'event_change_q75':
                        np.nanquantile(
                            change[valid],
                            0.75
                        ),

                    'rebound_median':
                        np.nanmedian(
                            rebound[valid]
                        ),

                    'residual_median':
                        np.nanmedian(
                            residual[valid]
                        ),

                    'decline_pixel_pct':
                        100 * n_decline / n_valid,

                    'decline_pixel_area_ha':
                        n_decline * pixel_area_ha,

                    'rebound_among_declined_pct':

                        100
                        * rebound_pixels.sum()
                        / n_decline

                        if n_decline > 0

                        else np.nan,

                    'persistent_deficit_among_declined_pct':

                        100
                        * residual_pixels.sum()
                        / n_decline

                        if n_decline > 0

                        else np.nan
                })

                for threshold in (
                    CHANGE_THRESHOLDS
                ):

                    affected = (

                        valid

                        & (
                            change <= threshold
                        )
                    )

                    threshold_rows.append({

                        'event': event_name,

                        'scenario': scenario,

                        'unit_id': unit.unit_id,

                        'index': index,

                        'threshold': threshold,

                        'valid_pixels': n_valid,

                        'decline_pixels':
                            int(affected.sum()),

                        'decline_pct':

                            100
                            * affected.sum()
                            / n_valid,

                        'decline_area_ha':

                            affected.sum()
                            * pixel_area_ha
                    })

spatial_summary = pd.DataFrame(
    summary_rows
)

threshold_sensitivity = pd.DataFrame(
    threshold_rows
)

assert not spatial_summary.empty

print('\nPRIMARY SPATIAL RESULTS')

display(

    spatial_summary.loc[

        (
            spatial_summary['scenario']
            == 'Primary'
        )

        &

        spatial_summary['unit_id'].isin([
            'FRAG_01',
            'FRAG_02'
        ])

    ].round(4)
)

# ============================================================
# 17. MULTI-INDEX SPATIAL AGREEMENT
# ============================================================

agreement_rows = []

for (
    event_name,
    scenario
), path in raster_products.items():

    with rasterio.open(path) as src:

        data = src.read().astype(float)

        data[
            data == -9999
        ] = np.nan

        nbr = data[
            BAND_NAMES.index(
                'NBR_event_change'
            )
        ]

        ndmi = data[
            BAND_NAMES.index(
                'NDMI_event_change'
            )
        ]

        for unit in units.itertuples():

            inside = geometry_mask(

                [mapping(unit.geometry)],

                out_shape=(
                    src.height,
                    src.width
                ),

                transform=src.transform,

                invert=True,

                all_touched=False
            )

            valid = (

                inside

                & np.isfinite(nbr)

                & np.isfinite(ndmi)
            )

            n_valid = int(
                valid.sum()
            )

            if n_valid == 0:
                continue

            for threshold in (
                CHANGE_THRESHOLDS
            ):

                nbr_decline = (

                    valid

                    & (nbr <= threshold)
                )

                ndmi_decline = (

                    valid

                    & (ndmi <= threshold)
                )

                both = (

                    nbr_decline

                    & ndmi_decline
                )

                either = (

                    nbr_decline

                    | ndmi_decline
                )

                agreement_rows.append({

                    'event': event_name,

                    'scenario': scenario,

                    'unit_id': unit.unit_id,

                    'threshold': threshold,

                    'n_valid_pixels': n_valid,

                    'NBR_decline_pct':

                        100
                        * nbr_decline.sum()
                        / n_valid,

                    'NDMI_decline_pct':

                        100
                        * ndmi_decline.sum()
                        / n_valid,

                    'both_decline_pct':

                        100
                        * both.sum()
                        / n_valid,

                    'either_decline_pct':

                        100
                        * either.sum()
                        / n_valid,

                    'jaccard_overlap':

                        both.sum()
                        / either.sum()

                        if either.sum() > 0

                        else np.nan
                })

spatial_agreement = pd.DataFrame(
    agreement_rows
)

print('\nMULTI-INDEX SPATIAL AGREEMENT')

display(

    spatial_agreement.loc[

        (
            spatial_agreement['scenario']
            == 'Primary'
        )

        &

        (
            spatial_agreement['threshold']
            == PRIMARY_THRESHOLD
        )

        &

        spatial_agreement['unit_id'].isin([
            'FRAG_01',
            'FRAG_02'
        ])

    ].round(4)
)

# ============================================================
# 18. SPATIAL MAP FIGURES
# ============================================================

plt.rcParams.update({

    'font.family': 'DejaVu Sans',

    'font.size': 10,

    'pdf.fonttype': 42,

    'ps.fonttype': 42
})

figure_paths = []

for event_name in EVENTS:

    key = (
        event_name,
        'Primary'
    )

    if key not in raster_products:
        continue

    path = raster_products[key]

    with rasterio.open(path) as src:

        data = src.read().astype(float)

        data[
            data == -9999
        ] = np.nan

        extent = [

            src.bounds.left,

            src.bounds.right,

            src.bounds.bottom,

            src.bounds.top
        ]

    fig, axes = plt.subplots(

        2,

        3,

        figsize=(15, 10)
    )

    columns = [

        (
            'event_change',
            'Event minus pre-event'
        ),

        (
            'rebound',
            'Post-event minus event'
        ),

        (
            'residual',
            'Post-event minus pre-event'
        )
    ]

    # Common color scale across all panels
    # and both historical events.

    norm = TwoSlopeNorm(

        vmin=-0.40,

        vcenter=0,

        vmax=0.40
    )

    cmap = plt.get_cmap(
        'RdBu'
    ).copy()

    cmap.set_bad(
        '#E5E5E5'
    )

    for i, index in enumerate(
        ['NBR', 'NDMI']
    ):

        for j, (
            suffix,
            title
        ) in enumerate(columns):

            ax = axes[i, j]

            name = (
                f'{index}_{suffix}'
            )

            raster = data[
                BAND_NAMES.index(name)
            ]

            im = ax.imshow(

                raster,

                extent=extent,

                origin='upper',

                cmap=cmap,

                norm=norm
            )

            fragments.boundary.plot(

                ax=ax,

                color='black',

                linewidth=1.2
            )

            units.loc[

                units['unit_id'].isin([
                    'P01',
                    'P02'
                ])

            ].boundary.plot(

                ax=ax,

                color='#F5D547',

                linewidth=1.2,

                linestyle='--'
            )

            for row in (
                fragments.itertuples()
            ):

                point = (
                    row.geometry
                    .representative_point()
                )

                label = (

                    'Secondary'

                    if row.unit_id == 'FRAG_01'

                    else 'Mature'
                )

                ax.text(

                    point.x,

                    point.y,

                    label,

                    ha='center',

                    va='center',

                    fontsize=8,

                    bbox={

                        'facecolor': 'white',

                        'alpha': 0.75,

                        'edgecolor': 'none',

                        'pad': 2
                    }
                )

            ax.set_xlim(
                extent[0],
                extent[1]
            )

            ax.set_ylim(
                extent[2],
                extent[3]
            )

            ax.set_aspect('equal')

            ax.set_title(

                f'{index}: {title}',

                fontweight='bold'
            )

            ax.set_xticks([])
            ax.set_yticks([])

    fig.suptitle(

        f'Spatial spectral disturbance '
        f'diagnostics — {event_name[-4:]}',

        fontsize=17,

        fontweight='bold',

        y=0.98
    )

    fig.subplots_adjust(

        left=0.04,

        right=0.96,

        top=0.91,

        bottom=0.15,

        wspace=0.08,

        hspace=0.18
    )

    colorbar_ax = fig.add_axes([

        0.25,

        0.075,

        0.50,

        0.025
    ])

    colorbar = fig.colorbar(

        im,

        cax=colorbar_ax,

        orientation='horizontal',

        extend='both'
    )

    colorbar.set_label(

        'Spectral index difference'
    )

    fig.text(

        0.5,

        0.025,

        'Black outlines: forest fragments | '
        'Yellow dashed outlines: field plots | '
        'Gray: insufficient valid observations',

        ha='center',

        fontsize=9
    )

    stem = (

        FIGURE_DIR

        / (
            f'FIG_SPATIAL_DISTURBANCE_'
            f'{event_name[-4:]}'
        )
    )

    for extension in [
        'png',
        'pdf',
        'svg'
    ]:

        fig.savefig(

            f'{stem}.{extension}',

            dpi=600,

            bbox_inches='tight'
        )

    figure_paths.append(
        str(stem)
    )

    plt.show()

# ============================================================
# 19. EXTREME-YEAR VARIANCE SENSITIVITY
# ============================================================

# Additional audit of Cell 93.
# Does shared variance remain dominant
# after excluding extreme historical years?

ANNUAL_PATH = (

    LANDSAT

    / '06_HISTORICAL_DISTURBANCE'

    / 'DISTURBANCE_ANNUAL_SERIES.csv'
)

annual_data = pd.read_csv(
    ANNUAL_PATH
)

annual_data = annual_data.loc[

    (
        annual_data['scenario']
        == 'Primary_fragment'
    )

    &

    annual_data['adequate_support']

].copy()

variance_rows = []

for (
    sensor,
    index
), group in annual_data.groupby([
    'sensor',
    'index'
]):

    if len(group) < 8:
        continue

    variance_scenarios = {

        'All_years': group.copy(),

        'Exclude_1991_2023':

            group.loc[

                ~group['year'].isin([
                    1991,
                    2023
                ])
            ].copy(),

        'Exclude_1991_2021_2023':

            group.loc[

                ~group['year'].isin([
                    1991,
                    2021,
                    2023
                ])
            ].copy()
    }

    for scenario, subset in (
        variance_scenarios.items()
    ):

        if len(subset) < 8:
            continue

        secondary = subset[
            'secondary'
        ].to_numpy()

        mature = subset[
            'mature'
        ].to_numpy()

        common = (
            secondary + mature
        ) / 2

        differential = (
            secondary - mature
        ) / 2

        var_common = np.var(
            common,
            ddof=1
        )

        var_diff = np.var(
            differential,
            ddof=1
        )

        total = (
            var_common + var_diff
        )

        variance_rows.append({

            'sensor': sensor,

            'index': index,

            'scenario': scenario,

            'n_years': len(subset),

            'common_variance_fraction':

                var_common / total

                if total > 0

                else np.nan,

            'differential_variance_fraction':

                var_diff / total

                if total > 0

                else np.nan
        })

variance_sensitivity = pd.DataFrame(
    variance_rows
)

print('\nSHARED VARIANCE SENSITIVITY')

display(

    variance_sensitivity.pivot(

        index=[
            'sensor',
            'index'
        ],

        columns='scenario',

        values='common_variance_fraction'

    ).round(4)
)

# ============================================================
# 20. EXPORT ALL TABLES
# ============================================================

exports = {

    'SPATIAL_EVENT_SCENE_SUPPORT':
        event_support,

    'SPATIAL_PRODUCT_SUPPORT':
        product_support,

    'SPATIAL_DISTURBANCE_SUMMARY':
        spatial_summary,

    'SPATIAL_THRESHOLD_SENSITIVITY':
        threshold_sensitivity,

    'SPATIAL_MULTI_INDEX_AGREEMENT':
        spatial_agreement,

    'SHARED_VARIANCE_EXTREME_YEAR_SENSITIVITY':
        variance_sensitivity
}

for name, frame in exports.items():

    frame.to_csv(

        OUT / f'{name}.csv',

        index=False,

        encoding='utf-8-sig'
    )

WORKBOOK = (

    TABLE_DIR

    / 'SATIPO_SPATIAL_DISTURBANCE.xlsx'
)

with pd.ExcelWriter(

    WORKBOOK,

    engine='openpyxl'

) as writer:

    sheets = {

        'Scene_support':
            event_support,

        'Product_support':
            product_support,

        'Spatial_summary':
            spatial_summary,

        'Threshold_sensitivity':
            threshold_sensitivity,

        'Multi_index_agreement':
            spatial_agreement,

        'Variance_sensitivity':
            variance_sensitivity
    }

    for sheet_name, frame in (
        sheets.items()
    ):

        frame.to_excel(

            writer,

            sheet_name=sheet_name,

            index=False
        )

# ============================================================
# 21. MANIFEST
# ============================================================

manifest = {

    'project':
        'SATIPO_FOREST_PROJECT',

    'cell':
        '94',

    'created_utc':

        datetime.now(
            timezone.utc
        ).isoformat(),

    'events':
        EVENTS,

    'scenarios':
        SPATIAL_SCENARIOS,

    'geometry_source':
        geometry_source,

    'screened_input':
        str(SCREENED_PATH),

    'primary_scene_count':
        len(primary_scenes),

    'crs':
        CRS,

    'grid_transform':
        TRANSFORM,

    'minimum_pixel_observations':
        MIN_PIXEL_OBSERVATIONS,

    'thresholds':
        CHANGE_THRESHOLDS,

    'primary_threshold':
        PRIMARY_THRESHOLD,

    'raster_products': [

        str(path)

        for path in raster_products.values()
    ],

    'figures':
        figure_paths,

    'workbook':
        str(WORKBOOK),

    'methodological_limits': [

        'Spatial composites use the same sensor '
        'within each historical event.',

        'Index-specific physical reflectance '
        'screening is applied to NBR and NDMI.',

        'Annual medians receive equal weight '
        'within multi-year reference periods.',

        'Pixel statistics use common valid '
        'pre-event, event and post-event support.',

        'Area estimates are based on 30 m pixel '
        'centers and are not exact polygon areas.',

        'Spectral thresholds are diagnostic '
        'rather than calibrated ecological '
        'disturbance thresholds.',

        'Pixels are not independent ecological '
        'replicates.',

        'Spectral rebound does not establish '
        'ecological recovery.',

        'Historical disturbances remain '
        'unconfirmed without independent evidence.'
    ]
}

MANIFEST_PATH = (

    OUT

    / 'SPATIAL_DISTURBANCE_MANIFEST.json'
)

with open(

    MANIFEST_PATH,

    'w',

    encoding='utf-8'

) as f:

    json.dump(

        manifest,

        f,

        indent=2,

        ensure_ascii=False,

        default=str
    )

# ============================================================
# 22. FINAL REPORT
# ============================================================

print('\n' + '=' * 78)

print(
    'SATIPO — SPATIAL DISTURBANCE REPORT'
)

print('=' * 78)

print(
    'Primary scenes:',
    len(primary_scenes)
)

print(
    'Spatial raster products:',
    len(raster_products)
)

print(
    'Spatial summary rows:',
    len(spatial_summary)
)

print(
    'Threshold sensitivity rows:',
    len(threshold_sensitivity)
)

print(
    'Multi-index agreement rows:',
    len(spatial_agreement)
)

print('\nPRODUCT SUPPORT')

display(

    product_support[
        [
            'event',
            'scenario',
            'pre_years',
            'event_scenes',
            'post_years',
            'status'
        ]
    ]
)

print('\nPRIMARY FRAGMENT RESULTS')

display(

    spatial_summary.loc[

        (
            spatial_summary['scenario']
            == 'Primary'
        )

        &

        spatial_summary['unit_id'].isin([
            'FRAG_01',
            'FRAG_02'
        ]),

        [

            'event',
            'unit_id',
            'index',

            'valid_grid_fraction',

            'event_change_median',

            'rebound_median',

            'residual_median',

            'decline_pixel_pct',

            'rebound_among_declined_pct',

            'persistent_deficit_among_declined_pct'
        ]

    ].round(4)
)

print('\nOUTPUT DIRECTORY:')
print(OUT)

print('\nWORKBOOK:')
print(WORKBOOK)

print('\nMANIFEST:')
print(MANIFEST_PATH)

print('\n' + '=' * 78)

print('CELL 94 COMPLETED')

print(
    'NEXT: CELL 95 — '
    'HYDROCLIMATIC CONTEXT AND '
    'DISTURBANCE ATTRIBUTION'
)

print('=' * 78)

In [ ]:
# ============================================================
# CELL 95
# HYDROCLIMATIC CONTEXT AND DISTURBANCE ATTRIBUTION
# SATIPO FOREST PROJECT
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import ee
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr
from shapely.geometry import mapping

import json
import time

# ============================================================
# 1. CONFIGURATION
# ============================================================

ROOT = Path(
    '/content/drive/MyDrive/SATIPO_FOREST_PROJECT'
)

REMOTE = ROOT / '04_REMOTE_SENSING'

OUT = REMOTE / '09_HYDROCLIMATIC_CONTEXT'

CACHE = OUT / 'ANNUAL_CACHE'

TABLE_DIR = REMOTE / '08_OUTPUTS' / 'TABLES'

FIGURE_DIR = REMOTE / '08_OUTPUTS' / 'FIGURES'

for directory in [
    OUT, CACHE, TABLE_DIR, FIGURE_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

START_YEAR = 1985

END_YEAR = 2025

BASELINE_START = 1985

BASELINE_END = 2019

BUFFER_M = 10000

EVENT_YEARS = [
    1991,
    2002,
    2016,
    2021,
    2023
]

CHIRPS_ID = 'UCSB-CHG/CHIRPS/DAILY'

ERA5_ID = 'ECMWF/ERA5_LAND/MONTHLY_AGGR'

CRS = 'EPSG:32718'

# ============================================================
# 2. EARTH ENGINE CONNECTION
# ============================================================

try:

    ee.Number(1).getInfo()

    print('Earth Engine: ACTIVE')

except Exception:

    ee.Authenticate()

    ee.Initialize()

# ============================================================
# 3. RECOVER VALIDATED GEOMETRIES
# ============================================================

required_units = {
    'FRAG_01',
    'FRAG_02'
}

geometry_candidates = []

for name, obj in list(globals().items()):

    if isinstance(obj, gpd.GeoDataFrame):

        if 'unit_id' not in obj.columns:
            continue

        ids = set(
            obj['unit_id'].astype(str)
        )

        if required_units.issubset(ids):

            geometry_candidates.append(
                (name, obj.copy())
            )

assert geometry_candidates, (
    'Validated geometry not found in memory. '
    'Restore the geometry from Cell 94.'
)

geometry_source, climate_units = (
    geometry_candidates[0]
)

climate_fragments = climate_units.loc[

    climate_units['unit_id'].isin(
        required_units
    )

].copy()

assert len(climate_fragments) == 2

assert climate_fragments.crs is not None

climate_fragments = (
    climate_fragments.to_crs(CRS)
)

climate_geometry_utm = (

    climate_fragments.geometry
    .union_all()
    .buffer(BUFFER_M)
)

climate_geometry_wgs = gpd.GeoSeries(

    [climate_geometry_utm],

    crs=CRS

).to_crs(
    'EPSG:4326'
).iloc[0]

climate_region = ee.Geometry(

    mapping(climate_geometry_wgs),

    proj='EPSG:4326',

    geodesic=False
)

climate_area_km2 = (

    climate_geometry_utm.area
    / 1e6
)

print('\nREGIONAL CLIMATE DOMAIN')

print(
    'Geometry source:',
    geometry_source
)

print(
    'Buffer:',
    BUFFER_M / 1000,
    'km'
)

print(
    'Area:',
    round(climate_area_km2, 2),
    'km²'
)

# Export climate analysis region.

climate_region_gdf = gpd.GeoDataFrame(

    {
        'region_id': ['SATIPO_CLIMATE_10KM'],
        'buffer_m': [BUFFER_M],
        'area_km2': [climate_area_km2]
    },

    geometry=[climate_geometry_wgs],

    crs='EPSG:4326'
)

climate_region_gdf.to_file(

    OUT / 'SATIPO_CLIMATE_REGION.geojson',

    driver='GeoJSON'
)

# ============================================================
# 4. DATASET INITIALIZATION
# ============================================================

chirps = ee.ImageCollection(
    CHIRPS_ID
).select(
    'precipitation'
)

era5 = ee.ImageCollection(
    ERA5_ID
)

ERA5_BANDS = [

    'temperature_2m',

    'dewpoint_temperature_2m',

    'volumetric_soil_water_layer_1'
]

available_bands = (

    ee.Image(era5.first())

    .bandNames()

    .getInfo()
)

assert set(ERA5_BANDS).issubset(

    set(available_bands)

), available_bands

chirps_projection = (

    ee.Image(chirps.first())

    .projection()
)

era5_projection = (

    ee.Image(era5.first())

    .select('temperature_2m')

    .projection()
)

print('\nDATASET VALIDATION')

print(
    'CHIRPS scale:',
    chirps_projection
    .nominalScale()
    .getInfo()
)

print(
    'ERA5-Land scale:',
    era5_projection
    .nominalScale()
    .getInfo()
)

# ============================================================
# 5. MONTHLY CLIMATE EXTRACTION
# ============================================================

def saturation_vapor_pressure(
    temperature_c
):

    # Tetens equation, kPa.

    return temperature_c.expression(

        '0.6108 * exp(17.27 * T / (T + 237.3))',

        {
            'T': temperature_c
        }
    )


def extract_monthly_climate(
    year,
    month
):

    start = ee.Date.fromYMD(

        year,
        month,
        1
    )

    end = start.advance(
        1,
        'month'
    )

    precipitation_collection = (

        chirps.filterDate(
            start,
            end
        )
    )

    n_chirps_days = (

        precipitation_collection.size()
    )

    expected_days = end.difference(

        start,
        'day'
    )

    precipitation = (

        precipitation_collection

        .sum()

        .rename('precip_mm')
    )

    era_month = era5.filterDate(

        start,
        end
    )

    n_era_images = era_month.size()

    era_image = ee.Image(

        era_month.first()
    )

    temperature = (

        era_image

        .select('temperature_2m')

        .subtract(273.15)

        .rename('temperature_c')
    )

    dewpoint = (

        era_image

        .select(
            'dewpoint_temperature_2m'
        )

        .subtract(273.15)

        .rename('dewpoint_c')
    )

    soil_moisture = (

        era_image

        .select(
            'volumetric_soil_water_layer_1'
        )

        .rename('soil_moisture_m3_m3')
    )

    # Monthly-mean-state VPD proxy.
    # This is NOT the mean of hourly VPD.

    es = saturation_vapor_pressure(
        temperature
    )

    ea = saturation_vapor_pressure(
        dewpoint
    )

    vpd = (

        es.subtract(ea)

        .max(0)

        .rename('vpd_proxy_kpa')
    )

    relative_humidity = (

        ea.divide(es)

        .multiply(100)

        .clamp(0, 100)

        .rename('rh_proxy_pct')
    )

    era_climate = (

        temperature

        .addBands(dewpoint)

        .addBands(soil_moisture)

        .addBands(vpd)

        .addBands(relative_humidity)
    )

    precipitation_stats = (

        precipitation.reduceRegion(

            reducer=ee.Reducer.mean(),

            geometry=climate_region,

            crs=chirps_projection,

            maxPixels=1e8,

            tileScale=2
        )
    )

    era_stats = (

        era_climate.reduceRegion(

            reducer=ee.Reducer.mean(),

            geometry=climate_region,

            crs=era5_projection,

            maxPixels=1e8,

            tileScale=2
        )
    )

    properties = ee.Dictionary({

        'year': year,

        'month': month,

        'date':
            start.format('YYYY-MM-dd'),

        'n_chirps_days':
            n_chirps_days,

        'expected_days':
            expected_days,

        'n_era_images':
            n_era_images

    }).combine(

        precipitation_stats,

        overwrite=True

    ).combine(

        era_stats,

        overwrite=True
    )

    return ee.Feature(
        None,
        properties
    )


# ============================================================
# 6. TEST EXTRACTION
# ============================================================

test_climate = extract_monthly_climate(

    2021,
    7

).getInfo()['properties']

print('\nTEST: JULY 2021')

display(

    pd.DataFrame(
        [test_climate]
    )
)

# ============================================================
# 7. EXTRACT 1985–2025
# ============================================================

monthly_frames = []

failed_years = []

for year in range(

    START_YEAR,

    END_YEAR + 1
):

    cache_path = (

        CACHE

        / f'CLIMATE_MONTHLY_{year}_10KM.csv'
    )

    if cache_path.exists():

        cached = pd.read_csv(
            cache_path
        )

        if (

            len(cached) == 12

            and set(cached['month'])
            == set(range(1, 13))

            and (
                cached['year'] == year
            ).all()

        ):

            monthly_frames.append(
                cached
            )

            print(
                year,
                'CACHED'
            )

            continue

    success = False

    for attempt in range(1, 4):

        try:

            features = [

                extract_monthly_climate(

                    year,
                    month
                )

                for month in range(
                    1, 13
                )
            ]

            collection = (

                ee.FeatureCollection(
                    features
                )
            )

            result = collection.getInfo()

            records = [

                feature['properties']

                for feature in result['features']
            ]

            frame = pd.DataFrame(
                records
            )

            assert len(frame) == 12

            frame.to_csv(

                cache_path,

                index=False
            )

            monthly_frames.append(
                frame
            )

            print(
                year,
                'EXTRACTED'
            )

            success = True

            break

        except Exception as error:

            print(

                year,

                'Attempt:',

                attempt,

                str(error)[:200]
            )

            time.sleep(
                5 * attempt
            )

    if not success:

        failed_years.append(
            year
        )

assert not failed_years, (
    'Failed climate years:',
    failed_years
)

climate_monthly = pd.concat(

    monthly_frames,

    ignore_index=True
)

climate_monthly['date'] = pd.to_datetime(

    climate_monthly['date']
)

climate_monthly = (

    climate_monthly

    .sort_values([
        'year',
        'month'
    ])

    .reset_index(drop=True)
)

# ============================================================
# 8. QUALITY CONTROL
# ============================================================

expected_rows = (

    END_YEAR - START_YEAR + 1

) * 12

assert len(climate_monthly) == expected_rows

assert not climate_monthly.duplicated(

    ['year', 'month']

).any()

climate_variables = [

    'precip_mm',

    'temperature_c',

    'dewpoint_c',

    'soil_moisture_m3_m3',

    'vpd_proxy_kpa',

    'rh_proxy_pct'
]

for variable in climate_variables:

    climate_monthly[variable] = (

        pd.to_numeric(

            climate_monthly[variable],

            errors='coerce'
        )
    )

climate_monthly['complete_chirps'] = (

    climate_monthly['n_chirps_days']

    ==

    climate_monthly['expected_days']
)

climate_monthly['complete_era5'] = (

    climate_monthly['n_era_images']
    == 1
)

assert climate_monthly[
    'complete_chirps'
].all()

assert climate_monthly[
    'complete_era5'
].all()

assert climate_monthly[
    climate_variables
].notna().all().all()

assert (

    climate_monthly['precip_mm'] >= 0

).all()

assert (

    climate_monthly[
        'soil_moisture_m3_m3'
    ].between(0, 1)

).all()

assert (

    climate_monthly['vpd_proxy_kpa'] >= 0

).all()

assert (

    climate_monthly['rh_proxy_pct']
    .between(0, 100)

).all()

climate_monthly['days_in_month'] = (

    climate_monthly['date']
    .dt.days_in_month
)

climate_qa = (

    climate_monthly[climate_variables]

    .describe()

    .T

    .reset_index()

    .rename(
        columns={
            'index': 'variable'
        }
    )
)

print('\nMONTHLY CLIMATE QA')

print(
    'Monthly observations:',
    len(climate_monthly)
)

print(
    'Complete CHIRPS months:',
    climate_monthly[
        'complete_chirps'
    ].sum()
)

print(
    'Complete ERA5 months:',
    climate_monthly[
        'complete_era5'
    ].sum()
)

display(
    climate_qa.round(4)
)

# ============================================================
# 9. SEASONAL AGGREGATION
# ============================================================

SEASONS = {

    'Annual':
        list(range(1, 13)),

    'Antecedent_JanApr':
        [1, 2, 3, 4],

    'Landsat_MaySep':
        [5, 6, 7, 8, 9],

    'Core_JunSep':
        [6, 7, 8, 9]
}

season_rows = []

for year, group in (

    climate_monthly.groupby('year')
):

    for season, months in (
        SEASONS.items()
    ):

        subset = group.loc[

            group['month'].isin(
                months
            )
        ].copy()

        assert len(subset) == len(months)

        weights = subset[
            'days_in_month'
        ].to_numpy()

        row = {

            'year': year,

            'season': season,

            'n_months': len(subset),

            'precip_mm':

                subset[
                    'precip_mm'
                ].sum()
        }

        for variable in climate_variables:

            if variable == 'precip_mm':
                continue

            row[variable] = np.average(

                subset[variable],

                weights=weights
            )

        season_rows.append(
            row
        )

climate_seasonal = pd.DataFrame(
    season_rows
)

assert len(climate_seasonal) == (

    END_YEAR - START_YEAR + 1

) * len(SEASONS)

# ============================================================
# 10. BASELINE CLIMATOLOGY
# ============================================================

baseline = climate_seasonal.loc[

    climate_seasonal['year'].between(

        BASELINE_START,

        BASELINE_END
    )
].copy()

baseline_rows = []

for season, group in (

    baseline.groupby('season')
):

    for variable in climate_variables:

        baseline_rows.append({

            'season': season,

            'variable': variable,

            'baseline_start':
                BASELINE_START,

            'baseline_end':
                BASELINE_END,

            'n_years': len(group),

            'baseline_mean':
                group[variable].mean(),

            'baseline_sd':
                group[variable].std(ddof=1),

            'baseline_median':
                group[variable].median(),

            'baseline_q10':
                group[variable].quantile(0.10),

            'baseline_q90':
                group[variable].quantile(0.90)
        })

climate_baseline = pd.DataFrame(
    baseline_rows
)

# ============================================================
# 11. STANDARDIZED CLIMATE ANOMALIES
# ============================================================

anomaly_rows = []

for row in climate_seasonal.itertuples():

    for variable in climate_variables:

        reference = climate_baseline.loc[

            (
                climate_baseline['season']
                == row.season
            )

            &

            (
                climate_baseline['variable']
                == variable
            )

        ].iloc[0]

        value = getattr(
            row,
            variable
        )

        anomaly = (

            value

            - reference['baseline_mean']
        )

        z = (

            anomaly

            / reference['baseline_sd']

            if reference['baseline_sd'] > 0

            else np.nan
        )

        if z <= -2:

            category = 'Extremely_low'

        elif z <= -1:

            category = 'Low'

        elif z >= 2:

            category = 'Extremely_high'

        elif z >= 1:

            category = 'High'

        else:

            category = 'Near_baseline'

        anomaly_rows.append({

            'year': row.year,

            'season': row.season,

            'variable': variable,

            'value': value,

            'baseline_mean':
                reference['baseline_mean'],

            'anomaly': anomaly,

            'z_score': z,

            'category': category,

            'anomaly_pct':

                100 * anomaly
                / reference['baseline_mean']

                if variable == 'precip_mm'

                else np.nan
        })

climate_anomalies = pd.DataFrame(
    anomaly_rows
)

# ============================================================
# 12. EVENT-SPECIFIC CLIMATE DIAGNOSTICS
# ============================================================

event_climate = climate_anomalies.loc[

    climate_anomalies['year'].isin(
        EVENT_YEARS
    )

    &

    climate_anomalies['season'].isin([

        'Annual',

        'Antecedent_JanApr',

        'Landsat_MaySep'
    ])

].copy()

event_matrix = event_climate.pivot(

    index=[
        'year',
        'season'
    ],

    columns='variable',

    values='z_score'
).reset_index()

print('\nEVENT CLIMATE ANOMALIES')

display(
    event_matrix.round(3)
)

# ============================================================
# 13. CLIMATE STRESS DIAGNOSTIC
# ============================================================

# Positive values indicate conditions
# consistent with greater atmospheric
# or soil-water stress.
#
# This is an exploratory composite,
# not a validated drought index.

stress_rows = []

for row in event_matrix.itertuples():

    components = [

        -row.precip_mm,

        -row.soil_moisture_m3_m3,

        row.vpd_proxy_kpa
    ]

    stress_score = np.mean(
        components
    )

    n_stress_signals = int(

        np.sum(
            np.asarray(components) >= 1
        )
    )

    stress_rows.append({

        'year': row.year,

        'season': row.season,

        'precipitation_z':
            row.precip_mm,

        'soil_moisture_z':
            row.soil_moisture_m3_m3,

        'vpd_proxy_z':
            row.vpd_proxy_kpa,

        'temperature_z':
            row.temperature_c,

        'exploratory_stress_score':
            stress_score,

        'n_stress_signals_ge1':
            n_stress_signals
    })

event_stress = pd.DataFrame(
    stress_rows
)

print('\nEVENT STRESS DIAGNOSTICS')

display(

    event_stress.loc[

        event_stress['season']
        == 'Landsat_MaySep'

    ].round(3)
)

# ============================================================
# 14. RECOVER HISTORICAL SPECTRAL SERIES
# ============================================================

ANNUAL_PATH = (

    REMOTE

    / '03_LANDSAT'

    / '06_HISTORICAL_DISTURBANCE'

    / 'DISTURBANCE_ANNUAL_SERIES.csv'
)

assert ANNUAL_PATH.exists(), ANNUAL_PATH

spectral_annual = pd.read_csv(
    ANNUAL_PATH
)

required_columns = {

    'scenario',

    'sensor',

    'year',

    'index',

    'adequate_support',

    'secondary',

    'mature'
}

assert required_columns.issubset(

    spectral_annual.columns

), spectral_annual.columns.tolist()

adequate_mask = (

    spectral_annual['adequate_support']

    .astype(str)

    .str.lower()

    .isin(['true', '1', '1.0'])
)

spectral_primary = spectral_annual.loc[

    (
        spectral_annual['scenario']
        == 'Primary_fragment'
    )

    & adequate_mask

].copy()

spectral_primary['common_component'] = (

    spectral_primary['secondary']

    + spectral_primary['mature']

) / 2

spectral_primary['signed_contrast'] = (

    spectral_primary['secondary']

    - spectral_primary['mature']
)

spectral_primary['absolute_contrast'] = (

    spectral_primary['signed_contrast']
    .abs()
)

assert not spectral_primary.duplicated([

    'sensor',

    'year',

    'index'

]).any()

# ============================================================
# 15. CLIMATE-SPECTRAL DATA INTEGRATION
# ============================================================

climate_wide = climate_anomalies.pivot(

    index='year',

    columns=[
        'season',
        'variable'
    ],

    values='z_score'
)

climate_wide.columns = [

    f'{season}__{variable}_z'

    for season, variable
    in climate_wide.columns
]

climate_wide = (

    climate_wide.reset_index()
)

spectral_climate = spectral_primary.merge(

    climate_wide,

    on='year',

    how='left',

    validate='many_to_one'
)

print('\nSPECTRAL-CLIMATE INTEGRATION')

print(
    'Rows:',
    len(spectral_climate)
)

print(
    'Sensors:',
    spectral_climate[
        'sensor'
    ].nunique()
)

# ============================================================
# 16. WITHIN-SENSOR ASSOCIATIONS
# ============================================================

# Correlations are exploratory.
# Sensors are not pooled.
# No ecological replication is implied.

CLIMATE_PREDICTORS = [

    'Antecedent_JanApr__precip_mm_z',

    'Landsat_MaySep__precip_mm_z',

    'Landsat_MaySep__temperature_c_z',

    'Landsat_MaySep__soil_moisture_m3_m3_z',

    'Landsat_MaySep__vpd_proxy_kpa_z'
]

RESPONSES = [

    'common_component',

    'signed_contrast',

    'absolute_contrast'
]

correlation_rows = []

for (sensor, index), group in (

    spectral_climate.groupby([

        'sensor',

        'index'
    ])
):

    subsets = {

        'All_years':
            group.copy(),

        'Exclude_extreme_years':

            group.loc[

                ~group['year'].isin([

                    1991,

                    2021,

                    2023
                ])
            ].copy()
    }

    for scenario, subset in (

        subsets.items()
    ):

        for predictor in CLIMATE_PREDICTORS:

            for response in RESPONSES:

                valid = subset[

                    [
                        predictor,
                        response
                    ]

                ].dropna()

                n = len(valid)

                if (

                    n < 8

                    or valid[predictor].nunique() < 2

                    or valid[response].nunique() < 2

                ):

                    rho = np.nan

                    status = 'INSUFFICIENT_SUPPORT'

                else:

                    rho = spearmanr(

                        valid[predictor],

                        valid[response]

                    ).statistic

                    status = 'ESTIMATED'

                correlation_rows.append({

                    'sensor': sensor,

                    'index': index,

                    'scenario': scenario,

                    'predictor': predictor,

                    'response': response,

                    'n_years': n,

                    'spearman_rho': rho,

                    'status': status
                })

climate_correlations = pd.DataFrame(
    correlation_rows
)

print('\nCOMMON SPECTRAL COMPONENT ASSOCIATIONS')

display(

    climate_correlations.loc[

        (
            climate_correlations['scenario']
            == 'All_years'
        )

        &

        (
            climate_correlations['response']
            == 'common_component'
        )

        &

        climate_correlations['index'].isin([

            'NBR',

            'NDMI'
        ])

    ].round(3)
)

# ============================================================
# 17. DETRENDED ASSOCIATION SENSITIVITY
# ============================================================

from scipy.stats import theilslopes

detrended_rows = []

for (sensor, index), group in (

    spectral_climate.groupby([

        'sensor',

        'index'
    ])
):

    for predictor in CLIMATE_PREDICTORS:

        for response in RESPONSES:

            valid = group[

                [
                    'year',
                    predictor,
                    response
                ]

            ].dropna()

            if len(valid) < 8:
                continue

            if (

                valid[predictor].nunique() < 2

                or valid[response].nunique() < 2

            ):
                continue

            years = valid[
                'year'
            ].to_numpy(dtype=float)

            years = years - years.mean()

            x = valid[
                predictor
            ].to_numpy()

            y = valid[
                response
            ].to_numpy()

            x_fit = theilslopes(
                x,
                years
            )

            y_fit = theilslopes(
                y,
                years
            )

            x_residual = x - (

                x_fit.intercept

                + x_fit.slope * years
            )

            y_residual = y - (

                y_fit.intercept

                + y_fit.slope * years
            )

            rho_raw = spearmanr(
                x,
                y
            ).statistic

            rho_detrended = spearmanr(

                x_residual,

                y_residual

            ).statistic

            detrended_rows.append({

                'sensor': sensor,

                'index': index,

                'predictor': predictor,

                'response': response,

                'n_years': len(valid),

                'rho_raw': rho_raw,

                'rho_detrended':
                    rho_detrended,

                'rho_difference':

                    rho_detrended - rho_raw
            })

climate_detrended = pd.DataFrame(
    detrended_rows
)

print('\nDETRENDED ASSOCIATIONS')

display(

    climate_detrended.loc[

        (
            climate_detrended['response']
            == 'common_component'
        )

        &

        climate_detrended['index'].isin([

            'NBR',

            'NDMI'
        ])

    ].round(3)
)

# ============================================================
# 18. EVENT-SPECIFIC SPECTRAL-CLIMATE TABLE
# ============================================================

event_spectral_climate = (

    spectral_climate.loc[

        spectral_climate['year'].isin(
            EVENT_YEARS
        )

        &

        spectral_climate['index'].isin([

            'NBR',

            'NDMI'
        ])

    ].copy()
)

event_columns = [

    'sensor',

    'year',

    'index',

    'secondary',

    'mature',

    'common_component',

    'signed_contrast',

    'absolute_contrast'

] + CLIMATE_PREDICTORS

event_spectral_climate = (

    event_spectral_climate[
        event_columns
    ]
)

print('\nEVENT SPECTRAL-CLIMATE TABLE')

display(

    event_spectral_climate.round(3)
)

# ============================================================
# 19. FIGURE 1
# LONG-TERM HYDROCLIMATIC ANOMALIES
# ============================================================

plt.rcParams.update({

    'font.family': 'DejaVu Sans',

    'font.size': 10,

    'pdf.fonttype': 42,

    'ps.fonttype': 42
})

plot_variables = [

    (
        'precip_mm',
        'Precipitation',
        '#527B98'
    ),

    (
        'temperature_c',
        'Air temperature',
        '#B65C45'
    ),

    (
        'soil_moisture_m3_m3',
        'Surface soil moisture',
        '#24634F'
    ),

    (
        'vpd_proxy_kpa',
        'VPD proxy',
        '#DB9140'
    )
]

fig, axes = plt.subplots(

    4,

    1,

    figsize=(14, 11),

    sharex=True
)

for ax, (

    variable,

    title,

    color

) in zip(

    axes,

    plot_variables
):

    subset = climate_anomalies.loc[

        (
            climate_anomalies['season']
            == 'Landsat_MaySep'
        )

        &

        (
            climate_anomalies['variable']
            == variable
        )

    ].sort_values('year')

    ax.axhspan(

        -1,

        1,

        color='#E8E8E8',

        alpha=0.6
    )

    ax.axhline(

        0,

        color='gray',

        linewidth=0.8
    )

    ax.plot(

        subset['year'],

        subset['z_score'],

        color=color,

        linewidth=1.8,

        marker='o',

        markersize=3
    )

    for event_year in EVENT_YEARS:

        event_point = subset.loc[

            subset['year']
            == event_year
        ]

        ax.axvline(

            event_year,

            color='#B0B0B0',

            linestyle=':',

            linewidth=0.8
        )

        ax.scatter(

            event_point['year'],

            event_point['z_score'],

            s=55,

            facecolor=color,

            edgecolor='black',

            zorder=5
        )

    ax.set_title(

        title,

        loc='left',

        fontweight='bold'
    )

    ax.set_ylabel(
        'Anomaly (z)'
    )

    ax.spines[
        ['top', 'right']
    ].set_visible(False)

axes[-1].set_xlabel('Year')

axes[-1].set_xlim(
    1984.5,
    2025.5
)

fig.suptitle(

    'Regional hydroclimatic context '
    'of historical spectral anomalies',

    fontsize=17,

    fontweight='bold'
)

fig.text(

    0.5,

    0.015,

    'May–September | Reference period: '
    '1985–2019 | Regional 10 km buffer | '
    'VPD estimated from monthly mean temperature '
    'and dewpoint',

    ha='center',

    fontsize=9
)

fig.tight_layout(

    rect=[0, 0.04, 1, 0.95]
)

climate_figure = (

    FIGURE_DIR

    / 'FIG_HYDROCLIMATIC_ANOMALIES'
)

for extension in [
    'png',
    'pdf',
    'svg'
]:

    fig.savefig(

        f'{climate_figure}.{extension}',

        dpi=600,

        bbox_inches='tight'
    )

plt.show()

# ============================================================
# 20. FIGURE 2
# EVENT CLIMATE ANOMALY MATRIX
# ============================================================

heat_variables = [

    'precip_mm',

    'temperature_c',

    'soil_moisture_m3_m3',

    'vpd_proxy_kpa'
]

heat_labels = [

    'Precipitation',

    'Temperature',

    'Soil moisture',

    'VPD proxy'
]

heat_seasons = [

    'Antecedent_JanApr',

    'Landsat_MaySep'
]

fig, axes = plt.subplots(

    1,

    2,

    figsize=(13, 5.5)
)

for ax, season in zip(

    axes,

    heat_seasons
):

    subset = event_matrix.loc[

        event_matrix['season']
        == season

    ].set_index(

        'year'

    ).reindex(
        EVENT_YEARS
    )

    matrix = subset[
        heat_variables
    ].to_numpy()

    image = ax.imshow(

        matrix,

        cmap='RdBu_r',

        vmin=-3,

        vmax=3,

        aspect='auto'
    )

    ax.set_xticks(

        np.arange(
            len(heat_labels)
        ),

        heat_labels,

        rotation=25,

        ha='right'
    )

    ax.set_yticks(

        np.arange(
            len(EVENT_YEARS)
        ),

        EVENT_YEARS
    )

    for i in range(
        matrix.shape[0]
    ):

        for j in range(
            matrix.shape[1]
        ):

            value = matrix[i, j]

            if np.isfinite(value):

                ax.text(

                    j,

                    i,

                    f'{value:.2f}',

                    ha='center',

                    va='center',

                    color=(

                        'white'

                        if abs(value) > 1.8

                        else 'black'
                    ),

                    fontsize=10
                )

    ax.set_title(

        season.replace('_', ' '),

        fontweight='bold'
    )

fig.suptitle(

    'Hydroclimatic conditions during '
    'candidate spectral disturbance years',

    fontsize=16,

    fontweight='bold'
)

fig.subplots_adjust(

    left=0.07,

    right=0.88,

    bottom=0.22,

    top=0.85,

    wspace=0.30
)

cax = fig.add_axes([

    0.91,

    0.25,

    0.018,

    0.55
])

fig.colorbar(

    image,

    cax=cax,

    label='Standardized anomaly'
)

event_figure = (

    FIGURE_DIR

    / 'FIG_EVENT_HYDROCLIMATIC_MATRIX'
)

for extension in [
    'png',
    'pdf',
    'svg'
]:

    fig.savefig(

        f'{event_figure}.{extension}',

        dpi=600,

        bbox_inches='tight'
    )

plt.show()

# ============================================================
# 21. FIGURE 3
# CLIMATE-SPECTRAL ASSOCIATION MATRIX
# ============================================================

predictor_labels = {

    'Antecedent_JanApr__precip_mm_z':
        'Antecedent precipitation',

    'Landsat_MaySep__precip_mm_z':
        'Seasonal precipitation',

    'Landsat_MaySep__temperature_c_z':
        'Temperature',

    'Landsat_MaySep__soil_moisture_m3_m3_z':
        'Soil moisture',

    'Landsat_MaySep__vpd_proxy_kpa_z':
        'VPD proxy'
}

plot_correlations = climate_correlations.loc[

    (
        climate_correlations['scenario']
        == 'All_years'
    )

    &

    (
        climate_correlations['response']
        == 'common_component'
    )

    &

    climate_correlations['sensor'].isin([

        'Landsat_5',

        'Landsat_7',

        'Landsat_8'
    ])

].copy()

fig, axes = plt.subplots(

    1,

    3,

    figsize=(17, 6)
)

index_order = [

    'NDVI',

    'EVI',

    'NDMI',

    'NBR',

    'NBR2'
]

for ax, sensor in zip(

    axes,

    [
        'Landsat_5',

        'Landsat_7',

        'Landsat_8'
    ]
):

    subset = plot_correlations.loc[

        plot_correlations['sensor']
        == sensor
    ]

    matrix_df = subset.pivot(

        index='index',

        columns='predictor',

        values='spearman_rho'

    ).reindex(

        index=index_order,

        columns=CLIMATE_PREDICTORS
    )

    matrix = matrix_df.to_numpy()

    image = ax.imshow(

        matrix,

        cmap='RdBu_r',

        vmin=-1,

        vmax=1,

        aspect='auto'
    )

    ax.set_xticks(

        np.arange(
            len(CLIMATE_PREDICTORS)
        ),

        [

            predictor_labels[p]

            for p in CLIMATE_PREDICTORS
        ],

        rotation=55,

        ha='right'
    )

    ax.set_yticks(

        np.arange(
            len(index_order)
        ),

        index_order
    )

    for i in range(
        matrix.shape[0]
    ):

        for j in range(
            matrix.shape[1]
        ):

            value = matrix[i, j]

            if np.isfinite(value):

                ax.text(

                    j,

                    i,

                    f'{value:.2f}',

                    ha='center',

                    va='center',

                    color=(

                        'white'

                        if abs(value) > 0.65

                        else 'black'
                    )
                )

    ax.set_title(

        sensor.replace('_', ' '),

        fontweight='bold'
    )

fig.suptitle(

    'Within-sensor associations between '
    'regional climate and shared spectral variability',

    fontsize=16,

    fontweight='bold'
)

fig.subplots_adjust(

    left=0.06,

    right=0.91,

    bottom=0.32,

    top=0.84,

    wspace=0.20
)

cax = fig.add_axes([

    0.94,

    0.35,

    0.015,

    0.45
])

fig.colorbar(

    image,

    cax=cax,

    label='Spearman correlation'
)

fig.text(

    0.5,

    0.025,

    'Exploratory associations based on '
    'within-sensor annual observations; '
    'correlation does not establish attribution.',

    ha='center',

    fontsize=9
)

association_figure = (

    FIGURE_DIR

    / 'FIG_CLIMATE_SPECTRAL_ASSOCIATIONS'
)

for extension in [
    'png',
    'pdf',
    'svg'
]:

    fig.savefig(

        f'{association_figure}.{extension}',

        dpi=600,

        bbox_inches='tight'
    )

plt.show()

# ============================================================
# 22. EXPORT TABLES
# ============================================================

exports = {

    'CLIMATE_MONTHLY_MASTER':
        climate_monthly,

    'CLIMATE_QUALITY_CONTROL':
        climate_qa,

    'CLIMATE_SEASONAL_MASTER':
        climate_seasonal,

    'CLIMATE_BASELINE':
        climate_baseline,

    'CLIMATE_STANDARDIZED_ANOMALIES':
        climate_anomalies,

    'CLIMATE_EVENT_DIAGNOSTICS':
        event_climate,

    'CLIMATE_EVENT_MATRIX':
        event_matrix,

    'CLIMATE_EVENT_STRESS':
        event_stress,

    'CLIMATE_SPECTRAL_MASTER':
        spectral_climate,

    'CLIMATE_SPECTRAL_CORRELATIONS':
        climate_correlations,

    'CLIMATE_SPECTRAL_DETRENDED':
        climate_detrended,

    'CLIMATE_EVENT_SPECTRAL_INTEGRATION':
        event_spectral_climate
}

for name, frame in exports.items():

    frame.to_csv(

        OUT / f'{name}.csv',

        index=False,

        encoding='utf-8-sig'
    )

WORKBOOK = (

    TABLE_DIR

    / 'SATIPO_HYDROCLIMATIC_CONTEXT.xlsx'
)

with pd.ExcelWriter(

    WORKBOOK,

    engine='openpyxl'

) as writer:

    sheets = {

        'Monthly_climate':
            climate_monthly,

        'Climate_QA':
            climate_qa,

        'Seasonal_climate':
            climate_seasonal,

        'Baseline':
            climate_baseline,

        'Anomalies':
            climate_anomalies,

        'Event_matrix':
            event_matrix,

        'Event_stress':
            event_stress,

        'Spectral_climate':
            spectral_climate,

        'Correlations':
            climate_correlations,

        'Detrended':
            climate_detrended,

        'Event_integration':
            event_spectral_climate
    }

    for sheet_name, frame in sheets.items():

        frame.to_excel(

            writer,

            sheet_name=sheet_name,

            index=False
        )

# ============================================================
# 23. MANIFEST
# ============================================================

manifest = {

    'project':
        'SATIPO_FOREST_PROJECT',

    'cell':
        '95',

    'created_utc':

        datetime.now(
            timezone.utc
        ).isoformat(),

    'datasets': {

        'precipitation':
            CHIRPS_ID,

        'other_climate_variables':
            ERA5_ID
    },

    'period': [

        START_YEAR,

        END_YEAR
    ],

    'baseline': [

        BASELINE_START,

        BASELINE_END
    ],

    'regional_buffer_m':
        BUFFER_M,

    'regional_area_km2':
        climate_area_km2,

    'geometry_source':
        geometry_source,

    'seasons':
        SEASONS,

    'event_years':
        EVENT_YEARS,

    'monthly_records':
        len(climate_monthly),

    'seasonal_records':
        len(climate_seasonal),

    'correlation_records':
        len(climate_correlations),

    'workbook':
        str(WORKBOOK),

    'figures': [

        str(climate_figure),

        str(event_figure),

        str(association_figure)
    ],

    'limitations': [

        'Climate products characterize '
        'regional conditions, not '
        'individual forest microclimates.',

        'ERA5-Land VPD is estimated from '
        'monthly mean temperature and '
        'dewpoint; it is not mean hourly VPD.',

        'Soil moisture represents the '
        'ERA5-Land upper soil layer, '
        'not directly measured root-zone moisture.',

        'The exploratory stress score '
        'is not a calibrated drought index.',

        'Climate-spectral correlations '
        'are descriptive and sensor-specific.',

        'Detrending reduces sensitivity '
        'to shared monotonic trends but '
        'does not establish causality.',

        'Temporal autocorrelation and '
        'multiple comparisons limit '
        'inferential interpretation.',

        'The 1991 event remains '
        'radiometrically unresolved.',

        'Climate agreement cannot by itself '
        'confirm disturbance attribution.'
    ]
}

MANIFEST_PATH = (

    OUT

    / 'HYDROCLIMATIC_MANIFEST.json'
)

with open(

    MANIFEST_PATH,

    'w',

    encoding='utf-8'

) as f:

    json.dump(

        manifest,

        f,

        indent=2,

        ensure_ascii=False,

        default=str
    )

# ============================================================
# 24. FINAL REPORT
# ============================================================

print('\n' + '=' * 78)

print(
    'SATIPO — HYDROCLIMATIC CONTEXT REPORT'
)

print('=' * 78)

print(
    'Monthly climate observations:',
    len(climate_monthly)
)

print(
    'Seasonal climate observations:',
    len(climate_seasonal)
)

print(
    'Complete CHIRPS months:',
    int(
        climate_monthly[
            'complete_chirps'
        ].sum()
    )
)

print(
    'Complete ERA5 months:',
    int(
        climate_monthly[
            'complete_era5'
        ].sum()
    )
)

print(
    'Climate domain km²:',
    round(
        climate_area_km2,
        2
    )
)

print(
    'Spectral-climate observations:',
    len(spectral_climate)
)

print(
    'Estimated correlations:',
    (
        climate_correlations['status']
        == 'ESTIMATED'
    ).sum()
)

print('\nEVENT CLIMATE CONDITIONS')

display(

    event_matrix.loc[

        event_matrix['season']
        == 'Landsat_MaySep'

    ].round(3)
)

print('\nEVENT STRESS DIAGNOSTICS')

display(

    event_stress.loc[

        event_stress['season']
        == 'Landsat_MaySep'

    ].round(3)
)

print('\nNBR AND NDMI — COMMON COMPONENT')

display(

    climate_detrended.loc[

        (
            climate_detrended['response']
            == 'common_component'
        )

        &

        climate_detrended['index'].isin([

            'NBR',

            'NDMI'
        ])

    ].round(3)
)

print('\nOUTPUT DIRECTORY:')

print(OUT)

print('\nWORKBOOK:')

print(WORKBOOK)

print('\nMANIFEST:')

print(MANIFEST_PATH)

print('\n' + '=' * 78)

print('CELL 95 COMPLETED')

print(

    'NEXT: CELL 96 — '
    'SENTINEL-2 CANOPY SPECTRAL STRUCTURE'
)

print('=' * 78)

In [ ]:
# ============================================================
# CELL 96
# SENTINEL-2 CANOPY SPECTRAL STRUCTURE
# SATIPO FOREST PROJECT
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import ee
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

from shapely.geometry import mapping

import json
import hashlib
import time

# ============================================================
# 1. CONFIGURATION
# ============================================================

ROOT = Path(
    '/content/drive/MyDrive/SATIPO_FOREST_PROJECT'
)

REMOTE = ROOT / '04_REMOTE_SENSING'

S2_DIR = REMOTE / '04_SENTINEL2'

OUT = S2_DIR / '01_CANOPY_SPECTRAL_STRUCTURE'

CACHE = OUT / 'ANNUAL_CACHE'

TABLE_DIR = REMOTE / '08_OUTPUTS' / 'TABLES'

FIGURE_DIR = REMOTE / '08_OUTPUTS' / 'FIGURES'

for directory in [
    OUT, CACHE, TABLE_DIR, FIGURE_DIR
]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )

START_YEAR = 2019

END_YEAR = 2025

FIELD_YEAR = 2019

CRS = 'EPSG:32718'

S2_ID = 'COPERNICUS/S2_SR_HARMONIZED'

CLOUD_ID = (
    'GOOGLE/CLOUD_SCORE_PLUS/V1/'
    'S2_HARMONIZED'
)

CLOUD_THRESHOLD = 0.60

MIN_OBSERVATIONS = 3

MIN_VALID_FRACTION = 0.70

EPS = 1e-6

UNIT_IDS = [

    'FRAG_01',

    'FRAG_02',

    'P01',

    'P02',

    'FRAG_01_CORE30',

    'FRAG_02_CORE30'
]

INDEX_SCALE = {

    'NDVI': 10,

    'EVI': 10,

    'GNDVI': 10,

    'NDRE': 20,

    'NDMI': 20,

    'NBR': 20,

    'NBR2': 20,

    'MTCI': 20
}

INDEX_NAMES = list(
    INDEX_SCALE
)

SCENARIOS = {

    'Primary': {

        'cloud_threshold': 0.60,

        'min_observations': 3
    },

    'Strict_cloud': {

        'cloud_threshold': 0.70,

        'min_observations': 3
    },

    'Strict_temporal': {

        'cloud_threshold': 0.60,

        'min_observations': 5
    }
}

# ============================================================
# 2. EARTH ENGINE CONNECTION
# ============================================================

try:

    ee.Number(1).getInfo()

    print('Earth Engine: ACTIVE')

except Exception:

    ee.Authenticate()

    ee.Initialize()

# ============================================================
# 3. RECOVER VALIDATED GEOMETRIES
# ============================================================

geometry_candidates = []

for name, obj in list(
    globals().items()
):

    if not isinstance(
        obj,
        gpd.GeoDataFrame
    ):

        continue

    if 'unit_id' not in obj.columns:

        continue

    if set(UNIT_IDS).issubset(

        set(
            obj['unit_id'].astype(str)
        )
    ):

        geometry_candidates.append(
            (name, obj.copy())
        )

assert geometry_candidates, (
    'Validated six-unit geometry not found. '
    'Restore geometry from Cell 94.'
)

geometry_source, s2_units = (
    geometry_candidates[0]
)

s2_units = s2_units.loc[

    s2_units['unit_id'].isin(
        UNIT_IDS
    )

].copy()

assert len(s2_units) == 6

assert s2_units['unit_id'].is_unique

assert s2_units.crs is not None

assert s2_units.geometry.is_valid.all()

assert not s2_units.geometry.is_empty.any()

s2_units = s2_units.to_crs(
    CRS
)

s2_units['area_ha'] = (

    s2_units.geometry.area

    / 10000
)

forest_lookup = {

    'FRAG_01':
        'Late_secondary',

    'FRAG_02':
        'Mature',

    'P01':
        'Late_secondary',

    'P02':
        'Mature',

    'FRAG_01_CORE30':
        'Late_secondary',

    'FRAG_02_CORE30':
        'Mature'
}

scale_lookup = {

    'FRAG_01':
        'Fragment',

    'FRAG_02':
        'Fragment',

    'P01':
        'Plot',

    'P02':
        'Plot',

    'FRAG_01_CORE30':
        'Fragment_core',

    'FRAG_02_CORE30':
        'Fragment_core'
}

s2_units['forest_type'] = (

    s2_units['unit_id']

    .map(forest_lookup)
)

s2_units['spatial_scale'] = (

    s2_units['unit_id']

    .map(scale_lookup)
)

print('\nVALIDATED SENTINEL-2 GEOMETRIES')

display(

    s2_units[

        [

            'unit_id',

            'forest_type',

            'spatial_scale',

            'area_ha'
        ]

    ].round(4)
)

s2_units_wgs = s2_units.to_crs(
    'EPSG:4326'
)

features = []

for row in s2_units_wgs.itertuples():

    geometry = ee.Geometry(

        mapping(row.geometry),

        proj='EPSG:4326',

        geodesic=False
    )

    properties = {

        'unit_id':
            row.unit_id,

        'forest_type':
            row.forest_type,

        'spatial_scale':
            row.spatial_scale,

        'polygon_area_ha':
            float(row.area_ha)
    }

    features.append(

        ee.Feature(
            geometry,
            properties
        )
    )

s2_fc = ee.FeatureCollection(
    features
)

s2_region = s2_fc.geometry().bounds()

# ============================================================
# 4. COMMON ANALYSIS GRIDS
# ============================================================

def grid_transform(scale):

    return [

        scale,

        0,

        0,

        0,

        -scale,

        10000000
    ]


grid_reference_rows = []

for resolution in [10, 20]:

    grid_area = (

        ee.Image.pixelArea()

        .rename('grid_area_m2')

        .reduceRegions(

            collection=s2_fc,

            reducer=ee.Reducer.sum(),

            crs=CRS,

            crsTransform=grid_transform(
                resolution
            ),

            tileScale=2
        )
    )

    grid_info = grid_area.getInfo()

    for feature in grid_info['features']:

        props = feature['properties']

        grid_reference_rows.append({

            'unit_id':
                props['unit_id'],

            'resolution_m':
                resolution,

            'grid_area_m2':
                props['sum']
        })

s2_grid_reference = pd.DataFrame(
    grid_reference_rows
)

assert len(s2_grid_reference) == 12

assert (
    s2_grid_reference['grid_area_m2'] > 0
).all()

print('\nGRID AREA REFERENCE')

display(
    s2_grid_reference.round(3)
)

# ============================================================
# 5. DATASET INITIALIZATION
# ============================================================

s2_collection = (

    ee.ImageCollection(S2_ID)

    .filterBounds(s2_region)

    .filterDate(

        f'{START_YEAR}-01-01',

        f'{END_YEAR + 1}-01-01'
    )
)

cloud_collection = (

    ee.ImageCollection(CLOUD_ID)

    .filterBounds(s2_region)

    .filterDate(

        f'{START_YEAR}-01-01',

        f'{END_YEAR + 1}-01-01'
    )
)

# Cloud Score+ linked by system:index.

s2_linked = s2_collection.linkCollection(

    cloud_collection,

    ['cs_cdf']
)

# ============================================================
# 6. PIXEL-LEVEL PREPROCESSING
# ============================================================

def physical_mask(image):

    return (

        image.gte(0)

        .And(image.lte(1))

        .reduce(
            ee.Reducer.min()
        )
    )


def normalized_index(
    a,
    b,
    name
):

    denominator = a.add(b)

    result = (

        a.subtract(b)

        .divide(denominator)

        .updateMask(
            denominator.gt(EPS)
        )

        .rename(name)
    )

    return result.updateMask(

        result.gte(-1)

        .And(
            result.lte(1)
        )
    )


def prepare_s2(
    image,
    cloud_threshold
):

    scl = image.select(
        'SCL'
    )

    # Exclude:
    # 0 no data
    # 1 saturated/defective
    # 3 cloud shadow
    # 6 water
    # 7 unclassified
    # 8/9 cloud
    # 10 cirrus
    # 11 snow/ice
    #
    # Keep vegetation and non-vegetated
    # land classes; do not mask low
    # vegetation index values.

    scl_mask = (

        scl.eq(4)

        .Or(
            scl.eq(5)
        )
    )

    clear_mask = (

        image.select('cs_cdf')

        .gte(cloud_threshold)
    )

    mask = scl_mask.And(
        clear_mask
    )

    bands10 = (

        image.select([

            'B2',

            'B3',

            'B4',

            'B8'
        ])

        .multiply(0.0001)

        .rename([

            'Blue',

            'Green',

            'Red',

            'NIR'
        ])

        .updateMask(mask)
    )

    bands20 = (

        image.select([

            'B5',

            'B6',

            'B7',

            'B8A',

            'B11',

            'B12'
        ])

        .multiply(0.0001)

        .rename([

            'RE1',

            'RE2',

            'RE3',

            'NIRn',

            'SWIR1',

            'SWIR2'
        ])

        .updateMask(mask)
    )

    blue = bands10.select('Blue')

    green = bands10.select('Green')

    red = bands10.select('Red')

    nir = bands10.select('NIR')

    re1 = bands20.select('RE1')

    re2 = bands20.select('RE2')

    nirn = bands20.select('NIRn')

    swir1 = bands20.select('SWIR1')

    swir2 = bands20.select('SWIR2')

    # NDVI — 10 m

    ndvi = normalized_index(

        nir,

        red,

        'NDVI'

    ).updateMask(

        physical_mask(

            nir.addBands(red)
        )
    )

    # GNDVI — 10 m

    gndvi = normalized_index(

        nir,

        green,

        'GNDVI'

    ).updateMask(

        physical_mask(

            nir.addBands(green)
        )
    )

    # EVI — 10 m

    evi_denominator = (

        nir

        .add(
            red.multiply(6)
        )

        .subtract(
            blue.multiply(7.5)
        )

        .add(1)
    )

    evi = (

        nir.subtract(red)

        .multiply(2.5)

        .divide(evi_denominator)

        .updateMask(

            evi_denominator.gt(EPS)
        )

        .updateMask(

            physical_mask(

                nir

                .addBands(red)

                .addBands(blue)
            )
        )

        .rename('EVI')
    )

    # NDRE — 20 m
    # Narrow NIR B8A and red-edge B5.

    ndre = normalized_index(

        nirn,

        re1,

        'NDRE'

    ).updateMask(

        physical_mask(

            nirn.addBands(re1)
        )
    )

    # NDMI — 20 m

    ndmi = normalized_index(

        nirn,

        swir1,

        'NDMI'

    ).updateMask(

        physical_mask(

            nirn.addBands(swir1)
        )
    )

    # NBR — 20 m

    nbr = normalized_index(

        nirn,

        swir2,

        'NBR'

    ).updateMask(

        physical_mask(

            nirn.addBands(swir2)
        )
    )

    # NBR2 — 20 m

    nbr2 = normalized_index(

        swir1,

        swir2,

        'NBR2'

    ).updateMask(

        physical_mask(

            swir1.addBands(swir2)
        )
    )

    # MTCI — 20 m.
    #
    # B4 is aggregated from 10 m to
    # the native red-edge grid.
    #
    # Positive denominator > 0.005
    # avoids unstable near-zero ratios.

    red20 = (

        red.reduceResolution(

            reducer=ee.Reducer.mean(),

            maxPixels=16
        )

        .reproject(
            re1.projection()
        )
    )

    mtci_denominator = re1.subtract(
        red20
    )

    mtci = (

        re2.subtract(re1)

        .divide(mtci_denominator)

        .updateMask(

            mtci_denominator.gt(0.005)
        )

        .updateMask(

            physical_mask(

                re2

                .addBands(re1)

                .addBands(red20)
            )
        )

        .rename('MTCI')
    )

    result = (

        ndvi

        .addBands(evi)

        .addBands(gndvi)

        .addBands(ndre)

        .addBands(ndmi)

        .addBands(nbr)

        .addBands(nbr2)

        .addBands(mtci)
    )

    return result.copyProperties(

        image,

        [

            'system:time_start',

            'system:index',

            'MGRS_TILE'
        ]
    )

# ============================================================
# 7. ANNUAL SCENE SUPPORT
# ============================================================

def get_year_collection(year):

    return (

        s2_linked

        .filterDate(

            f'{year}-05-01',

            f'{year}-10-01'
        )

        .filter(

            ee.Filter.lte(

                'CLOUDY_PIXEL_PERCENTAGE',

                90
            )
        )
    )


scene_support_rows = []

for year in range(

    START_YEAR,

    END_YEAR + 1
):

    year_collection = get_year_collection(
        year
    )

    n_granules = year_collection.size().getInfo()

    dates = (

        year_collection

        .aggregate_array(
            'system:time_start'
        )

        .getInfo()
    )

    unique_dates = sorted(

        set(

            pd.to_datetime(

                dates,

                unit='ms',

                utc=True

            ).strftime(
                '%Y-%m-%d'
            )
        )
    )

    scene_support_rows.append({

        'year':
            year,

        'n_candidate_granules':
            n_granules,

        'n_candidate_dates':
            len(unique_dates),

        'first_date':

            unique_dates[0]

            if unique_dates

            else None,

        'last_date':

            unique_dates[-1]

            if unique_dates

            else None
    })

    print(

        year,

        'Granules:',

        n_granules,

        '| Dates:',

        len(unique_dates)
    )

s2_scene_support = pd.DataFrame(
    scene_support_rows
)

assert (

    s2_scene_support[
        'n_candidate_dates'
    ] >= 5

).all()

print('\nSENTINEL-2 SCENE SUPPORT')

display(
    s2_scene_support
)

# ============================================================
# 8. DAILY MOSAICS
# ============================================================

# Prevent multiple granules from
# the same date from being counted
# as independent temporal observations.

def build_daily_collection(

    year,

    cloud_threshold
):

    collection = get_year_collection(
        year
    )

    prepared = collection.map(

        lambda image:

            prepare_s2(

                image,

                cloud_threshold
            )
    )

    dates = ee.List(

        prepared.aggregate_array(
            'system:time_start'
        )

    ).map(

        lambda timestamp:

            ee.Date(timestamp).format(
                'YYYY-MM-dd'
            )

    ).distinct().sort()

    def daily_mosaic(date_string):

        start = ee.Date(
            date_string
        )

        end = start.advance(
            1,
            'day'
        )

        daily = prepared.filterDate(

            start,

            end
        )

        return (

            daily.median()

            .set({

                'system:time_start':
                    start.millis(),

                'date':
                    date_string,

                'n_granules':
                    daily.size()
            })
        )

    return ee.ImageCollection.fromImages(

        dates.map(
            daily_mosaic
        )
    )

# ============================================================
# 9. ANNUAL CANOPY COMPOSITES
# ============================================================

def build_annual_composite(

    year,

    scenario
):

    config = SCENARIOS[
        scenario
    ]

    daily = build_daily_collection(

        year,

        config['cloud_threshold']
    )

    composites = {}

    observation_counts = {}

    for index, resolution in (

        INDEX_SCALE.items()
    ):

        transform = grid_transform(
            resolution
        )

        # Explicit common grid for
        # all daily observations.

        index_collection = daily.map(

            lambda image:

                image.select(index)

                .reproject(

                    crs=CRS,

                    crsTransform=transform
                )
        )

        count = (

            index_collection

            .count()

            .rename(
                f'{index}_nobs'
            )
        )

        composite = (

            index_collection

            .median()

            .updateMask(

                count.gte(

                    config[
                        'min_observations'
                    ]
                )
            )

            .rename(index)
        )

        composites[index] = composite

        observation_counts[index] = count

    return composites, observation_counts

# ============================================================
# 10. SPATIAL SUMMARY REDUCERS
# ============================================================

spatial_reducer = (

    ee.Reducer.mean()

    .combine(

        reducer2=ee.Reducer.median(),

        sharedInputs=True
    )

    .combine(

        reducer2=ee.Reducer.stdDev(),

        sharedInputs=True
    )

    .combine(

        reducer2=ee.Reducer.percentile(

            [10, 25, 75, 90]
        ),

        sharedInputs=True
    )

    .combine(

        reducer2=ee.Reducer.count(),

        sharedInputs=True
    )
)


def summarize_annual_index(

    year,

    scenario,

    index,

    composite,

    count
):

    resolution = INDEX_SCALE[
        index
    ]

    transform = grid_transform(
        resolution
    )

    stats = composite.reduceRegions(

        collection=s2_fc,

        reducer=spatial_reducer,

        crs=CRS,

        crsTransform=transform,

        tileScale=4
    )

    valid_area = (

        ee.Image.pixelArea()

        .updateMask(
            composite.mask()
        )

        .unmask(0, False)

        .rename('valid_area_m2')
    )

    area_stats = valid_area.reduceRegions(

        collection=s2_fc,

        reducer=ee.Reducer.sum(),

        crs=CRS,

        crsTransform=transform,

        tileScale=4
    )

    count_stats = (

        count

        .updateMask(
            composite.mask()
        )

        .reduceRegions(

            collection=s2_fc,

            reducer=ee.Reducer.mean(),

            crs=CRS,

            crsTransform=transform,

            tileScale=4
        )
    )

    stats_info = stats.getInfo()

    area_info = area_stats.getInfo()

    count_info = count_stats.getInfo()

    area_lookup = {

        feature['properties']['unit_id']:

            feature['properties'].get(
                'sum'
            )

        for feature in area_info['features']
    }

    count_lookup = {

        feature['properties']['unit_id']:

            feature['properties'].get(
                'mean'
            )

        for feature in count_info['features']
    }

    rows = []

    for feature in stats_info['features']:

        props = feature['properties']

        unit_id = props[
            'unit_id'
        ]

        rows.append({

            'year':
                year,

            'scenario':
                scenario,

            'unit_id':
                unit_id,

            'forest_type':
                props['forest_type'],

            'spatial_scale':
                props['spatial_scale'],

            'polygon_area_ha':
                props['polygon_area_ha'],

            'index':
                index,

            'resolution_m':
                resolution,

            'mean':
                props.get('mean'),

            'median':
                props.get('median'),

            'sd':
                props.get('stdDev'),

            'p10':
                props.get('p10'),

            'p25':
                props.get('p25'),

            'p75':
                props.get('p75'),

            'p90':
                props.get('p90'),

            'n_valid_pixels':
                props.get('count'),

            'valid_area_m2':
                area_lookup[unit_id],

            'mean_valid_dates':
                count_lookup[unit_id]
        })

    return pd.DataFrame(
        rows
    )

# ============================================================
# 11. CACHE CONFIGURATION
# ============================================================

s2_config = {

    'version':
        'CELL96_V01',

    'dataset':
        S2_ID,

    'cloud_dataset':
        CLOUD_ID,

    'years': [

        START_YEAR,

        END_YEAR
    ],

    'months':
        [5, 6, 7, 8, 9],

    'scenarios':
        SCENARIOS,

    'index_resolution':
        INDEX_SCALE,

    'crs':
        CRS,

    'geometry':

        {

            row.unit_id:
                row.geometry.wkt

            for row in s2_units.itertuples()
        }
}

config_hash = hashlib.sha256(

    json.dumps(

        s2_config,

        sort_keys=True

    ).encode('utf-8')

).hexdigest()[:12]

print(
    '\nConfiguration hash:',
    config_hash
)

# ============================================================
# 12. TEST EXTRACTION
# ============================================================

test_composites, test_counts = (

    build_annual_composite(

        FIELD_YEAR,

        'Primary'
    )
)

s2_test = summarize_annual_index(

    FIELD_YEAR,

    'Primary',

    'NDVI',

    test_composites['NDVI'],

    test_counts['NDVI']
)

assert len(s2_test) == 6

print('\nTEST: 2019 NDVI')

display(
    s2_test.round(4)
)

# ============================================================
# 13. FULL SENTINEL-2 EXTRACTION
# ============================================================

s2_frames = []

s2_failures = []

for year in range(

    START_YEAR,

    END_YEAR + 1
):

    for scenario in SCENARIOS:

        print(

            '\nPROCESSING:',

            year,

            scenario
        )

        composites = None

        counts = None

        for index in INDEX_NAMES:

            cache_path = (

                CACHE

                / (

                    f'S2_{year}_{scenario}_'

                    f'{index}_{config_hash}.csv'
                )
            )

            if cache_path.exists():

                cached = pd.read_csv(
                    cache_path
                )

                if (

                    len(cached) == 6

                    and set(
                        cached['unit_id']
                    ) == set(UNIT_IDS)

                    and (
                        cached['year'] == year
                    ).all()

                    and (
                        cached['scenario'] == scenario
                    ).all()

                    and (
                        cached['index'] == index
                    ).all()
                ):

                    s2_frames.append(
                        cached
                    )

                    print(
                        index,
                        'CACHED'
                    )

                    continue

            if composites is None:

                composites, counts = (

                    build_annual_composite(

                        year,

                        scenario
                    )
                )

            success = False

            for attempt in range(1, 4):

                try:

                    frame = summarize_annual_index(

                        year,

                        scenario,

                        index,

                        composites[index],

                        counts[index]
                    )

                    assert len(frame) == 6

                    temp_path = cache_path.with_suffix(
                        '.tmp'
                    )

                    frame.to_csv(

                        temp_path,

                        index=False
                    )

                    temp_path.replace(
                        cache_path
                    )

                    s2_frames.append(
                        frame
                    )

                    print(

                        index,

                        'EXTRACTED'
                    )

                    success = True

                    break

                except Exception as error:

                    print(

                        index,

                        'Attempt:',

                        attempt,

                        str(error)[:200]
                    )

                    time.sleep(
                        5 * attempt
                    )

            if not success:

                s2_failures.append({

                    'year':
                        year,

                    'scenario':
                        scenario,

                    'index':
                        index
                })

if s2_failures:

    with open(

        OUT / 'S2_EXTRACTION_FAILURES.json',

        'w'

    ) as f:

        json.dump(

            s2_failures,

            f,

            indent=2
        )

assert not s2_failures, s2_failures

s2_master = pd.concat(

    s2_frames,

    ignore_index=True
)

expected_rows = (

    (END_YEAR - START_YEAR + 1)

    * len(SCENARIOS)

    * len(INDEX_NAMES)

    * len(UNIT_IDS)
)

assert len(s2_master) == expected_rows

assert not s2_master.duplicated([

    'year',

    'scenario',

    'index',

    'unit_id'

]).any()

# ============================================================
# 14. QUALITY CONTROL
# ============================================================

s2_master = s2_master.merge(

    s2_grid_reference,

    on=[

        'unit_id',

        'resolution_m'
    ],

    how='left',

    validate='many_to_one'
)

numeric_columns = [

    'mean',

    'median',

    'sd',

    'p10',

    'p25',

    'p75',

    'p90',

    'n_valid_pixels',

    'valid_area_m2',

    'mean_valid_dates'
]

for column in numeric_columns:

    s2_master[column] = pd.to_numeric(

        s2_master[column],

        errors='coerce'
    )

s2_master['valid_fraction'] = (

    s2_master['valid_area_m2']

    / s2_master['grid_area_m2']
)

assert (

    s2_master['valid_fraction']

    .dropna()

    .between(0, 1.001)

).all()

s2_master['valid_fraction'] = (

    s2_master['valid_fraction']

    .clip(0, 1)
)

s2_master['adequate_support'] = (

    s2_master['valid_fraction']

    .ge(MIN_VALID_FRACTION)

    &

    s2_master['mean'].notna()

    &

    s2_master['sd'].notna()

    &

    s2_master['n_valid_pixels'].ge(10)
)

s2_master['iqr'] = (

    s2_master['p75']

    - s2_master['p25']
)

s2_master['p90_p10'] = (

    s2_master['p90']

    - s2_master['p10']
)

# CV is not a primary metric because
# spectral indices can approach zero.

s2_master['cv'] = np.where(

    s2_master['mean'] > 0.05,

    s2_master['sd']

    / s2_master['mean'],

    np.nan
)

normalized_indices = [

    'NDVI',

    'GNDVI',

    'NDRE',

    'NDMI',

    'NBR',

    'NBR2'
]

normalized_subset = s2_master.loc[

    s2_master['index'].isin(
        normalized_indices
    )
]

assert (

    normalized_subset['mean']

    .dropna()

    .between(-1.00001, 1.00001)

).all()

assert (

    s2_master['iqr']

    .dropna()

    .ge(-1e-8)

).all()

s2_qa = (

    s2_master.groupby([

        'scenario',

        'index',

        'resolution_m'
    ])

    .agg(

        n_records=(
            'unit_id',
            'size'
        ),

        n_adequate=(
            'adequate_support',
            'sum'
        ),

        mean_valid_fraction=(
            'valid_fraction',
            'mean'
        ),

        min_valid_fraction=(
            'valid_fraction',
            'min'
        ),

        mean_valid_dates=(
            'mean_valid_dates',
            'mean'
        ),

        minimum_mean=(
            'mean',
            'min'
        ),

        maximum_mean=(
            'mean',
            'max'
        ),

        maximum_sd=(
            'sd',
            'max'
        )
    )

    .reset_index()
)

print('\nSENTINEL-2 QUALITY CONTROL')

display(
    s2_qa.round(4)
)

# ============================================================
# 15. MATCHED FOREST COMPARISONS
# ============================================================

paired_rows = []

group_columns = [

    'year',

    'scenario',

    'spatial_scale',

    'index',

    'resolution_m'
]

for keys, group in (

    s2_master.groupby(
        group_columns
    )
):

    if len(group) != 2:

        continue

    secondary = group.loc[

        group['forest_type']
        == 'Late_secondary'
    ]

    mature = group.loc[

        group['forest_type']
        == 'Mature'
    ]

    if (

        len(secondary) != 1

        or len(mature) != 1
    ):

        continue

    s = secondary.iloc[0]

    m = mature.iloc[0]

    row = dict(

        zip(

            group_columns,

            keys
        )
    )

    row.update({

        'secondary_unit':
            s['unit_id'],

        'mature_unit':
            m['unit_id'],

        'secondary_coverage':
            s['valid_fraction'],

        'mature_coverage':
            m['valid_fraction'],

        'secondary_n_dates':
            s['mean_valid_dates'],

        'mature_n_dates':
            m['mean_valid_dates'],

        'paired_adequate':

            bool(

                s['adequate_support']

                and

                m['adequate_support']
            )
    })

    for metric in [

        'mean',

        'median',

        'sd',

        'iqr',

        'p90_p10',

        'cv'
    ]:

        row[
            f'secondary_{metric}'
        ] = s[metric]

        row[
            f'mature_{metric}'
        ] = m[metric]

        row[
            f'difference_{metric}'
        ] = (

            s[metric]

            - m[metric]
        )

        row[
            f'ratio_{metric}'
        ] = (

            s[metric] / m[metric]

            if pd.notna(m[metric])

            and m[metric] > EPS

            else np.nan
        )

    paired_rows.append(
        row
    )

s2_paired = pd.DataFrame(
    paired_rows
)

assert len(s2_paired) == (

    (END_YEAR - START_YEAR + 1)

    * len(SCENARIOS)

    * 3

    * len(INDEX_NAMES)
)

s2_primary = s2_paired.loc[

    (
        s2_paired['scenario']
        == 'Primary'
    )

    &

    s2_paired['paired_adequate']

].copy()

# ============================================================
# 16. FIELD-YEAR COMPARISON
# ============================================================

s2_field_year = s2_primary.loc[

    s2_primary['year']
    == FIELD_YEAR

].copy()

field_columns = [

    'spatial_scale',

    'index',

    'resolution_m',

    'secondary_mean',

    'mature_mean',

    'difference_mean',

    'secondary_sd',

    'mature_sd',

    'ratio_sd',

    'secondary_iqr',

    'mature_iqr',

    'ratio_iqr',

    'secondary_coverage',

    'mature_coverage'
]

print('\n2019 FIELD-PLOT COMPARISON')

display(

    s2_field_year.loc[

        s2_field_year['spatial_scale']
        == 'Plot',

        field_columns

    ].round(4)
)

print('\n2019 FRAGMENT COMPARISON')

display(

    s2_field_year.loc[

        s2_field_year['spatial_scale']
        == 'Fragment',

        field_columns

    ].round(4)
)

# ============================================================
# 17. TEMPORAL CONSISTENCY
# ============================================================

temporal_rows = []

for (scale, index), group in (

    s2_primary.groupby([

        'spatial_scale',

        'index'
    ])
):

    group = group.sort_values(
        'year'
    )

    n_years = len(group)

    for metric in [

        'mean',

        'sd',

        'iqr',

        'p90_p10'
    ]:

        differences = group[

            f'difference_{metric}'
        ].dropna()

        n = len(differences)

        if n == 0:

            continue

        n_positive = int(

            (differences > EPS).sum()
        )

        n_negative = int(

            (differences < -EPS).sum()
        )

        n_ties = (

            n

            - n_positive

            - n_negative
        )

        temporal_rows.append({

            'spatial_scale':
                scale,

            'index':
                index,

            'metric':
                metric,

            'n_years':
                n_years,

            'n_valid_years':
                n,

            'median_difference':
                differences.median(),

            'mean_difference':
                differences.mean(),

            'minimum_difference':
                differences.min(),

            'maximum_difference':
                differences.max(),

            'n_secondary_higher':
                n_positive,

            'n_mature_higher':
                n_negative,

            'n_ties':
                n_ties,

            'directional_consistency_pct':

                100

                * max(
                    n_positive,
                    n_negative
                )

                / n
        })

s2_temporal_consistency = pd.DataFrame(
    temporal_rows
)

print('\nTEMPORAL CONSISTENCY — FRAGMENTS')

display(

    s2_temporal_consistency.loc[

        (
            s2_temporal_consistency[
                'spatial_scale'
            ] == 'Fragment'
        )

        &

        s2_temporal_consistency[
            'metric'
        ].isin([

            'mean',

            'sd',

            'iqr'
        ])

    ].round(4)
)

# ============================================================
# 18. MASKING SENSITIVITY
# ============================================================

sensitivity_rows = []

for keys, group in (

    s2_paired.groupby([

        'year',

        'spatial_scale',

        'index'
    ])
):

    adequate = group.loc[

        group['paired_adequate']
    ]

    if not set(SCENARIOS).issubset(

        set(
            adequate['scenario']
        )
    ):

        continue

    indexed = adequate.set_index(
        'scenario'
    )

    for metric in [

        'mean',

        'sd',

        'iqr'
    ]:

        primary_value = indexed.loc[

            'Primary',

            f'difference_{metric}'
        ]

        for scenario in [

            'Strict_cloud',

            'Strict_temporal'
        ]:

            value = indexed.loc[

                scenario,

                f'difference_{metric}'
            ]

            sensitivity_rows.append({

                'year':
                    keys[0],

                'spatial_scale':
                    keys[1],

                'index':
                    keys[2],

                'metric':
                    metric,

                'scenario':
                    scenario,

                'primary_difference':
                    primary_value,

                'sensitivity_difference':
                    value,

                'change_in_difference':

                    value - primary_value,

                'same_direction':

                    bool(

                        np.sign(value)

                        ==

                        np.sign(primary_value)
                    )
            })

s2_sensitivity = pd.DataFrame(
    sensitivity_rows
)

if not s2_sensitivity.empty:

    s2_sensitivity_summary = (

        s2_sensitivity.groupby([

            'spatial_scale',

            'index',

            'metric',

            'scenario'
        ])

        .agg(

            n_matched_years=(
                'year',
                'size'
            ),

            same_direction_pct=(

                'same_direction',

                lambda x:

                    100 * x.mean()
            ),

            median_change=(

                'change_in_difference',

                'median'
            ),

            median_absolute_change=(

                'change_in_difference',

                lambda x:

                    x.abs().median()
            )
        )

        .reset_index()
    )

else:

    s2_sensitivity_summary = pd.DataFrame()

print('\nMASKING SENSITIVITY')

display(

    s2_sensitivity_summary.round(4)
)

# ============================================================
# 19. SPATIAL SCALE SENSITIVITY
# ============================================================

scale_rows = []

for keys, group in (

    s2_primary.groupby([

        'year',

        'index'
    ])
):

    if not {

        'Fragment',

        'Fragment_core',

        'Plot'

    }.issubset(

        set(
            group['spatial_scale']
        )
    ):

        continue

    indexed = group.set_index(
        'spatial_scale'
    )

    for metric in [

        'mean',

        'sd',

        'iqr'
    ]:

        fragment = indexed.loc[

            'Fragment',

            f'difference_{metric}'
        ]

        core = indexed.loc[

            'Fragment_core',

            f'difference_{metric}'
        ]

        plot = indexed.loc[

            'Plot',

            f'difference_{metric}'
        ]

        scale_rows.append({

            'year':
                keys[0],

            'index':
                keys[1],

            'metric':
                metric,

            'fragment_difference':
                fragment,

            'core_difference':
                core,

            'plot_difference':
                plot,

            'core_minus_fragment':
                core - fragment,

            'plot_minus_fragment':
                plot - fragment,

            'fragment_core_same_direction':

                bool(

                    np.sign(fragment)

                    == np.sign(core)
                ),

            'fragment_plot_same_direction':

                bool(

                    np.sign(fragment)

                    == np.sign(plot)
                )
        })

s2_scale_sensitivity = pd.DataFrame(
    scale_rows
)

print('\nSPATIAL SCALE SENSITIVITY — 2019')

if not s2_scale_sensitivity.empty:

    display(

        s2_scale_sensitivity.loc[

            s2_scale_sensitivity['year']
            == FIELD_YEAR

        ].round(4)
    )

# ============================================================
# 20. FIELD INVENTORY INTEGRATION
# ============================================================

# Values from the completed strict
# field biomass inventory.
#
# These are plot-level values.
# They must NOT be assigned to
# every pixel or entire fragment.

field_inventory_summary = pd.DataFrame([

    {

        'forest_type':
            'Late_secondary',

        'unit_id':
            'P01',

        'field_year':
            2019,

        'n_trees_biomass':
            541,

        'agb_mg_ha':
            355.2266,

        'mean_height_m':
            27.9667,

        'mean_dbh_cm':
            21.9578,

        'hill_q1_coverage_standardized':
            25.5663,

        'hill_q2_coverage_standardized':
            8.5519
    },

    {

        'forest_type':
            'Mature',

        'unit_id':
            'P02',

        'field_year':
            2019,

        'n_trees_biomass':
            461,

        'agb_mg_ha':
            234.3626,

        'mean_height_m':
            22.8915,

        'mean_dbh_cm':
            21.5054,

        'hill_q1_coverage_standardized':
            65.5537,

        'hill_q2_coverage_standardized':
            41.8154
    }
])

field_s2 = s2_master.loc[

    (
        s2_master['year']
        == FIELD_YEAR
    )

    &

    (
        s2_master['scenario']
        == 'Primary'
    )

    &

    (
        s2_master['spatial_scale']
        == 'Plot'
    )

    &

    s2_master['adequate_support']

].copy()

field_s2 = field_s2.merge(

    field_inventory_summary,

    on=[

        'unit_id',

        'forest_type'
    ],

    how='left',

    validate='many_to_one'
)

print('\nFIELD–SENTINEL INTEGRATION')

display(

    field_s2[

        [

            'unit_id',

            'index',

            'mean',

            'sd',

            'iqr',

            'agb_mg_ha',

            'mean_height_m',

            'hill_q1_coverage_standardized'
        ]

    ].round(4)
)

# ============================================================
# 21. FIGURE 1
# 2019 SPECTRAL MEANS AND HETEROGENEITY
# ============================================================

plt.rcParams.update({

    'font.family':
        'DejaVu Sans',

    'font.size':
        10,

    'pdf.fonttype':
        42,

    'ps.fonttype':
        42
})

COLORS = {

    'Late_secondary':
        '#DB9140',

    'Mature':
        '#24634F'
}

# Exclude MTCI from this panel because
# its numerical scale differs substantially.

plot_indices = [

    'NDVI',

    'EVI',

    'GNDVI',

    'NDRE',

    'NDMI',

    'NBR',

    'NBR2'
]

fig, axes = plt.subplots(

    2,

    2,

    figsize=(14, 9)
)

for row_number, scale in enumerate([

    'Plot',

    'Fragment'
]):

    subset = s2_field_year.loc[

        s2_field_year['spatial_scale']
        == scale

    ].set_index(

        'index'

    ).reindex(
        plot_indices
    )

    x = np.arange(
        len(plot_indices)
    )

    width = 0.36

    for col_number, metric in enumerate([

        'mean',

        'sd'
    ]):

        ax = axes[
            row_number,
            col_number
        ]

        ax.bar(

            x - width / 2,

            subset[
                f'secondary_{metric}'
            ],

            width=width,

            color=COLORS[
                'Late_secondary'
            ],

            label='Late secondary'
        )

        ax.bar(

            x + width / 2,

            subset[
                f'mature_{metric}'
            ],

            width=width,

            color=COLORS[
                'Mature'
            ],

            label='Mature'
        )

        ax.set_xticks(

            x,

            plot_indices,

            rotation=25
        )

        metric_label = (

            'Spectral mean'

            if metric == 'mean'

            else 'Spatial standard deviation'
        )

        ax.set_title(

            f'{scale}: {metric_label}',

            loc='left',

            fontweight='bold'
        )

        ax.set_ylabel(
            metric_label
        )

        ax.spines[

            ['top', 'right']

        ].set_visible(False)

axes[0, 0].legend(
    frameon=False
)

fig.suptitle(

    'Canopy spectral properties and '
    'spatial heterogeneity in 2019',

    fontsize=17,

    fontweight='bold'
)

fig.text(

    0.5,

    0.015,

    'Sentinel-2 seasonal median composites | '
    'May–September | Spatial SD is descriptive, '
    'not uncertainty of the mean',

    ha='center',

    fontsize=9
)

fig.tight_layout(

    rect=[0, 0.04, 1, 0.95]
)

figure1 = (

    FIGURE_DIR

    / 'FIG_S2_FIELD_YEAR_SPECTRAL_STRUCTURE'
)

for extension in [

    'png',

    'pdf',

    'svg'
]:

    fig.savefig(

        f'{figure1}.{extension}',

        dpi=600,

        bbox_inches='tight'
    )

plt.show()

# ============================================================
# 22. FIGURE 2
# TEMPORAL SPECTRAL CONTRASTS
# ============================================================

selected_indices = [

    'NDVI',

    'EVI',

    'NDRE',

    'NDMI',

    'NBR',

    'NBR2'
]

fig, axes = plt.subplots(

    2,

    3,

    figsize=(16, 9),

    sharex=True
)

scale_colors = {

    'Fragment':
        '#24634F',

    'Fragment_core':
        '#527B98',

    'Plot':
        '#DB9140'
}

for ax, index in zip(

    axes.flat,

    selected_indices
):

    subset = s2_primary.loc[

        s2_primary['index']
        == index
    ]

    ax.axhline(

        0,

        color='gray',

        linewidth=0.8
    )

    for scale in [

        'Fragment',

        'Fragment_core',

        'Plot'
    ]:

        group = subset.loc[

            subset['spatial_scale']
            == scale

        ].sort_values('year')

        # Reindex to avoid connecting
        # across unsupported years.

        group = group.set_index(

            'year'

        ).reindex(

            range(

                START_YEAR,

                END_YEAR + 1
            )
        )

        ax.plot(

            group.index,

            group['difference_mean'],

            marker='o',

            markersize=4,

            linewidth=1.6,

            color=scale_colors[scale],

            label=scale.replace(
                '_',
                ' '
            )
        )

    ax.set_title(

        index,

        fontweight='bold'
    )

    ax.set_ylabel(

        'Secondary − mature'
    )

    ax.spines[

        ['top', 'right']

    ].set_visible(False)

for ax in axes[-1]:

    ax.set_xlabel(
        'Year'
    )

handles, labels = axes[0, 0].get_legend_handles_labels()

fig.legend(

    handles,

    labels,

    loc='lower center',

    ncol=3,

    frameon=False
)

fig.suptitle(

    'Temporal consistency of '
    'between-forest spectral contrasts',

    fontsize=17,

    fontweight='bold'
)

fig.tight_layout(

    rect=[0, 0.06, 1, 0.94]
)

figure2 = (

    FIGURE_DIR

    / 'FIG_S2_TEMPORAL_SPECTRAL_CONTRASTS'
)

for extension in [

    'png',

    'pdf',

    'svg'
]:

    fig.savefig(

        f'{figure2}.{extension}',

        dpi=600,

        bbox_inches='tight'
    )

plt.show()

# ============================================================
# 23. FIGURE 3
# SPATIAL HETEROGENEITY CONTRASTS
# ============================================================

fig, axes = plt.subplots(

    2,

    3,

    figsize=(16, 9),

    sharex=True
)

for ax, index in zip(

    axes.flat,

    selected_indices
):

    subset = s2_primary.loc[

        s2_primary['index']
        == index
    ]

    ax.axhline(

        1,

        color='gray',

        linewidth=0.8
    )

    for scale in [

        'Fragment',

        'Fragment_core',

        'Plot'
    ]:

        group = subset.loc[

            subset['spatial_scale']
            == scale

        ].set_index(

            'year'

        ).reindex(

            range(

                START_YEAR,

                END_YEAR + 1
            )
        )

        ax.plot(

            group.index,

            group['ratio_sd'],

            marker='o',

            markersize=4,

            linewidth=1.6,

            color=scale_colors[scale],

            label=scale.replace(
                '_',
                ' '
            )
        )

    ax.set_title(

        index,

        fontweight='bold'
    )

    ax.set_ylabel(

        'Spatial SD ratio (S/M)'
    )

    ax.spines[

        ['top', 'right']

    ].set_visible(False)

for ax in axes[-1]:

    ax.set_xlabel(
        'Year'
    )

handles, labels = axes[0, 0].get_legend_handles_labels()

fig.legend(

    handles,

    labels,

    loc='lower center',

    ncol=3,

    frameon=False
)

fig.suptitle(

    'Relative canopy spectral heterogeneity '
    'across spatial scales',

    fontsize=17,

    fontweight='bold'
)

fig.tight_layout(

    rect=[0, 0.06, 1, 0.94]
)

figure3 = (

    FIGURE_DIR

    / 'FIG_S2_SPATIAL_HETEROGENEITY'
)

for extension in [

    'png',

    'pdf',

    'svg'
]:

    fig.savefig(

        f'{figure3}.{extension}',

        dpi=600,

        bbox_inches='tight'
    )

plt.show()

# ============================================================
# 24. EXPORT TABLES
# ============================================================

exports = {

    'S2_SCENE_SUPPORT':
        s2_scene_support,

    'S2_GRID_REFERENCE':
        s2_grid_reference,

    'S2_CANOPY_MASTER':
        s2_master,

    'S2_CANOPY_QA':
        s2_qa,

    'S2_PAIRED_COMPARISONS':
        s2_paired,

    'S2_PRIMARY_COMPARISONS':
        s2_primary,

    'S2_FIELD_YEAR_2019':
        s2_field_year,

    'S2_TEMPORAL_CONSISTENCY':
        s2_temporal_consistency,

    'S2_MASKING_SENSITIVITY':
        s2_sensitivity,

    'S2_MASKING_SENSITIVITY_SUMMARY':
        s2_sensitivity_summary,

    'S2_SPATIAL_SCALE_SENSITIVITY':
        s2_scale_sensitivity,

    'S2_FIELD_INVENTORY_INTEGRATION':
        field_s2
}

for name, frame in exports.items():

    frame.to_csv(

        OUT / f'{name}.csv',

        index=False,

        encoding='utf-8-sig'
    )

WORKBOOK = (

    TABLE_DIR

    / 'SATIPO_SENTINEL2_CANOPY_STRUCTURE.xlsx'
)

with pd.ExcelWriter(

    WORKBOOK,

    engine='openpyxl'

) as writer:

    sheets = {

        'Scene_support':
            s2_scene_support,

        'Grid_reference':
            s2_grid_reference,

        'Canopy_master':
            s2_master,

        'Quality_control':
            s2_qa,

        'Paired_comparisons':
            s2_paired,

        'Primary_comparisons':
            s2_primary,

        'Field_year_2019':
            s2_field_year,

        'Temporal_consistency':
            s2_temporal_consistency,

        'Masking_sensitivity':
            s2_sensitivity,

        'Masking_summary':
            s2_sensitivity_summary,

        'Scale_sensitivity':
            s2_scale_sensitivity,

        'Field_integration':
            field_s2
    }

    for sheet_name, frame in sheets.items():

        frame.to_excel(

            writer,

            sheet_name=sheet_name,

            index=False
        )

# ============================================================
# 25. MANIFEST
# ============================================================

manifest = {

    'project':
        'SATIPO_FOREST_PROJECT',

    'cell':
        96,

    'created_utc':

        datetime.now(
            timezone.utc
        ).isoformat(),

    'configuration':
        s2_config,

    'configuration_hash':
        config_hash,

    'geometry_source':
        geometry_source,

    'field_reference_year':
        FIELD_YEAR,

    'master_rows':
        len(s2_master),

    'paired_rows':
        len(s2_paired),

    'adequate_primary_pairs':
        len(s2_primary),

    'field_year_pairs':
        len(s2_field_year),

    'workbook':
        str(WORKBOOK),

    'figures': [

        str(figure1),

        str(figure2),

        str(figure3)
    ],

    'limitations': [

        'Spectral indices are indirect '
        'canopy proxies, not direct '
        'measurements of biomass or '
        'taxonomic diversity.',

        'The two field plots are not '
        'independent replicates of '
        'forest successional categories.',

        'Pixels within forest fragments '
        'are spatially autocorrelated.',

        'Spatial standard deviation '
        'describes spectral heterogeneity '
        'and is not a standard error.',

        'Indices at 10 and 20 m '
        'have different spatial support.',

        'Annual seasonal composites '
        'may differ in the distribution '
        'of valid observation dates.',

        'Cloud Score+ and SCL screening '
        'may introduce observation '
        'selection effects.',

        'MTCI is sensitive to small '
        'red-edge denominators; '
        'a minimum denominator of '
        '0.005 reflectance units is used.',

        'Field biomass and diversity '
        'are linked only to the '
        '2019 field plots.',

        'No pixel-level biomass model '
        'or biodiversity regression '
        'is fitted using two plots.'
    ]
}

MANIFEST_PATH = (

    OUT

    / 'S2_CANOPY_STRUCTURE_MANIFEST.json'
)

with open(

    MANIFEST_PATH,

    'w',

    encoding='utf-8'

) as f:

    json.dump(

        manifest,

        f,

        indent=2,

        ensure_ascii=False,

        default=str
    )

# ============================================================
# 26. FINAL REPORT
# ============================================================

print('\n' + '=' * 78)

print(
    'SATIPO — SENTINEL-2 CANOPY STRUCTURE REPORT'
)

print('=' * 78)

print(

    'Years:',

    START_YEAR,

    '–',

    END_YEAR
)

print(

    'Candidate granules:',

    int(

        s2_scene_support[
            'n_candidate_granules'
        ].sum()
    )
)

print(

    'Candidate dates:',

    int(

        s2_scene_support[
            'n_candidate_dates'
        ].sum()
    )
)

print(

    'Spectral indices:',

    len(INDEX_NAMES)
)

print(

    'Spatial units:',

    len(UNIT_IDS)
)

print(

    'Screening scenarios:',

    len(SCENARIOS)
)

print(

    'Master observations:',

    len(s2_master)
)

print(

    'Paired comparisons:',

    len(s2_paired)
)

print(

    'Adequate primary comparisons:',

    len(s2_primary)
)

print('\n2019 FIELD-PLOT RESULTS')

display(

    s2_field_year.loc[

        s2_field_year['spatial_scale']
        == 'Plot',

        field_columns

    ].round(4)
)

print('\n2019 FRAGMENT RESULTS')

display(

    s2_field_year.loc[

        s2_field_year['spatial_scale']
        == 'Fragment',

        field_columns

    ].round(4)
)

print('\nFRAGMENT TEMPORAL CONSISTENCY')

display(

    s2_temporal_consistency.loc[

        (
            s2_temporal_consistency[
                'spatial_scale'
            ] == 'Fragment'
        )

        &

        s2_temporal_consistency[
            'metric'
        ].isin([

            'mean',

            'sd'
        ])

    ].round(4)
)

print('\nOUTPUT DIRECTORY:')

print(OUT)

print('\nWORKBOOK:')

print(WORKBOOK)

print('\nMANIFEST:')

print(MANIFEST_PATH)

print('\n' + '=' * 78)

print('CELL 96 COMPLETED')

print(

    'NEXT: CELL 97 — '
    'SENTINEL-2 SPATIAL SPECTRAL '
    'DIVERSITY AND TEXTURE'
)

print('=' * 78)

In [ ]:

# ============================================================
# CELL 97 — GEDI CANOPY HEIGHT AVAILABILITY
# Final structural remote-sensing diagnostic
# ============================================================

import ee
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

from pathlib import Path
import json

ROOT = Path("/content/drive/MyDrive/SATIPO_FOREST_PROJECT")
REMOTE = ROOT / "04_REMOTE_SENSING"

GEDI_DIR = REMOTE / "05_GEDI"
GEDI_DIR.mkdir(parents=True, exist_ok=True)

TABLE_DIR = REMOTE / "08_OUTPUTS" / "TABLES"
FIGURE_DIR = REMOTE / "08_OUTPUTS" / "FIGURES"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Recover validated geometries
# ------------------------------------------------------------

REQUIRED_UNITS = {
    "FRAG_01", "FRAG_02",
    "P01", "P02",
    "FRAG_01_CORE30", "FRAG_02_CORE30"
}

gedi_geometry = None

for name, obj in list(globals().items()):
    if isinstance(obj, gpd.GeoDataFrame):
        if "unit_id" in obj.columns:
            if REQUIRED_UNITS.issubset(
                set(obj["unit_id"].astype(str))
            ):
                gedi_geometry = obj.copy()
                print("Geometry recovered:", name)
                break

# Fallback: recover saved spatial inputs

if gedi_geometry is None:

    for path in (REMOTE / "02_GEOMETRY_VALIDATION").rglob("*"):

        if path.suffix.lower() not in {
            ".gpkg", ".geojson", ".shp"
        }:
            continue

        try:
            layers = (
                gpd.list_layers(path)["name"].tolist()
                if path.suffix.lower() == ".gpkg"
                else [None]
            )

            for layer in layers:

                candidate = gpd.read_file(
                    path, layer=layer
                )

                if "unit_id" not in candidate.columns:
                    continue

                if REQUIRED_UNITS.issubset(
                    set(candidate["unit_id"].astype(str))
                ):
                    gedi_geometry = candidate.copy()
                    print("Geometry loaded:", path)
                    break

            if gedi_geometry is not None:
                break

        except Exception:
            continue

assert gedi_geometry is not None, \
    "Validated geometry layer not found."

gedi_geometry = (
    gedi_geometry[
        gedi_geometry["unit_id"].isin(REQUIRED_UNITS)
    ]
    .drop_duplicates("unit_id")
    .to_crs("EPSG:4326")
)

assert len(gedi_geometry) == 6
assert gedi_geometry.geometry.is_valid.all()

forest_lookup = {
    "FRAG_01": "Late_secondary",
    "FRAG_02": "Mature",
    "P01": "Late_secondary",
    "P02": "Mature",
    "FRAG_01_CORE30": "Late_secondary",
    "FRAG_02_CORE30": "Mature"
}

gedi_features = []

for _, row in gedi_geometry.iterrows():

    geom = ee.Geometry(
        row.geometry.__geo_interface__,
        proj="EPSG:4326",
        geodesic=False
    )

    gedi_features.append(
        ee.Feature(
            geom,
            {
                "unit_id": row["unit_id"],
                "forest_type": forest_lookup[row["unit_id"]]
            }
        )
    )

gedi_fc = ee.FeatureCollection(gedi_features)

# ------------------------------------------------------------
# 2. GEDI L2A monthly raster collection
# ------------------------------------------------------------

gedi_collection = (
    ee.ImageCollection("LARSE/GEDI/GEDI02_A_002_MONTHLY")
    .filterBounds(gedi_fc.geometry())
    .filterDate("2019-01-01", "2026-01-01")
)

n_images = gedi_collection.size().getInfo()

print("\nGEDI monthly images:", n_images)

assert n_images > 0, "No GEDI monthly images available."

# ------------------------------------------------------------
# 3. Quality filtering
# ------------------------------------------------------------

def mask_gedi(image):

    quality = image.select("quality_flag").eq(1)
    degradation = image.select("degrade_flag").eq(0)
    sensitivity = image.select("sensitivity").gte(0.90)

    rh98 = image.select("rh98")
    rh95 = image.select("rh95")

    physical = (
        rh98.gt(0)
        .And(rh98.lte(80))
        .And(rh95.gte(0))
        .And(rh95.lte(rh98))
    )

    mask = (
        quality
        .And(degradation)
        .And(sensitivity)
        .And(physical)
    )

    return (
        image.select(["rh95", "rh98", "sensitivity"])
        .updateMask(mask)
        .copyProperties(image, ["system:time_start"])
    )

gedi_clean = gedi_collection.map(mask_gedi)

# ------------------------------------------------------------
# 4. Extract monthly valid raster observations
# ------------------------------------------------------------

def extract_gedi_month(image):

    image = ee.Image(image)

    date = ee.Date(image.get("system:time_start"))

    def extract_unit(feature):

        feature = ee.Feature(feature)

        samples = image.sample(
            region=feature.geometry(),
            scale=25,
            projection=image.select("rh98").projection(),
            geometries=True,
            dropNulls=True,
            tileScale=4
        )

        def add_metadata(sample):

            sample = ee.Feature(sample)

            coords = sample.geometry().coordinates()

            return sample.set({
                "unit_id": feature.get("unit_id"),
                "forest_type": feature.get("forest_type"),
                "date": date.format("YYYY-MM-dd"),
                "year": date.get("year"),
                "longitude": coords.get(0),
                "latitude": coords.get(1)
            })

        return samples.map(add_metadata)

    return ee.FeatureCollection(
        gedi_fc.map(extract_unit)
    ).flatten()

gedi_rows = []

for year in range(2019, 2026):

    annual = gedi_clean.filterDate(
        f"{year}-01-01",
        f"{year + 1}-01-01"
    )

    extracted = ee.FeatureCollection(
        annual.map(extract_gedi_month)
    ).flatten()

    result = extracted.getInfo()

    rows = [
        feature["properties"]
        for feature in result["features"]
    ]

    gedi_rows.extend(rows)

    print(year, "| Valid pixel-month records:", len(rows))

# ------------------------------------------------------------
# 5. Availability summary
# ------------------------------------------------------------

gedi_master = pd.DataFrame(gedi_rows)

if not gedi_master.empty:

    gedi_master = (
        gedi_master
        .drop_duplicates(
            subset=[
                "unit_id", "date",
                "longitude", "latitude"
            ]
        )
        .reset_index(drop=True)
    )

    gedi_master["pixel_id"] = (
        gedi_master["longitude"].round(6).astype(str)
        + "_"
        + gedi_master["latitude"].round(6).astype(str)
    )

    gedi_summary = (
        gedi_master.groupby(
            ["unit_id", "forest_type"]
        )
        .agg(
            n_pixel_months=("rh98", "size"),
            n_unique_pixels=("pixel_id", "nunique"),
            n_months=("date", "nunique"),
            n_years=("year", "nunique"),
            rh95_median=("rh95", "median"),
            rh98_mean=("rh98", "mean"),
            rh98_median=("rh98", "median"),
            rh98_sd=("rh98", "std"),
            rh98_p25=("rh98", lambda x: x.quantile(0.25)),
            rh98_p75=("rh98", lambda x: x.quantile(0.75)),
            sensitivity_mean=("sensitivity", "mean")
        )
        .reset_index()
    )

else:

    gedi_summary = pd.DataFrame(
        columns=[
            "unit_id", "forest_type",
            "n_pixel_months", "n_unique_pixels",
            "n_months", "n_years",
            "rh95_median", "rh98_mean",
            "rh98_median", "rh98_sd",
            "rh98_p25", "rh98_p75",
            "sensitivity_mean"
        ]
    )

all_units = pd.DataFrame({
    "unit_id": list(forest_lookup),
    "forest_type": list(forest_lookup.values())
})

gedi_summary = all_units.merge(
    gedi_summary,
    on=["unit_id", "forest_type"],
    how="left"
)

count_columns = [
    "n_pixel_months",
    "n_unique_pixels",
    "n_months",
    "n_years"
]

gedi_summary[count_columns] = (
    gedi_summary[count_columns]
    .fillna(0)
    .astype(int)
)

# Exploratory support criterion, not a statistical
# sample-size or independence guarantee.

gedi_summary["support_status"] = np.where(
    (gedi_summary["n_unique_pixels"] >= 10)
    & (gedi_summary["n_months"] >= 3),
    "DESCRIPTIVE_SUPPORT",
    "LIMITED_SUPPORT"
)

print("\nGEDI AVAILABILITY AND HEIGHT SUMMARY")
display(gedi_summary.round(4))

# ------------------------------------------------------------
# 6. Decide whether to retain GEDI
# ------------------------------------------------------------

fragment_support = gedi_summary[
    gedi_summary["unit_id"].isin(
        ["FRAG_01", "FRAG_02"]
    )
]

plot_support = gedi_summary[
    gedi_summary["unit_id"].isin(
        ["P01", "P02"]
    )
]

retain_fragment_gedi = bool(
    len(fragment_support) == 2
    and fragment_support["support_status"]
    .eq("DESCRIPTIVE_SUPPORT").all()
)

retain_plot_gedi = bool(
    len(plot_support) == 2
    and plot_support["support_status"]
    .eq("DESCRIPTIVE_SUPPORT").all()
)

print("\nRETAIN GEDI FRAGMENT COMPARISON:",
      retain_fragment_gedi)

print("RETAIN GEDI PLOT COMPARISON:",
      retain_plot_gedi)

# ------------------------------------------------------------
# 7. Optional descriptive figure
# ------------------------------------------------------------

if retain_fragment_gedi:

    fig, ax = plt.subplots(figsize=(7, 5))

    plot_data = [
        gedi_master.loc[
            gedi_master["unit_id"] == unit, "rh98"
        ].dropna().values
        for unit in ["FRAG_01", "FRAG_02"]
    ]

    bp = ax.boxplot(
        plot_data,
        patch_artist=True,
        widths=0.45,
        showfliers=False
    )

    for box, color in zip(
        bp["boxes"], ["#DB9140", "#24634F"]
    ):
        box.set_facecolor(color)
        box.set_alpha(0.8)

    ax.set_xticks([1, 2])
    ax.set_xticklabels(["Late secondary", "Mature"])
    ax.set_ylabel("GEDI RH98 (m)")
    ax.set_title("GEDI canopy-height distributions")
    ax.spines[["top", "right"]].set_visible(False)

    fig.tight_layout()

    for ext in ["png", "pdf", "svg"]:

        fig.savefig(
            FIGURE_DIR / f"FIG_GEDI_CANOPY_HEIGHT.{ext}",
            dpi=300,
            bbox_inches="tight"
        )

    plt.show()

# ------------------------------------------------------------
# 8. Export
# ------------------------------------------------------------

gedi_master.to_csv(
    GEDI_DIR / "GEDI_VALID_PIXEL_MONTHS.csv",
    index=False
)

gedi_summary.to_csv(
    GEDI_DIR / "GEDI_AVAILABILITY_SUMMARY.csv",
    index=False
)

with pd.ExcelWriter(
    TABLE_DIR / "SATIPO_GEDI_SUMMARY.xlsx"
) as writer:

    gedi_summary.to_excel(
        writer,
        sheet_name="Availability",
        index=False
    )

    gedi_master.to_excel(
        writer,
        sheet_name="Valid_records",
        index=False
    )

manifest = {
    "cell": 97,
    "collection": "LARSE/GEDI/GEDI02_A_002_MONTHLY",
    "period": "2019-2025",
    "quality_flag": 1,
    "degrade_flag": 0,
    "minimum_sensitivity": 0.90,
    "n_valid_records": int(len(gedi_master)),
    "retain_fragment_comparison": retain_fragment_gedi,
    "retain_plot_comparison": retain_plot_gedi,
    "interpretation": (
        "Monthly raster pixel observations; "
        "not independent laser footprints. "
        "RH98 is not equivalent to mean field tree height."
    )
}

with open(
    GEDI_DIR / "GEDI_MANIFEST.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(manifest, f, indent=2)

print("\n" + "=" * 75)
print("CELL 97 COMPLETED")
print("NEXT: CELL 98 — TOPOGRAPHIC CONTEXT")
print("=" * 75)


In [ ]:

# ============================================================
# CELL 98 — TOPOGRAPHIC CONTEXT
# Final environmental characterization
# SATIPO FOREST PROJECT
# ============================================================

import ee
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import json

from pathlib import Path

ROOT = Path("/content/drive/MyDrive/SATIPO_FOREST_PROJECT")
REMOTE = ROOT / "04_REMOTE_SENSING"

TOPO_DIR = REMOTE / "06_TOPOGRAPHY"
TABLE_DIR = REMOTE / "08_OUTPUTS" / "TABLES"
FIGURE_DIR = REMOTE / "08_OUTPUTS" / "FIGURES"

for directory in [TOPO_DIR, TABLE_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Recover validated geometries from Cell 97
# ------------------------------------------------------------

assert "gedi_geometry" in globals(), \
    "Run Cell 97 or recover validated geometries."

topo_geometry = gedi_geometry.copy().to_crs("EPSG:4326")

assert len(topo_geometry) == 6
assert topo_geometry.geometry.is_valid.all()

topo_features = []

for _, row in topo_geometry.iterrows():

    unit_id = str(row["unit_id"])

    if unit_id.startswith("P"):
        spatial_scale = "Plot"
    elif "CORE" in unit_id:
        spatial_scale = "Fragment_core"
    else:
        spatial_scale = "Fragment"

    geom = ee.Geometry(
        row.geometry.__geo_interface__,
        proj="EPSG:4326",
        geodesic=False
    )

    topo_features.append(
        ee.Feature(
            geom,
            {
                "unit_id": unit_id,
                "forest_type": forest_lookup[unit_id],
                "spatial_scale": spatial_scale
            }
        )
    )

topo_fc = ee.FeatureCollection(topo_features)

# ------------------------------------------------------------
# 2. NASADEM and terrain derivatives
# ------------------------------------------------------------

dem = (
    ee.Image("NASA/NASADEM_HGT/001")
    .select("elevation")
    .rename("elevation_m")
)

terrain = ee.Terrain.products(dem)

slope = terrain.select("slope").rename("slope_deg")

aspect = terrain.select("aspect")

# Circular representation of aspect.
# Avoid arithmetic averages of compass degrees.

aspect_rad = aspect.multiply(np.pi / 180)

northness = aspect_rad.cos().rename("northness")
eastness = aspect_rad.sin().rename("eastness")

# Mask aspect where slope is effectively flat.

aspect_mask = slope.gte(1)

northness = northness.updateMask(aspect_mask)
eastness = eastness.updateMask(aspect_mask)

topo_image = (
    dem
    .addBands(slope)
    .addBands(northness)
    .addBands(eastness)
)

print("DEM projection:", dem.projection().getInfo())

# ------------------------------------------------------------
# 3. Descriptive zonal statistics
# ------------------------------------------------------------

topo_reducer = (
    ee.Reducer.mean()
    .combine(
        reducer2=ee.Reducer.median(),
        sharedInputs=True
    )
    .combine(
        reducer2=ee.Reducer.stdDev(),
        sharedInputs=True
    )
    .combine(
        reducer2=ee.Reducer.minMax(),
        sharedInputs=True
    )
    .combine(
        reducer2=ee.Reducer.percentile([25, 75]),
        sharedInputs=True
    )
    .combine(
        reducer2=ee.Reducer.count(),
        sharedInputs=True
    )
)

topo_stats = topo_image.reduceRegions(
    collection=topo_fc,
    reducer=topo_reducer,
    crs="EPSG:32718",
    crsTransform=[30, 0, 0, 0, -30, 10000000],
    tileScale=4
)

topo_result = topo_stats.getInfo()

topo_master = pd.DataFrame([
    feature["properties"]
    for feature in topo_result["features"]
])

assert len(topo_master) == 6
assert topo_master["unit_id"].is_unique

# ------------------------------------------------------------
# 4. Geometry area and elevation range
# ------------------------------------------------------------

geometry_area = topo_geometry.to_crs("EPSG:32718")

area_lookup = dict(zip(
    geometry_area["unit_id"],
    geometry_area.geometry.area / 10000
))

topo_master["area_ha"] = (
    topo_master["unit_id"].map(area_lookup)
)

topo_master["elevation_range_m"] = (
    topo_master["elevation_m_max"]
    - topo_master["elevation_m_min"]
)

# Mean aspect direction from circular components.
# Resultant length near zero means weak directional
# concentration; direction should then be interpreted
# cautiously.

topo_master["aspect_resultant_length"] = np.sqrt(
    topo_master["northness_mean"] ** 2
    + topo_master["eastness_mean"] ** 2
)

topo_master["mean_aspect_deg"] = (
    np.degrees(
        np.arctan2(
            topo_master["eastness_mean"],
            topo_master["northness_mean"]
        )
    ) + 360
) % 360

# ------------------------------------------------------------
# 5. Area coverage audit
# ------------------------------------------------------------

grid_area_image = ee.Image.pixelArea().rename(
    "grid_area_m2"
)

valid_area_image = (
    grid_area_image
    .updateMask(dem.mask())
    .rename("valid_area_m2")
)

area_image = grid_area_image.addBands(valid_area_image)

area_stats = area_image.reduceRegions(
    collection=topo_fc,
    reducer=ee.Reducer.sum(),
    crs="EPSG:32718",
    crsTransform=[30, 0, 0, 0, -30, 10000000],
    tileScale=4
)

area_df = pd.DataFrame([
    feature["properties"]
    for feature in area_stats.getInfo()["features"]
])

topo_master = topo_master.merge(
    area_df[
        ["unit_id", "grid_area_m2", "valid_area_m2"]
    ],
    on="unit_id",
    how="left",
    validate="one_to_one"
)

topo_master["valid_fraction"] = (
    topo_master["valid_area_m2"]
    / topo_master["grid_area_m2"]
)

assert topo_master["valid_fraction"].between(
    0, 1.001
).all()

topo_master["valid_fraction"] = (
    topo_master["valid_fraction"].clip(0, 1)
)

# ------------------------------------------------------------
# 6. Manuscript-ready summary
# ------------------------------------------------------------

topo_summary_columns = [
    "unit_id",
    "forest_type",
    "spatial_scale",
    "area_ha",
    "elevation_m_mean",
    "elevation_m_median",
    "elevation_m_stdDev",
    "elevation_m_min",
    "elevation_m_max",
    "elevation_range_m",
    "slope_deg_mean",
    "slope_deg_median",
    "slope_deg_stdDev",
    "slope_deg_min",
    "slope_deg_max",
    "mean_aspect_deg",
    "aspect_resultant_length",
    "valid_fraction"
]

topo_summary = topo_master[
    topo_summary_columns
].copy()

unit_order = [
    "FRAG_01",
    "FRAG_02",
    "P01",
    "P02",
    "FRAG_01_CORE30",
    "FRAG_02_CORE30"
]

topo_summary = (
    topo_summary
    .set_index("unit_id")
    .loc[unit_order]
    .reset_index()
)

print("\nTOPOGRAPHIC SUMMARY")
display(topo_summary.round(4))

# ------------------------------------------------------------
# 7. Paired contrasts
# ------------------------------------------------------------

topo_pairs = [
    ("Fragment", "FRAG_01", "FRAG_02"),
    ("Plot", "P01", "P02"),
    (
        "Fragment_core",
        "FRAG_01_CORE30",
        "FRAG_02_CORE30"
    )
]

topo_contrast_rows = []

for scale, secondary_id, mature_id in topo_pairs:

    secondary = topo_summary.loc[
        topo_summary["unit_id"] == secondary_id
    ].iloc[0]

    mature = topo_summary.loc[
        topo_summary["unit_id"] == mature_id
    ].iloc[0]

    for variable in [
        "elevation_m_mean",
        "elevation_range_m",
        "slope_deg_mean",
        "slope_deg_stdDev"
    ]:

        topo_contrast_rows.append({
            "spatial_scale": scale,
            "variable": variable,
            "secondary": float(secondary[variable]),
            "mature": float(mature[variable]),
            "difference_secondary_minus_mature": float(
                secondary[variable] - mature[variable]
            )
        })

topo_contrasts = pd.DataFrame(topo_contrast_rows)

print("\nTOPOGRAPHIC CONTRASTS")
display(topo_contrasts.round(4))

# ------------------------------------------------------------
# 8. Compact supplementary figure
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 2,
    figsize=(10, 4.5)
)

colors = {
    "Late_secondary": "#DB9140",
    "Mature": "#24634F"
}

scales = ["Fragment", "Plot", "Fragment_core"]

x = np.arange(len(scales))
width = 0.34

for ax, variable, ylabel, title in [
    (
        axes[0],
        "elevation_m_mean",
        "Elevation (m a.s.l.)",
        "Mean elevation"
    ),
    (
        axes[1],
        "slope_deg_mean",
        "Slope (degrees)",
        "Mean slope"
    )
]:

    for offset, forest, label in [
        (-width / 2, "Late_secondary", "Late secondary"),
        (width / 2, "Mature", "Mature")
    ]:

        values = []

        for scale in scales:

            value = topo_summary.loc[
                (topo_summary["spatial_scale"] == scale)
                & (topo_summary["forest_type"] == forest),
                variable
            ].iloc[0]

            values.append(value)

        ax.bar(
            x + offset,
            values,
            width=width,
            color=colors[forest],
            label=label
        )

    ax.set_xticks(x)
    ax.set_xticklabels(
        ["Fragment", "Plot", "Fragment core"]
    )

    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)

axes[0].legend(frameon=False)

fig.suptitle(
    "Topographic context of the studied forests",
    fontsize=14,
    fontweight="bold"
)

fig.tight_layout()

for ext in ["png", "pdf", "svg"]:

    fig.savefig(
        FIGURE_DIR / f"FIG_TOPOGRAPHIC_CONTEXT.{ext}",
        dpi=300,
        bbox_inches="tight"
    )

plt.show()

# ------------------------------------------------------------
# 9. Export
# ------------------------------------------------------------

topo_master.to_csv(
    TOPO_DIR / "TOPOGRAPHIC_MASTER.csv",
    index=False
)

topo_summary.to_csv(
    TOPO_DIR / "TOPOGRAPHIC_SUMMARY.csv",
    index=False
)

topo_contrasts.to_csv(
    TOPO_DIR / "TOPOGRAPHIC_CONTRASTS.csv",
    index=False
)

workbook_path = (
    TABLE_DIR / "SATIPO_TOPOGRAPHIC_CONTEXT.xlsx"
)

with pd.ExcelWriter(workbook_path) as writer:

    topo_summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    topo_contrasts.to_excel(
        writer,
        sheet_name="Paired_contrasts",
        index=False
    )

    topo_master.to_excel(
        writer,
        sheet_name="Full_statistics",
        index=False
    )

manifest = {
    "cell": 98,
    "dataset": "NASA/NASADEM_HGT/001",
    "nominal_resolution_m": 30,
    "analysis_crs": "EPSG:32718",
    "n_spatial_units": 6,
    "variables": [
        "elevation",
        "slope",
        "northness",
        "eastness"
    ],
    "minimum_valid_fraction": float(
        topo_summary["valid_fraction"].min()
    ),
    "interpretation": (
        "Descriptive topographic characterization. "
        "Spatial units are nested and do not constitute "
        "independent forest-type replicates. "
        "NASADEM is a terrain-elevation proxy and "
        "may retain vegetation-related elevation biases."
    )
}

manifest_path = TOPO_DIR / "TOPOGRAPHIC_MANIFEST.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("\n" + "=" * 75)
print("SATIPO — TOPOGRAPHIC CONTEXT REPORT")
print("=" * 75)

print("Spatial units:", len(topo_summary))
print("Minimum valid fraction:",
      topo_summary["valid_fraction"].min())

print("\nFIELD-PLOT TOPOGRAPHY")
display(
    topo_summary.loc[
        topo_summary["spatial_scale"] == "Plot"
    ].round(4)
)

print("\nOUTPUT DIRECTORY:", TOPO_DIR)
print("WORKBOOK:", workbook_path)
print("MANIFEST:", manifest_path)

print("\n" + "=" * 75)
print("CELL 98 COMPLETED")
print("SCIENTIFIC DATA EXTRACTION COMPLETED")
print("NEXT: CELL 99 — FINAL INTEGRATION AND ARCHIVE")
print("=" * 75)


In [ ]:

# ============================================================
# CELL 99 — FINAL INTEGRATION AND SCIENTIFIC ARCHIVE
# SATIPO FOREST PROJECT
# NO NEW SCIENTIFIC ANALYSES
# ============================================================

import pandas as pd
import numpy as np
import shutil
import hashlib
import json
import zipfile

from pathlib import Path
from datetime import datetime, timezone

ROOT = Path("/content/drive/MyDrive/SATIPO_FOREST_PROJECT")

REMOTE = ROOT / "04_REMOTE_SENSING"
OUTPUTS = ROOT / "05_OUTPUTS"

FINAL = ROOT / "08_FINAL_MANUSCRIPT_PACKAGE"

FOLDERS = {
    "main_figures": FINAL / "01_MAIN_FIGURES",
    "main_tables": FINAL / "02_MAIN_TABLES",
    "supp_figures": FINAL / "03_SUPPLEMENTARY_FIGURES",
    "supp_tables": FINAL / "04_SUPPLEMENTARY_TABLES",
    "remote": FINAL / "05_REMOTE_SENSING_RESULTS",
    "uncertainty": FINAL / "06_UNCERTAINTY",
    "metadata": FINAL / "07_METADATA",
    "results": FINAL / "08_RESULTS_SUMMARY",
    "review": FINAL / "09_REVIEW_REQUIRED"
}

for folder in FOLDERS.values():
    folder.mkdir(parents=True, exist_ok=True)

print("FINAL PACKAGE:", FINAL)

# ------------------------------------------------------------
# 1. Verify previous scientific checkpoint
# ------------------------------------------------------------

checkpoint_pointer = (
    ROOT / "06_CHECKPOINTS"
    / "LATEST_FINAL_CHECKPOINT.json"
)

assert checkpoint_pointer.exists(), (
    "Previous scientific checkpoint not found."
)

with open(checkpoint_pointer, encoding="utf-8") as f:
    previous_checkpoint = json.load(f)

shutil.copy2(
    checkpoint_pointer,
    FOLDERS["metadata"]
    / "STAGE_01_CHECKPOINT_POINTER.json"
)

print("Previous scientific checkpoint: FOUND")

# ------------------------------------------------------------
# 2. Recover original editorial package
# ------------------------------------------------------------

EDITORIAL = ROOT / "07_MANUSCRIPT_READY"

editorial_mapping = {
    "MAIN_FIGURES": "main_figures",
    "MAIN_TABLES": "main_tables",
    "SUPPLEMENTARY_FIGURES": "supp_figures",
    "SUPPLEMENTARY_TABLES": "supp_tables",
    "FIGURE_CAPTIONS": "metadata",
    "RESULTS_SUMMARY": "results",
    "METADATA": "metadata"
}

copied_records = []

def copy_verified(source, destination, category):

    source = Path(source)
    destination = Path(destination)

    if not source.is_file():
        return False

    destination.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    shutil.copy2(source, destination)

    def sha256_file(path):

        digest = hashlib.sha256()

        with open(path, "rb") as f:

            for chunk in iter(
                lambda: f.read(1024 * 1024),
                b""
            ):
                digest.update(chunk)

        return digest.hexdigest()

    source_hash = sha256_file(source)
    destination_hash = sha256_file(destination)

    assert source_hash == destination_hash, (
        f"Copy verification failed: {source}"
    )

    copied_records.append({
        "category": category,
        "source": str(source.relative_to(ROOT)),
        "destination": str(
            destination.relative_to(FINAL)
        ),
        "size_bytes": destination.stat().st_size,
        "sha256": destination_hash,
        "verified": True
    })

    return True


for source_folder, destination_key in (
    editorial_mapping.items()
):

    source_dir = EDITORIAL / source_folder

    if not source_dir.exists():
        print("Missing editorial folder:", source_dir)
        continue

    for source in sorted(source_dir.rglob("*")):

        if source.is_file():

            destination = (
                FOLDERS[destination_key]
                / source.relative_to(source_dir)
            )

            copy_verified(
                source,
                destination,
                destination_key
            )

print(
    "Original editorial files recovered:",
    len(copied_records)
)

# ------------------------------------------------------------
# 3. Recover integrated uncertainty outputs
# ------------------------------------------------------------

UNCERTAINTY_DIR = (
    ROOT / "03_ANALYSIS"
    / "INTEGRATED_UNCERTAINTY"
)

uncertainty_required = [
    "ALLOMETRIC_RESIDUAL_MC_SUMMARY.csv",
    "ALLOMETRIC_RESIDUAL_VALIDATION.csv",
    "EFFECTIVE_BIOMASS_CONTRIBUTORS.csv",
    "ALLOMETRIC_UNCERTAINTY_COMPARISON.csv",
    "INTEGRATED_MC_SUMMARY.csv",
    "INTEGRATED_MC_GAP_SUMMARY.csv",
    "INTEGRATED_MC_PRIMARY_SCENARIO.csv",
    "INTEGRATED_MC_STRESS_SCENARIO.csv",
    "ALLOMETRIC_CORRELATION_ANALYTICAL.csv",
    "SATIPO_INTEGRATED_UNCERTAINTY.xlsx",
    "INTEGRATED_MC_CONFIG.json"
]

missing_required = []

for filename in uncertainty_required:

    source = UNCERTAINTY_DIR / filename

    success = copy_verified(
        source,
        FOLDERS["uncertainty"] / filename,
        "uncertainty"
    )

    if not success:
        missing_required.append(str(source))

# Keep large simulation draws in the original project.
# Record their existence and dimensions.

simulation_files = [
    "ALLOMETRIC_RESIDUAL_MC_DRAWS.csv",
    "INTEGRATED_MC_DRAWS.csv"
]

simulation_inventory = []

for filename in simulation_files:

    path = UNCERTAINTY_DIR / filename

    if path.exists():

        simulation_inventory.append({
            "file": str(path.relative_to(ROOT)),
            "size_MB": path.stat().st_size / 1e6,
            "exists": True
        })

    else:

        missing_required.append(str(path))

pd.DataFrame(simulation_inventory).to_csv(
    FOLDERS["metadata"]
    / "SIMULATION_DATA_INVENTORY.csv",
    index=False
)

# Verify final Monte Carlo dimensions.

mc_draws_path = (
    UNCERTAINTY_DIR
    / "INTEGRATED_MC_DRAWS.csv"
)

mc_summary_path = (
    UNCERTAINTY_DIR
    / "INTEGRATED_MC_GAP_SUMMARY.csv"
)

assert mc_draws_path.exists()
assert mc_summary_path.exists()

mc_archive_draws = pd.read_csv(mc_draws_path)
mc_archive_summary = pd.read_csv(mc_summary_path)

assert len(mc_archive_draws) == 180000
assert len(mc_archive_summary) == 18

print("Integrated simulations: 180,000")
print("Integrated scenarios: 18")

# ------------------------------------------------------------
# 4. Recover Figure 8 — integrated uncertainty
# ------------------------------------------------------------

for ext in ["png", "pdf", "svg"]:

    source = (
        OUTPUTS / "FIGURES"
        / f"FIG_INTEGRATED_MONTE_CARLO.{ext}"
    )

    destination = (
        FOLDERS["main_figures"]
        / f"Fig_08_Integrated_Uncertainty.{ext}"
    )

    if not copy_verified(
        source,
        destination,
        "main_figures"
    ):
        missing_required.append(str(source))

# ------------------------------------------------------------
# 5. Supplementary Tables S27–S30
# ------------------------------------------------------------

supplementary_plan = {
    "Table_S27_Allometric_Residual_Uncertainty":
        "ALLOMETRIC_RESIDUAL_MC_SUMMARY.csv",

    "Table_S28A_Analytical_Validation":
        "ALLOMETRIC_RESIDUAL_VALIDATION.csv",

    "Table_S28B_Effective_Biomass_Contributors":
        "EFFECTIVE_BIOMASS_CONTRIBUTORS.csv",

    "Table_S29_Integrated_Monte_Carlo":
        "INTEGRATED_MC_GAP_SUMMARY.csv",

    "Table_S30_Residual_Correlation":
        "ALLOMETRIC_CORRELATION_ANALYTICAL.csv"
}

new_supp_tables = {}

for table_name, filename in (
    supplementary_plan.items()
):

    source = UNCERTAINTY_DIR / filename

    if source.exists():

        new_supp_tables[table_name] = pd.read_csv(
            source
        )

        copy_verified(
            source,
            FOLDERS["supp_tables"]
            / f"{table_name}.csv",
            "supp_tables"
        )

    else:

        missing_required.append(str(source))

with pd.ExcelWriter(
    FOLDERS["supp_tables"]
    / "SATIPO_SUPPLEMENTARY_TABLES_S27_S30.xlsx"
) as writer:

    for name, df in new_supp_tables.items():

        df.to_excel(
            writer,
            sheet_name=name[:31],
            index=False
        )

print("New uncertainty tables:", len(new_supp_tables))

# ------------------------------------------------------------
# 6. Archive remote sensing results
# ------------------------------------------------------------

remote_tables_dir = (
    REMOTE / "08_OUTPUTS" / "TABLES"
)

remote_figures_dir = (
    REMOTE / "08_OUTPUTS" / "FIGURES"
)

remote_workbooks = []

if remote_tables_dir.exists():

    for source in sorted(
        remote_tables_dir.glob("*.xlsx")
    ):

        copy_verified(
            source,
            FOLDERS["remote"]
            / "TABLES"
            / source.name,
            "remote_tables"
        )

        remote_workbooks.append(source.name)

# All remote-sensing figures are archived.
# Their final editorial placement remains provisional.

remote_figure_records = []

if remote_figures_dir.exists():

    for source in sorted(
        remote_figures_dir.rglob("*")
    ):

        if (
            source.is_file()
            and source.suffix.lower()
            in [".png", ".pdf", ".svg", ".tif", ".tiff"]
        ):

            copy_verified(
                source,
                FOLDERS["remote"]
                / "FIGURES"
                / source.relative_to(remote_figures_dir),
                "remote_figures"
            )

            remote_figure_records.append({
                "filename": source.name,
                "stem": source.stem,
                "extension": source.suffix,
                "editorial_status":
                    "Archived_pending_final_selection"
            })

# Preserve remote-sensing manifests.

for source in sorted(REMOTE.rglob("*MANIFEST*.json")):

    copy_verified(
        source,
        FOLDERS["remote"]
        / "MANIFESTS"
        / source.relative_to(REMOTE),
        "remote_manifest"
    )

# Verify essential completed remote-sensing workbooks.

essential_remote = [
    "SATIPO_HISTORICAL_DISTURBANCE.xlsx",
    "SATIPO_SPATIAL_DISTURBANCE.xlsx",
    "SATIPO_HYDROCLIMATIC_CONTEXT.xlsx",
    "SATIPO_TOPOGRAPHIC_CONTEXT.xlsx"
]

for filename in essential_remote:

    if filename not in remote_workbooks:
        missing_required.append(
            str(remote_tables_dir / filename)
        )

print("Remote sensing workbooks:", len(remote_workbooks))
print("Remote sensing figure files:",
      len(remote_figure_records))

pd.DataFrame(remote_figure_records).to_csv(
    FOLDERS["metadata"]
    / "REMOTE_FIGURE_CATALOGUE.csv",
    index=False
)

# ------------------------------------------------------------
# 7. Consolidated remote-sensing workbook
# ------------------------------------------------------------

remote_table_catalogue = []

consolidated_path = (
    FOLDERS["remote"]
    / "SATIPO_REMOTE_SENSING_CONSOLIDATED.xlsx"
)

with pd.ExcelWriter(
    consolidated_path,
    engine="openpyxl"
) as writer:

    sheet_counter = 0

    for filename in remote_workbooks:

        source = remote_tables_dir / filename

        workbook = pd.ExcelFile(source)

        for original_sheet in workbook.sheet_names:

            df = pd.read_excel(
                workbook,
                sheet_name=original_sheet
            )

            sheet_counter += 1

            clean_name = "".join(
                character
                if character.isalnum()
                else "_"
                for character in original_sheet
            )

            sheet_name = (
                f"R{sheet_counter:03d}_{clean_name}"
            )[:31]

            # Excel worksheet limits.
            if (
                len(df) > 1048575
                or len(df.columns) > 16384
            ):

                remote_table_catalogue.append({
                    "source_workbook": filename,
                    "source_sheet": original_sheet,
                    "consolidated_sheet": None,
                    "rows": len(df),
                    "columns": len(df.columns),
                    "status": "Original_file_retained"
                })

                continue

            df.to_excel(
                writer,
                sheet_name=sheet_name,
                index=False
            )

            remote_table_catalogue.append({
                "source_workbook": filename,
                "source_sheet": original_sheet,
                "consolidated_sheet": sheet_name,
                "rows": len(df),
                "columns": len(df.columns),
                "status": "Consolidated"
            })

    pd.DataFrame(remote_table_catalogue).to_excel(
        writer,
        sheet_name="TABLE_INDEX",
        index=False
    )

pd.DataFrame(remote_table_catalogue).to_csv(
    FOLDERS["metadata"]
    / "REMOTE_TABLE_CATALOGUE.csv",
    index=False
)

# ------------------------------------------------------------
# 8. Final editorial figure plan
# ------------------------------------------------------------

figure_plan_final = pd.DataFrame([

    {
        "figure": "Figure 1",
        "content": "Study area and forest plot locations",
        "placement": "Main",
        "status": "Final cartographic layout required"
    },
    {
        "figure": "Figure 2",
        "content": "Taxonomic diversity",
        "placement": "Main",
        "status": "Available"
    },
    {
        "figure": "Figure 3",
        "content": "Biomass distribution by diameter classes",
        "placement": "Main",
        "status": "Available"
    },
    {
        "figure": "Figure 4",
        "content": "Height-diameter allometry",
        "placement": "Main",
        "status": "Available"
    },
    {
        "figure": "Figure 5",
        "content": "Height architecture counterfactual",
        "placement": "Main",
        "status": "Available"
    },
    {
        "figure": "Figure 6",
        "content": "Taxonomic-biomass decoupling",
        "placement": "Main",
        "status": "Available"
    },
    {
        "figure": "Figure 7",
        "content": "Integrated biomass decomposition",
        "placement": "Main",
        "status": "Available"
    },
    {
        "figure": "Figure 8",
        "content": "Integrated biomass uncertainty",
        "placement": "Main",
        "status": "Available"
    },
    {
        "figure": "Figure 9",
        "content": "Historical spectral trajectories",
        "placement": "Main candidate",
        "status": "Select final Landsat figure"
    },
    {
        "figure": "Figure 10",
        "content": "Scale-dependent Sentinel-2 contrasts",
        "placement": "Main candidate",
        "status": "Select final Sentinel-2 figure"
    },
    {
        "figure": "Supplementary",
        "content": "Radiometric QA and sensor sensitivity",
        "placement": "Supplementary",
        "status": "Available"
    },
    {
        "figure": "Supplementary",
        "content": "Historical disturbance spatial diagnostics",
        "placement": "Supplementary",
        "status": "Available"
    },
    {
        "figure": "Supplementary",
        "content": "Hydroclimatic context",
        "placement": "Supplementary",
        "status": "Available"
    },
    {
        "figure": "Supplementary",
        "content": "GEDI canopy-height distributions",
        "placement": "Supplementary",
        "status": "Available; descriptive only"
    },
    {
        "figure": "Supplementary",
        "content": "Topographic context",
        "placement": "Supplementary",
        "status": "Available"
    }

])

figure_plan_final.to_csv(
    FOLDERS["metadata"]
    / "FINAL_EDITORIAL_FIGURE_PLAN.csv",
    index=False
)

# ------------------------------------------------------------
# 9. Consolidated scientific results
# Values already obtained in completed analyses.
# No new inferential calculations.
# ------------------------------------------------------------

key_results = pd.DataFrame([

    ["Inventory", "Measured structural trees",
     545, 464, "trees", "Original structural inventory"],

    ["Inventory", "Strict biomass inventory",
     541, 461, "trees", "Final biomass inventory"],

    ["Diversity", "Operational taxonomic richness",
     91, 111, "taxa", "Harmonized floristic inventory"],

    ["Diversity", "Coverage-standardized Hill q0",
     73.0568, 110.3110, "effective taxa",
     "Common coverage 0.898"],

    ["Diversity", "Coverage-standardized Hill q1",
     25.5663, 65.5537, "effective taxa",
     "Common coverage 0.898"],

    ["Diversity", "Coverage-standardized Hill q2",
     8.5519, 41.8154, "effective taxa",
     "Common coverage 0.898"],

    ["Structure", "Mean tree height",
     27.9667, 22.8915, "m",
     "Strict biomass inventory"],

    ["Biomass", "Aboveground biomass",
     355.2266, 234.3626, "Mg ha-1",
     "Chave et al. 2014"],

    ["Wood density", "Abundance-weighted CWM",
     0.5637, 0.5397, "g cm-3",
     "Assigned wood density"],

    ["Wood density", "Basal-area-weighted CWM",
     0.5620, 0.5059, "g cm-3",
     "Assigned wood density"],

    ["GEDI", "Mean RH98",
     24.1858, 27.7384, "m",
     "Fragment scale; 2019-2021"],

    ["GEDI", "Median RH98",
     24.69, 27.55, "m",
     "Fragment scale; descriptive"],

    ["GEDI", "Valid pixel-month records",
     12, 31, "records",
     "Fragment scale"],

    ["Topography", "Plot mean elevation",
     694.1045, 734.7096, "m a.s.l.",
     "NASADEM"],

    ["Topography", "Plot mean slope",
     12.5346, 20.1274, "degrees",
     "NASADEM"],

    ["Geometry", "Analyzed fragment area",
     18.9090, 39.6686, "ha",
     "Validated spatial polygons"]

], columns=[
    "block",
    "metric",
    "late_secondary",
    "mature",
    "unit",
    "context"
])

key_results.to_csv(
    FOLDERS["results"]
    / "SATIPO_KEY_RESULTS.csv",
    index=False
)

# Biomass decomposition and uncertainty summary.

mechanism_results = pd.DataFrame([

    ["Full biomass difference",
     120.8641, "Mg ha-1"],

    ["Secondary/mature biomass ratio",
     1.515714, "ratio"],

    ["Common-support architecture contribution",
     49.9457, "Mg ha-1"],

    ["Common-support architecture fraction",
     30.9827, "%"],

    ["Shapley abundance contribution",
     46.5117, "Mg ha-1"],

    ["Shapley diameter contribution",
     -6.8236, "Mg ha-1"],

    ["Shapley height contribution",
     52.0807, "Mg ha-1"],

    ["Shapley wood-density contribution",
     29.0953, "Mg ha-1"],

    ["Primary MC median biomass difference",
     120.73093, "Mg ha-1"],

    ["Primary MC lower 95% interval",
     87.200206, "Mg ha-1"],

    ["Primary MC upper 95% interval",
     156.113633, "Mg ha-1"],

    ["Stress-test MC median difference",
     118.250236, "Mg ha-1"],

    ["Stress-test MC lower 95% interval",
     -49.783379, "Mg ha-1"],

    ["Stress-test MC upper 95% interval",
     312.000734, "Mg ha-1"],

    ["Stress-test positive simulated differences",
     91.85, "%"]

], columns=["metric", "value", "unit"])

mechanism_results.to_csv(
    FOLDERS["results"]
    / "SATIPO_MECHANISM_RESULTS.csv",
    index=False
)

with pd.ExcelWriter(
    FOLDERS["results"]
    / "SATIPO_FINAL_KEY_RESULTS.xlsx"
) as writer:

    key_results.to_excel(
        writer,
        sheet_name="Forest_comparison",
        index=False
    )

    mechanism_results.to_excel(
        writer,
        sheet_name="Biomass_mechanisms",
        index=False
    )

# ------------------------------------------------------------
# 10. Scientific interpretation safeguards
# ------------------------------------------------------------

limitations = [

    {
        "issue": "Forest-type replication",
        "status": "Mandatory limitation",
        "instruction":
            "One inventory plot per forest type. "
            "Subplots, trees and pixels are nested "
            "observations, not independent forest replicates."
    },

    {
        "issue": "Disturbance causality",
        "status": "Mandatory limitation",
        "instruction":
            "Historical disturbance differences cannot "
            "be isolated from site characteristics."
    },

    {
        "issue": "Historical 1991 anomaly",
        "status": "Cautious interpretation",
        "instruction":
            "Shared spectral decline and rebound, "
            "with insufficient support under stricter "
            "scene-screening scenarios. "
            "Do not identify a confirmed disturbance cause."
    },

    {
        "issue": "Historical 2021 anomaly",
        "status": "Cautious interpretation",
        "instruction":
            "Moderate spatial spectral decline; "
            "no definitive disturbance attribution."
    },

    {
        "issue": "Hydroclimatic associations",
        "status": "Exploratory",
        "instruction":
            "Regional climate correlations are "
            "descriptive and do not establish attribution."
    },

    {
        "issue": "GEDI validation",
        "status": "Mandatory limitation",
        "instruction":
            "No direct plot-scale validation. "
            "GEDI RH98 and individual tree heights "
            "measure different structural properties."
    },

    {
        "issue": "Topographic confounding",
        "status": "Mandatory limitation",
        "instruction":
            "Elevation and slope differ between plots. "
            "Their effects cannot be separated from "
            "forest history using the current design."
    },

    {
        "issue": "Biomass decomposition",
        "status": "Mandatory limitation",
        "instruction":
            "Algebraic and counterfactual contributions "
            "are not identified causal effects."
    },

    {
        "issue": "Biomass uncertainty",
        "status": "Mandatory limitation",
        "instruction":
            "The high-error correlated-residual stress "
            "scenario includes zero in its 95% interval. "
            "Avoid claiming robustness under all scenarios."
    },

    {
        "issue": "Fragment area discrepancy",
        "status": "Author verification required",
        "instruction":
            "Historical areas: 23 and 35 ha. "
            "Analyzed polygons: 18.9090 and 39.6686 ha. "
            "Confirm internal forest boundary and "
            "document the distinction."
    },

    {
        "issue": "Taxonomic identification",
        "status": "Mandatory limitation",
        "instruction":
            "Preserve unresolved taxonomic labels and "
            "report sensitivity to identification resolution."
    },

    {
        "issue": "Recovery trajectories",
        "status": "Mandatory limitation",
        "instruction":
            "A single field census does not establish "
            "temporal biomass or biodiversity recovery rates."
    }

]

pd.DataFrame(limitations).to_csv(
    FOLDERS["review"]
    / "SCIENTIFIC_INTERPRETATION_SAFEGUARDS.csv",
    index=False
)

# ------------------------------------------------------------
# 11. Final archive quality control
# ------------------------------------------------------------

archive_manifest = pd.DataFrame(copied_records)

assert len(archive_manifest) > 0
assert archive_manifest["verified"].all()

archive_manifest.to_csv(
    FOLDERS["metadata"]
    / "FINAL_FILE_MANIFEST.csv",
    index=False
)

missing_df = pd.DataFrame({
    "missing_required_file": sorted(
        set(missing_required)
    )
})

missing_df.to_csv(
    FOLDERS["review"]
    / "MISSING_REQUIRED_FILES.csv",
    index=False
)

# All current files in the editorial package.

package_files = [
    path for path in FINAL.rglob("*")
    if path.is_file()
]

package_size_mb = sum(
    path.stat().st_size
    for path in package_files
) / 1e6

final_report = {

    "project": "SATIPO_FOREST_PROJECT",

    "scientific_analysis_last_cell": 98,

    "archive_cell": 99,

    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "scientific_data_extraction_completed": True,

    "verified_copied_files": len(archive_manifest),

    "package_file_count_before_final_report":
        len(package_files),

    "package_size_MB_before_final_report":
        package_size_mb,

    "remote_workbooks": len(remote_workbooks),

    "remote_figure_files":
        len(remote_figure_records),

    "remote_table_sheets":
        len(remote_table_catalogue),

    "integrated_mc_draws": 180000,

    "integrated_mc_scenarios": 18,

    "missing_required_files":
        sorted(set(missing_required)),

    "editorial_archive_complete":
        len(missing_required) == 0,

    "pending_editorial_tasks": [
        "Verify historical versus analyzed fragment areas",
        "Finalize study-area map",
        "Select final main-text figures",
        "Prepare Methods, Results and Discussion",
        "Verify references and data provenance",
        "Confirm authorship and contributions"
    ],

    "new_scientific_analyses_required": False

}

report_path = (
    FOLDERS["metadata"]
    / "FINAL_PROJECT_REPORT.json"
)

with open(
    report_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_report,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 12. Create portable editorial ZIP
# ------------------------------------------------------------

archive_path = (
    ROOT / "SATIPO_FINAL_MANUSCRIPT_PACKAGE.zip"
)

temporary_zip = archive_path.with_suffix(".tmp")

with zipfile.ZipFile(
    temporary_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as archive:

    for source in sorted(FINAL.rglob("*")):

        if source.is_file():

            archive.write(
                source,
                arcname=str(
                    source.relative_to(FINAL)
                )
            )

with zipfile.ZipFile(temporary_zip, "r") as archive:

    assert archive.testzip() is None

    archived_files = len(archive.namelist())

temporary_zip.replace(archive_path)

# ------------------------------------------------------------
# 13. Publish final archive pointer
# ------------------------------------------------------------

final_pointer = {
    "project": "SATIPO_FOREST_PROJECT",
    "last_scientific_cell": 98,
    "archive_cell": 99,
    "final_package": str(FINAL),
    "zip_archive": str(archive_path),
    "final_report": str(report_path),
    "editorial_archive_complete":
        len(missing_required) == 0,
    "missing_required_count":
        len(set(missing_required)),
    "previous_scientific_checkpoint":
        str(checkpoint_pointer)
}

pointer_path = (
    ROOT / "06_CHECKPOINTS"
    / "LATEST_MANUSCRIPT_ARCHIVE.json"
)

temporary_pointer = pointer_path.with_suffix(".tmp")

with open(
    temporary_pointer,
    "w",
    encoding="utf-8"
) as f:

    json.dump(final_pointer, f, indent=2)

temporary_pointer.replace(pointer_path)

# ------------------------------------------------------------
# 14. Final report
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("SATIPO — FINAL PROJECT ARCHIVE")
print("=" * 75)

print("Scientific processing: COMPLETED")
print("Verified copied files:", len(archive_manifest))
print("Archived files:", archived_files)

print(
    "ZIP size (MB):",
    round(archive_path.stat().st_size / 1e6, 2)
)

print("Remote workbooks:", len(remote_workbooks))
print("Remote table sheets:", len(remote_table_catalogue))
print("Integrated Monte Carlo draws: 180000")
print("Integrated uncertainty scenarios: 18")

print(
    "Missing required files:",
    len(set(missing_required))
)

if missing_required:

    print("\nFILES REQUIRING REVIEW:")

    for filename in sorted(set(missing_required)):
        print("-", filename)

else:

    print("\nEDITORIAL ARCHIVE: VERIFIED")

print("\nFINAL PACKAGE:", FINAL)
print("ZIP ARCHIVE:", archive_path)
print("FINAL REPORT:", report_path)
print("ARCHIVE POINTER:", pointer_path)

print("\n" + "=" * 75)
print("CELL 99 COMPLETED")
print("END OF SCIENTIFIC PROCESSING")
print("NEXT PHASE: MANUSCRIPT WRITING")
print("=" * 75)


In [ ]:

# ============================================================
# CELL 100 — PUBLICATION-QUALITY THEMATIC CARTOGRAPHY
# SATIPO FOREST PROJECT
#
# A. Sentinel-2 spectral atlas
# B. NASADEM topographic atlas
# C. GEDI canopy-height cartography
#
# NO NEW STATISTICAL INFERENCE
# ============================================================

import ee
import os
import json
import shutil
import warnings
import requests
import zipfile
import io
import re

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.colors as mcolors

from pathlib import Path
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.colors import LightSource
from matplotlib.ticker import MaxNLocator
from mpl_toolkits.axes_grid1 import make_axes_locatable
from rasterio.io import MemoryFile
from rasterio.plot import plotting_extent
from rasterio.features import geometry_mask
from shapely.geometry import mapping
from PIL import Image

warnings.filterwarnings("ignore")

# ============================================================
# 1. PROJECT CONFIGURATION
# ============================================================

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

REMOTE = ROOT / "04_REMOTE_SENSING"

CARTO = REMOTE / "10_FINAL_CARTOGRAPHY"

CARTO_DIRS = {
    "rasters": CARTO / "01_RASTERS",
    "figures": CARTO / "02_FIGURES",
    "tables": CARTO / "03_TABLES",
    "metadata": CARTO / "04_METADATA"
}

for folder in CARTO_DIRS.values():
    folder.mkdir(parents=True, exist_ok=True)

FINAL_PACKAGE = ROOT / "08_FINAL_MANUSCRIPT_PACKAGE"

CRS = "EPSG:32718"

SECONDARY_COLOR = "#DB9140"
MATURE_COLOR = "#24634F"

DARK = "#24332E"
LIGHT = "#F3F1EC"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
    "axes.edgecolor": "#555555",
    "axes.linewidth": 0.7,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# Verify existing Earth Engine session.

try:
    ee.Number(1).getInfo()
    print("Earth Engine: CONNECTED")

except Exception:
    raise RuntimeError(
        "Initialize Earth Engine using the same "
        "project employed in Cells 86–98."
    )

# ============================================================
# 2. RECOVER VALIDATED SPATIAL GEOMETRIES
# ============================================================

REQUIRED_UNITS = {
    "FRAG_01",
    "FRAG_02",
    "P01",
    "P02",
    "FRAG_01_CORE30",
    "FRAG_02_CORE30"
}

geometry_candidates = []

# First recover existing GeoDataFrames.

for variable_name, obj in list(globals().items()):

    if isinstance(obj, gpd.GeoDataFrame):

        if "unit_id" in obj.columns:

            available_ids = set(
                obj["unit_id"].astype(str)
            )

            if REQUIRED_UNITS.issubset(available_ids):

                geometry_candidates.append(
                    (variable_name, obj.copy())
                )

# Fallback: recover saved spatial data.

if not geometry_candidates:

    vector_dirs = [
        REMOTE / "02_GEOMETRY_VALIDATION",
        REMOTE / "01_VECTOR_INPUTS"
    ]

    for directory in vector_dirs:

        if not directory.exists():
            continue

        for path in directory.rglob("*"):

            if path.suffix.lower() not in [
                ".gpkg", ".geojson", ".shp", ".json"
            ]:
                continue

            try:

                if path.suffix.lower() == ".gpkg":

                    layers = gpd.list_layers(path)

                    for layer in layers["name"]:

                        candidate = gpd.read_file(
                            path,
                            layer=layer
                        )

                        if (
                            "unit_id" in candidate.columns
                            and REQUIRED_UNITS.issubset(
                                set(candidate["unit_id"])
                            )
                        ):

                            geometry_candidates.append(
                                (
                                    f"{path.name}:{layer}",
                                    candidate
                                )
                            )

                else:

                    candidate = gpd.read_file(path)

                    if (
                        "unit_id" in candidate.columns
                        and REQUIRED_UNITS.issubset(
                            set(candidate["unit_id"])
                        )
                    ):

                        geometry_candidates.append(
                            (str(path), candidate)
                        )

            except Exception:
                continue

assert geometry_candidates, (
    "Validated spatial geometries not found."
)

geometry_source, carto_units = geometry_candidates[0]

carto_units = carto_units.loc[
    carto_units["unit_id"].isin(REQUIRED_UNITS)
].copy()

carto_units = carto_units.drop_duplicates("unit_id")

assert len(carto_units) == 6
assert carto_units.crs is not None
assert carto_units.geometry.is_valid.all()

carto_units = carto_units.to_crs(CRS)

fragments = carto_units.loc[
    carto_units["unit_id"].isin(
        ["FRAG_01", "FRAG_02"]
    )
].copy()

plots = carto_units.loc[
    carto_units["unit_id"].isin(["P01", "P02"])
].copy()

secondary = fragments.loc[
    fragments["unit_id"] == "FRAG_01"
]

mature = fragments.loc[
    fragments["unit_id"] == "FRAG_02"
]

print("Geometry source:", geometry_source)
print("Spatial units:", len(carto_units))

# Map extent: forest polygons + contextual buffer.

xmin, ymin, xmax, ymax = fragments.total_bounds

BUFFER = 180

MAP_BOUNDS = (
    xmin - BUFFER,
    ymin - BUFFER,
    xmax + BUFFER,
    ymax + BUFFER
)

xmin, ymin, xmax, ymax = MAP_BOUNDS

# Use a rectangular projected grid.

region = ee.Geometry.Rectangle(
    [xmin, ymin, xmax, ymax],
    proj=CRS,
    geodesic=False
)

print("Map CRS:", CRS)
print("Map bounds:", MAP_BOUNDS)

# ============================================================
# 3. CARTOGRAPHIC UTILITIES
# ============================================================

def download_ee_raster(
    image,
    bands,
    filename,
    scale=10
):

    output_path = (
        CARTO_DIRS["rasters"] / filename
    )

    # Reuse previously downloaded rasters.

    if output_path.exists():

        with MemoryFile(
            output_path.read_bytes()
        ) as mem:

            with mem.open() as src:

                data = src.read(masked=True)
                transform = src.transform
                extent = plotting_extent(src)
                raster_crs = src.crs

        return (
            data,
            transform,
            extent,
            raster_crs
        )

    image = image.select(bands).toFloat()

    url = image.getDownloadURL({
        "name": output_path.stem,
        "bands": bands,
        "region": region,
        "scale": scale,
        "crs": CRS,
        "format": "GEO_TIFF"
    })

    response = requests.get(
        url,
        timeout=240
    )

    response.raise_for_status()

    content = response.content

    # Handle zipped Earth Engine responses.

    if content[:2] == b"PK":

        with zipfile.ZipFile(
            io.BytesIO(content)
        ) as archive:

            tif_files = [
                name for name in archive.namelist()
                if name.lower().endswith(".tif")
            ]

            assert len(tif_files) == 1, (
                "Expected a multiband GeoTIFF."
            )

            content = archive.read(tif_files[0])

    output_path.write_bytes(content)

    with MemoryFile(content) as mem:

        with mem.open() as src:

            data = src.read(masked=True)
            transform = src.transform
            extent = plotting_extent(src)
            raster_crs = src.crs

    print(
        "Raster downloaded:",
        output_path.name,
        data.shape
    )

    return data, transform, extent, raster_crs


def float_array(array):

    return np.ma.asarray(
        array,
        dtype=float
    ).filled(np.nan)


def robust_limits(array, lower=2, upper=98):

    values = float_array(array)

    values = values[np.isfinite(values)]

    assert len(values) > 0, (
        "Raster contains no valid values."
    )

    vmin, vmax = np.percentile(
        values,
        [lower, upper]
    )

    if np.isclose(vmin, vmax):
        vmax = vmin + 1e-6

    return float(vmin), float(vmax)


def add_north_arrow(ax):

    ax.annotate(
        "N",
        xy=(0.92, 0.95),
        xytext=(0.92, 0.82),
        xycoords="axes fraction",
        ha="center",
        va="center",
        fontsize=10,
        fontweight="bold",
        color=DARK,
        arrowprops={
            "arrowstyle": "-|>",
            "color": DARK,
            "lw": 1.7,
            "mutation_scale": 15
        },
        zorder=30
    )


def add_scale_bar(ax, length=200):

    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()

    x = x0 + 0.07 * (x1 - x0)
    y = y0 + 0.06 * (y1 - y0)

    height = 0.012 * (y1 - y0)

    for i in range(2):

        ax.add_patch(
            plt.Rectangle(
                (x + i * length / 2, y),
                length / 2,
                height,
                facecolor=(
                    DARK if i == 0 else "white"
                ),
                edgecolor=DARK,
                linewidth=0.7,
                zorder=25
            )
        )

    ax.text(
        x + length / 2,
        y + height * 1.7,
        f"{length} m",
        ha="center",
        va="bottom",
        fontsize=8,
        color=DARK,
        fontweight="bold",
        path_effects=[
            pe.withStroke(
                linewidth=2.5,
                foreground="white"
            )
        ],
        zorder=26
    )


def add_boundaries(ax, labels=True):

    # White halo enhances visibility over dark rasters.

    fragments.boundary.plot(
        ax=ax,
        color="white",
        linewidth=3.1,
        zorder=12
    )

    secondary.boundary.plot(
        ax=ax,
        color=SECONDARY_COLOR,
        linewidth=1.9,
        zorder=13
    )

    mature.boundary.plot(
        ax=ax,
        color=MATURE_COLOR,
        linewidth=1.9,
        zorder=13
    )

    plots.boundary.plot(
        ax=ax,
        color="black",
        linewidth=2.5,
        zorder=14
    )

    plots.boundary.plot(
        ax=ax,
        color="#FFE85B",
        linewidth=1.5,
        linestyle="--",
        zorder=15
    )

    if labels:

        for _, row in fragments.iterrows():

            point = row.geometry.representative_point()

            name = (
                "Secondary"
                if row["unit_id"] == "FRAG_01"
                else "Mature"
            )

            ax.text(
                point.x,
                point.y,
                name,
                ha="center",
                va="center",
                fontsize=8,
                fontweight="bold",
                color=DARK,
                bbox={
                    "boxstyle": "round,pad=0.25",
                    "facecolor": "white",
                    "edgecolor": "none",
                    "alpha": 0.85
                },
                zorder=20
            )

        for _, row in plots.iterrows():

            point = row.geometry.centroid

            ax.annotate(
                row["unit_id"],
                xy=(point.x, point.y),
                xytext=(10, -13),
                textcoords="offset points",
                fontsize=7,
                fontweight="bold",
                color="white",
                path_effects=[
                    pe.withStroke(
                        linewidth=2.4,
                        foreground="black"
                    )
                ],
                zorder=21
            )


def format_map(ax, coordinates=False):

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_aspect("equal")

    ax.set_facecolor(LIGHT)

    if coordinates:

        ax.xaxis.set_major_locator(
            MaxNLocator(3)
        )

        ax.yaxis.set_major_locator(
            MaxNLocator(4)
        )

        ax.ticklabel_format(
            style="plain",
            useOffset=False
        )

        ax.set_xlabel("Easting (m)")
        ax.set_ylabel("Northing (m)")

    else:

        ax.set_xticks([])
        ax.set_yticks([])

    for spine in ax.spines.values():

        spine.set_color("#777777")
        spine.set_linewidth(0.65)


def add_colorbar(
    fig,
    ax,
    image,
    label,
    ticks=None
):

    divider = make_axes_locatable(ax)

    cax = divider.append_axes(
        "right",
        size="4%",
        pad=0.10
    )

    cb = fig.colorbar(
        image,
        cax=cax,
        ticks=ticks,
        extend="both"
    )

    cb.set_label(
        label,
        fontsize=8
    )

    cb.ax.tick_params(
        labelsize=8,
        length=2
    )

    cb.outline.set_linewidth(0.5)

    return cb


def forest_legend():

    return [

        Line2D(
            [0], [0],
            color=SECONDARY_COLOR,
            linewidth=2.6,
            label="Late-secondary forest"
        ),

        Line2D(
            [0], [0],
            color=MATURE_COLOR,
            linewidth=2.6,
            label="Mature forest"
        ),

        Line2D(
            [0], [0],
            color="#D4AD00",
            linestyle="--",
            linewidth=2,
            label="Permanent field plots"
        )
    ]


exported_figures = []


def export_figure(fig, stem):

    base = CARTO_DIRS["figures"] / stem

    for extension in ["png", "pdf", "svg"]:

        path = base.with_suffix(
            f".{extension}"
        )

        fig.savefig(
            path,
            dpi=600,
            bbox_inches="tight",
            pad_inches=0.12
        )

        exported_figures.append(str(path))

    # LZW-compressed TIFF for journal submission.

    png_path = base.with_suffix(".png")
    tiff_path = base.with_suffix(".tiff")

    with Image.open(png_path) as image:

        image.convert("RGB").save(
            tiff_path,
            compression="tiff_lzw",
            dpi=(600, 600)
        )

    exported_figures.append(str(tiff_path))

    print("Figure exported:", stem)


# ============================================================
# 4. SENTINEL-2 CARTOGRAPHIC COMPOSITE
# ============================================================

# Match the field census year.
# Seasonal median: May–September 2019.
#
# This is a cartographic reconstruction.
# It does not replace the statistical dataset from Cell 96.

S2_COLLECTION = (
    "COPERNICUS/S2_SR_HARMONIZED"
)

S2_BANDS = [
    "B2", "B3", "B4",
    "B5", "B8", "B8A",
    "B11", "B12"
]

s2_collection = (
    ee.ImageCollection(S2_COLLECTION)
    .filterBounds(region)
    .filterDate(
        "2019-05-01",
        "2019-10-01"
    )
    .filter(
        ee.Filter.lt(
            "CLOUDY_PIXEL_PERCENTAGE",
            80
        )
    )
)

print(
    "Sentinel-2 candidate acquisitions:",
    s2_collection.size().getInfo()
)


def prepare_s2_cartography(image):

    scl = image.select("SCL")

    # Retain vegetation, bare soil and water.
    # Exclude cloud, shadow, cirrus, snow,
    # saturated pixels and unclassified pixels.

    clear = (
        scl.eq(4)
        .Or(scl.eq(5))
        .Or(scl.eq(6))
    )

    reflectance = (
        image.select(S2_BANDS)
        .multiply(0.0001)
        .updateMask(clear)
    )

    physical = (
        reflectance.gte(0)
        .And(reflectance.lte(1))
        .reduce(ee.Reducer.min())
    )

    reflectance = reflectance.updateMask(
        physical
    )

    blue = reflectance.select("B2")
    green = reflectance.select("B3")
    red = reflectance.select("B4")
    rededge = reflectance.select("B5")
    nir = reflectance.select("B8")
    nir_narrow = reflectance.select("B8A")
    swir1 = reflectance.select("B11")
    swir2 = reflectance.select("B12")

    def normalized(a, b, name):

        denominator = a.add(b)

        return (
            a.subtract(b)
            .divide(denominator)
            .updateMask(
                denominator.gt(1e-6)
            )
            .rename(name)
        )

    ndvi = normalized(
        nir, red, "NDVI"
    )

    gndvi = normalized(
        nir, green, "GNDVI"
    )

    ndre = normalized(
        nir_narrow, rededge, "NDRE"
    )

    ndmi = normalized(
        nir, swir1, "NDMI"
    )

    nbr = normalized(
        nir, swir2, "NBR"
    )

    nbr2 = normalized(
        swir1, swir2, "NBR2"
    )

    evi_denominator = (
        nir
        .add(red.multiply(6))
        .subtract(blue.multiply(7.5))
        .add(1)
    )

    evi = (
        nir.subtract(red)
        .multiply(2.5)
        .divide(evi_denominator)
        .updateMask(
            evi_denominator.gt(1e-6)
        )
        .rename("EVI")
    )

    return (
        reflectance
        .addBands([
            ndvi, evi, gndvi,
            ndre, ndmi, nbr, nbr2
        ])
        .copyProperties(
            image,
            ["system:time_start"]
        )
    )


s2_processed = s2_collection.map(
    prepare_s2_cartography
)

SPECTRAL_INDICES = [
    "NDVI", "EVI", "NDRE",
    "NDMI", "NBR", "NBR2"
]

s2_median = s2_processed.median()

# Require at least three valid observations
# per index in the cartographic composite.

s2_count = (
    s2_processed
    .select(SPECTRAL_INDICES)
    .count()
)

spectral_composite = (
    s2_median
    .select(SPECTRAL_INDICES)
    .updateMask(s2_count.gte(3))
)

# Common 20 m display grid avoids suggesting
# that red-edge and SWIR bands contain native
# 10 m spatial information.

spectral_data, spectral_transform, spectral_extent, spectral_crs = (
    download_ee_raster(
        spectral_composite,
        SPECTRAL_INDICES,
        "SATIPO_S2_SPECTRAL_ATLAS_2019_20M.tif",
        scale=20
    )
)

rgb_data, rgb_transform, rgb_extent, rgb_crs = (
    download_ee_raster(
        s2_median,
        ["B4", "B3", "B2"],
        "SATIPO_S2_RGB_2019_10M.tif",
        scale=10
    )
)

# ============================================================
# 5. TRUE-COLOR BACKGROUND
# ============================================================

rgb_array = np.moveaxis(
    float_array(rgb_data),
    0,
    -1
)

rgb_display = np.zeros_like(
    rgb_array,
    dtype=float
)

for band in range(3):

    vmin, vmax = robust_limits(
        rgb_array[:, :, band],
        2,
        98
    )

    rgb_display[:, :, band] = np.clip(
        (
            rgb_array[:, :, band] - vmin
        ) / (vmax - vmin),
        0,
        1
    )

# Gentle gamma enhancement.

rgb_display = np.power(
    rgb_display,
    0.85
)

rgb_display = np.nan_to_num(
    rgb_display,
    nan=0.94
)

# ============================================================
# 6. FIGURE A — CANOPY SPECTRAL ATLAS
# ============================================================

# Sequential, visually distinct palettes.

SPECTRAL_STYLES = {

    "NDVI": {
        "title": "Vegetation greenness",
        "cmap": "YlGn"
    },

    "EVI": {
        "title": "Enhanced vegetation response",
        "cmap": "viridis"
    },

    "NDRE": {
        "title": "Red-edge response",
        "cmap": "YlGnBu"
    },

    "NDMI": {
        "title": "Canopy moisture response",
        "cmap": "GnBu"
    },

    "NBR": {
        "title": "Near-infrared–SWIR contrast",
        "cmap": "BrBG"
    },

    "NBR2": {
        "title": "SWIR spectral contrast",
        "cmap": "PuBuGn"
    }
}

# Compute color limits from the actual forests,
# not from unrelated surrounding land cover.

forest_mask = geometry_mask(
    [
        mapping(geometry)
        for geometry in fragments.geometry
    ],
    out_shape=spectral_data.shape[1:],
    transform=spectral_transform,
    invert=True
)

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15.8, 13.0)
)

fig.subplots_adjust(
    left=0.055,
    right=0.96,
    bottom=0.095,
    top=0.87,
    wspace=0.32,
    hspace=0.24
)

fig.suptitle(
    "Spectral architecture of contrasting Amazonian forests",
    fontsize=21,
    fontweight="bold",
    color=DARK,
    y=0.975
)

fig.text(
    0.5,
    0.935,
    "Sentinel-2 canopy spectral atlas | Satipo, Peru | 2019",
    ha="center",
    fontsize=12,
    color="#596761"
)

spectral_map_metadata = []

for i, index in enumerate(SPECTRAL_INDICES):

    ax = axes.flat[i]

    values = float_array(
        spectral_data[i]
    )

    forest_values = values[forest_mask]

    vmin, vmax = robust_limits(
        forest_values,
        2,
        98
    )

    cmap = plt.get_cmap(
        SPECTRAL_STYLES[index]["cmap"]
    ).copy()

    cmap.set_bad("#E5E5E5")

    # Background image outside the study forests.

    ax.imshow(
        rgb_display,
        extent=rgb_extent,
        origin="upper",
        alpha=0.42,
        interpolation="nearest"
    )

    # Thematic data within the study forests.

    mapped_values = np.where(
        forest_mask,
        values,
        np.nan
    )

    im = ax.imshow(
        mapped_values,
        extent=spectral_extent,
        origin="upper",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        interpolation="nearest",
        zorder=5
    )

    add_boundaries(ax)
    format_map(ax)

    ax.set_title(
        f"({chr(97+i)}) {index}",
        loc="left",
        pad=10,
        fontsize=13
    )

    ax.text(
        0.5,
        1.012,
        SPECTRAL_STYLES[index]["title"],
        transform=ax.transAxes,
        ha="center",
        va="bottom",
        fontsize=8,
        color="#65716C"
    )

    add_colorbar(
        fig,
        ax,
        im,
        "Index value"
    )

    if i == 0:

        add_north_arrow(ax)
        add_scale_bar(ax)

    spectral_map_metadata.append({
        "index": index,
        "year": 2019,
        "season": "May–September",
        "display_resolution_m": 20,
        "color_min_p02": vmin,
        "color_max_p98": vmax,
        "valid_forest_pixels": int(
            np.isfinite(forest_values).sum()
        ),
        "color_limits":
            "Pooled forest pixels; 2nd–98th percentiles"
    })

fig.legend(
    handles=forest_legend(),
    loc="lower center",
    bbox_to_anchor=(0.5, 0.045),
    ncol=3,
    frameon=False,
    fontsize=10
)

fig.text(
    0.5,
    0.018,
    "Seasonal median composites | "
    "Minimum three valid observations | "
    "WGS 84 / UTM zone 18S | "
    "20 m display grid",
    ha="center",
    fontsize=9,
    color="#68746D"
)

export_figure(
    fig,
    "FIG_CARTO_01_SENTINEL2_SPECTRAL_ATLAS"
)

plt.show()
plt.close(fig)

pd.DataFrame(
    spectral_map_metadata
).to_csv(
    CARTO_DIRS["tables"]
    / "SPECTRAL_ATLAS_DISPLAY_METADATA.csv",
    index=False
)

# ============================================================
# 7. NASADEM TERRAIN PRODUCTS
# ============================================================

dem = (
    ee.Image("NASA/NASADEM_HGT/001")
    .select("elevation")
)

terrain = ee.Terrain.products(dem)

terrain_image = (
    dem.rename("Elevation")
    .addBands(
        terrain.select("slope").rename("Slope")
    )
    .addBands(
        terrain.select("aspect").rename("Aspect")
    )
    .addBands(
        terrain.select("hillshade").rename("Hillshade")
    )
)

terrain_data, terrain_transform, terrain_extent, terrain_crs = (
    download_ee_raster(
        terrain_image,
        [
            "Elevation",
            "Slope",
            "Aspect",
            "Hillshade"
        ],
        "SATIPO_NASADEM_TERRAIN_30M.tif",
        scale=30
    )
)

elevation = float_array(
    terrain_data[0]
)

slope = float_array(
    terrain_data[1]
)

aspect = float_array(
    terrain_data[2]
)

hillshade = float_array(
    terrain_data[3]
)

# ============================================================
# 8. FIGURE B — GEOMORPHOLOGICAL ATLAS
# ============================================================

# Terrain-inspired elevation palette.

elevation_cmap = (
    mcolors.LinearSegmentedColormap.from_list(
        "satipo_elevation",
        [
            "#163D36",
            "#387F63",
            "#8CBF88",
            "#E0D4A5",
            "#C99B68",
            "#8A5D43",
            "#F2E7D5"
        ],
        N=256
    )
)

slope_cmap = (
    mcolors.LinearSegmentedColormap.from_list(
        "satipo_slope",
        [
            "#F7F5E8",
            "#EAD18C",
            "#DB9140",
            "#BA5839",
            "#73313C",
            "#392B46"
        ],
        N=256
    )
)

terrain_styles = [
    (
        elevation,
        elevation_cmap,
        "Elevation (m a.s.l.)",
        "(a) Elevation"
    ),
    (
        slope,
        slope_cmap,
        "Slope (degrees)",
        "(b) Slope"
    )
]

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12.8, 9.5)
)

fig.subplots_adjust(
    left=0.06,
    right=0.92,
    bottom=0.11,
    top=0.84,
    wspace=0.30
)

fig.suptitle(
    "Topographic setting of the studied forests",
    fontsize=21,
    fontweight="bold",
    color=DARK,
    y=0.97
)

fig.text(
    0.5,
    0.92,
    "NASADEM-derived elevation and terrain slope",
    ha="center",
    fontsize=12,
    color="#596761"
)

for i, (
    values,
    cmap,
    label,
    panel_title
) in enumerate(terrain_styles):

    ax = axes[i]

    vmin, vmax = robust_limits(
        values,
        1,
        99
    )

    im = ax.imshow(
        values,
        extent=terrain_extent,
        origin="upper",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        interpolation="nearest"
    )

    # Subtle relief shading.

    ax.imshow(
        hillshade,
        extent=terrain_extent,
        origin="upper",
        cmap="gray",
        vmin=0,
        vmax=255,
        alpha=0.20,
        interpolation="nearest"
    )

    add_boundaries(ax)
    format_map(ax, coordinates=True)

    ax.set_title(
        panel_title,
        loc="left",
        pad=12,
        fontsize=13
    )

    add_colorbar(
        fig,
        ax,
        im,
        label
    )

    add_north_arrow(ax)
    add_scale_bar(ax)

fig.legend(
    handles=forest_legend(),
    loc="lower center",
    bbox_to_anchor=(0.5, 0.045),
    ncol=3,
    frameon=False,
    fontsize=10
)

fig.text(
    0.5,
    0.016,
    "NASADEM | 30 m display grid | "
    "WGS 84 / UTM zone 18S | "
    "Shaded relief for cartographic context",
    ha="center",
    fontsize=9,
    color="#68746D"
)

export_figure(
    fig,
    "FIG_CARTO_02_TOPOGRAPHIC_ATLAS"
)

plt.show()
plt.close(fig)

# ============================================================
# 9. RECOVER EXISTING GEDI RECORDS
# ============================================================

# Do not resample GEDI or create artificial footprints.
# Recover the previously extracted pixel-month records.

gedi_candidates = []


def find_column(df, aliases):

    lookup = {
        str(column).strip().lower(): column
        for column in df.columns
    }

    for alias in aliases:

        if alias.lower() in lookup:

            return lookup[alias.lower()]

    return None


def inspect_gedi_candidate(name, df):

    if not isinstance(df, pd.DataFrame):
        return

    unit_col = find_column(
        df,
        ["unit_id"]
    )

    height_col = find_column(
        df,
        ["rh98", "rh98_m", "RH98"]
    )

    lon_col = find_column(
        df,
        ["longitude", "lon", "lng", "long"]
    )

    lat_col = find_column(
        df,
        ["latitude", "lat"]
    )

    x_col = find_column(
        df,
        ["x_utm", "easting", "utm_x"]
    )

    y_col = find_column(
        df,
        ["y_utm", "northing", "utm_y"]
    )

    has_geometry = isinstance(
        df,
        gpd.GeoDataFrame
    )

    has_coordinates = (
        (lon_col is not None and lat_col is not None)
        or
        (x_col is not None and y_col is not None)
        or
        has_geometry
    )

    if (
        unit_col is not None
        and height_col is not None
        and has_coordinates
    ):

        subset = df.loc[
            df[unit_col].isin(
                ["FRAG_01", "FRAG_02"]
            )
        ].copy()

        if len(subset) > 0:

            gedi_candidates.append({
                "name": name,
                "data": subset,
                "unit": unit_col,
                "height": height_col,
                "lon": lon_col,
                "lat": lat_col,
                "x": x_col,
                "y": y_col,
                "has_geometry": has_geometry
            })


# Search current notebook objects.

for name, obj in list(globals().items()):

    if isinstance(obj, pd.DataFrame):

        inspect_gedi_candidate(
            name,
            obj
        )

# Search previously exported GEDI tables.

GEDI_DIR = REMOTE / "05_GEDI"

if GEDI_DIR.exists():

    for path in GEDI_DIR.rglob("*"):

        try:

            if path.suffix.lower() == ".csv":

                df = pd.read_csv(path)

                inspect_gedi_candidate(
                    str(path),
                    df
                )

            elif path.suffix.lower() == ".xlsx":

                workbook = pd.ExcelFile(path)

                for sheet in workbook.sheet_names:

                    df = pd.read_excel(
                        workbook,
                        sheet_name=sheet
                    )

                    inspect_gedi_candidate(
                        f"{path.name}:{sheet}",
                        df
                    )

            elif path.suffix.lower() in [
                ".geojson",
                ".gpkg",
                ".shp"
            ]:

                df = gpd.read_file(path)

                inspect_gedi_candidate(
                    str(path),
                    df
                )

        except Exception:
            continue

# Prioritize exact support from Cell 97:
# 12 secondary + 31 mature records.

valid_gedi_candidates = []

for candidate in gedi_candidates:

    counts = (
        candidate["data"]
        [candidate["unit"]]
        .value_counts()
    )

    if (
        counts.get("FRAG_01", 0) == 12
        and counts.get("FRAG_02", 0) == 31
    ):

        valid_gedi_candidates.append(
            candidate
        )

gedi_status = "NOT_EXPORTED"
gedi_source = None

# ============================================================
# 10. FIGURE C — GEDI CANOPY HEIGHT
# ============================================================

if valid_gedi_candidates:

    selected_gedi = valid_gedi_candidates[0]

    gedi_source = selected_gedi["name"]

    gedi_map = selected_gedi["data"].copy()

    gedi_map["unit_id"] = (
        gedi_map[selected_gedi["unit"]]
    )

    gedi_map["rh98_m"] = pd.to_numeric(
        gedi_map[selected_gedi["height"]],
        errors="coerce"
    )

    assert gedi_map["rh98_m"].notna().all()

    if selected_gedi["has_geometry"]:

        assert gedi_map.crs is not None

        gedi_map = gedi_map.to_crs(CRS)

    elif selected_gedi["lon"] is not None:

        gedi_map = gpd.GeoDataFrame(
            gedi_map,
            geometry=gpd.points_from_xy(
                gedi_map[selected_gedi["lon"]],
                gedi_map[selected_gedi["lat"]]
            ),
            crs="EPSG:4326"
        ).to_crs(CRS)

    else:

        gedi_map = gpd.GeoDataFrame(
            gedi_map,
            geometry=gpd.points_from_xy(
                gedi_map[selected_gedi["x"]],
                gedi_map[selected_gedi["y"]]
            ),
            crs=CRS
        )

    # Validate the original Cell 97 summaries.

    gedi_check = (
        gedi_map
        .groupby("unit_id")["rh98_m"]
        .agg(["count", "mean", "median"])
    )

    print("\nGEDI SOURCE VALIDATION")
    print(gedi_check)

    assert np.isclose(
        gedi_check.loc["FRAG_01", "median"],
        24.69,
        atol=0.02
    )

    assert np.isclose(
        gedi_check.loc["FRAG_02", "median"],
        27.55,
        atol=0.02
    )

    # Verify that coordinates belong to their
    # assigned forest, allowing a 30 m tolerance
    # for raster pixel-centre representation.

    spatial_checks = []

    for _, row in gedi_map.iterrows():

        forest_geometry = fragments.loc[
            fragments["unit_id"] == row["unit_id"],
            "geometry"
        ].iloc[0]

        spatial_checks.append(
            forest_geometry.buffer(30).covers(
                row.geometry
            )
        )

    assert all(spatial_checks), (
        "GEDI coordinates do not match "
        "their assigned forest polygons."
    )

    fig = plt.figure(
        figsize=(13.0, 9.2)
    )

    gs = fig.add_gridspec(
        1,
        2,
        width_ratios=[1.25, 0.85],
        left=0.06,
        right=0.95,
        bottom=0.13,
        top=0.84,
        wspace=0.35
    )

    ax_map = fig.add_subplot(gs[0, 0])
    ax_box = fig.add_subplot(gs[0, 1])

    fig.suptitle(
        "Spaceborne LiDAR observations of canopy height",
        fontsize=20,
        fontweight="bold",
        color=DARK,
        y=0.97
    )

    fig.text(
        0.5,
        0.925,
        "GEDI RH98 | Forest-fragment scale | 2019–2021",
        ha="center",
        fontsize=12,
        color="#596761"
    )

    ax_map.imshow(
        rgb_display,
        extent=rgb_extent,
        origin="upper",
        interpolation="nearest"
    )

    add_boundaries(
        ax_map,
        labels=False
    )

    gedi_cmap = (
        mcolors.LinearSegmentedColormap.from_list(
            "gedi_height",
            [
                "#443983",
                "#31688E",
                "#21918C",
                "#90D743",
                "#FDE725"
            ],
            N=256
        )
    )

    gedi_norm = mcolors.Normalize(
        vmin=10,
        vmax=45
    )

    points = ax_map.scatter(
        gedi_map.geometry.x,
        gedi_map.geometry.y,
        c=gedi_map["rh98_m"],
        cmap=gedi_cmap,
        norm=gedi_norm,
        s=62,
        edgecolors="white",
        linewidths=0.9,
        alpha=0.98,
        zorder=25
    )

    format_map(ax_map)

    add_north_arrow(ax_map)
    add_scale_bar(ax_map)

    ax_map.set_title(
        "(a) Spatial distribution of GEDI observations",
        loc="left",
        fontsize=12,
        pad=12
    )

    add_colorbar(
        fig,
        ax_map,
        points,
        "RH98 (m)",
        ticks=[10, 20, 30, 40]
    )

    # Right panel: distribution of the exact
    # records shown in the map.

    secondary_heights = (
        gedi_map.loc[
            gedi_map["unit_id"] == "FRAG_01",
            "rh98_m"
        ].to_numpy()
    )

    mature_heights = (
        gedi_map.loc[
            gedi_map["unit_id"] == "FRAG_02",
            "rh98_m"
        ].to_numpy()
    )

    height_groups = [
        secondary_heights,
        mature_heights
    ]

    colors = [
        SECONDARY_COLOR,
        MATURE_COLOR
    ]

    box = ax_box.boxplot(
        height_groups,
        positions=[1, 2],
        widths=0.47,
        patch_artist=True,
        showfliers=False,
        medianprops={
            "color": DARK,
            "linewidth": 2.0
        },
        whiskerprops={
            "color": DARK,
            "linewidth": 1.2
        },
        capprops={
            "color": DARK,
            "linewidth": 1.2
        }
    )

    for patch, color in zip(
        box["boxes"],
        colors
    ):

        patch.set_facecolor(color)
        patch.set_alpha(0.34)
        patch.set_edgecolor(color)
        patch.set_linewidth(1.5)

    rng = np.random.default_rng(20261115)

    for i, (
        values,
        color
    ) in enumerate(
        zip(height_groups, colors),
        start=1
    ):

        jitter = rng.uniform(
            -0.13,
            0.13,
            size=len(values)
        )

        ax_box.scatter(
            i + jitter,
            values,
            s=40,
            color=color,
            edgecolor="white",
            linewidth=0.55,
            alpha=0.82,
            zorder=5
        )

        ax_box.text(
            i,
            44.5,
            f"n = {len(values)}",
            ha="center",
            fontsize=10,
            fontweight="bold",
            color=color
        )

    ax_box.set_xticks(
        [1, 2],
        [
            "Late secondary",
            "Mature"
        ]
    )

    ax_box.set_ylabel(
        "GEDI RH98 (m)"
    )

    ax_box.set_ylim(
        8,
        47
    )

    ax_box.set_xlim(
        0.5,
        2.5
    )

    ax_box.set_title(
        "(b) Canopy-height distributions",
        loc="left",
        fontsize=12,
        pad=12
    )

    ax_box.grid(
        axis="y",
        color="#E2E6E3",
        linewidth=0.7
    )

    ax_box.set_axisbelow(True)

    ax_box.spines["top"].set_visible(False)
    ax_box.spines["right"].set_visible(False)

    fig.legend(
        handles=forest_legend(),
        loc="lower center",
        bbox_to_anchor=(0.5, 0.055),
        ncol=3,
        frameon=False,
        fontsize=10
    )

    fig.text(
        0.5,
        0.024,
        "GEDI L2A monthly raster observations | "
        "Points represent valid pixel-month records | "
        "Descriptive fragment-scale comparison; "
        "no spatial interpolation",
        ha="center",
        fontsize=9,
        color="#68746D"
    )

    export_figure(
        fig,
        "FIG_CARTO_03_GEDI_CANOPY_HEIGHT"
    )

    plt.show()
    plt.close(fig)

    gedi_map.to_file(
        CARTO_DIRS["rasters"]
        / "SATIPO_GEDI_CARTOGRAPHIC_POINTS.geojson",
        driver="GeoJSON"
    )

    gedi_check.to_csv(
        CARTO_DIRS["tables"]
        / "GEDI_CARTOGRAPHIC_VALIDATION.csv"
    )

    gedi_status = "EXPORTED_AND_VALIDATED"

else:

    print("\nGEDI CARTOGRAPHY NOT EXPORTED")

    print(
        "The existing 43 GEDI records with "
        "coordinates could not be recovered."
    )

    print(
        "Sentinel-2 and topographic figures "
        "remain successfully exported."
    )

    print(
        "Available GEDI candidate tables:",
        [
            candidate["name"]
            for candidate in gedi_candidates
        ]
    )

    print(
        "Do not fabricate coordinates or "
        "interpolate canopy heights."
    )

# ============================================================
# 11. CARTOGRAPHIC MANIFEST
# ============================================================

cartography_manifest = {

    "project": "SATIPO_FOREST_PROJECT",

    "cell": 100,

    "purpose":
        "Publication-quality thematic cartography",

    "new_statistical_inference": False,

    "geometry_source": geometry_source,

    "projection": CRS,

    "map_extent": list(MAP_BOUNDS),

    "sentinel2": {
        "collection": S2_COLLECTION,
        "year": 2019,
        "start_date": "2019-05-01",
        "end_date_exclusive": "2019-10-01",
        "composite": "Median of scene-level indices",
        "minimum_valid_observations": 3,
        "display_resolution_m": 20,
        "rgb_resolution_m": 10,
        "indices": SPECTRAL_INDICES,
        "purpose":
            "Cartographic reconstruction; "
            "not replacement of Cell 96 statistics"
    },

    "topography": {
        "source": "NASA/NASADEM_HGT/001",
        "display_resolution_m": 30,
        "variables": [
            "Elevation",
            "Slope",
            "Hillshade"
        ]
    },

    "gedi": {
        "status": gedi_status,
        "source": gedi_source,
        "expected_secondary_records": 12,
        "expected_mature_records": 31,
        "metric": "RH98",
        "interpolation": False
    },

    "figure_formats": [
        "PNG",
        "PDF",
        "SVG",
        "TIFF"
    ],

    "raster_export_dpi": 600,

    "exported_figures": exported_figures
}

manifest_path = (
    CARTO_DIRS["metadata"]
    / "FINAL_CARTOGRAPHY_MANIFEST.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        cartography_manifest,
        file,
        indent=2,
        ensure_ascii=False
    )

# ============================================================
# 12. COPY MAPS TO EDITORIAL PACKAGE
# ============================================================

editorial_carto = (
    FINAL_PACKAGE
    / "05_REMOTE_SENSING_RESULTS"
    / "THEMATIC_CARTOGRAPHY"
)

editorial_carto.mkdir(
    parents=True,
    exist_ok=True
)

for path in CARTO_DIRS["figures"].glob("*"):

    if path.is_file():

        shutil.copy2(
            path,
            editorial_carto / path.name
        )

shutil.copy2(
    manifest_path,
    editorial_carto / manifest_path.name
)

# Preserve the original Cell 99 archive.
# Create a separate cartographic ZIP.

carto_zip = (
    ROOT
    / "SATIPO_FINAL_THEMATIC_CARTOGRAPHY.zip"
)

with zipfile.ZipFile(
    carto_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as archive:

    for path in sorted(
        CARTO.rglob("*")
    ):

        if path.is_file():

            archive.write(
                path,
                arcname=str(
                    path.relative_to(CARTO)
                )
            )

with zipfile.ZipFile(
    carto_zip,
    "r"
) as archive:

    assert archive.testzip() is None

# ============================================================
# 13. FINAL REPORT
# ============================================================

print("\n" + "=" * 76)

print(
    "SATIPO — FINAL THEMATIC CARTOGRAPHY"
)

print("=" * 76)

print("Geometry validation: PASSED")
print("Sentinel-2 spectral atlas: EXPORTED")
print("Topographic atlas: EXPORTED")
print("GEDI canopy-height atlas:", gedi_status)

print(
    "Figure files:",
    len(exported_figures)
)

print(
    "Cartographic ZIP size (MB):",
    round(
        carto_zip.stat().st_size / 1e6,
        2
    )
)

print("\nOUTPUT DIRECTORY:")
print(CARTO)

print("\nFIGURES:")
print(CARTO_DIRS["figures"])

print("\nEDITORIAL COPY:")
print(editorial_carto)

print("\nCARTOGRAPHIC ZIP:")
print(carto_zip)

print("\nMANIFEST:")
print(manifest_path)

print("\n" + "=" * 76)
print("CELL 100 COMPLETED")
print("THEMATIC CARTOGRAPHY FINISHED")
print("NEXT PHASE: MANUSCRIPT WRITING")
print("=" * 76)


In [ ]:

# ============================================================
# CELL 101 — FINAL EDITORIAL SELECTION AND DOWNLOAD
# SATIPO FOREST PROJECT
# ============================================================

from pathlib import Path
import pandas as pd
import shutil
import zipfile
import hashlib
import json
from datetime import datetime
from IPython.display import display, FileLink
from google.colab import files

# ------------------------------------------------------------
# 1. PROJECT DIRECTORIES
# ------------------------------------------------------------

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

assert ROOT.exists(), "Project directory not found."

EDITORIAL = ROOT / "09_EDITORIAL_SELECTION"

FOLDERS = {
    "main": EDITORIAL / "01_MAIN_FIGURES",
    "supp": EDITORIAL / "02_SUPPLEMENTARY_FIGURES",
    "tables": EDITORIAL / "03_SOURCE_TABLES",
    "metadata": EDITORIAL / "04_EDITORIAL_MANIFEST",
}

for folder in FOLDERS.values():
    folder.mkdir(parents=True, exist_ok=True)

print("=" * 76)
print("SATIPO — FINAL EDITORIAL SELECTION")
print("=" * 76)

# ------------------------------------------------------------
# 2. FINAL MAIN FIGURE PLAN
# ------------------------------------------------------------

MAIN_PLAN = [
    {
        "number": "Fig_02",
        "source": "FIG_TAXONOMIC_DIVERSITY",
        "title": "Taxonomic_Diversity",
        "role": "Taxonomic diversity and dominance"
    },
    {
        "number": "Fig_03",
        "source": "Fig_Biomass_Diameter_Classes",
        "title": "Biomass_Diameter_Classes",
        "role": "Biomass distribution across diameter classes"
    },
    {
        "number": "Fig_04",
        "source": "FIG_HEIGHT_DIAMETER_ALLOMETRY",
        "title": "Height_Diameter_Allometry",
        "role": "Forest architectural differences"
    },
    {
        "number": "Fig_05",
        "source": "FIG_TAXONOMIC_BIOMASS_DECOUPLING",
        "title": "Taxonomic_Biomass_Decoupling",
        "role": "Taxonomic versus biomass concentration"
    },
    {
        "number": "Fig_06",
        "source": "FIG_INTEGRATED_BIOMASS_DECOMPOSITION",
        "title": "Integrated_Biomass_Decomposition",
        "role": "Integrated decomposition of biomass differences"
    },
    {
        "number": "Fig_07",
        "source": "FIG_HISTORICAL_NATIVE_TRAJECTORIES",
        "title": "Historical_Spectral_Trajectories",
        "role": "Long-term sensor-specific Landsat trajectories"
    },
    {
        "number": "Fig_08",
        "source": "FIG_CARTO_01_SENTINEL2_SPECTRAL_ATLAS",
        "title": "Sentinel2_Spectral_Atlas",
        "role": "Spatial spectral structure in the field census year"
    },
    {
        "number": "Fig_09",
        "source": "FIG_S2_FIELD_YEAR_SPECTRAL_STRUCTURE",
        "title": "Multiscale_Spectral_Properties",
        "role": "Plot- and fragment-scale spectral contrasts"
    }
]

# Figure 1 is reserved for the manually prepared study-area map.

# ------------------------------------------------------------
# 3. FINAL SUPPLEMENTARY FIGURE PLAN
# ------------------------------------------------------------

SUPP_PLAN = [
    ("Fig_S01", "FIG_INEXT_COVERAGE",
     "Coverage_Standardized_Diversity"),

    ("Fig_S02", "FIG_BETA_DIVERSITY",
     "Taxonomic_Beta_Diversity"),

    ("Fig_S03", "FIG_ALLOMETRY_BOOTSTRAP",
     "Allometric_Bootstrap"),

    ("Fig_S04", "FIG_WOOD_DENSITY_DISTRIBUTION",
     "Wood_Density_Distribution"),

    ("Fig_S05", "FIG_SYSTEMATIC_HEIGHT_BIAS",
     "Systematic_Height_Bias"),

    ("Fig_S06", "FIG_ARCHITECTURE_DECOMPOSITION",
     "Counterfactual_Architecture"),

    ("Fig_S07", "FIG_INTEGRATED_MONTE_CARLO",
     "Integrated_Biomass_Uncertainty"),

    ("Fig_S08", "FIG_LANDSAT_PIXEL_SCREENING",
     "Landsat_Radiometric_Screening"),

    ("Fig_S09", "FIG_HISTORICAL_CONTRAST_TRAJECTORIES",
     "Historical_Spectral_Contrasts"),

    ("Fig_S10", "FIG_HISTORICAL_FINAL_TREND_ROBUSTNESS",
     "Historical_Trend_Robustness"),

    ("Fig_S11", "FIG_HISTORICAL_DISTURBANCE_DIAGNOSTICS",
     "Historical_Disturbance_Diagnostics"),

    ("Fig_S12", "FIG_EVENT_HYDROCLIMATIC_MATRIX",
     "Hydroclimatic_Context"),

    ("Fig_S13", "FIG_S2_TEMPORAL_SPECTRAL_CONTRASTS",
     "Sentinel2_Temporal_Consistency"),

    ("Fig_S14", "FIG_S2_SPATIAL_HETEROGENEITY",
     "Spectral_Spatial_Heterogeneity"),

    ("Fig_S15", "FIG_CARTO_03_GEDI_CANOPY_HEIGHT",
     "GEDI_Canopy_Height_Atlas"),

    ("Fig_S16", "FIG_CARTO_02_TOPOGRAPHIC_ATLAS",
     "Topographic_Atlas"),
]

SUPP_PLAN = [
    {
        "number": number,
        "source": source,
        "title": title,
        "role": title.replace("_", " ")
    }
    for number, source, title in SUPP_PLAN
]

# ------------------------------------------------------------
# 4. INVENTORY ALL ORIGINAL FIGURE FILES
# ------------------------------------------------------------

FORMATS = {
    ".png", ".pdf", ".svg", ".tif", ".tiff"
}

EXCLUDED_DIRS = {
    "09_EDITORIAL_SELECTION",
    "06_CHECKPOINTS"
}

all_figures = []

for path in ROOT.rglob("*"):

    if not path.is_file():
        continue

    if path.suffix.lower() not in FORMATS:
        continue

    relative = path.relative_to(ROOT)

    if any(
        part in EXCLUDED_DIRS
        for part in relative.parts
    ):
        continue

    all_figures.append(path)

print(
    f"Available figure files: {len(all_figures)}"
)

# ------------------------------------------------------------
# 5. FILE MATCHING AND VERIFICATION
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(block)

    return h.hexdigest()


def source_priority(path):

    relative = path.relative_to(ROOT)

    first = relative.parts[0]

    priority = {
        "05_OUTPUTS": 0,
        "04_REMOTE_SENSING": 0,
        "07_MANUSCRIPT_READY": 1,
        "08_FINAL_MANUSCRIPT_PACKAGE": 2
    }

    return (
        priority.get(first, 3),
        -path.stat().st_mtime,
        len(str(path))
    )


def find_figure(source, extension):

    exact = [
        p for p in all_figures
        if p.stem.casefold() == source.casefold()
        and p.suffix.lower() == extension
    ]

    if exact:
        return sorted(
            exact, key=source_priority
        )[0]

    # Fallback for already numbered editorial copies

    fallback = [
        p for p in all_figures
        if p.stem.casefold().endswith(
            source.casefold()
        )
        and p.suffix.lower() == extension
    ]

    if fallback:
        return sorted(
            fallback, key=source_priority
        )[0]

    return None


manifest_rows = []
missing_figures = []


def export_figure_plan(plan, destination, section):

    for item in plan:

        number = item["number"]
        source = item["source"]
        title = item["title"]

        copied_formats = []

        for extension in sorted(FORMATS):

            original = find_figure(
                source, extension
            )

            if original is None:
                continue

            target = destination / (
                f"{number}_{title}{extension}"
            )

            shutil.copy2(original, target)

            source_hash = sha256_file(original)
            target_hash = sha256_file(target)

            assert source_hash == target_hash, (
                f"Integrity error: {target.name}"
            )

            copied_formats.append(extension)

            manifest_rows.append({
                "section": section,
                "figure": number,
                "title": title,
                "scientific_role": item["role"],
                "original_file": str(
                    original.relative_to(ROOT)
                ),
                "editorial_file": str(
                    target.relative_to(EDITORIAL)
                ),
                "format": extension,
                "size_MB": round(
                    target.stat().st_size / 1024**2,
                    3
                ),
                "sha256": target_hash
            })

        if copied_formats:

            print(
                f"[OK] {number} — {title} "
                f"| {', '.join(copied_formats)}"
            )

        else:

            missing_figures.append({
                "section": section,
                "figure": number,
                "source": source
            })

            print(
                f"[MISSING] {number} — {source}"
            )


print("\nMAIN FIGURES")
print("-" * 76)

export_figure_plan(
    MAIN_PLAN,
    FOLDERS["main"],
    "Main"
)

print("\nSUPPLEMENTARY FIGURES")
print("-" * 76)

export_figure_plan(
    SUPP_PLAN,
    FOLDERS["supp"],
    "Supplementary"
)

# ------------------------------------------------------------
# 6. GENERATE TIFF FROM ORIGINAL PDF IF NECESSARY
# ------------------------------------------------------------

try:

    import fitz
    from PIL import Image

    print("\nGENERATING MISSING TIFF FILES")

    for section, folder in [
        ("Main", FOLDERS["main"]),
        ("Supplementary", FOLDERS["supp"])
    ]:

        for pdf in sorted(folder.glob("*.pdf")):

            tif = pdf.with_suffix(".tiff")

            if tif.exists() or pdf.with_suffix(
                ".tif"
            ).exists():
                continue

            with fitz.open(pdf) as doc:

                page = doc[0]

                pix = page.get_pixmap(
                    dpi=600,
                    colorspace=fitz.csRGB,
                    alpha=False
                )

                image = Image.frombytes(
                    "RGB",
                    (pix.width, pix.height),
                    pix.samples
                )

                image.save(
                    tif,
                    compression="tiff_lzw",
                    dpi=(600, 600)
                )

            manifest_rows.append({
                "section": section,
                "figure": pdf.stem.split("_")[0]
                          + "_"
                          + pdf.stem.split("_")[1],
                "title": pdf.stem,
                "scientific_role":
                    "600-dpi TIFF from PDF",
                "original_file": str(
                    pdf.relative_to(EDITORIAL)
                ),
                "editorial_file": str(
                    tif.relative_to(EDITORIAL)
                ),
                "format": ".tiff",
                "size_MB": round(
                    tif.stat().st_size / 1024**2,
                    3
                ),
                "sha256": sha256_file(tif)
            })

            print("[TIFF]", tif.name)

except ImportError:

    print(
        "TIFF conversion skipped: "
        "PyMuPDF or Pillow unavailable."
    )

# ------------------------------------------------------------
# 7. SELECT SOURCE WORKBOOKS
# ------------------------------------------------------------

# These are source workbooks, not finished editorial tables.
# No raw simulation draws are included in this download.

TABLE_NAMES = [
    "SATIPO_MAIN_TABLES.xlsx",
    "SATIPO_SUPPLEMENTARY_TABLES.xlsx",
    "SATIPO_INTEGRATED_UNCERTAINTY.xlsx",
    "SATIPO_LANDSAT_RADIOMETRIC_QA.xlsx",
    "SATIPO_LANDSAT_PIXEL_SCREENING.xlsx",
    "SATIPO_LANDSAT_FINAL_TREND_AUDIT.xlsx",
    "SATIPO_HISTORICAL_DISTURBANCE.xlsx",
    "SATIPO_SPATIAL_DISTURBANCE.xlsx",
    "SATIPO_HYDROCLIMATIC_CONTEXT.xlsx",
    "SATIPO_TOPOGRAPHIC_CONTEXT.xlsx"
]

all_workbooks = [
    p for p in ROOT.rglob("*.xlsx")
    if "09_EDITORIAL_SELECTION"
       not in p.relative_to(ROOT).parts
    and "06_CHECKPOINTS"
       not in p.relative_to(ROOT).parts
]

# Include actual Sentinel-2, GEDI, geometry,
# and cross-sensor workbooks by name.

keywords = [
    "SENTINEL",
    "GEDI",
    "GEOSPATIAL",
    "GEOMETRY",
    "HARMONIZATION",
    "CROSS_SENSOR",
    "HISTORICAL_TRAJECTORIES",
    "LANDSAT_AVAILABILITY"
]

for workbook in all_workbooks:

    name = workbook.name.upper()

    if any(k in name for k in keywords):

        if workbook.name not in TABLE_NAMES:
            TABLE_NAMES.append(workbook.name)

table_manifest = []
missing_workbooks = []

print("\nSOURCE WORKBOOKS")
print("-" * 76)

for filename in TABLE_NAMES:

    matches = [
        p for p in all_workbooks
        if p.name.casefold()
        == filename.casefold()
    ]

    if not matches:

        missing_workbooks.append(filename)

        print("[NOT FOUND]", filename)

        continue

    original = sorted(
        matches, key=source_priority
    )[0]

    target = FOLDERS["tables"] / filename

    shutil.copy2(original, target)

    assert (
        sha256_file(original)
        == sha256_file(target)
    )

    table_manifest.append({
        "workbook": filename,
        "source": str(
            original.relative_to(ROOT)
        ),
        "destination": str(
            target.relative_to(EDITORIAL)
        ),
        "size_MB": round(
            target.stat().st_size / 1024**2,
            3
        ),
        "sha256": sha256_file(target)
    })

    print("[OK]", filename)

# ------------------------------------------------------------
# 8. EDITORIAL TABLE PLAN
# ------------------------------------------------------------

main_table_plan = pd.DataFrame([
    ["Table 1",
     "Structural, taxonomic and functional characteristics",
     "Retain current Table 1"],

    ["Table 2",
     "Coverage-standardized Hill diversity",
     "Retain current Table 2"],

    ["Table 3",
     "Counterfactual aboveground biomass",
     "Current Table 4"],

    ["Table 4",
     "Integrated LMDI and Shapley decomposition",
     "Current Table 5"],

    ["Table 5",
     "Biomass robustness and integrated uncertainty",
     "Consolidate current Tables 6–7 and Table S29"],

    ["Table 6",
     "Recent Landsat spectral convergence",
     "Final Landsat trend audit"]
], columns=[
    "Final_number",
    "Content",
    "Editorial_action"
])

supp_table_plan = pd.DataFrame([
    ["S31",
     "Geometry and satellite observation availability"],

    ["S32",
     "Landsat radiometric QA and retention"],

    ["S33",
     "Cross-sensor comparison and validation"],

    ["S34",
     "Sensor-specific historical spectral trends"],

    ["S35",
     "Sensitivity and robustness of recent convergence"],

    ["S36",
     "Historical spectral anomalies and candidate events"],

    ["S37",
     "Hydroclimatic context and spectral associations"],

    ["S38",
     "Sentinel-2 multiscale spectral properties"],

    ["S39",
     "GEDI availability and canopy-height metrics"],

    ["S40",
     "Topographic characteristics"]
], columns=[
    "Final_number",
    "Content"
])

# ------------------------------------------------------------
# 9. SAVE MANIFESTS
# ------------------------------------------------------------

figure_manifest = pd.DataFrame(manifest_rows)

table_manifest_df = pd.DataFrame(
    table_manifest
)

figure_manifest.to_csv(
    FOLDERS["metadata"]
    / "EDITORIAL_FIGURE_MANIFEST.csv",
    index=False
)

table_manifest_df.to_csv(
    FOLDERS["metadata"]
    / "EDITORIAL_WORKBOOK_MANIFEST.csv",
    index=False
)

main_table_plan.to_csv(
    FOLDERS["metadata"]
    / "MAIN_TABLE_PLAN.csv",
    index=False
)

supp_table_plan.to_csv(
    FOLDERS["metadata"]
    / "SUPPLEMENTARY_TABLE_PLAN.csv",
    index=False
)

with pd.ExcelWriter(
    FOLDERS["metadata"]
    / "SATIPO_EDITORIAL_PLAN.xlsx",
    engine="openpyxl"
) as writer:

    figure_manifest.to_excel(
        writer,
        sheet_name="Figure_manifest",
        index=False
    )

    table_manifest_df.to_excel(
        writer,
        sheet_name="Source_workbooks",
        index=False
    )

    main_table_plan.to_excel(
        writer,
        sheet_name="Main_tables",
        index=False
    )

    supp_table_plan.to_excel(
        writer,
        sheet_name="Supplementary_tables",
        index=False
    )

# ------------------------------------------------------------
# 10. EDITORIAL README
# ------------------------------------------------------------

readme = """
SATIPO FOREST PROJECT
FINAL EDITORIAL SELECTION

MAIN MANUSCRIPT
---------------
9 figures planned, including the manually prepared
study-area map (Figure 1).

6 main tables planned.

SUPPLEMENTARY MATERIAL
----------------------
16 supplementary figures.
40 supplementary tables planned.

IMPORTANT
---------
The figure numbering in this package supersedes
the previous editorial numbering.

Figure 1 must be supplied manually.

The workbooks are original scientific sources.
They are not final formatted manuscript tables.

Required editorial corrections:
1. Restore the original inventory QA table as S1.
2. Restore the taxonomic validation table as S2.
3. Provide the complete 163-entry taxonomic checklist.
4. Complete the missing S6 beta-diversity row.
5. Update the integrated uncertainty table.
6. Update Figure 8's old caption to the new numbering.
7. Confirm mapped versus historically reported areas.
8. Correct overlapping labels in thematic maps.
9. Document the GEDI background image metadata.

SCIENTIFIC INTERPRETATION
-------------------------
The comparison is conditional on two sampled
forest plots and their corresponding fragments.

Satellite observations do not constitute independent
forest-type replication.

Spectral convergence does not demonstrate convergence
in biodiversity or aboveground biomass.

GEDI RH98 and mean individual-tree height are
different structural metrics.

The complete scientific archive remains unchanged.
"""

(EDITORIAL / "README.txt").write_text(
    readme.strip(),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 11. FINAL QUALITY CONTROL
# ------------------------------------------------------------

main_found = figure_manifest.loc[
    figure_manifest["section"] == "Main",
    "figure"
].nunique()

supp_found = figure_manifest.loc[
    figure_manifest["section"] == "Supplementary",
    "figure"
].nunique()

report = {
    "created_at": datetime.now().isoformat(),
    "main_figures_expected": len(MAIN_PLAN),
    "main_figures_found": int(main_found),
    "supplementary_figures_expected":
        len(SUPP_PLAN),
    "supplementary_figures_found":
        int(supp_found),
    "figure_1_manual": True,
    "source_workbooks_found":
        len(table_manifest),
    "missing_figures": missing_figures,
    "missing_workbooks": missing_workbooks,
    "original_archive_modified": False
}

with open(
    FOLDERS["metadata"]
    / "EDITORIAL_SELECTION_REPORT.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report, f,
        indent=2,
        ensure_ascii=False
    )

print("\n" + "=" * 76)
print("EDITORIAL SELECTION SUMMARY")
print("=" * 76)

print(
    "Main figures:",
    main_found,
    "/",
    len(MAIN_PLAN)
)

print(
    "Supplementary figures:",
    supp_found,
    "/",
    len(SUPP_PLAN)
)

print(
    "Source workbooks:",
    len(table_manifest)
)

print(
    "Missing figures:",
    len(missing_figures)
)

if missing_figures:

    display(
        pd.DataFrame(missing_figures)
    )

display(main_table_plan)
display(supp_table_plan)

# ------------------------------------------------------------
# 12. CREATE TWO DOWNLOADABLE PACKAGES
# ------------------------------------------------------------

DOWNLOAD_DIR = Path("/content")

FIGURE_ZIP = (
    DOWNLOAD_DIR
    / "SATIPO_SELECTED_FIGURES.zip"
)

FULL_ZIP = (
    DOWNLOAD_DIR
    / "SATIPO_EDITORIAL_SELECTION.zip"
)


def make_zip(destination, source_folders):

    with zipfile.ZipFile(
        destination,
        "w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=6
    ) as archive:

        for folder in source_folders:

            for path in sorted(folder.rglob("*")):

                if path.is_file():

                    archive.write(
                        path,
                        arcname=str(
                            path.relative_to(
                                EDITORIAL
                            )
                        )
                    )

        archive.write(
            EDITORIAL / "README.txt",
            arcname="README.txt"
        )

    with zipfile.ZipFile(destination) as archive:

        bad_file = archive.testzip()

        assert bad_file is None, (
            f"ZIP integrity failure: {bad_file}"
        )

    return round(
        destination.stat().st_size / 1024**2,
        2
    )


fig_zip_mb = make_zip(
    FIGURE_ZIP,
    [
        FOLDERS["main"],
        FOLDERS["supp"]
    ]
)

full_zip_mb = make_zip(
    FULL_ZIP,
    list(FOLDERS.values())
)

# Persistent copies in Google Drive

shutil.copy2(
    FIGURE_ZIP,
    ROOT / FIGURE_ZIP.name
)

shutil.copy2(
    FULL_ZIP,
    ROOT / FULL_ZIP.name
)

print("\n" + "=" * 76)
print("DOWNLOAD PACKAGES READY")
print("=" * 76)

print(
    f"Figures only: {fig_zip_mb} MB"
)

print(
    f"Complete editorial selection: "
    f"{full_zip_mb} MB"
)

print("\nPersistent Google Drive copies:")

print(ROOT / FIGURE_ZIP.name)
print(ROOT / FULL_ZIP.name)

# ------------------------------------------------------------
# 13. DOWNLOAD AUTOMATICALLY
# ------------------------------------------------------------

# Change to "figures" if you only want the figures.

DOWNLOAD_MODE = "complete"

download_target = (
    FULL_ZIP
    if DOWNLOAD_MODE == "complete"
    else FIGURE_ZIP
)

print("\nStarting download:")
print(download_target.name)

files.download(str(download_target))

print("\n" + "=" * 76)
print("CELL 101 COMPLETED")
print("EDITORIAL SELECTION ARCHIVED")
print("NEXT: MANUSCRIPT WRITING")
print("=" * 76)


In [ ]:

# =============================================================================
# CELL 102 — PUBLICATION-QUALITY STUDY AREA MAP
# SATIPO FOREST PROJECT
# =============================================================================

!pip -q install geopandas rasterio requests pillow pymupdf pyogrio

import os
import io
import re
import json
import zipfile
import requests
import warnings
import unicodedata
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

from pathlib import Path
from shapely.geometry import box
from matplotlib.patches import Rectangle, Patch
from matplotlib.lines import Line2D
from matplotlib.ticker import ScalarFormatter
from rasterio.io import MemoryFile
from rasterio.plot import plotting_extent
from PIL import Image

import ee

warnings.filterwarnings("ignore")

# =============================================================================
# 1. CONFIGURATION
# =============================================================================

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

REMOTE = ROOT / "04_REMOTE_SENSING"

OUT = ROOT / "09_STUDY_AREA_MAP"

OUT.mkdir(parents=True, exist_ok=True)

CACHE = OUT / "CARTOGRAPHIC_CACHE"
CACHE.mkdir(exist_ok=True)

CRS_UTM = "EPSG:32718"
CRS_GEO = "EPSG:4326"

COLORS = {
    "secondary": "#DB9140",
    "mature": "#24634F",
    "plot": "#FFD43B",
    "ink": "#24352E",
    "water": "#E5EEF2",
    "land": "#E5E8DE",
    "boundary": "#87958B",
    "highlight": "#E4B46F",
    "muted": "#687970"
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
    "axes.edgecolor": "#65746B",
    "axes.linewidth": 0.8,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "savefig.facecolor": "white",
    "figure.facecolor": "white",
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

REQUIRED_UNITS = {
    "FRAG_01", "FRAG_02", "P01", "P02"
}

print("=" * 78)
print("SATIPO — STUDY AREA CARTOGRAPHY")
print("=" * 78)

# =============================================================================
# 2. RECOVER VERIFIED PROJECT GEOMETRIES
# =============================================================================

def valid_units(gdf):

    if not isinstance(gdf, gpd.GeoDataFrame):
        return False

    if "unit_id" not in gdf.columns:
        return False

    if gdf.crs is None:
        return False

    return REQUIRED_UNITS.issubset(
        set(gdf["unit_id"].astype(str))
    )


geometry_candidates = []

# Recover existing GeoDataFrames from the notebook

for name, obj in list(globals().items()):

    if valid_units(obj):

        geometry_candidates.append(
            (name, obj.copy())
        )

# If notebook memory is unavailable, recover from Drive

if not geometry_candidates:

    search_dirs = [
        REMOTE / "02_GEOMETRY_VALIDATION",
        REMOTE / "01_VECTOR_INPUTS",
        REMOTE
    ]

    checked = set()

    for folder in search_dirs:

        if not folder.exists():
            continue

        for path in folder.rglob("*"):

            if path in checked:
                continue

            checked.add(path)

            if path.suffix.lower() not in [
                ".gpkg", ".geojson", ".shp"
            ]:
                continue

            try:

                if path.suffix.lower() == ".gpkg":

                    layers = gpd.list_layers(path)

                    for layer in layers["name"]:

                        gdf = gpd.read_file(
                            path,
                            layer=layer
                        )

                        if valid_units(gdf):

                            geometry_candidates.append(
                                (f"{path.name}:{layer}", gdf)
                            )

                else:

                    gdf = gpd.read_file(path)

                    if valid_units(gdf):

                        geometry_candidates.append(
                            (path.name, gdf)
                        )

            except Exception:
                continue

        if geometry_candidates:
            break

if not geometry_candidates:

    raise RuntimeError(
        "Could not recover the verified project geometries. "
        "Restore the project checkpoint or inspect "
        "02_GEOMETRY_VALIDATION."
    )

geometry_source, units = geometry_candidates[0]

units = units[
    units["unit_id"].isin(REQUIRED_UNITS)
].copy()

units = units.drop_duplicates("unit_id")

assert len(units) == 4
assert units.geometry.notna().all()
assert units.geometry.is_valid.all()
assert not units.geometry.is_empty.any()

units = units.to_crs(CRS_UTM)

units["area_ha"] = units.geometry.area / 10000

fragments = units[
    units["unit_id"].isin(["FRAG_01", "FRAG_02"])
].copy()

plots = units[
    units["unit_id"].isin(["P01", "P02"])
].copy()

units_geo = units.to_crs(CRS_GEO)

study_union = fragments.geometry.union_all()

study_center = gpd.GeoSeries(
    [study_union.centroid],
    crs=CRS_UTM
).to_crs(CRS_GEO).iloc[0]

lon = study_center.x
lat = study_center.y

print("Geometry source:", geometry_source)
print(f"Study center: {lat:.6f}, {lon:.6f}")

display(
    units[["unit_id", "area_ha"]]
    .sort_values("unit_id")
)

# =============================================================================
# 3. ADMINISTRATIVE BOUNDARIES
# =============================================================================

def download_file(url, destination):

    destination = Path(destination)

    if destination.exists() and destination.stat().st_size > 0:
        return destination

    response = requests.get(
        url,
        timeout=180
    )

    response.raise_for_status()

    destination.write_bytes(response.content)

    return destination


def get_geoboundaries(level):

    local_file = CACHE / f"PER_{level}.geojson"

    if not local_file.exists():

        api = (
            "https://www.geoboundaries.org/api/current/"
            f"gbOpen/PER/{level}/"
        )

        response = requests.get(
            api,
            timeout=90
        )

        response.raise_for_status()

        metadata = response.json()

        url = metadata["gjDownloadURL"]

        download_file(url, local_file)

        (CACHE / f"PER_{level}_metadata.json").write_text(
            json.dumps(metadata, indent=2),
            encoding="utf-8"
        )

    return gpd.read_file(local_file).to_crs(CRS_GEO)


peru = get_geoboundaries("ADM0")
departments = get_geoboundaries("ADM1")
provinces = get_geoboundaries("ADM2")


def normalize_name(value):

    value = unicodedata.normalize(
        "NFKD", str(value)
    )

    return "".join(
        c for c in value
        if not unicodedata.combining(c)
    ).lower().strip()


def select_admin(gdf, target):

    target = normalize_name(target)

    names = gdf["shapeName"].map(normalize_name)

    exact = gdf[names == target]

    if len(exact):
        return exact.copy()

    partial = gdf[
        names.str.contains(
            target,
            regex=False
        )
    ]

    if len(partial):
        return partial.copy()

    raise ValueError(
        f"Administrative unit not found: {target}"
    )


junin = select_admin(
    departments,
    "Junin"
)

satipo = select_admin(
    provinces,
    "Satipo"
)

junin_union = junin.geometry.union_all()

junin_provinces = provinces[
    provinces.geometry.representative_point()
    .within(junin_union)
].copy()

# Verify geographical consistency

assert junin_union.covers(study_center), (
    "Study centroid is outside the recovered Junin boundary."
)

assert satipo.geometry.union_all().covers(study_center), (
    "Study centroid is outside the recovered Satipo boundary."
)

print("Administrative boundaries: VERIFIED")

# Natural Earth — South American context

NE_URL = (
    "https://naturalearth.s3.amazonaws.com/"
    "110m_cultural/ne_110m_admin_0_countries.zip"
)

NE_ZIP = CACHE / "natural_earth_countries.zip"

download_file(NE_URL, NE_ZIP)

world = gpd.read_file(
    f"zip://{NE_ZIP}"
).to_crs(CRS_GEO)

south_america = world[
    world["CONTINENT"] == "South America"
].copy()

# =============================================================================
# 4. LOCAL MAP EXTENT
# =============================================================================

xmin, ymin, xmax, ymax = fragments.total_bounds

width = xmax - xmin
height = ymax - ymin

padding_x = max(130, width * 0.25)
padding_y = max(130, height * 0.10)

extent_utm = (
    xmin - padding_x,
    xmax + padding_x,
    ymin - padding_y,
    ymax + padding_y
)

left, right, bottom, top = extent_utm

local_bounds = box(
    left, bottom, right, top
)

# =============================================================================
# 5. SENTINEL-2 NATURAL-COLOR BACKGROUND
# =============================================================================

# Reuse a cached image if this cell was executed previously.
# Otherwise download a small 2019 seasonal composite.

RGB_PATH = CACHE / "SATIPO_SENTINEL2_RGB_2019.tif"

if not RGB_PATH.exists():

    try:
        ee.Number(1).getInfo()

    except Exception:

        ee.Authenticate()

        EE_PROJECT = input(
            "Enter your Earth Engine Cloud project ID: "
        ).strip()

        ee.Initialize(project=EE_PROJECT)

    region_ee = ee.Geometry.Rectangle(
        [left, bottom, right, top],
        proj=CRS_UTM,
        geodesic=False
    )

    def mask_s2(image):

        scl = image.select("SCL")

        valid = (
            scl.neq(0)
            .And(scl.neq(1))
            .And(scl.neq(3))
            .And(scl.neq(7))
            .And(scl.neq(8))
            .And(scl.neq(9))
            .And(scl.neq(10))
            .And(scl.neq(11))
        )

        return (
            image.select(["B4", "B3", "B2"])
            .multiply(0.0001)
            .updateMask(valid)
            .copyProperties(
                image,
                ["system:time_start"]
            )
        )

    s2 = (
        ee.ImageCollection(
            "COPERNICUS/S2_SR_HARMONIZED"
        )
        .filterBounds(region_ee)
        .filterDate("2019-05-01", "2019-10-01")
        .filter(
            ee.Filter.lt(
                "CLOUDY_PIXEL_PERCENTAGE", 80
            )
        )
        .map(mask_s2)
    )

    n_images = s2.size().getInfo()

    print("Sentinel-2 candidate images:", n_images)

    if n_images == 0:

        raise RuntimeError(
            "No Sentinel-2 observations available "
            "for the requested period."
        )

    rgb_composite = (
        s2.median()
        .updateMask(s2.select("B4").count().gte(3))
        .clip(region_ee)
        .toFloat()
    )

    url = rgb_composite.getDownloadURL({
        "name": "SATIPO_RGB_2019",
        "region": region_ee,
        "crs": CRS_UTM,
        "scale": 10,
        "format": "GEO_TIFF",
        "filePerBand": False
    })

    response = requests.get(
        url,
        timeout=180
    )

    response.raise_for_status()

    RGB_PATH.write_bytes(response.content)

    (CACHE / "RGB_metadata.json").write_text(
        json.dumps({
            "collection": "COPERNICUS/S2_SR_HARMONIZED",
            "start_date": "2019-05-01",
            "end_date_exclusive": "2019-10-01",
            "candidate_images": n_images,
            "minimum_valid_observations": 3,
            "bands": ["B4", "B3", "B2"],
            "resolution_m": 10,
            "crs": CRS_UTM
        }, indent=2),
        encoding="utf-8"
    )

# Read and stretch the natural-color image

import rasterio

with rasterio.open(RGB_PATH) as src:

    rgb_raw = src.read(
        [1, 2, 3],
        masked=True
    ).astype("float32")

    rgb_raw = rgb_raw.filled(np.nan)

    raster_extent = plotting_extent(src)

    raster_crs = src.crs

assert raster_crs.to_epsg() == 32718

rgb = np.moveaxis(
    rgb_raw,
    0,
    -1
)

valid_rgb = (
    np.isfinite(rgb).all(axis=2)
    & (rgb > 0).any(axis=2)
)

# Shared reflectance stretch preserves relative band colors

valid_values = rgb[valid_rgb]

if valid_values.size == 0:

    raise RuntimeError(
        "The Sentinel-2 background has no valid pixels."
    )

vmin = max(
    0,
    np.nanpercentile(valid_values, 2)
)

vmax = np.nanpercentile(
    valid_values, 98
)

rgb_display = np.clip(
    (rgb - vmin) / max(vmax - vmin, 1e-6),
    0,
    1
)

# Gentle gamma correction for publication display

rgb_display = rgb_display ** 0.85

rgb_display[~valid_rgb] = [
    0.88, 0.89, 0.87
]

print("Sentinel-2 background: READY")

# =============================================================================
# 6. CARTOGRAPHIC HELPERS
# =============================================================================

def north_arrow(
    ax,
    x=0.91,
    y=0.88,
    color=COLORS["ink"]
):

    ax.annotate(
        "",
        xy=(x, y + 0.075),
        xytext=(x, y - 0.015),
        xycoords="axes fraction",
        arrowprops=dict(
            arrowstyle="-|>",
            color=color,
            lw=1.8,
            mutation_scale=18
        ),
        zorder=30
    )

    ax.text(
        x,
        y - 0.035,
        "N",
        transform=ax.transAxes,
        ha="center",
        va="top",
        color=color,
        fontsize=11,
        fontweight="bold",
        zorder=30
    )


def scale_bar(
    ax,
    length_m,
    location=(0.07, 0.06),
    label=None
):

    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()

    x = x0 + location[0] * (x1 - x0)
    y = y0 + location[1] * (y1 - y0)

    h = (y1 - y0) * 0.012

    for i, color in enumerate(
        [COLORS["ink"], "white"]
    ):

        ax.add_patch(
            Rectangle(
                (x + i * length_m / 2, y),
                length_m / 2,
                h,
                facecolor=color,
                edgecolor=COLORS["ink"],
                linewidth=0.8,
                zorder=40
            )
        )

    if label is None:

        label = (
            f"{length_m / 1000:g} km"
            if length_m >= 1000
            else f"{length_m:g} m"
        )

    ax.text(
        x + length_m / 2,
        y + h * 1.7,
        label,
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
        color=COLORS["ink"],
        path_effects=[
            pe.withStroke(
                linewidth=3,
                foreground="white"
            )
        ],
        zorder=41
    )


def clean_axis(ax):

    ax.set_xticks([])
    ax.set_yticks([])

    for spine in ax.spines.values():

        spine.set_color("#9AA69D")
        spine.set_linewidth(0.7)


def label_halo(
    ax, x, y, text,
    color=COLORS["ink"],
    fontsize=10,
    **kwargs
):

    return ax.text(
        x, y, text,
        color=color,
        fontsize=fontsize,
        fontweight="bold",
        path_effects=[
            pe.withStroke(
                linewidth=3.0,
                foreground="white"
            )
        ],
        zorder=35,
        **kwargs
    )

# =============================================================================
# 7. CREATE THREE-PANEL PUBLICATION MAP
# =============================================================================

fig = plt.figure(
    figsize=(17, 10.5),
    facecolor="white"
)

gs = fig.add_gridspec(
    1, 3,
    width_ratios=[1.00, 1.10, 1.30],
    left=0.045,
    right=0.975,
    bottom=0.16,
    top=0.85,
    wspace=0.12
)

ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[0, 1])
ax_c = fig.add_subplot(gs[0, 2])

fig.suptitle(
    "Geographical setting of the studied Amazonian forests",
    fontsize=23,
    fontweight="bold",
    color=COLORS["ink"],
    y=0.965
)

fig.text(
    0.5, 0.915,
    "Central Peruvian Amazon | Satipo, Junín | Permanent forest inventory plots",
    ha="center",
    fontsize=12,
    color=COLORS["muted"]
)

# -----------------------------------------------------------------------------
# PANEL A — PERU IN SOUTH AMERICA
# -----------------------------------------------------------------------------

ax_a.set_facecolor(COLORS["water"])

south_america.plot(
    ax=ax_a,
    color="#E3E7DE",
    edgecolor="#A9B3A9",
    linewidth=0.5,
    zorder=1
)

peru.plot(
    ax=ax_a,
    color="#BCCBB3",
    edgecolor=COLORS["ink"],
    linewidth=1.2,
    zorder=3
)

departments.boundary.plot(
    ax=ax_a,
    color="#829582",
    linewidth=0.45,
    zorder=4
)

junin.plot(
    ax=ax_a,
    color=COLORS["highlight"],
    edgecolor="#A57539",
    linewidth=1.1,
    zorder=5
)

ax_a.scatter(
    lon, lat,
    s=48,
    color=COLORS["ink"],
    edgecolor="white",
    linewidth=1.2,
    zorder=10
)

ax_a.annotate(
    "Junín",
    xy=(lon, lat),
    xytext=(-70.8, -7.2),
    fontsize=10,
    fontweight="bold",
    color=COLORS["ink"],
    arrowprops=dict(
        arrowstyle="-",
        lw=1.0,
        color=COLORS["ink"]
    ),
    zorder=15
)

ax_a.text(
    -74.6, -13.3,
    "PERU",
    fontsize=13,
    fontweight="bold",
    color=COLORS["ink"],
    ha="center",
    path_effects=[
        pe.withStroke(
            linewidth=2,
            foreground="white"
        )
    ]
)

ax_a.text(
    -82.0, -15.2,
    "Pacific Ocean",
    fontsize=10,
    color="#71919D",
    rotation=65,
    ha="center"
)

ax_a.text(
    -69.2, -4.2,
    "BRAZIL",
    fontsize=9,
    color="#7C897C",
    ha="center"
)

ax_a.text(
    -69.4, -16.8,
    "BOLIVIA",
    fontsize=8,
    color="#7C897C",
    ha="center"
)

ax_a.set_xlim(-84.0, -67.5)
ax_a.set_ylim(-19.2, 1.5)

ax_a.set_aspect(
    1 / np.cos(np.deg2rad(-9))
)

ax_a.set_title(
    "(a) National context",
    loc="left",
    pad=13,
    color=COLORS["ink"]
)

clean_axis(ax_a)
north_arrow(ax_a)

# South America locator inset

inset = ax_a.inset_axes(
    [0.025, 0.025, 0.34, 0.30]
)

inset.set_facecolor("white")

south_america.plot(
    ax=inset,
    color="#D9DFD5",
    edgecolor="#9DA99D",
    linewidth=0.35
)

peru.plot(
    ax=inset,
    color=COLORS["mature"],
    edgecolor="white",
    linewidth=0.5
)

inset.set_xlim(-83, -33)
inset.set_ylim(-57, 14)
inset.set_aspect("equal")

clean_axis(inset)

inset.set_title(
    "South America",
    fontsize=7,
    pad=3
)

# -----------------------------------------------------------------------------
# PANEL B — JUNIN AND SATIPO
# -----------------------------------------------------------------------------

ax_b.set_facecolor("#F4F5EF")

# Use metric coordinates for the regional panel

junin_utm = junin.to_crs(CRS_UTM)

provinces_utm = junin_provinces.to_crs(CRS_UTM)

satipo_utm = satipo.to_crs(CRS_UTM)

junin_utm.plot(
    ax=ax_b,
    color="#DFE7D8",
    edgecolor=COLORS["ink"],
    linewidth=1.4,
    zorder=2
)

provinces_utm.boundary.plot(
    ax=ax_b,
    color="#9AAE98",
    linewidth=0.65,
    zorder=3
)

satipo_utm.plot(
    ax=ax_b,
    color="#E9C58F",
    edgecolor="#A57539",
    linewidth=1.3,
    zorder=4
)

# Province labels

for _, row in provinces_utm.iterrows():

    point = row.geometry.representative_point()

    name = str(row["shapeName"])

    if normalize_name(name) == "satipo":
        continue

    ax_b.text(
        point.x,
        point.y,
        name,
        fontsize=7.3,
        color="#607263",
        ha="center",
        va="center",
        zorder=8
    )

satipo_label = (
    satipo_utm.geometry.union_all()
    .representative_point()
)

label_halo(
    ax_b,
    satipo_label.x,
    satipo_label.y,
    "Satipo",
    fontsize=10,
    ha="center"
)

site = study_union.centroid

ax_b.scatter(
    site.x,
    site.y,
    marker="*",
    s=240,
    color=COLORS["mature"],
    edgecolor="white",
    linewidth=1.4,
    zorder=20
)

jx0, jy0, jx1, jy1 = junin_utm.total_bounds

jwidth = jx1 - jx0
jheight = jy1 - jy0

ax_b.set_xlim(
    jx0 - jwidth * 0.07,
    jx1 + jwidth * 0.07
)

ax_b.set_ylim(
    jy0 - jheight * 0.07,
    jy1 + jheight * 0.07
)

ax_b.annotate(
    "Study site",
    xy=(site.x, site.y),
    xytext=(
        site.x - jwidth * 0.25,
        site.y + jheight * 0.12
    ),
    fontsize=10,
    fontweight="bold",
    color=COLORS["ink"],
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="white",
        edgecolor="#B9C4B7",
        alpha=0.95
    ),
    arrowprops=dict(
        arrowstyle="->",
        color=COLORS["ink"],
        lw=1.3
    ),
    zorder=25
)

ax_b.set_aspect("equal")

ax_b.set_title(
    "(b) Regional context",
    loc="left",
    pad=13,
    color=COLORS["ink"]
)

clean_axis(ax_b)

north_arrow(ax_b)

scale_bar(
    ax_b,
    length_m=50000,
    location=(0.06, 0.045)
)

# -----------------------------------------------------------------------------
# PANEL C — FOREST FRAGMENTS AND PERMANENT PLOTS
# -----------------------------------------------------------------------------

ax_c.imshow(
    rgb_display,
    extent=raster_extent,
    interpolation="nearest",
    zorder=1
)

# White outer halo ensures legibility on satellite imagery

fragments.boundary.plot(
    ax=ax_c,
    color="white",
    linewidth=4.2,
    zorder=5
)

for unit_id, color in [
    ("FRAG_01", COLORS["secondary"]),
    ("FRAG_02", COLORS["mature"])
]:

    fragments[
        fragments["unit_id"] == unit_id
    ].boundary.plot(
        ax=ax_c,
        color=color,
        linewidth=2.5,
        zorder=6
    )

# Permanent inventory plots

plots.boundary.plot(
    ax=ax_c,
    color="#202A25",
    linewidth=3.5,
    zorder=9
)

plots.boundary.plot(
    ax=ax_c,
    color=COLORS["plot"],
    linewidth=2.0,
    linestyle="--",
    zorder=10
)

# Forest labels

for unit_id, text, color in [
    (
        "FRAG_01",
        "Late-secondary forest",
        COLORS["secondary"]
    ),
    (
        "FRAG_02",
        "Mature forest",
        COLORS["mature"]
    )
]:

    row = fragments[
        fragments["unit_id"] == unit_id
    ].iloc[0]

    point = row.geometry.representative_point()

    ax_c.text(
        point.x,
        point.y + height * 0.045,
        text,
        fontsize=9,
        fontweight="bold",
        color="white",
        ha="center",
        va="center",
        bbox=dict(
            boxstyle="round,pad=0.35",
            facecolor=color,
            edgecolor="white",
            linewidth=0.6,
            alpha=0.96
        ),
        zorder=20
    )

# Plot identifiers outside their polygons

for _, row in plots.iterrows():

    px0, py0, px1, py1 = row.geometry.bounds

    label_halo(
        ax_c,
        px1 + 13,
        py0,
        row["unit_id"],
        color="#202A25",
        fontsize=9,
        ha="left",
        va="center"
    )

ax_c.set_xlim(left, right)
ax_c.set_ylim(bottom, top)

ax_c.set_aspect("equal")

ax_c.set_title(
    "(c) Forest fragments and field plots",
    loc="left",
    pad=13,
    color=COLORS["ink"]
)

ax_c.set_xlabel("Easting (m)")
ax_c.set_ylabel("Northing (m)")

formatter = ScalarFormatter(
    useOffset=False
)

formatter.set_scientific(False)

ax_c.xaxis.set_major_formatter(formatter)
ax_c.yaxis.set_major_formatter(formatter)

ax_c.locator_params(
    axis="x",
    nbins=4
)

ax_c.locator_params(
    axis="y",
    nbins=5
)

ax_c.tick_params(
    axis="both",
    labelsize=8,
    length=3
)

north_arrow(
    ax_c,
    x=0.90,
    y=0.86,
    color="white"
)

scale_bar(
    ax_c,
    length_m=200,
    location=(0.06, 0.045)
)

# =============================================================================
# 8. COMMON LEGEND AND SOURCE NOTES
# =============================================================================

legend_elements = [

    Patch(
        facecolor=COLORS["highlight"],
        edgecolor="#A57539",
        label="Administrative area of interest"
    ),

    Line2D(
        [0], [0],
        color=COLORS["secondary"],
        linewidth=3,
        label="Late-secondary forest"
    ),

    Line2D(
        [0], [0],
        color=COLORS["mature"],
        linewidth=3,
        label="Mature forest"
    ),

    Line2D(
        [0], [0],
        color="#C5A500",
        linewidth=2,
        linestyle="--",
        label="Permanent field plots"
    ),

    Line2D(
        [0], [0],
        marker="*",
        markersize=12,
        color="none",
        markerfacecolor=COLORS["mature"],
        markeredgecolor=COLORS["mature"],
        label="Study site"
    )
]

fig.legend(
    handles=legend_elements,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.095),
    ncol=5,
    frameon=False,
    fontsize=10,
    columnspacing=1.7,
    handlelength=2.2
)

fig.text(
    0.5,
    0.060,
    (
        "Administrative boundaries: geoBoundaries and Natural Earth | "
        "Local background: Sentinel-2 SR, May–September 2019"
    ),
    ha="center",
    fontsize=9,
    color=COLORS["muted"]
)

fig.text(
    0.5,
    0.038,
    (
        "Local map: WGS 84 / UTM zone 18S | "
        "10 m natural-color composite | "
        "Forest and plot boundaries from project geometries"
    ),
    ha="center",
    fontsize=9,
    color=COLORS["muted"]
)

# =============================================================================
# 9. EXPORT PUBLICATION FORMATS
# =============================================================================

STEM = "Fig_01_Study_Area"

for ext in ["png", "pdf", "svg"]:

    path = OUT / f"{STEM}.{ext}"

    fig.savefig(
        path,
        dpi=600,
        bbox_inches="tight",
        pad_inches=0.18,
        facecolor="white"
    )

    print("Saved:", path.name)

plt.show()

# TIFF — 600 dpi, LZW lossless compression

png_path = OUT / f"{STEM}.png"

tiff_path = OUT / f"{STEM}.tiff"

with Image.open(png_path) as img:

    img.convert("RGB").save(
        tiff_path,
        format="TIFF",
        compression="tiff_lzw",
        dpi=(600, 600)
    )

print("Saved:", tiff_path.name)

# =============================================================================
# 10. SAVE MAP METADATA
# =============================================================================

metadata = {
    "figure": "Figure 1",
    "title": (
        "Geographical setting of the studied Amazonian forests"
    ),
    "geometry_source": geometry_source,
    "study_center_latitude": lat,
    "study_center_longitude": lon,
    "local_crs": CRS_UTM,
    "local_background": "Sentinel-2 SR Harmonized",
    "background_period": "May–September 2019",
    "background_resolution_m": 10,
    "administrative_boundaries": "geoBoundaries",
    "continental_context": "Natural Earth",
    "export_dpi": 600,
    "units": units[
        ["unit_id", "area_ha"]
    ].to_dict("records")
}

(OUT / "FIGURE_01_METADATA.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)

# =============================================================================
# 11. DOWNLOAD PACKAGE
# =============================================================================

ZIP_PATH = ROOT / "SATIPO_FIGURE_01_LOCATION_MAP.zip"

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    for path in OUT.glob(f"{STEM}.*"):

        z.write(
            path,
            arcname=path.name
        )

    z.write(
        OUT / "FIGURE_01_METADATA.json",
        arcname="FIGURE_01_METADATA.json"
    )

print("\n" + "=" * 78)
print("FIGURE 1 — PUBLICATION MAP COMPLETED")
print("=" * 78)

print("Output folder:", OUT)
print("ZIP package:", ZIP_PATH)

print(
    "ZIP size (MB):",
    round(ZIP_PATH.stat().st_size / 1024**2, 2)
)

print("\nEXPORTED FILES")

for path in sorted(OUT.glob(f"{STEM}.*")):

    print(
        f"{path.name:40s}",
        f"{path.stat().st_size / 1024**2:.2f} MB"
    )

print("=" * 78)

from google.colab import files

files.download(str(ZIP_PATH))


In [ ]:

# ============================================================
# CELL 103 — FIGURE 02: TAXONOMIC DIVERSITY
# PUBLICATION-QUALITY REDESIGN
#
# Original scientific sources: Cells 34, 35 and 36
# No statistical recalculation or modification of results
# ============================================================

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from PIL import Image

# ============================================================
# 1. PATHS AND ORIGINAL SCIENTIFIC OBJECTS
# ============================================================

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

SOURCE = (
    ROOT
    / "03_ANALYSIS"
    / "TAXONOMIC_DIVERSITY"
)

OUT = (
    ROOT
    / "10_PUBLICATION_FIGURES"
    / "FIG_02"
)

OUT.mkdir(parents=True, exist_ok=True)

SCENARIO = "All_operational_taxa"

FORESTS = [
    "Late_secondary",
    "Mature"
]

# ------------------------------------------------------------
# Recover original results without recalculation
# ------------------------------------------------------------

def load_original_table(variable, filename, columns):

    obj = globals().get(variable)

    if isinstance(obj, pd.DataFrame):

        if set(columns).issubset(obj.columns):

            print(f"Recovered object: {variable}")

            return obj.copy(deep=True)

    path = SOURCE / filename

    if not path.exists():

        raise FileNotFoundError(
            f"Original scientific table not found: {path}"
        )

    df = pd.read_csv(path)

    assert set(columns).issubset(df.columns), (
        f"Unexpected columns in {filename}"
    )

    print(f"Recovered file: {path.name}")

    return df


hill_original = load_original_table(

    "hill_summary",

    "HILL_DIVERSITY_SUMMARY.csv",

    [
        "scenario",
        "forest_type",
        "n_individuals",
        "hill_q0",
        "hill_q1",
        "hill_q2"
    ]

)

profiles_original = load_original_table(

    "hill_profiles",

    "HILL_DIVERSITY_PROFILES.csv",

    [
        "scenario",
        "forest_type",
        "q",
        "effective_taxa"
    ]

)

rarefaction_original = load_original_table(

    "rarefaction_curves",

    "RAREFACTION_CURVES.csv",

    [
        "scenario",
        "forest_type",
        "sample_size",
        "expected_richness"
    ]

)

standardized_original = load_original_table(

    "standardized_richness",

    "STANDARDIZED_RICHNESS.csv",

    [
        "scenario",
        "forest_type",
        "common_sample_size",
        "rarefied_richness"
    ]

)

# ============================================================
# 2. EXTRACT THE EXACT ORIGINAL FIGURE DATA
# ============================================================

hill_plot = hill_original.loc[
    hill_original["scenario"].eq(SCENARIO)
].copy()

profiles_plot = profiles_original.loc[
    profiles_original["scenario"].eq(SCENARIO)
].copy()

rarefaction_plot = rarefaction_original.loc[
    rarefaction_original["scenario"].eq(SCENARIO)
].copy()

standardized_plot = standardized_original.loc[
    standardized_original["scenario"].eq(SCENARIO)
].copy()

hill_plot = (
    hill_plot
    .set_index("forest_type")
    .loc[FORESTS]
)

common_n = int(
    standardized_plot[
        "common_sample_size"
    ].iloc[0]
)

# ============================================================
# 3. SCIENTIFIC INTEGRITY CHECKS
# ============================================================

assert len(hill_plot) == 2

assert common_n == 464

assert (
    hill_plot.loc[
        "Late_secondary",
        "n_individuals"
    ] == 545
)

assert (
    hill_plot.loc[
        "Mature",
        "n_individuals"
    ] == 464
)

assert (
    hill_plot.loc[
        "Late_secondary",
        "hill_q0"
    ] == 91
)

assert (
    hill_plot.loc[
        "Mature",
        "hill_q0"
    ] == 111
)

assert (
    standardized_plot[
        "common_sample_size"
    ].nunique() == 1
)

# Verify that profile values match observed Hill numbers

for forest in FORESTS:

    p = profiles_plot.loc[
        profiles_plot["forest_type"].eq(forest)
    ]

    for q in [0, 1, 2]:

        profile_value = p.loc[
            np.isclose(p["q"], q),
            "effective_taxa"
        ].iloc[0]

        observed_value = hill_plot.loc[
            forest,
            f"hill_q{q}"
        ]

        assert np.isclose(
            profile_value,
            observed_value,
            rtol=1e-10
        )

    # Verify rarefaction endpoint identities

    r = (
        rarefaction_plot.loc[
            rarefaction_plot[
                "forest_type"
            ].eq(forest)
        ]
        .sort_values("sample_size")
    )

    assert np.isclose(
        r["expected_richness"].iloc[-1],
        hill_plot.loc[forest, "hill_q0"]
    )

    assert (
        r["sample_size"].iloc[-1]
        ==
        hill_plot.loc[
            forest,
            "n_individuals"
        ]
    )

print("\nORIGINAL SCIENTIFIC RESULTS VERIFIED")

display(
    hill_plot[
        [
            "n_individuals",
            "hill_q0",
            "hill_q1",
            "hill_q2"
        ]
    ].round(6)
)

display(
    standardized_plot[
        [
            "forest_type",
            "common_sample_size",
            "rarefied_richness"
        ]
    ].round(6)
)

# ============================================================
# 4. EDITORIAL VISUAL IDENTITY
# ============================================================

COLORS = {

    "Late_secondary": "#4477AA",

    "Mature": "#AA3377"

}

NAMES = {

    "Late_secondary": "Late-secondary forest",

    "Mature": "Mature forest"

}

MARKERS = {

    "Late_secondary": "o",

    "Mature": "D"

}

LINESTYLES = {

    "Late_secondary": "-",

    "Mature": "-"

}

TEXT = "#242A30"
MUTED = "#69737D"
GRID = "#E6E9ED"
AXIS = "#9BA3AA"

STYLE = {

    "font.family": "DejaVu Sans",

    "font.size": 10,

    "axes.labelsize": 10.5,

    "axes.titlesize": 11.5,

    "axes.titleweight": "medium",

    "axes.labelcolor": TEXT,

    "axes.edgecolor": AXIS,

    "axes.linewidth": 0.8,

    "axes.spines.top": False,

    "axes.spines.right": False,

    "xtick.color": TEXT,

    "ytick.color": TEXT,

    "xtick.labelsize": 9.5,

    "ytick.labelsize": 9.5,

    "xtick.major.size": 3.5,

    "ytick.major.size": 3.5,

    "xtick.major.width": 0.8,

    "ytick.major.width": 0.8,

    "legend.frameon": False,

    "figure.facecolor": "white",

    "axes.facecolor": "white",

    "savefig.facecolor": "white",

    "savefig.dpi": 600,

    "pdf.fonttype": 42,

    "ps.fonttype": 42,

    "svg.fonttype": "none"

}

# ============================================================
# 5. HELPER FUNCTIONS
# ============================================================

def style_axis(ax):

    ax.set_axisbelow(True)

    ax.grid(
        axis="y",
        color=GRID,
        linewidth=0.65,
        linestyle="-",
        alpha=0.9
    )

    ax.tick_params(
        axis="both",
        direction="out",
        pad=5
    )

    ax.spines["left"].set_color(AXIS)
    ax.spines["bottom"].set_color(AXIS)

    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)

    ax.margins(x=0.03)


def panel_title(ax, letter, title):

    ax.text(

        -0.10,
        1.065,

        letter,

        transform=ax.transAxes,

        fontsize=14,

        fontweight="bold",

        color=TEXT,

        ha="left",

        va="bottom"

    )

    ax.text(

        0.045,
        1.075,

        title,

        transform=ax.transAxes,

        fontsize=11.5,

        fontweight="medium",

        color=TEXT,

        ha="left",

        va="bottom"

    )


# ============================================================
# 6. CREATE PUBLICATION FIGURE
# ============================================================

with plt.rc_context(STYLE):

    fig = plt.figure(
        figsize=(13.8, 4.9),
        facecolor="white"
    )

    gs = fig.add_gridspec(

        nrows=1,

        ncols=3,

        width_ratios=[1.0, 1.05, 1.10],

        left=0.075,

        right=0.985,

        bottom=0.18,

        top=0.76,

        wspace=0.40

    )

    axes = [
        fig.add_subplot(gs[0, i])
        for i in range(3)
    ]

    # ========================================================
    # PANEL A — OBSERVED HILL DIVERSITY
    # ========================================================

    ax = axes[0]

    x = np.arange(3)

    bar_width = 0.31

    separation = 0.025

    metrics = [
        "hill_q0",
        "hill_q1",
        "hill_q2"
    ]

    for i, forest in enumerate(FORESTS):

        values = hill_plot.loc[
            forest,
            metrics
        ].to_numpy(dtype=float)

        offset = (
            (i - 0.5)
            *
            (bar_width + separation)
        )

        bars = ax.bar(

            x + offset,

            values,

            width=bar_width,

            color=COLORS[forest],

            edgecolor="white",

            linewidth=0.45,

            zorder=3

        )

        ax.bar_label(

            bars,

            labels=[
                f"{v:.1f}"
                for v in values
            ],

            padding=4,

            fontsize=9,

            color=TEXT,

            fontweight="medium"

        )

    ax.set_xticks(
        x,
        [
            r"$q=0$",
            r"$q=1$",
            r"$q=2$"
        ]
    )

    ax.set_xlabel(
        "Diversity order",
        labelpad=9
    )

    ax.set_ylabel(
        "Effective number of taxa",
        labelpad=8
    )

    ax.set_ylim(0, 130)

    ax.set_xlim(-0.55, 2.55)

    ax.yaxis.set_major_locator(
        mticker.MultipleLocator(25)
    )

    style_axis(ax)

    panel_title(
        ax,
        "a",
        "Observed Hill diversity"
    )

    # ========================================================
    # PANEL B — CONTINUOUS DIVERSITY PROFILES
    # ========================================================

    ax = axes[1]

    for forest in FORESTS:

        d = (

            profiles_plot.loc[
                profiles_plot[
                    "forest_type"
                ].eq(forest)
            ]

            .sort_values("q")

        )

        # Original 61 profile values
        # No smoothing or interpolation

        ax.plot(

            d["q"],

            d["effective_taxa"],

            color=COLORS[forest],

            linewidth=2.5,

            linestyle=LINESTYLES[forest],

            solid_capstyle="round",

            zorder=3

        )

        # Original values at Hill orders 0, 1 and 2

        selected = d.loc[
            d["q"].isin([0, 1, 2])
        ]

        ax.scatter(

            selected["q"],

            selected["effective_taxa"],

            s=35,

            marker=MARKERS[forest],

            facecolor="white",

            edgecolor=COLORS[forest],

            linewidth=1.5,

            zorder=5

        )

    ax.set_xlabel(
        r"Diversity order ($q$)",
        labelpad=9
    )

    ax.set_ylabel(
        "Effective number of taxa",
        labelpad=8
    )

    ax.set_xlim(-0.06, 3.06)

    ax.set_ylim(0, 130)

    ax.xaxis.set_major_locator(
        mticker.MultipleLocator(0.5)
    )

    ax.yaxis.set_major_locator(
        mticker.MultipleLocator(25)
    )

    style_axis(ax)

    panel_title(
        ax,
        "b",
        "Diversity profiles"
    )

    # ========================================================
    # PANEL C — INDIVIDUAL-BASED RAREFACTION
    # ========================================================

    ax = axes[2]

    # Common sample size reference

    ax.axvline(

        common_n,

        color=MUTED,

        linewidth=1.15,

        linestyle=(0, (4, 3)),

        zorder=1

    )

    for forest in FORESTS:

        d = (

            rarefaction_plot.loc[
                rarefaction_plot[
                    "forest_type"
                ].eq(forest)
            ]

            .sort_values("sample_size")

        )

        # Plot original rarefaction curve
        # No extrapolation or smoothing

        ax.plot(

            d["sample_size"],

            d["expected_richness"],

            color=COLORS[forest],

            linewidth=2.5,

            solid_capstyle="round",

            zorder=3

        )

        # Observed richness endpoint

        ax.scatter(

            d["sample_size"].iloc[-1],

            d["expected_richness"].iloc[-1],

            s=44,

            marker=MARKERS[forest],

            facecolor=COLORS[forest],

            edgecolor="white",

            linewidth=0.8,

            zorder=6

        )

        # Exact original richness at common sample size

        common_value = standardized_plot.loc[

            standardized_plot[
                "forest_type"
            ].eq(forest),

            "rarefied_richness"

        ].iloc[0]

        # Secondary richness at n=464
        # The mature endpoint already represents n=464

        if forest == "Late_secondary":

            ax.scatter(

                common_n,

                common_value,

                s=52,

                marker="o",

                facecolor="white",

                edgecolor=COLORS[forest],

                linewidth=1.6,

                zorder=7

            )

        # Label original full-sample richness

        ax.annotate(

            f"{d['expected_richness'].iloc[-1]:.0f}",

            xy=(

                d["sample_size"].iloc[-1],

                d["expected_richness"].iloc[-1]

            ),

            xytext=(7, 1),

            textcoords="offset points",

            color=COLORS[forest],

            fontsize=9,

            fontweight="medium",

            ha="left",

            va="center"

        )

    # Common sample size annotation

    ax.text(

        common_n - 12,

        124,

        r"Common $n=464$",

        fontsize=9,

        color=MUTED,

        ha="right",

        va="center",

        bbox=dict(

            facecolor="white",

            edgecolor="none",

            pad=2

        )

    )

    ax.set_xlabel(
        "Number of individuals",
        labelpad=9
    )

    ax.set_ylabel(
        "Expected taxon richness",
        labelpad=8
    )

    ax.set_xlim(0, 595)

    ax.set_ylim(0, 130)

    ax.xaxis.set_major_locator(
        mticker.MultipleLocator(100)
    )

    ax.yaxis.set_major_locator(
        mticker.MultipleLocator(25)
    )

    style_axis(ax)

    panel_title(
        ax,
        "c",
        "Individual-based rarefaction"
    )

    # ========================================================
    # 7. SHARED LEGEND
    # ========================================================

    handles = [

        Patch(

            facecolor=COLORS["Late_secondary"],

            edgecolor="none",

            label="Late-secondary forest"

        ),

        Patch(

            facecolor=COLORS["Mature"],

            edgecolor="none",

            label="Mature forest"

        )

    ]

    fig.legend(

        handles=handles,

        loc="upper center",

        bbox_to_anchor=(0.52, 0.965),

        ncol=2,

        fontsize=10.5,

        handlelength=2.0,

        handleheight=0.85,

        handletextpad=0.65,

        columnspacing=2.4,

        frameon=False

    )

    # ========================================================
    # 8. EXPORT — PRESERVE ORIGINAL FIGURE FILES
    # ========================================================

    basename = "Fig_02_Taxonomic_Diversity_FINAL"

    for extension in [
        "pdf",
        "svg",
        "png"
    ]:

        output_path = (
            OUT / f"{basename}.{extension}"
        )

        fig.savefig(

            output_path,

            dpi=600,

            bbox_inches="tight",

            pad_inches=0.15,

            facecolor="white"

        )

    # TIFF with lossless LZW compression

    png_path = (
        OUT / f"{basename}.png"
    )

    tiff_path = (
        OUT / f"{basename}.tiff"
    )

    with Image.open(png_path) as im:

        im.convert("RGB").save(

            tiff_path,

            compression="tiff_lzw",

            dpi=(600, 600)

        )

    plt.show()

# ============================================================
# 9. EXPORT EXACT FIGURE DATA FOR TRACEABILITY
# ============================================================

hill_plot.reset_index().to_csv(

    OUT / "FIG02_SOURCE_HILL_DIVERSITY.csv",

    index=False

)

profiles_plot.to_csv(

    OUT / "FIG02_SOURCE_DIVERSITY_PROFILES.csv",

    index=False

)

rarefaction_plot.to_csv(

    OUT / "FIG02_SOURCE_RAREFACTION.csv",

    index=False

)

standardized_plot.to_csv(

    OUT / "FIG02_SOURCE_COMMON_SAMPLE_SIZE.csv",

    index=False

)

# ============================================================
# 10. FIGURE PROVENANCE AND VALIDATION MANIFEST
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            h.update(chunk)

    return h.hexdigest()


metadata = {

    "figure": "Figure 2",

    "title": "Taxonomic diversity",

    "original_pipeline_cells": [34, 35, 36],

    "redesign_cell": 103,

    "scenario": SCENARIO,

    "scientific_recalculation": False,

    "original_results_modified": False,

    "curve_smoothing": False,

    "extrapolation": False,

    "inventory": {

        "Late_secondary": 545,

        "Mature": 464

    },

    "observed_richness": {

        "Late_secondary": 91,

        "Mature": 111

    },

    "common_sample_size": common_n,

    "palette": COLORS,

    "export_dpi": 600,

    "validation": {

        "inventory_counts_verified": True,

        "observed_richness_verified": True,

        "hill_profile_identity_verified": True,

        "rarefaction_endpoints_verified": True

    },

    "files": []

}

for path in sorted(OUT.glob("*")):

    if path.is_file():

        if path.suffix.lower() in [

            ".png",

            ".pdf",

            ".svg",

            ".tiff",

            ".csv"

        ]:

            metadata["files"].append({

                "filename": path.name,

                "size_bytes": path.stat().st_size,

                "sha256": sha256_file(path)

            })

with open(

    OUT / "FIG02_PROVENANCE.json",

    "w",

    encoding="utf-8"

) as f:

    json.dump(
        metadata,
        f,
        indent=2
    )

# ============================================================
# 11. FINAL REPORT
# ============================================================

print("\n" + "=" * 70)

print("SATIPO — FIGURE 02 PUBLICATION REMAKE")

print("=" * 70)

print("Original scientific results: PRESERVED")

print("Hill diversity validation: PASSED")

print("Diversity profile validation: PASSED")

print("Rarefaction validation: PASSED")

print("Scientific recalculation: NONE")

print("Common sample size:", common_n)

print("Output directory:", OUT)

print("\nEXPORTED FILES")

for path in sorted(OUT.glob("*")):

    if path.is_file():

        print(

            f"  {path.name:55s}"

            f"{path.stat().st_size / 1024:10.1f} KB"

        )

print("\nCELL 103 COMPLETED SUCCESSFULLY")


In [ ]:

# ============================================================
# CELL 104B — FIGURE 03: FINAL LOLLIPOP DESIGN
# Publication-quality forest structure visualization
#
# Scientific source: Original validated fig03_data
# No recalculation of biomass or abundance
# ============================================================

from pathlib import Path
import json
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from matplotlib.lines import Line2D
from PIL import Image

# ============================================================
# 1. PATHS AND ORIGINAL DATA
# ============================================================

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

OUT = (
    ROOT
    / "10_PUBLICATION_FIGURES"
    / "FIG_03"
    / "LOLLIPOP_FINAL"
)

OUT.mkdir(parents=True, exist_ok=True)

# Recover validated original data

if (
    "fig03_data" in globals()
    and isinstance(fig03_data, pd.DataFrame)
):

    data = fig03_data.copy(deep=True)

    source = "Validated fig03_data from Cell 104"

else:

    source_path = (
        ROOT
        / "10_PUBLICATION_FIGURES"
        / "FIG_03"
        / "FIG03_SOURCE_DIAMETER_BIOMASS.csv"
    )

    if not source_path.exists():

        raise FileNotFoundError(
            "Run Cell 104 first to recover "
            "the original scientific data."
        )

    data = pd.read_csv(source_path)

    source = str(source_path)

# ============================================================
# 2. SCIENTIFIC INTEGRITY
# ============================================================

FORESTS = [
    "Late_secondary",
    "Mature"
]

CLASSES = [
    "10-<20",
    "20-<30",
    "30-<40",
    "40-<60",
    "60-<80",
    ">=80"
]

DISPLAY_CLASSES = [
    "10–<20",
    "20–<30",
    "30–<40",
    "40–<60",
    "60–<80",
    "≥80"
]

REQUIRED = [
    "forest_type",
    "dbh_class",
    "n_trees",
    "agb_Mg",
    "abundance_pct",
    "biomass_pct"
]

assert set(REQUIRED).issubset(data.columns)

data["dbh_class"] = (
    data["dbh_class"].astype(str)
)

assert len(data) == 12

assert not data.duplicated(
    ["forest_type", "dbh_class"]
).any()

assert set(data["forest_type"]) == set(FORESTS)

assert set(data["dbh_class"]) == set(CLASSES)

assert data[REQUIRED].notna().all().all()

assert (
    data[
        [
            "n_trees",
            "agb_Mg",
            "abundance_pct",
            "biomass_pct"
        ]
    ] >= 0
).all().all()

EXPECTED_N = {
    "Late_secondary": 541,
    "Mature": 461
}

EXPECTED_AGB = {
    "Late_secondary": 355.2266,
    "Mature": 234.3626
}

tables = {}

for forest in FORESTS:

    d = (
        data.loc[
            data["forest_type"].eq(forest)
        ]
        .set_index("dbh_class")
        .reindex(CLASSES)
        .copy()
    )

    assert not d[REQUIRED[2:]].isna().any().any()

    assert d["n_trees"].sum() == EXPECTED_N[forest]

    assert np.isclose(
        d["agb_Mg"].sum(),
        EXPECTED_AGB[forest],
        atol=0.001,
        rtol=0
    )

    assert np.isclose(
        d["abundance_pct"].sum(),
        100
    )

    assert np.isclose(
        d["biomass_pct"].sum(),
        100
    )

    assert np.allclose(
        d["abundance_pct"],
        100 * d["n_trees"] / d["n_trees"].sum()
    )

    assert np.allclose(
        d["biomass_pct"],
        100 * d["agb_Mg"] / d["agb_Mg"].sum()
    )

    tables[forest] = d

print("Original scientific data: VERIFIED")

print("Source:", source)

display(
    data[
        [
            "forest_type",
            "dbh_class",
            "n_trees",
            "abundance_pct",
            "biomass_pct",
            "agb_Mg"
        ]
    ].round(4)
)

# ============================================================
# 3. EDITORIAL VISUAL IDENTITY
# ============================================================

COLORS = {

    "Late_secondary": "#C46D5E",

    "Mature": "#285B78"

}

LABELS = {

    "Late_secondary": "Late-secondary forest",

    "Mature": "Mature forest"

}

MARKERS = {

    "Late_secondary": "o",

    "Mature": "D"

}

TEXT = "#26333D"

MUTED = "#6D7882"

GRID = "#E6EAED"

AXIS = "#A4ADB5"

STYLE = {

    "font.family": "DejaVu Sans",

    "font.size": 10,

    "axes.labelsize": 10.5,

    "axes.labelcolor": TEXT,

    "axes.edgecolor": AXIS,

    "axes.linewidth": 0.8,

    "axes.spines.top": False,

    "axes.spines.right": False,

    "xtick.color": TEXT,

    "ytick.color": TEXT,

    "xtick.labelsize": 9,

    "ytick.labelsize": 9.5,

    "xtick.major.size": 0,

    "ytick.major.size": 3,

    "ytick.major.width": 0.8,

    "legend.frameon": False,

    "figure.facecolor": "white",

    "axes.facecolor": "white",

    "savefig.facecolor": "white",

    "pdf.fonttype": 42,

    "ps.fonttype": 42,

    "svg.fonttype": "none"

}

# ============================================================
# 4. PANEL DEFINITIONS
# ============================================================

PANELS = [

    {
        "letter": "a",
        "title": "Stem abundance",
        "metric": "abundance_pct",
        "ylabel": "Relative stem abundance (%)",
        "ylim": (0, 70),
        "step": 10
    },

    {
        "letter": "b",
        "title": "Biomass allocation",
        "metric": "biomass_pct",
        "ylabel": "Relative aboveground biomass (%)",
        "ylim": (0, 40),
        "step": 10
    },

    {
        "letter": "c",
        "title": "Absolute biomass",
        "metric": "agb_Mg",
        "ylabel": r"Aboveground biomass (Mg ha$^{-1}$)",
        "ylim": (0, 140),
        "step": 20
    }

]

# ============================================================
# 5. PUBLICATION FIGURE
# ============================================================

with plt.rc_context(STYLE):

    fig = plt.figure(
        figsize=(14.6, 5.0),
        facecolor="white"
    )

    gs = fig.add_gridspec(

        1,
        3,

        left=0.075,

        right=0.985,

        bottom=0.20,

        top=0.76,

        wspace=0.36

    )

    axes = [
        fig.add_subplot(gs[0, i])
        for i in range(3)
    ]

    x = np.arange(len(CLASSES))

    # Horizontal separation between forest types
    # This is graphical only.

    offsets = {
        "Late_secondary": -0.13,
        "Mature": 0.13
    }

    for ax, panel in zip(axes, PANELS):

        metric = panel["metric"]

        # ----------------------------------------------------
        # Subtle horizontal reference grid
        # ----------------------------------------------------

        ax.set_axisbelow(True)

        ax.grid(

            axis="y",

            color=GRID,

            linewidth=0.65,

            linestyle="-"

        )

        # ----------------------------------------------------
        # Lollipop stems and endpoints
        # ----------------------------------------------------

        for forest in FORESTS:

            values = (
                tables[forest][metric]
                .to_numpy(dtype=float)
            )

            positions = x + offsets[forest]

            # Thin stems extending from zero

            ax.vlines(

                positions,

                ymin=0,

                ymax=values,

                color=COLORS[forest],

                linewidth=2.0,

                alpha=0.85,

                zorder=3

            )

            # Solid endpoints with white separation

            ax.scatter(

                positions,

                values,

                s=62,

                marker=MARKERS[forest],

                facecolor=COLORS[forest],

                edgecolor="white",

                linewidth=0.85,

                zorder=5,

                clip_on=False

            )

            # ------------------------------------------------
            # Highlight the maximum of each forest
            # without overcrowding all six classes
            # ------------------------------------------------

            peak = int(np.argmax(values))

            # Opposite horizontal alignment prevents
            # overlap when both peaks occur in one class.

            if forest == "Late_secondary":

                text_offset = (-5, 9)

                alignment = "right"

            else:

                text_offset = (5, 9)

                alignment = "left"

            ax.annotate(

                f"{values[peak]:.1f}",

                xy=(
                    positions[peak],
                    values[peak]
                ),

                xytext=text_offset,

                textcoords="offset points",

                ha=alignment,

                va="bottom",

                fontsize=9,

                fontweight="medium",

                color=COLORS[forest],

                zorder=7

            )

        # ----------------------------------------------------
        # Axis formatting
        # ----------------------------------------------------

        ax.set_xlim(
            -0.55,
            len(CLASSES) - 0.45
        )

        ax.set_ylim(panel["ylim"])

        ax.set_xticks(x)

        ax.set_xticklabels(

            DISPLAY_CLASSES,

            rotation=0,

            ha="center",

            fontsize=8.7

        )

        ax.set_ylabel(

            panel["ylabel"],

            labelpad=9

        )

        ax.yaxis.set_major_locator(

            mticker.MultipleLocator(
                panel["step"]
            )

        )

        ax.tick_params(

            axis="both",

            direction="out",

            pad=6

        )

        ax.spines["left"].set_color(AXIS)

        ax.spines["bottom"].set_color(AXIS)

        ax.spines["left"].set_linewidth(0.8)

        ax.spines["bottom"].set_linewidth(0.8)

        # ----------------------------------------------------
        # Editorial panel titles
        # ----------------------------------------------------

        ax.text(

            -0.10,

            1.075,

            panel["letter"],

            transform=ax.transAxes,

            fontsize=15,

            fontweight="bold",

            color=TEXT,

            ha="left",

            va="bottom"

        )

        ax.text(

            0.045,

            1.085,

            panel["title"],

            transform=ax.transAxes,

            fontsize=11.5,

            fontweight="medium",

            color=TEXT,

            ha="left",

            va="bottom"

        )

    # ========================================================
    # 6. SHARED LEGEND
    # ========================================================

    legend_handles = [

        Line2D(

            [0],
            [0],

            color=COLORS[forest],

            linewidth=2.0,

            marker=MARKERS[forest],

            markersize=7,

            markerfacecolor=COLORS[forest],

            markeredgecolor="white",

            markeredgewidth=0.7,

            label=LABELS[forest]

        )

        for forest in FORESTS

    ]

    fig.legend(

        handles=legend_handles,

        loc="upper center",

        bbox_to_anchor=(0.525, 0.97),

        ncol=2,

        fontsize=10.5,

        handlelength=2.2,

        handletextpad=0.7,

        columnspacing=2.6,

        frameon=False

    )

    # Shared x-axis label

    fig.supxlabel(

        "Diameter class (cm)",

        x=0.525,

        y=0.065,

        fontsize=11,

        color=TEXT

    )

    # ========================================================
    # 7. EXPORT FINAL FIGURE
    # ========================================================

    BASENAME = (
        "Fig_03_Biomass_Diameter_Lollipop_FINAL"
    )

    for extension in [
        "pdf",
        "svg",
        "png"
    ]:

        fig.savefig(

            OUT / f"{BASENAME}.{extension}",

            dpi=600,

            bbox_inches="tight",

            pad_inches=0.14,

            facecolor="white"

        )

    # Lossless publication TIFF

    with Image.open(
        OUT / f"{BASENAME}.png"
    ) as im:

        im.convert("RGB").save(

            OUT / f"{BASENAME}.tiff",

            compression="tiff_lzw",

            dpi=(600, 600)

        )

    plt.show()

# ============================================================
# 8. PRESERVE EXACT FIGURE DATA
# ============================================================

data.to_csv(

    OUT / "FIG03_LOLLIPOP_SOURCE_DATA.csv",

    index=False

)

# ============================================================
# 9. PROVENANCE
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            h.update(chunk)

    return h.hexdigest()


metadata = {

    "figure": "Figure 3",

    "title": "Forest structure and biomass allocation",

    "design": "Grouped lollipop plots",

    "redesign_cell": "104B",

    "source": source,

    "original_results_modified": False,

    "scientific_recalculation": False,

    "diameter_classes_modified": False,

    "palette": COLORS,

    "markers": MARKERS,

    "export_dpi": 600,

    "scientific_validation": {

        "tree_counts": EXPECTED_N,

        "biomass_totals_Mg": {

            forest: float(
                tables[forest]["agb_Mg"].sum()
            )

            for forest in FORESTS

        },

        "relative_abundance_sum_verified": True,

        "relative_biomass_sum_verified": True

    },

    "files": [

        {

            "filename": path.name,

            "size_bytes": path.stat().st_size,

            "sha256": sha256_file(path)

        }

        for path in sorted(OUT.glob("*"))

        if path.is_file()

        and path.suffix.lower() in [
            ".pdf",
            ".svg",
            ".png",
            ".tiff",
            ".csv"
        ]

    ]

}

with open(

    OUT / "FIG03_LOLLIPOP_PROVENANCE.json",

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        metadata,

        f,

        indent=2,

        ensure_ascii=False

    )

# ============================================================
# 10. FINAL REPORT
# ============================================================

print("\n" + "=" * 70)

print("SATIPO — FIGURE 03 FINAL LOLLIPOP")

print("=" * 70)

print("Scientific results: PRESERVED")

print("Original biomass totals: VERIFIED")

print("Original diameter classes: VERIFIED")

print("Scientific recalculation: NONE")

print("Figure style: LOLLIPOP")

print("Export resolution: 600 dpi")

print("Output directory:", OUT)

print("\nEXPORTED FILES")

for path in sorted(OUT.glob("*")):

    if path.is_file():

        print(

            f"  {path.name:60s}"

            f"{path.stat().st_size / 1024:10.1f} KB"

        )

print("\nCELL 104B COMPLETED SUCCESSFULLY")


In [ ]:

# ============================================================
# CELL 104A — RECOVER ORIGINAL FIGURE 3 DATA
# Independent recovery from verified scientific checkpoint
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

!pip -q install dill

from pathlib import Path
import json
import hashlib
import dill
import pandas as pd
import numpy as np

# ============================================================
# 1. PROJECT PATHS
# ============================================================

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

CHECKPOINT_ROOT = ROOT / "06_CHECKPOINTS"

OUT = (
    ROOT
    / "10_PUBLICATION_FIGURES"
    / "FIG_03"
)

OUT.mkdir(parents=True, exist_ok=True)

assert ROOT.exists(), (
    f"Project directory not found: {ROOT}"
)

# ============================================================
# 2. SHA256 INTEGRITY FUNCTION
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            h.update(chunk)

    return h.hexdigest()

# ============================================================
# 3. RECOVER ORIGINAL SCIENTIFIC OBJECT
# ============================================================

REQUIRED_COLUMNS = [
    "forest_type",
    "dbh_class",
    "n_trees",
    "agb_Mg",
    "abundance_pct",
    "biomass_pct"
]

fig03_data = None
source_description = None

# ------------------------------------------------------------
# Option A: Original object already in memory
# ------------------------------------------------------------

original = globals().get("dbh_biomass")

if (
    isinstance(original, pd.DataFrame)
    and set(REQUIRED_COLUMNS).issubset(original.columns)
):

    fig03_data = original.copy(deep=True)

    source_description = (
        "Original dbh_biomass object in memory"
    )

    print("Original object recovered from memory.")

# ------------------------------------------------------------
# Option B: Verified final scientific checkpoint
# ------------------------------------------------------------

if fig03_data is None:

    pointer_path = (
        CHECKPOINT_ROOT
        / "LATEST_FINAL_CHECKPOINT.json"
    )

    if pointer_path.exists():

        with open(
            pointer_path,
            "r",
            encoding="utf-8"
        ) as f:

            pointer = json.load(f)

        assert pointer.get("verified") is True, (
            "The final checkpoint is not verified."
        )

        manifest_path = Path(
            pointer["manifest"]
        )

        # Resolve relative path if necessary

        if not manifest_path.is_absolute():

            manifest_path = (
                CHECKPOINT_ROOT / manifest_path
            )

        assert manifest_path.exists(), (
            f"Manifest not found: {manifest_path}"
        )

        with open(
            manifest_path,
            "r",
            encoding="utf-8"
        ) as f:

            manifest = json.load(f)

        checkpoint_dir = manifest_path.parent

        # Locate the exact object record

        records = [

            record

            for record in manifest["saved_objects"]

            if record["name"] == "dbh_biomass"

        ]

        assert len(records) == 1, (
            "dbh_biomass was not uniquely located "
            "in the scientific checkpoint."
        )

        record = records[0]

        object_path = (
            checkpoint_dir / record["file"]
        )

        assert object_path.exists(), (
            f"Saved object not found: {object_path}"
        )

        # Verify original object integrity

        actual_hash = sha256_file(object_path)

        assert actual_hash == record["sha256"], (
            "Checkpoint object SHA256 mismatch."
        )

        print("Checkpoint SHA256: VERIFIED")

        # Restore only the original biomass table

        try:

            with open(object_path, "rb") as f:

                restored = dill.load(f)

            assert isinstance(
                restored,
                pd.DataFrame
            )

            fig03_data = restored.copy(deep=True)

            source_description = str(object_path)

            print(
                "Original dbh_biomass recovered "
                "from verified checkpoint."
            )

        except Exception as error:

            print(
                "Binary recovery unavailable:",
                str(error)
            )

            # Portable backup of the same object

            csv_relative = record.get("csv_backup")

            if csv_relative:

                csv_path = (
                    checkpoint_dir / csv_relative
                )

                if csv_path.exists():

                    fig03_data = pd.read_csv(
                        csv_path,
                        encoding="utf-8-sig"
                    )

                    # Checkpoint CSV was saved with index=True

                    fig03_data = fig03_data.loc[
                        :,
                        ~fig03_data.columns.str.startswith(
                            "Unnamed:"
                        )
                    ].copy()

                    source_description = str(csv_path)

                    print(
                        "Original table recovered "
                        "from checkpoint CSV backup."
                    )

# ------------------------------------------------------------
# Option C: Locate original portable checkpoint CSV
# ------------------------------------------------------------

if fig03_data is None:

    candidates = sorted(

        CHECKPOINT_ROOT.rglob("dbh_biomass.csv")

    )

    final_candidates = [

        p for p in candidates

        if "STAGE_01_FINAL" in str(p)

    ]

    if len(final_candidates) == 1:

        csv_path = final_candidates[0]

    elif len(candidates) == 1:

        csv_path = candidates[0]

    else:

        csv_path = None

    if csv_path is not None:

        fig03_data = pd.read_csv(
            csv_path,
            encoding="utf-8-sig"
        )

        fig03_data = fig03_data.loc[
            :,
            ~fig03_data.columns.str.startswith(
                "Unnamed:"
            )
        ].copy()

        source_description = str(csv_path)

        print(
            "Original portable table recovered."
        )

# ------------------------------------------------------------
# Stop rather than generate approximate scientific values
# ------------------------------------------------------------

if fig03_data is None:

    raise FileNotFoundError(

        "The original dbh_biomass object could not "
        "be recovered. Check the final checkpoint "
        "directory and manifest."

    )

# ============================================================
# 4. VALIDATE ORIGINAL FIGURE DATA
# ============================================================

assert set(REQUIRED_COLUMNS).issubset(
    fig03_data.columns
)

fig03_data["dbh_class"] = (
    fig03_data["dbh_class"].astype(str)
)

EXPECTED_CLASSES = [
    "10-<20",
    "20-<30",
    "30-<40",
    "40-<60",
    "60-<80",
    ">=80"
]

EXPECTED_COUNTS = {
    "Late_secondary": 541,
    "Mature": 461
}

EXPECTED_AGB = {
    "Late_secondary": 355.2266,
    "Mature": 234.3626
}

assert len(fig03_data) == 12

assert not fig03_data.duplicated(
    ["forest_type", "dbh_class"]
).any()

assert set(
    fig03_data["dbh_class"]
) == set(EXPECTED_CLASSES)

assert set(
    fig03_data["forest_type"]
) == set(EXPECTED_COUNTS)

assert fig03_data[
    REQUIRED_COLUMNS
].notna().all().all()

assert (
    fig03_data[
        [
            "n_trees",
            "agb_Mg",
            "abundance_pct",
            "biomass_pct"
        ]
    ] >= 0
).all().all()

validation_rows = []

for forest in EXPECTED_COUNTS:

    d = fig03_data.loc[
        fig03_data["forest_type"].eq(forest)
    ]

    n = int(d["n_trees"].sum())

    agb = float(d["agb_Mg"].sum())

    assert n == EXPECTED_COUNTS[forest]

    assert np.isclose(
        agb,
        EXPECTED_AGB[forest],
        atol=0.001,
        rtol=0
    )

    assert np.isclose(
        d["abundance_pct"].sum(),
        100
    )

    assert np.isclose(
        d["biomass_pct"].sum(),
        100
    )

    assert np.allclose(

        d["abundance_pct"],

        100 * d["n_trees"] / n

    )

    assert np.allclose(

        d["biomass_pct"],

        100 * d["agb_Mg"] / agb

    )

    validation_rows.append({

        "forest_type": forest,

        "n_trees": n,

        "agb_Mg": agb,

        "abundance_sum_pct":
            d["abundance_pct"].sum(),

        "biomass_sum_pct":
            d["biomass_pct"].sum()

    })

fig03_validation = pd.DataFrame(
    validation_rows
)

# ============================================================
# 5. SAVE ORIGINAL DATA FOR CELL 104B
# ============================================================

source_csv = (
    OUT
    / "FIG03_SOURCE_DIAMETER_BIOMASS.csv"
)

fig03_data.to_csv(
    source_csv,
    index=False
)

fig03_validation.to_csv(

    OUT / "FIG03_SCIENTIFIC_VALIDATION.csv",

    index=False

)

# Verify that the portable copy preserves values

reloaded = pd.read_csv(source_csv)

for column in [
    "n_trees",
    "agb_Mg",
    "abundance_pct",
    "biomass_pct"
]:

    assert np.allclose(

        reloaded[column],

        fig03_data[column],

        rtol=1e-12,

        atol=1e-12

    )

# ============================================================
# 6. PROVENANCE
# ============================================================

metadata = {

    "figure": "Figure 3",

    "original_object": "dbh_biomass",

    "original_pipeline_cells": [30, 31],

    "recovery_cell": "104A",

    "source": source_description,

    "scientific_recalculation": False,

    "original_results_modified": False,

    "validation": "PASSED",

    "source_csv_sha256": sha256_file(
        source_csv
    )

}

with open(

    OUT / "FIG03_DATA_RECOVERY.json",

    "w",

    encoding="utf-8"

) as f:

    json.dump(
        metadata,
        f,
        indent=2
    )

# ============================================================
# 7. FINAL REPORT
# ============================================================

print("\n" + "=" * 70)

print("SATIPO — FIGURE 03 DATA RECOVERY")

print("=" * 70)

print("Original data source:")

print(source_description)

print("\nScientific validation:")

display(
    fig03_validation.round(6)
)

print("\nOriginal diameter-class results:")

display(
    fig03_data[
        REQUIRED_COLUMNS
    ].round(6)
)

print("\nOriginal scientific results: PRESERVED")

print("Scientific recalculation: NONE")

print("Data integrity: VERIFIED")

print("Portable source CSV:", source_csv)

print("\nCELL 104A COMPLETED SUCCESSFULLY")

print("YOU CAN NOW RUN CELL 104B")


In [ ]:

# ============================================================
# CELL 105 — FIGURE 04: HEIGHT–DIAMETER ALLOMETRY
# Publication-quality asymmetric layout
#
# Original scientific source: Cell 48
# No model refitting, no smoothing, no new simulations
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib
import json
import dill

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from matplotlib.lines import Line2D
from PIL import Image

# ============================================================
# 1. PROJECT PATHS
# ============================================================

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

CHECKPOINT_ROOT = ROOT / "06_CHECKPOINTS"

OUT = (
    ROOT
    / "10_PUBLICATION_FIGURES"
    / "FIG_04"
)

OUT.mkdir(parents=True, exist_ok=True)

FORESTS = [
    "Late_secondary",
    "Mature"
]

# ============================================================
# 2. RECOVER EXACT ORIGINAL SCIENTIFIC OBJECTS
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            h.update(chunk)

    return h.hexdigest()


pointer_path = (
    CHECKPOINT_ROOT
    / "LATEST_FINAL_CHECKPOINT.json"
)

with open(
    pointer_path,
    "r",
    encoding="utf-8"
) as f:

    pointer = json.load(f)

assert pointer.get("verified") is True

manifest_path = Path(pointer["manifest"])

if not manifest_path.is_absolute():

    manifest_path = (
        CHECKPOINT_ROOT / manifest_path
    )

with open(
    manifest_path,
    "r",
    encoding="utf-8"
) as f:

    manifest = json.load(f)

checkpoint_dir = manifest_path.parent

sources = {}


def recover_original(name, required_columns):

    records = [

        r for r in manifest["saved_objects"]

        if r["name"] == name

    ]

    assert len(records) == 1, (
        f"Original object not uniquely found: {name}"
    )

    record = records[0]

    object_path = (
        checkpoint_dir / record["file"]
    )

    assert object_path.exists(), (
        f"Missing original object: {object_path}"
    )

    actual_hash = sha256_file(object_path)

    assert actual_hash == record["sha256"], (
        f"SHA256 mismatch: {name}"
    )

    with open(object_path, "rb") as f:

        obj = dill.load(f)

    assert isinstance(obj, pd.DataFrame)

    assert set(required_columns).issubset(
        obj.columns
    ), f"Unexpected columns in {name}"

    sources[name] = {

        "path": str(object_path),

        "sha256": actual_hash,

        "verified": True

    }

    print(
        f"Recovered and verified: {name}"
    )

    return obj.copy(deep=True)


# Observed inventory

fig04_inventory = recover_original(

    "allometry_data",

    [
        "forest_type",
        "dbh_cm",
        "height_m"
    ]

)

# Original continuous predictions

fig04_curves = recover_original(

    "allometry_curves",

    [
        "dbh_cm",
        "height_difference_m",
        "height_ratio"
    ]

)

# Original reference-diameter contrasts

fig04_reference = recover_original(

    "height_contrast",

    [
        "dbh_cm",
        "height_difference_m",
        "height_implied_AGB_ratio"
    ]

)

# Recover original model predictions from the
# bootstrap summary, which stores the same
# original fitted curves used in Cell 48.

fig04_predictions = recover_original(

    "allometry_bootstrap",

    [
        "dbh_cm",
        "secondary_height_m",
        "mature_height_m",
        "height_difference_m",
        "height_implied_AGB_ratio"
    ]

)

# ============================================================
# 3. SCIENTIFIC INTEGRITY CHECKS
# ============================================================

EXPECTED_COUNTS = {

    "Late_secondary": 541,

    "Mature": 461

}

counts = (

    fig04_inventory

    .groupby("forest_type")

    .size()

    .to_dict()

)

assert counts == EXPECTED_COUNTS

assert len(fig04_inventory) == 1002

assert fig04_inventory[
    ["dbh_cm", "height_m"]
].notna().all().all()

assert (
    fig04_inventory["dbh_cm"] >= 10
).all()

assert (
    fig04_inventory["height_m"] > 0
).all()

# Original common diameter support

COMMON_MIN = 10.0
COMMON_MAX = 89.2

fig04_curves = (

    fig04_curves

    .sort_values("dbh_cm")

    .reset_index(drop=True)

)

fig04_predictions = (

    fig04_predictions

    .sort_values("dbh_cm")

    .reset_index(drop=True)

)

fig04_reference = (

    fig04_reference

    .sort_values("dbh_cm")

    .reset_index(drop=True)

)

assert np.isclose(
    fig04_curves["dbh_cm"].min(),
    COMMON_MIN
)

assert np.isclose(
    fig04_curves["dbh_cm"].max(),
    COMMON_MAX
)

# Confirm that stored predictions reproduce
# the original height differences.

assert np.allclose(

    fig04_predictions[
        "secondary_height_m"
    ]

    -

    fig04_predictions[
        "mature_height_m"
    ],

    fig04_predictions[
        "height_difference_m"
    ],

    rtol=1e-9,

    atol=1e-9

)

# Original transformation from Cell 48:
# height ratio raised to the Chave exponent.
# This reproduces the original plotted metric.

fig04_curves["height_implied_AGB_ratio"] = (

    fig04_curves["height_ratio"] ** 0.976

)

# Cross-check the original predictions
# against the original Cell 48 curves.

assert len(fig04_curves) == len(
    fig04_predictions
)

assert np.allclose(

    fig04_curves["dbh_cm"],

    fig04_predictions["dbh_cm"]

)

assert np.allclose(

    fig04_curves["height_difference_m"],

    fig04_predictions["height_difference_m"],

    rtol=1e-9,

    atol=1e-9

)

assert np.allclose(

    fig04_curves[
        "height_implied_AGB_ratio"
    ],

    fig04_predictions[
        "height_implied_AGB_ratio"
    ],

    rtol=1e-9,

    atol=1e-9

)

# Verify reference-diameter values

assert (

    fig04_reference["dbh_cm"]

    .between(COMMON_MIN, COMMON_MAX)

    .all()

)

print("\nORIGINAL SCIENTIFIC RESULTS VERIFIED")

display(

    fig04_inventory

    .groupby("forest_type")

    .agg(

        n_trees=("dbh_cm", "size"),

        mean_dbh_cm=("dbh_cm", "mean"),

        mean_height_m=("height_m", "mean"),

        max_dbh_cm=("dbh_cm", "max")

    )

    .round(6)

)

display(fig04_reference.round(6))

# ============================================================
# 4. EDITORIAL VISUAL IDENTITY
# ============================================================

COLORS = {

    "Late_secondary": "#C46D5E",

    "Mature": "#285B78"

}

LABELS = {

    "Late_secondary":
        "Late-secondary forest",

    "Mature":
        "Mature forest"

}

MARKERS = {

    "Late_secondary": "o",

    "Mature": "D"

}

# Derived contrasts have their own neutral
# colors, rather than representing a forest.

CONTRAST_COLOR = "#46566B"

RATIO_COLOR = "#79628D"

TEXT = "#26333D"

MUTED = "#69757F"

GRID = "#E6EAED"

AXIS = "#A4ADB5"

STYLE = {

    "font.family": "DejaVu Sans",

    "font.size": 10,

    "axes.labelsize": 10.5,

    "axes.labelcolor": TEXT,

    "axes.edgecolor": AXIS,

    "axes.linewidth": 0.8,

    "axes.spines.top": False,

    "axes.spines.right": False,

    "xtick.labelsize": 9.5,

    "ytick.labelsize": 9.5,

    "xtick.color": TEXT,

    "ytick.color": TEXT,

    "xtick.major.size": 3.5,

    "ytick.major.size": 3.5,

    "xtick.major.width": 0.8,

    "ytick.major.width": 0.8,

    "legend.frameon": False,

    "figure.facecolor": "white",

    "axes.facecolor": "white",

    "savefig.facecolor": "white",

    "pdf.fonttype": 42,

    "ps.fonttype": 42,

    "svg.fonttype": "none"

}

# ============================================================
# 5. FORMATTING HELPERS
# ============================================================

def style_axis(ax):

    ax.set_axisbelow(True)

    ax.grid(

        axis="y",

        color=GRID,

        linewidth=0.65,

        linestyle="-"

    )

    ax.tick_params(

        axis="both",

        direction="out",

        pad=5

    )

    for side in ["left", "bottom"]:

        ax.spines[side].set_color(AXIS)

        ax.spines[side].set_linewidth(0.8)


def panel_title(ax, letter, title):

    ax.text(

        -0.10,

        1.055,

        letter,

        transform=ax.transAxes,

        fontsize=15,

        fontweight="bold",

        color=TEXT,

        ha="left",

        va="bottom"

    )

    ax.text(

        0.035,

        1.065,

        title,

        transform=ax.transAxes,

        fontsize=11.5,

        fontweight="medium",

        color=TEXT,

        ha="left",

        va="bottom"

    )


# ============================================================
# 6. BUILD ASYMMETRIC MULTIPANEL FIGURE
# ============================================================

with plt.rc_context(STYLE):

    fig = plt.figure(

        figsize=(12.8, 7.1),

        facecolor="white"

    )

    gs = fig.add_gridspec(

        nrows=2,

        ncols=2,

        width_ratios=[1.48, 1.0],

        height_ratios=[1, 1],

        left=0.085,

        right=0.975,

        bottom=0.115,

        top=0.81,

        wspace=0.35,

        hspace=0.48

    )

    ax_a = fig.add_subplot(gs[:, 0])

    ax_b = fig.add_subplot(gs[0, 1])

    ax_c = fig.add_subplot(

        gs[1, 1],

        sharex=ax_b

    )

    # ========================================================
    # PANEL A — OBSERVED HEIGHT AND FITTED ALLOMETRY
    # ========================================================

    prediction_columns = {

        "Late_secondary":
            "secondary_height_m",

        "Mature":
            "mature_height_m"

    }

    # Mark the region beyond common support.
    # Observations are retained, but fitted curves
    # are not extrapolated into this region.

    max_observed = float(

        fig04_inventory["dbh_cm"].max()

    )

    ax_a.axvspan(

        COMMON_MAX,

        max_observed + 5,

        color="#EDF0F2",

        alpha=0.65,

        linewidth=0,

        zorder=0

    )

    for forest in FORESTS:

        d = fig04_inventory.loc[

            fig04_inventory[
                "forest_type"
            ].eq(forest)

        ]

        # All original observed individuals.
        # No jitter or synthetic points.

        ax_a.scatter(

            d["dbh_cm"],

            d["height_m"],

            color=COLORS[forest],

            marker=MARKERS[forest],

            alpha=0.25,

            s=18,

            edgecolors="none",

            rasterized=True,

            zorder=2

        )

        # Original Weibull predictions.
        # No refitting or interpolation.

        ax_a.plot(

            fig04_predictions["dbh_cm"],

            fig04_predictions[
                prediction_columns[forest]
            ],

            color=COLORS[forest],

            linewidth=3.0,

            solid_capstyle="round",

            zorder=5

        )

    ax_a.axvline(

        COMMON_MAX,

        color=MUTED,

        linestyle=(0, (4, 4)),

        linewidth=1.0,

        zorder=1

    )

    # Indicate the boundary of shared support

    ax_a.annotate(

        "Common support limit\n89.2 cm",

        xy=(COMMON_MAX, 0.94),

        xycoords=(

            "data",

            "axes fraction"

        ),

        xytext=(-8, 0),

        textcoords="offset points",

        ha="right",

        va="top",

        fontsize=9,

        color=MUTED,

        linespacing=1.35

    )

    ax_a.text(

        (COMMON_MAX + max_observed + 5) / 2,

        0.04,

        "Outside shared\nsupport",

        transform=ax_a.get_xaxis_transform(),

        ha="center",

        va="bottom",

        fontsize=8.5,

        color=MUTED

    )

    # Sample sizes are original inventory counts

    ax_a.text(

        0.035,

        0.96,

        "Late-secondary: n = 541\nMature: n = 461",

        transform=ax_a.transAxes,

        ha="left",

        va="top",

        fontsize=9,

        color=MUTED,

        linespacing=1.5

    )

    ax_a.set_xlabel(

        "Diameter (cm)",

        labelpad=9

    )

    ax_a.set_ylabel(

        "Tree height (m)",

        labelpad=9

    )

    ax_a.set_xlim(

        5,

        max_observed + 5

    )

    max_height = max(

        fig04_inventory["height_m"].max(),

        fig04_predictions[
            "secondary_height_m"
        ].max(),

        fig04_predictions[
            "mature_height_m"
        ].max()

    )

    ax_a.set_ylim(

        0,

        np.ceil(
            (max_height + 3) / 10
        ) * 10

    )

    ax_a.xaxis.set_major_locator(

        mticker.MultipleLocator(20)

    )

    ax_a.yaxis.set_major_locator(

        mticker.MultipleLocator(10)

    )

    style_axis(ax_a)

    panel_title(

        ax_a,

        "a",

        "Height–diameter allometry"

    )

    # ========================================================
    # PANEL B — HEIGHT CONTRAST
    # ========================================================

    ax_b.axhline(

        0,

        color=MUTED,

        linestyle=(0, (4, 3)),

        linewidth=1.0,

        zorder=1

    )

    ax_b.plot(

        fig04_curves["dbh_cm"],

        fig04_curves["height_difference_m"],

        color=CONTRAST_COLOR,

        linewidth=2.7,

        solid_capstyle="round",

        zorder=4

    )

    ax_b.scatter(

        fig04_reference["dbh_cm"],

        fig04_reference[
            "height_difference_m"
        ],

        s=43,

        color=CONTRAST_COLOR,

        edgecolor="white",

        linewidth=0.8,

        zorder=5

    )

    # Annotate the original reference at 80 cm

    ref80 = fig04_reference.loc[

        np.isclose(

            fig04_reference["dbh_cm"],

            80

        )

    ]

    assert len(ref80) == 1

    diff80 = float(

        ref80[
            "height_difference_m"
        ].iloc[0]

    )

    ax_b.annotate(

        f"+{diff80:.1f} m",

        xy=(80, diff80),

        xytext=(-10, 12),

        textcoords="offset points",

        ha="right",

        va="bottom",

        color=CONTRAST_COLOR,

        fontsize=10,

        fontweight="medium",

        arrowprops=dict(

            arrowstyle="-",

            color=CONTRAST_COLOR,

            linewidth=0.8

        )

    )

    ax_b.set_ylabel(

        r"$H_{\mathrm{S}}-H_{\mathrm{M}}$ (m)",

        labelpad=8

    )

    ax_b.set_ylim(

        -0.8,

        max(

            18,

            fig04_curves[
                "height_difference_m"
            ].max() + 1.5

        )

    )

    ax_b.yaxis.set_major_locator(

        mticker.MultipleLocator(5)

    )

    ax_b.tick_params(

        axis="x",

        labelbottom=False

    )

    style_axis(ax_b)

    panel_title(

        ax_b,

        "b",

        "Height contrast at equal diameter"

    )

    # ========================================================
    # PANEL C — HEIGHT-IMPLIED BIOMASS RATIO
    # ========================================================

    ax_c.axhline(

        1.0,

        color=MUTED,

        linestyle=(0, (4, 3)),

        linewidth=1.0,

        zorder=1

    )

    ax_c.plot(

        fig04_curves["dbh_cm"],

        fig04_curves[
            "height_implied_AGB_ratio"
        ],

        color=RATIO_COLOR,

        linewidth=2.7,

        solid_capstyle="round",

        zorder=4

    )

    ax_c.scatter(

        fig04_reference["dbh_cm"],

        fig04_reference[
            "height_implied_AGB_ratio"
        ],

        s=43,

        color=RATIO_COLOR,

        edgecolor="white",

        linewidth=0.8,

        zorder=5

    )

    ratio80 = float(

        ref80[
            "height_implied_AGB_ratio"
        ].iloc[0]

    )

    ax_c.annotate(

        f"{ratio80:.2f}×",

        xy=(80, ratio80),

        xytext=(-10, 11),

        textcoords="offset points",

        ha="right",

        va="bottom",

        color=RATIO_COLOR,

        fontsize=10,

        fontweight="medium",

        arrowprops=dict(

            arrowstyle="-",

            color=RATIO_COLOR,

            linewidth=0.8

        )

    )

    # Equality reference

    ax_c.text(

        COMMON_MIN + 2,

        1.012,

        "Equal height-implied biomass",

        fontsize=8,

        color=MUTED,

        ha="left",

        va="bottom"

    )

    ax_c.set_xlabel(

        "Diameter (cm)",

        labelpad=9

    )

    ax_c.set_ylabel(

        r"$(H_{\mathrm{S}}/H_{\mathrm{M}})^{0.976}$",

        labelpad=8

    )

    ax_c.set_ylim(

        0.98,

        max(

            1.42,

            fig04_curves[
                "height_implied_AGB_ratio"
            ].max() + 0.04

        )

    )

    ax_c.yaxis.set_major_locator(

        mticker.MultipleLocator(0.10)

    )

    ax_c.yaxis.set_major_formatter(

        mticker.FormatStrFormatter("%.1f")

    )

    style_axis(ax_c)

    panel_title(

        ax_c,

        "c",

        "Height-implied biomass ratio"

    )

    # Shared support for the comparative panels

    ax_b.set_xlim(

        COMMON_MIN - 2,

        COMMON_MAX + 2

    )

    ax_c.set_xticks(

        [10, 30, 50, 70, 90]

    )

    # ========================================================
    # 7. SHARED FOREST LEGEND
    # ========================================================

    legend_handles = [

        Line2D(

            [0],

            [0],

            color=COLORS[forest],

            linewidth=2.8,

            marker=MARKERS[forest],

            markersize=6.5,

            markerfacecolor=COLORS[forest],

            markeredgecolor="white",

            markeredgewidth=0.6,

            label=LABELS[forest]

        )

        for forest in FORESTS

    ]

    fig.legend(

        handles=legend_handles,

        loc="upper center",

        bbox_to_anchor=(0.52, 0.965),

        ncol=2,

        fontsize=11,

        handlelength=2.4,

        handletextpad=0.7,

        columnspacing=2.5,

        frameon=False

    )

    # ========================================================
    # 8. EXPORT PUBLICATION FILES
    # ========================================================

    BASENAME = (

        "Fig_04_Height_Diameter_Allometry_FINAL"

    )

    for extension in [

        "pdf",

        "svg",

        "png"

    ]:

        fig.savefig(

            OUT / f"{BASENAME}.{extension}",

            dpi=600,

            bbox_inches="tight",

            pad_inches=0.14,

            facecolor="white"

        )

    with Image.open(

        OUT / f"{BASENAME}.png"

    ) as im:

        im.convert("RGB").save(

            OUT / f"{BASENAME}.tiff",

            compression="tiff_lzw",

            dpi=(600, 600)

        )

    plt.show()

# ============================================================
# 9. EXPORT EXACT ORIGINAL SCIENTIFIC SOURCES
# ============================================================

fig04_inventory.to_csv(

    OUT / "FIG04_SOURCE_OBSERVED_TREES.csv",

    index=False

)

fig04_curves.to_csv(

    OUT / "FIG04_SOURCE_CONTINUOUS_CONTRASTS.csv",

    index=False

)

fig04_reference.to_csv(

    OUT / "FIG04_SOURCE_REFERENCE_DIAMETERS.csv",

    index=False

)

fig04_predictions.to_csv(

    OUT / "FIG04_SOURCE_WEIBULL_PREDICTIONS.csv",

    index=False

)

# ============================================================
# 10. PROVENANCE MANIFEST
# ============================================================

metadata = {

    "figure": "Figure 4",

    "title": (
        "Height–diameter allometry and "
        "height-implied biomass contrast"
    ),

    "original_pipeline_cells": [

        48,

        51

    ],

    "redesign_cell": 105,

    "design": "Asymmetric 1+2 panel layout",

    "original_model": "Weibull",

    "model_refitting": False,

    "scientific_results_modified": False,

    "synthetic_observations": False,

    "curve_smoothing": False,

    "extrapolation": False,

    "bootstrap_recalculation": False,

    "inventory_counts": EXPECTED_COUNTS,

    "common_diameter_support_cm": [

        COMMON_MIN,

        COMMON_MAX

    ],

    "height_implied_biomass_exponent": 0.976,

    "reference_at_80cm": {

        "height_difference_m": diff80,

        "height_implied_AGB_ratio": ratio80

    },

    "sources": sources,

    "palette": COLORS,

    "export_dpi": 600,

    "files": [

        {

            "filename": path.name,

            "size_bytes": path.stat().st_size,

            "sha256": sha256_file(path)

        }

        for path in sorted(OUT.glob("*"))

        if path.is_file()

        and path.suffix.lower() in [

            ".pdf",

            ".svg",

            ".png",

            ".tiff",

            ".csv"

        ]

    ]

}

with open(

    OUT / "FIG04_PROVENANCE.json",

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        metadata,

        f,

        indent=2,

        ensure_ascii=False

    )

# ============================================================
# 11. FINAL REPORT
# ============================================================

print("\n" + "=" * 72)

print("SATIPO — FIGURE 04 PUBLICATION REMAKE")

print("=" * 72)

print("Original checkpoint objects: VERIFIED")

print("Original tree inventory: 1002 individuals")

print("Original Weibull predictions: PRESERVED")

print("Original height contrasts: PRESERVED")

print("Original biomass ratios: PRESERVED")

print("Common diameter support: 10–89.2 cm")

print("Model refitting: NONE")

print("Scientific recalculation: NONE")

print("Curve extrapolation: NONE")

print(
    f"Height difference at 80 cm: {diff80:.6f} m"
)

print(
    f"Height-implied AGB ratio at 80 cm: {ratio80:.6f}"
)

print("\nOutput directory:", OUT)

print("\nEXPORTED FILES")

for path in sorted(OUT.glob("*")):

    if path.is_file():

        print(

            f"  {path.name:60s}"

            f"{path.stat().st_size / 1024:10.1f} KB"

        )

print("\nCELL 105 COMPLETED SUCCESSFULLY")


In [ ]:

# ============================================================
# CELL 106A — EXACT TAXONOMIC DATA RECOVERY
# Inspect original scientific checkpoint objects
# ============================================================

from pathlib import Path
import json
import hashlib
import dill
import pandas as pd
import numpy as np

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

CHECKPOINT_ROOT = ROOT / "06_CHECKPOINTS"

OUT = (
    ROOT
    / "10_PUBLICATION_FIGURES"
    / "FIG_05"
    / "SOURCE_DATA"
)

OUT.mkdir(parents=True, exist_ok=True)

# ============================================================
# 1. LOAD VERIFIED CHECKPOINT
# ============================================================

with open(
    CHECKPOINT_ROOT
    / "LATEST_FINAL_CHECKPOINT.json",
    encoding="utf-8"
) as f:
    pointer = json.load(f)

assert pointer.get("verified") is True

manifest_path = Path(pointer["manifest"])

if not manifest_path.is_absolute():
    manifest_path = CHECKPOINT_ROOT / manifest_path

with open(
    manifest_path,
    encoding="utf-8"
) as f:
    manifest = json.load(f)

checkpoint_dir = manifest_path.parent

records = {
    r["name"]: r
    for r in manifest["saved_objects"]
}

# ============================================================
# 2. VERIFIED OBJECT RECOVERY
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


def recover_exact(name):

    assert name in records, (
        f"Object not found: {name}"
    )

    record = records[name]

    path = checkpoint_dir / record["file"]

    assert path.exists()

    actual_hash = sha256_file(path)

    assert actual_hash == record["sha256"], (
        f"SHA256 mismatch: {name}"
    )

    with open(path, "rb") as f:
        obj = dill.load(f)

    print("\n" + "=" * 72)

    print("OBJECT:", name)

    print("SHA256: VERIFIED")

    print("TYPE:", type(obj).__name__)

    if isinstance(obj, pd.DataFrame):

        print("SHAPE:", obj.shape)

        print("COLUMNS:")

        print(obj.columns.tolist())

        print("\nDATA TYPES:")

        print(obj.dtypes.to_string())

        print("\nFIRST 8 ROWS:")

        display(obj.head(8))

        # Preserve the exact original object

        obj.to_csv(
            OUT / f"{name}_ORIGINAL.csv",
            index=False
        )

        # Preserve original index information
        # in a separate portable copy

        obj.to_csv(
            OUT / f"{name}_WITH_INDEX.csv",
            index=True
        )

        print("\nNUMERIC COLUMN SUMS:")

        print(
            obj.select_dtypes(
                include="number"
            ).sum().to_string()
        )

        if "forest_type" in obj.columns:

            print("\nFOREST GROUPS:")

            display(
                obj.groupby(
                    "forest_type",
                    dropna=False
                ).size()
            )

    elif isinstance(obj, dict):

        print("DICTIONARY KEYS:")

        print(list(obj.keys()))

        for key, value in obj.items():

            print(
                key,
                type(value).__name__,
                getattr(value, "shape", None)
            )

            if isinstance(value, pd.DataFrame):

                print(value.columns.tolist())

                display(value.head(5))

    else:

        print(repr(obj)[:3000])

    return obj


# ============================================================
# 3. RECOVER EXACT TAXONOMIC OBJECTS
# ============================================================

fig05_original_objects = {}

for name in [

    "taxon_biomass",

    "taxon_contributions",

    "taxon_summary"

]:

    fig05_original_objects[name] = recover_exact(
        name
    )

# Preserve original names in the current session

taxon_biomass = fig05_original_objects[
    "taxon_biomass"
]

taxon_contributions = fig05_original_objects[
    "taxon_contributions"
]

taxon_summary = fig05_original_objects[
    "taxon_summary"
]

# ============================================================
# 4. LOCATE ORIGINAL EXCLUSION AND CONCENTRATION
# ============================================================

print("\n" + "=" * 72)

print("RELATED CHECKPOINT OBJECTS")

print("=" * 72)

related_names = [

    name for name in records

    if any(

        term in name.lower()

        for term in [

            "exclusion",

            "dominant",

            "concentration",

            "cumulative",

            "taxon_gap",

            "biomass_gap",

            "species_gap"

        ]

    )

]

for name in sorted(related_names):

    print(name)

# ============================================================
# 5. LOCATE EXACT ORIGINAL CSV EXPORTS
# ============================================================

print("\n" + "=" * 72)

print("ORIGINAL PIPELINE CSV FILES")

print("=" * 72)

expected_files = [

    "DOMINANT_SPECIES_EXCLUSION.csv",

    "TAXONOMIC_BIOMASS_CONCENTRATION.csv",

    "TAXONOMIC_BIOMASS_GAP.csv"

]

search_roots = [

    ROOT / "03_ANALYSIS",

    ROOT / "05_OUTPUTS",

    ROOT / "07_MANUSCRIPT_READY",

    ROOT / "08_FINAL_MANUSCRIPT_PACKAGE"

]

fig05_original_csvs = {}

for filename in expected_files:

    found = []

    for folder in search_roots:

        if folder.exists():

            found.extend(
                folder.rglob(filename)
            )

    found = sorted(set(found))

    print("\nFILE:", filename)

    if not found:

        print("NOT FOUND")

        continue

    for path in found:

        print("PATH:", path)

    # Inspect first original exported copy

    path = found[0]

    df = pd.read_csv(path)

    fig05_original_csvs[filename] = df

    print("SHAPE:", df.shape)

    print("COLUMNS:", df.columns.tolist())

    display(df.head(8))

    df.to_csv(
        OUT / filename,
        index=False
    )

# ============================================================
# 6. FINAL REPORT
# ============================================================

print("\n" + "=" * 72)

print("SATIPO — FIGURE 05 SOURCE AUDIT")

print("=" * 72)

print(
    "Verified checkpoint objects:",
    len(fig05_original_objects)
)

print(
    "Original CSV tables recovered:",
    len(fig05_original_csvs)
)

print("Scientific recalculation: NONE")

print("Original objects modified: NONE")

print("Source directory:", OUT)

print("\nCELL 106A COMPLETED SUCCESSFULLY")


In [ ]:

# ============================================================
# CELL 106B — FIGURE 05 FINAL PUBLICATION REMAKE
# TAXONOMIC–BIOMASS DECOUPLING
#
# Exact scientific sources confirmed by Cell 106A
#
# a) Abundance–biomass decoupling
# b) Cumulative taxonomic biomass concentration
# c) Diverging taxonomic biomass contributions
# d) Dominant taxon exclusion waterfall
#
# NO biomass recalculation
# NO taxonomic reassignment
# NO cumulative curve recalculation
# ============================================================

from pathlib import Path
import json
import hashlib
import dill

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patheffects as pe

from matplotlib.lines import Line2D
from PIL import Image

# ============================================================
# 1. PROJECT PATHS
# ============================================================

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

CHECKPOINT_ROOT = ROOT / "06_CHECKPOINTS"

FUNCTIONAL_DIR = (
    ROOT
    / "03_ANALYSIS"
    / "FUNCTIONAL_ECOLOGY"
)

OUT = (
    ROOT
    / "10_PUBLICATION_FIGURES"
    / "FIG_05"
)

OUT.mkdir(parents=True, exist_ok=True)

FORESTS = [
    "Late_secondary",
    "Mature"
]

EXPECTED_N = {
    "Late_secondary": 541,
    "Mature": 461
}

EXPECTED_AGB = {
    "Late_secondary": 355.226643,
    "Mature": 234.362563
}

EXPECTED_RICHNESS = {
    "Late_secondary": 91,
    "Mature": 110
}

DOMINANT = "Pseudosenefeldera inclinata"

# ============================================================
# 2. VERIFIED CHECKPOINT RECOVERY
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


pointer_path = (
    CHECKPOINT_ROOT
    / "LATEST_FINAL_CHECKPOINT.json"
)

with open(
    pointer_path,
    encoding="utf-8"
) as f:

    pointer = json.load(f)

assert pointer.get("verified") is True

manifest_path = Path(pointer["manifest"])

if not manifest_path.is_absolute():

    manifest_path = (
        CHECKPOINT_ROOT / manifest_path
    )

with open(
    manifest_path,
    encoding="utf-8"
) as f:

    manifest = json.load(f)

checkpoint_dir = manifest_path.parent

records = {
    r["name"]: r
    for r in manifest["saved_objects"]
}

sources = {}


def recover_exact(name):

    assert name in records, (
        f"Missing checkpoint object: {name}"
    )

    record = records[name]

    path = checkpoint_dir / record["file"]

    assert path.exists()

    actual_hash = sha256_file(path)

    assert actual_hash == record["sha256"], (
        f"SHA256 mismatch: {name}"
    )

    with open(path, "rb") as f:

        obj = dill.load(f)

    assert isinstance(obj, pd.DataFrame)

    sources[name] = {

        "path": str(path),

        "sha256": actual_hash,

        "checkpoint_verified": True

    }

    print(
        f"Recovered and verified: {name}"
    )

    return obj.copy(deep=True)


# Exact original table:
# 201 forest–taxon combinations, 14 columns.

fig05_taxa = recover_exact(
    "taxon_biomass"
)

# ============================================================
# 3. RECOVER ORIGINAL CSV TABLES
# ============================================================

gap_path = (
    FUNCTIONAL_DIR
    / "TAXONOMIC_BIOMASS_GAP.csv"
)

exclusion_path = (
    FUNCTIONAL_DIR
    / "DOMINANT_SPECIES_EXCLUSION.csv"
)

assert gap_path.exists(), str(gap_path)

assert exclusion_path.exists(), str(exclusion_path)

fig05_gap = pd.read_csv(gap_path)

fig05_exclusion_all = pd.read_csv(
    exclusion_path
)

sources["taxonomic_biomass_gap"] = {

    "path": str(gap_path),

    "sha256": sha256_file(gap_path),

    "source_type": "Original pipeline CSV"

}

sources["dominant_species_exclusion"] = {

    "path": str(exclusion_path),

    "sha256": sha256_file(exclusion_path),

    "source_type": "Original pipeline CSV"

}

print("Original biomass gap CSV recovered.")

print("Original exclusion CSV recovered.")

# ============================================================
# 4. VALIDATE EXACT ORIGINAL SCHEMAS
# ============================================================

required_taxon_columns = [

    "forest_type",

    "functional_taxon",

    "n_trees",

    "agb_Mg",

    "abundance_pct",

    "biomass_pct",

    "biomass_rank",

    "cumulative_biomass_pct"

]

required_gap_columns = [

    "functional_taxon",

    "Late_secondary",

    "Mature",

    "biomass_difference_Mg"

]

required_exclusion_columns = [

    "scenario",

    "forest_type",

    "n_trees",

    "AGB_Mg"

]

assert set(
    required_taxon_columns
).issubset(fig05_taxa.columns)

assert set(
    required_gap_columns
).issubset(fig05_gap.columns)

assert set(
    required_exclusion_columns
).issubset(fig05_exclusion_all.columns)

assert fig05_taxa.shape == (201, 14)

assert fig05_gap.shape == (161, 9)

assert fig05_exclusion_all.shape == (4, 19)

assert not fig05_taxa.duplicated(

    ["forest_type", "functional_taxon"]

).any()

assert not fig05_gap[
    "functional_taxon"
].duplicated().any()

assert fig05_taxa[
    required_taxon_columns
].notna().all().all()

# ============================================================
# 5. SCIENTIFIC INTEGRITY CHECKS
# ============================================================

validation_rows = []

for forest in FORESTS:

    d = fig05_taxa.loc[

        fig05_taxa[
            "forest_type"
        ].eq(forest)

    ].copy()

    n = int(d["n_trees"].sum())

    agb = float(d["agb_Mg"].sum())

    richness = len(d)

    assert n == EXPECTED_N[forest]

    assert richness == EXPECTED_RICHNESS[forest]

    assert np.isclose(

        agb,

        EXPECTED_AGB[forest],

        atol=1e-6,

        rtol=0

    )

    assert np.isclose(

        d["abundance_pct"].sum(),

        100,

        atol=1e-8

    )

    assert np.isclose(

        d["biomass_pct"].sum(),

        100,

        atol=1e-8

    )

    # Validate existing percentages without
    # replacing the original values.

    assert np.allclose(

        d["abundance_pct"],

        100 * d["n_trees"] / n

    )

    assert np.allclose(

        d["biomass_pct"],

        100 * d["agb_Mg"] / agb

    )

    # Validate original biomass ranks and
    # cumulative biomass values.

    ranked = d.sort_values(
        "biomass_rank"
    )

    assert np.array_equal(

        ranked["biomass_rank"].to_numpy(),

        np.arange(1, richness + 1)

    )

    assert np.allclose(

        ranked[
            "cumulative_biomass_pct"
        ].to_numpy(),

        ranked[
            "biomass_pct"
        ].cumsum().to_numpy()

    )

    assert np.isclose(

        ranked[
            "cumulative_biomass_pct"
        ].iloc[-1],

        100

    )

    top10 = float(

        ranked.loc[

            ranked[
                "biomass_rank"
            ].eq(10),

            "cumulative_biomass_pct"

        ].iloc[0]

    )

    validation_rows.append({

        "forest_type": forest,

        "n_trees": n,

        "n_taxa": richness,

        "AGB_Mg": agb,

        "abundance_sum_pct":
            d["abundance_pct"].sum(),

        "biomass_sum_pct":
            d["biomass_pct"].sum(),

        "top10_biomass_pct": top10

    })

fig05_validation = pd.DataFrame(
    validation_rows
)

# ============================================================
# 6. VALIDATE ORIGINAL TAXONOMIC BIOMASS GAP
# ============================================================

# The original gap table already contains
# both forest biomass values and differences.
# We use these directly.

assert np.allclose(

    fig05_gap["Late_secondary"]

    - fig05_gap["Mature"],

    fig05_gap["biomass_difference_Mg"],

    atol=1e-8

)

assert np.isclose(

    fig05_gap["Late_secondary"].sum(),

    EXPECTED_AGB["Late_secondary"],

    atol=1e-6

)

assert np.isclose(

    fig05_gap["Mature"].sum(),

    EXPECTED_AGB["Mature"],

    atol=1e-6

)

assert np.isclose(

    fig05_gap[
        "biomass_difference_Mg"
    ].sum(),

    EXPECTED_AGB["Late_secondary"]

    - EXPECTED_AGB["Mature"],

    atol=1e-6

)

# Cross-check all 161 taxa against the
# verified checkpoint biomass table.

wide = fig05_taxa.pivot(

    index="functional_taxon",

    columns="forest_type",

    values="agb_Mg"

).fillna(0)

gap_indexed = fig05_gap.set_index(
    "functional_taxon"
)

assert set(wide.index) == set(
    gap_indexed.index
)

for forest in FORESTS:

    assert np.allclose(

        wide[forest].sort_index(),

        gap_indexed[forest].sort_index(),

        atol=1e-8

    )

# ============================================================
# 7. EXACT DOMINANT TAXON EXCLUSION SCENARIOS
# ============================================================

def select_scenario(scenario, forest):

    d = fig05_exclusion_all.loc[

        fig05_exclusion_all[
            "scenario"
        ].eq(scenario)

        & fig05_exclusion_all[
            "forest_type"
        ].eq(forest)

    ]

    assert len(d) == 1, (
        f"Scenario not uniquely identified: "
        f"{scenario}, {forest}"
    )

    return d.copy()


secondary_full = select_scenario(

    "Full_inventory",

    "Late_secondary"

)

mature_full = select_scenario(

    "Full_inventory",

    "Mature"

)

secondary_reduced = select_scenario(

    "Without_Pseudosenefeldera",

    "Late_secondary"

)

mature_reduced = select_scenario(

    "Without_Pseudosenefeldera",

    "Mature"

)

full_secondary = float(

    secondary_full["AGB_Mg"].iloc[0]

)

full_mature = float(

    mature_full["AGB_Mg"].iloc[0]

)

reduced_secondary = float(

    secondary_reduced["AGB_Mg"].iloc[0]

)

assert int(

    secondary_full["n_trees"].iloc[0]

) == 541

assert int(

    secondary_reduced["n_trees"].iloc[0]

) == 368

assert int(

    mature_full["n_trees"].iloc[0]

) == 461

assert np.isclose(

    full_secondary,

    355.226643,

    atol=1e-6

)

assert np.isclose(

    full_mature,

    234.362563,

    atol=1e-6

)

assert np.isclose(

    reduced_secondary,

    316.272084,

    atol=1e-6

)

# Confirm that the mature inventory
# remains unchanged under exclusion.

assert np.isclose(

    mature_reduced["AGB_Mg"].iloc[0],

    full_mature

)

# Recover the exact original dominant taxon.

dominant_row = fig05_taxa.loc[

    fig05_taxa[
        "forest_type"
    ].eq("Late_secondary")

    & fig05_taxa[
        "functional_taxon"
    ].eq(DOMINANT)

]

assert len(dominant_row) == 1

dominant_n = int(

    dominant_row["n_trees"].iloc[0]

)

dominant_agb = float(

    dominant_row["agb_Mg"].iloc[0]

)

assert dominant_n == 173

assert np.isclose(

    dominant_agb,

    38.954559,

    atol=1e-6

)

removed_agb = (

    full_secondary - reduced_secondary

)

assert np.isclose(

    removed_agb,

    dominant_agb,

    atol=1e-8

)

# Display-only arithmetic for annotation.

remaining_excess = (

    100

    * (reduced_secondary / full_mature - 1)

)

fig05_exclusion = pd.concat(

    [

        secondary_full,

        secondary_reduced,

        mature_full

    ],

    ignore_index=True

)

print("\nSCIENTIFIC VALIDATION: PASSED")

display(
    fig05_validation.round(6)
)

print(
    f"Dominant taxon: {DOMINANT}"
)

print(
    f"Dominant stems: {dominant_n}"
)

print(
    f"Dominant biomass: {dominant_agb:.6f} Mg"
)

print(
    f"Remaining secondary biomass: "
    f"{reduced_secondary:.6f} Mg"
)

print(
    f"Remaining excess vs. mature: "
    f"{remaining_excess:.4f}%"
)

# ============================================================
# 8. PRESERVE ORIGINAL PANEL C SELECTION
# ============================================================

# Same selection as the original Figure 5:
# eight largest positive contributions and
# four largest negative contributions.

positive = (

    fig05_gap.loc[

        fig05_gap[
            "biomass_difference_Mg"
        ].gt(0)

    ]

    .nlargest(

        8,

        "biomass_difference_Mg"

    )

)

negative = (

    fig05_gap.loc[

        fig05_gap[
            "biomass_difference_Mg"
        ].lt(0)

    ]

    .nsmallest(

        4,

        "biomass_difference_Mg"

    )

)

fig05_selected_gap = (

    pd.concat(

        [positive, negative],

        ignore_index=True

    )

    .sort_values(

        "biomass_difference_Mg",

        ascending=False

    )

    .reset_index(drop=True)

)

assert len(fig05_selected_gap) == 12

print("\nSelected taxonomic contributors:")

display(

    fig05_selected_gap[

        [

            "functional_taxon",

            "Late_secondary",

            "Mature",

            "biomass_difference_Mg"

        ]

    ].round(6)

)

# ============================================================
# 9. PUBLICATION VISUAL IDENTITY
# ============================================================

COLORS = {

    "Late_secondary": "#C46D5E",

    "Mature": "#285B78"

}

LABELS = {

    "Late_secondary":
        "Late-secondary forest",

    "Mature":
        "Mature forest"

}

MARKERS = {

    "Late_secondary": "o",

    "Mature": "D"

}

TEXT = "#26333D"

MUTED = "#69757F"

GRID = "#E6EAED"

AXIS = "#A4ADB5"

STYLE = {

    "font.family": "DejaVu Sans",

    "font.size": 10,

    "axes.labelsize": 10.5,

    "axes.labelcolor": TEXT,

    "axes.edgecolor": AXIS,

    "axes.linewidth": 0.8,

    "axes.spines.top": False,

    "axes.spines.right": False,

    "xtick.labelsize": 9.5,

    "ytick.labelsize": 9.5,

    "xtick.color": TEXT,

    "ytick.color": TEXT,

    "legend.frameon": False,

    "figure.facecolor": "white",

    "axes.facecolor": "white",

    "savefig.facecolor": "white",

    "pdf.fonttype": 42,

    "ps.fonttype": 42,

    "svg.fonttype": "none"

}

# ============================================================
# 10. GRAPHICAL HELPERS
# ============================================================

def style_axis(ax, grid_axis="y"):

    ax.set_axisbelow(True)

    ax.grid(

        axis=grid_axis,

        color=GRID,

        linewidth=0.65

    )

    ax.tick_params(

        direction="out",

        pad=5,

        length=3

    )


def panel_title(ax, letter, title):

    ax.text(

        -0.09,

        1.065,

        letter,

        transform=ax.transAxes,

        fontsize=15,

        fontweight="bold",

        color=TEXT,

        ha="left",

        va="bottom"

    )

    ax.text(

        0.025,

        1.075,

        title,

        transform=ax.transAxes,

        fontsize=11.5,

        fontweight="medium",

        color=TEXT,

        ha="left",

        va="bottom"

    )


def scientific_label(name, abbreviated=False):

    parts = str(name).split()

    if len(parts) >= 2:

        genus = parts[0]

        species = parts[1]

        if abbreviated:

            genus = genus[0] + "."

        if species.lower() in [

            "sp.",

            "spp."

        ]:

            remainder = " ".join(
                parts[1:]
            )

            return (

                rf"$\it{{{genus}}}$"

                f" {remainder}"

            )

        if len(parts) == 2:

            return (

                rf"$\it{{{genus}\ {species}}}$"

            )

    return str(name)


# ============================================================
# 11. BUILD PUBLICATION FIGURE
# ============================================================

with plt.rc_context(STYLE):

    fig = plt.figure(

        figsize=(13.8, 10.3),

        facecolor="white"

    )

    gs = fig.add_gridspec(

        2,

        2,

        left=0.19,

        right=0.975,

        bottom=0.09,

        top=0.865,

        wspace=0.36,

        hspace=0.47,

        width_ratios=[1.08, 1.0]

    )

    ax_a = fig.add_subplot(gs[0, 0])

    ax_b = fig.add_subplot(gs[0, 1])

    ax_c = fig.add_subplot(gs[1, 0])

    ax_d = fig.add_subplot(gs[1, 1])

    # ========================================================
    # PANEL A — ABUNDANCE–BIOMASS DECOUPLING
    # ========================================================

    for forest in FORESTS:

        d = fig05_taxa.loc[

            fig05_taxa[
                "forest_type"
            ].eq(forest)

        ]

        ax_a.scatter(

            d["abundance_pct"],

            d["biomass_pct"],

            s=32,

            marker=MARKERS[forest],

            color=COLORS[forest],

            alpha=0.55,

            edgecolor="white",

            linewidth=0.45,

            zorder=3

        )

    limit = 36

    ax_a.plot(

        [0, limit],

        [0, limit],

        color=MUTED,

        linestyle=(0, (4, 4)),

        linewidth=1.0,

        zorder=1

    )

    ax_a.text(

        23,

        25,

        "Equal shares",

        color=MUTED,

        fontsize=8.5,

        rotation=43,

        ha="center"

    )

    # Highlight the original three taxa.
    # Coordinates are taken from the checkpoint.

    highlighted = [

        (

            "Late_secondary",

            "Pseudosenefeldera inclinata",

            (-9, 13),

            "right"

        ),

        (

            "Late_secondary",

            "Meliosma frondosa",

            (18, 8),

            "left"

        ),

        (

            "Mature",

            "Ficus insipida",

            (5, 24),

            "left"

        )

    ]

    for forest, taxon, offset, align in highlighted:

        d = fig05_taxa.loc[

            fig05_taxa[
                "forest_type"
            ].eq(forest)

            & fig05_taxa[
                "functional_taxon"
            ].eq(taxon)

        ]

        assert len(d) == 1

        x = float(

            d["abundance_pct"].iloc[0]

        )

        y = float(

            d["biomass_pct"].iloc[0]

        )

        ax_a.scatter(

            x,

            y,

            s=72,

            marker=MARKERS[forest],

            color=COLORS[forest],

            edgecolor="white",

            linewidth=0.9,

            zorder=6

        )

        ax_a.annotate(

            scientific_label(

                taxon,

                abbreviated=True

            ),

            xy=(x, y),

            xytext=offset,

            textcoords="offset points",

            ha=align,

            va="bottom",

            fontsize=9,

            color=TEXT,

            arrowprops=dict(

                arrowstyle="-",

                color=COLORS[forest],

                linewidth=0.8

            ),

            zorder=8

        )

    ax_a.set_xlim(-0.5, limit)

    ax_a.set_ylim(-0.5, limit)

    ax_a.set_xlabel(

        "Relative abundance (%)",

        labelpad=8

    )

    ax_a.set_ylabel(

        "Relative biomass (%)",

        labelpad=8

    )

    ax_a.xaxis.set_major_locator(

        mticker.MultipleLocator(10)

    )

    ax_a.yaxis.set_major_locator(

        mticker.MultipleLocator(10)

    )

    ax_a.set_aspect(

        "equal",

        adjustable="box"

    )

    style_axis(ax_a)

    panel_title(

        ax_a,

        "a",

        "Abundance–biomass decoupling"

    )

    # ========================================================
    # PANEL B — ORIGINAL CUMULATIVE BIOMASS CURVES
    # ========================================================

    # Use the original precomputed columns.
    # No new ranking or cumulative calculations.

    cumulative_plot = {}

    for forest in FORESTS:

        d = (

            fig05_taxa.loc[

                fig05_taxa[
                    "forest_type"
                ].eq(forest)

                & fig05_taxa[
                    "biomass_rank"
                ].le(30)

            ]

            .sort_values(
                "biomass_rank"
            )

            .copy()

        )

        assert len(d) == 30

        cumulative_plot[forest] = d

        ax_b.plot(

            d["biomass_rank"],

            d["cumulative_biomass_pct"],

            color=COLORS[forest],

            linewidth=2.8,

            solid_capstyle="round",

            zorder=4

        )

        ref = d.loc[

            d["biomass_rank"].isin(

                [5, 10, 20, 30]

            )

        ]

        ax_b.scatter(

            ref["biomass_rank"],

            ref["cumulative_biomass_pct"],

            color=COLORS[forest],

            marker=MARKERS[forest],

            s=32,

            edgecolor="white",

            linewidth=0.6,

            zorder=5

        )

    secondary_curve = cumulative_plot[
        "Late_secondary"
    ]

    mature_curve = cumulative_plot[
        "Mature"
    ]

    assert np.array_equal(

        secondary_curve[
            "biomass_rank"
        ].to_numpy(),

        mature_curve[
            "biomass_rank"
        ].to_numpy()

    )

    # Shading represents the difference between
    # the original curves, not uncertainty.

    ax_b.fill_between(

        secondary_curve[
            "biomass_rank"
        ].to_numpy(),

        secondary_curve[
            "cumulative_biomass_pct"
        ].to_numpy(),

        mature_curve[
            "cumulative_biomass_pct"
        ].to_numpy(),

        color=COLORS["Late_secondary"],

        alpha=0.10,

        linewidth=0,

        zorder=2

    )

    ax_b.axhline(

        50,

        color=MUTED,

        linestyle=(0, (4, 3)),

        linewidth=0.9,

        zorder=1

    )

    ax_b.text(

        29.5,

        51.5,

        "50% of total biomass",

        color=MUTED,

        fontsize=8.5,

        ha="right",

        va="bottom"

    )

    # Exact original cumulative values at rank 10.

    for forest, offset in [

        ("Late_secondary", (5, 12)),

        ("Mature", (8, -18))

    ]:

        d = cumulative_plot[forest]

        value = float(

            d.loc[

                d["biomass_rank"].eq(10),

                "cumulative_biomass_pct"

            ].iloc[0]

        )

        ax_b.annotate(

            f"{value:.1f}%",

            xy=(10, value),

            xytext=offset,

            textcoords="offset points",

            color=COLORS[forest],

            fontsize=9.5,

            fontweight="medium"

        )

    ax_b.set_xlim(1, 30.8)

    ax_b.set_ylim(0, 100)

    ax_b.set_xticks(

        [5, 10, 15, 20, 25, 30]

    )

    ax_b.yaxis.set_major_locator(

        mticker.MultipleLocator(20)

    )

    ax_b.set_xlabel(

        "Number of highest-biomass taxa",

        labelpad=8

    )

    ax_b.set_ylabel(

        "Cumulative biomass (%)",

        labelpad=8

    )

    style_axis(ax_b)

    panel_title(

        ax_b,

        "b",

        "Taxonomic biomass concentration"

    )

    # ========================================================
    # PANEL C — DIVERGING LOLLIPOP
    # ========================================================

    y_positions = np.arange(

        len(fig05_selected_gap)

    )

    ax_c.axvline(

        0,

        color=MUTED,

        linewidth=1.0,

        zorder=1

    )

    for i, row in (
        fig05_selected_gap.iterrows()
    ):

        value = float(

            row["biomass_difference_Mg"]

        )

        forest = (

            "Late_secondary"

            if value > 0

            else "Mature"

        )

        color = COLORS[forest]

        ax_c.hlines(

            y=i,

            xmin=min(0, value),

            xmax=max(0, value),

            color=color,

            linewidth=2.5,

            zorder=3

        )

        ax_c.scatter(

            value,

            i,

            color=color,

            marker=MARKERS[forest],

            s=55,

            edgecolor="white",

            linewidth=0.8,

            zorder=5

        )

        # Numeric annotations on each contribution.

        ax_c.annotate(

            f"{value:+.1f}",

            xy=(value, i),

            xytext=(

                6 if value > 0 else -6,

                0

            ),

            textcoords="offset points",

            ha=(

                "left"

                if value > 0

                else "right"

            ),

            va="center",

            fontsize=8.1,

            color=color

        )

    ax_c.set_yticks(y_positions)

    ax_c.set_yticklabels(

        [

            scientific_label(name)

            for name in fig05_selected_gap[
                "functional_taxon"
            ]

        ],

        fontsize=8.7

    )

    ax_c.set_ylim(

        len(fig05_selected_gap) - 0.5,

        -0.5

    )

    gap_min = float(

        fig05_selected_gap[
            "biomass_difference_Mg"
        ].min()

    )

    gap_max = float(

        fig05_selected_gap[
            "biomass_difference_Mg"
        ].max()

    )

    ax_c.set_xlim(

        np.floor(gap_min / 10) * 10 - 9,

        np.ceil(gap_max / 10) * 10 + 9

    )

    ax_c.xaxis.set_major_locator(

        mticker.MultipleLocator(10)

    )

    ax_c.set_xlabel(

        "Biomass difference: secondary − mature (Mg)",

        labelpad=8

    )

    style_axis(

        ax_c,

        grid_axis="x"

    )

    ax_c.tick_params(

        axis="y",

        length=0,

        pad=7

    )

    panel_title(

        ax_c,

        "c",

        "Taxonomic contributors to biomass gap"

    )

    # ========================================================
    # PANEL D — DOMINANT TAXON EXCLUSION WATERFALL
    # ========================================================

    secondary_color = COLORS[
        "Late_secondary"
    ]

    mature_color = COLORS[
        "Mature"
    ]

    # Full inventory

    ax_d.bar(

        0,

        full_secondary,

        width=0.53,

        color=secondary_color,

        edgecolor="none",

        zorder=3

    )

    # Removed biomass represented as
    # a floating downward waterfall step.

    ax_d.bar(

        1,

        removed_agb,

        bottom=reduced_secondary,

        width=0.53,

        color="#E4C5BE",

        edgecolor=secondary_color,

        linewidth=0.8,

        hatch="///",

        zorder=3

    )

    # Remaining inventory

    ax_d.bar(

        2,

        reduced_secondary,

        width=0.53,

        color=secondary_color,

        edgecolor="none",

        zorder=3

    )

    # Waterfall connectors

    ax_d.hlines(

        full_secondary,

        0.265,

        0.735,

        color=MUTED,

        linewidth=1,

        linestyle=(0, (3, 3)),

        zorder=2

    )

    ax_d.hlines(

        reduced_secondary,

        1.265,

        1.735,

        color=MUTED,

        linewidth=1,

        linestyle=(0, (3, 3)),

        zorder=2

    )

    # Mature forest benchmark

    ax_d.axhline(

        full_mature,

        color=mature_color,

        linewidth=2.0,

        linestyle=(0, (5, 3)),

        zorder=5

    )

    ax_d.text(

        2.43,

        full_mature - 9,

        f"Mature forest: {full_mature:.1f} Mg",

        color=mature_color,

        fontsize=9,

        ha="right",

        va="top",

        path_effects=[

            pe.withStroke(

                linewidth=3.2,

                foreground="white"

            )

        ],

        zorder=7

    )

    # Exact original values

    annotations = [

        (

            0,

            full_secondary,

            f"{full_secondary:.1f}",

            TEXT

        ),

        (

            1,

            full_secondary,

            f"−{removed_agb:.1f}",

            secondary_color

        ),

        (

            2,

            reduced_secondary,

            f"{reduced_secondary:.1f}",

            TEXT

        )

    ]

    for x, y, label, color in annotations:

        ax_d.text(

            x,

            y + 9,

            label,

            ha="center",

            va="bottom",

            fontsize=11,

            fontweight="medium",

            color=color

        )

    # Remaining biomass contrast

    ax_d.text(

        2,

        reduced_secondary * 0.43,

        f"+{remaining_excess:.1f}%\nvs. mature",

        ha="center",

        va="center",

        color="white",

        fontsize=10,

        fontweight="medium",

        linespacing=1.4,

        zorder=7

    )

    ax_d.set_xticks([0, 1, 2])

    ax_d.set_xticklabels(

        [

            "Secondary\nfull inventory",

            "Excluded\n"

            + scientific_label(

                DOMINANT,

                abbreviated=True

            ),

            "Secondary\nremaining inventory"

        ],

        fontsize=9

    )

    ax_d.set_xlim(-0.48, 2.48)

    ax_d.set_ylim(0, 400)

    ax_d.yaxis.set_major_locator(

        mticker.MultipleLocator(100)

    )

    ax_d.set_ylabel(

        "Aboveground biomass (Mg)",

        labelpad=8

    )

    style_axis(ax_d)

    panel_title(

        ax_d,

        "d",

        "Dominant taxon exclusion"

    )

    # ========================================================
    # 12. SHARED FOREST LEGEND
    # ========================================================

    handles = [

        Line2D(

            [0],

            [0],

            color=COLORS[forest],

            linewidth=2.5,

            marker=MARKERS[forest],

            markersize=7,

            markerfacecolor=COLORS[forest],

            markeredgecolor="white",

            markeredgewidth=0.7,

            label=LABELS[forest]

        )

        for forest in FORESTS

    ]

    fig.legend(

        handles=handles,

        loc="upper center",

        bbox_to_anchor=(0.57, 0.965),

        ncol=2,

        fontsize=11,

        columnspacing=2.5,

        handlelength=2.4,

        frameon=False

    )

    # ========================================================
    # 13. EXPORT PUBLICATION-QUALITY FILES
    # ========================================================

    BASENAME = (

        "Fig_05_Taxonomic_Biomass_Decoupling_FINAL"

    )

    for extension in [

        "pdf",

        "svg",

        "png"

    ]:

        fig.savefig(

            OUT / f"{BASENAME}.{extension}",

            dpi=600,

            bbox_inches="tight",

            pad_inches=0.16,

            facecolor="white"

        )

    with Image.open(

        OUT / f"{BASENAME}.png"

    ) as im:

        im.convert("RGB").save(

            OUT / f"{BASENAME}.tiff",

            compression="tiff_lzw",

            dpi=(600, 600)

        )

    plt.show()

# ============================================================
# 14. EXPORT EXACT ORIGINAL SOURCE DATA
# ============================================================

# Preserve all original columns and values.

source_tables = {

    "FIG05_SOURCE_TAXONOMIC_BIOMASS.csv":
        fig05_taxa,

    "FIG05_SOURCE_TAXONOMIC_GAP.csv":
        fig05_gap,

    "FIG05_SOURCE_SELECTED_CONTRIBUTORS.csv":
        fig05_selected_gap,

    "FIG05_SOURCE_DOMINANT_EXCLUSION.csv":
        fig05_exclusion,

    "FIG05_SOURCE_ALL_EXCLUSION_SCENARIOS.csv":
        fig05_exclusion_all,

    "FIG05_SCIENTIFIC_VALIDATION.csv":
        fig05_validation

}

# Preserve the exact original top-30 curves.

fig05_cumulative = pd.concat(

    [

        cumulative_plot["Late_secondary"],

        cumulative_plot["Mature"]

    ],

    ignore_index=True

)

source_tables[

    "FIG05_SOURCE_CUMULATIVE_BIOMASS.csv"

] = fig05_cumulative

for filename, table in source_tables.items():

    table.to_csv(

        OUT / filename,

        index=False

    )

# ============================================================
# 15. PROVENANCE MANIFEST
# ============================================================

metadata = {

    "figure": "Figure 5",

    "title":
        "Taxonomic–biomass decoupling",

    "redesign_cell": "106B",

    "source_audit_cell": "106A",

    "scientific_results_modified": False,

    "tree_level_biomass_reestimated": False,

    "taxonomic_assignments_modified": False,

    "cumulative_curves_recalculated": False,

    "original_checkpoint_object":
        "taxon_biomass",

    "original_taxon_column":
        "functional_taxon",

    "original_taxon_rows": 201,

    "original_gap_rows": 161,

    "inventory_counts":
        EXPECTED_N,

    "taxonomic_richness":
        EXPECTED_RICHNESS,

    "original_biomass_Mg": {

        "secondary_full":
            full_secondary,

        "mature_full":
            full_mature,

        "secondary_without_dominant":
            reduced_secondary

    },

    "dominant_taxon": {

        "name": DOMINANT,

        "n_trees": dominant_n,

        "agb_Mg": dominant_agb,

        "abundance_pct": float(

            dominant_row[
                "abundance_pct"
            ].iloc[0]

        ),

        "biomass_pct": float(

            dominant_row[
                "biomass_pct"
            ].iloc[0]

        )

    },

    "remaining_secondary_excess_pct":
        remaining_excess,

    "panel_c_selection": {

        "positive_contributors": 8,

        "negative_contributors": 4

    },

    "design": {

        "a":
            "Abundance–biomass scatter",

        "b":
            "Original cumulative curves with "
            "difference shading",

        "c":
            "Diverging lollipop plot",

        "d":
            "Dominant taxon exclusion waterfall"

    },

    "sources": sources,

    "palette": COLORS,

    "export_dpi": 600,

    "files": [

        {

            "filename": path.name,

            "size_bytes": path.stat().st_size,

            "sha256": sha256_file(path)

        }

        for path in sorted(OUT.glob("*"))

        if path.is_file()

        and path.suffix.lower() in [

            ".pdf",

            ".svg",

            ".png",

            ".tiff",

            ".csv"

        ]

    ]

}

with open(

    OUT / "FIG05_PROVENANCE.json",

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        metadata,

        f,

        indent=2,

        ensure_ascii=False

    )

# ============================================================
# 16. FINAL SCIENTIFIC REPORT
# ============================================================

print("\n" + "=" * 72)

print("SATIPO — FIGURE 05 FINAL PUBLICATION REMAKE")

print("=" * 72)

print("Original checkpoint object: VERIFIED")

print("Original taxonomic biomass table: VERIFIED")

print("Original biomass gap table: VERIFIED")

print("Original exclusion scenarios: VERIFIED")

print("Original cumulative curves: PRESERVED")

print("Taxonomic assignments modified: NONE")

print("Biomass recalculation: NONE")

print("New statistical models: NONE")

print("\nORIGINAL SCIENTIFIC RESULTS")

print(

    f"Secondary full AGB: "
    f"{full_secondary:.6f} Mg"

)

print(

    f"Mature full AGB: "
    f"{full_mature:.6f} Mg"

)

print(

    f"Dominant taxon AGB: "
    f"{dominant_agb:.6f} Mg"

)

print(

    f"Secondary remaining AGB: "
    f"{reduced_secondary:.6f} Mg"

)

print(

    f"Remaining secondary excess: "
    f"{remaining_excess:.2f}%"

)

print("\nOUTPUT DIRECTORY")

print(OUT)

print("\nEXPORTED FILES")

for path in sorted(OUT.glob("*")):

    if path.is_file():

        print(

            f"  {path.name:65s}"

            f"{path.stat().st_size / 1024:10.1f} KB"

        )

print("\nCELL 106B COMPLETED SUCCESSFULLY")


In [ ]:

# ============================================================
# CELL 107A — EXACT SCIENTIFIC SOURCE RECOVERY
# FIGURE 06 — INTEGRATED BIOMASS DECOMPOSITION
#
# Direct recovery using verified original column names.
# No scientific recalculation.
# ============================================================

from pathlib import Path
import hashlib
import pandas as pd
import numpy as np

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

OUT = (
    ROOT
    / "10_PUBLICATION_FIGURES"
    / "FIG_06"
    / "SOURCE_DATA"
)

OUT.mkdir(parents=True, exist_ok=True)

ANALYSIS = ROOT / "03_ANALYSIS"

# ============================================================
# 1. RECOVER ORIGINAL CSV FILES
# ============================================================

def recover_original_csv(filename):

    matches = sorted(
        ANALYSIS.rglob(filename)
    )

    if not matches:

        raise FileNotFoundError(
            f"Original file not found: {filename}"
        )

    path = matches[0]

    df = pd.read_csv(path)

    print("\n" + "=" * 72)
    print("ORIGINAL FILE:", filename)
    print("PATH:", path)
    print("SHAPE:", df.shape)
    print("COLUMNS:", df.columns.tolist())
    print("=" * 72)

    display(df.head(10))

    return df, path


# ============================================================
# 2. EXACT MULTIPLICATIVE FACTORIZATION
# ============================================================

factorization, factorization_path = (
    recover_original_csv(
        "BIOMASS_EXACT_FACTORIZATION.csv"
    )
)

required_columns = [

    "forest_type",
    "N",
    "Diameter_moment",
    "Height_factor",
    "Wood_density_factor",
    "Observed_AGB_Mg",
    "Reconstructed_AGB_Mg"

]

assert set(required_columns).issubset(
    factorization.columns
), "Unexpected original factorization schema."

assert len(factorization) == 2

assert set(
    factorization["forest_type"]
) == {

    "Late_secondary",
    "Mature"

}

# Use only the exact original factor columns.
# Effective diameter, height and WD are NOT used.

FACTOR_COLUMNS = {

    "N": "N",

    "D": "Diameter_moment",

    "H": "Height_factor",

    "WD": "Wood_density_factor"

}

FACTORS = ["N", "D", "H", "WD"]

factor_index = factorization.set_index(
    "forest_type"
)

# Recover original factor values.

factor_rows = []

for code, column in FACTOR_COLUMNS.items():

    secondary_value = float(

        factor_index.loc[
            "Late_secondary",
            column
        ]

    )

    mature_value = float(

        factor_index.loc[
            "Mature",
            column
        ]

    )

    factor_rows.append({

        "factor": code,

        "original_column": column,

        "Late_secondary": secondary_value,

        "Mature": mature_value,

        "factor_ratio":
            secondary_value / mature_value

    })

fig06_factors = pd.DataFrame(
    factor_rows
)

ratio_series = (

    fig06_factors

    .set_index("factor")

    ["factor_ratio"]

    .reindex(FACTORS)

)

# ============================================================
# 3. ORIGINAL BIOMASS TOTALS
# ============================================================

secondary_agb = float(

    factor_index.loc[

        "Late_secondary",

        "Observed_AGB_Mg"

    ]

)

mature_agb = float(

    factor_index.loc[

        "Mature",

        "Observed_AGB_Mg"

    ]

)

biomass_gap = (

    secondary_agb - mature_agb

)

biomass_ratio = (

    secondary_agb / mature_agb

)

# Verify original biomass reconstruction.

assert np.allclose(

    factorization["Observed_AGB_Mg"],

    factorization["Reconstructed_AGB_Mg"],

    atol=1e-6,

    rtol=1e-8

), "Original biomass reconstruction mismatch."

# Verify multiplicative identity.

assert np.isclose(

    ratio_series.prod(),

    biomass_ratio,

    atol=1e-8,

    rtol=1e-8

), "Multiplicative identity mismatch."

# Independent reference checks.

assert np.isclose(

    secondary_agb,

    355.226643,

    atol=1e-5,

    rtol=0

)

assert np.isclose(

    mature_agb,

    234.362563,

    atol=1e-5,

    rtol=0

)

# ============================================================
# 4. DISPLAY VERIFIED FACTORS
# ============================================================

print("\n" + "=" * 72)
print("EXACT MULTIPLICATIVE FACTORIZATION")
print("=" * 72)

display(
    fig06_factors.round(9)
)

print(
    "\nMature AGB:",
    f"{mature_agb:.9f} Mg"
)

print(
    "Secondary AGB:",
    f"{secondary_agb:.9f} Mg"
)

print(
    "Observed biomass gap:",
    f"{biomass_gap:.9f} Mg"
)

print(
    "Observed biomass ratio:",
    f"{biomass_ratio:.9f}"
)

print(
    "Product of original factors:",
    f"{ratio_series.prod():.9f}"
)

print(
    "\nMultiplicative identity: VERIFIED"
)

# ============================================================
# 5. RECOVER ORIGINAL SHAPLEY SENSITIVITY TABLE
# ============================================================

fig06_sensitivity_raw, sensitivity_path = (

    recover_original_csv(

        "BIOMASS_INTEGRATED_SENSITIVITY.csv"

    )

)

print("\nFULL ORIGINAL SENSITIVITY TABLE")

display(
    fig06_sensitivity_raw
)

print("\nSENSITIVITY DATA TYPES")

print(
    fig06_sensitivity_raw.dtypes
)

# ============================================================
# 6. RECOVER OTHER ORIGINAL DECOMPOSITION TABLES
# ============================================================

candidate_files = [

    "BIOMASS_SHAPLEY_DECOMPOSITION.csv",

    "BIOMASS_LMDI_DECOMPOSITION.csv",

    "BIOMASS_INTEGRATED_DECOMPOSITION.csv",

    "BIOMASS_DECOMPOSITION_ORDER_SENSITIVITY.csv",

    "BIOMASS_SHAPLEY_SUMMARY.csv",

    "BIOMASS_DECOMPOSITION_SUMMARY.csv"

]

recovered_decomposition = {}

for filename in candidate_files:

    matches = sorted(

        ANALYSIS.rglob(filename)

    )

    if not matches:

        continue

    df, path = recover_original_csv(
        filename
    )

    recovered_decomposition[filename] = df

# ============================================================
# 7. DISCOVER ADDITIONAL DECOMPOSITION FILES
# ============================================================

print("\n" + "=" * 72)
print("RELATED ORIGINAL PIPELINE FILES")
print("=" * 72)

related_files = []

for path in ANALYSIS.rglob("*.csv"):

    name = path.name.upper()

    if any(

        token in name

        for token in [

            "SHAPLEY",

            "LMDI",

            "DECOMPOSITION",

            "FACTORIZATION",

            "INTEGRATED_SENSITIVITY"

        ]

    ):

        related_files.append(path)

for path in sorted(related_files):

    print(
        path.relative_to(ROOT)
    )

# ============================================================
# 8. PRESERVE EXACT RECOVERED SOURCES
# ============================================================

factorization.to_csv(

    OUT / "FIG06_ORIGINAL_FACTORIZATION.csv",

    index=False

)

fig06_factors.to_csv(

    OUT / "FIG06_SOURCE_FACTOR_RATIOS.csv",

    index=False

)

fig06_sensitivity_raw.to_csv(

    OUT / "FIG06_ORIGINAL_SENSITIVITY.csv",

    index=False

)

for filename, df in (
    recovered_decomposition.items()
):

    df.to_csv(

        OUT / f"FIG06_ORIGINAL_{filename}",

        index=False

    )

# ============================================================
# 9. SOURCE INTEGRITY MANIFEST
# ============================================================

source_manifest = []

for path in [

    factorization_path,

    sensitivity_path

]:

    digest = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

    source_manifest.append({

        "source_file": path.name,

        "original_path": str(path),

        "sha256": digest,

        "size_bytes": path.stat().st_size

    })

pd.DataFrame(
    source_manifest
).to_csv(

    OUT / "FIG06_SOURCE_MANIFEST.csv",

    index=False

)

# ============================================================
# 10. FINAL RECOVERY REPORT
# ============================================================

print("\n" + "=" * 72)

print("SATIPO — FIGURE 06 SOURCE RECOVERY")

print("=" * 72)

print("Original factor columns: VERIFIED")

print("Original biomass totals: VERIFIED")

print("Multiplicative identity: VERIFIED")

print("Original sensitivity table: RECOVERED")

print("Biomass reestimation: NONE")

print("Shapley recalculation: NONE")

print("Original scientific files modified: NONE")

print("\nSOURCE DIRECTORY:")

print(OUT)

print("\nCELL 107A COMPLETED SUCCESSFULLY")

print(
    "Ready for Cell 107B — "
    "Final publication visualization."
)


In [ ]:

# ============================================================
# CELL 107B — FIGURE 06 PUBLICATION REMAKE
#
# DESIGN C — Integrated biomass architecture
#
# a) Exact multiplicative factorization
# b) Mature-to-secondary Shapley waterfall
# c) Sensitivity profiles across inventories
#
# Uses verified original pipeline outputs.
# NO scientific recalculation.
# ============================================================

from pathlib import Path
import json
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch
from PIL import Image

# ============================================================
# 1. PATHS
# ============================================================

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

OUT = (
    ROOT
    / "10_PUBLICATION_FIGURES"
    / "FIG_06"
)

SOURCE = OUT / "SOURCE_DATA"

OUT.mkdir(parents=True, exist_ok=True)

# ============================================================
# 2. LOAD VERIFIED ORIGINAL RESULTS
# ============================================================

factorization = pd.read_csv(
    SOURCE / "FIG06_ORIGINAL_FACTORIZATION.csv"
)

factor_ratios = pd.read_csv(
    SOURCE / "FIG06_SOURCE_FACTOR_RATIOS.csv"
)

sensitivity_raw = pd.read_csv(
    SOURCE / "FIG06_ORIGINAL_SENSITIVITY.csv"
)

# Exact original pipeline component names.

COMPONENT_MAP = {
    "N": "N",
    "Diameter_moment": "D",
    "Height_factor": "H",
    "Wood_density_factor": "WD"
}

FACTORS = ["N", "D", "H", "WD"]

SCENARIOS = [
    "Full_inventory",
    "Without_Pseudosenefeldera",
    "Common_diameter_support",
    "Species_level_WD_only"
]

FACTOR_LABELS = {
    "N": "Stem abundance",
    "D": "Diameter distribution",
    "H": "Height dimension",
    "WD": "Wood density"
}

SCENARIO_LABELS = [
    "Full\ninventory",
    "Without\n"
    r"$\it{P.\ inclinata}$",
    "Common\ndiameter support",
    "Species-level\nWD only"
]

# ============================================================
# 3. PREPARE PLOTTING TABLES
# ============================================================

ratio_series = (
    factor_ratios
    .set_index("factor")["factor_ratio"]
    .reindex(FACTORS)
    .astype(float)
)

sensitivity_raw["factor_code"] = (
    sensitivity_raw["component"]
    .map(COMPONENT_MAP)
)

assert sensitivity_raw[
    "factor_code"
].notna().all()

assert not sensitivity_raw.duplicated(
    ["scenario", "factor_code"]
).any()

sensitivity = (
    sensitivity_raw.pivot(
        index="scenario",
        columns="factor_code",
        values="Shapley_Mg"
    )
    .reindex(
        index=SCENARIOS,
        columns=FACTORS
    )
)

shapley = sensitivity.loc[
    "Full_inventory"
].copy()

forest_index = factorization.set_index(
    "forest_type"
)

mature_agb = float(
    forest_index.loc[
        "Mature",
        "Observed_AGB_Mg"
    ]
)

secondary_agb = float(
    forest_index.loc[
        "Late_secondary",
        "Observed_AGB_Mg"
    ]
)

biomass_gap = secondary_agb - mature_agb

biomass_ratio = secondary_agb / mature_agb

# ============================================================
# 4. SCIENTIFIC VALIDATION
# ============================================================

assert len(sensitivity_raw) == 16

assert sensitivity.shape == (4, 4)

assert not sensitivity.isna().any().any()

assert np.isclose(
    mature_agb,
    234.362562660,
    atol=1e-7,
    rtol=0
)

assert np.isclose(
    secondary_agb,
    355.226643085,
    atol=1e-7,
    rtol=0
)

assert np.isclose(
    ratio_series.prod(),
    biomass_ratio,
    atol=1e-8,
    rtol=1e-8
)

assert np.isclose(
    shapley.sum(),
    biomass_gap,
    atol=1e-7,
    rtol=1e-8
)

# Check every original scenario.

for scenario in SCENARIOS:

    original = sensitivity_raw.loc[
        sensitivity_raw["scenario"].eq(
            scenario
        )
    ]

    assert len(original) == 4

    assert original[
        "total_gap_Mg"
    ].nunique() == 1

    original_gap = float(
        original["total_gap_Mg"].iloc[0]
    )

    assert np.isclose(
        sensitivity.loc[scenario].sum(),
        original_gap,
        atol=1e-7,
        rtol=1e-8
    )

    original_ratio = (
        original["secondary_AGB_Mg"].iloc[0]
        /
        original["mature_AGB_Mg"].iloc[0]
    )

    assert np.isclose(
        original["factor_ratio"].prod(),
        original_ratio,
        atol=1e-8,
        rtol=1e-8
    )

assert (sensitivity["H"] > 0).all()

assert (sensitivity["WD"] > 0).all()

print("=" * 72)
print("FIGURE 06 — ORIGINAL DATA VALIDATION")
print("=" * 72)

print(
    f"Mature AGB: {mature_agb:.9f} Mg"
)

print(
    f"Secondary AGB: {secondary_agb:.9f} Mg"
)

print(
    f"Net biomass gap: {biomass_gap:.9f} Mg"
)

print(
    f"Biomass ratio: {biomass_ratio:.9f}"
)

print("\nOriginal factor ratios:")

display(ratio_series.to_frame("ratio"))

print("\nOriginal Shapley contributions:")

display(sensitivity.round(6))

print("\nScientific integrity: VERIFIED")

# ============================================================
# 5. PUBLICATION STYLE
# ============================================================

COLORS = {
    "N": "#C18B5D",
    "D": "#557B9D",
    "H": "#9273A6",
    "WD": "#468477"
}

MARKERS = {
    "N": "o",
    "D": "s",
    "H": "D",
    "WD": "^"
}

TEXT = "#26333D"
MUTED = "#68757F"
DARK = "#35434D"
GRID = "#E5E9EC"
AXIS = "#A4ADB5"

STYLE = {
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.labelcolor": TEXT,
    "axes.edgecolor": AXIS,
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "xtick.color": TEXT,
    "ytick.color": TEXT,
    "legend.frameon": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
}

# ============================================================
# 6. GRAPHICAL HELPERS
# ============================================================

def panel_heading(
    ax,
    letter,
    title,
    subtitle=None
):

    ax.text(
        -0.075,
        1.125,
        letter,
        transform=ax.transAxes,
        fontsize=18,
        fontweight="bold",
        color=TEXT,
        va="bottom"
    )

    ax.text(
        0.005,
        1.13,
        title,
        transform=ax.transAxes,
        fontsize=13,
        fontweight="medium",
        color=TEXT,
        va="bottom"
    )

    if subtitle is not None:

        ax.text(
            0.005,
            1.065,
            subtitle,
            transform=ax.transAxes,
            fontsize=9.5,
            color=MUTED,
            va="bottom"
        )


def style_axis(ax):

    ax.set_axisbelow(True)

    ax.grid(
        axis="y",
        color=GRID,
        linewidth=0.75
    )

    ax.tick_params(
        direction="out",
        length=3,
        pad=5
    )

# ============================================================
# 7. FIGURE LAYOUT
# ============================================================

with plt.rc_context(STYLE):

    fig = plt.figure(
        figsize=(14.5, 8.7),
        facecolor="white"
    )

    gs = fig.add_gridspec(
        2,
        2,
        height_ratios=[0.85, 2.25],
        width_ratios=[1.05, 1.0],
        left=0.085,
        right=0.97,
        bottom=0.17,
        top=0.92,
        hspace=0.63,
        wspace=0.30
    )

    ax_a = fig.add_subplot(gs[0, :])

    ax_b = fig.add_subplot(gs[1, 0])

    ax_c = fig.add_subplot(gs[1, 1])

    # ========================================================
    # PANEL A — EXACT MULTIPLICATIVE FACTORIZATION
    # ========================================================

    ax_a.set_xlim(0, 1)
    ax_a.set_ylim(0, 1)
    ax_a.axis("off")

    ax_a.text(
        0.0,
        1.14,
        "a",
        fontsize=18,
        fontweight="bold",
        color=TEXT
    )

    ax_a.text(
        0.035,
        1.145,
        "Multiplicative factorization",
        fontsize=13,
        fontweight="medium",
        color=TEXT
    )

    ax_a.text(
        0.0,
        0.94,
        "Secondary / mature biomass ratio",
        fontsize=10,
        color=MUTED
    )

    positions = [
        0.000,
        0.202,
        0.404,
        0.606,
        0.823
    ]

    card_width = 0.165
    card_y = 0.10
    card_height = 0.67

    for i, factor in enumerate(FACTORS):

        x = positions[i]
        color = COLORS[factor]

        card = FancyBboxPatch(
            (x, card_y),
            card_width,
            card_height,
            boxstyle=(
                "round,pad=0.006,"
                "rounding_size=0.012"
            ),
            facecolor="#F5F6F7",
            edgecolor="#E0E5E8",
            linewidth=0.8
        )

        ax_a.add_patch(card)

        # Factor-specific accent.

        ax_a.plot(
            [
                x + 0.012,
                x + card_width - 0.012
            ],
            [
                card_y + card_height - 0.035,
                card_y + card_height - 0.035
            ],
            color=color,
            linewidth=3,
            solid_capstyle="round"
        )

        # Factor name.

        ax_a.text(
            x + card_width / 2,
            card_y + 0.44,
            FACTOR_LABELS[factor],
            ha="center",
            va="center",
            fontsize=10,
            color=TEXT
        )

        # Original factor ratio.

        ax_a.text(
            x + card_width / 2,
            card_y + 0.19,
            f"{ratio_series[factor]:.3f}",
            ha="center",
            va="center",
            fontsize=26,
            fontweight="bold",
            color=color
        )

        if i < 3:

            ax_a.text(
                x + card_width + 0.0185,
                card_y + card_height / 2,
                "×",
                fontsize=22,
                color=MUTED,
                ha="center",
                va="center"
            )

    # Equality sign.

    ax_a.text(
        0.797,
        card_y + card_height / 2,
        "=",
        fontsize=23,
        color=MUTED,
        ha="center",
        va="center"
    )

    # Final observed biomass ratio.

    result_card = FancyBboxPatch(
        (positions[4], card_y),
        card_width,
        card_height,
        boxstyle=(
            "round,pad=0.006,"
            "rounding_size=0.012"
        ),
        facecolor=DARK,
        edgecolor="none"
    )

    ax_a.add_patch(result_card)

    ax_a.text(
        positions[4] + card_width / 2,
        card_y + 0.44,
        "Biomass ratio",
        ha="center",
        va="center",
        fontsize=10,
        color="white"
    )

    ax_a.text(
        positions[4] + card_width / 2,
        card_y + 0.19,
        f"{biomass_ratio:.3f}",
        ha="center",
        va="center",
        fontsize=27,
        fontweight="bold",
        color="white"
    )

    # ========================================================
    # PANEL B — SHAPLEY WATERFALL
    # ========================================================

    panel_heading(
        ax_b,
        "b",
        "Exact biomass decomposition",
        "Shapley contributions to the observed contrast"
    )

    bar_width = 0.63

    # Mature forest baseline.

    ax_b.bar(
        0,
        mature_agb,
        width=bar_width,
        color=DARK,
        edgecolor="none",
        zorder=3
    )

    ax_b.text(
        0,
        mature_agb + 9,
        f"{mature_agb:.1f}",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="medium",
        color=DARK
    )

    running_total = mature_agb

    waterfall_rows = [
        {
            "component": "Mature",
            "value_Mg": mature_agb,
            "bottom_Mg": 0.0,
            "top_Mg": mature_agb
        }
    ]

    for i, factor in enumerate(
        FACTORS,
        start=1
    ):

        value = float(shapley[factor])

        next_total = running_total + value

        bottom = min(
            running_total,
            next_total
        )

        top = max(
            running_total,
            next_total
        )

        # Waterfall connector.

        ax_b.hlines(
            running_total,
            i - 1 + bar_width / 2,
            i - bar_width / 2,
            color=MUTED,
            linewidth=1.0,
            linestyle=(0, (3, 2)),
            zorder=2
        )

        ax_b.bar(
            i,
            abs(value),
            bottom=bottom,
            width=bar_width,
            color=COLORS[factor],
            edgecolor="none",
            zorder=3
        )

        # External labels for small contributions.

        if abs(value) < 15:

            ax_b.text(
                i,
                top + 10,
                f"{value:+.1f}",
                ha="center",
                va="bottom",
                fontsize=10.5,
                fontweight="medium",
                color=COLORS[factor]
            )

        else:

            ax_b.text(
                i,
                bottom + abs(value) / 2,
                f"{value:+.1f}",
                ha="center",
                va="center",
                fontsize=10.5,
                fontweight="bold",
                color="white"
            )

        waterfall_rows.append({
            "component": factor,
            "value_Mg": value,
            "bottom_Mg": bottom,
            "top_Mg": top
        })

        running_total = next_total

    # Confirm the waterfall closes exactly.

    assert np.isclose(
        running_total,
        secondary_agb,
        atol=1e-7,
        rtol=1e-8
    )

    ax_b.hlines(
        running_total,
        4 + bar_width / 2,
        5 - bar_width / 2,
        color=MUTED,
        linewidth=1.0,
        linestyle=(0, (3, 2)),
        zorder=2
    )

    # Secondary forest endpoint.

    ax_b.bar(
        5,
        secondary_agb,
        width=bar_width,
        color=DARK,
        edgecolor="none",
        zorder=3
    )

    ax_b.text(
        5,
        secondary_agb + 9,
        f"{secondary_agb:.1f}",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="medium",
        color=DARK
    )

    waterfall_rows.append({
        "component": "Secondary",
        "value_Mg": secondary_agb,
        "bottom_Mg": 0.0,
        "top_Mg": secondary_agb
    })

    # Net biomass contrast annotation.

    ax_b.text(
        0.5,
        0.045,
        f"Net biomass difference: "
        f"+{biomass_gap:.1f} Mg",
        transform=ax_b.transAxes,
        ha="center",
        va="bottom",
        fontsize=10.5,
        fontweight="medium",
        color=TEXT,
        bbox={
            "facecolor": "white",
            "edgecolor": "#DDE2E6",
            "boxstyle": "round,pad=0.45"
        },
        zorder=8
    )

    ax_b.set_xticks(np.arange(6))

    ax_b.set_xticklabels([
        "Mature",
        "N",
        "D",
        "H",
        "WD",
        "Secondary"
    ])

    ax_b.set_xlim(-0.65, 5.65)

    ax_b.set_ylim(0, 410)

    ax_b.yaxis.set_major_locator(
        mticker.MultipleLocator(100)
    )

    ax_b.set_ylabel(
        "Aboveground biomass (Mg)",
        labelpad=9
    )

    style_axis(ax_b)

    # ========================================================
    # PANEL C — INVENTORY SENSITIVITY PROFILES
    # ========================================================

    panel_heading(
        ax_c,
        "c",
        "Robustness of biomass contributions",
        "Shapley decomposition across alternative inventories"
    )

    x = np.arange(len(SCENARIOS))

    # Zero-contribution reference.

    ax_c.axhline(
        0,
        color="#8C989F",
        linewidth=1.1,
        linestyle=(0, (4, 3)),
        zorder=1
    )

    # Original Shapley values for each scenario.

    for factor in FACTORS:

        values = sensitivity[
            factor
        ].to_numpy()

        ax_c.plot(
            x,
            values,
            color=COLORS[factor],
            linewidth=2.5,
            marker=MARKERS[factor],
            markersize=7.5,
            markerfacecolor=COLORS[factor],
            markeredgecolor="white",
            markeredgewidth=0.9,
            solid_capstyle="round",
            zorder=4
        )

    # Selected annotations highlight the robust
    # positive height and wood-density components.

    for factor, offset in [
        ("H", (9, 8)),
        ("WD", (9, -9))
    ]:

        value = float(
            sensitivity.loc[
                SCENARIOS[-1],
                factor
            ]
        )

        ax_c.annotate(
            f"{value:.1f}",
            xy=(3, value),
            xytext=offset,
            textcoords="offset points",
            fontsize=9,
            fontweight="medium",
            color=COLORS[factor],
            va="center"
        )

    # Stem abundance sign reversal following
    # dominant taxon exclusion.

    excluded_stem = float(
        sensitivity.loc[
            "Without_Pseudosenefeldera",
            "N"
        ]
    )

    ax_c.annotate(
        f"{excluded_stem:.1f}",
        xy=(1, excluded_stem),
        xytext=(9, -10),
        textcoords="offset points",
        fontsize=9,
        fontweight="medium",
        color=COLORS["N"],
        ha="left",
        va="top"
    )

    ax_c.set_xticks(x)

    ax_c.set_xticklabels(
        SCENARIO_LABELS,
        fontsize=9.5,
        linespacing=1.3
    )

    ax_c.set_xlim(-0.20, 3.42)

    ax_c.set_ylim(-80, 80)

    ax_c.yaxis.set_major_locator(
        mticker.MultipleLocator(20)
    )

    ax_c.set_ylabel(
        "Shapley contribution (Mg)",
        labelpad=9
    )

    style_axis(ax_c)

    # ========================================================
    # 8. SHARED LEGEND
    # ========================================================

    legend_handles = [
        Line2D(
            [0],
            [0],
            color=COLORS[factor],
            linewidth=2.5,
            marker=MARKERS[factor],
            markersize=7,
            markeredgecolor="white",
            markeredgewidth=0.8,
            label=FACTOR_LABELS[factor]
        )
        for factor in FACTORS
    ]

    fig.legend(
        handles=legend_handles,
        loc="lower center",
        bbox_to_anchor=(0.52, 0.055),
        ncol=4,
        fontsize=10.5,
        columnspacing=2.0,
        handlelength=2.4,
        frameon=False
    )

    # ========================================================
    # 9. EXPORT HIGH-RESOLUTION FIGURES
    # ========================================================

    BASENAME = (
        "Fig_06_Integrated_Biomass_"
        "Decomposition_FINAL"
    )

    for extension in [
        "pdf",
        "svg",
        "png"
    ]:

        fig.savefig(
            OUT / f"{BASENAME}.{extension}",
            dpi=600,
            bbox_inches="tight",
            pad_inches=0.18,
            facecolor="white"
        )

    with Image.open(
        OUT / f"{BASENAME}.png"
    ) as im:

        im.convert("RGB").save(
            OUT / f"{BASENAME}.tiff",
            compression="tiff_lzw",
            dpi=(600, 600)
        )

    plt.show()

# ============================================================
# 10. EXPORT EXACT FIGURE SOURCE DATA
# ============================================================

ratio_series.rename(
    "factor_ratio"
).to_csv(
    OUT / "FIG06_FINAL_FACTOR_RATIOS.csv"
)

shapley.rename(
    "Shapley_Mg"
).to_csv(
    OUT / "FIG06_FINAL_SHAPLEY.csv"
)

sensitivity.to_csv(
    OUT / "FIG06_FINAL_SENSITIVITY.csv"
)

pd.DataFrame(
    waterfall_rows
).to_csv(
    OUT / "FIG06_FINAL_WATERFALL.csv",
    index=False
)

# ============================================================
# 11. PROVENANCE AND REPRODUCIBILITY
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            h.update(chunk)

    return h.hexdigest()


original_sources = [
    SOURCE / "FIG06_ORIGINAL_FACTORIZATION.csv",
    SOURCE / "FIG06_SOURCE_FACTOR_RATIOS.csv",
    SOURCE / "FIG06_ORIGINAL_SENSITIVITY.csv"
]

metadata = {
    "figure": "Figure 6",
    "design": "C",
    "title": (
        "Integrated biomass decomposition"
    ),
    "source_recovery_cell": "107A",
    "visualization_cell": "107B",
    "scientific_results_modified": False,
    "biomass_reestimated": False,
    "Shapley_recalculated": False,
    "new_model_fitting": False,
    "mature_AGB_Mg": mature_agb,
    "secondary_AGB_Mg": secondary_agb,
    "net_gap_Mg": biomass_gap,
    "biomass_ratio": biomass_ratio,
    "factor_ratios": {
        f: float(ratio_series[f])
        for f in FACTORS
    },
    "Shapley_Mg": {
        f: float(shapley[f])
        for f in FACTORS
    },
    "scenarios": SCENARIOS,
    "palette": COLORS,
    "export_dpi": 600,
    "sources": [
        {
            "path": str(path),
            "sha256": sha256_file(path)
        }
        for path in original_sources
    ],
    "interpretation": [
        (
            "Multiplicative factors reconstruct "
            "the observed biomass ratio."
        ),
        (
            "Shapley contributions provide an "
            "algebraic allocation of the biomass gap."
        ),
        (
            "Waterfall positions are graphical "
            "accumulations, not temporal changes."
        ),
        (
            "Connected sensitivity profiles compare "
            "categorical inventory scenarios."
        ),
        (
            "No causal inference or forest-type "
            "replication is implied."
        )
    ],
    "exports": [
        {
            "filename": path.name,
            "sha256": sha256_file(path),
            "size_bytes": path.stat().st_size
        }
        for path in sorted(
            OUT.glob(f"{BASENAME}.*")
        )
    ]
}

with open(
    OUT / "FIG06_FINAL_PROVENANCE.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata,
        f,
        indent=2,
        ensure_ascii=False
    )

# ============================================================
# 12. FINAL REPORT
# ============================================================

print("\n" + "=" * 72)
print("SATIPO — FIGURE 06 FINAL PUBLICATION REMAKE")
print("=" * 72)

print("Selected design: C")

print("Original factor ratios: VERIFIED")

print("Original Shapley contributions: VERIFIED")

print("All four inventory scenarios: VERIFIED")

print("Multiplicative identity: VERIFIED")

print("Waterfall closure: VERIFIED")

print("Scientific recalculation: NONE")

print("Original scientific files modified: NONE")

print("\nEXACT MULTIPLICATIVE IDENTITY")

print(
    " × ".join(
        f"{ratio_series[f]:.9f}"
        for f in FACTORS
    ),
    "=",
    f"{biomass_ratio:.9f}"
)

print("\nEXACT SHAPLEY CONTRIBUTIONS")

for factor in FACTORS:

    print(
        f"{FACTOR_LABELS[factor]:25s}",
        f"{shapley[factor]:+14.9f} Mg"
    )

print(
    f"{'Net biomass gap':25s}",
    f"{biomass_gap:+14.9f} Mg"
)

print("\nOUTPUT DIRECTORY:")

print(OUT)

print("\nPUBLICATION FILES:")

for path in sorted(
    OUT.glob(f"{BASENAME}.*")
):

    print(
        f"{path.name:65s}",
        f"{path.stat().st_size / 1024:10.1f} KB"
    )

print("\nCELL 107B COMPLETED SUCCESSFULLY")


In [ ]:

# ============================================================
# CELL 108A — FIGURE 07 SOURCE RECOVERY
# Historical native Landsat spectral trajectories
#
# Recover original scientific tables.
# No spectral recalculation.
# No temporal aggregation.
# No cross-sensor harmonization.
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import pandas as pd
import numpy as np
import hashlib
import json
import shutil

# ============================================================
# 1. PROJECT PATHS
# ============================================================

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

REMOTE = ROOT / "04_REMOTE_SENSING"

PACKAGE = ROOT / "08_FINAL_MANUSCRIPT_PACKAGE"

OUT = (
    ROOT
    / "10_PUBLICATION_FIGURES"
    / "FIG_07"
)

SOURCE = OUT / "SOURCE_DATA"

SOURCE.mkdir(
    parents=True,
    exist_ok=True
)

assert ROOT.exists(), (
    "SATIPO project directory not found."
)

assert REMOTE.exists(), (
    "Remote sensing directory not found."
)

print("=" * 72)
print("SATIPO — FIGURE 07 SOURCE RECOVERY")
print("=" * 72)

print("Project:", ROOT)
print("Remote sensing:", REMOTE)
print("Output:", SOURCE)

# ============================================================
# 2. FILE INTEGRITY HELPER
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            h.update(chunk)

    return h.hexdigest()


# ============================================================
# 3. RECOVER ORIGINAL HISTORICAL WORKBOOK
# ============================================================

WORKBOOK_NAME = (
    "SATIPO_LANDSAT_HISTORICAL_TRAJECTORIES.xlsx"
)

matches = sorted(
    REMOTE.rglob(WORKBOOK_NAME)
)

# Use the original remote sensing directory first.
# Only search the editorial package if necessary.

if not matches and PACKAGE.exists():

    matches = sorted(
        PACKAGE.rglob(WORKBOOK_NAME)
    )

if not matches:

    matches = sorted(
        ROOT.rglob(WORKBOOK_NAME)
    )

if not matches:

    raise FileNotFoundError(
        f"Original workbook not found: "
        f"{WORKBOOK_NAME}"
    )

print("\nWORKBOOK CANDIDATES")

for path in matches:

    print(path)

workbook_path = matches[0]

# Verify duplicate copies, if any.

workbook_hash = sha256_file(
    workbook_path
)

for path in matches[1:]:

    other_hash = sha256_file(path)

    if other_hash != workbook_hash:

        print(
            "\nWARNING: Different workbook copy:"
        )

        print(path)

        print(
            "Using the original remote sensing "
            "workbook, not the alternate copy."
        )

print("\n" + "=" * 72)
print("ORIGINAL HISTORICAL WORKBOOK")
print("=" * 72)

print("PATH:", workbook_path)

print("SHA256:", workbook_hash)

# ============================================================
# 4. INSPECT ALL ORIGINAL WORKBOOK SHEETS
# ============================================================

historical_workbook = pd.ExcelFile(
    workbook_path
)

print("\nAVAILABLE SHEETS:")

for sheet in historical_workbook.sheet_names:

    print(" -", sheet)

fig07_workbook_tables = {}

sheet_inventory = []

for sheet in historical_workbook.sheet_names:

    df = pd.read_excel(
        historical_workbook,
        sheet_name=sheet
    )

    fig07_workbook_tables[sheet] = df

    sheet_inventory.append({

        "sheet": sheet,

        "n_rows": len(df),

        "n_columns": len(df.columns),

        "columns": " | ".join(
            map(str, df.columns)
        )

    })

    print("\n" + "=" * 72)

    print("SHEET:", sheet)

    print("SHAPE:", df.shape)

    print("COLUMNS:", df.columns.tolist())

    print("=" * 72)

    display(df.head(8))

    # Report the exact categories and ranges of
    # relevant temporal and spatial variables.

    for col in df.columns:

        col_name = str(col).lower()

        if any(
            token in col_name
            for token in [

                "year",
                "sensor",
                "mission",
                "satellite",
                "forest",
                "unit",
                "scale",
                "index",
                "season",
                "scenario",
                "acquisition",
                "scene",
                "coverage",
                "valid",
                "support"

            ]
        ):

            unique_values = (
                df[col]
                .dropna()
                .unique()
            )

            if len(unique_values) <= 20:

                print(
                    f"\n{col}:",
                    unique_values.tolist()
                )

            elif pd.api.types.is_numeric_dtype(
                df[col]
            ):

                print(
                    f"\n{col}: "
                    f"min={df[col].min()}, "
                    f"max={df[col].max()}, "
                    f"unique={len(unique_values)}"
                )

            else:

                print(
                    f"\n{col}: "
                    f"{len(unique_values)} "
                    f"unique values"
                )

# ============================================================
# 5. DISCOVER ORIGINAL LANDSAT CSV TABLES
# ============================================================

print("\n" + "=" * 72)
print("ORIGINAL LANDSAT CSV INVENTORY")
print("=" * 72)

landsat_csv_files = []

for path in REMOTE.rglob("*.csv"):

    relative = str(
        path.relative_to(REMOTE)
    ).lower()

    if "landsat" in relative:

        landsat_csv_files.append(path)

landsat_csv_files = sorted(
    landsat_csv_files
)

for path in landsat_csv_files:

    print(
        path.relative_to(ROOT)
    )

print(
    "\nTotal Landsat CSV files:",
    len(landsat_csv_files)
)

# ============================================================
# 6. IDENTIFY HISTORICAL SOURCE CANDIDATES
# ============================================================

# Candidate selection is only for source inspection.
# No candidate is automatically assumed to be the
# original Figure 7 plotting table.

KEYWORDS = [

    "historical",
    "trajectory",
    "trajectories",
    "annual",
    "seasonal",
    "native",
    "sensor_year",
    "sensor-year",
    "paired"

]

candidate_csv_files = [

    path

    for path in landsat_csv_files

    if any(

        token in path.name.lower()

        for token in KEYWORDS

    )

]

print("\n" + "=" * 72)
print("HISTORICAL TRAJECTORY CSV CANDIDATES")
print("=" * 72)

fig07_csv_tables = {}

csv_inventory = []

for path in candidate_csv_files:

    try:

        df = pd.read_csv(
            path,
            low_memory=False
        )

    except Exception as exc:

        print(
            "Could not inspect:",
            path.name,
            str(exc)
        )

        continue

    fig07_csv_tables[str(path)] = df

    csv_inventory.append({

        "filename": path.name,

        "path": str(path),

        "n_rows": len(df),

        "n_columns": len(df.columns),

        "columns": " | ".join(
            map(str, df.columns)
        ),

        "sha256": sha256_file(path)

    })

    print("\n" + "=" * 72)

    print("FILE:", path.name)

    print("PATH:", path)

    print("SHAPE:", df.shape)

    print("COLUMNS:", df.columns.tolist())

    print("=" * 72)

    display(df.head(5))

    # Display exact sensor, year, forest and
    # spatial unit categories when available.

    for col in df.columns:

        col_name = str(col).lower()

        if col_name in [

            "year",
            "sensor",
            "mission",
            "forest_type",
            "unit_id",
            "spatial_scale",
            "index",
            "spectral_index",
            "season",
            "scenario"

        ]:

            values = (
                df[col]
                .dropna()
                .unique()
            )

            if len(values) <= 25:

                print(
                    f"{col}:",
                    values.tolist()
                )

            elif pd.api.types.is_numeric_dtype(
                df[col]
            ):

                print(
                    f"{col}: "
                    f"{df[col].min()}–"
                    f"{df[col].max()}"
                )

# ============================================================
# 7. IDENTIFY TABLES WITH FIGURE 7 STRUCTURE
# ============================================================

print("\n" + "=" * 72)
print("FIGURE 07 — PLOTTING TABLE CANDIDATES")
print("=" * 72)

# The original figure requires:
# - year
# - sensor
# - forest or spatial unit
# - spectral index values
# - observation counts
#
# This step only reports potential sources.
# It does not aggregate or transform observations.

INDEX_NAMES = [

    "ndvi",
    "evi",
    "ndmi",
    "nbr",
    "nbr2"

]

def inspect_plotting_schema(df):

    columns = [

        str(c).lower().strip()

        for c in df.columns

    ]

    has_year = any(

        "year" in c

        for c in columns

    )

    has_sensor = any(

        any(

            token in c

            for token in [

                "sensor",
                "mission",
                "satellite"

            ]

        )

        for c in columns

    )

    has_spatial_unit = any(

        any(

            token in c

            for token in [

                "forest",
                "unit_id",
                "fragment",
                "spatial"

            ]

        )

        for c in columns

    )

    has_wide_indices = all(

        any(

            idx == c
            or c.startswith(idx + "_")
            or c.endswith("_" + idx)

            for c in columns

        )

        for idx in INDEX_NAMES

    )

    has_long_indices = any(

        c in [

            "index",
            "index_name",
            "spectral_index",
            "variable",
            "metric"

        ]

        for c in columns

    )

    has_counts = any(

        any(

            token in c

            for token in [

                "count",
                "n_scene",
                "n_acquisition",
                "n_obs",
                "n_image",
                "acquisition",
                "sample_size"

            ]

        )

        for c in columns

    )

    return {

        "has_year": has_year,

        "has_sensor": has_sensor,

        "has_spatial_unit": has_spatial_unit,

        "has_wide_indices": has_wide_indices,

        "has_long_indices": has_long_indices,

        "has_counts": has_counts

    }


schema_results = []

for sheet, df in (
    fig07_workbook_tables.items()
):

    result = inspect_plotting_schema(df)

    schema_results.append({

        "source_type": "Workbook",

        "source": sheet,

        "n_rows": len(df),

        **result

    })

for path, df in (
    fig07_csv_tables.items()
):

    result = inspect_plotting_schema(df)

    schema_results.append({

        "source_type": "CSV",

        "source": Path(path).name,

        "n_rows": len(df),

        **result

    })

schema_audit = pd.DataFrame(
    schema_results
)

display(schema_audit)

# Report candidates containing temporal,
# sensor and spatial identifiers.

likely_candidates = schema_audit.loc[

    schema_audit["has_year"]

    & schema_audit["has_sensor"]

    & schema_audit["has_spatial_unit"]

    & (

        schema_audit["has_wide_indices"]

        | schema_audit["has_long_indices"]

    )

].copy()

print("\nLIKELY FIGURE 07 SOURCE TABLES")

display(likely_candidates)

# ============================================================
# 8. INSPECT LIKELY WORKBOOK TABLES IN DETAIL
# ============================================================

for sheet, df in (
    fig07_workbook_tables.items()
):

    schema = inspect_plotting_schema(df)

    if not (

        schema["has_year"]

        and schema["has_sensor"]

        and schema["has_spatial_unit"]

    ):

        continue

    print("\n" + "=" * 72)

    print("DETAILED SOURCE:", sheet)

    print("=" * 72)

    print("SHAPE:", df.shape)

    print("\nDATA TYPES")

    print(df.dtypes)

    print("\nFIRST 15 ROWS")

    display(df.head(15))

    print("\nLAST 10 ROWS")

    display(df.tail(10))

    print("\nMISSING VALUES")

    display(
        df.isna().sum().to_frame(
            "missing"
        )
    )

# ============================================================
# 9. PRESERVE ORIGINAL WORKBOOK AND CSV SOURCES
# ============================================================

# Copy the workbook byte-for-byte.
# Original pipeline files remain untouched.

workbook_copy = (
    SOURCE / workbook_path.name
)

shutil.copy2(
    workbook_path,
    workbook_copy
)

assert sha256_file(
    workbook_copy
) == workbook_hash

source_manifest = [

    {

        "source_type": "Workbook",

        "original_path": str(
            workbook_path
        ),

        "copied_path": str(
            workbook_copy
        ),

        "sha256": workbook_hash,

        "size_bytes": (
            workbook_path.stat().st_size
        )

    }

]

# Preserve each candidate CSV byte-for-byte.
# Retain relative paths to avoid name collisions.

for path in candidate_csv_files:

    relative = path.relative_to(
        REMOTE
    )

    destination = (
        SOURCE
        / "ORIGINAL_CSV"
        / relative
    )

    destination.parent.mkdir(

        parents=True,

        exist_ok=True

    )

    shutil.copy2(
        path,
        destination
    )

    digest = sha256_file(path)

    assert sha256_file(
        destination
    ) == digest

    source_manifest.append({

        "source_type": "CSV",

        "original_path": str(path),

        "copied_path": str(
            destination
        ),

        "sha256": digest,

        "size_bytes": (
            path.stat().st_size
        )

    })

# ============================================================
# 10. EXPORT SOURCE AUDIT
# ============================================================

pd.DataFrame(
    sheet_inventory
).to_csv(

    SOURCE / "FIG07_WORKBOOK_INVENTORY.csv",

    index=False

)

pd.DataFrame(
    csv_inventory
).to_csv(

    SOURCE / "FIG07_CSV_INVENTORY.csv",

    index=False

)

schema_audit.to_csv(

    SOURCE / "FIG07_SCHEMA_AUDIT.csv",

    index=False

)

likely_candidates.to_csv(

    SOURCE / "FIG07_LIKELY_SOURCE_TABLES.csv",

    index=False

)

pd.DataFrame(
    source_manifest
).to_csv(

    SOURCE / "FIG07_SOURCE_MANIFEST.csv",

    index=False

)

metadata = {

    "figure": "Figure 7",

    "design": "A",

    "title": (
        "Historical native Landsat "
        "spectral trajectories"
    ),

    "original_workbook": str(
        workbook_path
    ),

    "workbook_sha256": workbook_hash,

    "workbook_sheets": (
        historical_workbook.sheet_names
    ),

    "original_csv_candidates": len(
        candidate_csv_files
    ),

    "likely_source_tables": (
        likely_candidates.to_dict(
            orient="records"
        )
    ),

    "original_scientific_files_modified": False,

    "spectral_indices_recalculated": False,

    "annual_aggregation_recalculated": False,

    "cross_sensor_harmonization_applied": False,

    "source_selection_finalized": False

}

with open(

    SOURCE / "FIG07_SOURCE_RECOVERY.json",

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        metadata,

        f,

        indent=2,

        ensure_ascii=False,

        default=str

    )

# ============================================================
# 11. FINAL RECOVERY REPORT
# ============================================================

print("\n" + "=" * 72)

print("SATIPO — FIGURE 07 SOURCE RECOVERY")

print("=" * 72)

print(
    "Historical workbook: RECOVERED"
)

print(
    "Workbook integrity: VERIFIED"
)

print(
    "Workbook sheets:",
    len(fig07_workbook_tables)
)

print(
    "Historical CSV candidates:",
    len(candidate_csv_files)
)

print(
    "Likely plotting tables:",
    len(likely_candidates)
)

print(
    "Original files preserved:",
    len(source_manifest)
)

print(
    "Scientific recalculation: NONE"
)

print(
    "Original pipeline modified: NO"
)

print(
    "Cross-sensor harmonization: NONE"
)

print("\nSOURCE DIRECTORY:")

print(SOURCE)

print("\nCELL 108A COMPLETED SUCCESSFULLY")

print(
    "Next step: identify the exact original "
    "annual native Landsat plotting table "
    "for Cell 108B."
)


In [ ]:

# ============================================================
# CELL 108B — FIGURE 07 FINAL PUBLICATION VISUALIZATION
# Design A — Historical native Landsat trajectories
#
# Original source:
# SATIPO_LANDSAT_HISTORICAL_TRAJECTORIES.xlsx
# Sheet: Annual_primary
#
# No scientific recalculation.
# No interpolation.
# No smoothing.
# No cross-sensor harmonization.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

from matplotlib.lines import Line2D
from matplotlib.ticker import (
    MaxNLocator,
    FormatStrFormatter,
    MultipleLocator
)

from matplotlib.patches import Rectangle

import hashlib
import json
import zipfile
import shutil

from PIL import Image
from IPython.display import display


# ============================================================
# 1. PROJECT PATHS
# ============================================================

ROOT = Path(
    "/content/drive/MyDrive/SATIPO_FOREST_PROJECT"
)

OUT = (
    ROOT
    / "10_PUBLICATION_FIGURES"
    / "FIG_07"
)

SOURCE = OUT / "SOURCE_DATA"

EXPORT = OUT / "FINAL"

EXPORT.mkdir(
    parents=True,
    exist_ok=True
)

SOURCE.mkdir(
    parents=True,
    exist_ok=True
)

WORKBOOK = (
    SOURCE
    / "SATIPO_LANDSAT_HISTORICAL_TRAJECTORIES.xlsx"
)

# Original pipeline fallback.

if not WORKBOOK.exists():

    WORKBOOK = (
        ROOT
        / "04_REMOTE_SENSING"
        / "08_OUTPUTS"
        / "TABLES"
        / "SATIPO_LANDSAT_HISTORICAL_TRAJECTORIES.xlsx"
    )

assert WORKBOOK.exists(), (
    "Original historical Landsat workbook not found."
)

SHEET = "Annual_primary"

print("=" * 72)
print("SATIPO — FIGURE 07")
print("DESIGN A — FINAL PUBLICATION FIGURE")
print("=" * 72)

print("Source:", WORKBOOK)
print("Sheet:", SHEET)


# ============================================================
# 2. VERIFY ORIGINAL SOURCE INTEGRITY
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            h.update(chunk)

    return h.hexdigest()


EXPECTED_SHA256 = (
    "aeb20a3e22a63cc86e3207461684b8e4"
    "abe224525f177a0955b9b0dd507a2098"
)

actual_sha256 = sha256_file(WORKBOOK)

assert actual_sha256 == EXPECTED_SHA256, (
    "Workbook integrity mismatch. "
    "The source differs from the audited original."
)

print("Original workbook SHA256: VERIFIED")


# ============================================================
# 3. LOAD EXACT ORIGINAL PLOTTING TABLE
# ============================================================

fig07_data = pd.read_excel(

    WORKBOOK,

    sheet_name=SHEET

)

required_columns = [

    "scenario",
    "spatial_scale",
    "sensor",
    "year",
    "index",
    "n_scenes",
    "adequate_support",
    "secondary_median",
    "mature_median",
    "median_coverage"

]

missing_columns = [

    c for c in required_columns

    if c not in fig07_data.columns

]

assert not missing_columns, (
    f"Missing source columns: {missing_columns}"
)

assert fig07_data.shape == (330, 17), (
    f"Unexpected source dimensions: "
    f"{fig07_data.shape}"
)

assert set(
    fig07_data["scenario"]
) == {"Primary_fragment"}

assert set(
    fig07_data["spatial_scale"]
) == {"Fragment"}

assert not fig07_data.duplicated(

    subset=[
        "sensor",
        "year",
        "index"
    ]

).any()

print(
    "Original annual table:",
    fig07_data.shape
)


# ============================================================
# 4. ORIGINAL CATEGORY DEFINITIONS
# ============================================================

INDEX_ORDER = [

    "NDVI",
    "EVI",
    "NDMI",
    "NBR",
    "NBR2"

]

SENSOR_ORDER = [

    "Landsat_5",
    "Landsat_7",
    "Landsat_8",
    "Landsat_9"

]

SENSOR_LABELS = {

    "Landsat_5": "Landsat 5",

    "Landsat_7": "Landsat 7",

    "Landsat_8": "Landsat 8",

    "Landsat_9": "Landsat 9"

}

FOREST_COLUMNS = [

    "secondary_median",

    "mature_median"

]

FOREST_LABELS = {

    "secondary_median":
        "Late-secondary forest",

    "mature_median":
        "Mature forest"

}

assert set(
    fig07_data["index"]
) == set(INDEX_ORDER)

assert set(
    fig07_data["sensor"]
) == set(SENSOR_ORDER)

assert fig07_data["year"].min() == 1985

assert fig07_data["year"].max() == 2025

assert fig07_data["year"].nunique() == 40

assert (
    fig07_data["median_coverage"] >= 0.70
).all()

assert (
    fig07_data["n_scenes"] >= 1
).all()


# ============================================================
# 5. VERIFY ORIGINAL SUPPORT FLAGS
# ============================================================

# Preserve the original adequate_support field.
# Convert its representation only, if necessary.

if pd.api.types.is_bool_dtype(
    fig07_data["adequate_support"]
):

    support_flags = (
        fig07_data["adequate_support"]
        .copy()
    )

else:

    support_flags = (

        fig07_data["adequate_support"]

        .astype(str)

        .str.strip()

        .str.lower()

        .map({

            "true": True,

            "false": False,

            "1": True,

            "0": False

        })

    )

assert support_flags.notna().all()

support_flags = support_flags.astype(bool)

# Confirm consistency with original acquisition counts.

assert np.array_equal(

    support_flags.to_numpy(),

    (
        fig07_data["n_scenes"] >= 3
    ).to_numpy()

), "Original support flags do not match scene counts."

# Work on a separate plotting copy.

plot_data = fig07_data.copy()

plot_data["adequate_support"] = support_flags

# No imputation or removal of spectral observations.

for col in FOREST_COLUMNS:

    assert pd.api.types.is_numeric_dtype(
        plot_data[col]
    )

    assert np.isfinite(
        plot_data[col]
    ).all(), (
        f"Nonfinite observations found in {col}"
    )


# ============================================================
# 6. VERIFY SENSOR-YEAR SUPPORT
# ============================================================

sensor_year = (

    plot_data[

        [
            "sensor",
            "year",
            "n_scenes",
            "adequate_support"
        ]

    ]

    .drop_duplicates()

    .sort_values(
        ["sensor", "year"]
    )

)

assert len(sensor_year) == 66

assert sensor_year["n_scenes"].sum() == 219

support_audit = (

    sensor_year

    .groupby("sensor")

    .agg(

        n_years=(
            "year",
            "nunique"
        ),

        adequate_years=(
            "adequate_support",
            "sum"
        ),

        n_scenes=(
            "n_scenes",
            "sum"
        ),

        first_year=(
            "year",
            "min"
        ),

        last_year=(
            "year",
            "max"
        )

    )

    .reindex(SENSOR_ORDER)

)

EXPECTED_SUPPORT = {

    "Landsat_5": (25, 15),

    "Landsat_7": (24, 18),

    "Landsat_8": (13, 12),

    "Landsat_9": (4, 0)

}

for sensor, expected in (
    EXPECTED_SUPPORT.items()
):

    actual = (

        int(
            support_audit.loc[
                sensor,
                "n_years"
            ]
        ),

        int(
            support_audit.loc[
                sensor,
                "adequate_years"
            ]
        )

    )

    assert actual == expected, (
        f"Unexpected support for {sensor}: "
        f"{actual}"
    )

print("\nORIGINAL SENSOR SUPPORT")

display(support_audit)

print(
    "Total sensor-year combinations:",
    len(sensor_year)
)

print(
    "Adequately supported sensor-years:",
    sensor_year["adequate_support"].sum()
)

print(
    "Original acquisitions:",
    sensor_year["n_scenes"].sum()
)


# ============================================================
# 7. EXPORT EXACT PLOTTING SOURCE
# ============================================================

plot_data.to_csv(

    SOURCE / "FIG07_EXACT_ANNUAL_PRIMARY.csv",

    index=False,

    float_format="%.17g"

)

support_audit.to_csv(

    SOURCE / "FIG07_SENSOR_SUPPORT.csv"

)

# Preserve a byte-identical original workbook.

source_workbook_copy = (

    SOURCE

    / "SATIPO_LANDSAT_HISTORICAL_TRAJECTORIES.xlsx"

)

if WORKBOOK.resolve() != (
    source_workbook_copy.resolve()
):

    shutil.copy2(

        WORKBOOK,

        source_workbook_copy

    )

assert sha256_file(
    source_workbook_copy
) == EXPECTED_SHA256


# ============================================================
# 8. PUBLICATION STYLE
# ============================================================

COLORS = {

    "Landsat_5": "#426B91",

    "Landsat_7": "#BC7953",

    "Landsat_8": "#287D78",

    "Landsat_9": "#8764A3"

}

TEXT = "#26333D"

MUTED = "#66747E"

GRID = "#E4E9ED"

SPINE = "#A4AFB7"

CENSUS = "#B16B72"

SECONDARY_HEADER = "#F2E9E4"

MATURE_HEADER = "#E5EFEB"

SECONDARY_TEXT = "#855639"

MATURE_TEXT = "#285E51"

mpl.rcParams.update({

    "font.family": "DejaVu Sans",

    "font.size": 9.5,

    "axes.labelsize": 10,

    "axes.titlesize": 11,

    "axes.titleweight": "semibold",

    "axes.labelcolor": TEXT,

    "text.color": TEXT,

    "xtick.color": MUTED,

    "ytick.color": MUTED,

    "xtick.labelsize": 8.5,

    "ytick.labelsize": 8.5,

    "axes.linewidth": 0.7,

    "axes.edgecolor": SPINE,

    "axes.spines.top": False,

    "axes.spines.right": False,

    "axes.unicode_minus": True,

    "legend.frameon": False,

    "pdf.fonttype": 42,

    "ps.fonttype": 42,

    "svg.fonttype": "none",

    "savefig.facecolor": "white",

    "figure.facecolor": "white"

})


# ============================================================
# 9. SHARED VERTICAL LIMITS BY SPECTRAL INDEX
# ============================================================

# Both forests use identical y limits within each row.
# Limits are derived from the complete original data.
# No observation is clipped.

INDEX_LIMITS = {}

for idx in INDEX_ORDER:

    d = plot_data.loc[
        plot_data["index"] == idx
    ]

    values = d[
        FOREST_COLUMNS
    ].to_numpy().ravel()

    ymin = float(values.min())

    ymax = float(values.max())

    span = ymax - ymin

    pad = max(
        span * 0.08,
        0.015
    )

    lower = ymin - pad

    upper = ymax + pad

    INDEX_LIMITS[idx] = (

        lower,

        upper

    )

print("\nSHARED INDEX LIMITS")

for idx, limits in INDEX_LIMITS.items():

    print(

        idx,

        f"{limits[0]:.4f}",

        f"{limits[1]:.4f}"

    )


# ============================================================
# 10. CREATE THE EDITORIAL FIGURE
# ============================================================

fig, axes = plt.subplots(

    nrows=5,

    ncols=2,

    figsize=(13.4, 13.0),

    sharex=True,

    sharey="row"

)

fig.subplots_adjust(

    left=0.095,

    right=0.975,

    bottom=0.095,

    top=0.835,

    wspace=0.10,

    hspace=0.20

)


# ============================================================
# 11. MAIN TITLE AND SUBTITLE
# ============================================================

fig.text(

    0.535,

    0.975,

    "Historical spectral trajectories",

    ha="center",

    va="top",

    fontsize=19,

    fontweight="bold",

    color=TEXT

)

fig.text(

    0.535,

    0.947,

    "Native Landsat observations, 1985–2025",

    ha="center",

    va="top",

    fontsize=11.5,

    color=MUTED

)

fig.text(

    0.535,

    0.925,

    "May–September  |  Paired valid coverage ≥70%",

    ha="center",

    va="top",

    fontsize=9.5,

    color=MUTED

)


# ============================================================
# 12. GLOBAL SENSOR LEGEND
# ============================================================

sensor_handles = []

for sensor in SENSOR_ORDER:

    sensor_handles.append(

        Line2D(

            [0],

            [0],

            marker="o",

            linestyle="none",

            markerfacecolor=COLORS[sensor],

            markeredgecolor=COLORS[sensor],

            markersize=6,

            label=SENSOR_LABELS[sensor]

        )

    )

fig.legend(

    handles=sensor_handles,

    loc="upper center",

    bbox_to_anchor=(0.535, 0.908),

    ncol=4,

    fontsize=9.5,

    columnspacing=2.6,

    handletextpad=0.45,

    frameon=False

)


# ============================================================
# 13. FOREST COLUMN HEADERS
# ============================================================

header_colors = [

    SECONDARY_HEADER,

    MATURE_HEADER

]

header_text_colors = [

    SECONDARY_TEXT,

    MATURE_TEXT

]

for col, forest_col in enumerate(
    FOREST_COLUMNS
):

    pos = axes[0, col].get_position()

    header_y = 0.853

    header_h = 0.026

    header = Rectangle(

        (pos.x0, header_y),

        pos.width,

        header_h,

        transform=fig.transFigure,

        facecolor=header_colors[col],

        edgecolor="none",

        zorder=1

    )

    fig.add_artist(header)

    fig.text(

        pos.x0 + pos.width / 2,

        header_y + header_h / 2,

        FOREST_LABELS[forest_col],

        ha="center",

        va="center",

        fontsize=11,

        fontweight="semibold",

        color=header_text_colors[col],

        zorder=2

    )


# ============================================================
# 14. DRAW SAME-SENSOR CONNECTIONS
# ============================================================

# Visual connections are permitted only between
# consecutive calendar years of the same sensor,
# when both observations have adequate support.
#
# This avoids connecting different sensors or
# visually bridging missing/low-support years.

def draw_supported_connections(

    ax,

    data,

    value_col,

    color

):

    d = data.sort_values(
        "year"
    ).reset_index(drop=True)

    for k in range(len(d) - 1):

        row1 = d.iloc[k]

        row2 = d.iloc[k + 1]

        consecutive = (

            int(row2["year"])

            - int(row1["year"])

        ) == 1

        supported = (

            bool(
                row1["adequate_support"]
            )

            and bool(
                row2["adequate_support"]
            )

        )

        if consecutive and supported:

            ax.plot(

                [

                    row1["year"],

                    row2["year"]

                ],

                [

                    row1[value_col],

                    row2[value_col]

                ],

                color=color,

                linewidth=1.0,

                alpha=0.48,

                solid_capstyle="round",

                zorder=2

            )


# ============================================================
# 15. PLOT ALL TEN PANELS
# ============================================================

plotted_records = []

for row, idx in enumerate(
    INDEX_ORDER
):

    for col, value_col in enumerate(
        FOREST_COLUMNS
    ):

        ax = axes[row, col]

        d_index = plot_data.loc[

            plot_data["index"] == idx

        ].copy()

        ax.set_facecolor("white")

        # Subtle horizontal reference grid.

        ax.set_axisbelow(True)

        ax.grid(

            axis="y",

            color=GRID,

            linewidth=0.65,

            alpha=0.95

        )

        ax.grid(

            axis="x",

            color=GRID,

            linewidth=0.5,

            alpha=0.45

        )

        # Original field census reference.

        ax.axvline(

            2019,

            color=CENSUS,

            linestyle=(0, (3, 3)),

            linewidth=1.15,

            alpha=0.95,

            zorder=1

        )

        # Each sensor is plotted independently.

        for sensor in SENSOR_ORDER:

            d_sensor = (

                d_index.loc[

                    d_index["sensor"] == sensor

                ]

                .sort_values("year")

                .copy()

            )

            color = COLORS[sensor]

            # Same-sensor connections only.

            draw_supported_connections(

                ax=ax,

                data=d_sensor,

                value_col=value_col,

                color=color

            )

            adequate = d_sensor.loc[

                d_sensor["adequate_support"]

            ]

            limited = d_sensor.loc[

                ~d_sensor["adequate_support"]

            ]

            # Adequate support: filled circles.

            ax.scatter(

                adequate["year"],

                adequate[value_col],

                s=29,

                marker="o",

                facecolors=color,

                edgecolors="white",

                linewidths=0.45,

                alpha=0.98,

                zorder=4

            )

            # Limited support: open circles.

            ax.scatter(

                limited["year"],

                limited[value_col],

                s=25,

                marker="o",

                facecolors="white",

                edgecolors=color,

                linewidths=0.95,

                alpha=0.85,

                zorder=3

            )

            # Record each original plotted observation.

            for _, record in d_sensor.iterrows():

                plotted_records.append({

                    "panel": chr(

                        97 + row * 2 + col

                    ),

                    "forest": FOREST_LABELS[
                        value_col
                    ],

                    "sensor": sensor,

                    "year": int(
                        record["year"]
                    ),

                    "index": idx,

                    "value": float(
                        record[value_col]
                    ),

                    "n_scenes": int(
                        record["n_scenes"]
                    ),

                    "adequate_support": bool(
                        record["adequate_support"]
                    )

                })

        # Shared y limits by index.

        ax.set_ylim(
            INDEX_LIMITS[idx]
        )

        ax.set_xlim(
            1984.3,
            2025.8
        )

        ax.yaxis.set_major_locator(

            MaxNLocator(
                nbins=5
            )

        )

        ax.yaxis.set_major_formatter(

            FormatStrFormatter("%.2f")

        )

        ax.xaxis.set_major_locator(

            MultipleLocator(5)

        )

        ax.tick_params(

            axis="both",

            direction="out",

            length=3,

            width=0.65,

            pad=4

        )

        # Panel labels (a–j).

        letter = chr(
            97 + row * 2 + col
        )

        ax.text(

            0.018,

            0.955,

            f"({letter})",

            transform=ax.transAxes,

            ha="left",

            va="top",

            fontsize=10,

            fontweight="bold",

            color=TEXT,

            bbox={

                "facecolor": "white",

                "edgecolor": "none",

                "pad": 1.5,

                "alpha": 0.90

            },

            zorder=6

        )

        # Show index labels once per row.

        if col == 0:

            ax.set_ylabel(

                idx,

                fontsize=11,

                fontweight="semibold",

                labelpad=12

            )

        else:

            ax.tick_params(

                axis="y",

                labelleft=False

            )

        # Only bottom row displays year labels.

        if row < len(INDEX_ORDER) - 1:

            ax.tick_params(

                axis="x",

                labelbottom=False

            )

        else:

            ax.set_xticks(

                np.arange(

                    1985,

                    2026,

                    5

                )

            )

            ax.tick_params(

                axis="x",

                labelbottom=True

            )

        # Minimal spines.

        for spine in ["left", "bottom"]:

            ax.spines[spine].set_color(
                SPINE
            )

            ax.spines[spine].set_linewidth(
                0.65
            )

        ax.spines["top"].set_visible(False)

        ax.spines["right"].set_visible(False)


# ============================================================
# 16. SHARED YEAR LABEL
# ============================================================

fig.text(

    0.535,

    0.064,

    "Year",

    ha="center",

    va="center",

    fontsize=11,

    fontweight="medium",

    color=TEXT

)


# ============================================================
# 17. GLOBAL SUPPORT LEGEND
# ============================================================

support_handles = [

    Line2D(

        [0],

        [0],

        marker="o",

        linestyle="none",

        markerfacecolor=MUTED,

        markeredgecolor=MUTED,

        markersize=5.5,

        label="≥3 acquisitions"

    ),

    Line2D(

        [0],

        [0],

        marker="o",

        linestyle="none",

        markerfacecolor="white",

        markeredgecolor=MUTED,

        markeredgewidth=1.0,

        markersize=5.5,

        label="1–2 acquisitions"

    ),

    Line2D(

        [0],

        [0],

        color=CENSUS,

        linestyle=(0, (3, 3)),

        linewidth=1.3,

        label="2019 field census"

    )

]

fig.legend(

    handles=support_handles,

    loc="lower center",

    bbox_to_anchor=(0.535, 0.025),

    ncol=3,

    fontsize=9,

    columnspacing=2.5,

    handletextpad=0.6,

    frameon=False

)


# ============================================================
# 18. VERIFY ALL PLOTTED VALUES
# ============================================================

plotted_data = pd.DataFrame(
    plotted_records
)

# 330 annual index records × 2 forests.

assert len(plotted_data) == 660

assert not plotted_data.duplicated(

    subset=[

        "forest",

        "sensor",

        "year",

        "index"

    ]

).any()

for forest_col in FOREST_COLUMNS:

    original = (

        plot_data[

            [

                "sensor",

                "year",

                "index",

                forest_col

            ]

        ]

        .rename(

            columns={

                forest_col: "value"

            }

        )

        .sort_values(

            [

                "sensor",

                "year",

                "index"

            ]

        )

        .reset_index(drop=True)

    )

    recovered = (

        plotted_data.loc[

            plotted_data["forest"]

            == FOREST_LABELS[forest_col],

            [

                "sensor",

                "year",

                "index",

                "value"

            ]

        ]

        .sort_values(

            [

                "sensor",

                "year",

                "index"

            ]

        )

        .reset_index(drop=True)

    )

    pd.testing.assert_frame_equal(

        original,

        recovered,

        check_dtype=False,

        check_exact=True

    )

assert (

    plotted_data["adequate_support"].sum()

    == 450

)

assert (

    (~plotted_data["adequate_support"]).sum()

    == 210

)

plotted_data.to_csv(

    SOURCE / "FIG07_PLOTTED_OBSERVATIONS.csv",

    index=False,

    float_format="%.17g"

)

print("\nPLOTTING INTEGRITY")

print("Original annual records: 330")

print("Total plotted observations: 660")

print("Filled symbols: 450")

print("Open symbols: 210")

print("Original spectral values: VERIFIED")


# ============================================================
# 19. EXPORT PUBLICATION FILES
# ============================================================

BASENAME = (

    "Fig_07_Historical_Spectral_Trajectories"

    "_Design_A"

)

png_path = EXPORT / f"{BASENAME}.png"

pdf_path = EXPORT / f"{BASENAME}.pdf"

svg_path = EXPORT / f"{BASENAME}.svg"

tiff_path = EXPORT / f"{BASENAME}.tiff"

preview_path = EXPORT / f"{BASENAME}_Preview.png"

# High-resolution PNG.

fig.savefig(

    png_path,

    dpi=600,

    facecolor="white",

    bbox_inches="tight",

    pad_inches=0.12

)

# Vector PDF.

fig.savefig(

    pdf_path,

    facecolor="white",

    bbox_inches="tight",

    pad_inches=0.12

)

# Editable vector SVG.

fig.savefig(

    svg_path,

    facecolor="white",

    bbox_inches="tight",

    pad_inches=0.12

)

# Smaller preview for Colab.

fig.savefig(

    preview_path,

    dpi=180,

    facecolor="white",

    bbox_inches="tight",

    pad_inches=0.12

)

# Lossless RGB TIFF at 600 dpi.

with Image.open(png_path) as im:

    rgb = im.convert("RGB")

    rgb.save(

        tiff_path,

        format="TIFF",

        compression="tiff_lzw",

        dpi=(600, 600)

    )

# Display final figure.

plt.show()

plt.close(fig)


# ============================================================
# 20. VERIFY EXPORTED FILES
# ============================================================

exported_files = [

    png_path,

    pdf_path,

    svg_path,

    tiff_path,

    preview_path

]

file_audit = []

for path in exported_files:

    assert path.exists(), (
        f"Missing export: {path}"
    )

    assert path.stat().st_size > 0

    file_audit.append({

        "filename": path.name,

        "format": path.suffix,

        "size_MB": (

            path.stat().st_size

            / 1024**2

        ),

        "sha256": sha256_file(path)

    })

file_audit = pd.DataFrame(
    file_audit
)

file_audit.to_csv(

    EXPORT / "FIG07_EXPORT_MANIFEST.csv",

    index=False

)

with Image.open(tiff_path) as im:

    tiff_dimensions = im.size

    tiff_mode = im.mode

    tiff_dpi = im.info.get("dpi")

    assert tiff_mode == "RGB"

    assert tiff_dpi is not None

    assert all(

        abs(x - 600) < 1

        for x in tiff_dpi

    )

print("\nTIFF VALIDATION")

print("Dimensions:", tiff_dimensions)

print("Color mode:", tiff_mode)

print("Resolution:", tiff_dpi)

display(file_audit)


# ============================================================
# 21. FIGURE CAPTION
# ============================================================

caption = (

    "Figure 7. Historical native Landsat spectral "

    "trajectories of the late-secondary and mature "

    "forest fragments from 1985 to 2025. Rows show "

    "the normalized difference vegetation index "

    "(NDVI), enhanced vegetation index (EVI), "

    "normalized difference moisture index (NDMI), "

    "normalized burn ratio (NBR), and normalized "

    "burn ratio 2 (NBR2). Points represent annual "

    "sensor-specific medians of May–September "

    "observations retained under a minimum paired "

    "valid coverage threshold of 70%. Filled "

    "symbols indicate sensor-years supported by "

    "at least three acquisitions, whereas open "

    "symbols indicate one or two acquisitions. "

    "Thin lines connect adequately supported "

    "observations from consecutive years within "

    "the same sensor. The vertical dashed line "

    "marks the 2019 forest inventory. Spectral "

    "values are displayed in their native "

    "sensor-specific form without cross-sensor "

    "harmonization, temporal interpolation, or "

    "smoothing. Identical vertical scales are "

    "used for the two forests within each "

    "spectral index."

)

caption_path = EXPORT / "FIG07_CAPTION.txt"

caption_path.write_text(

    caption,

    encoding="utf-8"

)


# ============================================================
# 22. REPRODUCIBILITY METADATA
# ============================================================

metadata = {

    "figure": "Figure 7",

    "design": "A",

    "source_workbook": str(WORKBOOK),

    "source_sha256": actual_sha256,

    "source_sheet": SHEET,

    "source_shape": list(
        fig07_data.shape
    ),

    "scenario": "Primary_fragment",

    "spatial_scale": "Fragment",

    "season": "May–September",

    "minimum_paired_valid_coverage": 0.70,

    "first_year": 1985,

    "last_year": 2025,

    "unique_calendar_years": 40,

    "sensor_year_combinations": 66,

    "original_acquisitions": 219,

    "adequate_sensor_years": 45,

    "limited_sensor_years": 21,

    "plotted_observations": 660,

    "filled_symbols": 450,

    "open_symbols": 210,

    "indices": INDEX_ORDER,

    "sensor_colors": COLORS,

    "shared_y_limits": {

        k: list(v)

        for k, v in INDEX_LIMITS.items()

    },

    "connection_rule": (

        "Same sensor, consecutive calendar years, "

        "both observations with >=3 acquisitions"

    ),

    "field_census_year": 2019,

    "spectral_recalculation": False,

    "temporal_reaggregation": False,

    "cross_sensor_harmonization": False,

    "interpolation": False,

    "smoothing": False,

    "observations_removed": 0,

    "original_scientific_files_modified": False,

    "tiff_resolution_dpi": 600,

    "tiff_dimensions": list(
        tiff_dimensions
    ),

    "tiff_color_mode": tiff_mode

}

metadata_path = (

    EXPORT / "FIG07_REPRODUCIBILITY.json"

)

with open(

    metadata_path,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        metadata,

        f,

        indent=2,

        ensure_ascii=False

    )


# ============================================================
# 23. CREATE FIGURE-ONLY ZIP
# ============================================================

ZIP_PATH = (

    ROOT

    / "SATIPO_FIGURE_07_DESIGN_A.zip"

)

with zipfile.ZipFile(

    ZIP_PATH,

    "w",

    compression=zipfile.ZIP_DEFLATED,

    compresslevel=6

) as z:

    for path in EXPORT.iterdir():

        if path.is_file():

            z.write(

                path,

                arcname=f"FINAL/{path.name}"

            )

    source_files = [

        "SATIPO_LANDSAT_HISTORICAL_TRAJECTORIES.xlsx",

        "FIG07_EXACT_ANNUAL_PRIMARY.csv",

        "FIG07_SENSOR_SUPPORT.csv",

        "FIG07_PLOTTED_OBSERVATIONS.csv"

    ]

    for name in source_files:

        path = SOURCE / name

        assert path.exists()

        z.write(

            path,

            arcname=f"SOURCE_DATA/{name}"

        )

with zipfile.ZipFile(
    ZIP_PATH,
    "r"
) as z:

    assert z.testzip() is None

    archived_files = z.namelist()


# ============================================================
# 24. FINAL REPORT
# ============================================================

print("\n" + "=" * 72)

print("SATIPO — FIGURE 07 FINAL PUBLICATION PACKAGE")

print("=" * 72)

print("Design: A")

print("Original workbook integrity: VERIFIED")

print("Original plotting values: VERIFIED")

print("Original annual records: 330")

print("Plotted observations: 660")

print("Panels: 10")

print("Spectral indices: 5")

print("Landsat missions: 4")

print("Original acquisitions: 219")

print("Adequate sensor-years: 45")

print("Limited sensor-years: 21")

print("Scientific recalculation: NONE")

print("Cross-sensor harmonization: NONE")

print("Original files modified: NONE")

print("Exported formats: PNG, PDF, SVG, TIFF")

print("Raster resolution: 600 dpi")

print("ZIP integrity: VERIFIED")

print("Archived files:", len(archived_files))

print(

    "ZIP size (MB):",

    round(

        ZIP_PATH.stat().st_size

        / 1024**2,

        2

    )

)

print("\nFINAL FIGURE DIRECTORY:")

print(EXPORT)

print("\nDOWNLOAD PACKAGE:")

print(ZIP_PATH)

print("\nCELL 108B COMPLETED SUCCESSFULLY")

# Optional automatic download:
#
# from google.colab import files
# files.download(str(ZIP_PATH))
